# Online softmax for long rows

When a row is too long to sit in registers (a 128k vocabulary of logits, say), the fused one-block softmax no longer works. The **online softmax** computes the max and the normaliser in a single streaming pass by rescaling the running sum each time the max grows, and then a second pass writes the probabilities. That is two reads instead of the three of a naive chunked version (max, sum, write).

Write `online_softmax(x, temperature)` returning the softmax of `x / temperature` along each row.

## Inputs
- `x`: contiguous `float32` CUDA tensor of shape `(M, N)` with **finite** entries; $N$ can be far larger than the block.
- `temperature`: Python float, $T > 0$.

## Output
- `float32` tensor `(M, N)`: $p_{ij} = e^{(x_{ij} - m_i)/T} / d_i$, where $m_i = \max_j x_{ij}$ and $d_i = \sum_j e^{(x_{ij} - m_i)/T}$.

## Requirements
- One program per row with a fixed `BLOCK` (use 1024 or smaller). Do not size `BLOCK` from `N`.
- **Pass 1** keeps two scalars, the running max $m$ and normaliser $d$, and per chunk updates
  $$m' = \max\bigl(m, \max(\text{chunk})\bigr), \qquad d' = d\, e^{(m - m')/T} + \sum_{\text{chunk}} e^{(x - m')/T}$$
- **Pass 2** re-reads the row and stores $e^{(x - m)/T} / d$.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def online_softmax(x, temperature):
    """x: (M, N) float32 CUDA tensor, finite. Return softmax(x / temperature) per row, two passes per row."""
    raise NotImplementedError


@triton.jit
def _online_softmax_kernel(x_ptr, y_ptr, N, stride_row, inv_t, BLOCK: tl.constexpr):
    # pass 1: running (m, d); pass 2: write exp((x - m) * inv_t) / d
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[0.09003057330846786, 0.2447284758090973, 0.6652409434318542], [0.3333333432674408, 0.3333333432674408, 0.3333333432674408]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float32"}, 0.5], "e": {"__nd__": [[0.01587623916566372, 0.11731042712926865, 0.8668133616447449]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 0.0009765625, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21, 4.1487833311900974e-21]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.5674793720245361, 0.1996816098690033, -0.6607606410980225, -0.6718775629997253, 0.2504635155200958, -0.9028443098068237, 1.18769109249115, -0.1955714374780655, 2.2722787857055664, -2.6281285285949707, -1.442639946937561, -3.4243321418762207, -6.679845333099365, -8.929997444152832, 0.24047866463661194, -3.322035789489746, -3.020548105239868, -2.3228564262390137, 1.6708945035934448, 2.2712342739105225, 2.744068145751953, 3.699186086654663, 11.930084228515625, 1.178876280784607, 0.6174687743186951, -8.728806495666504, -6.736242771148682, 8.771803855895996, 10.983145713806152, 1.7853819131851196, -0.32112759351730347, 2.2283554077148438, -8.141119003295898, -2.1224286556243896, -1.62167489528656, 6.941997051239014, -3.964961290359497, 1.4520831108093262, 2.896409273147583, 4.321261405944824, -1.716118574142456, -2.7482776641845703, -8.707015991210938, 4.590350151062012, 4.728799819946289, -1.9597269296646118, 0.9245304465293884, -5.5842413902282715, -0.837069571018219, 2.9102795124053955, -1.933603286743164, 0.2523476779460907, 4.399903297424316, 2.4017410278320312, -2.451324701309204, 6.458953857421875, 7.098419666290283, 4.013909816741943, 2.4026763439178467, 2.8630733489990234, 3.5273988246917725, -5.050688743591309, 4.318156719207764, -3.3619115352630615, 0.8002825975418091, -0.3955196142196655, 1.8988829851150513, 7.034401893615723, 2.131483554840088, -0.4742811918258667, 2.6782310009002686, 0.46354562044143677, -6.615039825439453, -7.2445878982543945, -2.1334359645843506, 7.96367883682251, -1.1239840984344482, 2.621748924255371, 5.109438896179199, -5.3213067054748535, 5.391299247741699, 3.403937816619873, 2.138577938079834, 3.8349592685699463, -8.284767150878906, 0.6403369307518005, -6.253019332885742, 1.317952275276184, -0.7621099352836609, -1.7907516956329346, 2.270329713821411, -7.155617713928223, 0.26667356491088867, 8.693388938903809, 4.110901832580566, -3.869352340698242, 1.6594724655151367, -0.7207563519477844, 1.9738446474075317, -4.766958713531494, 1.8844445943832397, 0.002575179096311331, -4.681841850280762, 2.794448137283325, 5.4844279289245605, 4.614811897277832, -10.56239128112793, 1.392535924911499, 1.4611035585403442, -0.7765969038009644, -0.7004993557929993, 1.7081347703933716, 2.6967055797576904, -2.9048354625701904, 0.8750523328781128, -0.9463425278663635, -3.0035245418548584, 1.4903173446655273, -2.302018642425537, -7.0621819496154785, -2.1537070274353027, -7.109389305114746, 3.6658785343170166, -1.1386635303497314, 3.99746036529541, -8.139955520629883, 6.464188098907471, 1.9220174551010132, -4.5877251625061035, 8.221007347106934, -2.8804705142974854, -4.608313083648682, -0.6516376733779907, 1.6176221370697021, 3.4808826446533203, 9.377510070800781, -3.4544496536254883, -7.257417678833008, 1.2397695779800415, -5.135021686553955, -5.392439365386963, -0.23889659345149994, 5.3175129890441895, -3.4441816806793213, -3.0022993087768555, -0.9436273574829102, -1.4253442287445068, 4.446293830871582, 0.045229144394397736, -5.079237937927246, 9.590180397033691, -2.556767225265503, -6.681589126586914, 2.3609962463378906, 0.16919201612472534, 2.502643585205078, -0.3546624779701233, 0.0850246399641037, 0.6722471714019775, 3.6098148822784424, -8.896167755126953, 0.519418478012085, 0.023272976279258728, 8.8339204788208, -2.9689297676086426, 5.00221061706543, 3.7953288555145264, -2.7413742542266846, 2.429809331893921, 3.590186357498169, 5.199538707733154, 3.7112436294555664, -0.20448926091194153, 2.2571771144866943, -2.4548916816711426, -3.5911312103271484, -1.765317678451538, -2.8019630908966064, -3.2884628772735596, 6.684847831726074, -7.103389263153076, 1.5547821521759033, 1.6954697370529175, -4.608396053314209, 5.570442199707031, -3.920247793197632, 0.8978854417800903, -6.787271976470947, -5.540034294128418, -3.2884974479675293, 1.2092236280441284, 6.292786121368408, -4.8430376052856445, 3.424616575241089, -2.3005430698394775, 2.999441623687744, 2.9322831630706787, 0.8580870628356934, 1.3483366966247559, -7.47971773147583, 2.883268356323242, 5.024896621704102, -0.40548214316368103, 3.37957763671875, 3.1856296062469482, 5.0229387283325195, -3.1064751148223877, -2.0677788257598877, 3.236621856689453, 3.098186731338501, 5.757003307342529, -0.93421471118927, 0.11685889959335327, -2.8156960010528564, -0.8307033777236938, 5.676552772521973, 0.08179382234811783, 2.2295432090759277, -0.1626194715499878, -1.171138048171997, -0.0095783406868577, -0.09410758316516876, -3.9421193599700928, -3.002761125564575, -0.7615147233009338, 2.3823800086975098, -0.3170161843299866, -6.301400184631348, 10.74077320098877, 1.2617952823638916, -1.258592128753662, 0.46749597787857056, 0.01486444566398859, 0.1691826730966568, 2.352108955383301, 1.9381986856460571, 0.6935204267501831, 4.516877174377441, -3.706955909729004, 8.403059959411621, 1.1574931144714355, -3.89874267578125, -7.775594234466553, 1.2075576782226562, 3.1445181369781494, -1.3066152334213257, -0.8153395652770996, -1.5363287925720215, 2.585946559906006, 1.2659225463867188, -5.8140645027160645, 1.8391802310943604, 0.3607919216156006, 1.995523452758789, -3.34132981300354, 3.4868781566619873, 5.738986015319824, -8.519108772277832, 3.981174945831299, -0.07173801958560944, -0.18955661356449127, -0.940615177154541, -1.35161554813385, 1.0781553983688354, -1.3887057304382324, -9.86345386505127, 2.7945542335510254, 0.6948121190071106, 2.7221789360046387, -7.297420978546143, 2.020904541015625, -2.7470006942749023, 3.369987726211548, 1.2267076969146729, 6.0266947746276855, 4.718021392822266, 4.440171241760254, -8.719868659973145, 5.276473522186279, 2.786139965057373, -0.14063365757465363, 5.100886821746826, 1.0624902248382568, 1.7686158418655396, 2.1523311138153076, -1.9166371822357178, 1.2562530040740967, -5.55683708190918, 3.4473471641540527, -0.18154165148735046, 2.0210816860198975, -3.4830756187438965, -2.8263280391693115, 1.3158634901046753, 5.953649520874023, -5.882022380828857, -6.20496940612793, -2.510232448577881, 4.6997246742248535, 0.5862757563591003, 5.264795303344727, -0.859827995300293, -3.5210189819335938, -0.9128612279891968, 1.765920877456665, -4.169559001922607, 0.9183756709098816, 0.3321968615055084, -0.5846869945526123, 2.303516149520874, -5.586666584014893, -3.3449769020080566, 2.0767645835876465, 0.8080386519432068, 2.1623127460479736, 3.236231803894043, 7.927944183349609, 3.3788223266601562, 5.598857879638672, -0.014752519316971302, 3.635969877243042, 1.4627875089645386, -2.1524345874786377, -0.6118459701538086, -1.26811945438385, -2.447268009185791, -2.132012128829956, 7.896612167358398, -3.7306020259857178, -0.5906836986541748, -9.442008972167969, -2.9137685298919678, 0.7955529093742371, 4.04531717300415, -5.322547912597656, 0.2488207370042801, -2.3889987468719482, 0.3812004029750824, -3.2771685123443604, -0.07236552983522415, 1.2079739570617676, -0.13585245609283447, 6.66262149810791, 2.430325508117676, 2.5117788314819336, -7.677155494689941, 4.708813190460205, 2.4210290908813477, 6.007256984710693, -1.9833154678344727, 1.8873370885849, 9.773272514343262, 1.049704670906067, 0.9804815053939819, 2.246819257736206, 5.068029880523682, -1.8968037366867065, 1.192257285118103, -5.295132637023926, -7.016523838043213, -3.352635383605957, -3.514181613922119, -6.9206862449646, -5.419537544250488, -3.941152572631836, 1.0693176984786987, 1.3110325336456299, 3.66867733001709, 0.09012586623430252, -3.0026299953460693, -4.343867301940918, 2.506319046020508, -0.21786746382713318, 2.5879528522491455, -3.003410816192627, -4.5672926902771, -1.3402624130249023, 11.16496753692627, 0.8840841054916382, -0.11596982926130295, 0.7439441680908203, -3.8332390785217285, 7.589745044708252, -4.94725227355957, -1.4639885425567627, -0.7372041940689087, -5.391904830932617, -2.308415651321411, -3.5110809803009033, 1.1665198802947998, -1.8870258331298828, 3.09676456451416, 4.0090651512146, -8.240760803222656, 2.24062180519104, 0.43086305260658264, -2.47544264793396, 3.9580800533294678, 0.923584520816803, -3.775733709335327, -6.011331081390381, -0.41144004464149475, 2.30690336227417, -3.3505468368530273, 8.11461067199707, -7.834535598754883, 1.4058458805084229, -2.6075398921966553, 5.027538776397705, 2.4739151000976562, 1.3788797855377197, -2.2025253772735596, -3.1175601482391357, 4.840752124786377, -2.761423110961914, -0.859387993812561, -4.06098747253418, -1.0621294975280762, -9.299092292785645, 8.309946060180664, 5.034212589263916, -5.496442794799805, -5.122890949249268, -0.3961825370788574, 1.7774789333343506, 1.606045126914978, -5.658324241638184, 0.7202881574630737, -1.0737113952636719, -3.282111406326294, 4.286749362945557, 1.1067301034927368, 2.0229179859161377, 0.7183612585067749, -0.8413519263267517, 7.9805216789245605, -1.6276923418045044, -7.593630313873291, -1.3110181093215942, -6.0732221603393555, 3.809880018234253, 5.390826225280762, -2.1308376789093018, 1.8311119079589844, -1.6589021682739258, 4.728142261505127, 5.5705461502075195, 1.0996394157409668, -0.5396105647087097, -3.435211420059204, -8.165670394897461, -3.178773880004883, -1.0032652616500854, -0.8194197416305542, 2.7335357666015625, 1.929439663887024, 0.7623690366744995, 4.951929092407227, -5.897436618804932, 3.3111343383789062, -3.946789503097534, 1.7833398580551147, 4.797173976898193, 6.513219833374023, -0.5358286499977112, -1.3876368999481201, 4.069233417510986, 2.968205213546753, 1.9557181596755981, -1.5581090450286865, -4.440598964691162, -4.91241455078125, 10.917109489440918, -2.083719253540039, 0.836323082447052, 5.64666748046875, -4.340797424316406, -2.551801919937134, 4.435094833374023, 2.1776607036590576, -5.823814868927002, -2.9608497619628906, -1.113969087600708, -1.0863327980041504, 2.747812271118164, -2.001112937927246, -4.610996246337891, -1.1311451196670532, 6.240698337554932, -3.4205482006073, -0.25365927815437317, 1.9267241954803467, -0.7055000066757202, 7.3053107261657715, -5.680450916290283, -1.2627938985824585, 2.764876127243042, 2.627619504928589, 4.132790565490723, -1.1128959655761719, 3.196722984313965, -4.1340460777282715, -2.663863182067871, 3.0358529090881348, -2.2620747089385986, 4.332817077636719, 0.3847053349018097, 0.941983163356781, -1.8085178136825562, -8.892668724060059, 7.287489414215088, -4.449097633361816, 3.0457143783569336, 1.034355640411377, -1.8500233888626099, -2.6698977947235107, 2.863351345062256, 1.8410440683364868, -1.393078327178955, 2.4058914184570312, 0.40032193064689636, -0.028476789593696594, -0.7861908674240112, -0.08630689978599548, 2.495286703109741, -3.9064877033233643, -1.0800096988677979, -2.95291805267334, 0.243473619222641, 2.203944206237793, -5.418646812438965, 5.926295280456543, -7.043507099151611, -5.829231262207031, -7.190652370452881, 3.72520112991333, -1.9755265712738037, 7.2573347091674805, -1.0239923000335693, 4.316403865814209, -4.622065544128418, 1.0823564529418945, -1.8483880758285522, -3.7277064323425293, 7.810258388519287, 4.751760005950928, -1.9814956188201904, -5.979815483093262, -0.3974958658218384, -1.0924116373062134, 0.41255617141723633, -1.5432839393615723, -2.0333967208862305, 1.2809371948242188, 7.074786186218262, 3.2528789043426514, -3.167365312576294, -0.2664657235145569, -0.9218732118606567, 2.933995008468628, 0.7533679604530334, 0.5757876634597778, 2.3428447246551514, -3.847914218902588, -1.3590662479400635, -0.3284030258655548, -4.139484882354736, 3.986990213394165, 4.166701316833496, -1.026127815246582, 1.8400137424468994, -6.392551898956299, -0.47668489813804626, -0.8662453889846802, 5.343291759490967, 0.7812066674232483, 4.285094261169434, 4.0166544914245605, 1.6045647859573364, -0.9979943633079529, 0.3935321569442749, -3.263834238052368, 7.055173397064209, -0.6318876147270203, 0.4652954339981079, -1.9436943531036377, -1.095873475074768, 1.2653404474258423, 4.181263446807861, 8.54881763458252, -1.4149938821792603, -1.103154182434082, -0.3821660578250885, -12.76987361907959, -3.646587371826172, -5.062060356140137, 1.4496999979019165, 3.080512285232544, -2.705618143081665, -0.36182254552841187, 1.5803935527801514, -7.720892906188965, -4.958178997039795, 2.128291606903076, -5.590853214263916, 3.7497353553771973, 2.3578522205352783, -2.1115095615386963, -2.1240668296813965, -6.036391258239746, 0.7925829887390137, 1.4323842525482178, 3.4145188331604004, 5.543142318725586, 6.112684726715088, -0.02304200269281864, -0.7967825531959534, -3.696946382522583, 7.7678985595703125, 3.8051676750183105, -1.598183274269104, -1.4719263315200806, 1.8917770385742188, 0.46018579602241516, 1.09257173538208, -5.364506721496582, -3.5958456993103027, 1.182047724723816, -3.6523358821868896, 2.1609482765197754, 2.1576180458068848, 1.2414617538452148, 3.0234060287475586, 0.37633106112480164, -0.17953525483608246, -0.38735759258270264, -3.655977725982666, -0.11696241050958633, 1.1558383703231812, -8.502275466918945, 3.483203649520874, -3.293757438659668, -4.735438346862793, -0.4272276759147644, -0.5062137842178345, -0.08029172569513321, -3.841496229171753, 2.1679632663726807, -8.902214050292969, -6.030672550201416, -0.5616554617881775, 3.967292308807373, 1.462778925895691, 1.360223650932312, 7.755697727203369, -5.965937614440918, -2.1582088470458984, -0.6885097026824951, 3.538623332977295, -7.001023292541504, -5.349472999572754, 0.9243950247764587, -7.477298736572266, -5.390230178833008, -6.662515163421631, -7.9104905128479, 2.274272918701172, 0.18014153838157654, 3.5244266986846924, -0.9154506325721741, 6.995693206787109, -5.024775505065918, -3.1722488403320312, -2.8740580081939697, 0.9718188047409058, 0.14222674071788788, 4.732521057128906, -2.708925247192383, 0.4237387180328369, -2.77329158782959, -1.6695129871368408, 3.0134432315826416, 3.6043901443481445, -2.981903314590454, -5.6681365966796875, 8.638663291931152, -0.9034020900726318, -2.4278531074523926, 4.881864070892334, -6.867212295532227, 3.5431020259857178, -0.6096177697181702, -0.8548116087913513, -1.7542352676391602, 4.669594764709473, -0.9677808284759521, -3.9081265926361084, 1.8990727663040161, -10.343935012817383, 6.458410739898682, 0.22830460965633392, 2.1499171257019043, -4.879738807678223, -8.155330657958984, 1.8019046783447266, -5.4884233474731445, 1.014872431755066, -4.876776695251465, 3.2912797927856445, -1.6211079359054565, -4.2620720863342285, 0.3614676892757416, -5.942958354949951, 6.620043754577637, 4.214755058288574, -1.5525327920913696, -2.8373117446899414, -4.42722225189209, -3.9665167331695557, 7.430038928985596, 5.5139570236206055, 6.989181041717529, -7.868403911590576, -0.6828742027282715, -10.992241859436035, -4.325770378112793, 2.390227794647217, -8.29710865020752, 5.335766315460205, 0.3633238971233368, -5.035940170288086, 0.5398414731025696, -4.617715358734131, -1.3023428916931152, 5.132554054260254, 7.595426559448242, -4.409748077392578, -2.3360097408294678, -0.9432160258293152, -0.08847010880708694, 2.058870553970337, -2.968076705932617, -4.01871395111084, -3.3265902996063232, 1.6815309524536133, 4.365261554718018, -1.867174744606018, 4.54371976852417, -0.5158112645149231, 5.751307010650635, -5.850824356079102, 0.9887468814849854, 4.170513153076172, -1.5678621530532837, -1.1830743551254272, -3.2833080291748047, 6.739695072174072, -2.6378228664398193, 3.3218202590942383, 9.006837844848633, 1.7342808246612549, 6.7266845703125, 2.9933931827545166, 2.6485934257507324, -1.6794370412826538, -3.0960636138916016, 3.1997642517089844, -0.17021042108535767, -5.9755778312683105, 1.035581350326538, 1.2710305452346802, 1.6155188083648682, -0.031096482649445534, -1.2479215860366821, 0.3648918569087982, 0.7774245142936707, 3.492072820663452, -1.5945879220962524, 0.002240997739136219, 0.9312005043029785, 2.685638189315796, -2.202864408493042, 1.6025742292404175, -2.876288652420044, -3.613081455230713, -1.8896793127059937, -2.0152761936187744, -2.337785243988037, 0.7842492461204529, 2.341801881790161, -2.6992156505584717, 0.04587268829345703, -2.4758243560791016, -2.95131778717041, -6.041429042816162, -5.05372953414917, 2.0925889015197754, -1.3078155517578125, 1.652103066444397, -1.5170342922210693, 6.959170341491699, -0.9868581295013428, 0.29776036739349365, -2.0727484226226807, 8.42498779296875, 0.855364203453064, -2.2419772148132324, 0.8545193076133728, -5.717690467834473, -0.020486980676651, -1.7072407007217407, 2.7859089374542236, -0.05828036740422249, -4.020427703857422, -6.605632781982422, -3.50449800491333, -0.17835494875907898, 2.5734922885894775, 2.465358257293701, -3.3416967391967773, -0.07987815141677856, 2.7239747047424316, 1.314925193786621, -5.3947930335998535, -3.607084035873413, -2.476421356201172, -4.314620494842529, -4.767633438110352, -3.0009779930114746, 3.7952263355255127, 0.7920199036598206, 3.102411985397339, 2.66770339012146, 2.9918406009674072, -6.158088207244873, 0.3879643976688385, -3.9551241397857666, -2.0177347660064697, 1.2617695331573486, 4.926982402801514, 1.7276372909545898, -0.039758093655109406, -5.899855136871338, 0.07746490836143494, 5.443936824798584, 0.9586333632469177, 1.0432331562042236, -3.494394063949585, 1.3019428253173828, -4.829946994781494, -2.5280537605285645, -2.701897144317627, -1.1068557500839233, -3.5191240310668945, -1.1143827438354492, 2.254187822341919, -1.3341470956802368, 5.405445098876953, -6.237886428833008, 0.7535927891731262, -5.148542404174805, 2.1508708000183105, -5.162124156951904, -5.577277183532715, 7.3221259117126465, -7.2607903480529785, 1.370274543762207, 1.2753459215164185, 0.03899324685335159, -1.2963426113128662, -6.04155158996582, -6.164407730102539, -2.2156176567077637, -0.3720383942127228, -5.908003330230713, -2.18511700630188, 7.206718444824219, 2.682455062866211, 4.6556854248046875, 2.2074360847473145, -6.153719425201416, 0.7998560070991516, 0.9972442388534546, 4.575558662414551, 9.941422462463379, -2.3184401988983154, 0.594558596611023, 3.1637961864471436, -3.821303367614746, -4.048011302947998, 4.801324844360352, 1.7793433666229248, 2.659306764602661, 2.6278843879699707, 2.8488199710845947, -3.3000783920288086, -2.062697410583496, -3.7477827072143555, 4.456705570220947, -3.7377257347106934, -1.8094698190689087, 2.521463632583618, 2.521500587463379, -1.0266779661178589, 1.5193625688552856, 2.843080997467041, -5.16776704788208, 3.6528360843658447, 5.4568867683410645, 8.4207763671875, 2.656583070755005, -2.6084401607513428, 0.021456770598888397, -2.675809383392334, 1.9250589609146118, -1.8983776569366455, 6.376595973968506, -7.480485916137695, 0.1508033275604248, 1.8685078620910645, 0.15186913311481476, 2.8168089389801025, 0.4408860206604004, -9.13662338256836, -3.07820463180542, 8.706807136535645, -1.849745512008667, 2.0688860416412354, 3.6958167552948, 3.244936943054199, 1.306127667427063, 5.735843658447266, 0.9561222791671753, 6.35758638381958, 1.653934359550476, -2.4286141395568848, -1.3505104780197144, -1.038912296295166, -4.12293815612793, -0.8142853379249573, -0.19443440437316895, 2.42350697517395, -5.3823370933532715, 1.7080280780792236, 3.2934012413024902, 3.139784336090088, -6.2775702476501465, -0.15695258975028992, -3.46462082862854, 14.014847755432129, 3.442941427230835, -2.0900819301605225, -3.626304864883423, -3.13055682182312, 1.1539276838302612, 2.272954225540161, -1.4708524942398071, -3.338853120803833, -7.687045097351074, 3.81252384185791, 2.2847392559051514, 2.4921717643737793, -5.945617198944092, -2.364001750946045, -0.6491163969039917, 0.38604098558425903, 1.7139019966125488, 8.486248970031738, 2.8390467166900635, 3.0795516967773438, -2.6105573177337646, -2.1522324085235596, 0.9194929003715515, 5.965847492218018, -6.19295597076416, -2.8714303970336914, 10.140605926513672, 6.435704231262207, 4.221846580505371, 3.545877456665039, -4.109844207763672, -3.7653989791870117, 1.7000060081481934, -0.9749566912651062, -2.4406862258911133, -2.595024347305298, 4.871514797210693, -1.8708535432815552, 0.823459267616272, -2.844721555709839, -3.359551191329956, -1.2376714944839478, -4.0545735359191895, 4.6819963455200195, -3.5464608669281006, -1.1720080375671387, 5.11557149887085, 0.3817898631095886, -2.3234167098999023, -7.6775803565979, 3.4435434341430664, -4.574400424957275, 1.145980954170227, -2.9378809928894043, 3.5667965412139893, -2.310509204864502, 1.0594494342803955, 1.682031512260437, -5.116830348968506, 3.173417091369629, -3.0751616954803467, 1.1808509826660156, -1.4720319509506226, -1.6134487390518188, -5.766695022583008, -1.6478601694107056, -9.236541748046875, -0.0393923781812191, -2.8309848308563232, 0.8054212927818298, 5.13139009475708, -0.18594706058502197, 0.26059404015541077, 1.2888033390045166, -2.438086748123169, 2.9706079959869385, -1.7150236368179321, 0.9101308584213257, -4.4852118492126465, 0.8580554723739624, 0.9677228331565857, -2.1269543170928955, 7.200921535491943, -1.2937992811203003, 4.778321743011475, 4.374963283538818, 2.378065824508667, -0.02904583141207695, -4.808977127075195, -0.34913170337677, -2.290954351425171, 1.4877495765686035, 2.592498779296875, -3.6502020359039307, 8.724068641662598, 6.3193535804748535, -3.8128607273101807, -1.0639009475708008, 0.14234739542007446, 1.1327950954437256, 3.234057664871216, -2.2855398654937744, -3.3599941730499268, -1.5404095649719238, 3.243016004562378, 4.717167854309082, 4.626905918121338, -3.969416856765747, 5.6482625007629395, -1.3684654235839844, -3.344019651412964, 7.443937301635742, 1.9973968267440796, -1.2603461742401123, 1.2552213668823242, 4.5799102783203125, 2.2309110164642334, -0.6762116551399231, 4.757078647613525, 7.428207874298096, -4.689023971557617, 2.1839001178741455, -0.503305196762085, -3.725757598876953, 0.33983126282691956, -9.32374382019043, -4.611819267272949, 3.62019419670105, 4.949585914611816, 0.9287075996398926, -4.661895751953125, 1.9537113904953003, 1.4612410068511963, -3.7338736057281494, -5.104677200317383, -0.6870222687721252, -1.3792340755462646, 0.2775340974330902, -3.0192551612854004, -4.019295692443848, 0.8874689340591431, 2.324068307876587, 1.7931292057037354, 2.265979051589966, 4.283535957336426, -0.6439920663833618, -1.8284931182861328, -5.037574768066406, -5.656304836273193, 5.137726306915283, 3.9713988304138184, -1.7642422914505005, 2.302739143371582, 0.9095534682273865, -3.197376012802124, -1.1876733303070068, 5.642032146453857, 0.5534972548484802, 3.9370920658111572, 0.38658514618873596, -2.9493014812469482, 2.184276819229126, 0.1439797282218933, 4.604503631591797, -0.7596428990364075, 6.499518871307373, -4.068032264709473, -0.6187137961387634, -5.006340503692627, 2.022749662399292, 7.224667072296143, 2.4192111492156982, -4.925371170043945, 5.294839382171631, -2.6791763305664062, 6.189233303070068, -6.07690954208374, 3.08019757270813, -5.879885673522949, 0.8590180277824402, -2.1520655155181885, -4.797708988189697, 3.2953500747680664, -0.7771359086036682, -2.533928632736206, -0.5648025274276733, 1.8345447778701782, 6.653201580047607, -1.833601951599121, -0.25624704360961914, -1.0905450582504272, -4.175707817077637, -2.8676984310150146, -4.541668891906738, 4.435765743255615, 0.520576536655426, -0.025302384048700333, 1.144655704498291, -2.019367218017578, 4.158712863922119, 5.452089309692383, -6.603769302368164, 4.083682537078857, 7.297513484954834, -2.3099076747894287, 2.3191819190979004, 1.2032800912857056, 5.457470893859863, 1.118571162223816, -4.87314510345459, 0.6101831793785095, 3.3619184494018555, 5.776053428649902, 0.3773591220378876, -5.409859657287598, 2.438265562057495, 5.094523906707764, -3.3732452392578125, -4.120641231536865, 1.1463743448257446, 1.5447324514389038, 0.19239504635334015, -2.8125646114349365, -1.7731989622116089, -2.2141308784484863, -3.6324172019958496, 5.990015029907227, -1.071936845779419, -1.778379201889038, 2.0271923542022705, 0.7082622647285461, 5.070164203643799, -4.674895286560059, 3.8702194690704346, -3.57673978805542, 4.161278247833252, -3.080526828765869, 4.71956205368042, -0.23779959976673126, 4.254449367523193, -4.4917449951171875, 4.127530097961426, 7.251884937286377, 3.4044930934906006, -0.6593784689903259, -3.5799355506896973, 5.242781639099121, 0.6024765372276306, -1.9053550958633423, -1.494991421699524, 6.889277458190918, 3.9809060096740723, 5.332960605621338, 5.959484577178955, -3.930041790008545, -3.4562854766845703, -0.48669078946113586, 0.3359532058238983, 0.7371718287467957, 0.37450677156448364, 3.344548463821411, -2.464369297027588, 8.735151290893555, -1.9898667335510254, -3.9355969429016113, -4.219017505645752, -3.1598916053771973, -1.5745421648025513, -0.6434420347213745, 2.160179615020752, 2.460176944732666, 4.442878723144531, -1.8569031953811646, 4.708817005157471, -1.125626802444458, 1.2567250728607178, 1.1437602043151855, 2.417332887649536, 0.6053122878074646, -3.40498423576355, -1.5099859237670898, -5.014101505279541, 3.9229416847229004, -0.2262665331363678, 4.057018280029297, -1.4918547868728638, -1.410487174987793, 5.050572395324707, 1.981163740158081, -2.5351483821868896, 1.4255646467208862, 0.5362971425056458, -0.8244016170501709, -0.7964215874671936, -2.997586488723755, -7.367116928100586, 3.947824716567993, -6.5869245529174805, -0.6496155261993408, 1.532036542892456, -2.631662607192993, 6.098929405212402, -2.680154800415039, 2.409122943878174, 0.8343443274497986, -0.8479244112968445, -1.72991144657135, 2.308802604675293, 4.501896858215332, 4.201180458068848, -2.804119825363159, 8.108827590942383, -4.221721649169922, 5.742293834686279, -4.214749336242676, -1.4872709512710571, 2.170868158340454, -5.262479305267334, -3.759812831878662, -1.1243640184402466, -0.8765255212783813, 1.3797980546951294, -1.3682420253753662, 3.140577793121338, -3.5884687900543213, -3.70615291595459, 6.380594730377197, -5.430027961730957, 6.775272369384766, 1.4330909252166748, -6.50149393081665, -4.477619647979736, 2.4572196006774902, 3.2519657611846924, 2.6403191089630127, -2.141336679458618, -1.6116712093353271, -2.8487541675567627, -0.7405709028244019, -1.8726154565811157, 3.3512003421783447, 3.880467653274536, 2.1159467697143555, 4.9955244064331055, -1.4411406517028809, -2.4044902324676514, 2.8682010173797607, -2.435377836227417, 3.925661563873291, 0.15933597087860107, -1.6839210987091064, -1.892797827720642, 1.3665872812271118, -3.0365331172943115, 0.5614849925041199, 4.8670268058776855, -1.3455946445465088, 1.2227964401245117, -6.4470930099487305, 3.14070725440979, 13.924612045288086, 4.134561061859131, 7.395681381225586, -4.149280548095703, 2.069627523422241, 0.5144659876823425, 4.649516582489014, 2.7247753143310547, -0.38725101947784424, -3.9530837535858154, -0.17631269991397858, 3.942487955093384, -3.2114572525024414, 10.654979705810547, -2.16347074508667, -3.5737154483795166, -1.48225998878479, -2.1872310638427734, 9.155926704406738, 8.707084655761719, -1.518758773803711, 2.4585328102111816, 1.9830715656280518, 10.523921012878418, 4.469118118286133, -3.521176338195801, 4.679776191711426, 2.0373525619506836, -0.006707964930683374, 3.9347474575042725, -3.9768950939178467, -0.5878661870956421, 3.4831743240356445, -2.494518518447876, 2.7214818000793457, 1.801222801208496, -7.789611339569092, 2.5092105865478516, -6.00398588180542, 6.511597156524658, -5.099417686462402, -2.783780336380005, -3.494536876678467, -2.715102195739746, 0.7402400374412537, -6.905966758728027, -0.5795865654945374, 6.156010150909424, -10.229684829711914, -0.556065559387207, 7.291704177856445, 2.3614447116851807, -2.772035837173462, 3.055542469024658, -3.237420082092285, -1.0153216123580933, -1.7435197830200195, 0.8837999701499939, -2.642922878265381, 4.436962127685547, 3.855618953704834, 5.825423717498779, -0.11932025849819183, -6.800678253173828, -0.19279822707176208, -1.8136903047561646, -1.6407885551452637, -4.964327335357666, 0.29364511370658875, -1.022041916847229, -1.360055923461914, 1.9504245519638062, 1.5104074478149414, 2.7953929901123047, -6.91551399230957, -1.1160573959350586, -2.058847665786743, 1.4853057861328125, -6.130035877227783, -2.254991292953491, -1.5604246854782104, -8.873291969299316, -1.20155930519104, -1.8392071723937988, -2.5775258541107178, -6.790748596191406, 0.20392745733261108, 12.126670837402344, -1.6710824966430664, -1.683685064315796, 2.6936144828796387, -6.257249355316162, 1.3279297351837158, 2.862872838973999, 6.372895240783691, 3.1652536392211914, 3.7484774589538574, 2.194742441177368, -7.2805914878845215, -0.7400447130203247, -3.348461389541626, -2.755157709121704, 2.3294341564178467, 2.1673107147216797, -0.7900969982147217, -2.2434933185577393, 9.0393648147583, 0.7634517550468445, -4.160862445831299, 3.16790509223938, -1.0914853811264038, -0.8085177540779114, 4.132262229919434, 5.571738243103027, -2.969770908355713, -1.7522261142730713, 6.0005292892456055, -8.20764446258545, -1.0010863542556763, 1.065935492515564, -3.1766655445098877, 2.468719482421875, 0.5237431526184082, 0.821262538433075, 6.95937967300415, 0.43943583965301514, 8.69616413116455, 1.886804461479187, -0.035170797258615494, -0.013756919652223587, -0.4446136951446533, -1.1672035455703735, 4.992193698883057, -3.127014398574829, -3.7324187755584717, -9.06248664855957, -1.1751065254211426, -1.4726415872573853, 0.31547650694847107, -3.358393669128418, -1.4128501415252686, -9.969565391540527, -2.5862202644348145, -6.03525972366333, 2.9215121269226074, 2.230246067047119, -1.1833363771438599, -6.122663974761963, -7.184488296508789, 4.070893287658691, -4.862310409545898, -9.409751892089844, -3.42988920211792, 5.269082069396973, 10.839966773986816, -5.522178649902344, 0.3719005286693573, 2.5197277069091797, 8.625548362731934, 2.881500720977783, 2.8244948387145996, 2.0640246868133545, 0.4998071491718292, -4.437690258026123, 2.0328447818756104, 2.956289052963257, 4.141237258911133, -9.039775848388672, 0.31212079524993896, -1.4035027027130127, 4.137152671813965, 6.641415596008301, -4.3888397216796875, -4.880905628204346, 3.3095765113830566, 3.1749260425567627, -0.1025535985827446, -0.757241427898407, -3.5141022205352783, -5.178467273712158, -1.5606653690338135, 2.6944236755371094, 9.411822319030762, 0.3319236636161804, -2.765237331390381, 1.2378634214401245, 6.355413436889648, 2.0779948234558105, -0.5281986594200134, 2.302462100982666, 6.139636516571045, -2.44333815574646, -1.1963791847229004, 3.7554235458374023, -4.718968391418457, 1.3708614110946655, 3.3056325912475586, -2.520751714706421, -0.24418121576309204, -7.753392219543457, 5.183927059173584, -1.2754358053207397, 4.697011470794678, 5.649054527282715, 3.4522106647491455, -2.6277780532836914, 5.246140003204346, 4.0061726570129395, 5.942925453186035, 0.031348567456007004, 5.996335983276367, -5.6389923095703125, -1.909117579460144, -3.9779210090637207, -3.7472383975982666, 1.7845304012298584, 1.8406634330749512, 0.2710030972957611, 0.7732173204421997, -3.104438304901123, 3.3511667251586914, 3.7494876384735107, 1.715333104133606, -1.3776073455810547, -1.4897874593734741, -0.401258647441864, -2.3205974102020264, -0.9743034243583679, 1.7811498641967773, 0.9345428347587585, 0.2414686381816864, -0.5561274290084839, -6.456784725189209, 2.3895950317382812, -1.944006085395813, 1.4910658597946167, -10.684698104858398, 1.3984191417694092, -2.2604551315307617, 0.7456296682357788, 4.80161190032959, 1.5530436038970947, -11.688247680664062, 2.10563063621521, -4.167305946350098, 0.9689788818359375, -3.9073517322540283, 0.0012085225898772478, -3.310927152633667, 3.789911985397339, -3.453772783279419, -1.2730709314346313, 0.5623123049736023, -4.623579978942871, 1.637736439704895, 0.3460519015789032, 3.214000701904297, 2.115647554397583, -5.733591079711914, 3.0516343116760254, 0.7847192883491516, 3.125563859939575, 0.8229250311851501, -6.688389301300049, -2.4963953495025635, -7.545653820037842, -4.6199798583984375, 1.1505883932113647, 1.5130900144577026, -1.8192427158355713, 0.8240283727645874, 2.9536540508270264, 0.5373314619064331, -3.3403759002685547, -1.2680588960647583, -3.802105188369751, 4.158269882202148, -5.365673542022705, 1.5728909969329834, -3.7727930545806885, -2.1449177265167236, -2.835440158843994, 0.7824632525444031, 3.604302167892456, 6.456173419952393, 2.949418306350708, 1.488851547241211, -2.666053295135498, -3.030146837234497, -1.160438895225525, 0.406345933675766, -9.426517486572266, 8.150449752807617, 0.9874918460845947, -1.482698678970337, 2.930803060531616, -4.982537269592285, -0.702333390712738, 4.665481090545654, 7.388277530670166, 1.8673205375671387, -2.8192505836486816, 3.4450924396514893, -2.5456292629241943, 0.4987155497074127, -2.1637303829193115, 7.086857795715332, 0.5974637269973755, -3.4988811016082764, -2.154165744781494, 2.2642829418182373, 4.966980457305908, -1.375942349433899, -3.630598783493042, -6.105612277984619, 0.43728306889533997, 0.8629388213157654, -0.14446716010570526, 1.382616639137268, -5.267916202545166, -6.795093536376953, -1.1130341291427612, 6.892241477966309, 6.5080647468566895, 3.6601598262786865, 6.073751449584961, -4.047155380249023, 0.41626307368278503, 4.783798694610596, -0.6996498107910156, -1.2366218566894531, -2.1260621547698975, -3.706392288208008, 2.900036573410034, -4.155953884124756, -2.8265950679779053, -3.5439810752868652, -1.1829547882080078, 1.8285988569259644, -0.084394171833992, -0.39736634492874146, 0.3381912410259247, -1.689919352531433, 3.643089532852173, 3.122415781021118, 8.67130184173584, -2.036356210708618, -2.3892369270324707, 10.572416305541992, -0.8845421671867371, 0.9865248799324036, 0.3470189571380615, -8.369503021240234, 2.598548412322998, -7.709123134613037, -1.6881595849990845, -7.5096001625061035, 0.1907605528831482, -4.704705238342285, -0.5352625250816345, -1.1555626392364502, -3.4069149494171143, 5.653170108795166, 2.94592547416687, -5.485291957855225, -10.883878707885742, 1.319441318511963, -6.3386712074279785, -1.371429681777954, 1.2792915105819702, -0.6722732186317444, -2.385727882385254, -0.23817679286003113, -7.830713272094727, -2.268540620803833, 1.7034984827041626, 2.2457761764526367, -0.029298609122633934, 1.3315705060958862, 1.153122067451477, -2.2878079414367676, -3.6123154163360596, -5.782115459442139, -5.141838073730469, 2.7780587673187256, 0.20210233330726624, -2.7824366092681885, -1.8830024003982544, -4.553726673126221, -8.301339149475098, 2.1407806873321533, 3.388052225112915, -4.924345016479492, -6.893936634063721, 0.4220001995563507, -3.3720498085021973, -2.1793525218963623, 1.036465048789978, 2.925124406814575, 5.531796932220459, 3.4411563873291016, 1.5882407426834106, -0.10629477351903915, 5.178136348724365, -2.675751209259033, 3.5628435611724854, 7.94532585144043, 5.938393592834473, 5.353598594665527, -3.50656795501709, 2.2331185340881348, 0.9867839217185974, -1.7283657789230347, 1.000420093536377, -7.347286701202393, -4.666788101196289, 2.298144817352295, -3.50911545753479, 4.569879055023193, -4.357761859893799, 2.355912446975708, 0.3199182152748108, 1.3017009496688843, 2.563641309738159, -1.3430347442626953, -6.02418327331543, -1.5717856884002686, -10.682673454284668, 6.673178195953369, 4.842413902282715, 2.59602427482605, -0.9996032118797302, -2.6924593448638916, -5.912561893463135, -1.50409996509552, 3.3897202014923096, -2.9357240200042725, 2.249494791030884, -3.45074725151062, 1.2003438472747803, 4.8641357421875, 3.7953274250030518, -11.98929214477539, 5.056582927703857, 2.53652286529541, -3.3931355476379395, 3.138012647628784, -1.2599177360534668, -0.8793669939041138, 5.010703086853027, 4.030500888824463, 5.631251811981201, 2.542587995529175, 10.724637031555176, 3.4718315601348877, -4.890426158905029, 0.5140888094902039, -4.135262966156006, -7.658721446990967, -1.301053762435913, -1.0177110433578491, -2.308697462081909, 2.561859607696533, -1.1997325420379639, 1.2513316869735718, 2.6834824085235596, -0.5123396515846252, 3.347428798675537, -3.8366193771362305, -7.375330924987793, 6.914796829223633, -5.985328674316406, -1.4943023920059204, 1.0117502212524414, -6.969280242919922, 0.23907925188541412, 3.284132480621338, 5.5118794441223145, 0.41922128200531006, -1.446085810661316, -3.1947379112243652, 1.0403192043304443, 0.38794517517089844, -9.477119445800781, -3.603280782699585, 1.4205511808395386, 2.140909433364868, 1.7402091026306152, 4.383549213409424, -5.664680480957031, 2.804553508758545, -2.283596992492676, -2.668308973312378, -0.36303865909576416, -5.910376071929932, -0.014896932058036327, 2.1970949172973633, -7.305847644805908, -6.248519420623779, -6.958498477935791, 1.2415292263031006, 1.038146734237671, -4.660369396209717, -10.827445983886719, -0.6447449922561646, 8.515336990356445, -4.175156593322754, -3.206833839416504, -3.4076287746429443, -2.545363187789917, 5.112656593322754, -7.317787170410156, -4.13167142868042, -3.3195250034332275, -1.8434325456619263, 1.76788330078125, -3.1536335945129395, 2.4285316467285156, 1.765629529953003, -4.262821197509766, 1.2556724548339844, -5.542593955993652, -10.647089958190918, -1.929147481918335, 3.3749120235443115, 5.686441421508789, 2.318486213684082, -5.224315643310547, -2.2041385173797607, 1.7128325700759888, -0.5613656640052795, 11.55738639831543, -10.2284517288208, -4.718055248260498, 0.828785240650177, -11.385858535766602, 9.70734977722168, 0.5663891434669495, 0.03411436080932617, 5.745387554168701, -0.2593132257461548, -2.042339324951172, 3.7842938899993896, 1.3446452617645264, 4.711481094360352, -4.920857906341553, 6.107081890106201, -1.2282888889312744, 5.460719108581543, -2.5782580375671387, 1.0166789293289185, -2.038487672805786, 7.4332780838012695, -1.663270115852356, 1.661362648010254, 1.3506606817245483, -8.582695960998535, 3.573389768600464, 0.4988521337509155, 5.007653713226318, -4.57588529586792, 0.2263001948595047, -2.490586519241333, -3.922212600708008, 0.05477789789438248, 6.871890068054199, -1.0011345148086548, -6.183771133422852, -7.2752227783203125, -3.845137596130371, 4.899603843688965, -2.775007486343384, -1.9136639833450317, 5.496081829071045, -2.137058973312378, 4.000496864318848, 1.9272942543029785, -4.961672782897949, -2.2593438625335693, 2.601174831390381, 0.03320057690143585, -6.568905830383301, 2.761367082595825, 4.541067123413086, 5.700826644897461, 1.819076418876648, -4.920014381408691, 0.9144213199615479, -4.205929279327393, 4.115294933319092, 0.428331583738327, 5.121705055236816, 0.05090916156768799, -3.5761754512786865, 10.068617820739746, 0.12789669632911682, -6.287322044372559, -2.130647659301758, -5.289813041687012, -4.98341703414917, 0.3269098699092865, -4.8384199142456055, -1.8578026294708252, -6.453385829925537, 2.6283106803894043, 1.9070329666137695, -1.3538175821304321, -8.778448104858398, 4.064876556396484, -5.662007808685303, 2.8739795684814453, -1.4142180681228638, -2.975722312927246, 8.139415740966797, 1.4371156692504883, 1.934988021850586, 0.6443833112716675, 1.7998038530349731, -0.17656053602695465, -6.51173210144043, -1.9934666156768799, -4.922342300415039, 6.748470306396484, -4.510034561157227, -0.651871383190155, -2.1875593662261963, 0.6701011061668396, -2.2381114959716797, 4.425313472747803, -0.08013454079627991, -0.0653378888964653, 2.5923686027526855, -4.4356689453125, -2.2365853786468506, -7.5625529289245605, 0.22234223783016205, 2.655155897140503, 1.217002034187317, 2.089083194732666, 4.614190101623535, -0.8252583742141724, -3.678117513656616, 1.1132858991622925, 0.29690787196159363, -0.11600645631551743, 5.2342963218688965, 6.049347877502441, 6.751418113708496, 0.8899610638618469, 7.459509372711182, 3.454789400100708, 4.758577823638916, -1.2947102785110474, -4.234502792358398, -1.1356621980667114, 0.45386067032814026, 5.796436309814453, 3.4230594635009766, 1.1977046728134155, -0.8908320665359497, 1.9736549854278564, 0.4545342028141022, 1.6273926496505737, -3.9489569664001465, -2.419139862060547, 4.7486724853515625, -5.81541633605957, 0.8782235980033875, -7.215085029602051, -3.330411672592163, 3.495741605758667, 2.269007921218872, 1.7318090200424194, 1.9335715770721436, -2.2283999919891357, -8.555006980895996, 0.06065910682082176, 1.5131430625915527, -0.8849525451660156, -3.3739442825317383, -1.3301472663879395, 1.540177822113037, -10.761433601379395, -5.452040672302246, 2.4744815826416016, 0.9437753558158875, -0.9214330911636353, 4.143890857696533, -0.16074779629707336, 3.632251739501953, 7.887497425079346, -0.7809873819351196, 3.4305155277252197, 1.8422446250915527, -7.51235818862915, -4.047491550445557, -2.6605403423309326, 5.860177993774414, -3.0861217975616455, -0.7640318870544434, -2.0127005577087402, -7.623769760131836, 2.1732664108276367, -0.97247713804245, 4.266846656799316, -0.46734073758125305, 1.0147130489349365, 2.650623321533203, 4.593570232391357, 3.7198867797851562, -3.259859800338745, 4.5559892654418945, -3.223020553588867, 5.342846870422363, 1.6084784269332886, -3.455803871154785, -3.0964372158050537, 3.703540563583374, -1.0293285846710205, 0.8176854252815247, -0.5660625100135803, 6.1828765869140625, -2.8866422176361084, 1.0056136846542358, -7.72433614730835, 1.4322737455368042, -4.643674850463867, -1.7041159868240356, 7.904592990875244, 2.2967710494995117, -4.084873199462891, -0.3286246061325073, 0.8883248567581177, -2.4842495918273926, -1.1995718479156494, -5.325623512268066, 6.253701210021973, -2.002267599105835, 1.5499573945999146, 1.7851356267929077, 2.464587688446045, 1.6226552724838257, 4.934544086456299, -5.493946552276611, 3.033823251724243, 4.038730621337891, -4.862608432769775, -1.4219157695770264, -1.6465195417404175, -0.5629113912582397, 5.834632873535156, 3.261183023452759, 1.489694595336914, 1.8212682008743286, -5.707651615142822, 5.076282978057861, -3.2754554748535156, -0.728153645992279, 6.700202465057373, -3.4340851306915283, -8.254742622375488, 1.9931882619857788, 2.8989930152893066, -0.6160255670547485, 6.466458797454834, 0.604564368724823, 0.9333739280700684, 3.3358166217803955, -4.976117134094238, -2.4687867164611816, 1.2453992366790771, 2.2584338188171387, 8.174860954284668, -2.9486749172210693, 1.8926180601119995, -1.9612969160079956, -3.5795562267303467, 6.707016468048096, 1.6448266506195068, -2.332136392593384, 6.512331962585449, -6.657138347625732, 5.984064102172852, -1.7169454097747803, -2.681227207183838, -3.878085136413574, 2.1728038787841797, 1.7133502960205078, 2.855670213699341, -7.006640434265137, 5.406508922576904, 0.045656152069568634, 2.248558282852173, -10.03207778930664, -1.9999446868896484, 1.1243385076522827, 5.385268211364746, 1.4455064535140991, -0.32609331607818604, -1.2096800804138184, -0.8700475692749023, 6.812799453735352, 8.582965850830078, -4.431519985198975, 1.4580721855163574, 7.784153938293457, 2.1485562324523926, 0.1466422975063324, -1.6870883703231812, 1.6046435832977295, 10.946562767028809, 0.9824900031089783, -2.8704776763916016, -1.1612603664398193, 2.1892149448394775, 3.5014328956604004, 1.677932858467102, -0.7507612109184265, -0.8920404314994812, 0.5249689817428589, 0.0544922761619091, 7.0644073486328125, 2.2082531452178955, 1.0296673774719238, 5.523838520050049, 1.6559488773345947, 2.846766710281372, -7.297183513641357, -0.6416099071502686, 2.1573638916015625, -1.0482975244522095, -0.7850630283355713, 4.881501197814941, -4.310402870178223, 1.9204344749450684, -0.5265887975692749, -7.000750541687012, -1.294744849205017, -0.6531049013137817, -4.932615280151367, 4.847635269165039, -2.221745729446411, -2.0747387409210205, 2.3588364124298096, 4.942697048187256, -5.757436275482178, 0.5869229435920715, 3.036224842071533, -3.276808023452759, 4.790078639984131, 6.5510382652282715, 1.7237945795059204, 0.8786940574645996, 4.3157172203063965, -1.8284813165664673, -4.49237585067749, -8.390671730041504, -5.885618209838867, -4.165548801422119, 9.089815139770508, -1.042283296585083, -1.4515665769577026, 1.9843723773956299, 3.0059974193573, -4.919268608093262, 1.2773215770721436, 4.5416083335876465, 1.592016339302063, -3.7933850288391113, 1.21329927444458, 8.333317756652832, 5.029923439025879, 2.577331066131592, 6.487137317657471, 5.741029739379883, -4.699593544006348, 5.374455451965332, 1.4070199728012085, -3.4073944091796875, -2.8051421642303467, -1.1775184869766235, 2.0785768032073975, -1.7349261045455933, 6.3834123611450195, -1.7055692672729492, 1.1448144912719727, -2.7336902618408203, -3.7878365516662598, -1.9597522020339966, -2.5175068378448486, -3.557527542114258, 2.002660036087036, 5.78717041015625, 1.512071967124939, 8.156357765197754, 2.6267757415771484, 0.30147117376327515, 5.032732963562012, -5.158117294311523, -7.784420490264893, -0.20623232424259186, 6.7678303718566895, -0.25403425097465515, 1.4852677583694458, 4.776488780975342, 8.010368347167969, -1.1132344007492065, -5.433014392852783, -5.0222368240356445, -0.20443011820316315, -3.2233004570007324, 1.164941430091858, 1.6198903322219849, -0.025106268003582954, 3.7475950717926025, -0.6506287455558777, 10.647534370422363, -4.46200704574585, 3.6165976524353027, -1.5043367147445679, 1.866972804069519, -6.894225120544434, 4.732973098754883, -1.2468798160552979, 3.820028066635132, 6.95004940032959, -5.431979656219482, -0.823724627494812, 3.523355007171631, 4.126221179962158, -4.433801651000977, 1.345823884010315, 5.326648235321045, 2.0042924880981445, 2.140500783920288, 0.7281148433685303, 3.408261299133301, 2.5279605388641357, -5.106314182281494, 3.312408685684204, -0.18786609172821045, -7.354307651519775, 0.36648455262184143, -2.0832412242889404, 2.7722556591033936, -4.513186931610107, -0.2554972469806671, -1.248545527458191, -3.170836925506592, -5.941205978393555, -4.433408260345459, 3.160813331604004, 1.049788475036621, -3.2650094032287598, 1.7197926044464111, 4.412259578704834, 2.698235034942627, 1.6453701257705688, -0.5929074883460999, 5.786446571350098, 4.17165470123291, -6.71880578994751, -0.6827935576438904, -5.9756550788879395, 8.364310264587402, -6.837276935577393, 6.641827583312988, -2.134026288986206, -4.769718170166016, 4.2039313316345215, -5.3976149559021, -4.590322494506836, -2.044321298599243, 5.614314079284668, 3.2985005378723145, 4.91796875, 4.9629950523376465, -2.1372804641723633, 1.0484565496444702, -0.17803652584552765, 1.048473834991455, -0.9970235228538513, 6.0953049659729, 2.3622353076934814, -2.72326397895813, 5.323722839355469, -0.9760727882385254, -3.1995880603790283, -1.7547123432159424, 6.042677402496338, -1.198283076286316, -2.1538476943969727, -5.0398850440979, 1.0886996984481812, 2.5366642475128174, -2.036170721054077, -0.1163502186536789, -1.8398878574371338, 8.043127059936523, -1.9133262634277344, -3.668843984603882, 0.9061962962150574, -3.0209670066833496, 2.728940486907959, 3.024014949798584, -1.823976993560791, -1.2363226413726807, 6.8697190284729, -3.7396767139434814, -1.861419677734375, 7.542415142059326, -1.7736932039260864, -3.97979998588562, -5.440239429473877, -0.376755028963089, -4.912232398986816, -0.11698876321315765, -4.5004072189331055, -2.580556869506836, 2.964782476425171, -3.906520128250122, 2.27087664604187, -7.939538955688477, -3.624708652496338, 1.692812442779541, 0.6493251919746399, -0.027322104200720787, 1.7430591583251953, -4.78243350982666, 1.9544768333435059, -2.3019871711730957, -6.783548355102539, 3.7999038696289062, -0.00962560623884201, -1.1452621221542358, -2.9731767177581787, -1.705932855606079, -1.0429459810256958, -8.544949531555176, 1.1310248374938965, 11.552108764648438, 0.5990709066390991, 4.0176777839660645, 4.993115425109863, 2.6814804077148438, 1.8058209419250488, 5.66281270980835, 1.5532095432281494, 0.5002292394638062, 2.9986379146575928, 4.822393894195557, 3.0249850749969482, 0.051037367433309555, -4.3213019371032715, 6.12349271774292, -3.030937671661377, -0.6673646569252014, -5.243350982666016, 3.28763484954834, -0.1692904829978943, 0.27854034304618835, -7.606487274169922, -1.580073356628418, 0.9312080144882202, 4.39285945892334, -2.494978427886963, 4.801512718200684, -0.4069230556488037, 5.799743175506592, 4.9508514404296875, -0.27978143095970154, -1.3453707695007324, 1.6874034404754639, -4.163302898406982, 7.4023566246032715, -2.3434746265411377, 0.11380383372306824, -4.0133771896362305, -0.0961681380867958, -1.1455456018447876, -3.4797327518463135, 0.5516285300254822, 2.8946852684020996, -0.9524406790733337, 2.473283290863037, -0.46566200256347656, 4.201991558074951, 9.190803527832031, -4.10166072845459, 2.6145248413085938, -4.3333210945129395, -10.066316604614258, 5.90712833404541, 1.229162573814392, 1.5890347957611084, 3.7988548278808594, 0.2082703560590744, 1.1532269716262817, -0.1480633169412613, 3.0015439987182617, 1.4703984260559082, 3.6122355461120605, -2.5033087730407715, -7.829967975616455, 1.6405916213989258, -0.972551703453064, 1.169185996055603, 0.7992430329322815, -3.6557226181030273, -5.867622375488281, 3.1818363666534424, 5.67602014541626, 4.662571430206299, 0.754984438419342, -6.660912036895752, 4.11381721496582, -1.0848017930984497, 2.897313117980957, 0.4391425848007202, -6.02005672454834, 6.7094597816467285, 1.0739831924438477, 2.191113233566284, -1.2788059711456299, -11.643240928649902, -1.7253388166427612, -2.543654680252075, -2.6697921752929688, 3.5347673892974854, -9.306900024414062, 1.7193585634231567, 0.2967631220817566, 3.4864487648010254, -1.9176617860794067, -2.2556631565093994, 0.20790663361549377, 6.1302008628845215, -2.599058151245117, -0.20562690496444702, -3.158360481262207, 0.7827246189117432, 2.0401744842529297, -4.919888019561768, 2.814311981201172, -4.623697280883789, -3.314800500869751, 0.3952968120574951, -0.44045647978782654, -3.2889115810394287, 0.5301169753074646, 0.5434563159942627, -2.3278191089630127, -3.496217966079712, -3.8125932216644287, -0.921141505241394, -5.992562294006348, 2.751356601715088, 4.721462249755859, -1.0816458463668823, 4.314762115478516, 4.910926818847656, 3.987626075744629, -4.322455883026123, -3.4444313049316406, -0.7265692353248596, -0.6288968920707703, -2.4244863986968994, 3.043347120285034, -1.4450573921203613, 6.0299224853515625, 2.4178929328918457, 4.352834224700928, 1.2116401195526123, -2.0045599937438965, -1.827947735786438, -2.5429232120513916, -4.766819953918457, -0.5293184518814087, -0.03198862448334694, -2.9161555767059326, 1.5817859172821045, 0.8555404543876648, 0.5791651606559753, -0.04541392624378204, 7.4237494468688965, -1.179744839668274, -0.13965554535388947, 3.781275510787964, 6.030603885650635, 5.385830402374268, -2.999098539352417, 1.5212706327438354, 0.7973183989524841, -3.6223015785217285, -2.2679765224456787, 9.2844877243042, -3.621795177459717, 0.615811288356781, -8.058978080749512, 0.2442784309387207, 1.950906753540039, 1.7058943510055542, -3.0965850353240967, -4.346750736236572, -4.876662731170654, 3.1833674907684326, -6.075690746307373, 6.520864009857178, 4.131553649902344, -2.9948394298553467, -1.7681784629821777, -2.5416994094848633, -0.1806161254644394, -0.8419643640518188, -3.2329344749450684, 2.8982999324798584, -2.677327871322632, 5.0372138023376465, 1.4670939445495605, -0.74336177110672, -0.07658500969409943, -5.065093994140625, 0.935063362121582, -0.9395626783370972, 8.309880256652832, 0.15223483741283417, 1.70085871219635, 5.031528472900391, 5.56155252456665, -8.36885929107666, -0.7050184011459351, 3.1881213188171387, 1.144158124923706, 4.104754447937012, -0.5490992069244385, -3.1198008060455322, 0.5394347906112671, 3.927647113800049, 3.7300705909729004, -0.604264497756958, -1.24281644821167, -1.6521745920181274, -2.0583858489990234, 5.840234279632568, -1.3140431642532349, -4.150579452514648, 6.059872627258301, -1.4337079524993896, -0.22010092437267303, -0.34846118092536926, 3.005837917327881, 6.590190887451172, 3.519144296646118, 2.0772287845611572, 7.105921745300293, 0.991863489151001, 0.01171970460563898, 0.483961820602417, 0.7965259552001953, -3.0121891498565674, -5.2061848640441895, -1.2297664880752563, 1.4685513973236084, 7.577480792999268, -0.7221730947494507, 2.172788143157959, 1.1503757238388062, 11.06449031829834, 0.37509456276893616, 0.8986213803291321, 5.319798946380615, 4.1669816970825195, 0.17816941440105438, -2.5655252933502197, 0.17341066896915436, -8.198439598083496, -1.2323163747787476, -5.1853837966918945, -1.306668996810913, -3.88386607170105, 1.5269736051559448, -4.245408535003662, -2.92053484916687, 0.8777890205383301, 0.710943341255188, 1.473694086074829, -6.199126243591309, -5.282108306884766, -1.591616153717041, 2.2619011402130127, -3.0154592990875244, -0.0416281633079052, -1.7432605028152466, -1.392175316810608, 2.3056275844573975, -6.253364562988281, 3.391411066055298, 7.216739654541016, 5.8512654304504395, -0.5989645719528198, 0.523233950138092, 0.5395336747169495, 0.17677457630634308, 3.521754503250122, -1.8290528059005737, 1.9394950866699219, -4.277503967285156, 6.184781551361084, -1.215156078338623, 1.543649673461914, 4.314940452575684, 2.809447765350342, 0.7226002216339111, 0.1498476266860962, -3.9465250968933105, 0.7286929488182068, 1.7119032144546509, -4.499544620513916, 6.452941417694092, -1.651060938835144, 0.15673471987247467, -2.917435646057129, -2.745204448699951, -4.03053092956543, 5.486003398895264, -3.278208017349243, 0.8816812634468079, 5.0666632652282715, -4.366138458251953, 7.9565911293029785, 2.986395835876465, -1.2542798519134521, 7.309987545013428, 0.11896234005689621, -2.1072707176208496, 6.680375576019287, 2.9896836280822754, 0.16464672982692719, -1.7852369546890259, -3.281660318374634, 0.016775229945778847, 0.29861509799957275, 4.401503086090088, 0.7873405814170837, -5.393865585327148, 5.199489593505859, 6.742525100708008, -1.344473958015442, 6.860543727874756, 1.0583385229110718, 2.493082046508789, 8.085800170898438, 7.832270622253418, 0.6203126907348633, 4.529691696166992, -0.43619075417518616, 8.118322372436523, 6.999195575714111, -6.480469226837158, -3.645026683807373, -3.124070644378662, -7.081855297088623, 5.55481481552124, 3.2751824855804443, -2.9055304527282715, 3.697796583175659, 3.6840837001800537, 1.7987885475158691, -0.5480392575263977, -2.2315237522125244, 1.5225768089294434, -1.661249041557312, 1.3312017917633057, 3.357741594314575, -3.6392738819122314, -0.36494290828704834, -0.5210966467857361, -0.10535655170679092, -4.30680513381958, 3.0966312885284424, -5.255748748779297, 3.9354143142700195, 2.0185792446136475, 4.6022515296936035, -2.197016716003418, -1.847145676612854, -0.44365549087524414, -0.4976125657558441, -7.827589511871338, -5.611841201782227, 8.9952974319458, 6.033016204833984, -6.505643844604492, -5.191788673400879, 1.967567801475525, 0.9072112441062927, -1.4258599281311035, 1.7747209072113037, 4.364049911499023, -3.7114481925964355, 2.3740358352661133, 8.353049278259277, 1.0258095264434814, -8.26777458190918, -1.1365331411361694, 1.851616382598877, -0.9838380217552185, 1.105347990989685, -4.472803115844727, -1.5248603820800781, 2.81404185295105, -0.11439866572618484, -2.1687064170837402, -0.06840161979198456, 2.8122270107269287, -11.788126945495605, 1.8213924169540405, -3.6233997344970703, 1.900183081626892, 3.5386507511138916, -0.7131619453430176, 0.24571022391319275, -4.1044511795043945, 2.9650723934173584, -3.8362808227539062, 5.558772087097168, -1.3534680604934692, -3.1696319580078125, 2.5147337913513184, -2.917290210723877, 0.14479051530361176, -2.073042631149292, 3.628631114959717, -7.767747402191162, 6.128629684448242, -5.254600524902344, 1.907823920249939, 2.1700189113616943, 11.81360149383545, 2.05558180809021, 1.2746773958206177, -0.9287351369857788, 2.7458064556121826, -1.377522349357605, -5.787901878356934, 0.8188395500183105, -0.67340487241745, -1.385627031326294, 1.8945708274841309, 1.0098531246185303, -0.44111502170562744, -4.673768043518066, -2.389014482498169, -1.55315363407135, 4.8527727127075195, 1.9444639682769775, -3.499498128890991, 3.940549373626709, 1.9240527153015137, 2.9317054748535156, -2.8113980293273926, 6.903346538543701, 1.0609792470932007, 5.129151821136475, 3.5777900218963623, -5.087172031402588, 3.032057285308838, 2.7624735832214355, 8.097668647766113, -1.0811039209365845, -6.133373260498047, 4.288854122161865, -4.3341169357299805, -1.8575626611709595, -0.39705273509025574, -2.763228416442871, 2.0262529850006104, -5.479310989379883, -9.595552444458008, -7.81815767288208, -3.664789915084839, -1.1519261598587036, -1.1074692010879517, -2.7271933555603027, 0.9839942455291748, -4.337057113647461, -9.809914588928223, -9.025444030761719, 0.48031550645828247, -2.9229023456573486, -2.4431936740875244, 1.0091615915298462, -2.1997318267822266, -1.2364675998687744, 4.664907455444336, 1.5905661582946777, -0.10990048944950104, 3.1229379177093506, -3.3224031925201416, 1.1241463422775269, -0.650778591632843, -6.90809965133667, 3.677741289138794, 11.044303894042969, 1.0628620386123657, -3.2559564113616943, -0.055548835545778275, -3.2101218700408936, -7.933637619018555, 4.904041767120361, -3.5873985290527344, 2.813386917114258, 1.6701440811157227, -4.746481418609619, -4.479650020599365, -12.434995651245117, 11.424172401428223, 1.4040080308914185, 0.44961607456207275, -1.1819106340408325, 6.896198749542236, 3.879713296890259, -5.334948539733887, -5.912851333618164, -1.3585405349731445, -0.3695913255214691, 1.5355865955352783, -4.860045909881592, 1.8630517721176147, -4.251411437988281, -3.601675033569336, -0.5965328216552734, -2.061396360397339, -2.387324810028076, 4.145805835723877, 0.47677144408226013, -3.860281229019165, -7.418298244476318, 2.3100485801696777, 7.496973991394043, 4.346551418304443, -0.1650605946779251, 2.262357711791992, -4.432473659515381, 6.5883708000183105, 4.2352423667907715, -8.527265548706055, -0.754088819026947, 1.0239629745483398, -3.426335096359253, -0.33846554160118103, 2.77250599861145, 3.107851028442383, 5.855661392211914, 4.217648506164551, -0.02556140348315239, -3.314767599105835, -0.21319259703159332, -4.794659614562988, -5.871333599090576, -2.865663766860962, -1.908133864402771, 1.6037710905075073, -0.7971652746200562, -0.5268163681030273, 1.0148749351501465, 2.7934653759002686, -2.439504861831665, -5.944119453430176, 7.0758466720581055, 7.372738838195801, -0.8189544677734375, -0.7050295472145081, -2.8637893199920654, 3.7685387134552, -6.208191871643066, 4.824843406677246, -4.060068130493164, -3.478184461593628, -2.495223045349121, 1.6814146041870117, 3.1558849811553955, -4.450930118560791, -5.792882442474365, -5.44488000869751, -1.303882360458374, -2.2957468032836914, -3.034987688064575, -3.0866270065307617, -0.5493963956832886, -2.739844799041748, 4.4532952308654785, -2.0142955780029297, 2.208740234375, -3.6433663368225098, 2.4239490032196045, -1.5691229104995728, 3.051478147506714, 0.9598895907402039, -0.868844747543335, -1.9029524326324463, 2.9481592178344727, 0.39169323444366455, 3.497654438018799, -1.028292179107666, 0.7092646360397339, -2.2615954875946045, -2.3969833850860596, 0.5924761295318604, 5.75134801864624, 6.006035327911377, -6.151210784912109, 5.70881462097168, 6.019462585449219, 1.853105068206787, 4.918721675872803, 2.734280586242676, -0.806331217288971, 4.24667501449585, 2.115079879760742, -1.8725502490997314, -8.302373886108398, 0.41784757375717163, 0.4867556095123291, -6.13624382019043, -0.008895071223378181, -3.9328114986419678, -1.3756576776504517, -6.5985426902771, -2.281644105911255, -0.641707181930542, -6.365920066833496, -5.1804656982421875, 0.07914415001869202, 0.006575816310942173, 3.0137476921081543, 0.8662028312683105, -4.017307758331299, -6.4966607093811035, 2.3866329193115234, -2.661876678466797, 1.1465047597885132, 2.0895402431488037, -3.9882731437683105, 8.27686595916748, 6.116746425628662, -0.5861064791679382, -0.8449421525001526, -3.5736114978790283, 5.7627644538879395, 4.911971569061279, -1.4129149913787842, -2.4062349796295166, -2.769163131713867, 1.3308801651000977, -2.888291120529175, 4.581024646759033, -6.015077114105225, -1.4810783863067627, -4.546298503875732, 0.23938463628292084, -3.6503772735595703, -5.665976047515869, 1.763435959815979, 0.5265220999717712, 5.131092548370361, 0.6504316329956055, 0.7750552296638489, 2.234351396560669, 6.433518409729004, 1.466240406036377, 0.8736621737480164, 3.2343804836273193, -2.439621686935425, 2.489118814468384, -0.23366834223270416, -2.029522180557251, -4.069546699523926, -4.572372913360596, 1.1600124835968018, -7.484377861022949, 1.8928027153015137, 4.939792156219482, -1.2827881574630737, -3.9168102741241455, 1.1474148035049438, -5.568197250366211, -0.9138965606689453, -3.1511008739471436, 1.3903638124465942, 0.7829759120941162, -7.763927459716797, 1.38309645652771, 2.8272242546081543, 4.078114032745361, 5.635037422180176, 1.8316837549209595, 2.1829488277435303, -1.9176857471466064, -2.086214303970337, -1.052860975265503, -3.579768419265747, 2.3236095905303955, 5.2466044425964355, 5.341033935546875, -5.73592472076416, 0.07602983713150024, 2.643162727355957, -4.354420185089111, 1.4967352151870728, 4.9589409828186035, -0.26632940769195557, -0.02770914137363434, -1.9156055450439453, -0.4415796995162964, 1.1302391290664673, 4.266408920288086, 0.3463946282863617, 1.5239919424057007, 1.7174956798553467, -3.644836664199829, 0.8145596981048584, -0.44102025032043457, -7.239109039306641, 2.4260823726654053, -3.382863759994507, -7.51132345199585, -10.644403457641602, 2.4771535396575928, -4.583957672119141, 1.619780421257019, 2.6353790760040283, 4.814759254455566, -2.9038989543914795, -2.4811129570007324, 0.6402863264083862, -6.322474479675293, -5.906279563903809, -2.554776668548584, 0.22225156426429749, -6.103703498840332, 0.9554444551467896, -2.234163999557495, -7.694026470184326, -5.789746284484863, -3.0274360179901123, -0.903750479221344, -2.4698729515075684, 7.476178169250488, 1.360288143157959, -4.383774280548096, 3.3968348503112793, -3.1434803009033203, 1.7447631359100342, -3.5555028915405273, -8.567834854125977, 10.775559425354004, 1.062333583831787, -0.6665388941764832, -1.0262277126312256, -0.4268909990787506, -0.6791917681694031, 4.983136177062988, 2.5668933391571045, 2.4982478618621826, 0.7172834873199463, 3.539794445037842, -0.6588286757469177, -3.448962688446045, -0.1701641082763672, -0.64890456199646, -7.512580394744873, 1.2836238145828247, 1.2115561962127686, -5.929060935974121, -3.37933611869812, -1.6122114658355713, 0.2890246510505676, -4.620176315307617, 0.2856300473213196, 1.2140281200408936, 8.6946382522583, 3.6450748443603516, -3.4932398796081543, 2.161670446395874, 3.78527569770813, -0.9567886590957642, -0.5006118416786194, -8.91743278503418, -0.5572839379310608, 5.5495285987854, -3.3801681995391846, -0.47490325570106506, -2.582873821258545, -1.0951566696166992, -0.39778709411621094, 1.5932012796401978, -4.276365756988525, -1.606132984161377, 3.6414167881011963, -3.056936264038086, 5.841316223144531, 4.939736366271973, -4.7213544845581055, 5.535749912261963, -3.6399986743927, 1.804250717163086, 3.2395575046539307, -1.5892781019210815, -0.26686424016952515, 5.470550060272217, -3.3138091564178467, 0.8138083219528198, 2.1758742332458496, 7.9159111976623535, 3.141145706176758, 1.4393136501312256, 2.711428165435791, -0.7197049260139465, 0.5000022053718567, -1.987744688987732, 1.238577127456665, -9.189894676208496, -4.527822971343994, -2.9568557739257812, -1.2168214321136475, 2.447597026824951, 3.3288586139678955, -7.535160064697266, -0.4013219475746155, 7.051377773284912, 0.3766666650772095, -1.1450132131576538, -6.688758373260498, -5.908144950866699, 5.0682692527771, -3.6955513954162598, -0.29083216190338135, -2.6749608516693115, 1.2351890802383423, 4.58750581741333, 3.9370827674865723, -2.754135847091675, -0.7516657710075378, -2.8614144325256348, 0.7470239400863647, 0.011231030337512493, 0.06802824139595032, -6.976865768432617, -2.8593244552612305, 1.726030945777893, -0.8267803192138672, 1.7880183458328247, 0.14343509078025818, -5.655760765075684, -2.021169900894165, 2.012087106704712, 9.50763988494873, -8.187108039855957, 1.1164109706878662, -5.859503746032715, 0.533869206905365, 3.4111762046813965, 6.482316017150879, -0.6785494685173035, -2.111720561981201, 0.0493319109082222, 2.0352466106414795, 6.318423271179199, 8.875650405883789, 3.257286548614502, -6.112401962280273, 1.9295622110366821, -5.126508712768555, 0.7910372614860535, 2.386963129043579, 4.182068824768066, -2.843956708908081, -1.8884466886520386, 2.0495564937591553, -4.044996738433838, 1.974341869354248, -0.7683444619178772, -2.3350465297698975, 3.0348198413848877, -2.1750144958496094, 6.192606449127197, -1.0397365093231201, -1.4332571029663086, 2.1419756412506104, 3.495408296585083, 6.375234127044678, 1.6472715139389038, 4.666950225830078, -0.27188438177108765, 0.7041217088699341, -0.559341311454773, 2.541138172149658, 0.1697889119386673, 3.3156096935272217, -2.2993271350860596, 6.469311237335205, 6.283664703369141, 3.40415620803833, 1.7268534898757935, -6.452000617980957, -0.5226753354072571, -1.010582447052002, 1.7294739484786987, -2.4140305519104004, -2.000734329223633, 3.5251431465148926, 1.7674132585525513, -2.0191140174865723, 1.9918744564056396, 4.64819860458374, -0.3941423296928406, -6.190061092376709, 3.72293758392334, -2.7731151580810547, 5.803101539611816, -3.3108973503112793, 3.1206414699554443, 1.5921088457107544, -2.3512072563171387, -2.5461208820343018, -0.9486057162284851, 0.9031660556793213, -0.24022150039672852, -9.599821090698242, 2.9939191341400146, 1.1564315557479858, 7.678339004516602, 1.5503454208374023, 0.5398869514465332, -3.5881669521331787, -7.3926897048950195, 0.31561997532844543, 1.3524748086929321, 0.19213779270648956, -0.758849024772644, -0.4287395477294922, 7.706624507904053, 1.3640451431274414, -7.459831714630127, -0.4766785502433777, -3.9669620990753174, -3.916360855102539, 0.29309001564979553, 0.1983247548341751, -4.622976779937744, -6.002518177032471, 3.9500515460968018, -4.081094264984131, 7.58807373046875, 5.084727764129639, 0.46561604738235474, 3.461787223815918, -5.95074462890625, -0.43340298533439636, -2.3888747692108154, 3.7185192108154297, 5.910148620605469, 1.2403526306152344, -4.958219051361084, -7.949931621551514, -8.101690292358398, -2.045846700668335, -7.806734085083008, 4.4667463302612305, 7.780048847198486, 2.7201735973358154, 0.4440014362335205, -3.9765915870666504, -1.7590060234069824, 1.17617928981781, 2.351801872253418, -1.6243051290512085, -10.01644515991211, 1.7619757652282715, 6.256722927093506, -1.346328616142273, -3.9699878692626953, 6.761013984680176, 3.6559503078460693, 1.6611764430999756, -0.4203668534755707, 1.542478084564209, -0.9039493203163147, -1.010318398475647, 2.5632452964782715, -2.969071388244629, 3.9633657932281494, -3.323505401611328, -2.465257406234741, -0.25952956080436707, 0.7826052904129028, -0.2848500907421112, -4.762516975402832, -0.7451720833778381, -6.199996471405029, 6.206334590911865, -6.0538177490234375, -1.9884885549545288, 7.480605125427246, 1.1432533264160156, 0.4676891267299652, 3.1992650032043457, 0.10212958604097366, 2.420043468475342, -2.447028636932373, -2.8003456592559814, 1.4517861604690552, 3.77717661857605, 0.11520029604434967, 3.579195261001587, -4.701601982116699, 3.9192631244659424, -3.2341880798339844, -3.816990852355957, -2.330380916595459, -0.05281665176153183, 4.580747127532959, 2.1046178340911865, 0.36571237444877625, -4.5786638259887695, -2.492569923400879, 8.557975769042969, 0.9551776647567749, 2.535407066345215, 1.2913907766342163, -2.981994867324829, -6.7480573654174805, -1.908461332321167, -1.1686469316482544, -3.0819246768951416, -10.009917259216309, -8.231402397155762, 4.852931499481201, -5.8065080642700195, -3.136150598526001, 1.36989164352417, -3.040144920349121, -1.2964235544204712, 1.7880749702453613, -5.615149021148682, -1.2097458839416504, 9.956522941589355, 6.109834671020508, -3.9802284240722656, -4.178228378295898, 1.0395324230194092, -4.662318229675293, -2.896153211593628, -10.413954734802246, 0.8926985859870911, -5.586953639984131, -6.934225082397461, -3.319119691848755, 2.877904176712036, -2.247363567352295, -0.8319501876831055, 6.1726813316345215, 1.6176058053970337, 6.627054691314697, -5.985231876373291, -0.4067838788032532, -1.864884853363037, 5.176833629608154, -4.99379825592041, 1.5559920072555542, 1.5102266073226929, -0.5081253051757812, 0.7673994302749634, 4.72402286529541, -5.221701622009277, -11.833256721496582, 4.894833564758301, -6.644870758056641, 4.7663960456848145, 3.1230995655059814, -2.065307378768921, -2.279627799987793, 1.7074437141418457, -2.225715160369873, -6.4306206703186035, 1.0895721912384033, 2.0893185138702393, 0.9315202832221985, 1.2274317741394043, -1.2287671566009521, -4.777979850769043, 1.4962457418441772, 1.4328510761260986, 2.7721762657165527, -1.7723056077957153, 0.2926252484321594, -1.9623630046844482, 3.5726959705352783, -2.199568510055542, 0.7968308329582214, -1.7937499284744263, -1.7844184637069702, 0.6463400721549988, 3.3298416137695312, -4.9379754066467285, -3.0940117835998535, 2.472454071044922, -2.162368059158325, -4.903634548187256, -2.756350040435791, -0.5853835940361023, -3.7533626556396484, 1.835789680480957, -0.5982366800308228, 0.6947843432426453, 0.14431160688400269, -5.732370376586914, 5.204922199249268, 2.731624126434326, -7.804807662963867, 8.372183799743652, 6.012350559234619, 2.5622363090515137, 6.2783684730529785, 6.576194763183594, -0.3402906060218811, -0.8491645455360413, -3.780832529067993, 0.9959352612495422, 1.8511313199996948, -0.06313465535640717, 2.1900458335876465, -1.0087306499481201, 3.198195695877075, 5.31550931930542, 0.7562852501869202, 5.053380012512207, -0.42153865098953247, -2.6687278747558594, 0.067711740732193, 1.5214653015136719, 4.517066478729248, -7.36578893661499, 3.340430736541748, 0.7979937791824341, -4.03697395324707, -3.3906896114349365, -4.719419956207275, 1.7916901111602783, 7.360705375671387, -1.434052586555481, -2.5249762535095215, 3.53743052482605, 6.986522197723389, 1.9875568151474, 2.9010753631591797, 0.4335858225822449, 2.6826517581939697, -6.2988386154174805, -4.163572311401367, -1.637442708015442, 4.210819721221924, 1.565721869468689, -9.074833869934082, -4.094221115112305, -2.090132474899292, -3.992272138595581, -2.9599833488464355, 1.2330946922302246, -1.999465823173523, 6.232028484344482, -5.903489112854004, -5.799933910369873, -6.718456745147705, 1.4396871328353882, -0.2862476110458374, 3.5541536808013916, 3.272653579711914, 2.9036755561828613, -4.235421180725098, -0.5599934458732605, -9.299148559570312, 1.63962984085083, 0.9062908291816711, -0.15734446048736572, -1.324434757232666, 1.2493834495544434, -0.5796714425086975, 0.8192890882492065, -1.6422420740127563, -4.254422187805176, 4.07415246963501, -1.9543920755386353, -1.6682945489883423, -3.064328193664551, -1.5696308612823486, -5.372488975524902, 2.7640037536621094, -1.1028333902359009, 8.395048141479492, -1.3337291479110718, -11.40800666809082, 6.096948623657227, -6.8351359367370605, -6.543881416320801, 5.169307231903076, -2.007766008377075, -1.8819537162780762, -1.7060025930404663, -1.3637861013412476, 0.4978940784931183, 1.0146608352661133, 2.4855449199676514, 3.808452844619751, 1.2922630310058594, 1.3482911586761475, -4.745826244354248, -2.0260815620422363, -2.3123292922973633, 4.834040641784668, -3.402899742126465, -6.286744594573975, -0.4849240183830261, 0.20628860592842102, 7.419374942779541, 1.0719003677368164, 7.803489685058594, 3.4816088676452637, 4.019524097442627, -3.4181885719299316, -0.3026708662509918, 1.3499881029129028, 6.466457843780518, 0.08398706465959549, 4.789733409881592, 0.2254740446805954, -0.6375702619552612, -5.7779364585876465, -5.612226963043213, -8.221365928649902, 14.538087844848633, 6.997439861297607, -2.2876644134521484, -2.3465003967285156, 2.0250930786132812, -3.7788546085357666, 3.892503023147583, -1.06727135181427, 8.737457275390625, -1.8946146965026855, -3.3468856811523438, 4.766256809234619, 0.5813278555870056, -3.425053596496582, -3.614152431488037, -2.1013591289520264, -0.1801154911518097, 1.6761538982391357, -2.9785068035125732, -0.9545745253562927, 4.36317777633667, 6.1986212730407715, 0.9640480279922485, -1.366702914237976, -2.9249637126922607, 2.1536307334899902, -1.442639708518982, 6.682255744934082, -2.5466740131378174, 5.279788017272949, 7.6618499755859375, 3.7060980796813965, 0.9771020412445068, -3.1986262798309326, 5.909243106842041, -1.3066222667694092, -0.9233875274658203, 2.8119590282440186, -1.0457980632781982, 3.9451041221618652, -0.9557945132255554, -1.9402531385421753, 2.90633487701416, -1.2123054265975952, -1.3497000932693481, -0.4647907316684723, 7.570278167724609, -2.3844549655914307, 3.74682354927063, 0.24322237074375153, 0.05521588400006294, -3.4631712436676025, 1.961099624633789, -2.3638598918914795, -6.873354434967041, -1.1479871273040771, -1.8061368465423584, 5.846573829650879, -3.813913345336914, 1.8323516845703125, -1.2540189027786255, -1.199119210243225, 0.007486729416996241, 5.831679344177246, -2.0471389293670654, -0.9339136481285095, -2.2026407718658447, -7.617559909820557, -5.000283718109131, 1.3013763427734375, 7.487520694732666, -0.6326262950897217, 2.248704195022583, 2.0830605030059814, -2.473761558532715, -8.883374214172363, -3.9778854846954346, -2.2670199871063232, -4.544637680053711, 6.486288070678711, -5.894854545593262, 5.252402305603027, -0.5167614817619324, -0.8562613129615784, -3.6815860271453857, 6.660952568054199, 2.194850444793701, 3.104313850402832, 1.0561918020248413, 5.562789440155029, 0.7237755656242371, -4.082076549530029, -3.9395437240600586, 1.7346373796463013, 7.408542156219482, 0.5986032485961914, -5.482876777648926, 1.0893884897232056, -1.7965044975280762, -5.274228096008301, 4.227222442626953, 4.179964065551758, -6.107416152954102, 6.86691427230835, 2.2001402378082275, -6.078089237213135, -3.368906021118164, 6.697403907775879, 4.093400955200195, -0.49943897128105164, -3.452086925506592, -10.721036911010742, 3.4037628173828125, 5.8677659034729, -0.31931596994400024, 0.5484944581985474, 1.255757212638855, -1.8159866333007812, -4.458183288574219, -2.4632985591888428, -6.8940839767456055, -4.657742500305176, 4.652827262878418, 1.142215609550476, 8.863388061523438, -5.412558555603027, -5.094632148742676, 3.2881228923797607, -10.391465187072754, -2.0587241649627686, 4.631063461303711, -1.502009391784668, 4.637125492095947, 3.3089301586151123, 3.0025136470794678, -1.3472603559494019, -0.15876735746860504, -10.29592227935791, -1.340169072151184, 5.193005084991455, 2.835646629333496, 3.361234188079834, -3.3124523162841797, 6.63493013381958, 2.0626068115234375, -4.469979763031006, 1.418097972869873, 4.32082986831665, -5.666170597076416, -0.6313958168029785, -2.6509716510772705, -1.6605433225631714, -1.8423079252243042, -0.33840447664260864, -2.328892469406128, 6.448664665222168, 4.734588146209717, -9.598650932312012, -1.770890712738037, -3.3113977909088135, 8.711259841918945, 3.5471551418304443, 2.083996534347534, 3.9346039295196533, -0.17747540771961212, -4.884770393371582, 0.8075256943702698, 2.8629140853881836, 2.852071523666382, 2.3571996688842773, -2.2669990062713623, -4.186361312866211, -4.413821697235107, 5.415802955627441, 5.969156742095947, 1.5232325792312622, -0.1458277553319931, -8.119549751281738, -3.7153677940368652, 7.118287563323975, -1.516182780265808, 0.5727280974388123, -3.9344937801361084, 4.193605899810791, 10.35253620147705, 1.0432969331741333, -4.133339881896973, 2.601288318634033, -6.808229923248291, 5.425459384918213, 0.8454734086990356, -5.774726867675781, -3.8371264934539795, -0.22484330832958221, -2.8930985927581787, 1.8625411987304688, -4.2458295822143555, -5.8294758796691895, -3.2308690547943115, -1.112384557723999, -0.6857373714447021, 3.065610885620117, 7.558928966522217, 0.029534533619880676, -5.772619247436523, 2.557957172393799, 4.123680591583252, -9.649105072021484, -2.7574024200439453, -3.0173354148864746, 1.7622896432876587, -3.48738431930542, -10.882214546203613, 1.6516497135162354, -0.8704515099525452, -0.908353328704834, -0.7700268030166626, 1.2125483751296997, 7.7120680809021, 0.9991253614425659, -5.446402072906494, -1.9477125406265259, 0.42899367213249207, 2.640622615814209, 0.8486297726631165, 2.0810651779174805, 2.2990591526031494, -0.6969627737998962, 2.243534803390503, 0.8646779656410217, 3.948843002319336, -1.3765925168991089, 3.128387689590454, 2.3718059062957764, -2.242824077606201, -6.0412139892578125, -0.40576064586639404, 0.6276906728744507, 0.8408938646316528, 1.8395466804504395, 0.2957805097103119, 2.576854944229126, -0.3217619061470032, -4.026121616363525, -7.3282670974731445, 5.607947826385498, 0.5883780121803284, -2.14764142036438, -0.6768196821212769, -0.19135381281375885, 2.276665449142456, 7.5353827476501465, 1.4502506256103516, -1.8243286609649658, -4.032968521118164, 6.681285381317139, -2.0150485038757324, 1.9308263063430786, -0.18700474500656128, -2.5926265716552734, 0.02624157816171646, -1.6615817546844482, -2.4905827045440674, -4.432045936584473, 0.09238212555646896, 6.636744499206543, 1.618607759475708, 7.288848400115967, -7.570218086242676, -14.64400863647461, 3.4421167373657227, -4.038915157318115, 0.7893269658088684, 4.0008769035339355, -1.9178214073181152, 2.692476511001587, 4.861386775970459, -3.722630023956299, -4.261334419250488, 3.238036632537842, -0.5847439765930176, 0.6454892754554749, -0.5545207262039185, -1.786470651626587, -3.3964829444885254, -1.44289231300354, 0.5272464752197266, 0.9412944912910461, 3.199577569961548, -4.840306758880615, 1.238051176071167, 6.912755966186523, 1.4411789178848267, 5.328447341918945, 2.729848861694336, -2.2040257453918457, 1.5901222229003906, 1.7879607677459717, 0.08556338399648666, 5.764017105102539, -7.231942653656006, 5.2411322593688965, -3.8500263690948486, -18.394664764404297, -4.085381984710693, 3.034846067428589, -12.36269474029541, -0.23963966965675354, 5.216573715209961, -1.3031988143920898, -2.442823648452759, 1.9780690670013428, -1.136830449104309, 1.7198625802993774, -1.4231663942337036, 0.9496955275535583, -3.4656167030334473, 2.763728618621826, -2.7584333419799805, -7.970800876617432, -3.9651713371276855, 4.855947494506836, 0.6672161221504211, -2.3990063667297363, -4.741089344024658, 0.367928683757782, -1.3345274925231934, -8.672796249389648, -1.4070625305175781, -0.5602525472640991, -0.8739935159683228, -0.4807547628879547, -5.535763740539551, 0.5967583060264587, 3.092834234237671, -1.751047134399414, -1.5103718042373657, -4.2154541015625, -0.7354303598403931, -4.1671857833862305, 9.823405265808105, -4.199910640716553, 2.3145084381103516, 5.725100517272949, 3.4513320922851562, -2.4409215450286865, -3.320793390274048, -5.587129592895508, -2.780088424682617, -1.838366150856018, 4.546183109283447, 1.7669651508331299, -1.247554898262024, -1.586596131324768, 0.16241595149040222, 1.458696722984314, 2.8633368015289307, 1.1929888725280762, -0.006496984511613846, 4.237854480743408, 1.4633738994598389, 1.8946928977966309, 2.779351234436035, -1.1564409732818604, 3.2622740268707275, -7.381618022918701, -6.140303134918213, -6.28934907913208, 8.853850364685059, 2.816101551055908, -4.858116149902344, -2.220336675643921, -1.2352317571640015, -6.275389671325684, -0.25288036465644836, -1.247975468635559, -4.603251934051514, -7.221822261810303, -4.855701923370361, 2.413750648498535, 3.773188591003418, -3.904778242111206, -5.962106227874756, -1.0815768241882324, -4.696006774902344, -5.73442268371582, -7.739081859588623, 2.681891918182373, -6.102260112762451, -7.023601055145264, -1.2674496173858643, -3.1319732666015625, -3.3783793449401855, 2.627122163772583, -2.9896998405456543, -1.2705079317092896, 6.513031959533691, 4.005086421966553, -2.7695765495300293, -0.9465361833572388, -4.369312286376953, 5.5799241065979, -0.5786644220352173, -1.9155383110046387, -4.294296741485596, 0.5359295606613159, 1.7691376209259033, -0.8719251751899719, 5.3120503425598145, -5.648223400115967, -4.177672863006592, 1.7019000053405762, 4.2660746574401855, -3.7101054191589355, 0.5800655484199524, 3.828906297683716, 0.25379621982574463, -2.381385087966919, -0.8871299624443054, -4.363224029541016, 3.887331962585449, 0.3383013606071472, -4.629364013671875, -1.6977245807647705, 1.1692473888397217, 2.765389919281006, 1.0957720279693604, 0.8772076368331909, 3.1313893795013428, 2.1371610164642334, 1.060610055923462, 3.3830857276916504, 0.662246584892273, -3.251770257949829, 0.6113535761833191, 1.233970046043396, -4.082359790802002, 1.115493655204773, -4.656980991363525, 2.092094898223877, -0.2536619305610657, 1.9259583950042725, 1.5007182359695435, 0.7196937203407288, -10.100836753845215, 0.30179017782211304, -1.1630487442016602, -4.998677730560303, 0.035900019109249115, -3.130659341812134, 2.199167490005493, -3.914691209793091, -0.6867420077323914, -6.853356838226318, 1.3979853391647339, -0.6350484490394592, -1.2042227983474731, -3.807854413986206, 1.3221114873886108, 3.057544231414795, -0.4249190092086792, -1.5537583827972412, -1.9164077043533325, 1.7162246704101562, -0.12743628025054932, 1.731605052947998, -2.8517699241638184, 0.8606682419776917, 7.655778408050537, 3.347194194793701, 8.183859825134277, -0.09987097233533859, -4.648459434509277, 4.735504627227783, -2.7599945068359375, 4.6895060539245605, -2.0157787799835205, 1.1559088230133057, 1.0470830202102661, 5.10150671005249, -1.0610237121582031, -2.455421209335327, -5.691798686981201, 0.22046713531017303, 5.34214973449707, -0.0339047871530056, -5.4069504737854, 0.1909659206867218, -1.9293339252471924, 1.7420591115951538, -1.6775227785110474, -1.202094316482544, -9.18761920928955, 3.5240066051483154, -10.047699928283691, 1.8881659507751465, 5.356713771820068, 1.3784468173980713, 1.324802041053772, -2.696542739868164, 0.30647340416908264, 2.8245506286621094, -2.02763295173645, -1.3553056716918945, 2.366286516189575, -2.2281715869903564, 4.251749515533447, -3.0561635494232178, 0.062173061072826385, 2.0286386013031006, -1.3872261047363281, -1.1351286172866821, -7.104176044464111, -0.58585524559021, 6.536484241485596, 2.474181652069092, -6.051513671875, 1.1092475652694702, -4.8228840827941895, -0.9180473685264587, -3.744586706161499, 1.529719591140747, 3.939483165740967, -2.6901943683624268, 3.7557215690612793, 1.2745583057403564, -0.23981641232967377, 0.5274416208267212, 4.833048343658447, 7.439905166625977, -3.0563061237335205, 9.403078079223633, -1.8372172117233276, -1.7272783517837524, 4.357143402099609, 5.565958023071289, -8.048966407775879, 10.819846153259277, 2.3451452255249023, -0.4553273320198059, -3.7560174465179443, 0.1859283596277237, -2.0115063190460205, -2.9662563800811768, -3.178565263748169, -1.5415515899658203, 1.7103900909423828, -1.3919739723205566, -1.5721745491027832, -3.7090275287628174, -1.3791652917861938, -3.3998301029205322, -0.14400698244571686, 4.1816935539245605, 2.741762638092041, -2.687056541442871, 4.811505317687988, -0.9682079553604126, 1.3287731409072876, 5.63394832611084, -3.188253402709961, 0.3075105547904968, -1.8619587421417236, -0.3590145409107208, -1.726821780204773, -3.3189423084259033, 2.087775945663452, -11.658589363098145, 0.5776987671852112, 7.277107238769531, 3.3995087146759033, -1.9996106624603271, 1.0881849527359009, 1.390786051750183, -2.742652654647827, 2.4183521270751953, 4.671694755554199, -2.896073341369629, 2.6291730403900146, -6.465404033660889, -1.0781021118164062, -0.0251018013805151, -9.6704683303833, 4.4976115226745605, -3.1120238304138184, -1.9302468299865723, 4.343933582305908, -2.257222890853882, -10.407700538635254, -4.311947822570801, 1.6459399461746216, 0.8936063051223755, -2.8669464588165283, 5.564271450042725, -0.7610363960266113, -0.31110140681266785, 5.190976142883301, 0.5710189342498779, 1.9905332326889038, -4.1623077392578125, -3.713057279586792, -3.6021556854248047, -1.277805209159851, 1.7712523937225342, 10.995473861694336, -3.8807334899902344, -3.3745124340057373, -2.80385422706604, -1.0321080684661865, -0.651530385017395, -1.4705405235290527, -3.763761520385742, 1.0810182094573975, -1.6288979053497314, -1.0984222888946533, -7.261445045471191, -2.80648136138916, 1.3267794847488403, -2.2495462894439697, 1.4114818572998047, -0.05545796453952789, 2.3022923469543457, 1.1923191547393799, -5.330544471740723, 0.5593627095222473, -2.8101727962493896, 5.070804595947266, 0.8297818899154663, -5.580000877380371, -3.0771827697753906, -2.644578218460083, 5.978724002838135, 0.33224251866340637, -0.8531008958816528, -3.7951340675354004, 0.9179643988609314, -1.5977729558944702, 5.030418395996094, 2.2401320934295654, -7.970476150512695, -1.0991743803024292, -2.6499695777893066, 0.1856912225484848, 3.2010622024536133, -1.3441013097763062, 0.6060115694999695, 0.49874478578567505, -4.677666664123535, 3.0130391120910645, 3.650468111038208, 4.016010284423828, -2.7106986045837402, -4.1550703048706055, 6.417754650115967, 0.04328541457653046, -1.8147753477096558, -4.571277618408203, -1.2921950817108154, 4.151396751403809, -4.883361339569092, -4.303219318389893, 1.1405144929885864, -4.8782525062561035, -0.13446179032325745, 0.25953376293182373, 2.045875310897827, -5.629411220550537, 9.153584480285645, -3.22707200050354, 0.4484710991382599, 4.594086647033691, -1.5121179819107056, -2.973304033279419, 3.646453857421875, 0.6142504811286926, 7.724366188049316, 0.021320322528481483, 1.9956104755401611, 0.3907249867916107, -2.648160457611084, 0.8198427557945251, 0.5840026140213013, 8.681300163269043, -3.227742910385132, 5.485305309295654, 9.068892478942871, -1.4183733463287354, 2.0011003017425537, -1.2721704244613647, 0.6250742077827454, -10.229001998901367, 0.9094676971435547, 3.5508270263671875, -5.5772576332092285, -5.730550765991211, 4.5183186531066895, -2.359825611114502, 1.3641246557235718, -7.781353950500488, 1.5748398303985596, 5.5408549308776855, 2.659174919128418, 2.3424324989318848, -0.9294213056564331, -4.381178379058838, 2.8995656967163086, -1.0668569803237915, 2.2907421588897705, -6.611273765563965, 3.632570266723633, -5.760362148284912, 0.08282473683357239, -2.8014791011810303, 2.2709338665008545, -1.067603349685669, 0.497952938079834, 1.6870770454406738, -1.0173027515411377, -3.0905508995056152, -1.430585265159607, -0.08389988541603088, 0.8839999437332153, -4.899669647216797, -6.363945484161377, -0.033996570855379105, 1.5077604055404663, 2.815237283706665, 0.026072727516293526, -2.5682382583618164, -1.1541500091552734, -3.0555543899536133, -4.718070983886719, -2.534935474395752, -0.493633896112442, 0.89731365442276, 6.3148064613342285, -4.422826766967773, -1.8569494485855103, -2.6182138919830322, 8.247138977050781, -1.9866735935211182, -2.81321382522583, -3.174252986907959, -5.489953994750977, 0.49167338013648987, -7.483124732971191, -1.032074213027954, -3.6807408332824707, -2.203531503677368, 0.8646919131278992, -3.083390712738037, 4.052406311035156, -4.1115031242370605, 0.16731354594230652, 0.7028554081916809, 1.1114288568496704, -0.12695059180259705, 4.891274452209473, 0.259173721075058, -7.612795352935791, -5.656490802764893, -1.558119535446167, 9.16270923614502, 8.884607315063477, -4.919598579406738, 1.2273237705230713, -5.975335121154785, -3.619483232498169, 7.087038993835449, 3.6911566257476807, -2.5669686794281006, 0.014312396757304668, 1.7898085117340088, -0.33517202734947205, -7.307314872741699, -5.364286422729492, 2.014904022216797, 2.4220166206359863, 0.48256316781044006, -3.4788739681243896, 6.1792144775390625, 5.750144958496094, -3.4643750190734863, -1.4782257080078125, -0.5007669925689697, -2.1788699626922607, 1.1027958393096924, -1.349086880683899, -5.064373016357422, -4.767037391662598, -2.492990732192993, 4.236795902252197, -3.483337640762329, -0.9160910844802856, 0.6688987016677856, -0.08336291462182999, -4.581903457641602, 0.9294869303703308, -0.35109952092170715, 4.780374050140381, 4.417998313903809, -0.967704176902771, 0.29528510570526123, -1.1464496850967407, 2.6209137439727783, 0.6252443194389343, 3.3624672889709473, -3.3556323051452637, -1.8720091581344604, 3.925766706466675, 1.0687813758850098, 5.46027946472168, -3.2856366634368896, 2.2102506160736084, 2.810077667236328, -7.137419700622559, -5.376327037811279, -2.1307690143585205, -1.8544906377792358, -4.7691569328308105, -0.6796576976776123, -6.147007465362549, 4.931107521057129, -2.086104154586792, 2.673269033432007, 3.0993857383728027, 0.6670435667037964, 4.494686126708984, -0.6206886172294617, 5.793783187866211, -7.306755065917969, 10.053759574890137, 5.084225654602051, 4.549222469329834, 2.0338776111602783, 3.3224143981933594, -1.815136194229126, 0.8768526315689087, -5.388794898986816, 2.8854360580444336, 1.967660903930664, -2.578179121017456, -2.49558424949646, -0.8006151914596558, -0.47534483671188354, 0.502509593963623, -4.294950008392334, 3.2753021717071533, 4.1256937980651855, 4.396928310394287, -1.8376529216766357, -1.426606297492981, 1.7611334323883057, 0.6680523753166199, 4.379079818725586, 0.5145115256309509, -1.2921947240829468, -1.8811333179473877, 0.45461317896842957, -4.063433647155762, -3.4117586612701416, -3.2024996280670166, 3.432706594467163, -1.3709138631820679, -4.136809825897217, 9.324701309204102, -6.7179765701293945, 3.2684175968170166, 4.192021369934082, -0.6803098320960999, 8.999297142028809, 3.6238458156585693, -5.865432262420654, -1.674880862236023, -0.3735233545303345, 3.238823890686035, 4.123253345489502, 5.6512579917907715, 1.941664457321167, -5.080161094665527, -0.2768605947494507, 11.923181533813477, -8.266776084899902, -4.223485946655273, -8.465792655944824, -0.14296883344650269, -5.890040874481201, 3.4928274154663086, 0.9859223365783691, -0.8679661154747009, 1.9875859022140503, -0.23376405239105225, -10.98690128326416, 4.474578380584717, 1.3570148944854736, -2.986937999725342, -3.4352447986602783, 5.320406913757324, -0.35106346011161804, 5.46710205078125, 3.3606858253479004, 1.3877017498016357, -2.9921374320983887, -0.308189332485199, -1.5799932479858398, -0.7486415505409241, -2.6011013984680176, 3.999265432357788, 6.753237724304199, 1.5998859405517578, 6.106776714324951, -4.2773213386535645, 0.5578562021255493, -2.524380683898926, 7.925806999206543, -0.0649983212351799, -0.449307382106781, -6.1113762855529785, 0.7628298997879028, -4.268307685852051, -0.05499395355582237, 0.886411726474762, 0.9488312602043152, -6.0652875900268555, 12.299409866333008, 0.9345702528953552, -0.7841411232948303, 2.0012471675872803, -0.033800847828388214, -3.353070020675659, 10.899373054504395, -1.696370005607605, 4.9999871253967285, -5.787021636962891, -1.7373003959655762, 0.5246288180351257, 2.1813998222351074, 2.9732553958892822, 2.102323293685913, -2.02642822265625, -4.7473530769348145, 1.2674381732940674, 1.1173449754714966, -1.652297019958496, -1.4583972692489624, -5.70762825012207, 1.455806851387024, -3.006340742111206, -5.3581695556640625, 2.0924575328826904, 0.27158260345458984, -1.5158103704452515, 1.0390527248382568, 3.849283456802368, -1.0020421743392944, -0.9419487118721008, 2.18865966796875, -0.08192327618598938, -2.004262924194336, 5.407803058624268, 1.9379056692123413, 0.19822219014167786, -0.9625829458236694, 7.4565348625183105, -2.3979127407073975, 8.825138092041016, 4.586421489715576, 4.470457553863525, -4.935445785522461, -1.3623671531677246, -4.972492694854736, -3.6949853897094727, -0.9012234807014465, 0.9717588424682617, 2.813915729522705, -3.5231637954711914, -5.43528413772583, 5.7361907958984375, -0.16289016604423523, 0.994167149066925, -0.5722334384918213, -0.06877396255731583, -1.2353228330612183, 4.744088172912598, -0.7207001447677612, -3.1489861011505127, -2.3284995555877686, 4.2109808921813965, -7.486570358276367, 5.942647933959961, -4.4606475830078125, 5.578979969024658, 5.747533321380615, -3.2579286098480225, 9.065558433532715, -1.1641032695770264, -0.48011812567710876, 1.395092248916626, 1.7309434413909912, -1.6344895362854004, 4.385524749755859, -1.0154379606246948, 4.716945648193359, 13.705028533935547, 2.5963242053985596, -0.7615442276000977, 5.182009696960449, -8.013867378234863, -2.0475783348083496, 0.2747591435909271, -4.52922248840332, 6.920011043548584, -7.174902439117432, -5.993127822875977, -7.327133655548096, -0.7054644227027893, 0.6508268117904663, 0.8176771402359009, -1.4699320793151855, -3.280292272567749, 2.363990545272827, 4.519075870513916, -1.8867740631103516, 3.4914560317993164, 4.16880989074707, 2.8494648933410645, 0.8498852849006653, -0.9087672829627991, 2.0218422412872314, 4.093885898590088, -3.5032176971435547, -3.3747036457061768, 0.9828996062278748, -2.953624725341797, -0.11342175304889679, 7.586081504821777, 0.5515782833099365, -2.1790647506713867, -1.1754957437515259, 0.6777307391166687, -0.04545160010457039, 1.285900354385376, 5.170234680175781, -1.8776320219039917, -4.074787616729736, 1.0392874479293823, 6.205573558807373, -9.146383285522461, 2.440215587615967, -1.6822923421859741, -7.569788455963135, 0.018100043758749962, 2.27160906791687, -3.602736473083496, -3.653472900390625, 0.7969871163368225, -4.871601104736328, -2.4223837852478027, 2.819855213165283, -1.7414233684539795, -1.3282924890518188, 0.5511824488639832, -8.818486213684082, -3.8337349891662598, 0.05050119012594223, -2.8930718898773193, -2.4572911262512207, -1.7590998411178589, -2.2560603618621826, 11.436006546020508, -5.887208938598633, -7.470371246337891, -10.264357566833496, -1.1050469875335693, 2.4839282035827637, -3.479144334793091, 4.254973888397217, -1.919450283050537, -2.386470079421997, -3.516451835632324, -0.054991595447063446, -0.29287147521972656, 2.9643137454986572, -1.95433509349823, 5.143054008483887, 2.5578601360321045, 0.5701910257339478, -0.29188066720962524, -0.44908711314201355, 10.146394729614258, 4.8896894454956055, -1.860824704170227, -6.320207595825195, -2.723816394805908, -2.190593719482422, -3.9626030921936035, -1.6495583057403564, 1.3697175979614258, 3.45770263671875, 2.111438751220703, 1.616000771522522, 1.3611986637115479, -2.4474189281463623, 1.1540712118148804, 0.5347926020622253, 9.456369400024414, -5.732061862945557, -2.8593523502349854, -0.19413620233535767, 0.6632919311523438, -0.2510087788105011, -5.396728515625, 0.46616968512535095, 1.75901198387146, 7.916433811187744, -4.879833221435547, 2.3781135082244873, -6.886427879333496, 2.0844616889953613, 1.9924792051315308, -4.3698320388793945, -6.7628655433654785, 3.7515194416046143, -7.619555950164795, -0.7968688607215881, -4.145376682281494, 4.368778705596924, 4.118210792541504, -1.0094507932662964, -1.0295933485031128, 3.261622428894043, 5.859659671783447, 3.037651538848877, 1.0217359066009521, -7.887892246246338, 2.255356788635254, -4.418866157531738, 4.224586486816406, -0.044600795954465866, -3.616743326187134, -1.9868113994598389, 6.169064044952393, -2.0832371711730957, 4.245199680328369, 1.9138821363449097, -7.6095404624938965, 0.998198390007019, -3.0458381175994873, 1.2751001119613647, 3.616450071334839, -2.025176763534546, 2.7502758502960205, -0.006050813477486372, -3.6412525177001953, 0.9364066123962402, -8.57966423034668, -0.4715186357498169, -7.961648464202881, -1.0494847297668457, 4.4498066902160645, -0.7443537712097168, -4.111330032348633, -1.9974174499511719, 5.672210216522217, 0.3175804316997528, 2.1965324878692627, 1.6950452327728271, 0.8348520398139954, 7.560482025146484, 4.282439708709717, 2.9892704486846924, 3.042814254760742, -4.051575183868408, 4.922932147979736, 7.030683994293213, 0.7357603311538696, 3.8476953506469727, 4.549134731292725, -3.630690574645996, 0.48223668336868286, -2.270738124847412, 7.1926164627075195, -5.338705062866211, 1.4451309442520142, 0.8538103699684143, -5.29454231262207, -7.212493419647217, 0.5089547038078308, -2.4310781955718994, 3.461793899536133, 1.6522650718688965, -0.5925753712654114, -5.912609577178955, 5.59470272064209, 1.5289433002471924, 0.6102772355079651, 3.412003517150879, -3.347771644592285, -5.701722621917725, 1.0377660989761353, -3.8649935722351074, 2.298133611679077, -2.115828275680542, 2.8588624000549316, -6.4124650955200195, -4.54806661605835, -2.866725444793701, -3.28717303276062, 2.427356481552124, 5.168914794921875, 1.9362128973007202, -5.409635543823242, 0.4246056377887726, -4.66653299331665, -3.9307172298431396, -1.9994187355041504, 1.2383496761322021, 0.9946343898773193, 7.026412487030029, 2.5618748664855957, -1.8435148000717163, 4.398613929748535, 1.7152788639068604, -3.550351858139038, 4.4694342613220215, 2.0722949504852295, -1.6534197330474854, -0.3964163362979889, 6.569073677062988, 0.3722429871559143, 4.109262943267822, 3.3130455017089844, 2.141841411590576, -7.1704792976379395, 2.8696706295013428, 6.199944496154785, -4.581465244293213, 0.6543178558349609, -0.535743236541748, 2.669637680053711, 2.1232707500457764, -6.46793270111084, 3.1911978721618652, -8.047598838806152, 3.8767993450164795, 7.261173248291016, -5.944914817810059, -0.6086012125015259, -6.556564807891846, 3.291670799255371, -2.3621127605438232, 0.521836519241333, 7.260562896728516, 3.5284671783447266, -0.005258309654891491, 1.2730263471603394, 1.6200135946273804, 2.8545267581939697, -2.6051323413848877, -7.142526149749756, 0.5652424693107605, 8.887432098388672, -0.1723509430885315, -0.9605560898780823, -3.9515492916107178, -0.4082624316215515, 1.279884696006775, -0.07214440405368805, -0.8989650011062622, -0.6893433928489685, -1.9165334701538086, 1.0615155696868896, -7.181141376495361, 1.7502251863479614, -6.223669528961182, 0.2818627953529358, 2.5917582511901855, -4.511792182922363, -0.6833139061927795, -1.3373372554779053, 0.8895847797393799, 3.6293740272521973, 2.0288355350494385, 7.146755218505859, 1.916658878326416, -2.43994402885437, 3.560014247894287, 4.768767833709717, 0.07672242075204849, 4.168753623962402, -1.5500198602676392, -0.9866117835044861, 2.629889726638794, -1.6103819608688354, 3.636453866958618, -0.5748530626296997, 0.4102957546710968, -2.73396897315979, -4.352607727050781, -3.4270713329315186, -1.010326623916626, 3.6806769371032715, -4.045448303222656, -4.513922691345215, -8.255719184875488, -0.34234389662742615, 5.833856582641602, -1.8872179985046387, -2.8297221660614014, -4.090031147003174, -3.176126480102539, 0.5923829674720764, -5.5043840408325195, 1.125303030014038, 2.3158388137817383, 9.514243125915527, -2.8399646282196045, -6.727274417877197, 3.6591484546661377, 0.00940193422138691, -1.3921931982040405, -0.9659103155136108, 0.9001306295394897, 0.5101720094680786, 3.4661080837249756, -1.7135236263275146, -0.6108493208885193, 3.944704294204712, 2.1327807903289795, -5.785102367401123, 0.9897937774658203, 5.931858062744141, -4.466623783111572, -3.9846038818359375, -3.46995210647583, 0.7262020707130432, -3.2097644805908203, -1.8436317443847656, -2.266322135925293, 3.6651785373687744, 0.4077635407447815, -0.6115157008171082, -3.696486711502075, 1.102196216583252, -5.8394598960876465, -11.54573917388916, -0.5944287180900574, -0.34393206238746643, 3.4893851280212402, -1.2071367502212524, -3.723097801208496, -2.2775156497955322, -1.755333423614502, 0.5682629346847534, 0.7261945605278015, -4.04098653793335, 8.1815824508667, -2.995422840118408, 1.202593207359314, -2.153531789779663, -9.320173263549805, 0.7864829897880554, -1.7221083641052246, -0.9388015270233154, 7.270392894744873, 2.1836740970611572, 3.2185451984405518, 4.398282051086426, 1.581984281539917, -3.3707022666931152, 13.743199348449707, 3.028035879135132, -8.5935640335083, -1.4749107360839844, 0.17705339193344116, -13.80412483215332, 3.1120755672454834, -2.0275909900665283, -5.297431468963623, -3.068542957305908, -3.758720636367798, 1.366448163986206, -2.380581855773926, 2.073476791381836, 6.889457702636719, 3.1626360416412354, 1.6684186458587646, 1.8865697383880615, 2.8809759616851807, 1.3546714782714844, -2.466731548309326, -1.6597821712493896, 0.6214208602905273, 6.122578144073486, 2.8990283012390137, 1.2380504608154297, 2.6301915645599365, -2.4282326698303223, 1.042188286781311, -6.5354180335998535, -0.39101141691207886, -1.7880134582519531, 2.0909171104431152, 0.48137640953063965, 6.4343671798706055, -3.819840669631958, 1.954664945602417, -3.8359947204589844, 2.8591737747192383, -3.6617000102996826, -6.7740936279296875, -2.591403007507324, -0.005778305232524872, 3.6026158332824707, -5.57417106628418, -4.52281379699707, -2.4286181926727295, -0.20236027240753174, -0.6861016750335693, 4.052128791809082, 3.932662010192871, 4.725568771362305, 2.1723527908325195, 3.4539639949798584, -7.342297077178955, 5.7751784324646, -4.44289493560791, 2.226116895675659, -7.766936779022217, -1.480054259300232, 7.520319938659668, 3.3626859188079834, 0.8680110573768616, -7.613036632537842, -4.450740814208984, -0.7302674651145935, 2.245978355407715, 1.010233759880066, -3.2024986743927, 2.3462376594543457, 7.2233171463012695, 4.535435199737549, -1.2196484804153442, -5.750782489776611, 5.691661357879639, -0.8404677510261536, -3.3022589683532715, -2.5226118564605713, -4.133430480957031], [1.5673928260803223, 1.021665096282959, -0.2452140897512436, -3.4480509757995605, 1.067432165145874, -3.152127265930176, -1.1695823669433594, -3.11152720451355, -2.903482437133789, 1.030356764793396, -6.564744472503662, 6.804678440093994, 7.581025123596191, 2.3191940784454346, 6.420464992523193, 8.246423721313477, 0.8371117115020752, 5.10627555847168, 5.350265026092529, -0.17667879164218903, -4.111140251159668, -4.413475513458252, 4.629796504974365, -5.099764823913574, 0.33285748958587646, -0.2485324740409851, -4.534420013427734, -3.6902711391448975, -4.141360759735107, 6.118192195892334, -1.626465916633606, 1.0965937376022339, 1.230689287185669, 0.3050999939441681, 3.0823378562927246, -3.3362746238708496, 1.1525993347167969, 2.7013704776763916, 3.489636182785034, -3.9164955615997314, -0.06351418048143387, -1.947082281112671, -1.9585580825805664, -0.7666288018226624, -4.602409839630127, 0.26497507095336914, 4.5062127113342285, 1.6289939880371094, -3.3219668865203857, 0.6895753145217896, 3.3459908962249756, -6.41338586807251, -6.232821941375732, -1.2075895071029663, -3.69651198387146, 6.867403984069824, 0.18489663302898407, -0.13931767642498016, -0.8036429286003113, -2.621574640274048, -0.9881219863891602, 6.900582313537598, -1.411076545715332, 5.085241794586182, 0.8863720893859863, -0.7294240593910217, -1.3131288290023804, -6.8742995262146, -8.965404510498047, 7.935150146484375, 2.8298144340515137, 8.78732967376709, 1.3735963106155396, 4.070159912109375, 0.24537871778011322, 7.6564717292785645, -3.5253167152404785, 6.347637176513672, -8.153568267822266, 0.5119619965553284, -0.460183322429657, -1.9731272459030151, -6.005072593688965, -2.9183597564697266, 0.9859002232551575, 7.045266151428223, -0.4025462567806244, 3.7242238521575928, -6.81047248840332, -0.19439521431922913, -0.0033163849730044603, 4.719519138336182, -0.22702060639858246, 3.763016939163208, 5.2678446769714355, 1.4624816179275513, 3.1214687824249268, -2.1277334690093994, -3.010875940322876, 6.979156494140625, -4.616318702697754, -0.6109616160392761, 4.46481990814209, 1.9934405088424683, -4.859885215759277, 0.4946211278438568, 2.1761770248413086, 0.46594706177711487, 0.03682245686650276, 4.01393461227417, 1.499491572380066, 0.06284232437610626, 2.228198766708374, -8.25337028503418, -2.546196460723877, 4.847684383392334, -0.93339604139328, 2.109049081802368, 3.1908087730407715, -9.521071434020996, -5.416541576385498, 0.6343809962272644, 1.4761972427368164, -2.2093915939331055, 7.413813591003418, 2.571840286254883, 0.6109789609909058, -2.829178810119629, -11.830512046813965, -1.6591264009475708, -3.666274309158325, -4.13026762008667, -2.808626174926758, 4.564178943634033, -3.941654682159424, -1.2367831468582153, -1.284024953842163, -4.438321590423584, -2.2497735023498535, -0.05474089831113815, -0.28464239835739136, -6.500472068786621, 7.357505798339844, 6.992262363433838, -0.38147562742233276, -0.3605634570121765, -3.1202197074890137, -4.753849983215332, 1.7525054216384888, -2.7837555408477783, -2.02547025680542, -0.19746141135692596, 0.031724318861961365, 1.1401475667953491, -0.34336596727371216, 7.383743762969971, -5.100597381591797, -3.2681143283843994, 4.370794296264648, -0.9711912870407104, 1.2059876918792725, -0.19806848466396332, -2.1319868564605713, -2.216693878173828, -4.49478816986084, -0.4487069249153137, 7.2651166915893555, 4.742243766784668, 8.744922637939453, 3.4521281719207764, 0.21050085127353668, 3.3761651515960693, 0.4013960361480713, -4.052923679351807, -3.3449642658233643, 5.131352424621582, -1.796395182609558, -5.31479024887085, -1.0920401811599731, 4.2960591316223145, 3.828826427459717, 0.6331626176834106, -5.570982933044434, 0.026636427268385887, -1.8785779476165771, 10.037566184997559, -0.6375584006309509, -0.10948526859283447, -3.9598405361175537, -6.527202606201172, -0.18677067756652832, 0.46898749470710754, 6.286142826080322, -1.6220786571502686, -0.4785519540309906, 6.593632221221924, 2.537184476852417, 2.9600117206573486, 2.9957432746887207, -8.175252914428711, 1.8853908777236938, 9.789854049682617, -2.2250938415527344, -3.7098138332366943, 4.439797401428223, 6.3402228355407715, 0.9404850602149963, -8.550922393798828, -4.928042888641357, -3.7087721824645996, -3.140955686569214, -4.261632442474365, -1.1732112169265747, -2.2162272930145264, -0.5217862129211426, 0.4965008497238159, -0.38093364238739014, -3.394291877746582, 2.017439842224121, -3.4558792114257812, -1.42487633228302, -1.7082544565200806, 2.800246238708496, -0.5788848400115967, 1.001230239868164, -4.832986831665039, 2.2044448852539062, 0.02110847271978855, -0.5597103834152222, 3.9081525802612305, -3.6005918979644775, 1.9802162647247314, -7.4207563400268555, -3.680711269378662, 4.693698406219482, -0.4969765245914459, -5.438411235809326, 4.87714958190918, 0.7927512526512146, 1.1128374338150024, -6.147387504577637, 2.9200003147125244, 2.1252191066741943, -1.3810153007507324, -0.22442646324634552, -3.0239086151123047, 0.20021653175354004, -5.1583709716796875, 1.8196380138397217, 0.7776167988777161, -4.0558295249938965, -0.07217174023389816, -3.4406919479370117, -0.4998478293418884, -5.48588228225708, -1.6248053312301636, 7.0467610359191895, 2.2849977016448975, 4.720138072967529, 7.510164260864258, -4.011256694793701, -0.03093133307993412, -3.37410569190979, 6.36868143081665, 5.1257500648498535, 6.67988920211792, -5.241864204406738, 4.908782482147217, 0.04696269333362579, -2.7826039791107178, 1.0727710723876953, -5.692926406860352, 2.6238343715667725, -4.492871284484863, -2.4780054092407227, 2.4761059284210205, 3.1665537357330322, 3.009425640106201, 0.2915486693382263, -1.3110945224761963, 0.5356672406196594, -1.1739972829818726, 3.5474047660827637, 1.5328099727630615, 0.09979184716939926, 2.95505428314209, -4.584600448608398, -3.6853322982788086, 3.843935966491699, 2.0917415618896484, 0.06464516371488571, 1.646456241607666, -1.1132984161376953, -2.3057432174682617, 1.0131129026412964, 3.757345676422119, 4.630652904510498, 1.6000577211380005, -0.16082391142845154, -4.954712390899658, 0.943220317363739, -1.6936407089233398, 2.965245008468628, -3.1621227264404297, -1.1861783266067505, -4.141902923583984, 3.4087915420532227, 2.3038907051086426, -7.145751953125, -3.079622983932495, 0.11414291709661484, 5.071383476257324, 7.494673252105713, 4.365484714508057, -0.7953228950500488, 2.0712826251983643, -3.1343092918395996, 6.068629741668701, -4.9068756103515625, -6.81429386138916, 1.3162811994552612, 3.9704160690307617, 5.12798547744751, 1.7711176872253418, 2.5228865146636963, -1.6701079607009888, -1.5797269344329834, -3.7979676723480225, 1.069219946861267, -4.311089515686035, -3.4879939556121826, -4.492855548858643, 1.7028931379318237, 1.5353906154632568, 0.5337480902671814, 3.8337857723236084, -6.332508087158203, 0.3433689773082733, 4.7518110275268555, 4.770438194274902, -3.9553749561309814, 1.1536498069763184, -1.894117832183838, -0.7860560417175293, -2.566348075866699, -3.039980411529541, 1.348739743232727, 6.52400016784668, -1.5438185930252075, -0.4096381962299347, -6.537024021148682, -0.4882749319076538, 0.015318678691983223, -0.4822089672088623, 3.0298566818237305, 5.007923603057861, -3.881793260574341, -3.520284652709961, -3.928314447402954, -1.7179677486419678, -3.862898111343384, 0.8993433117866516, -5.584717750549316, -5.956647872924805, -5.498337745666504, -4.800675392150879, 0.9657883644104004, 1.627017617225647, 0.9680983424186707, 1.3702110052108765, -0.01617780141532421, 4.4783501625061035, -1.1433885097503662, 1.973904013633728, 6.472840785980225, -3.4687116146087646, -3.4229328632354736, -0.6540822982788086, 7.19919490814209, 8.452848434448242, -2.385714530944824, 7.413795471191406, -0.9298218488693237, -4.3653669357299805, 1.1416828632354736, -0.6028639674186707, -0.7764106392860413, -4.907720565795898, -3.629535675048828, 1.1910784244537354, 2.390658378601074, -1.3908225297927856, -5.052415370941162, -0.4516477584838867, -0.17905548214912415, -3.9809136390686035, 0.8005894422531128, -1.426842451095581, 3.536287307739258, -2.742487668991089, 2.5924806594848633, 4.452141284942627, -5.168378829956055, -1.732192039489746, -5.722292423248291, -0.548431396484375, -0.6714637279510498, -3.750032901763916, 3.1596953868865967, 0.6364114284515381, 0.0165616013109684, -1.3305314779281616, -4.409512519836426, 4.543586730957031, -1.7854549884796143, 2.8605785369873047, -11.191427230834961, -0.424786776304245, -8.164766311645508, -2.355785608291626, 3.324122905731201, -0.16590435802936554, -4.535172939300537, 3.9316112995147705, -4.747236728668213, -6.062290668487549, -0.6158492565155029, -6.752955436706543, 11.211833953857422, 6.167544841766357, -3.572213649749756, 5.561446189880371, 2.2911741733551025, -5.423311710357666, 4.118572235107422, -0.8603367805480957, 1.2290626764297485, 2.0851807594299316, 2.486506700515747, -4.112441539764404, 4.069416522979736, -0.8028216361999512, 8.741003036499023, -1.762102484703064, -4.826205253601074, 0.8111924529075623, -7.150409698486328, -9.718806266784668, -0.4180072844028473, 3.257024049758911, 1.5282262563705444, -3.2092716693878174, -1.8293671607971191, -4.379526615142822, -1.2741962671279907, -1.7506846189498901, 3.1534879207611084, -1.287839651107788, 7.59215784072876, -3.4227077960968018, 1.5540045499801636, 2.80499005317688, 1.6592175960540771, 3.2195401191711426, 9.527331352233887, 4.217087745666504, 3.354764938354492, 4.147374153137207, -3.282078266143799, -5.498165130615234, -1.8816509246826172, -4.408395767211914, 2.3436665534973145, -9.801129341125488, 0.8541300296783447, 2.1002352237701416, -3.5655269622802734, 3.7078843116760254, 0.010875222273170948, 2.127730131149292, -2.480687141418457, 3.050906181335449, 8.503288269042969, 0.9170046448707581, 1.9239269495010376, 3.4456169605255127, 1.5134897232055664, -4.106265544891357, 2.3508543968200684, -8.31688117980957, 2.290128707885742, -3.477534055709839, 2.6824705600738525, 5.651301860809326, -4.576298713684082, -0.2037222534418106, -0.6680417656898499, -1.4832695722579956, -1.4055038690567017, -5.118603229522705, 6.616652965545654, -2.1616740226745605, -6.369150638580322, 2.8734681606292725, -3.7634189128875732, -0.7753806710243225, -0.00927414745092392, -6.440315246582031, 2.109480381011963, 1.5713765621185303, 4.859457015991211, -6.225643157958984, -0.03477240353822708, 1.088197112083435, 0.28847312927246094, 4.62483024597168, 0.14922253787517548, -0.7801573276519775, 0.005195942707359791, -0.9821752905845642, 1.770492672920227, -2.9583306312561035, -1.5936074256896973, 1.271348476409912, 4.3545403480529785, 0.6035388708114624, 4.245100975036621, -2.626453399658203, -0.9147382378578186, 3.2167980670928955, -2.5251903533935547, 5.2754974365234375, -9.792325019836426, 2.6534488201141357, -2.8273332118988037, 0.7721444964408875, 0.003928192425519228, 2.480574131011963, -6.105679512023926, -1.5325738191604614, 5.1259589195251465, 1.726326584815979, -1.9502581357955933, 2.032515287399292, -2.2272350788116455, -9.438458442687988, -2.8096132278442383, 0.801717221736908, -6.353212356567383, -5.201216697692871, 1.1268231868743896, 5.221533298492432, 0.2850039005279541, -5.1363301277160645, 0.889684796333313, 0.023375598713755608, -0.7733626961708069, 0.05319579690694809, -3.6371636390686035, -3.656815528869629, -4.873100280761719, 6.847079753875732, 4.34736967086792, -4.87593412399292, -2.6567423343658447, -4.027915954589844, 2.729308605194092, 2.5827202796936035, 5.361629962921143, 1.152932047843933, -1.9223655462265015, 0.8477039337158203, 2.1374666690826416, -1.7075810432434082, -2.802431106567383, -4.228662967681885, 4.542755603790283, -0.1232355386018753, 6.255697727203369, 3.692073106765747, 0.576008677482605, 0.08371289819478989, 0.8305052518844604, -3.63706374168396, -3.2594940662384033, 1.1138979196548462, -4.9490132331848145, 6.47457218170166, 5.029089450836182, -1.7727919816970825, -0.3740074038505554, 1.253369688987732, -3.125516414642334, -2.999844789505005, -1.0634403228759766, 6.104457855224609, 0.717404842376709, 3.973849058151245, -8.956817626953125, 2.965956211090088, 1.9395887851715088, 1.0285319089889526, -5.9811530113220215, 2.072441339492798, 0.2614259719848633, 0.6868723630905151, -6.317885398864746, 7.519382476806641, -0.9452005624771118, -4.213018417358398, -0.4842630624771118, 1.6004587411880493, -0.7205463647842407, -4.393773555755615, 1.6333935260772705, 5.548994064331055, -2.1346874237060547, -0.025661440566182137, 1.3926546573638916, 1.1035815477371216, -3.258610248565674, 3.252005100250244, -5.751314163208008, -0.46156302094459534, 2.1966452598571777, 4.987859725952148, -8.066170692443848, 4.829063415527344, 4.445323944091797, -4.772644996643066, 7.158459186553955, -1.1389029026031494, -2.042843818664551, 3.0793747901916504, -3.8704898357391357, 4.338000297546387, -8.985934257507324, -1.1552352905273438, -8.076997756958008, 4.8420023918151855, 1.5201449394226074, -8.845754623413086, -2.216891288757324, 6.664656639099121, -0.5200067162513733, -13.491033554077148, 2.6922271251678467, -7.692379474639893, -1.0733574628829956, 9.285645484924316, 0.6107243299484253, 2.9161577224731445, 4.937912940979004, -4.800187587738037, -2.329779863357544, -1.4781076908111572, 3.804182767868042, -2.1509671211242676, -2.0907480716705322, -1.0183579921722412, -3.5500800609588623, 0.10670121014118195, -3.269721031188965, 0.3977290391921997, 2.232393980026245, -3.4906728267669678, -2.365100622177124, 0.9004907608032227, -4.728077411651611, 5.046500205993652, -3.81878399848938, 0.6208120584487915, 0.05872345715761185, -2.060260772705078, 2.451434373855591, 0.1313200742006302, -7.8471360206604, -2.2415595054626465, 4.501485824584961, -5.020169734954834, 11.712189674377441, 7.580137252807617, -2.9073383808135986, 6.338375091552734, -5.326608180999756, -2.591411828994751, 5.086503028869629, 8.751227378845215, 3.9200613498687744, 4.6441779136657715, -1.7276548147201538, -0.9184059500694275, -2.6105599403381348, -1.8073832988739014, 1.7375438213348389, 4.255209922790527, 11.633944511413574, -8.297882080078125, -3.8691773414611816, 2.1188089847564697, 3.614391326904297, 1.0802682638168335, -2.8860647678375244, -5.5510334968566895, 6.5423994064331055, -0.9290977120399475, -0.8924939632415771, 0.6728281378746033, 4.0891289710998535, 7.550107955932617, 6.16694450378418, 3.567418098449707, -0.22891846299171448, 2.7122442722320557, 0.9815307259559631, -3.6637887954711914, -0.37625864148139954, -3.4732723236083984, -5.8244309425354, 4.474827766418457, 0.045673616230487823, -5.9885478019714355, 0.9151152968406677, -1.1648330688476562, -3.4094293117523193, 0.6409425735473633, 9.24327278137207, 2.5542733669281006, -0.26468271017074585, -0.4699292480945587, -0.5605031847953796, -2.8925673961639404, -3.4948678016662598, 4.90846061706543, -3.8347578048706055, 1.9170773029327393, -5.24699592590332, 3.189725637435913, -5.259235858917236, 1.2905619144439697, 7.7719407081604, -4.062253475189209, 2.6127986907958984, 2.0603442192077637, -2.1413841247558594, -0.9381343722343445, 3.600538969039917, -5.0450758934021, -6.371430397033691, -2.469433546066284, -4.606680393218994, -1.923230767250061, -6.512256622314453, 0.5073844790458679, 0.19017642736434937, 2.765240430831909, -2.493056535720825, 4.1753740310668945, 3.882427215576172, 3.337737560272217, -1.217227816581726, -0.679282546043396, 8.786336898803711, -3.284301280975342, -3.8353898525238037, 1.928464651107788, -2.8917410373687744, -4.001391887664795, -2.434363603591919, 3.659341812133789, -6.806580066680908, -3.0735859870910645, 6.857624053955078, -2.9061107635498047, -5.61124849319458, -5.051110744476318, -0.9537689685821533, 1.5370076894760132, -3.6011080741882324, -6.240917205810547, -3.2637832164764404, 0.6210581660270691, 0.26266786456108093, 6.361326217651367, 0.40342801809310913, -0.45592200756073, 7.930065631866455, 4.167928695678711, 0.742557942867279, -3.4380033016204834, 2.4242217540740967, -4.2437334060668945, -1.0715981721878052, -0.9723413586616516, 1.9387385845184326, 5.120713233947754, -5.694023132324219, 8.824178695678711, -1.7056820392608643, 6.876741886138916, -6.073732376098633, 1.7573671340942383, -4.337268829345703, -6.614155292510986, -1.5080667734146118, -3.0257725715637207, -7.439698219299316, -4.004238128662109, 2.3751273155212402, 2.010437488555908, -1.9278442859649658, -2.1500847339630127, 2.574469804763794, -2.86541748046875, -4.961405277252197, 1.7268201112747192, -2.1580581665039062, 0.38831302523612976, 3.2521016597747803, 11.666329383850098, 0.4796091616153717, -3.4488375186920166, 4.79530143737793, -4.5545549392700195, 1.212073802947998, 3.5122756958007812, -2.18684983253479, 0.942990779876709, -0.9192986488342285, -4.360440731048584, 2.4415502548217773, 8.342597961425781, 5.400754928588867, -2.711474895477295, 1.1130738258361816, -5.816135406494141, 2.7699759006500244, -1.9721744060516357, -4.237377166748047, -4.543815612792969, 0.7795825600624084, 0.310879111289978, -5.594657897949219, 5.377716064453125, -7.763174057006836, -1.4148368835449219, 1.546222448348999, 2.0030670166015625, -7.2058820724487305, -4.418887615203857, 1.761263370513916, 4.30077600479126, -4.815794944763184, -1.7258392572402954, 3.8635332584381104, 1.0938911437988281, -1.5036109685897827, -1.7258172035217285, 3.6942923069000244, 2.568464517593384, 0.30694758892059326, 3.539409875869751, -2.5645339488983154, 0.4329717755317688, 4.211512565612793, 2.2710494995117188, 1.490431308746338, -6.275659084320068, 2.4591245651245117, -3.3288705348968506, 2.402813673019409, -7.4361371994018555, 3.569990634918213, -2.2991530895233154, -9.74860668182373, 4.877946376800537, -4.64197301864624, -4.025007247924805, -3.353640556335449, -1.2026853561401367, 6.486642837524414, 2.1350433826446533, 4.066198825836182, 6.553318023681641, -3.8407957553863525, -3.4336912631988525, -3.2334914207458496, 3.616590976715088, 1.2053186893463135, -1.509886384010315, 5.499511241912842, 0.4394606351852417, 2.021867036819458, 1.0549345016479492, -0.6058447957038879, -1.752550482749939, -0.6159448027610779, -3.802795171737671, 4.351324081420898, 2.7792627811431885, 2.9461922645568848, 1.8408149480819702, -0.14528758823871613, -11.162677764892578, 2.933173894882202, 1.7159013748168945, -3.1032726764678955, 0.09361647814512253, 0.46268805861473083, 5.330077171325684, 0.15129806101322174, -2.9731807708740234, 5.91453742980957, -5.767411708831787, -1.8102202415466309, -9.121774673461914, 10.764324188232422, 0.363089919090271, -6.183386325836182, -5.155502796173096, 2.1306586265563965, -1.519300103187561, -4.345438480377197, 5.134735107421875, 0.23481814563274384, -0.05274278298020363, 11.285526275634766, 6.699343204498291, -4.466416358947754, -1.3070077896118164, -1.323137640953064, -1.4143915176391602, -4.032263278961182, 5.253758907318115, 2.5622410774230957, 0.36170798540115356, -5.5281291007995605, 0.5666712522506714, -2.3298490047454834, 7.398110866546631, 0.8365305066108704, -0.15205638110637665, 2.5337131023406982, -0.5631424188613892, -2.5199129581451416, 14.076078414916992, -2.0924324989318848, 4.45900821685791, -7.072745323181152, 2.5188546180725098, 4.8530426025390625, 2.1351869106292725, 4.193291664123535, -7.001460075378418, -2.3355565071105957, -4.158974647521973, 2.869650363922119, 1.699684977531433, 2.2822723388671875, -2.7453227043151855, 1.9508953094482422, -0.19454911351203918, -0.7937448024749756, -4.2427754402160645, 1.4654606580734253, 2.484431028366089, 1.6009124517440796, -3.701716184616089, 0.38550323247909546, 9.06085205078125, -2.757429361343384, 2.75547456741333, 0.4183570146560669, -4.451892375946045, -5.819429397583008, 3.3611767292022705, 0.6583946347236633, -2.2934622764587402, 9.67335033416748, -1.9029055833816528, 2.035651206970215, -1.8729479312896729, -1.7302308082580566, 0.8336336612701416, -4.303611755371094, 5.266904830932617, -5.9952287673950195, 4.146162033081055, -1.224390983581543, -0.7585652470588684, 2.4801666736602783, 3.1675212383270264, -3.8041775226593018, -4.935699939727783, 3.229050397872925, 2.2320902347564697, 3.750108003616333, 6.979434967041016, 3.1327083110809326, -5.310080051422119, 0.6398404836654663, -0.05637088418006897, -0.13196150958538055, 5.104734897613525, 0.003265523351728916, 5.488870143890381, -1.993074655532837, 4.799756050109863, -5.8797926902771, 0.5405968427658081, -0.5323115587234497, -5.283357620239258, -3.6790049076080322, 1.4363155364990234, 4.00516414642334, 2.237502336502075, 1.600694179534912, 2.8176348209381104, -1.8473529815673828, 0.22550076246261597, -2.4518229961395264, 0.34636157751083374, -2.0096194744110107, -0.7077109813690186, -10.927046775817871, -5.569140911102295, 4.008894920349121, -1.7785488367080688, -2.5988690853118896, -1.609671711921692, 3.292083978652954, 5.305017948150635, 4.610468864440918, 4.833254337310791, -3.4462108612060547, 0.8955443501472473, -8.165555953979492, -2.4741709232330322, 0.7935766577720642, 3.980164051055908, 5.646790981292725, -8.389986038208008, -0.16502784192562103, 7.933780670166016, -3.507723569869995, -2.6763439178466797, 0.23394657671451569, 7.52199649810791, -8.080522537231445, -1.737617015838623, 7.629701137542725, -0.601951539516449, -4.439136028289795, -4.280712604522705, 5.266988754272461, 1.2915314435958862, 4.22315788269043, 4.80225944519043, 2.4834108352661133, 2.297197103500366, 0.5203728675842285, 3.991253614425659, -4.810044765472412, 3.409479856491089, -3.739999532699585, -1.8546024560928345, 2.9457590579986572, -1.9556587934494019, -9.789679527282715, -2.919313430786133, -2.083219289779663, -0.33814945816993713, -2.829505681991577, 4.664731979370117, -4.140864849090576, -0.756069004535675, -4.486568450927734, 2.3764588832855225, 2.9257302284240723, 3.054231643676758, 3.2209246158599854, 0.43588054180145264, 4.2784247398376465, -1.9079337120056152, -4.2988457679748535, -2.336723804473877, 3.3462603092193604, 3.2005367279052734, -1.328713059425354, -0.4156014919281006, -5.2656168937683105, -1.7479075193405151, -2.9383063316345215, 2.390270471572876, -1.928544044494629, 11.12682056427002, 3.920883893966675, -3.885636806488037, -0.72508305311203, -9.218992233276367, -2.5677688121795654, 1.1265583038330078, -7.267820358276367, -0.24224698543548584, -2.3946425914764404, -2.6660032272338867, 1.171880841255188, -9.156078338623047, -3.0879743099212646, -1.328673005104065, 5.281789302825928, -5.459028244018555, 6.82289457321167, 2.5257534980773926, -1.9466924667358398, 2.373086929321289, -0.6513568758964539, -2.7857582569122314, 5.575307369232178, -3.2764816284179688, 6.091463088989258, -0.8964825868606567, 0.6842899918556213, -0.9808529615402222, 3.037952184677124, -2.268432378768921, 1.3837336301803589, 1.1004935503005981, -1.7065560817718506, 2.572218418121338, -2.809279203414917, 3.0851857662200928, 3.948209762573242, 6.963723182678223, -0.028708012774586678, 0.16666319966316223, -1.1282461881637573, -3.0418312549591064, -6.005833148956299, 5.4290080070495605, -3.9851670265197754, -2.786881685256958, 0.980165421962738, 8.644708633422852, -6.720540523529053, -4.776172161102295, 1.05889093875885, 1.7602299451828003, -0.044820379465818405, -1.993499517440796, -1.6437344551086426, -0.8490923643112183, 3.4374969005584717, 4.50984525680542, -1.0104985237121582, 1.8122305870056152, 6.150050163269043, 1.5954190492630005, -4.128068923950195, 1.8160419464111328, -1.969854474067688, 0.09332618862390518, 0.6585847735404968, 3.0587315559387207, 0.2079792022705078, 4.776302337646484, 2.520730495452881, 4.25295877456665, -0.8265213370323181, 6.29111909866333, -2.092404365539551, 0.07437042891979218, -6.5903401374816895, -0.9750255346298218, 6.383248805999756, -5.195993900299072, 0.05196379870176315, -0.3777320981025696, 3.6128971576690674, 10.434975624084473, -0.9406882524490356, 0.12120182812213898, -3.9384043216705322, 3.947618246078491, 3.703336238861084, 0.9168413877487183, 4.212879657745361, 0.143487811088562, 2.253466844558716, -0.1372668743133545, -1.1132659912109375, 7.02243709564209, -7.350878715515137, -3.9080262184143066, -2.361628770828247, 0.7689488530158997, 0.4058183431625366, 4.6923017501831055, 1.3558050394058228, 1.8023768663406372, 5.838948726654053, 0.5361078977584839, 3.945218324661255, 5.8600921630859375, -1.9524487257003784, 5.367669105529785, 7.194510459899902, 10.005754470825195, 0.04729749634861946, 3.0362532138824463, 1.6284233331680298, 3.435147523880005, -3.7669036388397217, -3.2580244541168213, -2.3736298084259033, -1.3833003044128418, 0.05630705878138542, -9.322343826293945, 5.753911018371582, -0.6388518214225769, -4.556753635406494, -1.7748079299926758, -8.61517333984375, 2.7716383934020996, 6.777988433837891, -0.39182209968566895, 2.5966436862945557, -0.11757990717887878, 1.1410096883773804, 3.7675271034240723, 1.5565146207809448, -4.713886260986328, 0.1482713520526886, 1.7935107946395874, 7.516205310821533, -5.972966194152832, 6.5547966957092285, -2.5705249309539795, 4.364030838012695, 6.225240230560303, -4.363345146179199, -4.58922004699707, -3.16850209236145, -1.8499343395233154, 7.537951946258545, -3.9470698833465576, 3.392533540725708, -11.381135940551758, -6.013918876647949, -5.338399887084961, 4.663926601409912, -4.3104681968688965, 1.5440810918807983, 0.14558415114879608, 0.5839454531669617, -5.488344192504883, -4.138970851898193, -3.785465955734253, -2.0694777965545654, 2.5091872215270996, 1.7116990089416504, -0.9769219160079956, 2.9801642894744873, -3.990314483642578, -2.786949872970581, -5.627710342407227, 1.8079696893692017, -1.7823652029037476, -1.764000654220581, -9.260208129882812, 0.009113911539316177, 1.2725387811660767, 1.6857942342758179, -1.7116891145706177, 1.266914963722229, 3.7303531169891357, 2.19209623336792, 1.005886435508728, 9.414274215698242, 1.774458885192871, 5.540693283081055, -3.0569026470184326, 8.26517391204834, 1.032282829284668, 1.3371649980545044, -6.326581954956055, -1.3089152574539185, -1.779933214187622, -5.766397476196289, 2.5515592098236084, 0.6144720315933228, 1.152327060699463, 5.537269592285156, -2.3176774978637695, -2.129868268966675, 1.112745761871338, 8.86229419708252, 3.890683174133301, -3.079733371734619, -5.6031107902526855, -1.8163105249404907, 2.700690984725952, -4.268924236297607, 0.8654108047485352, -3.825305938720703, -1.1479277610778809, 1.7845945358276367, 0.281453937292099, -5.93057918548584, 0.8314539194107056, -4.929861545562744, 1.9428240060806274, 2.4879024028778076, -8.774589538574219, -5.614562511444092, 2.2984514236450195, -1.283864140510559, 2.091639280319214, 3.720845937728882, -0.16471566259860992, 0.21962574124336243, 1.9073758125305176, -0.3365601599216461, -5.43919563293457, -2.2586519718170166, -1.7444064617156982, -5.389688968658447, -1.6613470315933228, -1.4516258239746094, 1.3160558938980103, 1.7269091606140137, -2.4654617309570312, -3.300877094268799, -2.771150827407837, 3.9707701206207275, -8.335911750793457, 3.1499176025390625, -2.061415672302246, 1.1277284622192383, -4.056098937988281, 2.9359116554260254, -1.293628215789795, 0.9617539644241333, -0.3139302730560303, -2.6583240032196045, 6.094803810119629, 1.706507682800293, 1.399374008178711, 4.844048976898193, -10.595832824707031, -4.253883361816406, 0.6467029452323914, -0.3735904097557068, 4.934147834777832, 1.9204834699630737, -2.8827311992645264, 2.5672852993011475, -1.7419592142105103, -0.9613611698150635, 2.9401323795318604, 4.758574485778809, -1.1999313831329346, 3.458559513092041, -2.9629087448120117, 4.275797367095947, 1.935172200202942, 0.6078346371650696, 4.529831886291504, 3.3653664588928223, 2.5327470302581787, -1.1041386127471924, -1.2199488878250122, -0.3045956790447235, 4.8306708335876465, -3.3241891860961914, -1.227177619934082, -0.4330247938632965, 4.017642021179199, -1.0521042346954346, -3.895609140396118, 1.1538777351379395, -2.5200259685516357, 0.9500861763954163, 2.253024101257324, 5.902758598327637, -2.6001648902893066, -0.6123403906822205, -1.189205527305603, 3.8374364376068115, -0.24013878405094147, -0.389570027589798, 1.385870337486267, 8.914911270141602, 0.28375452756881714, -4.323137283325195, -0.4365943372249603, -3.592982530593872, -3.303884744644165, 0.5287347435951233, 1.6788846254348755, -0.7118668556213379, 0.08529666811227798, -0.9014124870300293, -2.854882001876831, 2.7919976711273193, 4.500263214111328, -0.9093681573867798, -1.3933311700820923, 4.872430801391602, 6.391364097595215, -0.9317191243171692, -1.1727232933044434, 1.1231520175933838, -1.9841341972351074, 1.7466553449630737, -6.855525970458984, 5.059107780456543, 3.2495012283325195, 4.606661796569824, 3.4502205848693848, 1.928635835647583, -1.59229576587677, 2.4146556854248047, 2.8412110805511475, -0.8151060342788696, -1.4044537544250488, 4.064333438873291, -2.9116621017456055, -7.6790690422058105, 0.6407334208488464, 1.9257426261901855, -0.1636093556880951, 4.691427707672119, 1.3439027070999146, 5.743106365203857, 0.3377952575683594, 3.7015323638916016, -3.6529781818389893, -1.1629066467285156, 6.510761737823486, 6.904986381530762, -4.122534275054932, -1.7494397163391113, -0.6011415719985962, -7.264029502868652, -0.9066028594970703, -2.2972052097320557, -3.707048177719116, -1.9995616674423218, 1.8250797986984253, 6.1925740242004395, -0.8473991751670837, -1.6556235551834106, -1.5444685220718384, 3.624858856201172, -2.21290922164917, 2.127345323562622, -0.9962575435638428, 2.5010664463043213, 0.3033401668071747, -5.230073928833008, -2.211418628692627, -6.144702434539795, -0.5650395750999451, 2.4650890827178955, -0.24482393264770508, -3.465130090713501, -0.8363103270530701, 2.2154006958007812, 2.586711883544922, 3.2482309341430664, 1.107287883758545, 1.4209115505218506, -1.710590124130249, 8.287549018859863, 6.766604900360107, 3.762909412384033, 1.302014708518982, 1.0879100561141968, -8.106000900268555, -2.1218643188476562, 0.38029593229293823, 1.9891185760498047, 3.251971483230591, -3.6285619735717773, 4.019166946411133, 4.370335578918457, 2.9312474727630615, 0.8667526245117188, 5.323152542114258, -0.22439227998256683, -2.8365190029144287, 0.10531792044639587, 0.9596018195152283, 6.262582778930664, -0.8040657043457031, -0.8940173983573914, 1.279365062713623, 3.860314130783081, -4.287530422210693, 2.641295909881592, -2.9533884525299072, 1.924698829650879, -1.9258129596710205, -2.7431161403656006, 3.32511830329895, -3.470857858657837, 2.167698860168457, 2.0837082862854004, 2.4847450256347656, -0.46833354234695435, -2.037736654281616, 2.3552017211914062, 8.62398910522461, 2.212663412094116, 8.252609252929688, 4.598397731781006, 4.64463472366333, 4.725462436676025, 5.977341651916504, 0.06669401377439499, -5.46036434173584, 0.6213329434394836, 4.561866283416748, -0.30143001675605774, 1.170884370803833, 7.212276458740234, 4.350790023803711, 0.30949512124061584, -2.1557869911193848, -0.8440299034118652, -1.4199998378753662, -2.6255578994750977, 1.2941985130310059, 0.7330605983734131, -1.7864365577697754, -2.754276752471924, 1.0842745304107666, -3.9880332946777344, -2.8653883934020996, -4.525102615356445, 2.663783073425293, 2.7592883110046387, -4.464242935180664, -8.117887496948242, 1.2718623876571655, -8.861675262451172, -6.165615081787109, -6.293528079986572, 5.440451145172119, 3.8294363021850586, -3.465745687484741, -1.036623239517212, 2.2981159687042236, -4.823515892028809, -1.683959722518921, 6.914425373077393, 1.312172532081604, -1.8278073072433472, 9.94943904876709, 3.723299980163574, -2.378207206726074, -3.843852996826172, 5.9114789962768555, -0.49582865834236145, 0.6436482667922974, 4.610016822814941, -2.7390494346618652, 1.7950292825698853, 2.5055510997772217, 1.5918155908584595, -1.1905252933502197, -2.3382558822631836, -3.363502025604248, 3.710904836654663, 7.623574256896973, 0.323481023311615, -3.910414934158325, -0.61351478099823, 3.110724925994873, 3.226439952850342, -6.2753071784973145, 0.45971235632896423, -3.913799524307251, -1.0905758142471313, 0.5342998504638672, 7.2197370529174805, -0.15158075094223022, -2.5546886920928955, 3.8012912273406982, 3.672797679901123, -1.802960753440857, 1.0766277313232422, -2.2826592922210693, -5.605356216430664, -3.5931849479675293, -0.8256904482841492, -4.852710723876953, -1.1164419651031494, -3.560668706893921, -0.11200636625289917, 4.097303867340088, 0.08306903392076492, -5.727970123291016, -3.4592647552490234, -1.1782996654510498, -0.8532646894454956, 2.9257943630218506, -3.0569679737091064, -8.331644058227539, 1.8518542051315308, 2.7890172004699707, 1.10080885887146, -11.003358840942383, 0.012187546119093895, -2.2931809425354004, -0.5268129110336304, 2.771214723587036, 4.819823741912842, -0.542609691619873, 2.536590099334717, 3.1864967346191406, -1.942059874534607, -7.691718101501465, 4.662512302398682, -3.9776268005371094, 2.0797719955444336, 0.7818053364753723, 5.602771759033203, -7.039890766143799, -0.8352661728858948, -2.342390298843384, -1.9618557691574097, 0.7191885709762573, 8.44581127166748, -2.409958839416504, -1.0062721967697144, 4.000070571899414, 0.7990374565124512, 0.7271641492843628, -6.680248260498047, 2.3226122856140137, 4.340424537658691, -5.534641742706299, 2.669430732727051, -7.72767972946167, 2.577824354171753, -2.2482783794403076, -2.071681261062622, -7.4546895027160645, -2.145305871963501, 4.416770935058594, -1.9027736186981201, -2.4263241291046143, -4.935459136962891, 0.2681371569633484, 2.1984994411468506, -3.293741226196289, -9.168089866638184, -0.15784437954425812, 6.953675270080566, 1.7796658277511597, -4.440215587615967, 4.315968036651611, -0.5527515411376953, -3.7399635314941406, -2.030144214630127, -10.958246231079102, -1.2222131490707397, -0.7575206756591797, -8.062371253967285, -4.548503875732422, -4.494212627410889, -2.134276866912842, 1.838768720626831, -3.6018972396850586, -6.600986003875732, 1.8870441913604736, -1.128618836402893, 3.887361526489258, 2.340613603591919, -0.9909878373146057, 3.9943392276763916, -1.5040944814682007, -0.051331013441085815, -3.400156259536743, 3.344343900680542, 1.7946434020996094, 0.17012973129749298, -1.4681141376495361, -1.9381192922592163, -2.720954656600952, -5.551854133605957, 3.6871044635772705, 5.050540924072266, 0.7999061942100525, 4.546779632568359, -1.1996634006500244, -7.802046775817871, -6.199962139129639, 6.684815406799316, -2.88808274269104, -1.2788631916046143, 1.330633521080017, -5.795176982879639, 1.273469090461731, -3.42328143119812, 5.716201305389404, 1.6793190240859985, 3.837857246398926, 1.217488408088684, -8.37520980834961, -0.9944579601287842, -3.102153778076172, 0.7465412616729736, -3.6214356422424316, 0.7877427339553833, -2.9520905017852783, 1.6375819444656372, -7.519476413726807, -1.730417251586914, 2.017963171005249, 1.1988602876663208, -6.233565807342529, 5.647126197814941, 6.480988502502441, 2.584965229034424, -3.3115968704223633, -1.0455734729766846, -1.6020811796188354, -3.3499197959899902, -8.289910316467285, -10.601082801818848, 4.1296916007995605, 0.22773785889148712, -2.417607307434082, 5.316751003265381, -1.7071287631988525, -3.4504828453063965, 3.8889949321746826, -4.226586818695068, 4.995980262756348, -1.5252635478973389, 9.200050354003906, -4.252678394317627, 0.9214689135551453, -6.499943256378174, 1.538255214691162, 4.0941691398620605, -1.5653867721557617, 2.641301393508911, -2.6979875564575195, 2.9084901809692383, 4.5283637046813965, -0.5917062163352966, -9.491081237792969, -1.3597995042800903, 2.442901372909546, 6.157749652862549, 1.9797371625900269, -6.1337480545043945, -4.087068557739258, 0.633750855922699, 7.126199245452881, 7.228173732757568, -2.2077531814575195, 0.7536293268203735, -0.2244524359703064, -3.62933349609375, -1.1333004236221313, -1.728359341621399, 1.9834163188934326, 7.620137691497803, -1.1118162870407104, -0.6253995299339294, 0.3044225871562958, -3.0707178115844727, -1.560514211654663, 0.0595218800008297, -0.19951488077640533, -2.481440544128418, 1.7974432706832886, 6.085599422454834, -8.417128562927246, -3.036212682723999, 5.61572265625, -8.179975509643555, 6.939552307128906, -3.811734199523926, 0.09285470843315125, -7.39219856262207, 3.992344856262207, 1.5724314451217651, 0.5483382940292358, 0.23930911719799042, -7.296383380889893, 4.2152838706970215, 4.556999683380127, 3.900832414627075, 1.773657202720642, 6.1533918380737305, -2.186861753463745, -4.446125030517578, -3.079526901245117, -0.6431897282600403, -1.570149540901184, 1.710292935371399, 6.196355819702148, -1.8162981271743774, -3.4415171146392822, -2.204115390777588, -1.7237449884414673, -0.017088040709495544, 0.9652794003486633, -4.234489917755127, -0.2029072642326355, -0.7061498165130615, 0.10928261280059814, -1.8775476217269897, 1.4572389125823975, 4.549441337585449, 2.4047248363494873, 5.542431831359863, 2.665469169616699, 1.6719292402267456, 1.9810470342636108, -5.963475704193115, 3.657404899597168, -0.2006261646747589, 0.950391948223114, 5.124527931213379, 2.292098045349121, 1.5342670679092407, -4.535551071166992, 0.49490633606910706, 1.2099473476409912, 0.8648000359535217, -0.4889702796936035, -5.182643413543701, -4.695838451385498, 0.02130812592804432, -2.000168800354004, -5.8374342918396, 2.1500675678253174, 0.516911506652832, 2.7266738414764404, 4.367819786071777, -6.086360931396484, 6.208395481109619, 0.8746402263641357, 6.659746170043945, 1.6472729444503784, -2.6305840015411377, 3.9820752143859863, 2.561657428741455, -4.486462116241455, 4.198654651641846, -0.8147858381271362, -5.443297863006592, -7.038202285766602, -2.3759167194366455, 0.662065327167511, 1.4587947130203247, 6.261116027832031, 0.9399833083152771, 2.556704044342041, 3.9947149753570557, -2.7211813926696777, -1.5776152610778809, -4.481451511383057, -3.8593857288360596, -2.8107070922851562, 0.7712437510490417, -5.189876556396484, 3.836259126663208, 0.09892331063747406, -6.562193393707275, 7.097995758056641, -1.592989444732666, -2.5330753326416016, 1.7928082942962646, -0.6541948914527893, 4.077259063720703, -3.2400641441345215, -0.6882137656211853, -5.397664546966553, 6.256500720977783, 3.52372670173645, -6.843411445617676, 1.4109820127487183, -1.7473163604736328, -0.1859995275735855, 3.916987657546997, 1.1078755855560303, -5.094050407409668, 0.8141615986824036, -6.00801420211792, -4.054000377655029, 4.2823405265808105, 0.777381956577301, 2.5741498470306396, -0.5276229977607727, 0.3277819752693176, -1.960246205329895, -1.8536109924316406, 1.859284520149231, 4.565649032592773, 7.034814834594727, -4.408811092376709, -1.9391602277755737, 10.412126541137695, -3.45531964302063, 0.9900420308113098, 3.445333957672119, 5.748619079589844, -2.137448787689209, -3.7922027111053467, -3.926571846008301, -3.252939462661743, -5.988283157348633, -7.780450820922852, 5.357047080993652, 4.842590808868408, -7.832240581512451, -5.391952037811279, -7.653048038482666, -0.5120560526847839, -3.9472904205322266, 2.336111307144165, -5.851122856140137, -2.180983781814575, 4.2297821044921875, 5.184370040893555, -3.5799107551574707, -0.5061399936676025, -2.8128950595855713, -2.631694793701172, -6.9422287940979, -0.26975807547569275, -5.990670680999756, 1.5061228275299072, -5.098076343536377, -1.563025712966919, 11.660787582397461, -1.1798851490020752, 3.5762271881103516, 3.128553628921509, -0.4689716696739197, 2.342399835586548, -1.8427568674087524, -2.380969524383545, 4.956806659698486, -6.326574802398682, -3.5255320072174072, 1.0902749300003052, 1.5550928115844727, 1.2550508975982666, 2.971813440322876, 7.358333587646484, 2.8032755851745605, -1.4765992164611816, 9.800492286682129, -3.584686279296875, 1.6183098554611206, -8.471322059631348, 6.312554836273193, -3.14080810546875, -1.4805183410644531, -8.226760864257812, 1.486033320426941, -0.5320845246315002, -2.527971029281616, -6.143538951873779, -1.4146767854690552, 9.538348197937012, -0.5012102127075195, 0.5678332448005676, -8.253545761108398, -4.166341781616211, 5.038133144378662, -5.798251628875732, -7.211392402648926, 2.9747252464294434, -0.8473024368286133, -1.3374009132385254, 5.437370777130127, 2.018486976623535, 1.3812124729156494, 0.3312629759311676, 3.042447090148926, -7.201252460479736, 8.411480903625488, -1.5041916370391846, -0.7570342421531677, -2.197960376739502, -4.181813716888428, 1.4263547658920288, 2.165654420852661, 0.31989458203315735, 11.498766899108887, 0.543677806854248, -4.035834789276123, 2.818718194961548, -1.8928067684173584, 4.189951419830322, 4.193042278289795, 1.772546648979187, 5.368799686431885, 2.005509853363037, -4.744051456451416, -4.03871488571167, 3.766610860824585, -0.6467959880828857, 1.8321740627288818, 3.8978445529937744, -2.7344515323638916, 8.171613693237305, -6.633342266082764, 1.4179633855819702, -0.20459520816802979, 6.922449111938477, -1.0825347900390625, -4.704334735870361, -2.3253371715545654, 5.776965141296387, 0.03045167587697506, 0.060756027698516846, -2.6399102210998535, 8.979750633239746, -5.498180866241455, -4.4062418937683105, 0.6484166979789734, 2.6881966590881348, -5.9786529541015625, -0.5855249762535095, -6.774179935455322, -3.597860336303711, 6.72602653503418, 0.5644819140434265, 2.4947428703308105, 1.8511546850204468, -3.489725112915039, -6.157440662384033, -1.1080360412597656, -4.046346664428711, -1.8325543403625488, 4.615281581878662, -6.341369152069092, 1.4558045864105225, 0.015875697135925293, -0.795115053653717, -7.179445266723633, 8.731461524963379, 2.179018259048462, 1.2885545492172241, 3.6487925052642822, -1.7249467372894287, 0.062244586646556854, 5.854935646057129, -4.496829986572266, 4.128938674926758, 2.0813629627227783, 0.6031373739242554, -5.235830307006836, -7.407009124755859, 2.2485928535461426, 3.2323901653289795, 1.0176416635513306, -1.0133846998214722, -2.2437150478363037, 5.0206298828125, 2.2159042358398438, 4.353507995605469, 2.296952247619629, 1.0183885097503662, 1.4297566413879395, 2.9911909103393555, 1.1831220388412476, -5.29019832611084, -3.6652469635009766, -10.667704582214355, -1.7580944299697876, -3.4237775802612305, -4.317468166351318, 8.218835830688477, 1.8091093301773071, -2.127871513366699, 8.668241500854492, -0.43571582436561584, -1.1416196823120117, -3.352558135986328, -4.077789306640625, 7.60820198059082, 4.747568607330322, -1.777112364768982, -0.7847754955291748, -3.120537281036377, -3.2063310146331787, 5.8573150634765625, -2.4459598064422607, 5.845763683319092, 0.9911595582962036, 3.2063772678375244, 3.0098109245300293, 0.8858032822608948, 4.8099260330200195, -3.1105129718780518, 2.6490871906280518, -7.699311256408691, -2.736720323562622, 1.2997736930847168, 1.0553247928619385, -0.5325906872749329, -1.033127784729004, 1.0309126377105713, 3.5978057384490967, 3.1053361892700195, 1.2879925966262817, -1.5353589057922363, 2.1132071018218994, 0.12328639626502991, -0.12627607583999634, -8.702878952026367, 1.6385420560836792, -7.552845001220703, -0.3813014030456543, 3.190420389175415, 7.536902904510498, -3.0501508712768555, -0.24702678620815277, -0.6377843022346497, 4.790213584899902, 0.6101669073104858, -1.8669559955596924, 0.980374813079834, -3.6267106533050537, -3.2638165950775146, -6.175434589385986, 0.18977372348308563, -0.35295528173446655, -5.776122570037842, -3.7022979259490967, -2.2285776138305664, -0.6947869062423706, -1.7860329151153564, 5.608710289001465, 3.9639241695404053, 1.1922863721847534, 10.322832107543945, 5.742486000061035, 4.537064075469971, -1.0845874547958374, 5.0195817947387695, -1.8428906202316284, 2.8577237129211426, -1.6604541540145874, 4.61178731918335, -4.787289619445801, 2.8535072803497314, 2.8654885292053223, -0.692145049571991, -0.12107082456350327, 3.702718734741211, -0.24108746647834778, 3.730454444885254, 1.5666046142578125, 3.685664176940918, 7.994147777557373, 3.667238473892212, 11.9415922164917, -2.105456829071045, -2.3763315677642822, -2.225156307220459, -1.6298688650131226, -1.4087517261505127, 1.5728265047073364, -2.5654776096343994, 0.12124358117580414, -0.2068375200033188, 2.9553723335266113, -8.155240058898926, 1.903152585029602, 4.856711387634277, 3.943253993988037, 1.5362375974655151, 7.0104241371154785, 7.114903926849365, 1.4459508657455444, -0.5607162117958069, -0.17183856666088104, -9.916131019592285, -3.2502708435058594, 2.147636890411377, -1.128097653388977, 0.3332504332065582, 5.979317665100098, 4.921454906463623, 1.542946696281433, 4.5461297035217285, -3.668821096420288, 4.08666467666626, -3.150639533996582, -1.8515914678573608, 0.943810760974884, 1.1988531351089478, -5.199594497680664, -5.257648944854736, 7.193859100341797, 0.30823057889938354, 1.3115376234054565, -6.1188883781433105, -6.164555549621582, -8.511260986328125, -1.288837194442749, 4.285510540008545, 3.7073686122894287, -0.9066312313079834, 6.858036994934082, -0.21756024658679962, -0.24836266040802002, -2.113464117050171, 3.238471031188965, -2.352792978286743, 0.7869508266448975, 2.9963109493255615, 3.2544193267822266, -2.851675271987915, -1.5385606288909912, -5.863203048706055, 0.26963722705841064, -6.0395827293396, 0.30165112018585205, -7.956053256988525, 1.4100288152694702, -3.2388265132904053, -3.5005056858062744, -2.757390260696411, -2.662503719329834, -1.8172591924667358, 4.0048933029174805, -2.675241708755493, 5.126858711242676, 7.656396865844727, 0.39393043518066406, -1.419517993927002, -0.21758317947387695, -6.451772212982178, -2.8170015811920166, 5.213846683502197, -0.5464401245117188, -5.08643102645874, 7.0325093269348145, 2.4831900596618652, -5.745737552642822, 4.796218395233154, -0.441121906042099, 0.3194368779659271, 0.008725968189537525, 3.502760410308838, 1.4418197870254517, -4.747080326080322, 1.4211363792419434, -3.618248224258423, 3.1998679637908936, 2.4857633113861084, 1.7125866413116455, -3.00260853767395, 7.807220458984375, -3.435736656188965, -4.797769069671631, 3.962467670440674, -1.712593674659729, 2.7778382301330566, -5.550487995147705, 0.6834749579429626, -0.1336793303489685, 4.306899070739746, -0.6416260004043579, 3.13203501701355, 3.470602512359619, -0.7611169219017029, 1.3048580884933472, -3.2693753242492676, -1.9658962488174438, -2.1368143558502197, -1.7433750629425049, -2.774719476699829, 6.19981575012207, 2.378803253173828, 2.9974894523620605, 0.057008832693099976, 6.560240745544434, 5.086698532104492, -3.744126319885254, 4.769658088684082, 1.00921630859375, 2.5772035121917725, 3.1186463832855225, 5.085007190704346, -3.081902265548706, 7.432277679443359, 3.2576022148132324, 3.383633613586426, -0.8201664090156555, 1.8965725898742676, -3.864290952682495, 2.6485536098480225, -9.672279357910156, -0.12514910101890564, -0.6564589142799377, 0.07845457643270493, -4.90526819229126, 1.4966719150543213, 0.8973211646080017, -1.6178970336914062, 2.8975729942321777, 2.921478033065796, -1.9925260543823242, -0.9945291876792908, 0.5987224578857422, 1.338379144668579, 5.181481838226318, -1.7625566720962524, 2.251120090484619, 4.731451988220215, 0.13779930770397186, -3.1993043422698975, 7.25069522857666, 0.07426527142524719, -2.7115867137908936, -3.8249452114105225, -2.6353869438171387, -4.734691143035889, 1.7218449115753174, -2.2196109294891357, -4.181297779083252, -2.1306777000427246, -0.10261576622724533, -1.3703839778900146, -4.800366401672363, -4.06580114364624, -1.8612323999404907, -3.298762083053589, -0.3266102969646454, 7.182770252227783, -0.04763702675700188, -2.8295671939849854, -5.313586235046387, -3.2618279457092285, -3.7567496299743652, -4.977385520935059, 1.2919590473175049, -2.248849630355835, -3.8952341079711914, -1.370333194732666, 6.275166034698486, 0.325342059135437, -1.305560827255249, -4.990128993988037, -2.764920711517334, 1.0637660026550293, 2.7585902214050293, 4.362942695617676, 5.5499372482299805, 4.860230445861816, -4.600265979766846, -1.7770687341690063, -2.487531900405884, -2.614957094192505, 4.124114036560059, -3.045076847076416, 3.1765172481536865, -1.2529031038284302, -4.006652355194092, -0.21717000007629395, 6.832910060882568, -11.361238479614258, 1.6224802732467651, 5.8008270263671875, -4.414536952972412, 12.885869026184082, 4.510411739349365, -0.7866367101669312, 1.5669540166854858, -0.49287867546081543, -2.1454293727874756, 3.4652302265167236, -9.787288665771484, -1.5329304933547974, 1.8325546979904175, -3.8563523292541504, -1.109473705291748, 2.2279603481292725, 1.9879783391952515, 5.785211563110352, -1.3174470663070679, -0.6051579713821411, 5.571281433105469, -4.435700416564941, -2.274508237838745, -8.458061218261719, -1.866883397102356, 5.607090473175049, 2.451232671737671, 0.8260130286216736, -1.522234320640564, 6.977531909942627, -0.09041648358106613, 2.5139167308807373, -2.5768990516662598, 2.1390647888183594, -0.11329260468482971, -1.949342131614685, 4.126571178436279, -8.99917984008789, 5.15548038482666, -0.11787013709545135, 5.287349224090576, 4.395702838897705, 4.510285377502441, 0.9753586649894714, 0.21419000625610352, 8.415878295898438, -2.7109525203704834, -5.935394763946533, -5.145033359527588, 6.171258449554443, -3.069344997406006, -11.774457931518555, -1.6807231903076172, -5.506864070892334, 3.16186785697937, -1.7130714654922485, 0.9589808583259583, 1.3150466680526733, 5.27842903137207, 3.743525981903076, -0.4054855406284332, -5.413845539093018, 1.7967764139175415, -7.662821292877197, -1.171987533569336, -4.250369071960449, 1.23900306224823, -2.6697499752044678, 4.240245819091797, 6.62734842300415, 2.620770215988159, -5.495089530944824, 7.622615814208984, 0.06647968292236328, 1.803540587425232, -0.012302023358643055, 3.010652780532837, 1.430580735206604, 1.7341371774673462, 0.9280514717102051, 2.3639655113220215, -2.814643621444702, -7.23334264755249, 4.969460964202881, -0.24955596029758453, -2.968322992324829, -2.368359327316284, 1.9253408908843994, 0.29709866642951965, -1.547020673751831, -2.5465574264526367, 1.4447321891784668, -1.187016487121582, -1.169740915298462, -4.835819721221924, 2.656989574432373, 5.641694068908691, -0.656141459941864, -3.174330949783325, -2.7004964351654053, 1.4941225051879883, 3.3138580322265625, -6.420677185058594, -2.7626662254333496, -0.40320709347724915, -4.244507312774658, -2.0085747241973877, -0.7575443387031555, 1.6578340530395508, 0.25043612718582153, -3.852173089981079, -1.0623263120651245, -0.028174834325909615, -4.567466735839844, 3.2212679386138916, -5.450566291809082, 3.0101418495178223, 1.8958790302276611, -2.3505375385284424, -1.9126126766204834, -1.7786197662353516, 0.4616662859916687, 4.604577541351318, 2.1715524196624756, 0.027947261929512024, -2.4372434616088867, 0.1555853635072708, -7.509633541107178, 2.0712058544158936, 1.5739927291870117, -2.362724542617798, 3.807929515838623, -6.233548641204834, 5.778908729553223, -1.6346527338027954, -4.7140021324157715, 3.916773796081543, -1.0103709697723389, -2.7498364448547363, -8.700460433959961, -6.799405574798584, 2.924210548400879, 2.608944892883301, 2.380645990371704, -2.8662986755371094, -0.2507290542125702, -1.9397128820419312, -6.854815483093262, -4.223145961761475, 0.8531346917152405, 2.2493560314178467, 4.060036659240723, -4.013060569763184, 0.8984450101852417, 4.317497730255127, -4.385526180267334, -1.1512092351913452, 4.476166248321533, -5.756980895996094, 4.861860752105713, -5.34890604019165, -1.7236602306365967, -1.292699933052063, 1.6976979970932007, 1.9205158948898315, -2.1991002559661865, -0.3169451057910919, -1.6725436449050903, 3.33583402633667, -6.201505184173584, 4.658911228179932, 3.168753147125244, -1.3393265008926392, -7.110053062438965, -9.21831226348877, 6.148360729217529, -0.8781632781028748, -3.531743049621582, -3.636981248855591, -2.4975686073303223, 1.4238044023513794, 2.4954640865325928, -4.510833740234375, 0.4823070466518402, -0.8070651888847351, 2.9099512100219727, 1.6319923400878906, 3.4201090335845947, 5.608550548553467, -8.200857162475586, -1.0171332359313965, -3.230908155441284, 5.704087734222412, -0.9769913554191589, -1.0290865898132324, -2.3148770332336426, -5.450995922088623, 3.182459592819214, 2.9605467319488525, 1.4050666093826294, 8.572734832763672, 1.793927550315857, 1.9838632345199585, -1.7301265001296997, -4.230650424957275, -2.190201759338379, 0.25855496525764465, 3.7832283973693848, -0.2114666849374771, -3.6169872283935547, 4.727106094360352, -3.353581190109253, 2.115651845932007, -3.530733823776245, -1.1172012090682983, 9.40978717803955, -2.8294570446014404, -3.5398895740509033, 8.332260131835938, -4.1501946449279785, -5.204939842224121, 4.512901782989502, 0.31696856021881104, 3.301752805709839, -0.18402199447155, 3.8453497886657715, -4.095466136932373, 6.388790130615234, 9.00070858001709, 3.1039042472839355, 3.821518898010254, -7.212145805358887, 5.046097278594971, -5.649393081665039, 2.257956027984619, -10.626338958740234, 0.5309605002403259, -4.705683708190918, -4.953606605529785, -1.0324302911758423, 3.238971710205078, -4.233463764190674, -1.1261061429977417, 3.2851688861846924, -2.8498315811157227, -1.3626089096069336, 3.0606493949890137, -1.6666287183761597, -8.256908416748047, 5.15770959854126, 3.437288284301758, 1.1991870403289795, 0.37783384323120117, -4.035772800445557, 2.1703813076019287, 4.434819221496582, 0.1368100643157959, -3.3689637184143066, -8.216279983520508, 4.013765335083008, 0.0029210506472736597, -5.590494155883789, -1.0539219379425049, -2.633633852005005, -1.760864019393921, 0.3481804132461548, 0.5552896857261658, -4.676180362701416, -0.18129263818264008, 8.176682472229004, 0.22569353878498077, 3.8273396492004395, -4.206307411193848, -0.16307027637958527, -1.324890375137329, -13.482576370239258, -3.5008363723754883, 2.6045901775360107, -1.8969926834106445, -4.526203155517578, 3.774345636367798, 0.0222113449126482, 2.402573347091675, -1.1171855926513672, 3.339233875274658, -2.165675163269043, -2.1874701976776123, 3.4318695068359375, -0.5685643553733826, 2.4152843952178955, -2.5382049083709717, -3.5620012283325195, 3.887723207473755, -2.795508623123169, -6.029917240142822, -9.356011390686035, 8.355192184448242, -5.8369927406311035, -1.3541854619979858, 1.4950436353683472, -5.2086501121521, -0.4718971252441406, 5.029430389404297, 1.6597129106521606, 4.824660301208496, 6.657822608947754, 2.8079309463500977, 1.7468928098678589, -4.773664951324463, -3.954557418823242, 0.7129620909690857, -1.1268612146377563, -3.9301609992980957, 0.6976323127746582, -0.39706918597221375, -4.940140247344971, -2.759812593460083, 3.161485433578491, 0.7352123260498047, -0.289567768573761, -0.18408358097076416, -1.6557447910308838, -3.426161050796509, -0.17635022103786469, 1.9798815250396729, -0.33868148922920227, -3.6234898567199707, -1.9381836652755737, -6.122326374053955, 4.128775596618652, 5.157315254211426, -0.6573875546455383, -1.6099258661270142, -0.8421297073364258, -6.0583176612854, -1.6567959785461426, 2.09517502784729, 5.716009616851807, 0.3838711380958557, 9.286092758178711, 0.9105355739593506, -2.2795369625091553, -6.4962897300720215, -0.0341472364962101, 6.569941997528076, -0.39771610498428345, -4.4248151779174805, -3.3343987464904785, 0.5985457301139832, -2.6481118202209473, 2.597439765930176, 2.0381789207458496, 0.7675281763076782, -0.17217513918876648, -1.1184052228927612, 1.3934266567230225, 6.173397064208984, -3.7979400157928467, -4.818065166473389, 0.11937083303928375, 2.4119722843170166, -4.525296688079834, 1.2226797342300415, 0.34452635049819946, -1.795954704284668, -3.234668731689453, -0.7285977602005005, 6.565043926239014, -0.18225273489952087, -1.8490079641342163, -2.2750232219696045, 4.589044570922852, 4.316098213195801, -2.0042195320129395, 7.282225131988525, 1.9470218420028687, 1.646631121635437, 2.7625732421875, 5.562941551208496, 1.3320289850234985, 1.3723819255828857, -3.5065364837646484, 0.7853918671607971, -7.457867622375488, -0.4781925678253174, -7.563292026519775, -3.1572792530059814, 0.9650384783744812, -2.135258913040161, 0.6887809038162231, -2.3310723304748535, 2.8406732082366943, -3.24359130859375, -5.168214321136475, 5.924657821655273, 4.584421634674072, -2.8293399810791016, -0.34294596314430237, -2.796090602874756, 2.1918013095855713, 3.0859289169311523, 2.923257827758789, 4.670997142791748, -0.4655391573905945, 2.881690740585327, -1.203473687171936, -7.72016716003418, 1.0384751558303833, -0.6360923647880554, -9.613788604736328, 1.8015748262405396, 0.489117294549942, 8.003068923950195, -1.5158549547195435, 2.5410995483398438, -1.5077886581420898, 8.005585670471191, -4.855740547180176, -4.355098724365234, 6.389568328857422, -8.341063499450684, 1.295414686203003, 1.1840627193450928, -10.098371505737305, -2.48453950881958, 2.0386440753936768, -3.270986795425415, -3.474903106689453, 3.2030837535858154, 5.308053016662598, -0.607987105846405, -7.532471656799316, -4.64726448059082, -3.8469138145446777, 1.2674237489700317, 6.746670722961426, -3.9301445484161377, 4.001992225646973, 7.23256778717041, -1.7550244331359863, -9.231703758239746, 4.601024150848389, 0.16272865235805511, 7.565744876861572, 0.792567789554596, 0.7205065488815308, 4.735269546508789, 6.757297992706299, 1.5273990631103516, 0.41197896003723145, 2.638761520385742, -1.4981921911239624, -8.75189208984375, 2.162080764770508, -2.5274415016174316, -3.2157630920410156, 2.5094356536865234, -0.6502118110656738, -1.2821375131607056, 1.9947712421417236, -0.07988056540489197, -5.065898895263672, 2.6180837154388428, 2.6642322540283203, -3.376739263534546, 5.289163112640381, -2.7968392372131348, -1.707627534866333, 6.332343101501465, -3.8458688259124756, -4.182923793792725, 0.13227535784244537, 3.0611889362335205, -5.085851192474365, -0.1956498622894287, -0.8600087761878967, 1.9693528413772583, -0.34220412373542786, 1.7169482707977295, -1.9341864585876465, 1.1340911388397217, 1.4211708307266235, 0.8135889172554016, 0.5184184908866882, -0.5870597958564758, 1.6019598245620728, -0.18795612454414368, -7.176060676574707, -4.906655311584473, 4.199221611022949, -4.075797080993652, 1.8820582628250122, -2.1690874099731445, -2.7629177570343018, 3.078392267227173, 4.006352424621582, 6.719066143035889, -2.9783873558044434, -9.392549514770508, 0.33902454376220703, 1.8141745328903198, 1.7680003643035889, 6.774322986602783, 3.2934775352478027, 5.379239082336426, 1.1533745527267456, 4.715883255004883, 3.39300799369812, -0.010700782760977745, 0.3584429621696472, -3.590219259262085, 0.05278299003839493, 4.445611953735352, 1.5337646007537842, 2.8017375469207764, -3.733654737472534, -2.9157321453094482, -4.133639812469482, 0.693938136100769, -1.4384124279022217, 0.15396027266979218, -1.4741647243499756, -1.178108811378479, 2.5430784225463867, -2.4067015647888184, -3.275657892227173, -10.091803550720215, 4.4380903244018555, 1.505307912826538, -0.6328661441802979, -1.325536847114563, 3.1660804748535156, -2.121676445007324, 1.2508774995803833, 0.7323095202445984, 3.6535892486572266, -4.297094821929932, -2.995781660079956, -0.07109308987855911, 2.937922477722168, -1.3897207975387573, 1.6182323694229126, -0.09157620370388031, -4.449053764343262, 1.6144956350326538, 2.363574743270874, 5.5789713859558105, 7.939913749694824, -3.9771997928619385, 6.0044403076171875, -6.125194072723389, 0.8598014712333679, -4.190886974334717, -5.407449722290039, 3.368126630783081, 4.513411045074463, -3.233160972595215, -2.4876692295074463, 0.7741085886955261, -5.222699165344238, 1.4206318855285645, -2.1377227306365967, -3.414412260055542, 1.9848579168319702, -3.2339529991149902, -4.107713222503662, -1.8208104372024536, 2.512787342071533, -2.3248422145843506, 0.9037238955497742, 5.883849143981934, 9.024957656860352, -1.8209383487701416, 2.434288501739502, -1.2142901420593262, 1.1880007982254028, 2.3459362983703613, 2.8003013134002686, 0.7855377793312073, 2.079977035522461, 2.572883367538452, -0.8172733783721924, -9.014735221862793, -7.822520732879639, -0.6717963814735413, -0.9297670125961304, 3.6144611835479736, 0.1214757189154625, -0.46298688650131226, 6.205636024475098, -0.9363787174224854, -4.81624174118042, 6.780958652496338, 2.1031768321990967, -0.4979707598686218, -9.464953422546387, -1.4335029125213623, -2.2659811973571777, -6.2751784324646, -2.4425277709960938, -3.664602279663086, -2.6588406562805176, -1.8563565015792847, -6.859441757202148, -4.141423225402832, -3.957043409347534, -2.6980602741241455, -3.153042793273926, -0.20275162160396576, -0.8484347462654114, -3.683155059814453, -1.7690916061401367, -1.7250159978866577, -1.964976191520691, -1.6555544137954712, 0.7935701608657837, 2.6993799209594727, -0.8022733330726624, -3.3564393520355225, -1.5218698978424072, 2.790797233581543, -1.4599992036819458, -4.206814289093018, 0.7935791611671448, -3.1064443588256836, -3.3544647693634033, 0.15766167640686035, -0.060755886137485504, -2.255368232727051, 3.622659683227539, 1.9723150730133057, 5.350442409515381, 0.7242594361305237, -1.6742104291915894, 4.210738182067871, 8.998152732849121, 1.3497031927108765, -1.1028003692626953, 1.1811039447784424, -1.2485395669937134, -0.7435665726661682, -2.394866466522217, 7.863254070281982, -0.39518269896507263, -1.2203525304794312, -3.730602741241455, 0.7951923608779907, 1.523218035697937, -5.638206958770752, 1.2698780298233032, 2.2000222206115723, -0.12105674296617508, 3.985814332962036, 5.6080451011657715, 0.4893067479133606, -2.951887369155884, 1.0870611667633057, -1.5325508117675781, -6.4136481285095215, -3.113694667816162, -1.4643018245697021, -2.89815616607666, -0.716530978679657, 1.262872576713562, 1.2414034605026245, -2.4809930324554443, 5.571200847625732, -4.401882171630859, 0.9340973496437073, -0.14702560007572174, 5.403519630432129, 3.862017869949341, -0.4385763704776764, 3.0882554054260254, -5.660780906677246, 1.0182380676269531, -1.3091344833374023, -5.55704927444458, 3.9362688064575195, -3.3405771255493164, -1.0669382810592651, -1.0405454635620117, -2.182539224624634, -3.2083661556243896, 0.4795941710472107, -1.1839948892593384, 3.934865713119507, -7.04354190826416, -3.1355910301208496, -1.6038353443145752, -0.5497668385505676, -4.025417804718018, -6.144962310791016, 5.379755973815918, 0.3463164269924164, -4.373260974884033, -0.7094570398330688, -4.639565467834473, -0.23603777587413788, 2.471508026123047, 2.0338973999023438, -4.804325103759766, -4.030459403991699, 6.337162494659424, 5.644020080566406, 2.659184694290161, 1.4109323024749756, 4.948895454406738, 1.8412219285964966, 0.29792746901512146, 3.307921886444092, -1.9749903678894043, 2.9735898971557617, 1.1871687173843384, -3.3978054523468018, 4.618474006652832, -7.537146091461182, -5.956830978393555, -4.568424224853516, 4.395219326019287, -1.1808570623397827, 4.414315223693848, -1.7713041305541992, 4.233335018157959, -4.959051132202148, -3.890611171722412, -0.0045169079676270485, 3.2695860862731934, -6.474069118499756, 2.0928971767425537, 1.4021892547607422, 1.3257601261138916, 0.32919639348983765, 2.796611785888672, 4.319540977478027, 0.6638617515563965, 5.128892421722412, -3.6423890590667725, -1.1219735145568848, 2.0086896419525146, 3.5667943954467773, 1.6078039407730103, 1.0497299432754517, 3.964905261993408, -0.09306623041629791, 0.4510214328765869, 2.263045072555542, -3.0498886108398438, 9.522932052612305, -2.8340916633605957, -0.9620347023010254, -1.505104899406433, 2.9518685340881348, 4.134641170501709, -1.21537184715271, 0.6239244937896729, 2.9540374279022217, -0.5003398060798645, 1.571002721786499, 3.9649176597595215, -1.1132405996322632, 1.2082254886627197, -2.0551764965057373, -0.3552820384502411, -3.694824457168579, 10.183741569519043, -1.5071920156478882, 3.557912588119507, 1.4855107069015503, 3.7928109169006348, 3.2359609603881836, -2.0484707355499268, 9.446697235107422, 3.1402151584625244, -3.388368844985962, -9.459004402160645, -2.539323091506958, 3.7449679374694824, 4.5916852951049805, -3.850971221923828, -0.15766161680221558, -3.47861647605896, 6.428493976593018, 3.7322115898132324, -6.204331874847412, -2.360727548599243, 0.11006661504507065, 0.09261847287416458, 7.333717346191406, 4.310199737548828, 1.025827169418335, -2.5796151161193848, 4.185834884643555, 2.2431862354278564, -0.0010730503126978874, 4.667330265045166, -0.5433646440505981, -7.574330806732178, -10.32003116607666, 1.875549077987671, -2.6710152626037598, 1.8120412826538086, 8.396349906921387, 1.297341227531433, 4.115220069885254, -1.754502296447754, -1.4519455432891846, -3.9762446880340576, 2.1799137592315674, 5.3550519943237305, -1.1461126804351807, 3.2239272594451904, 4.023193359375, -3.368644952774048, 5.625529766082764, -5.2443060874938965, 3.303830862045288, -2.9769935607910156, -4.271608352661133, -5.185838222503662, 2.480517625808716, -1.1560536623001099, 1.0079450607299805, 5.276244163513184, 3.5248100757598877, 5.943852424621582, 2.941335678100586, -0.08431090414524078, -3.2828786373138428, 0.9038587808609009, -0.3194063603878021, 2.0749621391296387, 1.2458956241607666, 0.5353503227233887, 5.578553676605225, -0.1902482658624649, 0.9383847117424011, -0.761437714099884, 3.854114055633545, 0.9465184211730957, -7.3440165519714355, -0.37387943267822266, 4.545851230621338, -7.583518981933594, 1.3909543752670288, 3.1129746437072754, 6.920587062835693, -7.166199207305908, -1.0214651823043823, -0.1348545402288437, 6.259291648864746, -0.8948636651039124, -3.6472251415252686, 2.5240933895111084, -1.3494614362716675, -0.5021639466285706, 6.920490741729736, 1.9784327745437622, 6.50987434387207, -1.7729690074920654, 5.043325424194336, 8.33073902130127, 10.713594436645508, -1.2808382511138916, 1.1887102127075195, -0.11904090642929077, -0.12300548702478409, -1.8225959539413452, -3.301276445388794, -1.668363332748413, 1.0516568422317505, 0.9861546158790588, -1.4275022745132446, -0.44316235184669495, 0.0921330451965332, 2.810411214828491, 1.9382848739624023, -2.6746432781219482, -4.207584857940674, 2.889585018157959, -2.394216537475586, 1.2963296175003052, 2.713620185852051, -2.0518712997436523, 0.8223062753677368, -5.549930572509766, -3.573601245880127, 0.4421485960483551, -5.293281555175781, 3.4692485332489014, -1.5612220764160156, -1.800047516822815, 2.8182365894317627, 0.33897459506988525, -4.839532375335693, -4.015065670013428, 0.43785518407821655, 2.583068609237671, 0.6569474339485168, 0.9016795754432678, -2.831087112426758, -3.308192253112793, 2.510288953781128, -1.6168994903564453, -7.518248081207275, -2.1033523082733154, -2.396775245666504, -5.936236381530762, 0.3865641951560974, -3.173588752746582, -1.5825750827789307, 0.02920461632311344, 7.272594928741455, 6.583729267120361, -1.518053412437439, -1.6300361156463623, 1.1462204456329346, -2.3499677181243896, 0.866611897945404, -2.176166534423828, 0.39632970094680786, 0.3461068272590637, -2.944366693496704, 1.3171601295471191, -4.0000762939453125, -3.132901668548584, -1.2483572959899902, -0.7522832155227661, -4.056065559387207, -1.5167295932769775, -0.584333598613739, -2.087510108947754, -3.252103328704834, 3.945240020751953, -0.9609133005142212, 3.29402756690979, 1.004528284072876, 6.700906276702881, 0.4600788652896881, 0.1468476504087448, 5.679025650024414, -4.665988922119141, 3.8873746395111084, 0.36639320850372314, 1.2650278806686401, -1.3908168077468872, 0.4300333559513092, 0.5455163717269897, 9.037322044372559, -5.621322154998779, 3.8066186904907227, 2.8680837154388428, 1.0705641508102417, -3.8114936351776123, -1.8510138988494873, -4.767049312591553, -2.1053545475006104, -1.7080684900283813, 6.794212818145752, -4.218125343322754, -2.5240225791931152, -2.6825733184814453, -1.2063654661178589, 0.5219414234161377, -0.11789066344499588, 3.479828119277954, 0.6137757897377014, -0.37867507338523865, 2.4574975967407227, 6.824585914611816, 7.304611682891846, -0.38024255633354187, 1.9611520767211914, -6.291898250579834, 3.8820924758911133, 10.42539119720459, -4.0153021812438965, -5.508450031280518, -0.6852695941925049, -4.561131000518799, 2.2570884227752686, 6.930522918701172, -2.60396409034729, 3.0200369358062744, 2.6399502754211426, 3.336822271347046, 6.965696811676025, 1.795613408088684, 0.4065043032169342, 2.5982894897460938, -3.1146864891052246, -2.0059256553649902, 3.121314525604248, -0.956924557685852, -0.7579851150512695, -2.2575042247772217, 1.3874582052230835, -5.637706279754639, 3.897522449493408, 3.271469831466675, -9.798735618591309, 5.145986080169678, 3.0441112518310547, -4.7207722663879395, -3.846858501434326, -3.5982792377471924, -4.065884590148926, 1.1839916706085205, -1.370081901550293, -4.565459251403809, 4.4553117752075195, 1.5340102910995483, 3.939924478530884, -1.1091638803482056, 9.850199699401855, 1.7203786373138428, 0.9784095287322998, -0.38621288537979126, 0.07052335888147354, 0.33046722412109375, -3.7176966667175293, 1.306574821472168, 1.9041650295257568, -1.8985350131988525, -2.16711163520813, 5.946319580078125, 2.531369924545288, -0.4848853051662445, -2.666020393371582, 0.08360622078180313, -5.677623748779297, -1.9825177192687988, -2.128864288330078, -1.2896274328231812, 0.2237219661474228, 3.3863937854766846, 3.5261197090148926, -0.200038343667984, 1.3358149528503418, -0.9544702172279358, -2.7756409645080566, -1.7050728797912598, 1.1080372333526611, 2.4401512145996094, 5.544007301330566, 8.217288970947266, -0.7024447917938232, -8.348831176757812, 1.9518051147460938, -0.9250679612159729, 2.7158915996551514, -5.407177925109863, 3.737872838973999, 3.3915350437164307, -6.143754005432129, 1.49422287940979, -3.5424375534057617, -7.133734703063965, -1.6423622369766235, -6.990301132202148, 7.628839492797852, -0.9405427575111389, 9.35854721069336, 3.5476863384246826, 1.4655611515045166, -3.4583487510681152, -3.699110984802246, 7.0310282707214355, -2.5964128971099854, -4.038856506347656, 4.438056468963623, 2.421858072280884, 3.01916766166687, 2.7754456996917725, -0.36213764548301697, 5.891875267028809, -1.5232690572738647, -3.1584255695343018, -4.2354044914245605, -2.230639934539795, -3.83103084564209, 6.291816234588623, -4.955231666564941, 6.056031227111816, -9.93351936340332, 8.692117691040039, -3.6203815937042236, 5.583033561706543, 3.192812442779541, 6.487146377563477, -6.046570777893066, 4.36566686630249, 6.57977294921875, 8.945822715759277, 1.5629287958145142, 2.437697172164917, -2.9738056659698486, -0.6608447432518005, -2.451427459716797, -1.8341519832611084, -2.681121349334717, -7.101574420928955, 0.23830857872962952, -5.001346111297607, -5.456809043884277, -1.4165488481521606, -4.717137336730957, -9.226116180419922, 2.868218183517456, 1.3808871507644653, -0.3248063921928406, 2.481977701187134, -6.716047286987305, -1.5501054525375366, 3.1484761238098145, -1.473537802696228, 6.626712322235107, -2.4150030612945557, 0.09541907161474228, 1.3061258792877197, -0.405049592256546, -2.430943727493286, 7.51046895980835, 5.055039882659912, 2.9838571548461914, -1.7158105373382568, -2.2526259422302246, -0.3775852918624878, 3.965641736984253, 6.228713035583496, 3.783501386642456, 2.6195762157440186, -0.804174542427063, -2.5688726902008057, -10.204623222351074, -3.9081056118011475, 1.3718496561050415, -5.460994243621826, -0.7762967348098755, -3.163214921951294, -4.5290141105651855, -1.0652164220809937, 2.897496223449707, 10.002838134765625, 4.4970927238464355, 4.416473388671875, -9.162105560302734, -3.4485979080200195, 2.5127580165863037, 2.7752974033355713, -2.5742974281311035, -4.710953235626221, 2.7673110961914062, 1.4201303720474243, -3.2364494800567627, 4.294226169586182, -1.1920015811920166, -0.8966804146766663, 2.402428388595581, 3.1242265701293945, -1.951648473739624, -5.352541923522949, -2.9037444591522217, 11.94019889831543, 1.6626386642456055, 1.6747044324874878, -1.8058738708496094, -4.711060523986816, -4.120615005493164, -0.10257793217897415, -4.905309677124023, 3.8796703815460205, -1.6160962581634521, 5.166106700897217, 3.9467873573303223, 2.3080050945281982, 4.761110782623291, 5.769235610961914, -2.564058542251587, -1.5118765830993652, 0.3675464987754822, -3.3994040489196777, -6.666548728942871, 1.6585230827331543, 5.499479293823242, 10.367974281311035, 1.2781213521957397, 0.3811975121498108, 6.711258888244629, 2.1002578735351562, 3.8514628410339355, -0.7689802646636963, 2.480614185333252, -4.363015174865723, -8.177467346191406, -1.9813441038131714, 3.8216562271118164, -1.6175538301467896, 2.9127376079559326, 3.1022191047668457, -7.114468097686768, -1.5716954469680786, -1.1424357891082764, -1.1918132305145264, 4.599025726318359, 0.3195691704750061, 6.788918972015381, -3.6205663681030273, 1.4520044326782227, -3.694385290145874, -0.4803687036037445, -10.987404823303223, -2.375671863555908, -0.3989101052284241, -9.042880058288574, -1.0234750509262085, -1.5956541299819946, 1.8795002698898315, 3.779780626296997, -1.4505685567855835, 0.8686944246292114, 1.2201483249664307, 3.4134440422058105, 3.310584306716919, 3.4514846801757812, -0.8395736813545227, -1.195709466934204, -7.626375198364258, -1.1240719556808472, 1.1294007301330566, -10.271592140197754, -1.288613200187683, -0.9567688703536987, 0.9738830327987671, -4.8680620193481445, 6.406441688537598, 1.0474117994308472, 3.605924367904663, -1.1243208646774292, 3.2098820209503174, 1.6776399612426758, 1.0480746030807495, -3.1441948413848877, 5.150700092315674, -4.528326511383057, 5.494778633117676, -0.254077672958374, 5.4149017333984375, 0.8752401471138, 0.050711482763290405, 2.304960250854492, -0.6507035493850708, 0.16339059174060822, -1.413412094116211, 2.2207071781158447, 2.706904649734497, -4.003711700439453, -3.8617494106292725, 5.1135172843933105, 2.5203444957733154, -4.955888271331787, -6.077216148376465, -4.1528544425964355, 5.464572906494141, -3.193105459213257, 2.76615047454834, 0.19510658085346222, 0.9329713582992554, -0.32562023401260376, -2.9876673221588135, -1.355556607246399, 4.283123970031738, 0.15535128116607666, -2.4063057899475098, 1.5639547109603882, 6.434854030609131, -2.6150436401367188, -3.569876194000244, 0.967221736907959, 3.8787221908569336, 0.08964066952466965, -5.663412094116211, 4.683732032775879, -2.1989595890045166, -6.750450134277344, -6.775708198547363, -3.4166557788848877, -4.164811611175537, 0.7560707926750183, -1.092706322669983, -5.704898357391357, -0.864687442779541, -2.7052247524261475, -2.6914849281311035, 0.3363702595233917, -8.986078262329102, 10.50152587890625, 4.9770684242248535, -4.545112133026123, 0.7090663909912109, -2.2834372520446777, 6.426222324371338, 0.3160603940486908, 0.7138487696647644, -4.137602806091309, -8.42054271697998, 3.5137476921081543, -3.176156759262085, 3.399056911468506, 2.0532047748565674, -1.0143312215805054, -2.0260467529296875, 1.5667555332183838, 3.8976049423217773, 4.546721935272217, 2.4730048179626465, -4.424755573272705, -2.0265414714813232, 2.046560764312744, 6.615581035614014, -0.3832976222038269, 4.458573818206787, 0.9050168991088867, 1.3632996082305908, 2.938602924346924, 4.283722400665283, 1.2787331342697144, 1.5107194185256958, -10.30030632019043, -1.8150495290756226, -5.2397541999816895, 1.2079347372055054, 1.4447652101516724, 0.8207260370254517, -4.681329727172852, -0.2592439353466034, -7.1368088722229, 0.7305979132652283, -3.888538360595703, -1.0380055904388428, 3.323108434677124, 1.6044405698776245, -10.354369163513184, -3.6377081871032715, -1.7044401168823242, -6.330015182495117, 1.4969815015792847, 2.4802331924438477, -0.807174801826477, 1.4899768829345703, 1.7160396575927734, 2.9579696655273438, -2.3775758743286133, -0.04320568963885307, -2.33915114402771, 2.023974895477295, -3.5578272342681885, 4.5425310134887695, -0.6502422094345093, -6.183797359466553, -4.601624488830566, 0.2049066126346588, 2.3601465225219727, 0.7821708917617798, 0.5144307613372803, 0.25348126888275146, -1.8392969369888306, -1.9615614414215088, -8.01047420501709, 0.9698448777198792, -0.67100590467453, 3.1290977001190186, 1.5044760704040527, 3.1286396980285645, 1.8496447801589966, 1.6190851926803589, 3.0148119926452637, -3.9204800128936768, -1.6738276481628418, 2.4347352981567383, 0.38837701082229614, -1.5085819959640503, -9.44607925415039, 5.388556480407715, -4.26462984085083, 9.351826667785645, -1.2169535160064697, 5.754694938659668, 4.455453872680664, -1.6223769187927246, 1.9103584289550781, -2.596376895904541, 9.118380546569824, -0.5081969499588013, -3.248086452484131, -1.6173397302627563, -0.021222051233053207, -4.11368989944458, -0.5966751575469971, 0.847758948802948, 2.1525912284851074, -3.854121685028076, -0.027380015701055527, -5.135239124298096, -3.830636501312256, 1.4054608345031738, -6.232161521911621, 3.9089856147766113, 1.9958678483963013, -2.5771617889404297, 0.43949052691459656, -8.212119102478027, -0.43365347385406494, 3.5387322902679443, -0.9559439420700073, 1.651086688041687, -3.6762402057647705, 2.4265434741973877, -2.495833396911621, 6.212327480316162, -1.816617488861084, -3.1592133045196533, 5.913361549377441, -0.6948938965797424, -5.240149021148682, -2.5579543113708496, -0.41489920020103455, 7.406309127807617, -4.469418048858643, 2.0848987102508545, 2.557302951812744, 0.21549946069717407, -0.8329675793647766, -0.5866039395332336, -5.079296588897705, 0.7276946306228638, -0.4649543762207031, 1.9889246225357056, -0.005258406978100538, -3.4003946781158447, 5.129576683044434, -0.9045013785362244, 0.16231007874011993, -8.82765007019043, -4.726353168487549, 2.1211867332458496, -0.0739956945180893, -2.877089262008667, -0.18225769698619843, -6.140417098999023, 3.5088160037994385, 2.829739570617676, -0.4653290808200836, 0.9073047041893005, 4.649367809295654, -0.33697736263275146, -8.175955772399902, 0.6207624673843384, 0.6805446147918701, -0.2483004331588745, -4.182929992675781, 4.64204216003418, -3.092447519302368, -0.22919081151485443, -1.3901947736740112, -2.168026924133301, 6.278404712677002, -5.3219685554504395, 6.488097190856934, -5.4399285316467285, -2.6587114334106445, 1.3138176202774048, 3.0550577640533447, 3.344543695449829, -2.2077996730804443, -4.260395526885986, 2.3997788429260254, 6.076557636260986, 0.1432010680437088, 0.7733669877052307, 2.3238489627838135, 3.717383623123169, 1.4450252056121826, 4.638178825378418, 2.234893798828125, -2.553008794784546, 2.1205780506134033, -3.7951784133911133, 0.015585812740027905, 3.3787522315979004, 0.22160477936267853, 5.193166732788086, -3.684882164001465, 2.8423502445220947, 4.503628253936768, 2.8079307079315186, -2.4193737506866455, 0.6811224222183228, 5.509571552276611, -7.655791759490967, -2.7036538124084473, 1.1578360795974731, 1.4786072969436646, 4.505475997924805, 4.70123291015625, 1.0929244756698608, -2.363218069076538, 0.796192467212677, -0.7543074488639832, 0.49336734414100647, 2.4478631019592285, -1.0745923519134521, -4.654496192932129, -1.865324854850769, 2.055586576461792, 0.6434301733970642, -3.3562443256378174, 3.176999568939209, -0.12049712240695953, 2.083491086959839, 2.9999868869781494, -0.6499712467193604, 0.25844377279281616, -2.7775473594665527, -1.9234659671783447, -1.4916800260543823, -1.1123862266540527, 7.889430046081543, -5.431883811950684, 1.5195086002349854, 4.181416034698486, 3.240497350692749, 0.23894867300987244, 0.4788767099380493, -1.4478950500488281, -0.7276879549026489, 0.6952234506607056, 0.21448780596256256, -1.0887775421142578, 3.912961721420288, 8.834977149963379, -0.941645085811615, 3.640531301498413, 0.6610009670257568, -0.596716582775116, -5.7769999504089355, -4.939451694488525, -2.4297969341278076, -1.034175992012024, -4.66432523727417, 1.285229206085205, -1.7375011444091797, -5.029722213745117, 2.1508920192718506, 1.985307216644287, 0.42718517780303955, 3.555955171585083, -5.5491132736206055, -0.727233350276947, 1.6641587018966675, 0.49924346804618835, -4.758100509643555, 2.184450149536133, 3.711745023727417, 3.310591697692871, 0.38203316926956177, -0.2749996781349182, -8.439212799072266, 5.715137481689453, -2.3054144382476807, 0.617077648639679, -2.877004384994507, 3.7037010192871094, 8.403504371643066, 4.612336158752441, 3.4370508193969727, -0.9197481870651245, -0.18651212751865387, -0.08181320875883102, 5.760818958282471, -2.2171313762664795, 3.2309460639953613, 0.9318108558654785, -0.7221094965934753, 7.0566205978393555, 4.439804553985596, 2.540175199508667, 4.015211582183838, 3.666830062866211, 2.8605151176452637, 0.521763801574707, -0.6888822913169861, 3.2861409187316895, 11.505017280578613, 1.3396825790405273, -11.601531028747559, 0.3168228268623352, 8.930831909179688, 2.4440841674804688, -5.534646511077881, 4.832211494445801, 0.7255435585975647, 1.873443841934204, -1.1381198167800903, 0.8617398142814636, -4.361931800842285, -8.365955352783203, 2.5713231563568115, -8.85836124420166, -4.6463942527771, -0.049333322793245316, 1.3684122562408447, -0.8989520072937012, 6.463078498840332, -2.3361806869506836, 1.0401279926300049, -6.537981986999512, -2.927605628967285, -1.5041515827178955, 7.048454761505127, -4.215394020080566, 2.0229320526123047, 4.329756259918213, 2.7176620960235596, 6.709245204925537, -0.9943687915802002, 1.7185956239700317, 1.4074918031692505, 2.315812587738037, -0.49415138363838196, 4.266697883605957, 5.179610729217529, 0.16419346630573273, 0.08549825847148895, 1.7480372190475464, 7.976264476776123, -1.4794976711273193, -0.6685064435005188, -4.363768100738525, 1.5812411308288574, -0.6528680324554443, -3.0201804637908936, -0.9429922699928284, -4.278687000274658, 4.546446323394775, 4.719151496887207, 2.00498104095459, 2.1581358909606934, -0.1869506984949112, 1.9928227663040161, -3.4432296752929688, -4.923100471496582, 4.834840297698975, -1.326517105102539, -0.007189098279923201, 3.8608970642089844, 1.6883121728897095, -7.014237880706787, 6.481567859649658, -2.8487942218780518, -7.025157451629639, 3.6671266555786133, -2.1165881156921387, 1.3006454706192017, 0.19306975603103638, 0.12392614781856537, 0.7028178572654724, -1.9851199388504028, 2.007188081741333, -7.978152275085449, 1.6422890424728394, -0.19555696845054626, -0.7913731932640076, 1.067361831665039, 2.251661777496338, 6.0439019203186035, -7.3381123542785645, 2.439040422439575, -10.915905952453613, -0.7834509611129761, 4.4672088623046875, 4.6517462730407715, -5.292917251586914, 5.087206840515137, 0.3389413058757782, -4.429730415344238, 3.6823041439056396, -1.6732420921325684, -5.590489864349365, -0.0960841029882431, 3.5088391304016113, 2.5329983234405518, 1.347537875175476, -1.4037292003631592, 2.2034826278686523, -2.3133294582366943, -2.6178901195526123, -4.796030044555664, 0.13019172847270966, 6.765104293823242, 3.067194938659668, 0.08763448894023895, 0.42537039518356323, -2.243981122970581, -1.784433364868164, -1.9304101467132568, 3.6926591396331787, 0.29165321588516235, 2.5698928833007812, -0.2756253480911255, 3.648374080657959, 5.388577461242676, 0.05413246154785156, 0.9376786947250366, 3.4707276821136475, 1.2548354864120483, 8.425705909729004, -10.424830436706543, 4.984389781951904, 1.1316726207733154, -4.744243621826172, 6.669086456298828, 3.8717660903930664, 4.297066688537598, 6.728643417358398, 2.609813928604126, -1.3806610107421875, -1.8101624250411987, 0.3059157729148865, -3.5411617755889893, 7.585602760314941, 2.717237710952759, 5.818455696105957, 7.561408042907715, 2.172964572906494, 0.6174072623252869, -4.3855881690979, -0.002963667968288064, -1.1904691457748413, 7.309203624725342, 5.418766975402832, 0.42385628819465637, 0.16976740956306458, -2.814976215362549, 10.610551834106445, 2.0280609130859375, 1.7099456787109375, -2.9533402919769287, 4.428030490875244, -0.29143619537353516, 2.9363231658935547, 1.780509352684021, 5.483903408050537, 3.5779151916503906, 2.1506803035736084, -5.890549659729004, 4.853794574737549, -1.3907145261764526, -0.02838023565709591, 3.8659183979034424, -0.4757688343524933, 1.8802255392074585, 2.854553461074829, -5.520732402801514, -5.848305702209473, 2.04240345954895, 0.06702793389558792, -8.736294746398926, -1.3026599884033203, 2.155228614807129, -0.02942139469087124, 4.821324825286865, -4.6168413162231445, -0.6679558157920837, 4.644160747528076, 2.181739330291748, 3.0152571201324463, -6.5570454597473145, -4.19488000869751, 4.591085433959961, -2.2968361377716064, -1.761533498764038, 3.8098318576812744, 1.88456130027771, -3.5775156021118164, -0.12297209352254868, 0.8698573708534241, 1.1229742765426636, -7.4831109046936035, -0.4349350333213806, 2.314049482345581, 5.323886871337891, 5.477612495422363, 2.512878894805908, 2.5291824340820312, -0.7343097925186157, 0.7065138220787048, 3.154982566833496, -6.211540699005127, -3.0888984203338623, 3.599081516265869, -6.255847930908203, 4.4375224113464355, -1.0599905252456665, -1.9928739070892334, 4.844018936157227, 2.6438369750976562, 8.17373275756836, -3.3538074493408203, -5.879334926605225, 0.5530800819396973, -2.1649365425109863, -1.7703176736831665, 3.4271748065948486, -0.881022036075592, 3.257477045059204, -2.6184048652648926, 0.5220563411712646, 0.00014112571079749614, 2.034830331802368, -2.6688339710235596, -3.625554084777832, -2.0343971252441406, 0.35367250442504883, -1.9349526166915894, 0.6656163930892944, 2.9120848178863525, -5.960458755493164, -2.8778884410858154, -1.9692411422729492, -7.2709574699401855, -0.9737854599952698, 0.5223340392112732, 3.6553993225097656, 1.473503828048706, 2.165569543838501, -4.395148754119873, -2.9219577312469482, 1.3461980819702148, 6.713266849517822, 1.500795841217041, -1.3877509832382202, 5.5486884117126465, 8.542767524719238, -3.9095253944396973, -4.035335063934326, 2.425889492034912, -5.60008430480957, 4.05031156539917, -1.0216132402420044, 0.027425436303019524, 0.6269644498825073, -4.1512298583984375, 1.2360483407974243, -1.8826690912246704, -3.6858012676239014, -1.731584072113037, -2.2675247192382812, -1.0424590110778809, -4.444879055023193, 3.5281453132629395, -6.187196254730225, 5.5148138999938965, -0.6728478670120239, -6.474975109100342, -4.375333786010742, -1.292675495147705, -4.646280765533447, 2.991523027420044, -5.306241989135742, -7.128340244293213, -0.905620813369751, -1.8022099733352661, 0.7734607458114624, 8.53091049194336, -7.6902360916137695, -2.009199619293213, -1.0995975732803345, 5.790193557739258, -7.040577411651611, 3.8746657371520996, 7.343690872192383, -1.9339882135391235, -1.0440647602081299, 0.37571951746940613, -1.1152477264404297, -4.182617664337158, 3.296584367752075, -1.973675012588501, -6.077042579650879, 2.5308492183685303, 5.788503646850586, -6.657881259918213, -8.20533561706543, 0.6890580058097839, -1.6780917644500732, -5.982058525085449, 7.549110412597656, -3.5290448665618896, 2.7243263721466064, 9.498916625976562, -0.41059187054634094, 3.238201856613159, 6.796655654907227, -5.846395969390869, 1.365803599357605, 0.830091655254364, -0.1146942675113678, 5.4291582107543945, 0.6743251085281372, -0.19989237189292908, 1.504094123840332, 2.9398789405822754, -1.1239256858825684, 3.293647289276123, -3.8643605709075928, -1.7795159816741943, -3.4631636142730713, 0.3570452332496643, 2.1767044067382812, -1.470695972442627, -0.8713372945785522, -0.6236701011657715, 0.3893686830997467, -5.1915693283081055, 0.9179535508155823, 2.986905336380005, 0.1430204063653946, -6.513429641723633, 1.816418170928955, -3.0611460208892822, -0.8546738028526306, -2.534128189086914, 0.9996607303619385, -9.421537399291992, -1.5207879543304443, -5.617082118988037, 6.759042263031006, -0.5575730204582214, -7.211270332336426, 3.983503818511963, -2.0906500816345215, -4.3869194984436035, 4.44828987121582, 0.1025744378566742, 1.4706089496612549, -5.731000900268555, 2.4058151245117188, 3.577073574066162, 3.544506788253784, -3.9077422618865967, 3.431541681289673, 9.960261344909668, 0.09452283382415771, 4.969851493835449, 6.059791564941406, 2.039349317550659, -0.02957841008901596, 4.5302300453186035, -0.7063801288604736, 3.2269816398620605, -1.8093388080596924, 4.238438606262207, -0.9786654710769653, -0.3351325988769531, -1.0863189697265625, -3.0832717418670654, 1.9989336729049683, 0.5333622097969055, -2.959681987762451, 1.3508127927780151, 4.281345844268799, 1.3192250728607178, -1.6507145166397095, 9.004179954528809, -4.238814830780029, 2.8751614093780518, -2.4865362644195557, 4.414027214050293, -5.28556489944458, 10.660248756408691, -4.157334804534912, 5.1563615798950195, -1.5108095407485962, -0.14705799520015717, -0.6967637538909912, -2.7261996269226074, 3.9991297721862793, 14.691606521606445, -0.549669086933136, -4.060488224029541, -8.439007759094238, 1.5659719705581665, 5.205344200134277, -4.472620487213135, 3.5001046657562256, -0.10486432164907455, -2.348825216293335, -0.38449034094810486, 3.4740569591522217, 3.1003403663635254, 4.820684432983398, -3.3014752864837646, 2.1379480361938477, -6.24431037902832, 1.145204782485962, 2.375903844833374, 1.9428263902664185, -2.4513309001922607, -1.90496027469635, -4.701852321624756, -2.031813144683838, -6.781307220458984, -2.382829427719116, 1.1079645156860352, 9.440983772277832, 0.3205375373363495, 0.5327185988426208, -4.8702616691589355, 4.394068717956543, 3.603644609451294, 1.1589142084121704, -2.800708770751953, -1.7921946048736572, -4.262498378753662, -10.32983112335205, -4.035037994384766, 4.983972549438477, -2.260622262954712, -0.13014620542526245, 1.5183320045471191, -5.0176897048950195, 4.574411392211914, -0.8798185586929321, 4.209293365478516, 4.980020523071289, 4.197794437408447, 7.464142322540283, 0.870404064655304, 6.734678745269775, 0.8745810389518738, 0.758812665939331, -3.7639129161834717, 4.512394905090332, -2.8440046310424805, -4.024503231048584, 6.580920219421387, -2.2240092754364014, -2.1752326488494873, -5.233373641967773, -3.420761823654175, -0.4891953468322754, 5.11487340927124, -5.294027805328369, -0.17144650220870972, 1.7519747018814087, -1.527323842048645, -0.669252336025238, 1.5970011949539185, -2.898104667663574, -2.28702974319458, -1.6047483682632446, -2.606933355331421, 3.5752899646759033, -6.011285781860352, -3.678288698196411, 0.5533438920974731, -1.8190888166427612, 0.16107425093650818, -3.5013699531555176, 2.6156117916107178, -4.744414806365967, 2.214296340942383, -6.877629280090332, -3.085878849029541, -3.2562644481658936, 0.004433425143361092, -2.188045024871826, 1.2641406059265137, 0.08458173274993896, 2.7106995582580566, 0.44606122374534607, 6.828910827636719, -4.106637001037598, 6.289612293243408, 6.43864107131958, -4.096782684326172, -3.3848023414611816, 9.501592636108398, -3.374663829803467, -6.198330402374268, -4.965705394744873, 0.6115278005599976, 0.4048902988433838, -8.325288772583008, 1.2129863500595093, 3.7592217922210693, 4.78240966796875, 2.2137391567230225, -1.142998456954956, 2.5636584758758545, -2.4241955280303955, 2.672987222671509, 11.5224609375, -7.409030914306641, 5.392453193664551, -0.13312222063541412, -3.071913003921509, 6.8186235427856445, -1.384553074836731, -3.7923126220703125, 4.658392429351807, 3.577057123184204, 5.228710651397705, 6.087014675140381, -2.7804465293884277, 0.908528208732605, 1.7698769569396973, -2.5112359523773193, -4.056619167327881, 4.749259948730469, 1.917221188545227, 6.437978267669678, -4.551833629608154, 6.345160961151123, -0.9289525151252747, -0.7172120213508606, -2.2550134658813477, 4.672389030456543, -1.094753623008728, 1.2422364950180054, -10.450758934020996, -6.152748107910156, -1.8917008638381958, 4.158297061920166, 2.0122225284576416, 5.939340114593506, 2.0416858196258545, 0.424641489982605, 2.549940347671509, 0.16885590553283691, -2.7862584590911865, 5.925870895385742, -1.565990924835205, -1.283122181892395, -0.9125782251358032, -4.307267665863037, 5.554440498352051, 6.065512180328369, -5.783591270446777, -5.1832661628723145, -1.1322931051254272, -8.896003723144531, 0.9902046918869019, 1.3385714292526245, -0.9065762162208557, 2.1470518112182617, 2.0669798851013184, -2.9739487171173096, -0.6011699438095093, -2.137606143951416, -1.1200743913650513, -3.860419988632202, -2.4155471324920654, 1.2645812034606934, -7.561835765838623, 2.115283727645874, 2.000448703765869, -0.6498413681983948, 6.934750080108643, -0.05395636707544327, -2.7603323459625244, 4.81540584564209, -4.197035789489746, -1.2760891914367676, -2.8762269020080566, -5.937302589416504, 8.395216941833496, 0.861950695514679, -0.7504265904426575, 0.7982367277145386, 2.3985137939453125, -2.726545572280884, -3.5130836963653564, 0.2626700699329376, 4.806743144989014, 1.5847861766815186, -0.6743803024291992, 5.832186698913574, 1.7166229486465454, 2.832228183746338, -0.30769768357276917, 7.069729328155518, -0.6963639259338379, -3.261667490005493, 0.2899886965751648, 8.891860008239746, -3.729832887649536, 4.480048656463623, 6.849186897277832, 4.6098504066467285, 7.027759075164795, 0.6091688275337219, 7.399050235748291, 0.3625592291355133, -1.3510364294052124, -1.1667256355285645, -3.747551679611206, 5.599231719970703, 2.1622960567474365, 0.12677401304244995, -0.14539410173892975, 1.564200520515442, -1.8490045070648193, 2.689775228500366, 2.361739158630371, 2.4140584468841553, 3.6716184616088867, 6.222778797149658, -1.1926872730255127, -1.2780897617340088, -10.183499336242676, 1.1979504823684692, 1.6879264116287231, -2.8107497692108154, 0.5895785689353943, 5.655277252197266, -2.461273670196533, -3.8616321086883545, 4.9082560539245605, -1.6213773488998413, 7.388005256652832, 3.2730367183685303, 0.4351942837238312, -0.17205645143985748, 2.833165407180786, -3.8882343769073486, -2.9593918323516846, -4.8886284828186035, -1.0898246765136719, -0.050833504647016525, -0.6588510274887085, -0.35692453384399414, 1.833814024925232, -4.379809856414795, 2.581693410873413, -4.0605340003967285, 0.4002682864665985, 2.476210594177246, -1.6398497819900513, -7.725653171539307, -2.5723562240600586, -11.219834327697754, -2.3388967514038086, 6.455606937408447, -3.274773597717285, -3.2721171379089355, 1.165742039680481, -5.9127888679504395, -8.694023132324219, -5.5598931312561035, -5.634285926818848, -1.8829164505004883, 2.1717984676361084, 0.37781578302383423, -0.03654563054442406, -0.23808887600898743, 0.13737773895263672, -0.7676801085472107, 3.886314630508423, 4.948970794677734, 0.12142634391784668, 2.2502427101135254, -5.374338626861572, 3.9437754154205322, 4.417194843292236, 2.329251527786255, -0.1754150390625, -0.7989802956581116, -4.377367973327637, -5.479428291320801, -4.244948387145996, -1.3780735731124878, -6.652400493621826, 7.18339729309082, -0.34754347801208496, 6.933096408843994, 0.024605758488178253, -0.3863297700881958, -1.2038971185684204, -1.451596975326538, -5.860392093658447, -3.40680193901062, 0.6064726710319519, 1.259369134902954, 0.07580133527517319, -4.840686798095703, -8.10198974609375, 5.0467987060546875, -3.207871437072754, 3.394035816192627, -5.318654537200928, -1.0990787744522095, 1.964214563369751, 0.07801225781440735, 4.449901103973389, 1.2998168468475342, 1.9061737060546875, -0.39899101853370667, 0.9796706438064575, -5.106626033782959, -0.2600323557853699, -4.60568904876709, -2.9606010913848877, -1.1474266052246094, 1.2689027786254883, 0.12240581214427948, 1.5652538537979126, -4.633008003234863, 4.411738395690918, -2.9373908042907715, -6.116312503814697, 7.033744812011719, 1.382534146308899, -2.828049421310425, 0.08452168107032776, 0.7672823071479797, 0.4666630029678345, -5.245623588562012, -0.27091386914253235, 2.2700283527374268, 6.071010112762451, -0.730822741985321, -1.5097017288208008, 2.7351911067962646, -1.7043616771697998, -0.9276430606842041, 0.7466723322868347, -1.8722093105316162, -6.924632549285889, -1.0046334266662598, 2.018338441848755, 3.963143825531006, -1.3818542957305908, -0.5308361649513245, -4.607804775238037, -3.216724157333374, 2.7460825443267822, -2.9299683570861816, 9.723034858703613, 9.859701156616211, -5.241680145263672, 1.4002177715301514, 6.3827619552612305, 0.3994736969470978, -4.129916191101074, -1.4416178464889526, 3.18570876121521, 0.8604716062545776, 0.6745785474777222, -0.15999148786067963, 1.0148051977157593, -8.458526611328125, -0.4793817698955536, -2.048290252685547, 3.9762039184570312, 10.828354835510254, -4.984121322631836, -1.2224633693695068, 1.545647382736206, 5.13327169418335, 4.0905890464782715, -0.3002653419971466, -4.125127792358398, 3.9832139015197754, -0.33277440071105957, -2.419443130493164, 1.9269518852233887, 3.2353339195251465, 6.380773067474365, -3.711402416229248, 0.48030540347099304, 7.156899452209473, 0.9668617248535156, 0.9617059230804443, 0.8696279525756836, 1.6964939832687378, 3.3503470420837402, -0.5946971774101257, -1.3705639839172363, 4.8649091720581055, 1.2409381866455078, -0.5657046437263489, -0.9037593007087708, -0.4565436840057373, -0.383148729801178, -1.9589202404022217, -0.0953967422246933, 0.2806074023246765, 1.9301725625991821, -4.8763017654418945, 5.368773937225342, -6.370540618896484, -1.1147645711898804, 4.205840587615967, -2.309316873550415, -0.503194272518158, 5.78449821472168, -1.882463812828064, -7.140459060668945, -5.294353485107422, -0.1780167669057846, 5.495716571807861, 2.063310146331787, -2.91133713722229, -0.8280583620071411, -2.5712835788726807, 8.224903106689453, 4.960316181182861, 1.59291410446167, -1.3888795375823975, -2.0162911415100098, -6.825069904327393, -0.9000536799430847, 6.519688129425049, -4.992694854736328, 6.62036657333374, 3.9488041400909424, 4.698577880859375, 6.446654796600342, -2.466264009475708, -1.160292148590088, -0.8623784780502319, -4.7360334396362305, -1.3140329122543335, 2.5436787605285645, -12.077898979187012, -2.5366604328155518, -4.792043685913086, -2.009998083114624, 3.492405414581299, -11.471609115600586, -6.18569803237915, 4.447691917419434, -3.0435922145843506, 9.645283699035645, -2.8104262351989746, 3.143732786178589, 0.7750411629676819, -1.2561204433441162, -9.478120803833008, -4.7967071533203125, -5.421937465667725, -2.953732967376709, 2.7027008533477783, -0.7233703136444092, -1.1228978633880615, -0.6674396395683289, 4.3867411613464355, 1.4190013408660889, -5.484578609466553, -2.35430645942688, 0.6159204840660095, -2.393282651901245, 2.677177906036377, -1.0717582702636719, -7.207386016845703, 1.075805425643921, -4.91562557220459, -0.3470952808856964, -0.43073245882987976, 0.10179679095745087, 3.11191725730896, 0.4236465096473694, 3.5520474910736084, 1.0170096158981323, 0.4788309633731842, 3.4553534984588623, -0.19723954796791077, -1.4931015968322754, -1.822654128074646, 1.7389589548110962, 4.251816272735596, 0.21482929587364197, 0.6788772344589233, 0.32354021072387695, -5.762580871582031, 4.756209850311279, -2.611783266067505, -12.54886531829834, -1.859559416770935, 2.9043824672698975, 2.715096950531006, 2.487797975540161, -3.2568979263305664, -2.7822940349578857, -0.09423591196537018, -3.6134326457977295, 5.7923688888549805, -3.04194712638855, 0.47056224942207336, -0.16208547353744507, -5.149474143981934, 2.298133373260498, -9.001626968383789, 5.314421653747559, -4.111522674560547, 4.003856658935547, 2.4278361797332764, 1.5537265539169312, 0.9175952672958374, -3.318150758743286, -5.810858249664307, 4.917415618896484, 1.854874849319458, -0.7167766690254211, 1.5256824493408203, -0.010252255946397781, 0.42919373512268066, -3.950557231903076, 1.9128774404525757, -1.0725337266921997, -1.289929986000061, 5.531262397766113, -0.42565521597862244, 1.9292633533477783, 1.4643456935882568, -1.5059382915496826, 3.492164134979248, 0.51128089427948, 0.8228589296340942, 6.032386302947998, 0.1455022245645523, 1.4470114707946777, -3.1592071056365967, -6.421200275421143, -1.4875421524047852, -1.2391557693481445, 0.4834055006504059, -5.275904655456543, 4.152148246765137, -0.4447048306465149, -4.2823662757873535, 2.2303640842437744, 2.6142985820770264, 0.8380455374717712, -9.932780265808105, -1.5536893606185913, 4.652459621429443, 1.4878567457199097, 6.972073554992676, -3.91780686378479, -0.8792678117752075, 1.4894694089889526, -6.020663261413574, 6.3413872718811035, -0.30755144357681274, 0.7071393132209778, -4.645143508911133, 2.6377651691436768, -5.743404865264893, -4.787646293640137, 2.206507682800293, 3.1217615604400635, 2.495249032974243, -1.6707991361618042, 1.0092289447784424, -2.813558578491211, -5.594827651977539, -7.206709861755371, -0.22334955632686615, 5.755426406860352, 0.0819287896156311, 2.4481234550476074, 7.088443279266357, -5.7538299560546875, 4.966275215148926, 3.7891764640808105, 6.665126800537109, 6.4073028564453125, -1.751833438873291, 0.6755478978157043, 1.4589263200759888, -4.3757710456848145, -5.3523359298706055, -2.459991455078125, 1.1940656900405884, 0.615085244178772, -0.8728243112564087, 4.56056022644043, 4.394916534423828, -0.2551691234111786, 5.257063865661621, 1.7531195878982544, 1.4093412160873413, -4.263075351715088, 4.3586249351501465, -3.96832013130188, -0.7236319780349731, 2.63297438621521, 0.0008194894180633128, -2.582993984222412, -3.295137405395508, 4.546160697937012, -0.23241175711154938, -5.081633567810059, -0.3412112891674042, -4.611753463745117, -1.0130285024642944, 6.470952987670898, -0.31099048256874084, -2.812880754470825, -1.0262320041656494, 1.0236653089523315, -4.834990501403809, -0.6937815546989441, 2.163055181503296, 0.989126443862915, -6.511294364929199, -6.782196998596191, 2.0325028896331787, -12.393967628479004, 6.852006435394287], [-0.22303493320941925, -2.9295427799224854, 2.439837694168091, 1.5854976177215576, 4.3460845947265625, -0.06867010146379471, -0.09565424919128418, -1.399667501449585, 0.5858905911445618, 6.762322425842285, 5.7277750968933105, 2.0228331089019775, 6.1254377365112305, -2.150374174118042, 0.18802410364151, -0.6987628936767578, 1.6517786979675293, -10.830252647399902, -0.8345130681991577, 1.3534761667251587, 3.5239503383636475, 1.0695345401763916, -0.5388920903205872, 0.911974310874939, -4.016080379486084, -1.6969751119613647, 1.6518949270248413, 0.1963420808315277, 2.924260139465332, -5.788838863372803, -4.775421142578125, 4.498601913452148, 4.484622001647949, -2.9076781272888184, -0.4212682843208313, 0.6386677622795105, -1.0011578798294067, -5.970231056213379, 0.18472185730934143, 0.16654717922210693, 4.5425333976745605, -8.015398025512695, 4.298511028289795, -9.318711280822754, 3.5725696086883545, 0.4533112645149231, -8.215797424316406, 0.19528774917125702, 2.4415619373321533, 2.5876693725585938, 0.17751191556453705, 4.800653457641602, -0.9458233118057251, -3.466052293777466, 0.7189978957176208, -6.013711452484131, -0.2553960382938385, 2.079282760620117, -5.3937225341796875, 0.5188246369361877, -1.3687076568603516, 0.19321079552173615, -1.61294424533844, -0.05223812907934189, 1.396291732788086, 2.3973727226257324, 6.452172756195068, -7.6614861488342285, -5.830544471740723, -1.2805415391921997, 0.14926226437091827, 0.3114296495914459, -3.8883466720581055, -7.040850639343262, 5.309798240661621, 4.358098030090332, -1.026193380355835, 5.010931491851807, -2.0620055198669434, 0.6993857622146606, -2.8398501873016357, 4.126023292541504, -1.811116337776184, -5.738470554351807, -5.97351598739624, -0.7864934802055359, -4.1445159912109375, -0.7462623715400696, 1.2973790168762207, 0.04286187142133713, 5.859098434448242, 3.2601211071014404, -8.308006286621094, -0.14852553606033325, 1.5588264465332031, -4.028072834014893, 0.3368368148803711, -2.2381019592285156, 6.041589260101318, -0.6186083555221558, 1.904354453086853, 5.501769065856934, 4.829223155975342, -6.090274333953857, 2.3859174251556396, -2.3339593410491943, 6.64938497543335, 6.850296974182129, 3.577688694000244, 1.9598654508590698, -1.4039279222488403, 8.575575828552246, 9.305397033691406, 1.3045891523361206, 3.577401876449585, 3.736560106277466, -4.753856658935547, -5.370552062988281, 2.778681755065918, 6.283965110778809, -5.031956672668457, -2.275014638900757, 3.3068597316741943, 0.8949029445648193, -3.39141845703125, -0.5110409259796143, -0.43095332384109497, 1.8740487098693848, -0.7210443019866943, -0.3462303578853607, -7.4760565757751465, -3.0758345127105713, -5.839061260223389, 4.010403633117676, 2.0544986724853516, -4.09764289855957, 0.20397181808948517, 2.539444923400879, 3.2200028896331787, 0.7135060429573059, -6.395229816436768, -6.1395263671875, -4.729579925537109, 4.135786056518555, 7.586536407470703, 0.10087767243385315, -2.044619083404541, 1.7553174495697021, 8.902204513549805, 1.2975223064422607, 0.6756592392921448, -2.964989185333252, -7.732368469238281, -7.3219428062438965, 5.823451995849609, 5.937488079071045, -3.3169760704040527, -1.3814228773117065, 2.8933427333831787, 0.36694464087486267, -5.386471748352051, -2.8570992946624756, -1.6594407558441162, 2.534430980682373, 0.8708536028862, 3.570537805557251, -0.8703548312187195, 1.4229202270507812, -3.2544639110565186, 5.281916618347168, 0.955419659614563, -3.2105872631073, 3.7295548915863037, -0.3410036861896515, -4.814244270324707, -2.307912826538086, -2.840026378631592, 7.361328125, 4.496275901794434, 1.5258708000183105, 2.135448932647705, -5.198720932006836, 1.920638918876648, 1.007400393486023, 1.1475409269332886, 3.062115430831909, -0.3380388915538788, 0.7406359314918518, 0.7138916850090027, 3.5135657787323, 10.369403839111328, -3.276362895965576, -1.8409106731414795, 1.9282289743423462, 4.487127304077148, -0.2307145893573761, 0.33155205845832825, 3.6661181449890137, -6.0069260597229, 0.22919660806655884, -4.296924591064453, 3.9590401649475098, 11.74447250366211, -3.242403268814087, -4.187382221221924, -1.1858881711959839, -5.83914041519165, -2.1884677410125732, -1.1463093757629395, -3.292006254196167, 3.082347869873047, -1.565659761428833, -0.33050158619880676, 2.511037826538086, 0.7778701782226562, -6.9257121086120605, 5.848964691162109, -5.830999374389648, 3.660221576690674, -2.129687547683716, -2.9765961170196533, -3.7275919914245605, 6.768689155578613, 6.328266143798828, 3.4277002811431885, -5.335082530975342, 0.5063076019287109, -1.4796671867370605, -3.1932435035705566, -2.0765957832336426, -0.041074540466070175, 1.7004081010818481, -2.6766767501831055, 3.637516736984253, 0.22987964749336243, -7.760553359985352, 0.09855950623750687, -1.5429707765579224, 2.601383686065674, -4.1679558753967285, -1.9715328216552734, 9.09931755065918, 4.466790676116943, -1.078530192375183, -2.431889772415161, 7.088432312011719, -3.131364583969116, -1.7732359170913696, 5.726644039154053, -2.0738513469696045, 3.4429404735565186, 1.1407734155654907, 6.792642116546631, 0.6740915179252625, 3.4944701194763184, -10.093551635742188, 3.2756574153900146, -2.148798942565918, 0.12663429975509644, -6.069510459899902, -0.8190681338310242, -6.307610988616943, -4.256088733673096, -4.333924770355225, 3.8371264934539795, 4.218504428863525, -3.132434844970703, -7.409122943878174, 7.333491802215576, -1.0769933462142944, -3.9714772701263428, 4.760777473449707, -1.0886493921279907, -4.116401672363281, -1.37546968460083, 2.728391647338867, -2.1623144149780273, -4.374964237213135, 2.9910647869110107, 4.008078098297119, -7.767066478729248, 2.261883497238159, -0.7775063514709473, -6.891054630279541, -2.5903635025024414, 3.063325881958008, -1.714571475982666, 1.4319372177124023, -7.996584415435791, -3.08274507522583, 2.8574817180633545, 1.02716064453125, -0.9859680533409119, -0.26164495944976807, -3.5526437759399414, 0.6343338489532471, -0.6889044046401978, 0.6449586153030396, -1.210169792175293, 3.110274314880371, 6.351968765258789, -1.3881217241287231, 7.689855575561523, 0.4142705500125885, -5.29312801361084, -2.234747886657715, 1.3823109865188599, -4.1204514503479, -3.7371156215667725, -3.795243501663208, 0.11316041648387909, -8.068910598754883, 0.6542307734489441, -2.0225110054016113, 1.0107908248901367, 0.8189387321472168, -2.666402578353882, 9.17276382446289, -0.15984271466732025, -2.5554678440093994, 0.20585136115550995, 0.5305916666984558, 3.594909906387329, 4.094384670257568, 11.907049179077148, -0.12103212624788284, 0.9828850030899048, 2.676334857940674, 2.92182993888855, 0.5067987442016602, -0.6763902306556702, -4.383016586303711, 3.461639404296875, -1.222031831741333, -2.142364740371704, 1.4991745948791504, 5.469122886657715, 2.007572650909424, -8.94229507446289, -4.1432671546936035, -3.7512316703796387, -2.4230639934539795, 2.508714437484741, 2.868074655532837, 2.886606454849243, 3.079545021057129, 0.42671892046928406, 0.6657066345214844, -2.7927324771881104, 1.6466392278671265, 2.7266504764556885, 5.474597454071045, -0.21518905460834503, 3.6685049533843994, 8.346874237060547, 0.33565136790275574, 3.0328056812286377, 1.9598135948181152, -6.30608606338501, -4.528814792633057, -4.344694137573242, -0.3410327732563019, -4.380029201507568, -4.555627346038818, -0.9282599687576294, -5.151888847351074, 4.393095016479492, 3.5461554527282715, 4.990703582763672, 0.716911792755127, 5.03045129776001, -5.175897121429443, -0.707549512386322, -2.3595006465911865, -1.5140101909637451, 4.5436553955078125, -1.8214635848999023, 3.7416043281555176, 4.048183441162109, 3.101545572280884, -4.655158519744873, -1.013344168663025, -3.949664354324341, 1.5343797206878662, -3.804964780807495, -2.4093613624572754, 0.48165467381477356, -0.3939090967178345, -4.8603034019470215, 2.09513521194458, 5.445056438446045, 0.43459030985832214, -6.303816795349121, 1.834141731262207, 3.786342144012451, 4.969610691070557, -2.66507887840271, -5.6522111892700195, -1.6137207746505737, 5.786820888519287, 0.3041583001613617, 4.898207187652588, 1.1214088201522827, -4.388513565063477, 1.430402159690857, -0.7123206853866577, -0.07346194982528687, -2.5733518600463867, -2.2919209003448486, -3.9429125785827637, -3.0675177574157715, 0.880243182182312, -1.6841449737548828, 0.3474486768245697, -5.736521244049072, -1.9313939809799194, -4.180243015289307, -6.808128833770752, 2.3028934001922607, -2.3089687824249268, 1.9537461996078491, 0.022674262523651123, -7.175349235534668, -1.3814767599105835, -0.9818577170372009, 1.7394120693206787, 1.9670426845550537, -1.5138963460922241, 2.8248467445373535, 3.5624608993530273, 6.750709533691406, 7.120349407196045, 3.952484130859375, -5.440261363983154, 1.0051674842834473, 7.524575710296631, -2.8262696266174316, -0.9878262877464294, 2.7785446643829346, -6.965980529785156, 1.5701019763946533, -2.563072443008423, 7.925508975982666, -0.21889358758926392, 2.9135916233062744, 5.375199317932129, -1.4474414587020874, 3.6878249645233154, 0.7346119284629822, 0.17603759467601776, 0.7232306599617004, 6.393179416656494, 9.771550178527832, 5.479940891265869, -5.878136157989502, 5.774479389190674, -6.159278392791748, -3.1601874828338623, 1.326721429824829, 2.1585288047790527, 1.7617912292480469, 3.5603809356689453, -3.089327573776245, 6.20782470703125, 4.777978420257568, 3.9506289958953857, -5.00419282913208, -5.883930206298828, 2.4336225986480713, 8.781133651733398, -2.6712114810943604, -1.2992852926254272, -2.921924591064453, -5.588456630706787, 6.926642417907715, -3.989656925201416, 8.758126258850098, 4.632102012634277, -6.83944034576416, 1.4027819633483887, -6.804934978485107, 0.5073471665382385, 3.6046135425567627, -0.8860851526260376, -10.306014060974121, -4.011938095092773, 0.6418179273605347, 0.01787080243229866, 2.636193037033081, -8.603410720825195, -4.279318332672119, -1.8686062097549438, -3.308638572692871, -1.6477621793746948, 7.0142412185668945, 5.164340972900391, -6.7088303565979, 2.4112770557403564, 5.114935874938965, -7.211352348327637, 1.924174427986145, -9.281494140625, 3.5940937995910645, 4.315380096435547, -6.874576091766357, -3.9708588123321533, 4.783157825469971, 5.522650718688965, 1.5792227983474731, 0.4554990530014038, -4.206459045410156, -1.5941095352172852, -0.8345873951911926, 3.6441190242767334, 4.701913356781006, -3.173408269882202, -1.5146070718765259, -3.384110689163208, 2.048427104949951, -0.8215948343276978, -4.079011917114258, -6.2351555824279785, -0.06451952457427979, -0.2957492768764496, 0.003142331028357148, -5.217920780181885, 3.2468185424804688, 3.6386351585388184, -0.7817214727401733, -4.498779773712158, -2.3031370639801025, -3.8350274562835693, -8.28359317779541, 1.3690720796585083, -1.6074490547180176, 1.4015772342681885, 4.443357944488525, -1.4447828531265259, 2.986534595489502, 2.6424479484558105, -1.0435677766799927, -3.9621291160583496, 4.622233867645264, -5.868508815765381, -2.153092384338379, 6.05381441116333, 1.2191905975341797, 3.7638437747955322, 0.5124474763870239, -3.337820291519165, 0.058101195842027664, 2.536710023880005, 1.8745808601379395, 0.00688779354095459, -3.3797662258148193, 2.0105483531951904, -0.3582919239997864, -0.9382091164588928, -3.132641077041626, 1.6681697368621826, -4.424604892730713, -6.013298988342285, 3.543583393096924, -5.229106903076172, 5.71635103225708, 2.6646296977996826, 1.3128362894058228, 2.020751714706421, 2.2736501693725586, -6.505106449127197, -1.8091257810592651, -2.4874014854431152, 2.3796072006225586, 2.043766736984253, -0.5413974523544312, 1.191519856452942, -3.8109145164489746, -1.2870582342147827, 3.0704684257507324, -0.06456585973501205, -5.963551044464111, 3.868865966796875, 2.4298057556152344, 5.686459541320801, -3.249239444732666, -3.580277442932129, 1.5860438346862793, 7.786318778991699, 0.9187485575675964, 6.0469183921813965, -1.9159491062164307, 2.0207467079162598, -5.5160298347473145, -8.145153045654297, 2.7999110221862793, -0.3593403697013855, -10.541447639465332, -1.3674969673156738, 1.5636152029037476, 0.8617830872535706, 2.681580066680908, 0.9612951874732971, 8.825063705444336, -0.09940973669290543, 5.537280082702637, -2.7271370887756348, -2.0281155109405518, -9.942113876342773, 5.559102535247803, -4.63961124420166, 7.908588409423828, 1.4064898490905762, 1.2216393947601318, -1.6012587547302246, 1.205245852470398, 3.4304566383361816, -3.6442854404449463, 1.6350830793380737, -4.343362331390381, -2.467472553253174, 7.421029090881348, -1.4445655345916748, 0.018438875675201416, -7.658326148986816, 4.151385307312012, 2.9175403118133545, -0.9858657121658325, 3.2382090091705322, 5.038069725036621, -0.18212902545928955, -0.005359945353120565, 6.874339580535889, -0.4521295726299286, 2.5264461040496826, -5.190147876739502, 0.8542201519012451, -3.2630908489227295, -9.267821311950684, -3.951484203338623, 0.19628432393074036, 4.48367166519165, 4.869450569152832, 4.307602405548096, 0.3109813332557678, -0.06319579482078552, -4.3293986320495605, -3.413205146789551, 7.706412315368652, 0.40802818536758423, 1.5117143392562866, 8.016325950622559, -1.9928263425827026, -5.403833389282227, -5.257024765014648, -7.969505786895752, 2.0211398601531982, 0.4629021883010864, 3.5261499881744385, -0.7621541619300842, 3.5771982669830322, 3.026218891143799, 1.1421644687652588, -0.16780388355255127, 3.0039825439453125, -0.9392443895339966, -2.331406593322754, -2.669215679168701, -0.5615164637565613, 2.5032384395599365, 0.46664348244667053, -1.8708394765853882, 0.41850370168685913, -0.6413848996162415, 1.4256556034088135, -7.665746212005615, 8.66456413269043, -1.5230517387390137, -3.4646313190460205, -6.753365516662598, 2.4323136806488037, 2.8432626724243164, -6.215477466583252, 1.093662142753601, 2.1054511070251465, -8.453901290893555, -0.10892360657453537, 1.0744131803512573, 1.091860294342041, 2.1734583377838135, -5.074984073638916, -0.3812789022922516, -0.41617608070373535, 2.0045065879821777, 5.023313999176025, -1.6991103887557983, -1.3994885683059692, -3.2891838550567627, -1.4808526039123535, -7.972951412200928, -3.523531198501587, -3.740067481994629, -7.748432159423828, -3.352271556854248, 0.2882392406463623, 3.1468966007232666, -6.754573345184326, 3.7557480335235596, 0.8578119874000549, -5.9053754806518555, 0.8566679954528809, 4.545948505401611, 3.0302658081054688, 1.7877840995788574, -3.6497104167938232, -1.2526156902313232, -4.041186809539795, -3.8283112049102783, 0.6419788599014282, -3.5118868350982666, 1.2277073860168457, -1.3207287788391113, -1.1761361360549927, -3.911546468734741, 9.049154281616211, -1.6349430084228516, 4.21345329284668, 2.1562893390655518, -3.8640549182891846, 0.031762488186359406, 1.2765077352523804, 9.615983963012695, 7.065719127655029, -0.8330856561660767, -0.9739441275596619, -3.717787981033325, -3.131826639175415, 1.03004789352417, 9.777871131896973, 2.1578528881073, 0.6651489734649658, 3.061572313308716, -2.600144863128662, 2.275333881378174, 4.547059535980225, 0.22196033596992493, 5.115807056427002, 1.2478909492492676, 9.913350105285645, -2.356868028640747, 1.6648138761520386, -1.873650312423706, 2.8687281608581543, 4.460535049438477, 3.359445810317993, -2.1550168991088867, -2.093114137649536, -9.383298873901367, 2.4074418544769287, 0.4818117320537567, 5.499084949493408, -0.9154762625694275, -2.5305562019348145, -3.4892568588256836, 0.36264559626579285, 2.8620526790618896, 2.0118069648742676, 6.552158832550049, -1.3835008144378662, 1.2666208744049072, -0.8039757013320923, -3.377042293548584, -1.7497594356536865, 5.811797142028809, -7.310700416564941, 4.9388813972473145, -2.6091694831848145, -0.5235874056816101, -1.7374156713485718, -2.438478946685791, 6.751169681549072, 4.875558376312256, -3.2202272415161133, 2.43583083152771, -14.269207954406738, 0.5294366478919983, -2.0225722789764404, 2.4499130249023438, -2.7563469409942627, -1.4928871393203735, -7.667629241943359, 0.7608356475830078, 2.49812388420105, -0.20038804411888123, 3.228454113006592, 0.4030250608921051, 7.468341827392578, -1.711470603942871, -3.4581637382507324, 3.3792290687561035, -1.8179985284805298, -2.495271682739258, -4.050050735473633, -3.8940651416778564, -7.073202133178711, -2.073399782180786, -1.815416693687439, -2.0623838901519775, 0.3420834541320801, -1.2350069284439087, 1.2935290336608887, -4.104233741760254, 0.38554516434669495, -3.4428513050079346, 2.10483980178833, -5.575341701507568, -0.3298746347427368, 0.45286208391189575, -0.7980682253837585, 2.747934579849243, -4.12664794921875, -1.3413735628128052, 1.9397461414337158, 3.058023691177368, 2.4235727787017822, -1.5131845474243164, 1.0455272197723389, -3.4360482692718506, 1.0106302499771118, -1.7470548152923584, 3.5921645164489746, -2.478306531906128, -0.889962911605835, 3.4749200344085693, -0.13514508306980133, -1.8910576105117798, 1.1196649074554443, 0.4087997376918793, -4.421050548553467, -1.0890487432479858, -2.5522756576538086, 5.25440788269043, -3.2467966079711914, 0.884016752243042, -6.203418254852295, -5.612985134124756, -0.6477083563804626, 0.5416092872619629, -1.0200244188308716, 1.1075003147125244, -2.992581605911255, -5.759013652801514, -4.686025142669678, 3.8360378742218018, -3.071185827255249, 9.311830520629883, -4.671426296234131, 3.9424328804016113, -1.0530915260314941, 4.240730285644531, 3.879336357116699, -2.235232353210449, -1.4728676080703735, 2.2907207012176514, 0.7074712514877319, -6.740653038024902, -0.4647328555583954, -2.596798896789551, 1.4588266611099243, -11.920415878295898, 2.4890050888061523, -1.6802985668182373, 5.206324100494385, -2.9132988452911377, -0.7782056331634521, -2.676182746887207, 5.410959243774414, 0.2742935121059418, 2.4643261432647705, -2.157492160797119, 6.965085506439209, 1.2300750017166138, -1.4586145877838135, 1.3200469017028809, 6.176385402679443, 2.6452744007110596, 1.131572961807251, -0.15107238292694092, -8.134587287902832, 6.783915996551514, -9.198856353759766, -1.4785447120666504, -4.581829071044922, 0.6207655072212219, -0.8874965906143188, 2.399566650390625, -7.300968647003174, 0.637509822845459, 1.1303081512451172, -7.665732383728027, 0.0779334008693695, 2.141347885131836, 1.5231395959854126, -2.405797004699707, -1.3553433418273926, -9.469416618347168, -2.8020617961883545, 1.3056493997573853, -2.2008683681488037, -6.566574573516846, 3.2837767601013184, 0.9912603497505188, -5.810544490814209, 2.4465744495391846, 8.12518310546875, -2.3215479850769043, 6.669584274291992, 7.59600830078125, 1.2329453229904175, 4.15085506439209, 2.092952013015747, -1.1480557918548584, 4.824236869812012, 2.3085362911224365, 0.17345009744167328, 1.3624826669692993, 5.379587173461914, -0.6165269017219543, -5.359006881713867, -1.4185022115707397, -5.218317985534668, -3.2482101917266846, -5.324460983276367, 2.917863130569458, 3.708017110824585, 7.939442157745361, -5.4973015785217285, -0.1680401712656021, 1.759541630744934, 0.362510621547699, 1.6086153984069824, 10.19666862487793, 2.6923954486846924, -6.332520961761475, -2.5749943256378174, 5.853054523468018, -4.733870029449463, 7.902295112609863, 8.304268836975098, -3.1685822010040283, 5.447793960571289, -7.278083801269531, -1.2454465627670288, 1.5933951139450073, -3.2272586822509766, 0.5026684999465942, -3.766071081161499, 3.210779905319214, 6.2678680419921875, 6.997981548309326, -2.434216022491455, 6.092165470123291, -4.341372013092041, -1.441346526145935, 2.690463066101074, -4.24518346786499, -0.3223668038845062, -5.105616569519043, 7.096149921417236, -4.75128173828125, 5.781386852264404, 5.531994342803955, -1.65941321849823, 1.1745717525482178, -3.6645658016204834, -2.9320595264434814, 8.900535583496094, 4.174498081207275, -3.7048397064208984, -4.399238586425781, -0.9793092012405396, -6.372477054595947, 2.059453248977661, 11.985467910766602, -1.1262071132659912, -0.37882310152053833, -4.259251594543457, 0.7086736559867859, -0.986107349395752, 2.077545166015625, 6.9014482498168945, -1.3691177368164062, 1.8498868942260742, -1.7778340578079224, -6.95380973815918, 2.1879632472991943, 3.065397262573242, 1.343993902206421, -0.1915321797132492, -8.842375755310059, 1.5522350072860718, 3.938450813293457, -5.236482620239258, -1.5760446786880493, 2.6432952880859375, -6.446264266967773, -1.3661041259765625, 0.2985832691192627, -6.712521076202393, 4.016028881072998, 5.485720157623291, -4.670966625213623, 0.8524903059005737, 2.361238718032837, 0.97484290599823, -1.2670607566833496, 2.603630304336548, -5.244337558746338, -6.5765910148620605, 0.11484652012586594, -3.370615243911743, 5.439602375030518, 2.726337432861328, -0.18234077095985413, -8.641392707824707, -2.588651418685913, 5.3261847496032715, 6.877721786499023, -1.2402156591415405, 0.36761024594306946, 6.084026336669922, 3.7289299964904785, -0.776394248008728, 4.92505407333374, -7.3756818771362305, -2.575117588043213, -3.1374001502990723, 4.032103061676025, 9.656683921813965, -1.6205724477767944, 1.6068280935287476, 2.1721863746643066, -0.22995279729366302, -4.635488033294678, 4.279252052307129, -2.6531457901000977, 7.234707355499268, 8.116456031799316, 3.6996846199035645, 1.6717472076416016, 4.60841703414917, -1.0766520500183105, 0.5897121429443359, 1.9726979732513428, -2.0339808464050293, -0.951574981212616, 7.270334243774414, 6.4565277099609375, -0.8551120758056641, 2.9816739559173584, 10.666383743286133, -0.8637233972549438, 5.790318965911865, 8.512382507324219, -1.8658028841018677, 4.441326141357422, -0.4250332713127136, -6.709505558013916, 4.184048175811768, 9.227314949035645, 6.5417938232421875, 3.4098708629608154, -1.438772201538086, -1.2025444507598877, -3.9070441722869873, 1.6679691076278687, 1.1550384759902954, 1.0936503410339355, -5.0014238357543945, 2.3096392154693604, 6.654849052429199, 2.8846402168273926, 2.4713850021362305, -0.007332976441830397, 3.846477508544922, 5.381290912628174, -2.453622817993164, -0.27839943766593933, 0.8208502531051636, 5.350468158721924, -5.43419075012207, 5.214661598205566, 5.111917018890381, 7.255100727081299, 6.299836158752441, -4.128713130950928, -2.9983630180358887, 6.4460601806640625, -3.7850685119628906, -1.6164530515670776, 7.759425163269043, 7.966913223266602, -3.6878480911254883, 2.3106706142425537, 1.1835472583770752, -6.51399564743042, -3.28216552734375, -0.23410291969776154, -6.96996545791626, -0.7896971106529236, 1.654879093170166, 0.570270836353302, 3.2216956615448, 0.6046664118766785, -1.2982783317565918, -2.7952890396118164, -1.8575555086135864, -0.10097227245569229, -1.5908663272857666, -4.790084362030029, 5.28767204284668, 0.21606817841529846, -2.997015953063965, 0.4874573051929474, 2.6383161544799805, 8.53492546081543, 5.123905181884766, -2.9408860206604004, -0.13458050787448883, 1.0010548830032349, -0.10388340801000595, -3.379314661026001, -2.017371654510498, 9.842531204223633, -0.11748749762773514, -2.363652229309082, 4.064765930175781, 4.227226257324219, -1.7725982666015625, -1.15089750289917, -0.1951252669095993, -6.216590404510498, -3.1253180503845215, 0.20886202156543732, 7.939123153686523, 4.978837490081787, 1.671840786933899, 5.3577561378479, 3.318227529525757, -4.454249382019043, 0.9703799486160278, -4.867368698120117, -4.1385498046875, -1.7858130931854248, -0.500370979309082, -1.459855079650879, -1.1962275505065918, -5.479813575744629, 0.04859491065144539, 4.482849597930908, -6.594423294067383, 1.5816441774368286, 1.1212531328201294, -6.969783306121826, 2.276380777359009, -1.6276746988296509, -2.7960562705993652, 2.2127938270568848, 0.903910756111145, 0.4859074652194977, -0.7660568952560425, -3.310149908065796, 5.730984210968018, -4.521478652954102, -2.5164949893951416, 0.5280892252922058, -3.979893207550049, 5.476309299468994, -1.723285436630249, -3.162116050720215, -1.881752610206604, 5.80950927734375, -1.8014599084854126, 4.320441246032715, 4.759818077087402, 4.804272174835205, 2.5482938289642334, 3.9364516735076904, 0.9459538459777832, -4.56960391998291, 1.5969197750091553, 3.9048361778259277, 5.104862689971924, -3.007356882095337, -0.08177952468395233, 3.3702762126922607, 2.180842161178589, 3.156313419342041, -0.23388108611106873, 3.7631208896636963, 0.5847364664077759, -6.6127424240112305, 1.3268070220947266, -6.119765758514404, -4.615312099456787, -2.2532424926757812, -8.411287307739258, 2.736332416534424, -0.03871702030301094, -1.6703126430511475, -5.16417932510376, 2.829742670059204, 0.6040674448013306, 1.2626605033874512, -7.752068042755127, -7.303426742553711, 1.9404758214950562, -7.557219505310059, -3.378955602645874, 1.6389312744140625, -8.6718168258667, -2.312084197998047, 1.3991056680679321, 2.596304178237915, 4.76291561126709, 3.1364266872406006, -2.123512029647827, -1.9368921518325806, 0.5066123604774475, 2.2608108520507812, -0.30018025636672974, -0.047147199511528015, 5.339582920074463, 3.9571428298950195, 2.2762458324432373, -1.4239094257354736, 3.4352900981903076, -4.086993217468262, 0.7123047709465027, -1.7733790874481201, -2.7666664123535156, 5.746349334716797, -3.9537463188171387, -0.238326758146286, -0.13937358558177948, -6.674131393432617, -2.6949853897094727, -6.044242858886719, 0.43450430035591125, -2.5851941108703613, 9.194499015808105, -1.34355628490448, 0.4436614513397217, 0.3242989182472229, 1.3164491653442383, -0.9922369122505188, 1.3626402616500854, -3.484902858734131, 5.361078262329102, 1.8060877323150635, 2.042093276977539, 6.471992015838623, 1.776123285293579, 0.08445531874895096, 0.9597672820091248, 2.4391286373138428, -0.2006753385066986, 1.570130705833435, -0.023650800809264183, -3.7966115474700928, -0.316203773021698, 1.9487488269805908, -1.5857572555541992, 1.4762639999389648, -0.13917621970176697, 4.272315979003906, 1.474843978881836, -0.008188816718757153, -3.1716794967651367, 1.2673646211624146, -0.4232836067676544, 2.2435503005981445, 1.9023436307907104, 0.856378972530365, 1.159593939781189, -2.208380937576294, 3.2737414836883545, -3.695026159286499, 2.558838129043579, 7.942007541656494, 6.428256511688232, -0.37335020303726196, -3.509002447128296, 0.5951444506645203, 4.073171138763428, -0.12720535695552826, 0.6038927435874939, -1.772200345993042, 2.927983045578003, -2.7270967960357666, -7.479854106903076, -0.8901984095573425, 5.053964138031006, 2.220899820327759, 4.304169654846191, 2.019779682159424, 0.39227551221847534, 1.5091614723205566, 0.6271183490753174, 1.5939399003982544, -9.676318168640137, -7.849095821380615, -3.0663857460021973, 5.659145832061768, 4.5754714012146, 7.802401542663574, -3.5640921592712402, -7.185250759124756, 2.416290283203125, -0.3156619668006897, 0.3707524836063385, 2.7786781787872314, -4.400331974029541, -1.1541881561279297, 1.503191590309143, -0.06182892993092537, -1.4131673574447632, 3.0435307025909424, -0.7507579326629639, -1.2804405689239502, 1.4820783138275146, -2.2186472415924072, 1.6334866285324097, -2.995093584060669, 2.3111977577209473, -2.648540735244751, -5.659026622772217, -3.6739776134490967, 4.110380172729492, -6.427298069000244, -7.37918758392334, 2.3473143577575684, -0.9245597124099731, 2.3808627128601074, -7.748246192932129, -4.014883041381836, -3.345980644226074, 0.17842765152454376, -1.6013438701629639, -7.456545352935791, 2.5230538845062256, 5.5059356689453125, 4.6111555099487305, -5.406944274902344, 7.047935962677002, 2.2104198932647705, 0.7961824536323547, 1.9009863138198853, 6.80850887298584, 2.5090889930725098, -5.070940971374512, -9.383490562438965, -0.41156262159347534, -0.3977733850479126, -4.964865684509277, 5.722718238830566, 3.038814067840576, 3.2974791526794434, 1.1616383790969849, -0.882125198841095, 2.91607403755188, -0.7764003276824951, -2.6484220027923584, 0.8583227396011353, 4.907254219055176, -4.119471549987793, 4.176466464996338, -4.921554088592529, -4.996910095214844, 2.1342837810516357, -0.5214643478393555, 2.1627511978149414, -6.161363124847412, 2.053675889968872, 5.716516017913818, 1.4830840826034546, -3.8728108406066895, -0.061614155769348145, -1.9109846353530884, -1.2212767601013184, 3.49231219291687, -0.6480247974395752, -5.758240222930908, -2.162165641784668, -9.577094078063965, 5.930229187011719, -0.7734993696212769, -5.4884772300720215, -2.9792869091033936, -2.7217953205108643, -1.1494886875152588, -2.409872055053711, -2.10666823387146, -1.6533355712890625, -6.567497253417969, 1.2459900379180908, -1.7532978057861328, 8.395538330078125, 2.3988466262817383, -1.161970853805542, 0.46359720826148987, -2.9470748901367188, 2.564403533935547, 0.09215427935123444, 9.500073432922363, -2.8672115802764893, -2.7664594650268555, -4.175514221191406, -3.344144582748413, 5.150026798248291, 0.14300449192523956, -2.3707258701324463, 1.1509913206100464, 1.7424451112747192, -9.946301460266113, 3.5355305671691895, 6.116411209106445, -2.2482411861419678, 1.3237313032150269, 6.015353202819824, 4.973628997802734, 2.251098155975342, -0.5908920764923096, -2.7853734493255615, -2.710148811340332, 8.695390701293945, -3.3057079315185547, -1.8556665182113647, 3.5939133167266846, 2.9017059803009033, -2.307793140411377, 1.6043469905853271, -0.27038052678108215, -2.7683327198028564, -1.5562759637832642, -0.5584322214126587, 5.300354957580566, 6.954543590545654, -5.595868110656738, 9.305331230163574, -0.3416815996170044, 1.9683620929718018, 5.611140251159668, 2.2730624675750732, -3.2086400985717773, -5.2120280265808105, 3.2004897594451904, 1.16778564453125, -2.0782101154327393, 8.607431411743164, 0.9230192303657532, 3.006098985671997, -6.499998092651367, 4.10886812210083, -5.9888834953308105, -3.9143013954162598, -3.6348612308502197, -4.171832084655762, -4.442319869995117, -0.5636915564537048, -4.93499231338501, 8.749290466308594, -8.674420356750488, 3.974177598953247, -1.448875904083252, -0.6756261587142944, -3.0289928913116455, -6.743870258331299, -2.254293203353882, 0.7030622959136963, 2.1957709789276123, 7.146844863891602, -0.9246237277984619, -5.391013145446777, -6.193376064300537, -0.7264910936355591, -7.0272135734558105, 0.8580055832862854, 5.305781364440918, -3.698756694793701, 4.683337211608887, 2.275070905685425, -4.588424205780029, 3.961947441101074, -0.6596919894218445, 2.0786406993865967, 1.3770294189453125, -5.063257217407227, 4.285556793212891, -13.029302597045898, 7.012674808502197, 1.149348497390747, -0.8135944604873657, 3.956916570663452, -0.8373414278030396, 1.2793900966644287, -0.5523983240127563, 0.4293749928474426, -3.29764461517334, -1.4752486944198608, -2.389345407485962, -8.222504615783691, -1.7431221008300781, 6.363802433013916, -0.14619438350200653, -4.757930755615234, -0.8009389638900757, -1.5513184070587158, 0.48548781871795654, 0.45683181285858154, -2.7564892768859863, 5.2051801681518555, -3.920243501663208, 5.953179836273193, -1.8656407594680786, -3.2753407955169678, -4.695207118988037, 3.764054775238037, -5.5262451171875, -4.928323745727539, -1.6984153985977173, -0.5983351469039917, 6.896854400634766, 4.098405361175537, -3.4571659564971924, -1.7172774076461792, 2.334432601928711, 1.180730938911438, -6.639459609985352, -1.4534211158752441, -6.095944881439209, -0.03495826944708824, 1.4835606813430786, 0.15057064592838287, 0.9874585270881653, 0.9989001750946045, -0.48621049523353577, 3.347630023956299, -1.71052885055542, -0.7778470516204834, -5.8312249183654785, 2.5341501235961914, -3.516350269317627, -1.4659701585769653, -0.3080380856990814, 5.922830104827881, 0.5042583346366882, -4.208769798278809, -1.1710011959075928, -6.763060569763184, -2.4979758262634277, 3.063196897506714, 3.686305046081543, 3.0510239601135254, -4.337707996368408, -4.52391242980957, 2.2522737979888916, 1.0962159633636475, 0.26092833280563354, -3.332794189453125, 4.083812236785889, 3.8374598026275635, 6.747626781463623, -0.843647837638855, -4.075489044189453, 0.8355395793914795, 4.179265975952148, -10.3452787399292, 2.221569061279297, -0.3837270438671112, 4.986822128295898, 7.221071243286133, -3.9939560890197754, -1.3542479276657104, 0.32684558629989624, -0.44699135422706604, 6.599478721618652, -5.394165992736816, 5.046844482421875, 3.21382737159729, 4.824982166290283, -1.771848201751709, -0.2533137798309326, 3.433687686920166, -3.5637800693511963, -10.206856727600098, -3.4231810569763184, 4.561209201812744, -1.573091745376587, -0.8207986354827881, 0.0683959499001503, -0.7646439671516418, 1.5696945190429688, 2.629415988922119, 7.293589115142822, 2.8479135036468506, 4.780595302581787, 5.2470479011535645, -1.2236415147781372, 5.9189863204956055, 5.74468469619751, 6.01762580871582, 4.153453350067139, -0.23457397520542145, -4.2004241943359375, -0.010412587784230709, -4.581971168518066, -5.702052593231201, 6.792773723602295, -0.8175419569015503, -4.59910249710083, 2.5118446350097656, -2.436206340789795, 1.2202250957489014, 6.852688312530518, 6.415832042694092, 0.22684019804000854, 3.9368700981140137, -5.663638114929199, 7.011969566345215, -0.35245898365974426, -3.159320831298828, 10.141592025756836, -1.4830129146575928, -6.770432949066162, -2.5302298069000244, 1.5213323831558228, 5.049140930175781, -2.2631099224090576, -0.5511547923088074, -2.8690342903137207, 6.044198989868164, -3.702695369720459, -3.211743116378784, -3.344350576400757, 0.1666349321603775, -0.8320091962814331, -2.2116007804870605, 3.4136533737182617, 8.940057754516602, 0.6636885404586792, -3.2904722690582275, 6.0269694328308105, 0.7552646398544312, -7.10627555847168, 1.784088134765625, -0.9442301988601685, 2.441551685333252, 6.50506591796875, -3.570734739303589, -0.8781317472457886, -6.192781925201416, 5.017988681793213, 3.550532102584839, 4.510079860687256, 2.2677054405212402, -6.93432092666626, -0.7985284328460693, -0.13653990626335144, -1.0246742963790894, -0.26828664541244507, 5.15087890625, -5.287407398223877, -2.3668394088745117, 0.21760588884353638, 3.583045244216919, -1.0059421062469482, -1.9591466188430786, 6.169253349304199, 6.247803211212158, -1.3820786476135254, 3.1256651878356934, -0.030682099983096123, -7.7884721755981445, -1.036549687385559, 6.76181697845459, -2.463719129562378, 0.9204004406929016, -1.9644320011138916, -2.282383918762207, -0.18781405687332153, 0.8383199572563171, 5.672987461090088, 7.070455074310303, 11.295600891113281, -5.201212406158447, -1.1515988111495972, -0.13803508877754211, 4.137884140014648, 0.8132331967353821, -4.011523723602295, 2.8402116298675537, 4.809595108032227, 8.955235481262207, -2.817143678665161, 11.930853843688965, -6.983738899230957, 1.7326124906539917, 3.6605770587921143, 0.014105227775871754, -2.6346890926361084, 4.885693550109863, -5.332205772399902, 1.2099555730819702, -6.12358283996582, -3.049546957015991, -2.2033305168151855, -3.422619581222534, -0.3092259466648102, -1.4074262380599976, 3.3543245792388916, 2.387150287628174, -1.926647663116455, -0.2969518303871155, -2.7799599170684814, 1.4723143577575684, -7.736991882324219, 3.832327365875244, -5.051450729370117, -1.9112943410873413, 0.5096052289009094, -3.021090269088745, 1.8918018341064453, -7.371149063110352, 4.637352466583252, -3.327791929244995, 0.9931925535202026, 2.6001217365264893, 1.1967240571975708, -6.109885215759277, -4.793529510498047, 7.990656852722168, 2.376018524169922, 5.9112043380737305, -5.5086445808410645, -0.8115556240081787, 0.46971940994262695, -2.2122602462768555, 1.371997594833374, -0.2659154534339905, -3.3459935188293457, -5.396206378936768, 1.4362764358520508, 1.1746810674667358, -7.006532192230225, -2.1546618938446045, -10.0663480758667, -3.685670852661133, 3.1878647804260254, -2.890718460083008, 0.1983201652765274, -7.446237564086914, -1.7453683614730835, 3.1558032035827637, 2.94826340675354, -1.1563390493392944, 1.585113525390625, 6.121309757232666, -2.3786206245422363, 6.107975482940674, -4.159648895263672, -4.42795991897583, -1.620658040046692, -2.9965436458587646, -0.7963505387306213, -1.01618230342865, -3.026557445526123, 3.794703245162964, -10.11375617980957, 5.598763465881348, 2.0577244758605957, 6.305401802062988, -1.919517159461975, -1.9686390161514282, 0.761572539806366, -6.720118999481201, 0.19493336975574493, 4.94659423828125, 4.934059143066406, -6.555160999298096, 3.105055332183838, 0.5105480551719666, -1.9421470165252686, 1.3797681331634521, -6.7229533195495605, 7.2887749671936035, -10.373615264892578, -2.0330142974853516, -5.070494174957275, 7.0648193359375, 0.9784853458404541, 6.3251051902771, -4.567362308502197, 8.558577537536621, -0.5399103760719299, -1.947510838508606, 3.9389588832855225, 8.631490707397461, 3.1413955688476562, -1.6434721946716309, -0.13922257721424103, 3.654320240020752, 1.1203628778457642, -7.872570991516113, -5.1602783203125, 6.224782943725586, 5.862907886505127, -2.3031721115112305, -3.6489365100860596, -5.4927873611450195, -2.4200429916381836, 7.892885684967041, -2.479085922241211, 2.672532796859741, 3.6709542274475098, 1.5111382007598877, 3.5136661529541016, -5.552769660949707, 6.294399261474609, -7.165128707885742, 2.0218098163604736, 1.178296685218811, 1.3204244375228882, 2.965740919113159, 2.8682944774627686, -0.2352045178413391, 1.162233591079712, 0.8120043277740479, 1.7093150615692139, 2.675104856491089, 2.17415452003479, -6.848560810089111, 1.9460315704345703, 1.7674486637115479, -7.903853893280029, 5.902999401092529, 1.1148455142974854, -7.579322338104248, -3.289956569671631, -2.5242104530334473, -3.8619871139526367, -1.421305537223816, -0.37196066975593567, 5.0255584716796875, 3.0621228218078613, 2.5540459156036377, -3.867140293121338, 5.738489627838135, -4.684326648712158, -1.8380686044692993, 5.001817226409912, -0.474290668964386, 2.2499780654907227, -4.131067276000977, 2.9245307445526123, -2.7744874954223633, -2.1124510765075684, 0.8214801549911499, 6.437611103057861, -6.117002010345459, 1.8369879722595215, 2.60837459564209, -1.6933737993240356, 4.263057708740234, 0.572517454624176, -4.44555139541626, -5.382095813751221, -2.8537466526031494, 2.7015256881713867, -7.082028388977051, 1.4016984701156616, 2.2957241535186768, -1.7027825117111206, -2.9157683849334717, 1.983201026916504, 5.0566582679748535, -2.506129741668701, -2.590076446533203, 6.20428991317749, 2.463392734527588, 1.8367990255355835, 6.2255048751831055, 6.815374374389648, -3.292436361312866, 5.648252010345459, -0.22854255139827728, 2.114494562149048, -0.3376905918121338, -2.6762311458587646, 0.5261914730072021, -6.201444149017334, 3.3057594299316406, -7.397517204284668, 1.5360661745071411, -5.817500591278076, -0.39226651191711426, -1.7603483200073242, -6.856926441192627, 0.1150432899594307, 7.475478172302246, 1.8834199905395508, -4.389329433441162, 1.3979617357254028, -4.237051486968994, -0.8713939189910889, -6.380879878997803, -1.7656004428863525, -5.239116191864014, -2.5299649238586426, 0.32469698786735535, 4.095602512359619, 4.436573028564453, 8.240490913391113, 7.984209060668945, -3.081514596939087, -3.379913330078125, 9.756974220275879, 1.769936203956604, 1.6086772680282593, 2.9570536613464355, 4.174402713775635, -1.5759472846984863, -1.69195556640625, 4.221753120422363, 0.11207085847854614, -3.7303659915924072, 3.3849756717681885, -2.119948148727417, -5.493259429931641, -0.05257024988532066, -1.4186583757400513, 3.7686309814453125, -5.098062992095947, 2.200752019882202, 7.096907138824463, -0.5324957966804504, -1.955500841140747, -4.2623677253723145, 4.762567043304443, -1.774522066116333, 5.808027267456055, -0.8953267335891724, 2.5743813514709473, 0.01940835639834404, -1.293174386024475, -1.018105387687683, 1.2362662553787231, 1.7400954961776733, -7.67531681060791, -1.3556574583053589, 3.003418207168579, -0.6400015950202942, -5.792720794677734, -1.711818814277649, 1.4738527536392212, 0.9052086472511292, 0.8376649022102356, 7.665730953216553, -3.3857975006103516, 1.5695271492004395, -1.6168426275253296, -4.882857799530029, 3.697526216506958, 6.910186767578125, 3.6493985652923584, 1.6975058317184448, -5.626504421234131, -0.8215540647506714, -3.0459039211273193, 1.598063349723816, -8.497212409973145, -3.12833309173584, -8.963473320007324, 1.8849830627441406, 3.0312259197235107, -1.4317902326583862, -0.6750146150588989, -6.04203987121582, 4.563961982727051, -3.0440938472747803, -3.0228686332702637, 0.36690109968185425, -6.518371105194092, -1.2506502866744995, 2.7102036476135254, 1.1693918704986572, -1.2596018314361572, 1.125618577003479, -1.5457314252853394, 4.6295270919799805, -0.8530590534210205, 5.470693588256836, 0.6639873385429382, 2.2990026473999023, 0.4358339309692383, 3.0061306953430176, -7.534064292907715, 8.03937816619873, 0.45406585931777954, -4.79348087310791, 0.60421222448349, -6.072110652923584, -1.4072966575622559, -0.07661695778369904, -0.085142120718956, -5.375612735748291, -6.024190425872803, -1.166785717010498, 4.0997490882873535, 1.9767487049102783, 3.864391326904297, 1.2419365644454956, 1.288497805595398, 0.6567592024803162, -4.706356525421143, -1.5826492309570312, 7.612435817718506, 2.992887258529663, 1.3424978256225586, 0.2318689078092575, 11.481559753417969, 2.7339160442352295, 0.5802072286605835, -6.515579700469971, 10.119511604309082, -5.689756393432617, 2.0896413326263428, -6.017428398132324, -5.00282096862793, 5.286297798156738, -9.340523719787598, 1.4544594287872314, 5.055803298950195, 3.063929319381714, -2.6784605979919434, 2.9470419883728027, -1.5830247402191162, -2.0670113563537598, 3.431317090988159, 3.6298134326934814, -5.061697006225586, -0.9869884252548218, 2.4533843994140625, -0.18317139148712158, 11.223323822021484, 6.879642963409424, 6.145890235900879, -1.1321898698806763, -1.7372454404830933, -1.3136337995529175, 6.632906913757324, 4.30640983581543, 7.283044815063477, -1.4585853815078735, 0.08303237706422806, -3.4274098873138428, -3.1608774662017822, -7.374345779418945, -4.163628578186035, 5.932287216186523, 0.44109898805618286, 4.61442232131958, 2.790985584259033, 0.3312969505786896, 6.758511066436768, 5.259975433349609, -6.946147918701172, 2.853719472885132, -0.8064945340156555, -3.672710657119751, 3.5545814037323, 0.5088616013526917, -8.352206230163574, -3.4852192401885986, -0.9259741306304932, -6.7005615234375, -2.0427567958831787, 4.0409650802612305, 2.5779335498809814, -2.1328351497650146, 6.160979747772217, 1.1772407293319702, -0.16279146075248718, 1.0090333223342896, -0.9164911508560181, 0.5658433437347412, -2.488499402999878, -1.4879947900772095, -8.297924041748047, -6.314846992492676, 2.903451681137085, -5.048532009124756, 5.545469284057617, 2.180168390274048, 2.5430846214294434, -1.1507526636123657, 4.277349948883057, -1.9609344005584717, -0.6166868805885315, 4.519242763519287, 1.426820993423462, -5.0326433181762695, -11.268203735351562, -7.579349517822266, 7.577589511871338, 8.464412689208984, -1.0548145771026611, -3.48956036567688, 4.195631980895996, -3.758699655532837, 8.205011367797852, 2.9449989795684814, 2.1610798835754395, -3.828065872192383, -5.970310211181641, -1.3043980598449707, 3.599416971206665, 1.4751484394073486, -5.2046942710876465, -1.364477515220642, 1.0884582996368408, -5.54764986038208, 8.974042892456055, 2.0702967643737793, 5.495149612426758, -6.221074104309082, 1.0419403314590454, 4.041945457458496, -0.5262858271598816, -4.075737953186035, 5.838623523712158, -2.098543167114258, 6.53286600112915, -0.8519101738929749, 5.856527805328369, -5.03773307800293, 0.5222567319869995, -1.8357754945755005, -9.179060935974121, 3.2093987464904785, 8.164799690246582, -2.3416452407836914, 2.0781288146972656, -2.3457281589508057, -0.6466826796531677, -9.214385032653809, -1.7893885374069214, -8.37643051147461, -0.7195413112640381, -1.0488979816436768, -2.7593157291412354, 3.031914472579956, 1.822118878364563, -1.8398637771606445, -4.883600234985352, 5.415258407592773, -2.602355718612671, -0.8556661009788513, -1.717928409576416, 2.580639362335205, 5.07402229309082, 4.430392742156982, -0.394527405500412, -3.4678616523742676, -5.161038398742676, -2.822636842727661, 3.435896873474121, -6.681433200836182, -9.128552436828613, 4.775633811950684, 3.2223775386810303, -2.749422073364258, -3.0267317295074463, 1.055841326713562, -4.030898571014404, -4.17132568359375, -3.7917399406433105, -0.9222846627235413, -7.808032035827637, 6.392777442932129, -12.25931453704834, -0.10071037709712982, -0.009292793460190296, 6.394657611846924, -7.5823540687561035, 3.6538147926330566, 1.2997217178344727, -3.996253252029419, -1.5606046915054321, 2.0230748653411865, 2.3552191257476807, -0.17515328526496887, -0.9420940279960632, -0.9963457584381104, 3.791963815689087, 1.026005744934082, 5.503830909729004, -1.1115541458129883, 3.0692265033721924, -1.0475836992263794, -2.041400671005249, -3.6931326389312744, -2.310551643371582, 0.9340940117835999, -6.5060834884643555, -3.4127213954925537, -2.1511878967285156, 3.3543717861175537, -1.8135536909103394, -2.7225122451782227, 0.5303158164024353, 2.0144283771514893, 0.12377554178237915, 4.530737400054932, 1.5944103002548218, 2.2856626510620117, -5.997112274169922, 0.6737675666809082, -2.7624857425689697, -5.61742639541626, -0.11317017674446106, -0.9369680881500244, 0.4606977105140686, 8.915536880493164, -2.0797910690307617, -1.085832118988037, -0.3904246985912323, -1.9648669958114624, -7.261447906494141, 9.973114967346191, -7.652570724487305, -4.379601955413818, -0.023840442299842834, -3.929941177368164, 1.960068702697754, 2.430896520614624, 2.3165283203125, 2.5990562438964844, 1.1770262718200684, 4.948964595794678, -2.311133861541748, 2.0566678047180176, -0.4336710274219513, 2.2148616313934326, -5.080059051513672, -1.7899106740951538, -0.49705755710601807, -1.4254473447799683, 2.7083017826080322, 5.100668907165527, -2.7568020820617676, 0.4583090841770172, 5.621680736541748, 0.2936409115791321, -0.7390734553337097, -2.8382627964019775, 7.255703449249268, 4.988760948181152, -3.0772311687469482, 5.8592939376831055, 0.46636566519737244, -4.984065055847168, -7.08194637298584, 1.0598855018615723, -2.389169216156006, -2.26259183883667, 4.300900936126709, 2.689831256866455, 4.64207124710083, -4.770129680633545, -2.453603744506836, -7.153185844421387, 8.80354118347168, -5.028295516967773, 3.3145642280578613, 1.2021903991699219, -3.7770836353302, 0.6222948431968689, 8.603737831115723, 1.3909707069396973, 1.4208316802978516, 1.197479486465454, 2.020982027053833, 1.6615588665008545, -1.0141485929489136, 3.4011363983154297, -2.061897039413452, -2.1866748332977295, -0.33960840106010437, -2.1604959964752197, -2.0319175720214844, 0.33388060331344604, 1.1459583044052124, 6.104428768157959, -7.689235210418701, 3.12150239944458, -2.028552293777466, 1.072731852531433, -1.3495609760284424, 9.321364402770996, -2.035010576248169, 1.331762671470642, -5.3478593826293945, -2.988621711730957, 9.942341804504395, -1.7198070287704468, 4.268372535705566, 5.714870452880859, 3.4320356845855713, -1.2513147592544556, 6.375427722930908, 4.539267539978027, 1.8730483055114746, -7.016918659210205, 0.3696548342704773, 3.04876446723938, -2.5059900283813477, 0.566727876663208, -5.360517978668213, -1.0380085706710815, -1.119497299194336, 5.616774082183838, 3.764394760131836, 3.3437561988830566, 1.053709626197815, -1.979087233543396, -7.789999961853027, -2.1370177268981934, 5.283775806427002, -0.18798193335533142, -4.844794750213623, 2.3261680603027344, 2.8169538974761963, -3.6774911880493164, 2.2427961826324463, 4.31050968170166, 2.1392858028411865, -6.5800933837890625, 3.107431650161743, -2.01918888092041, 2.0233051776885986, 2.8365771770477295, 1.991296648979187, 0.32805386185646057, 2.5663681030273438, 5.315837860107422, -1.4231363534927368, -4.7653069496154785, -8.915112495422363, -4.4170026779174805, -2.779912233352661, -0.7385025024414062, -3.5966289043426514, -2.7892162799835205, -4.343780994415283, -2.1877410411834717, 6.484835624694824, 2.1088707447052, -5.119302272796631, -2.5639073848724365, 1.1216627359390259, -3.535114049911499, 1.6425367593765259, -6.143850326538086, -2.136873483657837, 1.0599898099899292, -5.9449849128723145, 2.1084227561950684, 4.1859846115112305, 3.2773172855377197, 0.3265293836593628, 0.7165574431419373, -0.613764762878418, 0.08988135308027267, -3.2094385623931885, -0.8642469644546509, 1.8159621953964233, -0.6153980493545532, -4.406717300415039, 1.58732271194458, -2.3941218852996826, 5.772317409515381, -1.4623048305511475, 2.119497776031494, 3.094212770462036, 4.37177848815918, -4.7130045890808105, 5.333503723144531, 2.396500587463379, 0.002104981569573283, -4.661368370056152, 7.781375408172607, 2.223114490509033, 4.6141862869262695, 3.5732429027557373, 6.060824871063232, 1.065248966217041, 3.3629508018493652, -5.741907119750977, -0.43057993054389954, 3.700399160385132, 1.5997281074523926, 3.301856517791748, -4.701703071594238, 4.30288028717041, -3.8754332065582275, 0.48021820187568665, 6.271707534790039, 9.15466022491455, -2.349245548248291, 3.7655434608459473, 1.3353474140167236, 1.5980241298675537, 1.5211780071258545, -4.5553388595581055, 1.8846887350082397, -0.9220240712165833, -3.1935970783233643, 1.9445815086364746, -5.383281230926514, 1.3871173858642578, 3.6357853412628174, -4.253101825714111, -5.112510681152344, -0.6803107857704163, -3.2323193550109863, 6.951475620269775, -3.253425359725952, -2.5915937423706055, 2.138593912124634, 3.4118764400482178, 0.08222191780805588, 6.066452980041504, -2.4110777378082275, 6.555942058563232, -6.525144577026367, 6.0306243896484375, 4.226902008056641, -0.217062845826149, 1.2812151908874512, 1.9743518829345703, -17.868854522705078, -3.402740955352783, -3.375946044921875, 0.2629058063030243, 1.4518208503723145, 1.1789090633392334, 3.4670679569244385, -0.9601746797561646, -0.7619209289550781, -3.03412127494812, -7.708820819854736, 4.7827677726745605, -0.049768105149269104, 0.8680445551872253, 4.842201232910156, -3.967658042907715, 3.062525987625122, -2.3229336738586426, 2.5624513626098633, 4.283743381500244, 4.400167942047119, -0.17856132984161377, 0.11772599071264267, 5.265122890472412, -0.32462865114212036, 4.2607221603393555, 0.5684965252876282, 6.295039653778076, -0.28249433636665344, 2.0183446407318115, -6.465054035186768, -1.581089735031128, 5.572882652282715, -4.551069736480713, -0.7178032994270325, 4.8099822998046875, -6.736676216125488, 12.845300674438477, 3.183882474899292, 0.2988872528076172, 8.013337135314941, 1.754904866218567, -3.074676036834717, -1.9477440118789673, 6.735472679138184, 2.174246072769165, 0.0018757397774606943, 2.637281894683838, -7.287398338317871, 1.1848503351211548, -4.394652366638184, -4.156110763549805, 0.7561208605766296, 0.11275000125169754, 3.5539605617523193, -3.5711729526519775, -0.13306169211864471, -2.0022976398468018, 4.385655879974365, 2.286451578140259, -5.608823776245117, 1.9451924562454224, 3.5260019302368164, 0.9214933514595032, 1.106881022453308, 6.413078784942627, 3.6440656185150146, -2.073760509490967, -0.988207221031189, -2.6709275245666504, -4.01713228225708, -2.870513916015625, 3.149482488632202, 1.861291766166687, 1.1222219467163086, -7.360279083251953, -2.3655340671539307, 5.166437149047852, -5.299286842346191, 0.5812171101570129, -8.836554527282715, 0.7330142259597778, 6.256353855133057, 0.8782721757888794, 5.550612926483154, 4.256153106689453, 1.5255385637283325, 2.019605875015259, -4.62880277633667, -0.4221481382846832, -0.037215158343315125, 4.382065773010254, 3.0556893348693848, 0.5564652681350708, -1.2414430379867554, -5.220719337463379, 0.7945097088813782, -0.6629582643508911, -2.801457405090332, -2.5714259147644043, -5.128623962402344, 5.532969951629639, -3.81681489944458, -4.663612365722656, 2.0937092304229736, 7.409513473510742, 1.5629100799560547, 0.10309935361146927, -2.1511456966400146, -1.0450468063354492, -4.333910942077637, -4.494452953338623, 4.418142318725586, -2.2249562740325928, -5.568262100219727, -5.583387851715088, 3.0784003734588623, 4.421380996704102, 5.6460981369018555, -9.205315589904785, 1.538170576095581, 0.5000578761100769, 6.760022163391113, -0.6664509773254395, -5.556632995605469, 4.133902549743652, 6.650335788726807, -1.6254171133041382, -1.4065948724746704, 4.686763763427734, -0.28861990571022034, -9.055046081542969, 2.461580276489258, 7.635197639465332, 6.932966709136963, -2.2378437519073486, -5.322968482971191, 0.4842301607131958, 2.2774548530578613, 4.698623180389404, 5.618356704711914, 1.895561933517456, -6.3994903564453125, 0.7716164588928223, -3.1721320152282715, -3.218134641647339, 4.102287769317627, -1.746441125869751, 1.871933937072754, 0.06166073679924011, -3.6147830486297607, 4.681332588195801, 0.631746768951416, 0.06332255154848099, -0.6600582003593445, 0.026969123631715775, -7.623932361602783, -1.341784954071045, -3.4049365520477295, -9.947603225708008, 3.2602827548980713, 0.7546817660331726, 0.6626856923103333, -5.501628875732422, 1.5522032976150513, 0.8331151604652405, 3.6763417720794678, 6.6681294441223145, -3.020625591278076, 1.9404488801956177, -4.572497844696045, 1.6972911357879639, 0.7493177056312561, 3.362504005432129, -0.9385455250740051, 7.571264266967773, 2.605790138244629, -3.2899603843688965, -2.505394220352173, 1.3593854904174805, 2.4163126945495605, -5.095540523529053, -3.0553722381591797, 2.977264642715454, -5.437698841094971, -4.251937389373779, 0.7108485698699951, -0.2459535151720047, 5.205434799194336, 0.5799692869186401, -8.209927558898926, -3.658163547515869, -2.3642473220825195, -2.0414023399353027, -3.331494092941284, -3.2989418506622314, 0.6102532148361206, 5.143265724182129, -9.607592582702637, 7.235158920288086, 3.235928773880005, -4.360565662384033, -0.7096775770187378, -3.3043630123138428, 2.0840206146240234, -3.0757381916046143, -5.335222244262695, -7.43290901184082, 6.590363025665283, 9.551971435546875, -5.380490303039551, 1.1203458309173584, -2.058960437774658, 1.7636016607284546, 3.1310813426971436, 0.37356820702552795, -0.42811304330825806, 0.5435039401054382, 1.1230359077453613, 3.4618194103240967, 1.111154317855835, -1.4214463233947754, -4.880874156951904, 2.4812591075897217, 4.9194655418396, -6.008109092712402, 2.1194651126861572, 5.6094183921813965, -1.9480037689208984, 1.5223902463912964, -3.007826089859009, 1.2360193729400635, 4.606522560119629, -0.24990065395832062, -4.620708465576172, 0.23554660379886627, -5.414559364318848, 0.5188422799110413, 3.9325644969940186, 0.23625431954860687, -2.309732675552368, 1.077283263206482, -1.9936778545379639, -0.7679108381271362, -6.453717231750488, -2.431239366531372, 13.344497680664062, 3.569990873336792, -5.56859827041626, 1.111067533493042, -4.250611782073975, -1.7538517713546753, 0.7509289979934692, 1.3233802318572998, -2.6275248527526855, -4.569212913513184, -1.5055698156356812, -4.740199089050293, 2.5415191650390625, 0.9528040289878845, -1.0669461488723755, -2.296328067779541, 2.037795066833496, -5.120321750640869, 8.328838348388672, -0.26322561502456665, 0.6827778220176697, 0.4081137776374817, 3.6301157474517822, -1.9775049686431885, 2.2944250106811523, 1.3039034605026245, 3.661499261856079, -0.8673211336135864, 6.782901287078857, 0.17023779451847076, 3.5222671031951904, -1.7748206853866577, 0.5767679810523987, 4.104593753814697, -1.7326985597610474, -2.89321231842041, -0.5792690515518188, 3.570502519607544, 2.788402557373047, 5.006831645965576, 7.4312357902526855, -8.518152236938477, 4.581249713897705, 0.15944895148277283, 1.264741063117981, 0.22111834585666656, -3.991126298904419, -2.5377745628356934, 1.1475329399108887, -3.3261165618896484, 0.3773835599422455, 0.015315079130232334, 3.552140712738037, -3.4258310794830322, 0.45655393600463867, -0.30430251359939575, -1.8162099123001099, 5.892564296722412, -2.741422176361084, -0.22551387548446655, -0.32066231966018677, 0.7254282832145691, 5.002148151397705, 6.043678283691406, -2.8266677856445312, 5.117714881896973, 1.2034657001495361, -6.79599142074585, 3.6738498210906982, -1.2526410818099976, 5.252862930297852, -2.661292552947998, 4.021778106689453, 6.198101997375488, 6.444753646850586, 1.4861162900924683, -0.7262740135192871, -6.013160228729248, 1.4404209852218628, 2.2075650691986084, 2.6528525352478027, -1.7577953338623047, -0.26078036427497864, 8.793278694152832, 0.7306241393089294, 9.28000259399414, 4.535737991333008, 1.967281699180603, 1.5178600549697876, 0.5332925915718079, 3.0804708003997803, -0.1462884247303009, -0.19115692377090454, -1.5436891317367554, 2.6276724338531494, -9.554437637329102, -4.4858927726745605, 2.7648870944976807, 6.456412315368652, 0.700471818447113, -2.3359930515289307, -0.8341779112815857, -1.5248281955718994, 3.2220230102539062, -1.8395429849624634, -7.337327480316162, -0.7719336748123169, -2.7971043586730957, -1.296578288078308, 2.855637550354004, 1.7694381475448608, -1.9181612730026245, -5.248688697814941, -0.5978777408599854, 1.8311774730682373, -0.8747758269309998, -0.9471738338470459, -0.037086550146341324, -4.005558490753174, 0.6222702860832214, -1.3359959125518799, -7.815552234649658, -3.3888096809387207, -8.161759376525879, -3.4984476566314697, -3.7265748977661133, -2.351769208908081, -2.8677735328674316, -2.4038245677948, 2.389763355255127, 4.097215175628662, 4.260700702667236, -3.564826011657715, -1.444882869720459, 2.7262210845947266, 7.046666622161865, 0.01901905983686447, 6.528641700744629, 5.790400981903076, -1.6144284009933472, -2.497450828552246, -1.1015963554382324, -2.3573403358459473, 0.3141310214996338, 3.7995495796203613, 3.2888424396514893, 2.142256259918213, 2.393812656402588, 1.171852946281433, 4.410018444061279, 4.379431247711182, 2.275237560272217, 1.5293048620224, -2.362067937850952, 4.220844268798828, 4.174101829528809, 5.638635635375977, -0.40630802512168884, 3.0529139041900635, -2.568209409713745, 0.19221048057079315, -0.5759451985359192, -11.002280235290527, 3.3082635402679443, 4.398388862609863, 1.8399728536605835, -4.683109760284424, -9.924239158630371, 6.115902423858643, -1.1782933473587036, -1.788630485534668, 7.335194110870361, -0.6610979437828064, 0.9245668053627014, 3.3844990730285645, -0.8915926814079285, 4.895991325378418, 1.2740370035171509, -0.8448992371559143, 5.66909122467041, 2.4898507595062256, -6.144494533538818, 3.8218939304351807, -1.4961107969284058, -1.1522259712219238, 5.681329250335693, -5.419457912445068, -2.200164556503296, 4.077986240386963, 2.05759859085083, 6.017639636993408, -6.3675761222839355, 4.489130973815918, -0.37858709692955017, -2.8872599601745605, -2.438899517059326, 0.8663235306739807, -0.5129411220550537, -2.348050832748413, 1.761457920074463, -4.398797035217285, 1.7591843605041504, -7.152069091796875, 4.1055588722229, -1.7958803176879883, 2.615325450897217, -1.7223267555236816, 0.7296426892280579, 0.4878964424133301, -2.623976707458496, 2.6305909156799316, 8.540733337402344, -1.1355623006820679, -6.765233516693115, 7.414343357086182, 3.1603660583496094, -4.42905330657959, 4.084150791168213, 3.015850782394409, -1.0592674016952515, -0.44695237278938293, 0.27020567655563354, 0.7666457891464233, 2.5010826587677, 3.364793539047241, 2.2142858505249023, -4.131373405456543, -2.2814345359802246, 7.138725280761719, -1.4524660110473633, 2.1147332191467285, 3.6582155227661133, -1.0098199844360352, -2.712803363800049, -3.801813840866089, 1.5739097595214844, 4.448587894439697, 3.858759880065918, -0.3945634067058563, 1.635137915611267, 2.562575101852417, 5.973583221435547, 4.994359970092773, -1.07523512840271, 1.290026307106018, -10.82999038696289, 2.068467140197754, -9.460705757141113, 9.818265914916992, 0.633099377155304, -1.8235198259353638, 6.448036193847656, -0.7923789024353027, 4.040419578552246, 8.069906234741211, -11.809252738952637, -4.54115629196167, -4.823581218719482, 1.7293684482574463, 1.4882811307907104, -8.135749816894531, 1.0336250066757202, 1.2167432308197021, -0.9852476716041565, -0.20371095836162567, -1.7284845113754272, 0.09492817521095276, 0.34242409467697144, 1.1428358554840088, -1.054052710533142, 0.5199627876281738, -4.328185558319092, 1.7781976461410522, 4.901206970214844, -0.7753511667251587, -4.511044025421143, -1.5683767795562744, -4.983877182006836, -9.953170776367188, 3.348271131515503, 0.2587721049785614, 4.980926513671875, 2.7932424545288086, 4.021516799926758, -2.910343647003174, 1.6128976345062256, 1.1738390922546387, -0.14118316769599915, 0.2989822030067444, 1.73476243019104, 3.061965227127075, -5.420764923095703, 2.815239429473877, -2.0987133979797363, 1.2623323202133179, 4.803247928619385, -5.438216686248779, -5.180339336395264, 3.5680017471313477, -0.3856342136859894, 4.624582767486572, 1.2462856769561768, -3.427035093307495, -4.9728522300720215, 3.6365137100219727, 4.3573760986328125, -1.1140238046646118, -2.184638738632202, 3.293635368347168, -0.5901519656181335, 4.626108169555664, 2.308744192123413, -3.2793569564819336, -3.64723539352417, 0.10898464173078537, -5.96928071975708, -0.10258764773607254, 2.8169405460357666, -2.9509027004241943, 2.937025308609009, 1.1042487621307373, 0.03961505368351936, 4.048752784729004, -0.02441748045384884, 2.634129524230957, -3.2119827270507812, 5.607786178588867, -3.4198315143585205, -0.07702013105154037, -0.788755476474762, 7.962124824523926, 1.2162035703659058, -0.24325819313526154, 11.67383098602295, -2.228389263153076, -2.0328714847564697, 0.3074587881565094, -0.3206871449947357, -4.668708324432373, 6.147686004638672, 1.4691131114959717, 1.0556305646896362, -2.5103402137756348, 3.8446202278137207, 2.0281474590301514, 4.740992546081543, 1.5752695798873901, 3.7182202339172363, -2.4667422771453857, -4.3361358642578125, 4.805559158325195, -3.9402382373809814, 0.6882405877113342, -0.606889009475708, -6.148444175720215, 1.2588236331939697, -5.606506824493408, 5.169280529022217, 0.17944370210170746, 2.2352898120880127, -2.738804817199707, -0.9273915886878967, -9.865483283996582, 0.9376658797264099, 8.424686431884766, 2.1202967166900635, 2.4051172733306885, 2.3600382804870605, 4.1923604011535645, 3.958094358444214, -0.6140426993370056, 2.537442922592163, -0.8395732045173645, -0.6039071679115295, 7.7957563400268555, 1.408468246459961, -4.294949054718018, 5.913198947906494, -0.9975623488426208, -3.9838013648986816, -1.5389684438705444, 2.1516666412353516, -2.461667060852051, 2.4330506324768066, -0.15149050951004028, 1.6706390380859375, -1.7891967296600342, 2.4669439792633057, -5.0942792892456055, 7.5059356689453125, 3.295593500137329, -4.752513408660889, -2.4779064655303955, 7.272810935974121, -3.4156014919281006, -2.7584567070007324, 4.977059841156006, -0.43889319896698, -2.6104631423950195, 2.6842947006225586, -4.716468811035156, 3.484466791152954, -3.3312039375305176, 1.4519407749176025, -4.439187049865723, -3.8986546993255615, -2.659012794494629, 1.035297155380249, 0.027599893510341644, -10.617147445678711, -0.5664758086204529, 4.440251350402832, 0.7259777188301086, -2.9947447776794434, 4.5488481521606445, -3.1695780754089355, 1.9966094493865967, 6.549100875854492, -3.397195339202881, -7.559327602386475, 5.697739601135254, 1.799661636352539, 6.673712730407715, 2.5645406246185303, -1.1396386623382568, -2.443154811859131, 1.0427418947219849, 1.549289584159851, 6.973174571990967, 0.7546586394309998, 3.061541795730591, -1.0825681686401367, -5.199673652648926, -5.074838161468506, -3.187117099761963, 2.4164657592773438, 2.7726688385009766, -0.7374303340911865, -1.249078392982483, 7.648077964782715, -2.0500621795654297, -0.48335614800453186, -2.5515189170837402, 0.5341742038726807, 1.3718475103378296, 1.7468255758285522, 9.635129928588867, 2.1601574420928955, 5.146854877471924, -3.8537228107452393, 9.439705848693848, -3.427966594696045, 4.980741024017334, -2.719775676727295, 2.3901166915893555, -9.401845932006836, -1.4536429643630981, -6.180088520050049, 8.131160736083984, 0.5231993794441223, -1.9996556043624878, -0.37753012776374817, 1.9117505550384521, -3.330357313156128, 0.148013636469841, 0.47979703545570374, 0.948742151260376, 4.115829944610596, 0.8004859089851379, 4.466015815734863, 2.1230998039245605, 0.3995095491409302, 1.930067777633667, 4.717733860015869, -3.456246852874756, 1.3350892066955566, -0.5036409497261047, -3.5515666007995605, -2.5342016220092773, 0.8446288704872131, -0.5898644924163818, -1.2539479732513428, 3.5447452068328857, -3.1188292503356934, 2.408965587615967, -1.80194890499115, -4.078271389007568, 5.109155654907227, 0.2291291505098343, -2.2684340476989746, 1.1387927532196045, 4.664502143859863, 8.163830757141113, 2.952843427658081, 2.0486042499542236, 5.544877529144287, -2.883486032485962, 5.830504894256592, -4.337543964385986, 3.4854838848114014, 0.8085317015647888, -6.237563610076904, -1.3111274242401123, 6.983654975891113, -1.7742478847503662, 1.9184675216674805, 3.0127134323120117, -0.11417757719755173, 2.8787615299224854, 0.5448973774909973, 10.792524337768555, 5.857152938842773, -5.854300498962402, -1.811281681060791, 0.39150485396385193, -3.7948811054229736, 3.1184020042419434, -8.218001365661621, 4.45548152923584, 0.07604828476905823, -6.338414669036865, 1.913705825805664, 2.247497320175171, -6.284869194030762, 5.2866291999816895, -3.4515225887298584, 4.8687744140625, 4.619934558868408, 5.520453929901123, -2.6809372901916504, -5.179112434387207, 1.0902000665664673, -2.725166082382202, -4.821954250335693, 4.399900913238525, 6.555269718170166, -1.500901222229004, -5.062826156616211, -3.669185161590576, -5.227805137634277, -4.790604114532471, 3.865107774734497, -0.5381702780723572, -0.559779942035675, -3.065744161605835, -4.110890865325928, -2.7909815311431885, 2.8912246227264404, -2.9171228408813477, 4.302738189697266, 1.6609907150268555, 5.001265048980713, 2.3913776874542236, 1.7597148418426514, 11.833993911743164, 1.1878129243850708, 6.352297306060791, -2.571953535079956, 4.773012161254883, 0.976477861404419, -7.400980472564697, 0.038799088448286057, -3.728071451187134, -5.382633686065674, 0.4646410048007965, -1.5118036270141602, -0.19684913754463196, -1.6070222854614258, 0.13145160675048828, 3.904568672180176, -6.585187911987305, -3.4859678745269775, -3.6207644939422607, -5.448285102844238, 5.1907477378845215, -2.7591466903686523, -2.1969611644744873, 0.7954885959625244, -1.3949627876281738, -1.253875970840454, -1.688737392425537, 0.676642119884491, -0.06933032721281052, 1.5390669107437134, -3.4744608402252197, 4.017694473266602, -4.5888190269470215, -2.5333292484283447, 8.167953491210938, 5.833332538604736, -2.567128896713257, 0.6853596568107605, -1.4373582601547241, -0.6360330581665039, 8.239909172058105, -4.5725226402282715, -5.43401575088501, -0.6273406147956848, 2.424088478088379, -3.3579261302948, -0.12320636957883835, 0.2692566215991974, -6.039170265197754, -1.25223708152771, 1.9873679876327515, -0.06816421449184418, 11.985076904296875, 1.883887529373169, -1.1229243278503418, 5.016129493713379, -2.878964424133301, 0.624484658241272, 4.480905055999756, -1.5090581178665161, -4.804224967956543, 1.3530303239822388, -3.6725969314575195, -3.170680522918701, 1.5646021366119385, -2.279824733734131, 5.616921901702881, 0.7365607023239136, 1.9918800592422485, 6.600208759307861, -1.3164387941360474, 0.9848957657814026, -1.6399924755096436, -3.4414119720458984, 0.41079509258270264, -4.871209144592285, -4.0697021484375, 0.23911897838115692, -2.265467405319214, -1.6448071002960205, 2.2456982135772705, -8.303082466125488, 4.159125328063965, -1.4960460662841797, -1.6324975490570068, 10.128626823425293, -0.35377463698387146, 1.9517804384231567, -1.3444145917892456, 3.7888519763946533, 3.6814067363739014, 3.895390272140503, -3.323721170425415, 0.9811724424362183, 1.8952380418777466, -6.229525566101074, 3.569179058074951, 4.416162490844727, 0.3489396870136261, 0.8569595217704773, 9.730413436889648, -2.256303071975708, -4.225546360015869, -0.9643189311027527, -7.278341770172119, 3.3197500705718994, 2.6061620712280273, 3.0445804595947266, 10.315752029418945, -3.7598514556884766, 2.9087088108062744, 0.27317574620246887, -1.1002613306045532, 4.485308647155762, -11.257912635803223, 5.303991317749023, 4.3718791007995605, -1.2481483221054077, -7.779058456420898, 2.105375051498413, 2.0564112663269043, 3.289436101913452, 0.8298215866088867, -2.404980182647705, 2.6571364402770996, 2.013406276702881, -0.023259136825799942, -2.565108299255371, -1.7079565525054932, -3.0927000045776367, 1.7986770868301392, -0.5361630916595459, -4.503811836242676, 3.121295690536499, -0.4677712023258209, -1.639267086982727, -0.6397382020950317, -6.422754287719727, 1.7845414876937866, 1.7412490844726562, 6.720979690551758, 6.579208850860596, -5.061673164367676, 0.5277314782142639, -2.786515235900879, 4.157289981842041, 4.780968189239502, -2.4681570529937744, 3.714160442352295, -2.528717279434204, -7.137304306030273, -2.084794521331787, 0.6153706312179565, 3.050144910812378, -0.8873187899589539, 4.071995735168457, -2.747365713119507, -0.1583186388015747, 0.7916345000267029, 2.209412097930908, -0.7085537910461426, 4.769559860229492, 3.8187942504882812, 1.2721530199050903, -8.949912071228027, -0.6879557967185974, -4.420053482055664, 0.08440542221069336, -3.7811572551727295, -1.4845671653747559, 2.683582305908203, -2.8115227222442627, 1.1031429767608643, 6.516590595245361, 6.642096996307373, 4.931684494018555, -10.4376220703125, -2.6156985759735107, -2.8158645629882812, -1.8417896032333374, -0.37333619594573975, -1.2179135084152222, -4.6595635414123535, -8.066121101379395, 2.2050511837005615, -5.0300750732421875, 3.737375497817993, 1.2746738195419312, -1.587532639503479, -1.608765721321106, -1.4135751724243164, -2.454918622970581, 6.736624717712402, -1.9963078498840332, -1.7071017026901245, -1.6277934312820435, 1.227160096168518, 3.0685832500457764, 0.6492024064064026, 0.8741956949234009, 0.05806233733892441, -4.517494201660156, 1.7467222213745117, -1.2692127227783203, -2.471367597579956, -3.1542887687683105, 1.1175247430801392, 6.806571960449219, 4.97179651260376, -3.17694354057312, 2.0364203453063965, 3.507861375808716, -3.764178514480591, -2.5247812271118164, 1.5043987035751343, -3.091911792755127, 3.6773102283477783, -7.541584491729736, -3.013580083847046, -1.0240641832351685, 0.20489172637462616, 0.11183437705039978, -1.28466796875, -0.593770444393158, -5.501982688903809, -6.738165855407715, -2.2639055252075195, -4.541652679443359, -4.177960395812988, 9.187417030334473, -1.9337828159332275, -2.1246178150177, 0.07882678508758545, 3.4601566791534424, -0.8146457672119141, -0.6048451662063599, -1.5647329092025757, 0.05380230024456978, -3.117056131362915, -8.240869522094727, -3.1582279205322266, 1.4311273097991943, -7.083759307861328, -7.7301201820373535, -2.460944652557373, 3.3023107051849365, 5.921494483947754, 2.7051539421081543, 3.7317070960998535, 3.313922643661499, 1.130636215209961, 2.0325968265533447, 1.408274531364441, 0.036615341901779175, -6.866650581359863, -1.1260604858398438, -7.815325736999512, -0.1623723953962326, 2.2243542671203613, 1.0912528038024902, -0.3964903950691223, -2.7136898040771484, -5.866586685180664, 0.5758042335510254, -1.6402602195739746, -2.801689624786377, -4.63624906539917, 4.566791534423828, 1.6163374185562134, -3.7433533668518066, 4.164488315582275, -0.1551506370306015, -0.7892998456954956, -2.5192463397979736, 4.898558616638184, 4.373327732086182, 2.1163625717163086, 5.113631725311279, -1.1017603874206543, 2.4511666297912598, -3.6674575805664062, -3.5842339992523193, -2.5014209747314453, 5.436369895935059, 2.110652446746826, 2.4564785957336426, 3.7096683979034424, -0.23119774460792542, -3.420334815979004, -5.657466888427734, -5.0402140617370605, -1.9378968477249146, -0.9418075084686279, -4.269285202026367, -1.2921478748321533, -2.1636152267456055, 4.353931427001953, -0.7593770027160645, 4.691895484924316, -1.271487832069397, -2.4484245777130127, -2.489194631576538, -2.0857107639312744, -1.359156608581543, -1.0861247777938843, 2.646209955215454, 3.3884506225585938, 4.4753570556640625, 5.443004131317139, -1.7685375213623047, 0.23291265964508057, 0.0682881623506546, -3.072809934616089, -2.3887429237365723, 2.955099105834961, 4.046247482299805, -1.2171120643615723, 2.5734004974365234, 2.3947393894195557, -6.01833438873291, -4.069480895996094, -3.9805800914764404, -0.9606695175170898, 3.4226691722869873, 3.289008378982544, 3.7238354682922363, 5.957513332366943, 0.04945724457502365, -2.387110710144043, 2.6597373485565186, 1.212968349456787, -2.5673139095306396, 1.6637187004089355, 0.17250937223434448, -3.6354517936706543, 3.6785168647766113, -1.8286993503570557, 8.327596664428711, 0.528328001499176, 8.549819946289062, -2.866406202316284, -3.321826934814453, -1.3502721786499023, -0.9273405075073242, -1.0147223472595215, 1.785112738609314, -6.516103744506836, -0.43832671642303467, -0.9256888628005981, -4.668888092041016, -6.581782817840576, -3.2458717823028564, 2.3917107582092285, -1.214928150177002, -1.1729038953781128, 2.84370493888855, 4.006377696990967, 5.720691204071045, 2.3672680854797363, 1.2752745151519775, 1.6438288688659668, -0.4835048317909241, 2.4012205600738525, 4.70888090133667, 6.015044212341309, -2.484964609146118, -0.5054908990859985, -1.364937663078308, -5.317227840423584, 2.5534887313842773, 4.138842582702637, -0.15968868136405945, -4.02918815612793, 0.30060330033302307, 0.6687596440315247, -4.239698886871338, -1.1041814088821411, 0.1797107458114624, 1.7316265106201172, -5.862939834594727, -1.3002005815505981, -0.636064887046814, 2.9637033939361572, 0.985005795955658, 0.9742782115936279, 4.356374740600586, 7.913363933563232, 0.5430554747581482, 2.0983986854553223, -2.0862438678741455, -4.492230415344238, 4.58513879776001, 3.7210946083068848, 1.867406964302063, -0.8975257277488708, 2.0916907787323, 1.4155523777008057, 5.041417598724365, 0.1750394105911255, 0.9931824803352356, -2.5695981979370117, 2.627789258956909, 0.9667026400566101, 0.15250851213932037, -1.0317572355270386, 2.172661304473877, 4.086708068847656, -0.1824093759059906, 2.1543452739715576, 3.4096462726593018, 12.57344913482666, -1.0355936288833618, -5.737342834472656, -5.4502973556518555, 1.9711576700210571, -2.5312297344207764, -4.358642578125, 4.691364288330078, 0.151695117354393, 0.5662667155265808, -3.0947883129119873, 2.463980197906494, 7.53424072265625, 1.9396233558654785, -0.1867501586675644, -2.893232583999634, -2.498140573501587, -5.215712547302246, -3.4651196002960205, -1.4715691804885864, 4.719051837921143, -5.281475067138672, -0.9179823994636536, 0.5243252515792847, -0.2679561674594879, 2.6550283432006836, -0.5200847387313843, 4.149025917053223, -0.07464583963155746, 0.7027615904808044, -2.3167543411254883, -7.266899108886719, -0.9725466370582581, 5.571347713470459, 0.8662509918212891, 4.346497535705566, 2.635128974914551, -1.6982578039169312, 7.7532172203063965, 2.4761431217193604, -2.57071852684021, 3.4580726623535156, 0.0435921810567379, 2.6076436042785645, 3.9149365425109863, 2.119701623916626, 5.505668640136719, 1.594534158706665, -5.168620586395264, 1.0731271505355835, -1.9395085573196411, -2.9318618774414062, 2.3738720417022705, 1.1062768697738647, -1.8560305833816528, -3.949429750442505, -3.7680602073669434, 0.8694358468055725, -4.326231002807617, -3.296130418777466, -0.7293424010276794, 0.37478816509246826, -2.496105194091797, 4.74086332321167, 2.445037841796875, -0.14027327299118042, 1.0046582221984863, -1.1451804637908936, -7.0067009925842285, -5.5755205154418945, 3.587595224380493, 1.3464184999465942, 6.2019429206848145, 1.2343703508377075, 1.9075547456741333, 9.290166854858398, 1.2590136528015137, 0.9358172416687012, -2.1401031017303467, 1.680452823638916, 0.35663902759552, -5.961411476135254, 0.8285502195358276, -6.453078746795654, 5.871321678161621, -3.5815773010253906, 6.564764499664307, -2.645627498626709, -0.7780315279960632, -1.264322280883789, 1.3402793407440186, -1.6065903902053833, 1.6621794700622559, -3.5568933486938477, -0.3264053761959076, 1.5133939981460571, -3.992954730987549, 0.42366543412208557, 5.713160991668701, 1.6052829027175903, 0.3727024495601654, 2.9405412673950195, 1.3991212844848633, 0.6411596536636353, -4.987215995788574, 3.5309247970581055, 2.489318609237671, 1.4828814268112183, 10.451555252075195, 0.5494842529296875, -2.065603256225586, -1.794575572013855, 3.590336561203003, 4.299654006958008, -2.011051654815674, -7.989314079284668, 0.0762096717953682, 11.54179573059082, 7.850345134735107, 4.047591209411621, 4.041463375091553, -1.2366188764572144, 3.6184799671173096, -7.299182415008545, -1.4756327867507935, -0.5953258275985718, -11.385821342468262, -9.211404800415039, -4.372994899749756, -0.9087188839912415, -3.856109380722046, 1.4181382656097412, -1.7910157442092896, 2.912729024887085, 2.210993766784668, 3.9149856567382812, -1.8208937644958496, -1.0646402835845947, -2.7128682136535645, 1.1582306623458862, -3.424680233001709, -5.570173740386963, -3.3548295497894287, 0.933021605014801, 0.31833064556121826, 4.0551323890686035, -3.7433998584747314, 1.5892901420593262, 1.0736830234527588, -2.58602237701416, -2.2509336471557617, -0.3607111871242523, 2.6259500980377197, -2.077303409576416, -1.8063772916793823, 0.5192558765411377, -2.0665109157562256, -2.930265426635742, -4.668209075927734, 2.7555558681488037, -1.4528536796569824, 4.274331092834473, -1.093360424041748, -6.898499488830566, -6.026835918426514, 4.599808692932129, 3.3763296604156494, -4.481906414031982, -2.3900649547576904, -4.220810890197754, 5.509594440460205, 1.3865727186203003, 0.8896503448486328, 4.942854404449463, 1.6535035371780396, -1.067487359046936, -3.479466199874878, 2.944340705871582, -4.369993209838867, 9.380302429199219, 3.955472946166992, -0.2715708613395691, -1.3913222551345825, -4.042335510253906, -3.77311110496521, -7.9805097579956055, 4.821120262145996, 4.656015872955322, -0.2992871105670929, -2.9748456478118896, -5.052896022796631, 2.182504653930664, 3.391145944595337, 0.8943765759468079, 4.738028526306152, -1.3881733417510986, 4.549842357635498, -6.646263599395752, -1.455675721168518, 4.942747116088867, -0.31774649024009705, 3.7351460456848145, 0.0041290028020739555, -3.2592344284057617, -0.07849172502756119, -0.04641552269458771, 0.41398587822914124, -0.5275062322616577, 2.9795150756835938, -4.381175518035889, 0.6070955395698547, 2.4828901290893555, 6.663392066955566, -2.7762839794158936, -7.360020160675049, 4.87179708480835, -1.8552839756011963, 4.581404209136963, -3.333688497543335, 1.449215054512024, 1.1637980937957764, 5.000069618225098, 8.61984920501709, 0.9053061604499817, 1.0716066360473633, -0.33067768812179565, -5.570522308349609, -1.2840626239776611, -3.7540125846862793, 2.427075147628784, 5.023011207580566, 4.469087600708008, 5.329339027404785, 1.77555513381958, -0.24429672956466675, 1.732527256011963, -1.202065110206604, 0.842907726764679, -2.9101662635803223, -2.8062427043914795, 4.372280597686768, -1.3735857009887695, 0.6754000782966614, 0.5721951723098755, -1.8519734144210815, 3.8977091312408447, -2.01324200630188, -3.434936046600342, -7.017726421356201, 0.19694136083126068, 3.9098238945007324, -1.4076473712921143, -3.4238908290863037, 8.659551620483398, 4.4310126304626465, 3.8714756965637207, -4.161233425140381, -4.161624431610107, 4.573659896850586, 0.08576246351003647, -2.072036027908325, -2.9214649200439453, 0.0017077919328585267, 1.7413328886032104, 3.4922831058502197, -7.960124492645264, -3.3637678623199463, -0.8097176551818848, 1.6857845783233643, -6.982350826263428, 2.5383286476135254, -1.2663977146148682, -4.377884864807129, 4.756720066070557, 1.4372613430023193, -5.553700923919678, 2.0469446182250977, -4.84169340133667, -7.610934257507324, 1.5357424020767212, -3.31120228767395, 0.47715330123901367, -5.767539978027344, 1.9895836114883423, 2.5460047721862793, -7.647481918334961, 1.891412615776062, -3.2634117603302, -4.583089351654053, 0.6720004677772522, 1.071447730064392, -1.7528964281082153, -1.1216939687728882, 6.2640299797058105, 0.002695864997804165, 4.048658847808838, 4.048424243927002, -0.499891072511673, -1.7644073963165283, 2.2168288230895996, 1.025244116783142, 1.0583391189575195, 6.441064357757568, -2.633059024810791, -1.9933457374572754, 6.631217002868652, 0.399109810590744, 6.067688941955566, 2.161817789077759, 8.165904998779297, 3.9792110919952393, -0.580065906047821, -2.37886643409729, -3.5482070446014404, 4.223912715911865, -2.683077812194824, 2.3938591480255127, -2.0609123706817627, 2.540003776550293, -5.611589431762695, 2.2452147006988525, -5.445370197296143, -2.6655185222625732, 2.008970022201538, 4.9747161865234375, 4.644665241241455, -1.9897150993347168, 1.7542551755905151, -6.997623920440674, 2.892228603363037, 6.772154331207275, 1.0055838823318481, 3.477156400680542, 7.654871940612793, -2.12384295463562, 0.43460023403167725, 5.9196624755859375, 6.523628234863281, -8.14908218383789, 1.5886008739471436, -0.6245442032814026, -2.104931592941284, 3.8048272132873535, -5.5820770263671875, 3.518059015274048, 0.5590248703956604, 2.4034292697906494, 1.2597777843475342, 7.690920829772949, 4.153305530548096, 2.1441292762756348, -1.001198649406433, 4.719455242156982, 3.378274440765381, -5.835104942321777, 1.2897735834121704, -2.6646499633789062, 1.6824028491973877, -3.542840003967285, -4.428205490112305, 3.2122697830200195, 2.537916421890259, 0.622115969657898, -1.5266929864883423, -1.9229655265808105, -4.602848052978516, -0.21372966468334198, -1.113237977027893, -2.306840658187866, -1.7354015111923218, 9.345521926879883, 0.1866866797208786, 2.9452250003814697, -1.484151840209961, -5.3591461181640625, -5.478240489959717, 1.9646999835968018, -4.457594871520996, 0.5786588788032532, -7.358291149139404, 2.9246532917022705, -4.8727006912231445, 6.69101619720459, 4.448908805847168, -7.9174299240112305, 0.9146267771720886, -2.080235242843628, 2.1096599102020264, 4.36945915222168, 4.536357402801514, 8.572991371154785, 0.7961620092391968, -2.441628932952881, 5.894709587097168, -3.584787368774414, -1.0236448049545288, -0.7556185722351074, 0.47700759768486023, 2.019016981124878, -3.218212604522705, 2.5233635902404785, 6.837704658508301, 3.3069803714752197, -0.10776765644550323, 6.790495872497559, -1.3783661127090454, -0.9156754612922668, 3.341024160385132, 3.1735124588012695, 2.2286393642425537, -0.06570044159889221, -3.3393936157226562, -2.209280252456665, -1.0505303144454956, -2.078958749771118, -4.862795829772949, -1.443967342376709, -9.706315994262695, 1.2128299474716187, 8.634204864501953, -2.979494571685791, -6.0838398933410645, -0.6647745966911316, 2.93764066696167, -5.828458786010742, 0.9189414381980896, 7.592700004577637, -5.611225128173828, -1.718384861946106, 2.0355424880981445, -5.31088924407959, -0.03619980812072754, 5.412291526794434, 0.03694145753979683, 1.1168116331100464, -2.0876541137695312, 7.221043586730957, 7.1617255210876465, -4.6333160400390625, -6.611592769622803, -1.7438989877700806, -4.575526237487793, 4.657012939453125, 1.7203015089035034, -0.775445282459259, 3.3117170333862305, -8.23235034942627, -5.115141868591309, -3.920691967010498, 3.9134857654571533, 9.174201965332031, -0.5278289318084717, 4.702035427093506, 4.1021504402160645, 1.1181637048721313, 3.8456077575683594, 2.3250739574432373, 4.145468711853027, 2.5118818283081055, -4.008785724639893, 4.491809368133545, 0.14596255123615265, 4.979278087615967, 2.8241002559661865, -3.577648639678955, 8.524392127990723, 5.14605712890625, 1.9261173009872437, 12.529074668884277, 0.6837427616119385, -6.598036289215088, -4.8647871017456055, -1.6598743200302124, -1.784463882446289, 5.891591548919678, 3.2561237812042236, -0.9475657939910889, 5.03308629989624, 0.3342941403388977, -4.801060676574707, -3.887868642807007, 0.9356286525726318, 1.3610687255859375, -2.498911142349243, 2.136200428009033, 6.0325026512146, 1.6080619096755981, -0.6093912124633789, 4.473967552185059, -1.2898218631744385, 3.841583490371704, 3.41241455078125, -2.130366086959839, -1.2451915740966797, 2.8695473670959473, 1.8760597705841064, -3.7640790939331055, -3.1071460247039795, -1.9861273765563965, -0.5590274930000305, 1.232848048210144, 4.3035173416137695, -1.9919953346252441, 3.9989991188049316, 2.0125205516815186, -4.411407947540283, 5.1081109046936035, -2.415123701095581, -2.443176031112671, 1.4380381107330322, 1.6296412944793701, 4.608897686004639, -1.029098391532898, 1.5331623554229736, -1.0281630754470825, 0.6158255934715271, 11.457114219665527, 4.553156852722168, -4.086159706115723, 7.422923564910889, 0.97892165184021, -0.4578232765197754, -3.0193674564361572, 1.3997483253479004, 3.575545310974121, 6.149859428405762, 4.012585163116455, 0.6450254321098328, -5.294733047485352, -6.502798080444336, -1.9068297147750854, -1.618297815322876, 3.009915590286255, 0.6224789023399353, 1.5420385599136353, 4.562042236328125, 0.7131125926971436, -1.8463956117630005, -3.062268018722534, -5.923439025878906, -1.081587553024292, -0.22726790606975555, -6.842823028564453, 3.930194616317749, -1.036505103111267, 1.1488093137741089, -1.0366992950439453, -0.4154936969280243, 0.8542101383209229, -8.336344718933105, 3.2820332050323486, 4.913816452026367, 3.813429117202759, -6.798467636108398, -0.18499276041984558, -5.512316703796387, 3.9446163177490234, -0.45253705978393555, -6.684789180755615, 3.9428844451904297, -5.789239883422852, -3.523406982421875, 2.3228535652160645, 2.2032485008239746, 0.3547726571559906, -1.6716771125793457, -1.3222724199295044, 1.0581830739974976, 7.23809814453125, 0.018732888624072075, -4.055803298950195, 2.3626248836517334, 0.4286806285381317, 2.543527841567993, -5.045956611633301, -4.16410493850708, 1.54008948802948, -1.192138433456421, 3.022459030151367, -3.27571964263916, -2.0693657398223877, 2.4078524112701416, -10.024496078491211, -3.5948290824890137, 0.9068559408187866, 3.67590594291687, 0.13756734132766724, 6.428742408752441, -4.063574314117432, 0.11653073877096176, 0.6423245072364807, -0.16485385596752167, -0.673416793346405, -3.1106626987457275, 0.721156895160675, -5.410656452178955, 2.731827974319458, 0.15250523388385773, -4.2430572509765625, -0.45770731568336487, -1.2420741319656372, -5.640966892242432, 2.0905449390411377, -4.168120861053467, 0.630817174911499, -3.370617389678955, -4.040623664855957, 0.5969879627227783, 3.1919288635253906, -3.1565401554107666, -2.629077196121216, 4.81649923324585, 3.9964587688446045, 1.0421428680419922, 0.9754729866981506, -10.694389343261719, -4.257004261016846, -5.284170150756836, -4.251167297363281, 0.8063727021217346, 1.4049183130264282, -0.030543353408575058, 2.9060471057891846, 0.24717499315738678, 6.3285746574401855, 5.037240505218506, 8.614679336547852, -1.00652277469635, -8.890362739562988, 8.457345008850098, 4.543458938598633, -3.6590278148651123, 3.834296941757202, 2.9758834838867188, -6.409985542297363, -6.489977836608887, 2.0187363624572754, 3.4908454418182373, 3.5062434673309326, -5.088626384735107, -0.9395372867584229, -3.0531344413757324, -3.255519390106201, 6.346038818359375, 1.100934624671936, 5.965587139129639, 2.5025320053100586, -8.201638221740723, -5.0837273597717285, 3.7357897758483887, -0.21487924456596375, -3.992327928543091, 5.524008274078369, 2.57143235206604, 6.719398021697998, -3.7130026817321777, -3.780043125152588, -2.910590410232544, 8.206296920776367, 7.761684894561768, 4.122021198272705, 0.5889097452163696, -2.6682028770446777, 3.944366693496704, -2.340073347091675, 0.21415309607982635, 1.8352826833724976, -5.469898223876953, -4.321803092956543, 1.9469553232192993, -7.264435291290283, 5.105940818786621, -3.829721689224243, 2.077986001968384, 9.778780937194824, 4.3117995262146, 1.4651095867156982, 1.9065991640090942, 2.4645884037017822, 3.553584575653076, 5.568630695343018, -0.37971585988998413, 0.705437958240509, -2.7785372734069824, -5.017773151397705, -0.44627952575683594, 2.4752323627471924, 1.5919849872589111, -6.31550931930542, 4.301806926727295, 0.883769154548645, 3.906273365020752, -0.8485734462738037, -6.596418857574463, -3.706207036972046, -3.623600959777832, -4.092834949493408, 3.3229947090148926, -1.6938611268997192, 4.37894344329834, 3.8759772777557373, 4.820633888244629, 5.604427337646484, 2.5114195346832275, -2.523437023162842, 0.14795736968517303, 0.17553548514842987, -6.648873805999756, -6.7339348793029785, -1.734537959098816, 0.3076061010360718, -3.7477643489837646, 2.755091667175293, -2.7223076820373535, 0.05366387963294983, 4.969536304473877, 2.894479751586914, -3.2389233112335205, -0.7688528299331665, 5.031099319458008, 4.042995452880859, -5.373441696166992, -2.6510508060455322, 2.8315680027008057, -0.759692907333374, -4.42596960067749, -1.6171696186065674, 7.630183219909668, 1.1127475500106812, -0.7581803798675537, 0.5995675325393677, -7.246923446655273, 3.3953676223754883, 7.502946376800537, 4.333042144775391, -6.7556915283203125, -0.1594826728105545, -0.2431219220161438, -3.502997636795044, -0.6222642660140991, 1.6340543031692505, -2.8023252487182617, 5.004989147186279, -5.139439582824707, 3.8764002323150635, 5.730897426605225, -0.39829498529434204, 0.08823424577713013, -7.147934913635254, -2.8312859535217285, 8.713700294494629, -3.0285775661468506, 2.8950886726379395, 6.119121074676514, 1.2765569686889648, -4.046746253967285, -3.195638656616211, -5.149310111999512, -3.1876072883605957, -4.094123363494873, 5.05418062210083, 5.389270305633545, 1.6304627656936646, 1.0528285503387451, -5.160468101501465, 8.7105073928833, -10.619034767150879, 0.1091471016407013, 1.124362587928772, -0.948864758014679, 3.210688352584839, -1.9925321340560913, -1.4268381595611572, 4.599430084228516, -1.7784076929092407, -7.116497993469238, -4.262852668762207, -3.951650857925415, -1.1572890281677246, -2.1324498653411865, -4.938812732696533, -6.644619941711426, -0.8534780740737915, 8.277862548828125, 8.873356819152832, 0.1545386016368866, -0.5133952498435974, -2.650062322616577, 1.025103211402893, 4.302932262420654, 1.443806529045105, 0.7894922494888306, 1.994621992111206, -1.6972426176071167, 1.6904730796813965, 6.45281457901001, -7.476283073425293, 0.09349214285612106, -1.7760288715362549, -4.2328901290893555, -1.0957891941070557, 0.8571852445602417, -2.242861032485962, -1.6873078346252441, -4.4206743240356445, -3.0921292304992676, 1.8190946578979492, 5.603157997131348, 3.9552505016326904, 4.434813976287842, -0.6857951283454895, -3.600100517272949, -1.3571436405181885, 0.853751003742218, 4.740286827087402, 4.04133939743042, -8.414105415344238, 4.467604637145996, -2.3456919193267822, 3.3129985332489014, 3.9570770263671875, -2.1430225372314453, 4.61598539352417, -3.6839210987091064, -5.041170120239258, -2.5939059257507324, 4.551044940948486, -2.1837899684906006, 2.508439540863037, 1.5782289505004883, 1.8362785577774048, 0.05057821422815323, -7.3872199058532715, -0.5264286994934082, 4.404186725616455, -2.9748518466949463, 3.0021722316741943, 2.8004517555236816, 1.8643708229064941, 1.2719943523406982, 3.8080801963806152, -1.9301961660385132, 0.402005672454834, 4.672645568847656, 0.2020540088415146, -0.928382933139801, -1.9592972993850708, -1.3753678798675537, -0.07767727226018906, 0.3549574911594391, 5.477065563201904, -7.422252655029297, -0.48685306310653687, -2.1187055110931396, -4.00679874420166, -2.2911622524261475, -5.0527143478393555, 2.3337488174438477, -1.0866444110870361, -1.7513656616210938, -5.273879051208496, -1.6109862327575684, -1.1879414319992065, -6.2922587394714355, 1.8470975160598755, 0.30216121673583984, 0.034087859094142914, -3.7835073471069336, -1.1802235841751099, 2.9702961444854736, -2.319153308868408, -3.4222517013549805, -0.08591262996196747, 1.9916942119598389, -2.713785171508789, 8.107988357543945, 2.4165749549865723, -7.299186706542969, -2.54744291305542, -3.3533270359039307, -1.7641521692276, -2.326772928237915, 2.2656948566436768, -3.439460039138794, -6.379194736480713, 2.5958659648895264, -2.7362654209136963, -1.518778681755066, -1.6280667781829834, -1.861149549484253, -3.2006590366363525, 5.524765968322754, -1.224979043006897, 1.216130018234253, 1.3233307600021362, 9.228965759277344, -0.6064427495002747, 7.616734027862549, 3.6065144538879395, 2.328352689743042, 10.86946964263916, 7.428390026092529, 1.2294385433197021, 1.8727909326553345, 3.4321467876434326, -2.3052256107330322, 4.618257522583008, 1.5575629472732544, 5.267623424530029, 0.7532107830047607, -3.881643056869507, -0.23501154780387878, 5.264341354370117, -3.425625801086426, -3.8475894927978516, 9.359817504882812, 7.992315769195557, -6.115697860717773, 0.17720089852809906, -0.0016003421042114496, 3.4275355339050293, -1.215248465538025, 6.214675426483154, 3.188582420349121, -10.855264663696289, -2.993680715560913, -3.5430116653442383, -1.6688284873962402, -5.850743293762207, -8.392436027526855, 5.486448287963867, -4.57106876373291, -0.5027706027030945, 1.1869621276855469, -4.417117118835449, -0.8560835719108582, 1.082038164138794, 0.0419473797082901, -0.3382152318954468, -3.5391316413879395, -4.423760414123535, 1.8394776582717896, 1.8945245742797852, 4.169930934906006, -5.829341888427734, 3.4526922702789307, 0.32945480942726135, 0.062332164496183395, 0.3135842978954315, 1.1296242475509644, 1.2253484725952148, -5.810876846313477, 8.74892807006836, 7.473550319671631, 2.342918872833252, -0.22919292747974396, -1.9263496398925781, -3.3277697563171387, 2.8155040740966797, 1.3738369941711426, 7.026335716247559, 3.848158597946167, -2.134891986846924, -0.8613787889480591, -1.2974668741226196, -4.509699821472168, -4.648452281951904, -0.11443382501602173, -4.663383483886719, -2.1661183834075928, 0.12033572793006897, 3.5681910514831543, -1.7753549814224243, -7.169647693634033, 0.37021201848983765, -1.6535357236862183, -4.301065444946289, 1.1439411640167236, -6.695664405822754, -4.173680305480957, 4.9635725021362305, -3.886770486831665, -0.1215471550822258, 1.076705813407898, 0.2264215499162674, 2.2190792560577393, -4.061376571655273, 0.3392099440097809, -0.5334994196891785, -5.52263879776001, -3.4895377159118652, -3.3979651927948, -3.637817144393921, -0.5570884943008423, 0.6311697959899902, 0.24033871293067932, -0.44625359773635864, 0.7567760348320007, -2.9157979488372803, -2.6348142623901367, 0.8369070291519165, -2.128521203994751, 4.735208511352539, -2.1723976135253906, -4.336397647857666, -0.3889985978603363, -0.8605399131774902, 1.2540305852890015, 2.8020756244659424, -9.539628028869629, 9.332757949829102, -0.7916305065155029, 0.6887343525886536, -3.262711763381958, 6.13610315322876, -0.8477522730827332, -1.2194408178329468, 3.361382007598877, 3.6191763877868652, 10.344564437866211, 2.7188353538513184, 8.84830379486084, -2.7117371559143066, 0.6653559803962708, 2.425457239151001, -0.5656388401985168, 5.021584510803223, 0.28348079323768616, -2.690532684326172, 2.806471109390259, 5.414994239807129, -8.109492301940918, -1.1490898132324219, -3.381439685821533, -0.8363152146339417, 1.7179127931594849, 5.144472122192383, 1.2767213582992554, 5.3060479164123535, -3.3526647090911865, 0.8245455026626587, -1.7290234565734863, 4.452059268951416, -3.0123462677001953, -2.5856616497039795, -4.109310150146484, -2.5439743995666504, 4.848968505859375, 0.5148497223854065, -3.7012784481048584, 1.9557170867919922, 0.5513946413993835, 2.498215436935425, 1.2131205797195435, 1.3811264038085938, 2.6369454860687256, -0.30374813079833984, 3.932980537414551, -2.5126140117645264, 3.495851993560791, -0.5812864899635315, 3.6055009365081787, -3.9146151542663574, -5.772937297821045, -2.7467586994171143, -6.172548294067383, -2.311277151107788, -2.6666829586029053, -3.052927017211914, -3.213074207305908, -3.9018335342407227, 4.298943519592285, 3.392714738845825, -0.3957051932811737, 6.113037109375, 0.21251188218593597, 3.805370330810547, 0.41724351048469543, 5.800920009613037, -2.214726686477661, -1.441845417022705, -1.4051398038864136, 3.717574119567871, 1.1331843137741089, 2.3305294513702393, 0.18473032116889954, 12.138875961303711, 3.2496261596679688, 2.1825149059295654, 2.4052817821502686, 3.4918763637542725, 2.814683675765991, 5.7287445068359375, 2.9506070613861084, -2.980487108230591, 3.7885854244232178, -2.7067782878875732, -4.036281585693359, 4.287467956542969, -1.309786081314087, 0.3279431164264679, 0.2420024573802948, 6.050042629241943, -4.279921054840088, -2.287905216217041, 1.2120622396469116, -1.2424007654190063, -4.148731231689453, -5.002319812774658, 1.733561396598816, -0.26489949226379395, 0.9393881559371948, -2.0135726928710938, -0.16875143349170685, 3.7800378799438477, 7.577929496765137, -4.286220550537109, -0.05581963062286377, 0.7415553331375122, 5.253489971160889, 2.192464590072632, -0.10400012135505676, 2.0087389945983887, -6.25003719329834, 4.277376651763916, 2.5118086338043213, -0.47491559386253357, 7.30609130859375, -2.156325101852417, 2.0333993434906006, 1.0909875631332397, -0.19382546842098236, -1.7064061164855957, 3.2631776332855225, 0.5572822093963623, 6.031915664672852, 0.6898056864738464, -7.281960964202881, 1.1455366611480713, -2.831812620162964, -0.9449440836906433, 2.719097137451172, 0.8111013174057007, 5.301284313201904, -1.4989070892333984, 0.7758349180221558, -2.8536086082458496, 0.8279715180397034, 0.32585781812667847, 7.0002899169921875, -5.356002330780029, -5.159758567810059, 8.722105026245117, -1.7367833852767944, 0.46395987272262573, 1.2399945259094238, 2.1445698738098145, 7.2572832107543945, 0.26330533623695374, 1.3201220035552979, -2.899333953857422, 5.304329872131348, 5.685363292694092, -5.108716011047363, -3.6983611583709717, -3.832200527191162, 0.8216454982757568, -3.2907896041870117, -3.207693099975586, -2.5609683990478516, 1.9601026773452759, 4.047346591949463, -1.0086184740066528, -3.9465994834899902, -2.9165101051330566, 3.7937068939208984, -1.3105708360671997, 4.843310832977295, 3.769259452819824, -1.168041706085205, 6.346653938293457, -3.494445562362671, 4.58698034286499, 7.973365306854248, -1.148618221282959, -5.8457441329956055, 7.746114253997803, -0.17800872027873993, -1.8512556552886963, -4.687808513641357, -0.7193065881729126, -1.963653802871704, 5.681663990020752, -1.148488163948059, -2.1207122802734375, -3.8130552768707275, 1.3003535270690918, -4.367412567138672, 7.517362594604492, -2.029412269592285, -0.4892067313194275, 0.9130847454071045, -1.4485223293304443, 2.912736415863037, -5.182359218597412, 2.526785373687744, -4.118525981903076, -1.4151725769042969, -3.0729081630706787, -2.7613160610198975, 3.238205909729004, 2.5834121704101562, -0.08965186029672623, -6.113179683685303, -8.116501808166504, 9.15108585357666, 3.252286911010742, 1.1780893802642822, 1.6441766023635864, 8.110467910766602, 0.21103642880916595, 5.008411407470703, -2.2186808586120605, 2.5900447368621826, 8.509354591369629, -6.607850074768066, 0.8056206703186035, 3.8311572074890137, 0.1926051527261734, 0.23473331332206726, -2.5045735836029053, -3.512270450592041, -0.5603967905044556, 2.6729848384857178, -3.0050082206726074, 4.086874961853027, 3.351698160171509, 1.1847785711288452, -0.6960014700889587, 2.473454236984253, -3.7017016410827637, 0.00719493767246604, -5.990482330322266, 0.37521108984947205, 3.2077291011810303, 9.065576553344727, -1.9661246538162231, 3.1147384643554688, -0.6243956089019775, -7.046812534332275, -6.005576133728027, 1.3927836418151855, -4.727506160736084, 3.35632061958313, -2.5804035663604736, 5.772791385650635, 1.6963350772857666, 0.7214521169662476, -1.2593251466751099, -5.64439582824707, 1.3697527647018433, 6.897696495056152, 4.857677459716797, 2.9543912410736084, 1.669382929801941, 0.6097589731216431, 0.9761890172958374, -1.7358367443084717, -5.485265731811523, -6.4591264724731445, -2.847628116607666, -0.2955586910247803, -6.093325614929199, 4.564005374908447, -7.079384803771973, -4.464804172515869, 1.7447255849838257, -0.02453056536614895, -0.5088842511177063, -0.9569370150566101, -2.0620334148406982, 7.381457805633545, 5.601373195648193, 10.701010704040527, -5.4318695068359375, 4.71095085144043, 1.7896724939346313, -2.6262905597686768, 2.018251657485962, -2.6014273166656494, -6.361973762512207, 0.3572844862937927, -1.896257996559143, 1.5376918315887451, -1.7380961179733276, 3.299614906311035, -1.5226919651031494, 7.148155689239502, 6.901278018951416, -2.332617998123169, -5.298058986663818, -1.41575288772583, -2.5026566982269287, -0.6437125205993652, -1.2718055248260498, -2.744993209838867, -3.5368595123291016, -5.016861438751221, 2.262956380844116, 0.8929122686386108, -10.539142608642578, 1.7574511766433716, 6.502748966217041, -4.130748748779297, 0.49506333470344543, 1.8010852336883545, -5.747945308685303, -1.2343902587890625, -4.119860649108887, -2.2395401000976562, 3.0115551948547363, -2.2676620483398438, -4.7119832038879395, -5.982439994812012, -5.735598087310791, -4.635867595672607, 4.063874244689941, 0.3275362551212311, -0.8089475631713867, -2.8039350509643555, 1.480523943901062, 0.6633217930793762, -2.0293798446655273, -7.080756664276123, -2.674267292022705, 3.4343066215515137, 1.1655064821243286, -2.874863624572754, 3.4645028114318848, -4.4552412033081055]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[2.226929396442756e-08, 1.3036911639119353e-07, 5.514287337859969e-08, 5.4533249027599595e-08, 1.3716048385958857e-07, 4.328662939201422e-08, 3.5015617072531313e-07, 8.780484250792142e-08, 1.0358388635722804e-06, 7.710319493980933e-09, 2.5230363576156378e-08, 3.4776472812581005e-09, 1.3410164645399902e-10, 1.4132060105476185e-11, 1.3579777657923842e-07, 3.852230534562295e-09, 5.2077089307545066e-09, 1.0462858135440456e-08, 5.676940304510936e-07, 1.0347574743718724e-06, 1.660304405959323e-06, 4.315093065088149e-06, 0.01620405912399292, 3.47083158658279e-07, 1.9797819561517827e-07, 1.728150608726331e-11, 1.2674797322809184e-10, 0.0006886524497531354, 0.006285989657044411, 6.365544891195896e-07, 7.74444188778034e-08, 9.913261465044343e-07, 3.110362661873367e-11, 1.278483185984669e-08, 2.10945181322586e-08, 0.00011049053864553571, 2.025318135423504e-09, 4.5612696908392536e-07, 1.933519797603367e-06, 8.038112355279736e-06, 1.9193459266375612e-08, 6.837419963545699e-09, 1.7662211970748132e-11, 1.0520050636841916e-05, 1.2082192370144185e-05, 1.504372804106424e-08, 2.6913647843684885e-07, 4.010964893552682e-10, 4.6229519767848615e-08, 1.960525196409435e-06, 1.5441903755686326e-08, 1.374191640479694e-07, 8.695765245647635e-06, 1.179008563667594e-06, 9.201471584674437e-09, 6.816196400905028e-05, 0.0001291988301090896, 5.911163498240057e-06, 1.1801117807408446e-06, 1.8701267663345789e-06, 3.6339915823191404e-06, 6.838609678538887e-10, 8.013194928935263e-06, 3.7016423259927933e-09, 2.3769081280988757e-07, 7.189225215142869e-08, 7.130639687602525e-07, 0.00012118698941776529, 8.997992608783534e-07, 6.644715000447832e-08, 1.5545153928542277e-06, 1.6973420713384257e-07, 1.4307995066520363e-10, 7.623765507780433e-11, 1.2644877145362443e-08, 0.0003069275408051908, 3.469876119766013e-08, 1.4691466958538513e-06, 1.7678898075246252e-05, 5.217229426257575e-10, 2.3435033654095605e-05, 3.2119257866725093e-06, 9.062054573405476e-07, 4.942603027302539e-06, 2.6941726627427443e-11, 2.0255775723398983e-07, 2.0549548174209065e-10, 3.9887194702714623e-07, 4.9828056347678285e-08, 1.7813141184319647e-08, 1.0338219453842612e-06, 8.333142509364677e-11, 1.394019761846721e-07, 0.000636714743450284, 6.513224889204139e-06, 2.2285158163271035e-09, 5.61246736197063e-07, 5.193182417428943e-08, 7.685708851568052e-07, 9.082182228503655e-10, 7.028424420241208e-07, 1.070464961117068e-07, 9.88908288590551e-10, 1.7460934031987563e-06, 2.5722363716340624e-05, 1.0780562661238946e-05, 2.7622680618538675e-12, 4.2975875658157747e-07, 4.602600540692947e-07, 4.911140294439065e-08, 5.299453320617431e-08, 5.892337071600195e-07, 1.583501330060244e-06, 5.8465552399411536e-09, 2.56144176091766e-07, 4.144410326034631e-08, 5.297121408176508e-09, 4.739043220070016e-07, 1.0683169016090233e-08, 9.149291047005903e-11, 1.2391132564459895e-08, 8.727413236542247e-11, 4.173735305812443e-06, 3.419312477603853e-08, 5.814723408548161e-06, 3.113983723657121e-11, 6.8519679189194e-05, 7.297526281035971e-07, 1.0865015553918056e-09, 0.0003970015677623451, 5.990755891360777e-09, 1.0643614878347307e-09, 5.56482397939817e-08, 5.382430572353769e-07, 3.468823479124694e-06, 0.0012619872577488422, 3.374470702866006e-09, 7.526579359762309e-11, 3.688749643515621e-07, 6.28553875614557e-10, 4.859005975355046e-10, 8.408191831676959e-08, 2.1768104488728568e-05, 3.409298177103892e-09, 5.303615768781356e-09, 4.1556784680096825e-08, 2.5670535919175563e-08, 9.108669473789632e-06, 1.1171123048825393e-07, 6.646133643428698e-10, 0.0015610489062964916, 8.280645502622974e-09, 1.3386800001846666e-10, 1.131935619014257e-06, 1.2645419644741196e-07, 1.3041825468462775e-06, 7.489039433039579e-08, 1.162464826620635e-07, 2.0912565901198832e-07, 3.946178821934154e-06, 1.461832217775072e-11, 1.7948768515907432e-07, 1.0928521021469351e-07, 0.0007327857310883701, 5.483581144716254e-09, 1.588131817698013e-05, 4.750556399812922e-06, 6.884784742311467e-09, 1.21257016871823e-06, 3.869476586260134e-06, 1.9345727196196094e-05, 4.367437668406637e-06, 8.702529896709166e-08, 1.020313561639341e-06, 9.168708459128538e-09, 2.9433750992069463e-09, 1.827201145943036e-08, 6.480029401956244e-09, 3.983756435843588e-09, 8.543711737729609e-05, 8.779935112279702e-11, 5.054607186139037e-07, 5.818180852656951e-07, 1.0642731140819706e-09, 2.8032794944010675e-05, 2.1179324960485246e-09, 2.6206001280115743e-07, 1.2044237829300641e-10, 4.1922557092455293e-10, 3.983618768188535e-09, 3.577776794827514e-07, 5.772665099357255e-05, 8.416849439640828e-10, 3.2790358091006055e-06, 1.0698943953002527e-08, 2.143359552064794e-06, 2.0041418338223593e-06, 2.5183527441186015e-07, 4.111774387638434e-07, 6.02634400714841e-11, 1.9082779090240365e-06, 1.624571996217128e-05, 7.117957778746131e-08, 3.134627831968828e-06, 2.581995204309351e-06, 1.6213944036280736e-05, 4.778912376934841e-09, 1.3502964968381548e-08, 2.717071765800938e-06, 2.365807858950575e-06, 3.3782271202653646e-05, 4.1949789419959416e-08, 1.200066321871418e-07, 6.3916480996795144e-09, 4.652476448541165e-08, 3.1170919100986794e-05, 1.1587151504954818e-07, 9.925042832037434e-07, 9.074638995798523e-08, 3.3100555185683334e-08, 1.05753379386897e-07, 9.718151972037958e-08, 2.0721127036438247e-09, 5.301167060878242e-09, 4.985772150689627e-08, 1.1564012538656243e-06, 7.776348098786912e-08, 1.9579010634984684e-10, 0.004933016840368509, 3.770898331367789e-07, 3.032874573705158e-08, 1.7040605371221318e-07, 1.0837013775244486e-07, 1.264530169464706e-07, 1.1219203770451713e-06, 7.416569474116841e-07, 2.1362210134157067e-07, 9.774823411135003e-06, 2.6214621584585984e-09, 0.00047627376625314355, 3.397402110749681e-07, 2.1639721126121003e-09, 4.48287275356396e-11, 3.571821309833467e-07, 2.4779981231404236e-06, 2.8906685045626546e-08, 4.724508073650213e-08, 2.297391255012826e-08, 1.4174781881592935e-06, 3.7864938917664404e-07, 3.187410058114182e-10, 6.71738007440581e-07, 1.5315953305616858e-07, 7.854144996599643e-07, 3.778617863048339e-09, 3.4896834222308826e-06, 3.317905793664977e-05, 2.1313371206810316e-11, 5.720794888475211e-06, 9.937992473396662e-08, 8.833456632828529e-08, 4.168214928768066e-08, 2.7634708743562442e-08, 3.138275133096613e-07, 2.6628509175452564e-08, 5.556617232310712e-12, 1.7462787127442425e-06, 2.1389820403783233e-07, 1.624356627871748e-06, 7.231433507559615e-11, 8.056043157012027e-07, 6.846156530571079e-09, 3.1047109132487094e-06, 3.6408809478416515e-07, 4.423992140800692e-05, 1.195266486320179e-05, 9.053071153175551e-06, 1.74366580202312e-11, 2.0892837710562162e-05, 1.7316467619821196e-06, 9.27636136793808e-08, 1.7528351236251183e-05, 3.0894966585037764e-07, 6.259709266487334e-07, 9.187547789224482e-07, 1.5706127953762916e-08, 3.750056691842474e-07, 4.1224024194264075e-10, 3.3544238249305636e-06, 8.904540749199441e-08, 8.057469926825434e-07, 3.2792426551964127e-09, 6.324051504691397e-09, 3.980396456881863e-07, 4.112360693397932e-05, 2.977996738096067e-10, 2.1561059881936018e-10, 8.675089979703898e-09, 1.1735958651115652e-05, 1.9189798194929608e-07, 2.06502645596629e-05, 4.5189292308123186e-08, 3.1571480985093103e-09, 4.2855194948288045e-08, 6.242862582439557e-07, 1.65058389178796e-09, 2.674850918538141e-07, 1.4884194854403177e-07, 5.950147752287194e-08, 1.068706410478626e-06, 4.0012493318641873e-10, 3.7648621997732334e-09, 8.518859999639972e-07, 2.3954152084115776e-07, 9.279713708565396e-07, 2.716011977099697e-06, 0.00029615324456244707, 3.1322610993811395e-06, 2.8840790037065744e-05, 1.052076044061323e-07, 4.0507525227440055e-06, 4.6103576778477873e-07, 1.2406909277729028e-08, 5.790722568121964e-08, 3.004116777560739e-08, 9.238874554284848e-09, 1.266289384460606e-08, 0.00028701801784336567, 2.5602018283166217e-09, 5.914573009135893e-08, 8.469186289772335e-12, 5.7945603870734885e-09, 2.365692637340544e-07, 6.099763595557306e-06, 5.210757936247035e-10, 1.3693534128833562e-07, 9.793210686837028e-09, 1.563173981367072e-07, 4.029005573613631e-09, 9.931758171433103e-08, 3.573308333670866e-07, 9.320820026914589e-08, 8.355911268154159e-05, 1.213196242133563e-06, 1.3161511560610961e-06, 4.946611523171107e-11, 1.1843107131426223e-05, 1.2019701216559042e-06, 4.338829967309721e-05, 1.469302191026145e-08, 7.048783459140395e-07, 0.0018747026333585382, 3.0502471304316714e-07, 2.8462417844821175e-07, 1.009799802886846e-06, 1.696178333077114e-05, 1.6020743842659613e-08, 3.5175870038983703e-07, 5.355588195143923e-10, 9.576713727588171e-11, 3.736138953769341e-09, 3.178808771764352e-09, 1.0539942529863566e-10, 4.729103775247268e-10, 2.074117100292483e-09, 3.1106623055165983e-07, 3.961213792536e-07, 4.185433226666646e-06, 1.1684099376907398e-07, 5.301862060491658e-09, 1.386552983539957e-09, 1.3089847925584763e-06, 8.586881250494116e-08, 1.4203250202626805e-06, 5.297724037234275e-09, 1.1089298368460732e-09, 2.795023767987459e-08, 0.007539418991655111, 2.5846807716334297e-07, 9.507996878710401e-08, 2.2466991822511773e-07, 2.31046559662218e-09, 0.00021117304277140647, 7.583849659376085e-10, 2.469744053712475e-08, 5.1084644070442664e-08, 4.861603897232669e-10, 1.0615046619477653e-08, 3.188680430810109e-09, 3.428208401601296e-07, 1.6178161033053584e-08, 2.362445684411796e-06, 5.882594905415317e-06, 2.815380741039153e-11, 1.0035610102931969e-06, 1.6427652838046924e-07, 8.982206090024647e-09, 5.590187811321812e-06, 2.6888201887231844e-07, 2.447224201063136e-09, 2.6167737421367576e-10, 7.075676222711991e-08, 1.0723325658545946e-06, 3.743950482970604e-09, 0.0003569313557818532, 4.226282171448936e-11, 4.355170801773056e-07, 7.87070941754564e-09, 1.6288700862787664e-05, 1.2672484217546298e-06, 4.239298050379148e-07, 1.1800746158030506e-08, 4.726230518059538e-09, 1.3513443263946101e-05, 6.748127390210357e-09, 4.520917684658343e-08, 1.8398805812225305e-09, 3.6912808809574926e-08, 9.770338184333927e-12, 0.00043392812949605286, 1.639777110540308e-05, 4.379044071800564e-10, 6.362251281366582e-10, 7.184461026099598e-08, 6.315436280601716e-07, 5.320477498571563e-07, 3.7245609374458866e-10, 2.1941750105725077e-07, 3.648775503961588e-08, 4.009139686900198e-09, 7.765433110762388e-06, 3.2292439300363185e-07, 8.072279911175428e-07, 2.1899511182255083e-07, 4.603197112373891e-08, 0.0003121408517472446, 2.0967965141949207e-08, 5.37752377349765e-11, 2.8779693295177822e-08, 2.459728809522943e-10, 4.820187768928008e-06, 2.3423952370649204e-05, 1.2677774385849716e-08, 6.663399858553021e-07, 2.0323664529087182e-08, 1.2074250662408303e-05, 2.8035708965035155e-05, 3.2064272659226845e-07, 6.224495763262894e-08, 3.4400180481952702e-09, 3.034928558465211e-11, 4.4455972236789876e-09, 3.915087987138577e-08, 4.7052704843508764e-08, 1.6429092966063763e-06, 7.351891326834448e-07, 2.288477958245494e-07, 1.5102525139809586e-05, 2.9324451200629653e-10, 2.9272609936015215e-06, 2.0624582042216844e-09, 6.352559580591333e-07, 1.2937199244333897e-05, 7.196304068202153e-05, 6.248080808290979e-08, 2.665698595194499e-08, 6.2474055084749125e-06, 2.0774434688064503e-06, 7.547648692707298e-07, 2.2478944927684097e-08, 1.2587122455443023e-09, 7.852709593691998e-10, 0.005884295795112848, 1.3289428224538824e-08, 2.464135491209163e-07, 3.0253149816417135e-05, 1.3908161289322152e-09, 8.321863198545998e-09, 9.007230801216792e-06, 9.423237088412861e-07, 3.156482852872955e-10, 5.528067781312984e-09, 3.5048017821281974e-08, 3.6030122885222227e-08, 1.6665325119902263e-06, 1.4433836348359819e-08, 1.0615094359067712e-09, 3.4451172581384526e-08, 5.479676474351436e-05, 3.490831401720129e-09, 8.284975905326064e-08, 7.331954634537396e-07, 5.27301864394758e-08, 0.000158894996275194, 3.6430539140930307e-10, 3.020157990363259e-08, 1.6952139958448242e-06, 1.4777967862755759e-06, 6.657362519035814e-06, 3.508564816456783e-08, 2.6107977646461222e-06, 1.7102542715363711e-09, 7.439658666896776e-09, 2.2228402940527303e-06, 1.1118533649323581e-08, 8.131536560540553e-06, 1.5686623555666301e-07, 2.738748605679575e-07, 1.7499464988190994e-08, 1.4669560705060647e-11, 0.00015608835383318365, 1.2480602107345362e-09, 2.2448691652243724e-06, 3.00378616202579e-07, 1.6788007428658602e-08, 7.394898027257568e-09, 1.8706466562434798e-06, 6.729911774527864e-07, 2.651232655637159e-08, 1.1839121043522027e-06, 1.5933518682231806e-07, 1.0377357284596656e-07, 4.864248026592577e-08, 9.794256072837015e-08, 1.2946230754096177e-06, 2.147276800812392e-09, 3.6258665403465784e-08, 5.572089456507001e-09, 1.3620508809708554e-07, 9.67419623520982e-07, 4.733317626737232e-10, 4.00139506382402e-05, 9.321757948876908e-11, 3.139432325216518e-10, 8.04624838979251e-11, 4.42882355855545e-06, 1.480791045338492e-08, 0.0001514518226031214, 3.834774986444245e-08, 7.999161425686907e-06, 1.049824005505684e-09, 3.1514869647253363e-07, 1.6815482339893606e-08, 2.56762588968229e-09, 0.00026327293016947806, 1.236281059391331e-05, 1.4719785390582274e-08, 2.700556167578583e-10, 7.175031413453326e-08, 3.581176599709579e-08, 1.6129649793583667e-07, 2.2814679923044423e-08, 1.3975299140156494e-08, 3.843775573386665e-07, 0.0001261812140000984, 2.7616042643785477e-06, 4.496605754411576e-09, 8.179551258535867e-08, 4.2470720273968254e-08, 2.007575631068903e-06, 2.2679716948914574e-07, 1.8989585726103542e-07, 1.1115746474388288e-06, 2.276806743140014e-09, 2.742957683210534e-08, 7.68830261677067e-08, 1.7009778030541156e-09, 5.754160156357102e-06, 6.886990377097391e-06, 3.826594507927439e-08, 6.722981424900354e-07, 1.7873272306623278e-10, 6.628762605487282e-08, 4.4900222206933904e-08, 2.233655504824128e-05, 2.3319961428569513e-07, 7.752591045573354e-06, 5.927409802097827e-06, 5.312607527230284e-07, 3.935778281061175e-08, 1.5825699506422097e-07, 4.083089422124431e-09, 0.0001237305550603196, 5.67582212340767e-08, 1.7003146979277517e-07, 1.5286861554386633e-08, 3.5688007216094775e-08, 3.784290356634301e-07, 6.98801341059152e-06, 0.0005510086193680763, 2.593761472269307e-08, 3.5429113864893225e-08, 7.285871106432751e-08, 3.037864827314801e-13, 2.7845903360912416e-09, 6.761284310208282e-10, 4.5504125978368393e-07, 2.32436082114873e-06, 7.135411816960868e-09, 7.435608750938627e-08, 5.185734721635527e-07, 4.734922870452962e-11, 7.501433918477574e-10, 8.969317377705011e-07, 3.98453242622665e-10, 4.538825123745482e-06, 1.128382336901268e-06, 1.292519602458242e-08, 1.2763905488100136e-08, 2.5520116575528107e-10, 2.3586771646932903e-07, 4.4722972347699397e-07, 3.2460916372656357e-06, 2.7277854314888828e-05, 4.821246329811402e-05, 1.0433909380935802e-07, 4.812999421233144e-08, 2.6478335080071247e-09, 0.0002523536386433989, 4.797526798938634e-06, 2.1595930377316108e-08, 2.4502172735196837e-08, 7.080149089233601e-07, 1.6916489187224215e-07, 3.1838453651289456e-07, 4.996644209498413e-10, 2.9295312842236854e-09, 3.481856651887938e-07, 2.7686288817108107e-09, 9.26706036352698e-07, 9.236250093636045e-07, 3.6949967352484236e-07, 2.1953442228550557e-06, 1.5555808374756452e-07, 8.922425109858523e-08, 7.248144129334833e-08, 2.7585644879479787e-09, 9.49856371335045e-08, 3.391784844097856e-07, 2.16751824821948e-11, 3.476884103292832e-06, 3.962719929972991e-09, 9.373015696922948e-10, 6.964845056245395e-08, 6.435884358779731e-08, 9.853348359456504e-08, 2.2914663500017696e-09, 9.332297281616775e-07, 1.453020169461805e-11, 2.566647727686444e-10, 6.088779258561772e-08, 5.641923962684814e-06, 4.610318171671679e-07, 4.160942523867561e-07, 0.0002492933999747038, 2.7382954237431534e-10, 1.2335474863789386e-08, 5.3633744556691454e-08, 3.675011157611152e-06, 9.72631419848824e-11, 5.072329778421647e-10, 2.6910001338364964e-07, 6.040939970475279e-11, 4.869752379121906e-10, 1.3644591012607066e-10, 3.917163468591944e-11, 1.0379065997767611e-06, 1.2784641967300558e-07, 3.6232070215191925e-06, 4.274437159779154e-08, 0.00011658563016681, 7.018136627401361e-10, 4.474699721868092e-09, 6.029295285259195e-09, 2.821692248744512e-07, 1.230898902804256e-07, 1.2127236914238892e-05, 7.111853328467532e-09, 1.631103287991209e-07, 6.668510632579228e-09, 2.0109153453518047e-08, 2.1735811515100067e-06, 3.9248297980520874e-06, 5.4128990178980985e-09, 3.688193084272484e-10, 0.0006028064526617527, 4.326249225528045e-08, 9.419999003057455e-09, 1.4080585060582962e-05, 1.111889608162997e-10, 3.691507345138234e-06, 5.803639879786715e-08, 4.541654874401502e-08, 1.847563524393081e-08, 1.1387629456294235e-05, 4.056506952565542e-08, 2.1437605024487993e-09, 7.1319931294056e-07, 3.4366886506737027e-12, 6.812495121266693e-05, 1.341545754485196e-07, 9.165395908894425e-07, 8.113540950205334e-10, 3.0664717293182875e-11, 6.471595384027751e-07, 4.41430253461661e-10, 2.9458291805894987e-07, 8.137610030267695e-10, 2.8697147627099184e-06, 2.110648189557196e-08, 1.5047338930429532e-09, 1.5326305913276883e-07, 2.8019478404139875e-10, 8.007603173609823e-05, 7.226016350614373e-06, 2.2604643490353737e-08, 6.254970319474751e-09, 1.2756627976173718e-09, 2.0221704311040867e-09, 0.00018000268028117716, 2.649324778758455e-05, 0.00011582887236727402, 4.085541974174767e-11, 5.3936851429625676e-08, 1.797149018142874e-12, 1.411873840062583e-09, 1.1655122307274723e-06, 2.6611268744147765e-11, 2.2169093426782638e-05, 1.5354781623955205e-07, 6.940217289752582e-10, 1.8319104810871067e-07, 1.0544008999247012e-09, 2.903044915569808e-08, 1.8092308891937137e-05, 0.00021237623877823353, 1.298149809869642e-09, 1.0326138166760757e-08, 4.1573880338319213e-08, 9.773092557452401e-08, 8.367778718820773e-07, 5.488260956809654e-09, 1.919326253485565e-09, 3.834725426088426e-09, 5.737645096814958e-07, 8.399686521443073e-06, 1.650252556828491e-08, 1.0040757842944004e-05, 6.374411753995446e-08, 3.3590386010473594e-05, 3.072368748302523e-10, 2.869864488275198e-07, 6.913292509125313e-06, 2.226077100431212e-08, 3.270780624120562e-08, 4.004345299790657e-09, 9.02539977687411e-05, 7.635933663152628e-09, 2.9587092740257503e-06, 0.0008711121045053005, 6.048430236660352e-07, 8.90873561729677e-05, 2.130434722857899e-06, 1.5091193290572846e-06, 1.9910576298798333e-08, 4.828927924194204e-09, 2.618749931571074e-06, 9.006014067836077e-08, 2.7120244938672045e-10, 3.007470184002159e-07, 3.8058846030253335e-07, 5.371121574171411e-07, 1.0350207446663262e-07, 3.0654103255756127e-08, 1.537887612812483e-07, 2.323192802577978e-07, 3.507858309603762e-06, 2.1673713490599766e-08, 1.07010734495816e-07, 2.709376190068724e-07, 1.5660726830901694e-06, 1.1796745802428177e-08, 5.302042609400814e-07, 6.015861142572021e-09, 2.879471105998732e-09, 1.61352904370915e-08, 1.4230846723251034e-08, 1.0307820375032861e-08, 2.339102280757288e-07, 1.1104160648756078e-06, 7.181242711595814e-09, 1.1178314451854021e-07, 8.978777721324604e-09, 5.581013429178938e-09, 2.5391874713953655e-10, 6.81784684264386e-10, 8.65473737121647e-07, 2.8872008783764613e-08, 5.571258725467487e-07, 2.3421486972097227e-08, 0.00011240441381232813, 3.979853246960374e-08, 1.4380360369159462e-07, 1.3436027401780848e-08, 0.00048683275235816836, 2.51150481744844e-07, 1.1344249095657233e-08, 2.509383989490743e-07, 3.5098832174007555e-10, 1.0460602339890102e-07, 1.9364614800565505e-08, 1.7312466980001773e-06, 1.0072638190194994e-07, 1.9160397712880695e-09, 1.444322578203483e-10, 3.2097406954534335e-09, 8.932962458629845e-08, 1.3999340353620937e-06, 1.2564510143420193e-06, 3.777231860624397e-09, 9.857424032588824e-08, 1.6272762195512769e-06, 3.9766635495652736e-07, 4.847582890654678e-10, 2.896792583584329e-09, 8.973419340918554e-09, 1.4277040660815032e-09, 9.076056572965285e-10, 5.310628381494098e-09, 4.7500693654001225e-06, 2.357349444537249e-07, 2.3758252609695774e-06, 1.5382358924398432e-06, 2.127129619111656e-06, 2.2595936233216207e-10, 1.573783094954706e-07, 2.045339897449594e-09, 1.4195902231506352e-08, 3.770801129121537e-07, 1.4730427210452035e-05, 6.008380637467781e-07, 1.0260944804940664e-07, 2.9253616196101007e-10, 1.1537100164105141e-07, 2.4701641450519674e-05, 2.7847309524986485e-07, 3.030571065210097e-07, 3.2423359552780084e-09, 3.925370606339129e-07, 8.52775572379727e-10, 8.521857886023554e-09, 7.162012316541677e-09, 3.529821412939782e-08, 3.1631364194595335e-09, 3.503352274947247e-08, 1.0172681186304544e-06, 2.8121686312942984e-08, 2.3768898245180026e-05, 2.0862887806227803e-10, 2.268481580358639e-07, 6.201125724025758e-10, 9.174140700451971e-07, 6.117472639566301e-10, 4.038995526922662e-10, 0.0001615894288988784, 7.501237131446459e-11, 4.202974537292903e-07, 3.822344183390669e-07, 1.110167815454588e-07, 2.920516273263729e-08, 2.5388763313927143e-10, 2.2453591763671454e-10, 1.164725471625161e-08, 7.360034715020447e-08, 2.901621998230297e-10, 1.200797594691494e-08, 0.0001439766783732921, 1.5610955870215548e-06, 1.1230331438127905e-05, 9.708036259326036e-07, 2.2694869594719336e-10, 2.3758943257234932e-07, 2.894354622640094e-07, 1.0365588423155714e-05, 0.002217987086623907, 1.05091668700652e-08, 1.9349404567492456e-07, 2.5262324925279245e-06, 2.3382078495615133e-09, 1.8639108034790297e-09, 1.2991011317353696e-05, 6.327222195068316e-07, 1.5253739320542081e-06, 1.4781883237446891e-06, 1.8436601294524735e-06, 3.937751014149171e-09, 1.3571753498808903e-08, 2.5165916017755308e-09, 9.204001798934769e-06, 2.5420285876265325e-09, 1.7482813419178456e-08, 1.328959683633002e-06, 1.3290087963468977e-06, 3.8244898803441174e-08, 4.878708637079399e-07, 1.833109763538232e-06, 6.083049619576286e-10, 4.119653112866217e-06, 2.5023606212926097e-05, 0.0004847868112847209, 1.5212249309115577e-06, 7.863627082826952e-09, 1.0908691194799758e-07, 7.351311559489204e-09, 7.319755468415678e-07, 1.599554799724956e-08, 6.277323700487614e-05, 6.021716458803894e-11, 1.2415011951816268e-07, 6.917301789144403e-07, 1.242825078406895e-07, 1.7855772966868244e-06, 1.659313539903451e-07, 1.1493941215467984e-11, 4.9159423198830154e-09, 0.0006453159148804843, 1.6792672141718867e-08, 8.452007591586153e-07, 4.300578893889906e-06, 2.7397584290156374e-06, 3.941832176224125e-07, 3.307496081106365e-05, 2.7777471700574097e-07, 6.159121403470635e-05, 5.581470645665831e-07, 9.412833179567315e-09, 2.7665265633913805e-08, 3.777984858288619e-08, 1.7293575460541888e-09, 4.7294914651274667e-08, 8.790473771114193e-08, 1.2049521274093422e-06, 4.908341511011827e-10, 5.891708383387595e-07, 2.875809286706499e-06, 2.466295427439036e-06, 2.005118154846386e-10, 9.126209477017255e-08, 3.3403222410299804e-09, 0.13032421469688416, 3.3396775052096928e-06, 1.3205140980687702e-08, 2.841645363460543e-09, 4.665202890663522e-09, 3.38531037868961e-07, 1.0365388334321324e-06, 2.4528498343556748e-08, 3.7879881453761755e-09, 4.8979327538223316e-11, 4.832948434341233e-06, 1.0488266752872732e-06, 1.2905966286780313e-06, 2.794507958370218e-10, 1.0041096842883235e-08, 5.578872119826883e-08, 1.570759025071311e-07, 5.926417543378193e-07, 0.0005175891565158963, 1.8257293277201825e-06, 2.3221291485242546e-06, 7.846995941918067e-09, 1.240941838176468e-08, 2.6778408823702193e-07, 4.162830373388715e-05, 2.182164449138213e-10, 6.045158595924249e-09, 0.0027068443596363068, 6.659550126641989e-05, 7.277441909536719e-06, 3.701767127495259e-06, 1.7521505357720457e-09, 2.472646754014818e-09, 5.844634074492205e-07, 4.027501887549079e-08, 9.29988352993405e-09, 7.969834570076273e-09, 1.3935612514615059e-05, 1.644192693106561e-08, 2.432640258120955e-07, 6.2087934793453314e-09, 3.710389773203815e-09, 3.0969925290946776e-08, 1.8517193334233184e-09, 1.1529733455972746e-05, 3.077837540388373e-09, 3.307177109945769e-08, 1.7787649994716048e-05, 1.5640956974039e-07, 1.0456997934227275e-08, 4.944510426097004e-11, 3.341688625368988e-06, 1.1010757861029674e-09, 3.3585149594728136e-07, 5.656510371210288e-09, 3.7800202790094772e-06, 1.0592846599877248e-08, 3.080116357523366e-07, 5.740517963204184e-07, 6.400927565763936e-10, 2.5506544716336066e-06, 4.930924113466517e-09, 3.477692303022195e-07, 2.449958458328183e-08, 2.126876097463537e-08, 3.3420294309749465e-10, 2.0549322243823553e-08, 1.0400996904025295e-11, 1.0264698602213684e-07, 6.294670118478507e-09, 2.3891539058240596e-07, 1.8071261365548708e-05, 8.865399081514624e-08, 1.3855705560672504e-07, 3.8741305274925253e-07, 9.324089944584557e-09, 2.0824411421926925e-06, 1.9214486002283593e-08, 2.652887758358702e-07, 1.2037916219398426e-09, 2.5182731633321964e-07, 2.810158150623465e-07, 1.2727102927101441e-08, 0.00014314446889329702, 2.92795352407893e-08, 1.2695588338829111e-05, 8.481574695906602e-06, 1.1514231346154702e-06, 1.037145409554796e-07, 8.708470056184581e-10, 7.530574208658436e-08, 1.0802026828571343e-08, 4.7268900971175754e-07, 1.426796416126308e-06, 2.774543039762989e-09, 0.000656551739666611, 5.928085738560185e-05, 2.35803199188922e-09, 3.6847477957735464e-08, 1.23104740623603e-07, 3.314520711228397e-07, 2.7101134492113488e-06, 1.0860672361445722e-08, 3.7087464210827648e-09, 2.288035183539705e-08, 2.734500640144688e-06, 1.194246669911081e-05, 1.0911734534602147e-05, 2.016314448738399e-09, 3.0301443985081278e-05, 2.7172969652156098e-08, 3.768467760068006e-09, 0.0001825218932935968, 7.86887255799229e-07, 3.027559358770304e-08, 3.7461899182744673e-07, 1.0410793947812635e-05, 9.93862727227679e-07, 5.42974056827461e-08, 1.2428739864844829e-05, 0.00017967339954338968, 9.818312829423803e-10, 9.48221611452027e-07, 6.45463060777729e-08, 2.572634549835584e-09, 1.4998261121945689e-07, 9.532429533221087e-12, 1.060636134475601e-09, 3.987350737588713e-06, 1.5067178537719883e-05, 2.702630581552512e-07, 1.0088311297451469e-09, 7.53251754304074e-07, 4.603233207944868e-07, 2.551839628495145e-09, 6.479193292996399e-10, 5.371358113848146e-08, 2.6881922465804564e-08, 1.409242145200551e-07, 5.214446652246352e-09, 1.9182100352566067e-09, 2.5934443215192005e-07, 1.0908979675150476e-06, 6.415052098418528e-07, 1.029333930091525e-06, 7.740519322396722e-06, 5.607533637430606e-08, 1.7153375608813803e-08, 6.928881912671159e-10, 3.732089914887382e-10, 1.8186128727393225e-05, 5.6651401791896205e-06, 1.8291670400572002e-08, 1.0678763828764204e-06, 2.6513563966545917e-07, 4.3636640967292806e-09, 3.255772895727205e-08, 3.0113242246443406e-05, 1.8570982263099722e-07, 5.474083536682883e-06, 1.5716139500909776e-07, 5.592277751986785e-09, 9.485788723395672e-07, 1.2330585263953253e-07, 1.0670004485291429e-05, 4.995113656036665e-08, 7.098380592651665e-05, 1.8269645796209488e-09, 5.7510892048640017e-08, 7.148715508442649e-10, 8.070920785030467e-07, 0.00014658419240731746, 1.199786993311136e-06, 7.751621566853828e-10, 2.1280096916598268e-05, 7.326601547674727e-09, 5.204798799240962e-05, 2.450675495868637e-10, 2.3236295874085044e-06, 2.984366642699854e-10, 2.520698387797893e-07, 1.2411489613839422e-08, 8.8071527848399e-10, 2.8814190500270342e-06, 4.908493878019726e-08, 8.471939594301148e-09, 6.069647184858695e-08, 6.686314009130001e-07, 8.277568849734962e-05, 1.7065966417817435e-08, 8.263564410526669e-08, 3.587867425380864e-08, 1.6404659852753412e-09, 6.067760960348778e-09, 1.137712035692573e-09, 9.013276212499477e-06, 1.796956610178313e-07, 1.0410351336531676e-07, 3.3540669619469554e-07, 1.417274742010477e-08, 6.832192866568221e-06, 2.4903843950596638e-05, 1.447016534372736e-10, 6.3383304222952574e-06, 0.00015766086289659142, 1.0599220168217016e-08, 1.0855803793674568e-06, 3.5565750522437156e-07, 2.50382272497518e-05, 3.2677090189281444e-07, 8.167216347665374e-10, 1.9654105187782989e-07, 3.079758926105569e-06, 3.443199602770619e-05, 1.5571808376080298e-07, 4.77509365381934e-10, 1.2228673540448654e-06, 1.741717460390646e-05, 3.659925917887108e-09, 1.7333342539060936e-09, 3.359836284744233e-07, 5.004064291824761e-07, 1.2942261662374221e-07, 6.411693842522936e-09, 1.812857064464879e-08, 1.1664583965398378e-08, 2.8243292149454646e-09, 4.2646614019759e-05, 3.6552563642544555e-08, 1.8034903348507214e-08, 8.106857194434269e-07, 2.1679460360246594e-07, 1.6998023056657985e-05, 9.958016633504485e-10, 5.119989054946927e-06, 2.986040747998686e-09, 6.849742931080982e-06, 4.904539885330905e-09, 1.1971093954343814e-05, 8.417420360729011e-08, 7.5186171670793556e-06, 1.1959527812521742e-09, 6.622433829761576e-06, 0.00015062869351822883, 3.213709760530037e-06, 5.521914303585618e-08, 2.976513258090563e-09, 2.020064312091563e-05, 1.950322001675886e-07, 1.588432851917787e-08, 2.3943496074707582e-08, 0.0001048164049279876, 5.719256932934513e-06, 2.2106980395619757e-05, 4.136426650802605e-05, 2.097290563440879e-09, 3.3682814315483256e-09, 6.562766685647148e-08, 1.4940209780434088e-07, 2.2315352055102267e-07, 1.5527456298514153e-07, 3.026725380550488e-06, 9.082221197331819e-09, 0.0006638685590587556, 1.4597078212830183e-08, 2.0856720794881767e-09, 1.5709343825776045e-09, 4.530337882613367e-09, 2.2112564224130438e-08, 5.6106188139892765e-08, 9.259939588446287e-07, 1.2499577906055492e-06, 9.077615686692297e-06, 1.6672904834535984e-08, 1.1843152606161311e-05, 3.464181119738896e-08, 3.751827364339988e-07, 3.351064776779822e-07, 1.1975355391768971e-06, 1.9558603980840417e-07, 3.545587601294642e-09, 2.3587153563653374e-08, 7.093449161388321e-10, 5.397168933995999e-06, 8.515061011848957e-08, 6.1715568335785065e-06, 2.4018715905071986e-08, 2.6054772561678874e-08, 1.6668242096784525e-05, 7.742167440483172e-07, 8.461612743815294e-09, 4.441901637619594e-07, 1.8254290523600503e-07, 4.681887944002483e-08, 4.814737053493445e-08, 5.328669949733467e-09, 6.74459446625697e-11, 5.533151579584228e-06, 1.4715977048052054e-10, 5.576088213388175e-08, 4.940934559272137e-07, 7.683118141699197e-09, 4.75538254249841e-05, 7.319436168273796e-09, 1.1877441465912852e-06, 2.459264294429886e-07, 4.5730420339396005e-08, 1.893054424328966e-08, 1.0743710845417809e-06, 9.629484338802285e-06, 7.128589004423702e-06, 6.466068569466188e-09, 0.00035487316199578345, 1.5666921093782094e-09, 3.328899038024247e-05, 1.5776537853895434e-09, 2.4129066744649208e-08, 9.359445698464697e-07, 5.533352664954805e-10, 2.4864978964700413e-09, 3.468558062991178e-08, 4.444100554223951e-08, 4.243192677222396e-07, 2.717903946347633e-08, 2.468253114784602e-06, 2.951221933500392e-09, 2.6235680294917074e-09, 6.302475230768323e-05, 4.679752696468142e-10, 9.352279448648915e-05, 4.475458865726978e-07, 1.6028435523285367e-10, 1.21296583888153e-09, 1.2462667200452415e-06, 2.759083599812584e-06, 1.496683921686781e-06, 1.254536741157608e-08, 2.1306600928028274e-08, 6.183805911774698e-09, 5.0912944971059915e-08, 1.641298297272442e-08, 3.046925939997891e-06, 5.172729288460687e-06, 8.859273066263995e-07, 1.5775485735503025e-05, 2.5268219516760837e-08, 9.642668885589956e-09, 1.8797406937665073e-06, 9.349382601442358e-09, 5.411868642113404e-06, 1.2521397252385214e-07, 1.982149555601609e-08, 1.608504973660274e-08, 4.187505453501217e-07, 5.125125657201579e-09, 1.8719916283771454e-07, 1.3873210264137015e-05, 2.7801597468624095e-08, 3.6266681036067894e-07, 1.6924550649832781e-10, 2.4685725747986e-06, 0.11907929182052612, 6.669160029559862e-06, 0.00017392326844856143, 1.6843969552482463e-09, 8.458276852252311e-07, 1.7860097045740986e-07, 1.1161266229464673e-05, 1.6285795254589175e-06, 7.248916489288604e-08, 2.0495174446466535e-09, 8.951224117481615e-08, 5.503701231646119e-06, 4.302648903831141e-09, 0.0045274426229298115, 1.2270737315134284e-08, 2.9950852908910974e-09, 2.4250280006299363e-08, 1.1982617564854081e-08, 0.0010111661395058036, 0.0006454950198531151, 2.3381131697419733e-08, 1.2479043789426214e-06, 7.756952413728868e-07, 0.0039713201113045216, 9.318959200754762e-06, 3.156651384728093e-09, 1.1504163921927102e-05, 8.189644518097339e-07, 1.0605737088553724e-07, 5.461264208861394e-06, 2.001292020992196e-09, 5.9312611710993224e-08, 3.476782012512558e-06, 8.81248585216099e-09, 1.6232245343417162e-06, 6.467183766289963e-07, 4.420474056243684e-11, 1.3127753391017905e-06, 2.6360652549684005e-10, 7.184636342572048e-05, 6.513360406579238e-10, 6.5989316233583395e-09, 3.2418729922767398e-09, 7.0680590269489585e-09, 2.2383925113445002e-07, 1.0696232094487002e-10, 5.980573547503809e-08, 5.034720015828498e-05, 3.852639995222118e-12, 6.122910178874008e-08, 0.0001567476283526048, 1.1324433444315218e-06, 6.6768897077906786e-09, 2.2670405996905174e-06, 4.192377556222482e-09, 3.868169429210866e-08, 1.8674676027785608e-08, 2.5839466388788424e-07, 7.597090068145462e-09, 9.02406554814661e-06, 5.0457779252610635e-06, 3.61745769623667e-05, 9.47619440694325e-08, 1.1883846684490607e-10, 8.804868656397957e-08, 1.7409183428185315e-08, 2.0695154034910956e-08, 7.455454031912723e-10, 1.4321302899134025e-07, 3.842261619979581e-08, 2.740244475774034e-08, 7.507800319217495e-07, 4.835213758269674e-07, 1.7477440223956364e-06, 1.0594598809365863e-10, 3.497490297377226e-08, 1.3624102734866028e-08, 4.7153528726084915e-07, 2.3238780344492227e-10, 1.1197570870535856e-08, 2.242695096299485e-08, 1.4956581112501866e-11, 3.210875831882731e-08, 1.6970574279184802e-08, 8.110522031756773e-09, 1.2002436544644723e-10, 1.30923808683292e-07, 0.01972424052655697, 2.0077617790548175e-08, 1.982617625628791e-08, 1.5786141602802672e-06, 2.0462807837073882e-10, 4.0287159208673984e-07, 1.869751827143773e-06, 6.254135951166973e-05, 2.529917082938482e-06, 4.53311940873391e-06, 9.58558530328446e-07, 7.354165193484974e-11, 5.093974309033911e-08, 3.751766453063965e-09, 6.790539686107877e-09, 1.0967672778861015e-06, 9.326209351456782e-07, 4.8452847067892435e-08, 1.1327062843236035e-08, 0.0008999126148410141, 2.290957183959108e-07, 1.665000914918835e-09, 2.5366339286847506e-06, 3.5844951895569466e-08, 4.756848070996966e-08, 6.65384641251876e-06, 2.8069149266229942e-05, 5.478970610539591e-09, 1.8512793076297385e-08, 4.30973777838517e-05, 2.9101767870520767e-11, 3.9236276450083096e-08, 3.1001590627965925e-07, 4.4549799405047e-09, 1.2606813015736407e-06, 1.8026558734618447e-07, 2.427302376872831e-07, 0.0001124279442592524, 1.6569089211770915e-07, 0.0006384842563420534, 7.045030088193016e-07, 1.0308123421509663e-07, 1.0531240235422956e-07, 6.844800992666933e-08, 3.3231046359105676e-08, 1.572302971908357e-05, 4.681758092317523e-09, 2.5555548788247506e-09, 1.2378446358207729e-11, 3.2969456498221916e-08, 2.448465430404667e-08, 1.4637394940564263e-07, 3.714687224487534e-09, 2.5993278640612516e-08, 4.9972014373733664e-12, 8.040311527679478e-09, 2.5549010129743976e-10, 1.9826711650239304e-06, 9.932020930136787e-07, 3.269923709581235e-08, 2.341072613543105e-10, 8.095998871304744e-11, 6.257784207264194e-06, 8.256186845301272e-10, 8.746831384187637e-12, 3.4583753638628423e-09, 2.0738978491863236e-05, 0.0054474323987960815, 4.26778334894351e-10, 1.5487040627704118e-07, 1.326654796685034e-06, 0.0005949522601440549, 1.904907776406617e-06, 1.7993539813687676e-06, 8.411018939114001e-07, 1.760019898711107e-07, 1.2623787570831269e-09, 8.152810551109724e-07, 2.05283527066058e-06, 6.713833499816246e-06, 1.2662787485240301e-11, 1.4588357544198516e-07, 2.6237387373839738e-08, 6.686466349492548e-06, 8.180582517525181e-05, 1.325577758670704e-09, 8.104079629589478e-10, 2.922704425145639e-06, 2.554506181695615e-06, 9.636417530600738e-08, 5.00712360462785e-08, 3.179061014435547e-09, 6.01830696389527e-10, 2.2421554390916754e-08, 1.5798920003362582e-06, 0.0013060403289273381, 1.4880129128869157e-07, 6.72243727350974e-09, 3.681724933812802e-07, 6.145752558950335e-05, 8.529346473551414e-07, 6.295935861544422e-08, 1.0675805697246687e-06, 4.952954623149708e-05, 9.275253454177346e-09, 3.227551559348285e-08, 4.564716618915554e-06, 9.528666744529346e-10, 4.205441825888556e-07, 2.9112002266629133e-06, 8.584312816140027e-09, 8.363874570704866e-08, 4.5835144707462305e-11, 1.9046054148930125e-05, 2.982217850444613e-08, 1.1704159987857565e-05, 3.032545282621868e-05, 3.370777676536818e-06, 7.71302222091208e-09, 2.02685987460427e-05, 5.865604180144146e-06, 4.068495400133543e-05, 1.1017132806045993e-07, 4.291703589842655e-05, 3.797264447324977e-10, 1.582467668015397e-08, 1.9992401067980836e-09, 2.517961616987918e-09, 6.360127144944272e-07, 6.727350410073996e-07, 1.4000683279391524e-07, 2.313439182444199e-07, 4.788656138288161e-09, 3.046823394470266e-06, 4.537700988294091e-06, 5.934905402682489e-07, 2.6925688345613707e-08, 2.4068421922152083e-08, 7.148084080199624e-08, 1.0486520984898107e-08, 4.030134093113702e-08, 6.338662501548242e-07, 2.7184469786334375e-07, 1.359322681082631e-07, 6.122531459595848e-08, 1.6761315946300925e-10, 1.1647749715848477e-06, 1.528209736534336e-08, 4.7425919547094963e-07, 2.4442670110147446e-12, 4.32294598340377e-07, 1.1136555677637716e-08, 2.2504892172037216e-07, 1.2994741155125666e-05, 5.045827151661797e-07, 8.960095035717708e-13, 8.76834917562519e-07, 1.6543070247010405e-09, 2.8136901164543815e-07, 2.1454222842720583e-09, 1.0690030194382416e-07, 3.8952623349075566e-09, 4.724892733065644e-06, 3.3767555418506845e-09, 2.989278513609861e-08, 1.8735410378667439e-07, 1.0482353873797479e-09, 5.4917904890317e-07, 1.5091850968929066e-07, 2.656298420333769e-06, 8.856622457642516e-07, 3.4545152849396743e-10, 2.25819803745253e-06, 2.3402019166951504e-07, 2.431471784802852e-06, 2.431341101782891e-07, 1.329607673961064e-10, 8.795962180840888e-09, 5.641807507283936e-11, 1.0520159188232014e-09, 3.374024686308985e-07, 4.84820191104518e-07, 1.731278764793842e-08, 2.4340252480214986e-07, 2.0474330995057244e-06, 1.827318101277342e-07, 3.7822243115215315e-09, 3.004298676501094e-08, 2.383530928184996e-09, 6.829166977695422e-06, 4.990817203953668e-10, 5.14697376274853e-07, 2.454431324849793e-09, 1.2500521506808582e-08, 6.2666880573658545e-09, 2.334928268510339e-07, 3.924484644812765e-06, 6.797270907554775e-05, 2.0387790300446795e-06, 4.732102070192923e-07, 7.423382797355771e-09, 5.157960725199473e-09, 3.345660459785904e-08, 1.6029791538585414e-07, 8.601407780473025e-12, 0.0003699554654303938, 2.866264878775837e-07, 2.4239643181545034e-08, 2.0011777905892814e-06, 7.320919426234695e-10, 5.2897430435905335e-08, 1.1340880519128405e-05, 0.00017264032794628292, 6.909093599460903e-07, 6.368968463732472e-09, 3.3468691071902867e-06, 8.373389981386481e-09, 1.758099728021989e-07, 1.2267552307321239e-08, 0.00012771364708896726, 1.9405698026275786e-07, 3.2278200112756394e-09, 1.2385449998930653e-08, 1.0275895192535245e-06, 1.5331557733588852e-05, 2.697055734302012e-08, 2.829469769594084e-09, 2.381334296419624e-10, 1.653345833574349e-07, 2.53060079558054e-07, 9.240868337201391e-08, 4.255169301359274e-07, 5.503349997937335e-10, 1.1950400391480542e-10, 3.5080802263109945e-08, 0.00010512754670344293, 7.159302185755223e-05, 4.149935193709098e-06, 4.6371464122785255e-05, 1.865506860099231e-09, 1.6189552809464658e-07, 1.276531293115113e-05, 5.303957451019414e-08, 3.1002450384676195e-08, 1.2738462729089406e-08, 2.6229400873489794e-09, 1.9405460989219137e-06, 1.6731938057290563e-09, 6.322363077515547e-09, 3.0854794275114727e-09, 3.2711717778965976e-08, 6.646675387855794e-07, 9.81300800617646e-08, 7.1759608033517e-08, 1.4973684869801218e-07, 1.970295748776607e-08, 4.079695827385876e-06, 2.4238293008238543e-06, 0.0006228057318367064, 1.3934000619997278e-08, 9.790878330306896e-09, 0.004168656654655933, 4.4086164052714594e-08, 2.8634946147576557e-07, 1.5106452622148936e-07, 2.4752843871822527e-11, 1.43545412356616e-06, 4.790980806523848e-11, 1.9737660394980594e-08, 5.848926898366358e-11, 1.2921125858156302e-07, 9.665549471904455e-10, 6.251619311115064e-08, 3.3620146666635264e-08, 3.538748627462951e-09, 3.045051562367007e-05, 2.0316704194556223e-06, 4.4281472932894417e-10, 2.0028371322533545e-12, 3.9946630181475484e-07, 1.8862712780620683e-10, 2.7092541543538573e-08, 3.837455153643532e-07, 5.451167695014192e-08, 9.825295244070276e-09, 8.41424636632837e-08, 4.242467488424495e-11, 1.1046874526243755e-08, 5.865081789124815e-07, 1.0087470627695438e-06, 1.0368832192853006e-07, 4.0434102288600116e-07, 3.38258445253814e-07, 1.083606804286319e-08, 2.881677785282477e-09, 3.290888950235882e-10, 6.242839578618486e-10, 1.7177093241116381e-06, 1.3068508053493133e-07, 6.607804969860354e-09, 1.6243385303482683e-08, 1.1240760544595219e-09, 2.6498928051843507e-11, 9.082037877305993e-07, 3.1613055853085825e-06, 7.759580200605853e-10, 1.0825686180826466e-10, 1.6282700698866392e-07, 3.66430374931781e-09, 1.2077395972198701e-08, 3.010129034919373e-07, 1.9898459413525416e-06, 2.6970124963554554e-05, 3.3337214517814573e-06, 5.226588086770789e-07, 9.60043351483364e-08, 1.893608168757055e-05, 7.35173921739829e-09, 3.765107521758182e-06, 0.00030134586268104613, 4.050099232699722e-05, 2.256796506117098e-05, 3.2031035601676194e-09, 9.960591569324606e-07, 2.8642367055908835e-07, 1.8959825709430334e-08, 2.903561266975885e-07, 6.879676689441894e-11, 1.0039076236978417e-09, 1.0629814823914785e-06, 3.194953857033056e-09, 1.0306882359145675e-05, 1.3674207322011966e-09, 1.1261956842645304e-06, 1.4702554551604408e-07, 3.9244213212441537e-07, 1.3862110108675552e-06, 2.7872857799593476e-08, 2.5833576944300773e-10, 2.217360162148907e-08, 2.4492209475812654e-12, 8.444589911960065e-05, 1.3535917787521612e-05, 1.4318353578346432e-06, 3.9294512532705994e-08, 7.2299259912256275e-09, 2.888424777136578e-10, 2.37263950708666e-08, 3.16658292831562e-06, 5.6687241567487945e-09, 1.0125052085641073e-06, 3.386987579290235e-09, 3.5461474112707947e-07, 1.3833159755449742e-05, 4.750549578602659e-06, 6.630872066704097e-13, 1.676872852840461e-05, 1.349124318039685e-06, 3.5878480186823936e-09, 2.461929852870526e-06, 3.028856809805802e-08, 4.431490907563784e-08, 1.6016763765946962e-05, 6.0100542214058805e-06, 2.979035343741998e-05, 1.3573318256021594e-06, 0.0048540555872023106, 3.437568466324592e-06, 8.027290498979767e-10, 1.7853362521691452e-07, 1.7081743797220383e-09, 5.0386434607974095e-11, 2.9067896534229476e-08, 3.858938057987871e-08, 1.0612055234560103e-08, 1.3837434380548075e-06, 3.216746691236949e-08, 3.731646813776024e-07, 1.5627002767359954e-06, 6.39657926626569e-08, 3.0354558475664817e-06, 2.3026687223648423e-09, 6.689421239158833e-11, 0.00010752568050520495, 2.685708322403002e-10, 2.395999842974561e-08, 2.936646126272535e-07, 1.0040009795764249e-10, 1.3560786271682446e-07, 2.849277052519028e-06, 2.6438261556904763e-05, 1.6237514444128465e-07, 2.514357255734012e-08, 4.375191320349359e-09, 3.02175294564222e-07, 1.5737528258341626e-07, 8.176988600117063e-12, 2.907830642939757e-09, 4.419688082180073e-07, 9.083207714866148e-07, 6.084393362471019e-07, 8.55471080285497e-06, 3.700961759278698e-10, 1.763827867762302e-06, 1.0881794132444611e-08, 7.406656621355978e-09, 7.42657206842523e-08, 2.894745276815769e-10, 1.0519241300244175e-07, 9.60816123551922e-07, 7.170752880369946e-11, 2.0642228204525992e-10, 1.0148844264978862e-10, 3.6952462778572226e-07, 3.015195488842437e-07, 1.0103721193033266e-09, 2.119112611081997e-12, 5.603313013580191e-08, 0.0005328659317456186, 1.6413704839735033e-09, 4.322588065264199e-09, 3.5362235362157435e-09, 8.375618421041509e-09, 1.7735876099322923e-05, 7.085646652749134e-11, 1.7143203523417583e-09, 3.861914787961496e-09, 1.689901907298008e-08, 6.255125981624587e-07, 4.5587777997013745e-09, 1.2110218676752993e-06, 6.241044161470199e-07, 1.5036070166729587e-09, 3.7478801573342935e-07, 4.1815387263888226e-10, 2.5379418618770577e-12, 1.5510863704548683e-08, 3.1200370358419605e-06, 3.1480689358431846e-05, 1.0848253850781475e-06, 5.748607145861229e-10, 1.1781724928994208e-08, 5.920083481214533e-07, 6.090543536174664e-08, 0.01116254460066557, 3.8573935712271634e-12, 9.537372003265432e-10, 2.4456309688503097e-07, 1.2123793722423892e-12, 0.0017551026539877057, 1.8811947199992574e-07, 1.1047646353290475e-07, 3.339213435538113e-05, 8.238265536419931e-08, 1.3850880442589641e-08, 4.698422344517894e-06, 4.0966239112094627e-07, 1.1874745723616797e-05, 7.786685740640564e-10, 4.794309279532172e-05, 3.1261873090215886e-08, 2.5119688871200196e-05, 8.104586335377917e-09, 2.9511556931538507e-07, 1.3904331908065615e-08, 0.0001805866922950372, 2.023508649529049e-08, 5.623085712613829e-07, 4.121341135032708e-07, 2.000030349669224e-11, 3.805025244218996e-06, 1.7583397493581288e-07, 1.5967996660037898e-05, 1.0994420929222315e-09, 1.3388594766183814e-07, 8.847204746587067e-09, 2.1137751549105133e-09, 1.1278304867801126e-07, 0.00010300967551302165, 3.9234386406405974e-08, 2.202299592690693e-10, 7.393753664874936e-11, 2.2831374568710316e-09, 1.433260058547603e-05, 6.657077999960848e-09, 1.5752894100273807e-08, 2.6023883037851192e-05, 1.2599147503067343e-08, 5.8324067140347324e-06, 7.336135467994609e-07, 7.475271512902282e-10, 1.1148937772986756e-08, 1.439229208699544e-06, 1.1037555935899945e-07, 1.4983543572544278e-10, 1.689275791250111e-06, 1.0014158760895953e-05, 3.19368155032862e-05, 6.583683216376812e-07, 7.793256595611808e-10, 2.6642945272215e-07, 1.591630272024247e-09, 6.5419008024036884e-06, 1.6386120194056275e-07, 1.7897085854201578e-05, 1.1234756414069125e-07, 2.987726288594672e-09, 0.002518832217901945, 1.2133858717788826e-07, 1.985659692227415e-10, 1.2680184013902362e-08, 5.384153678456016e-10, 7.314481798026407e-10, 1.4805709724896587e-07, 8.4558060553519e-10, 1.6657915935525125e-08, 1.681838279754544e-10, 1.4788186035730178e-06, 7.188991730799899e-07, 2.7573923588875004e-08, 1.6444569664320063e-11, 6.2202457229432184e-06, 3.7108666139928914e-10, 1.890634393930668e-06, 2.5957746174753993e-08, 5.446459727664887e-09, 0.0003658958012238145, 4.4935075038665673e-07, 7.392795851046685e-07, 2.033790451605455e-07, 6.458013785959338e-07, 8.949006513603308e-08, 1.5865171676399115e-10, 1.4544625059897953e-08, 7.775135535403876e-10, 9.104948549065739e-05, 1.1742781191870222e-09, 5.563523686191729e-08, 1.1978684710811649e-08, 2.0867733496743313e-07, 1.1388187282079798e-08, 8.919557330955286e-06, 9.854897342620461e-08, 1.000180063215339e-07, 1.426610651833471e-06, 1.264933047195882e-09, 1.1405580480072786e-08, 5.547267159511371e-11, 1.333570764927572e-07, 1.5190554449873162e-06, 3.6057144825463183e-07, 8.624449492344866e-07, 1.0773861504276283e-05, 4.677878351344589e-08, 2.6981614720256175e-09, 3.2504837577107537e-07, 1.4368106349138543e-07, 9.507648712769878e-08, 2.0029960069223307e-05, 4.525353142526001e-05, 9.131828119279817e-05, 2.599915660539409e-07, 0.0001853863795986399, 3.3794813134591095e-06, 1.2447386325220577e-05, 2.9252873900986742e-08, 1.5467954694869945e-09, 3.429590478276623e-08, 1.6809828196073795e-07, 3.514101990731433e-05, 3.2739339985710103e-06, 3.5368009321246063e-07, 4.3809734506794484e-08, 7.68425138630846e-07, 1.682115424728181e-07, 5.435277330434474e-07, 2.057992887216642e-09, 9.502437059438762e-09, 1.2324699127930216e-05, 3.183104058113173e-10, 2.5695777594592073e-07, 7.852039157763002e-11, 3.820099347962014e-09, 3.5207515338697704e-06, 1.0324563390895491e-06, 6.033498038959806e-07, 7.382331546068599e-07, 1.1499322383201616e-08, 2.056183001641898e-11, 1.13448301419794e-07, 4.84845941173262e-07, 4.406807363466214e-08, 3.6573681860829765e-09, 2.823439260168925e-08, 4.981324082109495e-07, 2.263720677922465e-12, 4.577864198829218e-10, 1.2679664678216795e-06, 2.7436612981546205e-07, 4.248941820605978e-08, 6.731673238391522e-06, 9.091639441294319e-08, 4.035719484818401e-06, 0.00028441380709409714, 4.889625060400249e-08, 3.298435785836773e-06, 6.737996045558248e-07, 5.832817562279047e-11, 1.8648798061349225e-09, 7.464420193059595e-09, 3.745390131371096e-05, 4.877175996398364e-09, 4.973238176830819e-08, 1.4267548031909882e-08, 5.217866069773258e-11, 9.381919312545506e-07, 4.037501000198063e-08, 7.612407443957636e-06, 6.690993359370623e-08, 2.945359938166803e-07, 1.5121858041311498e-06, 1.0553980246186256e-05, 4.405349500302691e-06, 4.099349748543091e-09, 1.0164711966353934e-05, 4.253182694924362e-09, 2.2326621547108516e-05, 5.333439503374393e-07, 3.369903911476513e-09, 4.827124477913003e-09, 4.333924152888358e-06, 3.8143660674450075e-08, 2.418635176582029e-07, 6.06200458719286e-08, 5.171818338567391e-05, 5.953896486943222e-09, 2.9186804795244825e-07, 4.7186473478566526e-11, 4.471802981242945e-07, 1.0273814021743988e-09, 1.9425218766855323e-08, 0.0002893178316298872, 1.0615221981424838e-06, 1.79645431863662e-09, 7.686598735290318e-08, 2.595665193894092e-07, 8.903446868657738e-09, 3.2172636110772146e-08, 5.194756291793112e-10, 5.551393405767158e-05, 1.4417179450276763e-08, 5.030278202866612e-07, 6.363977718137903e-07, 1.2554831982924952e-06, 5.409589221017086e-07, 1.4842236851109192e-05, 4.389988650377319e-10, 2.2183332930580946e-06, 6.059719453332946e-06, 8.253726591078703e-10, 2.5758698285471837e-08, 2.0576889525614206e-08, 6.081136660895936e-08, 3.65092528227251e-05, 2.7846324428537628e-06, 4.73609304663114e-07, 6.598128834411909e-07, 3.545295723661468e-10, 1.7102349374908954e-05, 4.035913381272849e-09, 5.1549086776958575e-08, 8.67591006681323e-05, 3.4438947249526564e-09, 2.7762904822314916e-11, 7.835825499569182e-07, 1.938522018463118e-06, 5.766570154719375e-08, 6.867544288979843e-05, 1.9543982432423945e-07, 2.7152714210387785e-07, 3.000411652465118e-06, 7.368072263425063e-10, 9.042190107777515e-09, 3.709574514232372e-07, 1.0215965176030295e-06, 0.0003790976479649544, 5.595782504030922e-09, 7.086106279530213e-07, 1.502012914045281e-08, 2.9776425769512116e-09, 8.735229494050145e-05, 5.530866928893374e-07, 1.0366212777057626e-08, 7.189917960204184e-05, 1.3718153002439948e-10, 4.239358167978935e-05, 1.9177596399799768e-08, 7.311591332381795e-09, 2.2091393159229256e-09, 9.377581022818049e-07, 5.923149046793696e-07, 1.8563330286269775e-06, 9.671833473001712e-11, 2.3794198568793945e-05, 1.1175894343296022e-07, 1.0115574013980222e-06, 4.694378132746868e-12, 1.4450708185620442e-08, 3.286609455699363e-07, 2.3294122001971118e-05, 4.531370336735563e-07, 7.706080396019388e-08, 3.184906560704803e-08, 4.4729826953471274e-08, 9.709911682875827e-05, 0.0005701495683752, 1.2701920626412289e-09, 4.5886696398156346e-07, 0.0002564892638474703, 9.152931284006627e-07, 1.236345923416593e-07, 1.975881502858101e-08, 5.3130258947931e-07, 0.00606018491089344, 2.851964211458835e-07, 6.050920653422054e-09, 3.3429131462980877e-08, 9.532746503282397e-07, 3.540846137184417e-06, 5.71703765217535e-07, 5.0396760542525953e-08, 4.375682749468979e-08, 1.8048670824555302e-07, 1.1275083977579925e-07, 0.0001248783664777875, 9.715971600599005e-07, 2.9897367426201527e-07, 2.6756337319966406e-05, 5.59272621103446e-07, 1.8398785641693394e-06, 7.233151577690222e-11, 5.620907472803083e-08, 9.233903028871282e-07, 3.742693266417518e-08, 4.8697373244976916e-08, 1.4075476428843103e-05, 1.4337383502649459e-09, 7.285983087967907e-07, 6.306079569640133e-08, 9.728967631517094e-11, 2.9251863153945123e-08, 5.556665172434805e-08, 7.695670767304819e-10, 1.3606779248220846e-05, 1.1576097413978914e-08, 1.3409311883094688e-08, 1.1294933983663213e-06, 1.4963738976803143e-05, 3.373115953220207e-10, 1.9202221324121638e-07, 2.223667024736642e-06, 4.0304581894190505e-09, 1.2845730452681892e-05, 7.473668665625155e-05, 5.985336315461609e-07, 2.5707868189783767e-07, 7.993670806172304e-06, 1.7153578113493495e-08, 1.1951984957292439e-09, 2.423436451515215e-11, 2.967307510814976e-10, 1.6572163641370707e-09, 0.0009464782197028399, 3.7652707618462955e-08, 2.500614471045992e-08, 7.767048941786925e-07, 2.1574571746896254e-06, 7.79907083359177e-10, 3.8299032212307793e-07, 1.0019580258813221e-05, 5.246358796284767e-07, 2.4044064517170227e-09, 3.5923881114285905e-07, 0.00044418920879252255, 1.632758903724607e-05, 1.4053184713702649e-06, 7.011033449089155e-05, 3.324693534523249e-05, 9.715083182371131e-10, 2.3043605324346572e-05, 4.3602872779047175e-07, 3.5370524287259286e-09, 6.459461410202039e-09, 3.289003203121865e-08, 8.534311746188905e-07, 1.883585021289491e-08, 6.320258398773149e-05, 1.9397008443888808e-08, 3.354599584781681e-07, 6.937891150471387e-09, 2.417784417119151e-09, 1.504334790070061e-08, 8.612213164838067e-09, 3.0439639697732446e-09, 7.910397243904299e-07, 3.481691237539053e-05, 4.843269039156439e-07, 0.0003721476241480559, 1.4765504374736338e-06, 1.4433821604598052e-07, 1.637352761463262e-05, 6.142033548428572e-10, 4.4434799589820884e-11, 8.687374020155403e-08, 9.282938117394224e-05, 8.281870123028057e-08, 4.7151735316219856e-07, 1.2672339835262392e-05, 0.00032159764668904245, 3.507377499545328e-08, 4.665797748160116e-10, 7.035975690961038e-10, 8.703044329649856e-08, 4.251992535841964e-09, 3.422801455599256e-07, 5.39465304427722e-07, 1.041239343635425e-07, 4.529121270024916e-06, 5.5704411749957217e-08, 0.004493859596550465, 1.2320520159647685e-09, 3.973035745730158e-06, 2.3720778230540418e-08, 6.906691396579845e-07, 1.0822563678569708e-10, 1.2132720257795881e-05, 3.068605636258326e-08, 4.869352324021747e-06, 0.00011138383706565946, 4.670628328540261e-10, 4.6850583856894445e-08, 3.6193262076267274e-06, 6.61377134747454e-06, 1.2672972671268212e-09, 4.101455033378443e-07, 2.1967873180983588e-05, 7.923321163616492e-07, 9.079496408048726e-07, 2.2114154774044437e-07, 3.2258424198516877e-06, 1.3376219385463628e-06, 6.46859565911484e-10, 2.930993787231273e-06, 8.848402899275243e-08, 6.83154366409866e-11, 1.5403389852508553e-07, 1.3295783141131778e-08, 1.7077701386369881e-06, 1.170582186738045e-09, 8.269762474810705e-08, 3.063498255073682e-08, 4.481022219948727e-09, 2.8068622426324907e-10, 1.2677958682871804e-09, 2.518708242860157e-06, 3.0505026415994507e-07, 4.07829370274726e-09, 5.961430815659696e-07, 8.803878699836787e-06, 1.585925133440469e-06, 5.533873945751111e-07, 5.9014350739516885e-08, 3.479171937215142e-05, 6.921188742126105e-06, 1.2897745371720504e-10, 5.394119995116853e-08, 2.7118149392713065e-10, 0.000458171300124377, 1.1456776499718657e-10, 8.183953468687832e-05, 1.2637414670280123e-08, 9.057155025971042e-10, 7.1482259045296814e-06, 4.83392270655969e-10, 1.0836832542437946e-09, 1.3823455269346141e-08, 2.9290022212080657e-05, 2.890511268560658e-06, 1.4598249435948674e-05, 1.5270576113834977e-05, 1.2596356846472645e-08, 3.046442316190223e-07, 8.935807471743829e-08, 3.046495180569764e-07, 3.9396013562509324e-08, 4.738178176921792e-05, 1.1333389693390927e-06, 7.010605873603026e-09, 2.1903702872805297e-05, 4.0230094811022354e-08, 4.354022031805016e-09, 1.8466822737650546e-08, 4.495267421589233e-05, 3.221412470111318e-08, 1.2389389958400443e-08, 6.912892480670507e-10, 3.171541038682335e-07, 1.3493150845533819e-06, 1.3936585219198605e-08, 9.504380926728118e-08, 1.69590279597287e-08, 0.0003323072160128504, 1.575821606536465e-08, 2.7232993637937852e-09, 2.642470349201176e-07, 5.205528008644933e-09, 1.6353769751731306e-06, 2.1966814074403374e-06, 1.723101838990715e-08, 3.101173007280522e-08, 0.00010278628178639337, 2.5370738843122354e-09, 1.659777204565671e-08, 0.00020141107961535454, 1.8119612477107694e-08, 1.9954871088856407e-09, 4.632208783217351e-10, 7.325401440994028e-08, 7.854140116059227e-10, 9.498313602307462e-08, 1.1856378101526843e-09, 8.08597633294994e-09, 2.070345090032788e-06, 2.1472070788064457e-09, 1.0343875374019262e-06, 3.8050129019806533e-11, 2.8461848433636305e-09, 5.802741043225979e-07, 2.0438660897070804e-07, 1.0389346982719871e-07, 6.101759026932996e-07, 8.942719342819316e-10, 7.538285444752546e-07, 1.068350474753288e-08, 1.2089169942886002e-10, 4.772339707415085e-06, 1.057483842714646e-07, 3.396824155288414e-08, 5.4603419563648e-09, 1.938995808359323e-08, 3.7627764015724097e-08, 2.076967244024619e-11, 3.308658449441282e-07, 0.011103788390755653, 1.9436912168657727e-07, 5.933478405495407e-06, 1.5737528883619234e-05, 1.5595747981933528e-06, 6.496989044535439e-07, 3.0745559342904016e-05, 5.046664455221617e-07, 1.7607628421956178e-07, 2.1416376512206625e-06, 1.326762321696151e-05, 2.1988134903949685e-06, 1.123619668419451e-07, 1.4181967822324282e-09, 4.8736368626123294e-05, 5.15388309807463e-09, 5.4779906832891356e-08, 5.640215516855562e-10, 2.8592737635335652e-06, 9.014303259391454e-08, 1.4106608148267696e-07, 5.308827683014883e-11, 2.199059245810986e-08, 2.709396653699514e-07, 8.634729056211654e-06, 8.808433982210317e-09, 1.2993452401133254e-05, 7.107708910325528e-08, 3.525742067722604e-05, 1.5086257917573676e-05, 8.071356916161676e-08, 2.7807821822989354e-08, 5.771438509327709e-07, 1.6609424946523177e-09, 0.00017508813471067697, 1.0249341819701385e-08, 1.196405605696782e-07, 1.9295964825971623e-09, 9.698147351855368e-08, 3.395861369881459e-08, 3.2902232049991653e-09, 1.8536310619765572e-07, 1.93018945537915e-06, 4.119213770081842e-08, 1.266447952730232e-06, 6.702234855993083e-08, 7.134373390726978e-06, 0.001047054654918611, 1.7665480189776872e-09, 1.458571773582662e-06, 1.4012532245644138e-09, 4.536368676533176e-12, 3.925430792151019e-05, 3.6498298072729085e-07, 5.230739930084383e-07, 4.7673361223132815e-06, 1.3149363553566218e-07, 3.3829391554718313e-07, 9.207696649582431e-08, 2.1478704184119124e-06, 4.645580702344887e-07, 3.9557426134706475e-06, 8.735361767264749e-09, 4.245630583210591e-11, 5.50749291505781e-07, 4.0371997300781004e-08, 3.4373607604720746e-07, 2.3744384236579208e-07, 2.759268147300986e-09, 3.021190242424865e-10, 2.57221972788102e-06, 3.115432264166884e-05, 1.1307930435577873e-05, 2.2716407954703755e-07, 1.3666481835095112e-10, 6.532241059176158e-06, 3.608532850307711e-08, 1.9352683011675254e-06, 1.6564230520543788e-07, 2.5940399828172644e-10, 8.75659825396724e-05, 3.1252088206201734e-07, 9.550859658702393e-07, 2.9721840988372605e-08, 9.37257216282461e-13, 1.9017303287682807e-08, 8.389940298059173e-09, 7.3956791801776944e-09, 3.660867832877557e-06, 9.694351224553976e-12, 5.958843871667341e-07, 1.436602730109371e-07, 3.488185257083387e-06, 1.5690043042582147e-08, 1.1190049775677835e-08, 1.3144581600954552e-07, 4.906439789920114e-05, 7.937750901021445e-09, 8.69263487857097e-08, 4.537279885141743e-09, 2.335538624720357e-07, 8.212787747652328e-07, 7.794241363434651e-10, 1.7811244106269442e-06, 1.0481123746686194e-09, 3.880203713890751e-09, 1.5853650836561428e-07, 6.873315072652986e-08, 3.981969420863152e-09, 1.8141824398298922e-07, 1.8385445343938045e-07, 1.0411063122717223e-08, 3.2364277924301632e-09, 2.358662820611812e-09, 4.250180651865776e-08, 2.66635102885715e-10, 1.672449684519961e-06, 1.19938631542027e-05, 3.6199391928448676e-08, 7.986039236129727e-06, 1.4495810319203883e-05, 5.757819963037036e-06, 1.4165612016725504e-09, 3.4084473021778194e-09, 5.163082406056674e-08, 5.692822213632098e-08, 9.45176736877329e-09, 2.23956112677115e-06, 2.5169445194705986e-08, 4.438294854480773e-05, 1.1982064052062924e-06, 8.295946827274747e-06, 3.586432910651638e-07, 1.4384167634773348e-08, 1.7162733456643764e-08, 8.39608027547456e-09, 9.083442886748117e-10, 6.288890119776624e-08, 1.0340977496525738e-07, 5.780744771755053e-09, 5.192960088606924e-07, 2.5119476276813657e-07, 1.9053831579185498e-07, 1.0203074651826682e-07, 0.0001788741210475564, 3.281688876199951e-08, 9.285439261930151e-08, 4.6842619667586405e-06, 4.441320197656751e-05, 2.3307222363655455e-05, 5.320618612358885e-09, 4.888026410299062e-07, 2.3698729023635678e-07, 2.8530442452989746e-09, 1.105310776239321e-08, 0.0011498888488858938, 2.8544893115878267e-09, 1.9765032277518912e-07, 3.3766368173759886e-11, 1.3631475326292275e-07, 7.511421244998928e-07, 5.879150535292865e-07, 4.826410826552774e-09, 1.3825607325657074e-09, 8.138537621604769e-10, 2.576161023171153e-06, 2.453664216250928e-10, 7.251524948514998e-05, 6.649132956226822e-06, 5.343328002282988e-09, 1.8219813213704583e-08, 8.406360940682589e-09, 8.912786597647937e-08, 4.600378744612499e-08, 4.211225146377728e-09, 1.937178922162275e-06, 7.3401573708054e-09, 1.644705844228156e-05, 4.63025486396873e-07, 5.07710531394423e-08, 9.889939178719942e-08, 6.740804026073022e-10, 2.7198623797630717e-07, 4.172604306518224e-08, 0.0004338995786383748, 1.2432796836492344e-07, 5.849619810760487e-07, 1.63538170454558e-05, 2.778469570330344e-05, 2.4768782511119802e-11, 5.2755588342279225e-08, 2.588436927908333e-06, 3.3523986076033907e-07, 6.473308076238027e-06, 6.165713273276197e-08, 4.715652757170119e-09, 1.831165690191483e-07, 5.422624781203922e-06, 4.450442247616593e-06, 5.8347914944079093e-08, 3.0810998197239314e-08, 2.0460854344150903e-08, 1.3630395478969604e-08, 3.671432932605967e-05, 2.869276372052809e-08, 1.6822104820235495e-09, 4.5732329454040155e-05, 2.545673005727167e-08, 8.567723597252552e-08, 7.535625456966955e-08, 2.1571129309450043e-06, 7.772086246404797e-05, 3.604118319344707e-06, 8.522815164724307e-07, 0.00013017172750551254, 2.878822726870567e-07, 1.0802988015257142e-07, 1.732351506689156e-07, 2.3679956484556897e-07, 5.251422408036888e-09, 5.853784124099093e-10, 3.121571623410091e-08, 4.6370081463464885e-07, 0.00020859898359049112, 5.1858300764706655e-08, 9.377433229929011e-07, 3.3733073223629617e-07, 0.006818693596869707, 1.5536585351583199e-07, 2.62252939364771e-07, 2.1817922970512882e-05, 6.888921689096605e-06, 1.2759453227317863e-07, 8.208439261636613e-09, 1.2698878038008843e-07, 2.937088419696643e-11, 3.113622071282407e-08, 5.976824590803176e-10, 2.890513073339207e-08, 2.19640527987508e-09, 4.915982003694808e-07, 1.5300181122057666e-09, 5.7554849774987815e-09, 2.5684610704956867e-07, 2.1737662336818175e-07, 4.66091620410225e-07, 2.1687412976589826e-10, 5.425797588998194e-10, 2.1738220112865747e-08, 1.0251449111819966e-06, 5.234277455912206e-09, 1.0241774361929856e-07, 1.8679518376529813e-08, 2.6536278951994063e-08, 1.0709653679441544e-06, 2.054245523686049e-10, 3.171941671098466e-06, 0.0001454267476219684, 3.712157194968313e-05, 5.865797447768273e-08, 1.8017382785728842e-07, 1.8313467364805547e-07, 1.2741668342641788e-07, 3.613537955970969e-06, 1.7143777952810524e-08, 7.426190791193221e-07, 1.4816912141668581e-09, 5.1816798077197745e-05, 3.1675135403474997e-08, 4.998648819309892e-07, 7.987463504832704e-06, 1.7724815961628337e-06, 2.1992538279391738e-07, 1.2403152993556432e-07, 2.063003767815985e-09, 2.2126943122202647e-07, 5.91458388043975e-07, 1.1866609916921789e-09, 6.77533753332682e-05, 2.0483653884184605e-08, 1.2488868605942116e-07, 5.77334979823263e-09, 6.858465351200493e-09, 1.8967789561230575e-09, 2.5762919904082082e-05, 4.024819588721584e-09, 2.578477733550244e-07, 1.6938618500716984e-05, 1.3560141898238953e-09, 0.00030475982930511236, 2.1155792637728155e-06, 3.0459816002803564e-08, 0.0001596398651599884, 1.2025932960568753e-07, 1.2980099661774602e-08, 8.505587902618572e-05, 2.122546447935747e-06, 1.258807316162347e-07, 1.791164727649175e-08, 4.0109489063411274e-09, 1.0857741017389344e-07, 1.4392657021744526e-07, 8.709687790542375e-06, 2.3463444165372493e-07, 4.852080959238947e-10, 1.934477586473804e-05, 9.050978405866772e-05, 2.783277253115557e-08, 0.00010184750135522336, 3.0766966574446997e-07, 1.2917719232063973e-06, 0.0003467947244644165, 0.00026913240435533226, 1.9854202548685862e-07, 9.900888471747749e-06, 6.902697435862137e-08, 0.000358258665073663, 0.00011699467722792178, 1.6368996436089134e-10, 2.7889395237679082e-09, 4.695560384959663e-09, 8.971052373185628e-11, 2.759812014119234e-05, 2.8238898721610894e-06, 5.842493600027865e-09, 4.30910176874022e-06, 4.250414804118918e-06, 6.451460308198875e-07, 6.172252398073397e-08, 1.1463457738614125e-08, 4.894415042144828e-07, 2.0276024415011307e-08, 4.041919510200387e-07, 3.066921863137395e-06, 2.8050299860638006e-09, 7.412443636667376e-08, 6.340809477478615e-08, 9.609445328351285e-08, 1.4389058833330637e-09, 2.3621309992449824e-06, 5.570721106629151e-10, 5.464907189889345e-06, 8.037332008825615e-07, 1.0646001101122238e-05, 1.1865931348609138e-08, 1.6836386507179668e-08, 6.851362854831677e-08, 6.491479354053808e-08, 4.255740551628584e-11, 3.901776679526847e-10, 0.0008611169178038836, 4.4520467781694606e-05, 1.596205667642181e-10, 5.938666225446809e-10, 7.637617613909242e-07, 2.6451536427885003e-07, 2.565730206072203e-08, 6.298042194430309e-07, 8.389515642193146e-06, 2.6097122240997805e-09, 1.1467922149677179e-06, 0.00045304076047614217, 2.978224813432462e-07, 2.7403447566687333e-11, 3.426604777700959e-08, 6.801439553782984e-07, 3.991890906718254e-08, 3.2247839953924995e-07, 1.2188222653364278e-09, 2.3238904134359473e-08, 1.7806432879297063e-06, 9.522947408413529e-08, 1.2206659683045018e-08, 9.97120466195156e-08, 1.7774145817384124e-06, 8.108408150638013e-13, 6.598948516511882e-07, 2.849912750235717e-09, 7.13991596512642e-07, 3.6751118841493735e-06, 5.2327717270372887e-08, 1.3651006725012849e-07, 1.7616255121311042e-09, 2.070945356535958e-06, 2.303448320972734e-09, 2.7707550543709658e-05, 2.758356387744243e-08, 4.486425009275763e-09, 1.3200460671214387e-06, 5.774189570928456e-09, 1.2340586863501812e-07, 1.3432075007813182e-08, 4.021133918286068e-06, 4.5181875601985055e-11, 4.898737097391859e-05, 5.577120987254602e-10, 7.1946800517253e-07, 9.351500693810522e-07, 0.014422349631786346, 8.340304589182779e-07, 3.8197896401470643e-07, 4.2180285930726313e-08, 1.6631930748189916e-06, 2.6927976293222855e-08, 3.271901638512986e-10, 2.421428177967755e-07, 5.4450023156960015e-08, 2.6710615941283322e-08, 7.099957315404026e-07, 2.931080302914779e-07, 6.868790336511665e-08, 9.969247649621593e-10, 9.79305703197042e-09, 2.2590613824036154e-08, 1.3676863090950064e-05, 7.463182214451081e-07, 3.2258291593478816e-09, 5.49304195374134e-06, 7.312393677239015e-07, 2.0029845018143533e-06, 6.4191780779765395e-09, 0.00010630150063661858, 3.0848320875520585e-07, 1.8030857972917147e-05, 3.8218049667193554e-06, 6.593611212579731e-10, 2.2144190552353393e-06, 1.6911460534174694e-06, 0.00035093515180051327, 3.621901356609669e-08, 2.3161352002976088e-10, 7.781794920447282e-06, 1.4001385606476902e-09, 1.6661912738413776e-08, 7.178211802738588e-08, 6.735955793146786e-09, 8.099245860648807e-07, 4.4547113220438916e-10, 7.263711767485637e-12, 4.296070096887483e-11, 2.734362292144965e-09, 3.374262647071191e-08, 3.527656744495289e-08, 6.983112754710419e-09, 2.856257594885392e-07, 1.3960279598990155e-09, 5.862222599711808e-12, 1.284557443054224e-11, 1.726046292560568e-07, 5.741874975484507e-09, 9.276593715412673e-09, 2.9290541192494857e-07, 1.183375797353392e-08, 3.100723233728786e-08, 1.1334376722516026e-05, 5.238755989012134e-07, 9.56587911105089e-08, 2.4250950900750468e-06, 3.850815666339713e-09, 3.2859779253158194e-07, 5.5696066425525714e-08, 1.0673442685238399e-10, 4.2235419641656335e-06, 0.006682428531348705, 3.0906454639989533e-07, 4.115382257197098e-09, 1.0100189484774091e-07, 4.30839852683107e-09, 3.8275337760351746e-11, 1.4396348888112698e-05, 2.95438229436229e-09, 1.779477429408871e-06, 5.672682164004073e-07, 9.270078038525753e-10, 1.2105055846589607e-09, 4.246235524654751e-13, 0.009770327247679234, 4.3471740696077177e-07, 1.6738627550694218e-07, 3.274589133184236e-08, 0.0001055443863151595, 5.168828920432134e-06, 5.146540416056666e-10, 2.8875890567547913e-10, 2.744400084964127e-08, 7.378066868568567e-08, 4.95850599691039e-07, 8.274904095273428e-10, 6.879663487779908e-07, 1.5208611037209607e-09, 2.9125037936950093e-09, 5.880079001485683e-08, 1.3589422920290417e-08, 9.809618006784149e-09, 6.744576239725575e-06, 1.7199400303979928e-07, 2.2488229056705222e-09, 6.408082398046133e-11, 1.075710542863817e-06, 0.00019246355805080384, 8.243988304457162e-06, 9.052513405549689e-08, 1.0256130735797342e-06, 1.2689813644328751e-09, 7.757953426335007e-05, 7.375585028057685e-06, 2.1140230191396547e-11, 5.022933891041248e-08, 2.9727306127824704e-07, 3.470688625384355e-09, 7.611326680034836e-08, 1.7081977148336591e-06, 2.388782604612061e-06, 3.728511728695594e-05, 7.246954737638589e-06, 1.0407655537392202e-07, 3.880331167493978e-09, 8.627117153992003e-08, 8.834050158057494e-10, 3.009998639225131e-10, 6.0801195189696955e-09, 1.5840251776921832e-08, 5.308392587721755e-07, 4.811157694462054e-08, 6.304644983856633e-08, 2.9458365702339506e-07, 1.744378323564888e-06, 9.310876514234678e-09, 2.7986965522863727e-10, 0.00012631509162019938, 0.00016997844795696437, 4.707460377062489e-08, 5.275500214452222e-08, 6.091527282592324e-09, 4.624977918865625e-06, 2.1491691759578657e-10, 1.330016220890684e-05, 1.8415727831566642e-09, 3.295321349128244e-09, 8.806280149542545e-09, 5.736977755077532e-07, 2.506325699869194e-06, 1.2457752607275552e-09, 3.255646308097937e-10, 4.6107623274949106e-10, 2.8985791544755557e-08, 1.075038191800104e-08, 5.133052205508193e-09, 4.874712633551326e-09, 6.163881494103407e-08, 6.895322979261209e-09, 9.172666977974586e-06, 1.424480888800872e-08, 9.720705520521733e-07, 2.7935740387619035e-09, 1.2054848639309057e-06, 2.223272232981799e-08, 2.2578453808819177e-06, 2.7882316544491914e-07, 4.478366122384614e-08, 1.592253973115021e-08, 2.036213800238329e-06, 1.5796624097674794e-07, 3.527492481225636e-06, 3.818321303583616e-08, 2.1701202967960853e-07, 1.1123863608020201e-08, 9.715327209391944e-09, 1.9309152321511647e-07, 3.3591761166462675e-05, 4.3335327063687146e-05, 2.2751873995918714e-10, 3.21929510391783e-05, 4.3921125325141475e-05, 6.811572461629112e-07, 1.4609244317398407e-05, 1.644133362788125e-06, 4.767260364246795e-08, 7.460391316271853e-06, 8.851596362546843e-07, 1.641405411589858e-08, 2.6471522890370025e-11, 1.621522471850767e-07, 1.7371981186897756e-07, 2.3094962053882284e-10, 1.0582566289940587e-07, 2.0914898701818174e-09, 2.6978234757280006e-08, 1.4545993576309257e-10, 1.0903066005596429e-08, 5.6203607101679154e-08, 1.835566559860169e-10, 6.006291575211264e-10, 1.1556490164821298e-07, 1.0747560708068704e-07, 2.174243036279222e-06, 2.5388743551957305e-07, 1.922026982015268e-09, 1.6106092848300335e-10, 1.1613298056545318e-06, 7.454452166655301e-09, 3.36027454750365e-07, 8.628392151877051e-07, 1.9786503546725953e-09, 0.0004198085516691208, 4.840868496103212e-05, 5.9417075704004674e-08, 4.586700441677749e-08, 2.9953965974272023e-09, 3.397745604161173e-05, 1.451096250093542e-05, 2.5991592877971925e-08, 9.625859220818711e-09, 6.696098342473533e-09, 4.0406197854281345e-07, 5.944087444476054e-09, 1.0422401828691363e-05, 2.6069896241764923e-10, 2.4278950405687283e-08, 1.1324570170501147e-09, 1.3564928735831927e-07, 2.774056984122808e-09, 3.696170036704416e-10, 6.227368771760666e-07, 1.8076724472848582e-07, 1.806588625186123e-05, 2.0461287419948349e-07, 2.3176950492143078e-07, 9.972878842745558e-07, 6.645009852945805e-05, 4.626304246357904e-07, 2.5578833628969733e-07, 2.7109886104881298e-06, 9.309789383848965e-09, 1.2866624956586747e-06, 8.452266797576158e-08, 1.4029551742567037e-08, 1.8241997912227248e-09, 1.1033105540292354e-09, 3.4059721087942307e-07, 5.998326141343213e-11, 7.087414815032389e-07, 1.4920334251655731e-05, 2.9603718587623007e-08, 2.1252253290526824e-09, 3.3633338603067386e-07, 4.0758363351045546e-10, 4.2810849976149257e-08, 4.570338330012191e-09, 4.288262971385848e-07, 2.336125675128642e-07, 4.5354796307517375e-11, 4.257211685398943e-07, 1.8042718465949292e-06, 6.303133432083996e-06, 2.9903341783210635e-05, 6.667211209787638e-07, 9.473200179854757e-07, 1.5689668231289033e-08, 1.3256312492160305e-08, 3.725652675257152e-08, 2.9770108600502e-09, 1.0903976317422348e-06, 2.0278015654184856e-05, 2.228617995569948e-05, 3.446463114897824e-10, 1.1520555176502967e-07, 1.5009460412329645e-06, 1.3719978486648188e-09, 4.769555630446121e-07, 1.5208794138743542e-05, 8.18066681063101e-08, 1.0385326731920941e-07, 1.5722338986279283e-08, 6.865599289085367e-08, 3.30605985254806e-07, 7.609075964865042e-06, 1.50970237200454e-07, 4.901345960206527e-07, 5.947753720647597e-07, 2.7894695442398643e-09, 2.4110869389915024e-07, 6.869441193657622e-08, 7.665650059163198e-11, 1.2080593023711117e-06, 3.6248914980774316e-09, 5.838855787754227e-11, 2.544769082957199e-12, 1.2713589967461303e-06, 1.090602719244771e-09, 5.394060167418502e-07, 1.4893084880895913e-06, 1.3166715689294506e-05, 5.852033080344654e-09, 8.931418271629354e-09, 2.0254751120774017e-07, 1.9170715015448536e-10, 2.906627993848332e-10, 8.297145193125743e-09, 1.3334499726624927e-07, 2.3858839903745377e-10, 2.7758650844589283e-07, 1.143323125063489e-08, 4.863857580583719e-11, 3.265872294822003e-10, 5.171961969807626e-09, 4.324742164385498e-08, 9.032373071704569e-09, 0.00018850245396606624, 4.161210824804584e-07, 1.3323093739359138e-09, 3.1891922844806686e-06, 4.605300141236057e-09, 6.112165351623844e-07, 3.0501330350318767e-09, 2.029974972839188e-11, 0.005107637494802475, 3.0890126367921766e-07, 5.482516129973192e-08, 3.8262122359356e-08, 6.967190557816139e-08, 5.4135835370061614e-08, 1.5581261322950013e-05, 1.3907264246881823e-06, 1.2984622799194767e-06, 2.1875921163427847e-07, 3.679317615024047e-06, 5.5249508079668885e-08, 3.3930371845514173e-09, 9.006431156421968e-08, 5.58005410766782e-08, 5.831521376897797e-11, 3.854116243928729e-07, 3.58613192474877e-07, 2.841159529864967e-10, 3.6377014733801616e-09, 2.12950919120658e-08, 1.4255284952469083e-07, 1.0518093063183187e-09, 1.4206975151864754e-07, 3.5950074561696965e-07, 0.0006375107332132757, 4.08780306315748e-06, 3.246080515495464e-09, 9.273755381400406e-07, 4.703037575382041e-06, 4.101342554463372e-08, 6.472038904803412e-08, 1.4310744429757438e-11, 6.115454453947677e-08, 2.74526155408239e-05, 3.634675893593453e-09, 6.640583194439387e-08, 8.067263301825278e-09, 3.5713597412723175e-08, 7.172942417810191e-08, 5.252579171610705e-07, 1.4833786421419859e-09, 2.1424929386171243e-08, 4.072876890859334e-06, 5.021616455991307e-09, 3.6754074244527146e-05, 1.4919502064003609e-05, 9.505958242783663e-10, 2.707694875425659e-05, 2.802997611794922e-09, 6.486795314231131e-07, 2.725059630392934e-06, 2.1789103854530367e-08, 8.176292709549671e-08, 2.5367857233504765e-05, 3.8840521909833114e-09, 2.409275907666597e-07, 9.406417689206137e-07, 0.00029261098825372756, 2.4696553282410605e-06, 4.5033951323603105e-07, 1.606986984370451e-06, 5.198645425252835e-08, 1.7603632329610264e-07, 1.4628087185997174e-08, 3.684353373500926e-07, 1.089766731715569e-11, 1.1535742361346024e-09, 5.550190973480085e-09, 3.162243089604999e-08, 1.2343318758212263e-06, 2.979607188535738e-06, 5.7013230941871385e-11, 7.147631464476945e-08, 0.00012326180876698345, 1.5561029442778818e-07, 3.397669701143968e-08, 1.3291170941620578e-10, 2.901211215711186e-10, 1.6965843315119855e-05, 2.6515296625007068e-09, 7.982653471572121e-08, 7.357551901066017e-09, 3.6718918750011653e-07, 1.0490170097909868e-05, 5.474033059726935e-06, 6.797482132725463e-09, 5.035119343688166e-08, 6.106010808082374e-09, 2.2536291055530455e-07, 1.0797710103815916e-07, 1.1428740265273518e-07, 9.964139235929537e-11, 6.118785922382131e-09, 5.998736583023856e-07, 4.6707643974741586e-08, 6.382349511113716e-07, 1.232387063510032e-07, 3.734121067910934e-10, 1.4147221172322588e-08, 7.985321417436353e-07, 0.0014373734593391418, 2.970559215276225e-11, 3.260657592818461e-07, 3.0458180422243686e-10, 1.821002513224812e-07, 3.235259100620169e-06, 6.97731229593046e-05, 5.417061643697707e-08, 1.292246842865552e-08, 1.1217049689093983e-07, 8.172415846274816e-07, 5.922573109273799e-05, 0.0007640118128620088, 2.773803316813428e-06, 2.3652205194402143e-10, 7.352792295023391e-07, 6.339275770983477e-10, 2.355034069978501e-07, 1.1617133850450045e-06, 6.993643637542846e-06, 6.2135439016230976e-09, 1.61551909627633e-08, 8.290202799798863e-07, 1.8695380799016448e-09, 7.689531003052252e-07, 4.9518366296297245e-08, 1.0336089317775077e-08, 2.220544956799131e-06, 1.2129901527657694e-08, 5.222385152592324e-05, 3.774872325834622e-08, 2.5468210651524714e-08, 9.092897244045162e-07, 3.519578058330808e-06, 6.268780998652801e-05, 5.544405894397642e-07, 1.1357554285496008e-05, 8.135349105486966e-08, 2.1589880816463847e-07, 6.102885663494817e-08, 1.3553652706832509e-06, 1.2652969871851383e-07, 2.9403909138636664e-06, 1.0711961095921652e-08, 6.887160998303443e-05, 5.720249828300439e-05, 3.2126272344612516e-06, 6.003672865517728e-07, 1.6841696093283787e-10, 6.33080716738732e-08, 3.886545130171726e-08, 6.019425882186624e-07, 9.551111901373588e-09, 1.4439302198354653e-08, 3.6258038562664296e-06, 6.252186040001106e-07, 1.4176336549098778e-08, 7.825537409189565e-07, 1.1146566066599917e-05, 7.199133733593044e-08, 2.1884906387104053e-10, 4.41881002188893e-06, 6.6696870248961204e-09, 3.537602606229484e-05, 3.8953782421913274e-09, 2.419532393105328e-06, 5.246844239081838e-07, 1.017039341633108e-08, 8.36927505076801e-09, 4.135041464792266e-08, 2.634475322338403e-07, 8.39705904809307e-08, 7.232771673248983e-12, 2.1315554477041587e-06, 3.393797385342623e-07, 0.00023073545889928937, 5.03223077430448e-07, 1.8319937566957378e-07, 2.952112998499956e-09, 6.574303601514231e-11, 1.463949530489117e-07, 4.128824571125733e-07, 1.293893348019992e-07, 4.9990806161304135e-08, 6.954323339414259e-08, 0.00023735511058475822, 4.1768737446545856e-07, 6.147383990739996e-11, 6.628804527508692e-08, 2.0212700402311157e-09, 2.1261807869876748e-09, 1.4313354768091813e-07, 1.3019233335853642e-07, 1.0488678814368768e-09, 2.6399368802110246e-10, 5.545486601477023e-06, 1.803255877952381e-09, 0.00021082039165776223, 1.7247386495000683e-05, 1.700859968423174e-07, 3.4032132134598214e-06, 2.7802157798184624e-10, 6.921967354855951e-08, 9.794424826736758e-09, 4.399329100124305e-06, 3.937304427381605e-05, 3.690900882702408e-07, 7.501133603149412e-10, 3.7656731899371465e-11, 3.235450102279103e-11, 1.3802385012695595e-08, 4.345427837004756e-11, 9.296883035858627e-06, 0.00025543849915266037, 1.6211024558288045e-06, 1.6644909806018404e-07, 2.0018995350312707e-09, 1.838770202766682e-08, 3.4614834021340357e-07, 1.1215759059268748e-06, 2.1039108233367187e-08, 4.768340236549484e-12, 6.218282351255766e-07, 5.568193591898307e-05, 2.7781199563037262e-08, 2.0151633695064675e-09, 9.219877392752096e-05, 4.132502454012865e-06, 5.622038656838413e-07, 7.012793901139958e-08, 4.992795652469795e-07, 4.323882407675228e-08, 3.887571509153531e-08, 1.385662244501873e-06, 5.482804432688226e-09, 5.619814601232065e-06, 3.846573282118015e-09, 9.074159201816201e-09, 8.236483495238645e-08, 2.3352599498593918e-07, 8.030549736304238e-08, 9.122612665279917e-10, 5.0679222596272666e-08, 2.1668548899622664e-10, 5.294573202263564e-05, 2.5079244236891896e-10, 1.4617209664891107e-08, 0.00018933879618998617, 3.349366579641355e-07, 1.704389660517336e-07, 2.617442987684626e-06, 1.1825197532289167e-07, 1.2007859595541959e-06, 9.241087006728321e-09, 6.490518789092903e-09, 4.559915396384895e-07, 4.6651007323816884e-06, 1.1980775127540255e-07, 3.8271796256594826e-06, 9.695590996727788e-10, 5.377351499191718e-06, 4.20594936656471e-09, 2.3483130995316515e-09, 1.0384426651910417e-08, 1.0127823202310537e-07, 1.0419509635539725e-05, 8.759473075770075e-07, 1.5391499630368344e-07, 1.0963914220951665e-09, 8.829674769117446e-09, 0.000556078040972352, 2.775124414711172e-07, 1.3476197864292772e-06, 3.8841676541778725e-07, 5.412403414339906e-09, 1.2525930292994758e-10, 1.5835064814950783e-08, 3.318311669886498e-08, 4.897688921090548e-09, 4.799569162244888e-12, 2.8418519273931686e-11, 1.3679034964297898e-05, 3.2115868298099315e-10, 4.6391797070555185e-09, 4.2013655843220477e-07, 5.106647993358138e-09, 2.920280017804089e-08, 6.382711035257671e-07, 3.8888917086588037e-10, 3.184696950597754e-08, 0.0022517337929457426, 4.807524965144694e-05, 1.9946322371566794e-09, 1.636336288690643e-09, 3.019376606516744e-07, 1.0084050261482957e-09, 5.8975375694103604e-09, 3.2042840603097034e-12, 2.607042688396177e-07, 4.0001008061452126e-10, 1.0398205213535405e-10, 3.863480202426217e-09, 1.898068944683473e-06, 1.1283309397924768e-08, 4.6466794856314664e-08, 5.119358320371248e-05, 5.382343033488723e-07, 8.063941641012207e-05, 2.6859683921465205e-10, 7.108698696356441e-08, 1.6540356639893616e-08, 1.8911428924184293e-05, 7.238941113207886e-10, 5.060725811745215e-07, 4.834339506487595e-07, 6.423593390536553e-08, 2.300019019685351e-07, 1.2024614079564344e-05, 5.763653443402461e-10, 7.750611675898245e-13, 1.4264393030316569e-05, 1.3887477279261873e-10, 1.2545084246085025e-05, 2.4254873096651863e-06, 1.3536378240530667e-08, 1.092507151412292e-08, 5.888266514375573e-07, 1.1530238097634538e-08, 1.7205646629658844e-10, 3.1743095973979507e-07, 8.626479370832385e-07, 2.710242767989257e-07, 3.6435181982596987e-07, 3.1246923271055493e-08, 8.982635746335177e-10, 4.767221639667696e-07, 4.4743853777617915e-07, 1.7076345102395862e-06, 1.814477279538096e-08, 1.4306704088085098e-07, 1.5004124165329813e-08, 3.802386117968126e-06, 1.1835690649775188e-08, 2.368717701983769e-07, 1.7759813175644013e-08, 1.7926312878557837e-08, 2.0377738962906733e-07, 2.98253758046485e-06, 7.654531453127333e-10, 4.838846212606995e-09, 1.2653982821575482e-06, 1.2284275818785773e-08, 7.921960309964504e-10, 6.782447936615199e-09, 5.946004222323609e-08, 2.5025881367213287e-09, 6.694642706861487e-07, 5.870068875424295e-08, 2.1389226390056137e-07, 1.2334677990111231e-07, 3.458734687544762e-10, 1.9450155377853662e-05, 1.6397716535720974e-06, 4.353806898338419e-11, 0.0004617929516825825, 4.360986713436432e-05, 1.3842648058925988e-06, 5.690033867722377e-05, 7.664065196877345e-05, 7.597448359319969e-08, 4.56737438980781e-08, 2.4347779348232734e-09, 2.89056856672687e-07, 6.79814149862068e-07, 1.002386156301327e-07, 9.540671044305782e-07, 3.8937486124268617e-08, 2.614645609355648e-06, 2.172453241655603e-05, 2.2745976480109675e-07, 1.671510653977748e-05, 7.004581448200042e-08, 7.4035546582251754e-09, 1.1425123602748499e-07, 4.888977969130792e-07, 9.776674232853111e-06, 6.753557435512647e-11, 3.0142878131300677e-06, 2.3714740393643297e-07, 1.8845973670522653e-09, 3.596634323699277e-09, 9.524364630308924e-10, 6.405826979971607e-07, 0.00016794528346508741, 2.5447958407198712e-08, 8.548123986429346e-09, 3.670630121632712e-06, 0.00011552131036296487, 7.791822440594842e-07, 1.942562903423095e-06, 1.6472442609938298e-07, 1.5614027688570786e-06, 1.962922879794604e-10, 1.6604950747733938e-09, 2.076451366406218e-08, 7.197635568445548e-06, 5.110206302560982e-07, 1.2226546164395558e-11, 1.7797394669116784e-09, 1.3204473070516087e-08, 1.9707535603430415e-09, 5.532859503887266e-09, 3.664209486942127e-07, 1.4457629760045165e-08, 5.4323740187101066e-05, 2.9147501079407334e-10, 3.2327698851197795e-10, 1.290224871386414e-10, 4.505077413341496e-07, 8.019334529763e-08, 3.7325307857827283e-06, 2.816757614709786e-06, 1.9476206034596544e-06, 1.545375494238499e-09, 6.098907334717296e-08, 9.76978914435378e-12, 5.502198519025114e-07, 2.642720176027069e-07, 9.12263402597091e-08, 2.839614410277136e-08, 3.724383645931084e-07, 5.980066219990476e-08, 2.4225167294389394e-07, 2.0665094524474625e-08, 1.5162890942832519e-09, 6.278212367760716e-06, 1.5124198782245912e-08, 2.013367073061545e-08, 4.984633594773413e-09, 2.22214335821036e-08, 4.956918209231276e-10, 1.69373572589393e-06, 3.5440482548665386e-08, 0.00047247318434529006, 2.8133442242506135e-08, 1.1858225989688775e-12, 4.745972546515986e-05, 1.1481331857465804e-10, 1.5363228744735835e-10, 1.8769629605230875e-05, 1.4338125353674513e-08, 1.626042767099989e-08, 1.9388604499681605e-08, 2.7300417926312548e-08, 1.7566560472914716e-07, 2.94520589250169e-07, 1.2820722758988268e-06, 4.8133133532246575e-06, 3.8875572272445424e-07, 4.1115870885732875e-07, 9.276153734028014e-10, 1.4077905063913931e-08, 1.0573583786310792e-08, 1.3423051314020995e-05, 3.552986127530744e-09, 1.9868065526118528e-10, 6.574371980150318e-08, 1.312333068881344e-07, 0.00017809333803597838, 3.1187065019366855e-07, 0.0002614969271235168, 3.4713434615696315e-06, 5.944443728367332e-06, 3.499078138347045e-09, 7.888706221592656e-08, 4.1185700183632434e-07, 6.86753774061799e-05, 1.1612592487608708e-07, 1.2841296666010749e-05, 1.3377538721215387e-07, 5.643659761744857e-08, 3.30467042619631e-10, 3.900271772216968e-10, 2.8705177124166426e-11, 0.219920352101326, 0.00011678944429149851, 1.0837623243276084e-08, 1.021837636727696e-08, 8.089856464721379e-07, 2.439598523196196e-09, 5.235361186350929e-06, 3.6723495355772684e-08, 0.0006654011667706072, 1.605585175923352e-08, 3.7576826095175875e-09, 1.2543338016257621e-05, 1.9095082848252787e-07, 3.4751392874454723e-09, 2.8763891268823727e-09, 1.3057060321841618e-08, 8.917249516571246e-08, 5.706876322619792e-07, 5.4313153974305806e-09, 4.1104335934960545e-08, 8.382201485801488e-06, 5.253891504253261e-05, 2.7998504492643406e-07, 2.7220904641467314e-08, 5.730051100272249e-09, 9.199495707434835e-07, 2.5230368905226896e-08, 8.521595009369776e-05, 8.364646753022953e-09, 2.0962201233487576e-05, 0.00022696204541716725, 4.345022261986742e-06, 2.836639225733961e-07, 4.358212013499951e-09, 3.9337410271400586e-05, 2.8906482540946854e-08, 4.240645523623243e-08, 1.7769383475751965e-06, 3.752059640760308e-08, 5.518118541658623e-06, 4.1054217803093707e-08, 1.533955895638428e-08, 1.952806769622839e-06, 3.176555907202783e-08, 2.7687693915368072e-08, 6.708076938366503e-08, 0.0002071019116556272, 9.83781056618227e-09, 4.525628355622757e-06, 1.3617086835893133e-07, 1.1283245271442865e-07, 3.3451679204432594e-09, 7.58837586545269e-07, 1.0042521481068434e-08, 1.1050811654644832e-10, 3.387580349567543e-08, 1.754118095220747e-08, 3.694782208185643e-05, 2.3555513095629976e-09, 6.671666028523759e-07, 3.046776342330304e-08, 3.218720223685523e-08, 1.0757355539681157e-07, 3.640157956397161e-05, 1.3784561048169053e-08, 4.196241931708755e-08, 1.1799384580513106e-08, 5.2503692365979404e-11, 7.192145212719936e-10, 3.923147460227483e-07, 0.00019065271771978587, 5.671630987080789e-08, 1.0117049669133849e-06, 8.572663432460104e-07, 8.997318445835845e-09, 1.4806543144341155e-11, 1.9993109390270547e-09, 1.106368507919342e-08, 1.1343394001883667e-09, 7.005081715760753e-05, 2.9400268330981305e-10, 2.0395926185301505e-05, 6.368357219344034e-08, 4.535075603939731e-08, 2.6888191673180017e-09, 8.341977809322998e-05, 9.586619853507727e-07, 2.3803479507478187e-06, 3.0700988418175257e-07, 2.781908551696688e-05, 2.2018403456058877e-07, 1.8014854052950113e-09, 2.0774566511505554e-09, 6.050587444406119e-07, 0.0001761745079420507, 1.9427824327067356e-07, 4.438854839250439e-10, 3.173726383920439e-07, 1.7710959809846827e-08, 5.468722696910788e-10, 7.316669780266238e-06, 6.978938927204581e-06, 2.377042451762179e-10, 0.00010249839397147298, 9.637465154810343e-07, 2.44778614044705e-10, 3.6758416310789244e-09, 8.651663665659726e-05, 6.40022926745587e-06, 6.479633896105952e-08, 3.382452984368456e-09, 2.3570396936317684e-12, 3.2113637189468136e-06, 3.773917705984786e-05, 7.758485054409903e-08, 1.8478307595160004e-07, 3.7481979120457254e-07, 1.7369250926435598e-08, 1.2367721291539624e-09, 9.091951191919634e-09, 1.0824091623007348e-10, 1.0130297711796743e-09, 1.1198279025848024e-05, 3.345892878314771e-07, 0.0007547004497610033, 4.762223393406373e-10, 6.544605413161264e-10, 2.8606696105271112e-06, 3.277163262982441e-12, 1.362578494479294e-08, 1.0957194717775565e-05, 2.3776049573598357e-08, 1.102381975215394e-05, 2.9208158593974076e-06, 2.149954070773674e-06, 2.7755326925671397e-08, 9.109662357786874e-08, 3.6057188051302758e-12, 2.795284714807167e-08, 1.92197403521277e-05, 1.8195322581959772e-06, 3.0776523090025876e-06, 3.889325750350281e-09, 8.127699402393773e-05, 8.399101716349833e-07, 1.2222682865825618e-09, 4.4088588424529007e-07, 8.034644451981876e-06, 3.695451167295971e-10, 5.678614201087839e-08, 7.536187673906625e-09, 2.0290338298423194e-08, 1.691803497294586e-08, 7.611791374984023e-08, 1.0399894279089494e-08, 6.746422877768055e-05, 1.215233078255551e-05, 7.241240159577833e-12, 1.8170464244349205e-08, 3.8934291346492955e-09, 0.0006481957389041781, 3.70649991054961e-06, 8.580691428505816e-07, 5.460480224428466e-06, 8.940823192915559e-08, 8.07281963499662e-10, 2.3941868221299956e-07, 1.8698289068197482e-06, 1.849664727160416e-06, 1.1276463283138582e-06, 1.1063916893760961e-08, 1.623082002133458e-09, 1.2928724757443888e-09, 2.4016373572521843e-05, 4.176629227004014e-05, 4.897625558442087e-07, 9.228303810004945e-08, 3.178179594498509e-11, 2.5995030572545375e-09, 0.00013179139932617545, 2.344143901211737e-08, 1.8931574174985144e-07, 2.0879742379520394e-09, 7.074796940287342e-06, 0.003345826640725136, 3.030764332834224e-07, 1.7114624162317682e-09, 1.43939257668535e-06, 1.1794441812096323e-10, 2.424940794298891e-05, 2.4867864567568176e-07, 3.315294150318948e-10, 2.3015012118321465e-09, 8.52718855526291e-08, 5.915580025828149e-09, 6.876151701362687e-07, 1.5293740718291815e-09, 3.1386643284392335e-10, 4.2199324035152586e-09, 3.510359647407313e-08, 5.378264233968366e-08, 2.2899814666743623e-06, 0.00020476475765462965, 1.0997165844628398e-07, 3.322288832929843e-10, 1.3783539998257766e-06, 6.5969898059847765e-06, 6.8849531111025986e-12, 6.775314087548168e-09, 5.224466637088199e-09, 6.220234354259446e-07, 3.265143710962093e-09, 2.0061727886572234e-12, 5.56873374080169e-07, 4.4711761404414574e-08, 4.304881784378267e-08, 4.943512976751663e-08, 3.589691743854928e-07, 0.00023865068214945495, 2.899804485423374e-07, 4.603749881315622e-10, 1.522555947985893e-08, 1.6396973023802275e-07, 1.4971382142903167e-06, 2.4946481858023617e-07, 8.555575163882168e-07, 1.0639538459145115e-06, 5.318228346595788e-08, 1.0064885600513662e-06, 2.535005876325158e-07, 5.5387886277458165e-06, 2.6953026477372077e-08, 2.4383475647482555e-06, 1.1442377854109509e-06, 1.1334646110583435e-08, 2.539733701123481e-10, 7.115976075056096e-08, 2.0001228051569342e-07, 2.475424309977825e-07, 6.719841962876671e-07, 1.4351917343446985e-07, 1.4046494243302732e-06, 7.739531326933502e-08, 1.9051609179143725e-09, 7.011777269916308e-11, 2.910414696088992e-05, 1.923018260185927e-07, 1.2466520260545622e-08, 5.4264400972670046e-08, 8.817595897880892e-08, 1.0403927035440574e-06, 0.00019999964570160955, 4.5529191083915066e-07, 1.7224959236727955e-08, 1.8921610944744316e-09, 8.513329521520063e-05, 1.423408768630452e-08, 7.362093015217397e-07, 8.856027733372684e-08, 7.988967709593453e-09, 1.0961012009147453e-07, 2.0269277811735265e-08, 8.847238497367016e-09, 1.2695241524696144e-09, 1.1710491776284471e-07, 8.142459410009906e-05, 5.387738610806991e-07, 0.00015630062262061983, 5.5049090286196645e-11, 4.662747984485009e-14, 3.336924464747426e-06, 1.8809425128551993e-09, 2.351009698031703e-07, 5.834623607370304e-06, 1.5687538379438593e-08, 1.5768187040521298e-06, 1.3795184713671915e-05, 2.580693436726733e-09, 1.5058442270898809e-09, 2.7209184736420866e-06, 5.949808468130868e-08, 2.0360410246667016e-07, 6.132376029199804e-08, 1.7889563608264325e-08, 3.5758580541056517e-09, 2.5223997113243968e-08, 1.8089822617639584e-07, 2.736863109475962e-07, 2.618261078168871e-06, 8.439866028275844e-10, 3.6824161497861496e-07, 0.0001073064559022896, 4.511803126661107e-07, 2.20074307435425e-05, 1.6368632032026653e-06, 1.178305364391008e-08, 5.236431093180727e-07, 6.381981734193687e-07, 1.1630912410964811e-07, 3.4020045859506354e-05, 7.720782346787303e-11, 2.0167353795841336e-05, 2.272002808112461e-09, 1.0958539577079346e-15, 1.7955406050873535e-09, 2.22060339183372e-06, 4.564613118908378e-13, 8.401946161029628e-08, 1.9678103853948414e-05, 2.9005612134369585e-08, 9.280027413183234e-09, 7.718244887655601e-07, 3.425586214689247e-08, 5.961848046354135e-07, 2.5726503594114547e-08, 2.759952622000128e-07, 3.336997345115833e-09, 1.6932698372329469e-06, 6.768332561080115e-09, 3.687901026228069e-11, 2.0248929200050725e-09, 1.3720353308599442e-05, 2.0807617318041594e-07, 9.695693137246053e-09, 9.320197946749431e-10, 1.5425649735334446e-07, 2.8110990868412955e-08, 1.827706735957335e-11, 2.6144153508766976e-08, 6.097327087672966e-08, 4.455367275113531e-08, 6.601839430686596e-08, 4.210197190879228e-10, 1.9392014394270518e-07, 2.3531788428954314e-06, 1.853463160728097e-08, 2.3578053287565126e-08, 1.5765423411195911e-09, 5.117533774523508e-08, 1.6545057546224484e-09, 0.0019710822962224483, 1.6012386971908654e-09, 1.080518813978415e-06, 3.2721534807933494e-05, 3.3678174986562226e-06, 9.297695946486328e-09, 3.8570195926013184e-09, 3.9993972023033564e-10, 6.623339654510119e-09, 1.69848526354599e-08, 1.0065522474178579e-05, 6.249385364753834e-07, 3.066534759454953e-08, 2.184762060153389e-08, 1.2560022355501133e-07, 4.591536253428785e-07, 1.8706194850892643e-06, 3.520161158121482e-07, 1.0607974587628632e-07, 7.39487586542964e-06, 4.61306200350009e-07, 7.100824177541654e-07, 1.7199308786075562e-06, 3.359063072139179e-08, 2.7876722015207633e-06, 6.647496442191425e-11, 2.3001402171818341e-10, 1.9816387419879788e-10, 0.0007475365418940783, 1.7843146906670881e-06, 8.290887976158956e-10, 1.1592420356976163e-08, 3.1045576776023154e-08, 2.0094952091209706e-10, 8.29143189662318e-08, 3.0652451243895484e-08, 1.0697619456934149e-09, 7.799316054102334e-11, 8.310928056864952e-10, 1.1932534107472748e-06, 4.646533398044994e-06, 2.1509505288008768e-09, 2.7488070153403044e-10, 3.620188948616487e-08, 9.74999192493442e-10, 3.4516436930864813e-10, 4.649577944881855e-11, 1.5602167877659667e-06, 2.3893304001987303e-10, 9.509176085442661e-11, 3.0061297451311475e-08, 4.658599284113052e-09, 3.641183576874596e-09, 1.477062028243381e-06, 5.370861089204482e-09, 2.996949888256495e-08, 7.194952195277438e-05, 5.859236352989683e-06, 6.693330334428538e-09, 4.14360776801459e-08, 1.3517172936516886e-09, 2.8299862606218085e-05, 5.986090911846986e-08, 1.5723395918598726e-08, 1.4570172846006813e-09, 1.8247581579089456e-07, 6.262976626203454e-07, 4.464592251451904e-08, 2.16495172935538e-05, 3.76237291321857e-10, 1.637245561347811e-09, 5.855713993696554e-07, 7.606533017678885e-06, 2.6132187525007566e-09, 1.907099544951052e-07, 4.912776148557896e-06, 1.3761835759851238e-07, 9.868057482265158e-09, 4.397222497232178e-08, 1.3599720238843815e-09, 5.208358743402641e-06, 1.4975333328948182e-07, 1.0421898899437565e-09, 1.954976980300671e-08, 3.437571649556048e-07, 1.696085178082285e-06, 3.1940507483341207e-07, 2.566968362316402e-07, 2.4456776372971945e-06, 9.049223308466026e-07, 3.083693229655182e-07, 3.1456438591703773e-06, 2.0704470671262243e-07, 4.132645781140809e-09, 1.967712108807973e-07, 3.6674182979368197e-07, 1.800975257815196e-09, 3.257667913203477e-07, 1.0138014872040912e-09, 8.650462746118137e-07, 8.284953878501256e-08, 7.32634191535908e-07, 4.788590786120039e-07, 2.1928710225438408e-07, 4.3824444941054086e-12, 1.4438427342611249e-07, 3.336940324061288e-08, 7.203704854852333e-10, 1.1067390914831776e-07, 4.664724606584514e-09, 9.628095085645327e-07, 2.12973372271108e-09, 5.3728637539052215e-08, 1.1274025463192672e-10, 4.321071003232646e-07, 5.6579100515818936e-08, 3.202335108198895e-08, 2.3698667472871193e-09, 4.005343896551494e-07, 2.2715832983521977e-06, 6.980943112466775e-08, 2.2576955416298006e-08, 1.570973218179006e-08, 5.940198661846807e-07, 9.39959647894284e-08, 6.032267378941469e-07, 6.165185251205685e-09, 2.524861315578164e-07, 0.00022558821365237236, 3.034743940588669e-06, 0.0003825244784820825, 9.662303313007214e-08, 1.0224775470746295e-09, 1.216347300214693e-05, 6.75777434011593e-09, 1.1616643860179465e-05, 1.4223696886972448e-08, 3.3920238706741657e-07, 3.042260914298822e-07, 1.7539221516926773e-05, 3.695365080602642e-08, 9.163854564064877e-09, 3.601947073939016e-10, 1.3310726387771865e-07, 2.2311061911750585e-05, 1.0321181775907462e-07, 4.789005303429406e-10, 1.292377902473163e-07, 1.550797179561414e-08, 6.09566029652342e-07, 1.9948727114638132e-08, 3.209158450090399e-08, 1.0922492077458656e-11, 3.621685209509451e-06, 4.621611687100069e-12, 7.054628099467664e-07, 2.263837814098224e-05, 4.237463144818321e-07, 4.01613505118803e-07, 7.200463336687335e-09, 1.4506204593089933e-07, 1.7994543668464758e-06, 1.405608163196348e-08, 2.7532921720307968e-08, 1.1379397619748488e-06, 1.1501949614967089e-08, 7.498345439671539e-06, 5.025498239774606e-09, 1.1362018881300173e-07, 8.118590244521329e-07, 2.6667937191859892e-08, 3.431420836363941e-08, 8.773030218955924e-11, 5.9432004206882993e-08, 7.365684723481536e-05, 1.2675862990363385e-06, 2.513709518314755e-10, 3.237383623400092e-07, 8.588199595926937e-10, 4.263351982558561e-08, 2.524647380042211e-09, 4.929499937134096e-07, 5.487188445840729e-06, 7.2463199884964524e-09, 4.5660772229894064e-06, 3.8193346085790836e-07, 8.400461126711889e-08, 1.8093354015036311e-07, 1.3409738130576443e-05, 0.00018178742902819067, 5.024781479789908e-09, 0.0012946698116138577, 1.700438012619543e-08, 1.898045454140629e-08, 8.331772733072285e-06, 2.790737198665738e-05, 3.41061241748708e-11, 0.005338921677321196, 1.114134761337482e-06, 6.771859517584744e-08, 2.495953221881564e-09, 1.28588382608541e-07, 1.4284596616676026e-08, 5.498260513547848e-09, 4.446524926038364e-09, 2.2854237613501027e-08, 5.905641273784568e-07, 2.654162223336698e-08, 2.216498096174746e-08, 2.616036942626465e-09, 2.688377165327438e-08, 3.563909167780821e-09, 9.245121646017651e-08, 6.991019745328231e-06, 1.6564810039199074e-06, 7.2690933272667735e-09, 1.3123941243975423e-05, 4.0547746493757586e-08, 4.0321151573152747e-07, 2.9870792786823586e-05, 4.403654330076279e-09, 1.4521256730404275e-07, 1.658882631261349e-08, 7.45651789202384e-08, 1.8989123162782562e-08, 3.864165876166226e-09, 8.613182558292465e-07, 9.229816346975617e-13, 1.902591151292654e-07, 0.0001544762053526938, 3.197731302861939e-06, 1.4455536323509932e-08, 3.169909064126841e-07, 4.290074002710753e-07, 6.875988667331967e-09, 1.1987567631877027e-06, 1.14115682663396e-05, 5.898008748062011e-09, 1.4800943972659297e-06, 1.6617465736779025e-10, 3.632790068763825e-08, 1.0412439621632075e-07, 6.739428026064642e-12, 9.588306966179516e-06, 4.752469084934319e-09, 1.5493821337031477e-08, 8.222435099014547e-06, 1.1172609504228603e-08, 3.224387120151495e-12, 1.4315250096430532e-09, 5.537028187063697e-07, 2.60941021679173e-07, 6.072325753336827e-09, 2.7860343834618106e-05, 4.9881577979249414e-08, 7.822480085906136e-08, 1.9180784875061363e-05, 1.889924448050806e-07, 7.815048661541368e-07, 1.6625962828697993e-09, 2.6055162472005122e-09, 2.911104246550167e-09, 2.9751600294503078e-08, 6.276235353652737e-07, 0.006363963708281517, 2.2032964341889283e-09, 3.655290958803903e-09, 6.467786306529888e-09, 3.803778625410814e-08, 5.5654211905675766e-08, 2.4536150888820885e-08, 2.4766988460100947e-09, 3.1472723094339017e-07, 2.0942701794979257e-08, 3.559716077461417e-08, 7.496327864009444e-11, 6.450816769643097e-09, 4.024084603315714e-07, 1.1258707743877494e-08, 4.379785707442352e-07, 1.0101107505988693e-07, 1.0673993529053405e-06, 3.517804714192607e-07, 5.169256134252009e-10, 1.868022962980831e-07, 6.4270477828642925e-09, 1.700891152722761e-05, 2.4480695515194384e-07, 4.02800959253824e-10, 4.9209685215600985e-09, 7.584524119863545e-09, 4.2167797801084816e-05, 1.4884874133258563e-07, 4.5494306988302924e-08, 2.4002047016580264e-09, 2.673750998383184e-07, 2.1604792621587876e-08, 1.633567262615543e-05, 1.003069655780564e-06, 3.689098679315883e-11, 3.557039818247176e-08, 7.543743407723014e-09, 1.285579003251769e-07, 2.622151214382029e-06, 2.7843144678740828e-08, 1.9572286191760213e-07, 1.758151029207511e-07, 9.930457567364215e-10, 2.1727028070017695e-06, 4.109909241378773e-06, 5.923592652834486e-06, 7.099252741227247e-09, 1.674672844842462e-09, 6.54108080198057e-05, 1.1149430889645373e-07, 1.739030430769617e-08, 1.1045195869030522e-09, 2.932654474818719e-08, 6.782390300941188e-06, 8.084202751668101e-10, 1.444074748668811e-09, 3.3402056942577474e-07, 8.125609074483009e-10, 9.33379098455589e-08, 1.3841021484495286e-07, 8.259740980065544e-07, 3.833821038412566e-10, 0.0010088005801662803, 4.235986228451338e-09, 1.671947416070907e-07, 1.0559431757428683e-05, 2.3536918192235134e-08, 5.459646956751385e-09, 4.093444204045227e-06, 1.9734206091470696e-07, 0.00024160376051440835, 1.090720260776834e-07, 7.85482825449435e-07, 1.578133606017218e-07, 7.557403591818002e-09, 2.423858518341149e-07, 1.9146226293287327e-07, 0.0006290639867074788, 4.233144945686718e-09, 2.5744941012817435e-05, 0.0009268811554647982, 2.5850107832070535e-08, 7.898068474787578e-07, 2.9919718258497596e-08, 1.9948964791183244e-07, 3.855271570735175e-12, 2.6511290229791484e-07, 3.7201346003712388e-06, 4.0390746303131664e-10, 3.4650340929864853e-10, 9.788923307496589e-06, 1.0083117452097667e-08, 4.177205994437827e-07, 4.457127028567598e-11, 5.157014015821915e-07, 2.7215530280955136e-05, 1.5251729337251163e-06, 1.1111164894828107e-06, 4.215135618323984e-08, 1.335772492616627e-09, 1.9396325114939827e-06, 3.6738715181172665e-08, 1.0551416380621959e-06, 1.4361981048871542e-10, 4.037005055579357e-06, 3.363261058542122e-10, 1.1599102833770303e-07, 6.483166448134625e-09, 1.0344467682443792e-06, 3.671130599514072e-08, 1.7567593602052511e-07, 5.76955528686085e-07, 3.860513686504419e-08, 4.855622126598291e-09, 2.553634814717043e-08, 9.81785959197623e-08, 2.584463345556287e-07, 7.953432357155066e-10, 1.8391944911488878e-10, 1.0320234622440694e-07, 4.822431947104633e-07, 1.7827732108344208e-06, 1.0959161045320798e-07, 8.186200162185742e-09, 3.366767131751658e-08, 5.028560234876522e-09, 9.537222123157107e-10, 8.463413969650446e-09, 6.517358031032927e-08, 2.619102303924592e-07, 5.901191252632998e-05, 1.2812823024788145e-09, 1.6672133895667685e-08, 7.787144262749734e-09, 0.0004075125907547772, 1.4643763535104881e-08, 6.407532726626641e-09, 4.4657411102377864e-09, 4.407550990848108e-10, 1.745762290283892e-07, 6.005847208445658e-11, 3.8039075889173546e-08, 2.691092682027829e-09, 1.1788879206164893e-08, 2.535041119244852e-07, 4.890514215816211e-09, 6.1431592257576995e-06, 1.7492463033619288e-09, 1.2621687517366809e-07, 2.15625590271884e-07, 3.2444529551867163e-07, 9.40416313710557e-08, 1.4213714166544378e-05, 1.3836039158832136e-07, 5.275444664443185e-11, 3.7313960254969913e-10, 2.2478708672224457e-08, 0.0010180476820096374, 0.0007708857301622629, 7.796497891732201e-10, 3.6431245575840876e-07, 2.712682856120807e-10, 2.861096248807371e-09, 0.0001277367991860956, 4.280584107618779e-06, 8.19659984330201e-09, 1.0831033137037593e-07, 6.393785270120134e-07, 7.636435839231126e-08, 7.160239762216136e-11, 4.997744995627329e-10, 8.007847327462514e-07, 1.2031576943627442e-06, 1.7299302612627798e-07, 3.293049832819861e-09, 5.152913217898458e-05, 3.355137232574634e-05, 3.341143584023598e-09, 2.4348310034838505e-08, 6.471034907917783e-08, 1.2083225087167193e-08, 3.2165641528081323e-07, 2.7704677663109578e-08, 6.745665692697855e-10, 9.081467799987308e-10, 8.82596040696626e-09, 7.387051937257638e-06, 3.2783835646199577e-09, 4.271700149161006e-08, 2.0842657022512867e-07, 9.823133240161042e-08, 1.0928452587322113e-09, 2.704737482872588e-07, 7.515770050758874e-08, 1.2721670827886555e-05, 8.854547559167258e-06, 4.056817815012437e-08, 1.434480907391844e-07, 3.392792535805711e-08, 1.4679202422485105e-06, 1.9952358343289234e-07, 3.081449676756165e-06, 3.724959007911366e-09, 1.642293767645242e-08, 5.412437531049363e-06, 3.108994235390128e-07, 2.5108647605520673e-05, 3.995031416792472e-09, 9.735398407428875e-07, 1.773598455656611e-06, 8.48617703885779e-11, 4.937930064841112e-10, 1.2678645688879442e-08, 1.6713178396798867e-08, 9.062239847423825e-10, 5.411061820836949e-08, 2.2847708447404358e-10, 1.4791317880735733e-05, 1.3257772657482292e-08, 1.5468210676772287e-06, 2.3686461645411327e-06, 2.0804027656140534e-07, 9.560299076838419e-06, 5.7397429031880165e-08, 3.504791311570443e-05, 7.164249055113814e-11, 0.0024816833902150393, 1.7238728105439804e-05, 1.0096161531691905e-05, 8.161235314219084e-07, 2.960467554657953e-06, 1.7384030215339408e-08, 2.566057162312063e-07, 4.876746784177044e-10, 1.9124188384012086e-06, 7.638328725079191e-07, 8.105225823840101e-09, 8.803099582621599e-09, 4.794588193135496e-08, 6.63765149511164e-08, 1.764782666668907e-07, 1.456065712446275e-09, 2.8242279768164735e-06, 6.610284344787942e-06, 8.66993377712788e-06, 1.699697271817513e-08, 2.5638158263063815e-08, 6.213046503944497e-07, 2.0825025615067716e-07, 8.516561138094403e-06, 1.7860909906630695e-07, 2.9326553629971386e-08, 1.6273773439934303e-08, 1.682248154111221e-07, 1.8353853992181257e-09, 3.521649416526884e-09, 4.34136371296745e-09, 3.3056708161893766e-06, 2.7106519695507814e-08, 1.7055340473248748e-09, 0.001197072328068316, 1.2908445146120329e-10, 2.804850964821526e-06, 7.063595603540307e-06, 5.407534331425268e-08, 0.000864568050019443, 4.001937668363098e-06, 3.0278141105455347e-10, 2.0001499123623034e-08, 7.349113673171814e-08, 2.7230612431594636e-06, 6.594171736651333e-06, 3.0392347980523482e-05, 7.44231840599241e-07, 6.640001326552181e-10, 8.094966119642777e-08, 0.016092592850327492, 2.7430823237861723e-11, 1.5639304296044543e-09, 2.2480555611492647e-11, 9.254724631091449e-08, 2.954213262906791e-10, 3.5105063034279738e-06, 2.861769985429419e-07, 4.482302884412093e-08, 7.792048677401908e-07, 8.451458199942863e-08, 1.8067725050460726e-12, 9.369982763018925e-06, 4.1476124579276075e-07, 5.3857154291847564e-09, 3.439903251134524e-09, 2.183119249821175e-05, 7.516041478083935e-08, 2.528053846617695e-05, 3.0759649689571233e-06, 4.276862455299124e-07, 5.3577853265096564e-09, 7.84529277098045e-08, 2.1992354604094544e-08, 5.050369722425785e-08, 7.921548750289276e-09, 5.8252289818483405e-06, 9.148459503194317e-05, 5.287808448883879e-07, 4.79284644825384e-05, 1.4819618865402617e-09, 1.8652109190497868e-07, 8.553216801487906e-09, 0.000295520992949605, 1.000519773697306e-07, 6.812748409856795e-08, 2.367647744527801e-10, 2.2895329720995505e-07, 1.4953801530381838e-09, 1.0105795666959239e-07, 2.590703900295921e-07, 2.7575683247960114e-07, 2.4793231356845524e-10, 0.02344336360692978, 2.718521443512145e-07, 4.874229020401799e-08, 7.899228648966528e-07, 1.032225469543846e-07, 3.734515363618129e-09, 0.005780849605798721, 1.9576269494336884e-08, 1.5846046153455973e-05, 3.2747829448176446e-10, 1.8791181943811353e-08, 1.8042531735318335e-07, 9.458537419959612e-07, 2.0879615476587787e-06, 8.739397117096814e-07, 1.4073025411676099e-08, 9.262001166021605e-10, 3.792237066591042e-07, 3.263704684286495e-07, 2.045834968100735e-08, 2.4835916434540195e-08, 3.5453787128325587e-10, 4.578286336709425e-07, 5.282224879721298e-09, 5.028409355567476e-10, 8.653600502839254e-07, 1.4008799098519376e-07, 2.3450169805983023e-08, 3.0179285204212647e-07, 5.013911049900344e-06, 3.919879176805807e-08, 4.162660260931261e-08, 9.527454949420644e-07, 9.837285119829176e-08, 1.4388441549328945e-08, 2.3825010430300608e-05, 7.414396918647981e-07, 1.3017898936595884e-07, 4.077647020039876e-08, 0.0001848357787821442, 9.706302428469371e-09, 0.0007263782899826765, 1.0478801414137706e-05, 9.331450200988911e-06, 7.67391872269485e-10, 2.733918336161878e-08, 7.394825862760968e-10, 2.653030906074605e-09, 4.335684877787571e-08, 2.8215228553563065e-07, 1.780418756425206e-06, 3.150383953709479e-09, 4.655219543181488e-10, 3.308644227217883e-05, 9.072182649561e-08, 2.8854623224106035e-07, 6.024711751706491e-08, 9.96749278669995e-08, 3.104274881593483e-08, 1.226832864631433e-05, 5.193474095221973e-08, 4.580013701627195e-09, 1.0403981676176954e-08, 7.19879562893766e-06, 5.985189427404336e-11, 4.0673661715118214e-05, 1.2337281196650451e-09, 2.8273156203795224e-05, 3.346386438352056e-05, 4.107274076403655e-09, 0.0009237960330210626, 3.333423137519276e-08, 6.606043712054088e-08, 4.3085879042337183e-07, 6.028278107805818e-07, 2.0825924096357085e-08, 8.571627404307947e-06, 3.86771965565913e-08, 1.1939813703065738e-05, 0.09560318291187286, 1.4322648667075555e-06, 4.9856250683433245e-08, 1.9009570678463206e-05, 3.532447251375359e-11, 1.3778505447703537e-08, 1.4053370023248135e-07, 1.1519609710575196e-09, 0.00010808780643856153, 8.173979548775634e-11, 2.6648436235454653e-10, 7.019729242330186e-11, 5.273206582501189e-08, 2.046937481736677e-07, 2.4186149971683335e-07, 2.4551084720769722e-08, 4.016439625331714e-09, 1.1353300806149491e-06, 9.796338417800143e-06, 1.6182235995643168e-08, 3.5056953038292704e-06, 6.901527285663178e-06, 1.844849521148717e-06, 2.4977822477012523e-07, 4.303100098468349e-08, 8.063600489549572e-07, 6.403333827620372e-06, 3.21385273949204e-09, 3.654592184432204e-09, 2.853132627933519e-07, 5.56815304975089e-09, 9.53225480770925e-08, 0.00021040081628598273, 1.8535379808781727e-07, 1.2080872302533407e-08, 3.295662764912777e-08, 2.1027555874297832e-07, 1.0202690248206636e-07, 3.86290025744529e-07, 1.8787044609780423e-05, 1.633085311425475e-08, 1.8146643077088243e-09, 3.0186370736373647e-07, 5.290545232128352e-05, 1.1382306555618449e-11, 1.2252543228896684e-06, 1.985380748692478e-08, 5.5072744975515064e-11, 1.0872134481587636e-07, 1.0351453738621785e-06, 2.9094138209728726e-09, 2.765482731703628e-09, 2.3690878947491e-07, 8.179836252786288e-10, 9.471661677196153e-09, 1.7910249425767688e-06, 1.8713866012376457e-08, 2.8286811115663113e-08, 1.852804416557774e-07, 1.5799165795082892e-11, 2.3093200685053716e-09, 1.1230173413423472e-07, 5.9157376774976456e-09, 9.146734925025157e-09, 1.838597718517576e-08, 1.1185606219044075e-08, 0.009886637330055237, 2.9625912834063683e-10, 6.082933462492335e-11, 3.721347448942813e-12, 3.5362116790338405e-08, 1.280001242776052e-06, 3.2921596559987165e-09, 7.522561645600945e-06, 1.5662006802585893e-08, 9.818005963779797e-09, 3.171600315710066e-09, 1.0105819114869519e-07, 7.966390569436044e-08, 2.0693748865596717e-06, 1.512506031531302e-08, 1.8283277313457802e-05, 1.3782201904177782e-06, 1.8883604013808508e-07, 7.974288251944017e-08, 6.81424907611472e-08, 0.002722559031099081, 1.4191203263180796e-05, 1.6607650366040616e-08, 1.9214221880226035e-10, 7.0067343038715535e-09, 1.1942391964225862e-08, 2.0300998659905645e-09, 2.0514455911779805e-08, 4.2006342937384034e-07, 3.3893409181473544e-06, 8.81942526120838e-07, 5.373710791900521e-07, 4.165001428191317e-07, 9.237481002344339e-09, 3.38579638992087e-07, 1.8226847942059976e-07, 0.001365535892546177, 3.459801889427183e-10, 6.118614948036338e-09, 8.793095673809148e-08, 2.0726125171677268e-07, 8.306964360826896e-08, 4.838209277657768e-10, 1.7018018638736976e-07, 6.199879862833768e-07, 0.00029276395798660815, 8.112774896318342e-10, 1.1514780453580897e-06, 1.0907279940353121e-10, 8.58468354181241e-07, 7.830271329112293e-07, 1.3510148555440082e-09, 1.2341810906590922e-10, 4.546930085780332e-06, 5.239899833475725e-11, 4.812584109004092e-08, 1.6909854627655818e-09, 8.429281479038764e-06, 6.561003829119727e-06, 3.890945876605656e-08, 3.8133560309461245e-08, 2.7858561679749982e-06, 3.7434492696775123e-05, 2.2268418433668558e-06, 2.966117449432204e-07, 4.006692547187107e-11, 1.0184579650740488e-06, 1.2863670129092952e-09, 7.297408956219442e-06, 1.0211374501523096e-07, 2.8689461917252856e-09, 1.4641745593735322e-08, 5.100873386254534e-05, 1.329583643183696e-08, 7.449392796843313e-06, 7.238399462039524e-07, 5.2926434068734096e-11, 2.897117781230918e-07, 5.077657405649916e-09, 3.8214045616769e-07, 3.972449576394865e-06, 1.4090648647879789e-08, 1.6706432006685645e-06, 1.0612708933877002e-07, 2.7994853102342176e-09, 2.7235182642471045e-07, 2.006103096141576e-11, 6.663096741021945e-08, 3.721808972123597e-11, 3.7382527295903856e-08, 9.140723705058917e-06, 5.07207111866137e-08, 1.749549061180744e-09, 1.4487275379337916e-08, 3.10358518618159e-05, 1.466822254769795e-07, 9.602758836990688e-07, 5.815711574541638e-07, 2.460513144342258e-07, 0.00020508302259258926, 7.732038284302689e-06, 2.1216694676695624e-06, 2.2383680970960995e-06, 1.857279774419851e-09, 1.4670886230305769e-05, 0.00012073726975359023, 2.2283876432993566e-07, 5.005954790249234e-06, 1.0095275683852378e-05, 2.8292099774063217e-09, 1.7293655218963977e-07, 1.1022625479029102e-08, 0.00014196056872606277, 5.127243074554144e-10, 4.529669013209059e-07, 2.5076053589145886e-07, 5.358750665429568e-10, 7.872415219711826e-11, 1.7761935566795728e-07, 9.389667710024696e-09, 3.4032359508273657e-06, 5.572161398958997e-07, 5.9033954613596507e-08, 2.8882871094815243e-10, 2.8721200578729622e-05, 4.925674375044764e-07, 1.9655954019981436e-07, 3.237936880395864e-06, 3.754355049068181e-09, 3.5663783037875874e-10, 3.0140481044327316e-07, 2.2382504738516218e-09, 1.0629695452735177e-06, 1.2869495691347765e-08, 1.8622682773639099e-06, 1.7520877804155788e-10, 1.1304565061820426e-09, 6.073667790928994e-09, 3.988898544804442e-09, 1.2095995316485642e-06, 1.8762264517135918e-05, 7.401856692013098e-07, 4.776163908815079e-10, 1.6325179785781074e-07, 1.0041637521496227e-09, 2.095874584995272e-09, 1.4458310992893075e-08, 3.6835155015069176e-07, 2.886810648305982e-07, 0.00012022263399558142, 1.3837645838066237e-06, 1.689762996193167e-08, 8.684560270921793e-06, 5.934583100497548e-07, 3.0658848793052584e-09, 9.321905963588506e-06, 8.480868700644351e-07, 2.043539382157178e-08, 7.182781303072261e-08, 7.609682506881654e-05, 1.5492344118683832e-07, 6.502558790089097e-06, 2.9328607524803374e-06, 9.09167681584222e-07, 8.210214452741837e-11, 1.8825052165993839e-06, 5.2608480473281816e-05, 1.093324319967337e-09, 2.0540959155823657e-07, 6.248614425885535e-08, 1.5412141465276363e-06, 8.924396297516068e-07, 1.6575497918669413e-10, 2.5964125143218553e-06, 3.415279864471543e-11, 5.1537890612962656e-06, 0.00015203429211396724, 2.7964713877892677e-10, 5.809542358292674e-08, 1.5169601685904865e-10, 2.8708370791719062e-06, 1.0060082544782745e-08, 1.7992222467455576e-07, 0.0001519415236543864, 3.6378760341904126e-06, 1.0621122470411137e-07, 3.813487978732155e-07, 5.395318112277891e-07, 1.8542116322350921e-06, 7.889681796768855e-09, 8.44295328095157e-11, 1.8790389333389612e-07, 0.0007730664219707251, 8.986757649154242e-08, 4.085920224383699e-08, 2.052664704876861e-09, 7.098195453636436e-08, 3.839732301003096e-07, 9.933954459029337e-08, 4.3454878806414854e-08, 5.3589047865898465e-08, 1.5707756872984646e-08, 3.0864867994750966e-07, 8.123141048699267e-11, 6.145641577859351e-07, 2.1161611352127352e-10, 1.4153555127904838e-07, 1.4257401517170365e-06, 1.1722159909410834e-09, 5.391314061853336e-08, 2.8032117072029905e-08, 2.598937669517909e-07, 4.024122517876094e-06, 8.120189249893883e-07, 0.00013559710350818932, 7.25852601135557e-07, 9.306788228968799e-09, 3.7544698443525704e-06, 1.2574873835546896e-05, 1.1528537413596496e-07, 6.901138931425521e-06, 2.2661518883637655e-08, 3.980833795935723e-08, 1.4811555502092233e-06, 2.1334088273761154e-08, 4.052713848068379e-06, 6.008949782199124e-08, 1.609323163620502e-07, 6.9359575860517e-09, 1.3744867466414235e-09, 3.4681342242492974e-09, 3.887539534730422e-08, 4.235959295328939e-06, 1.8686940883583247e-09, 1.1697212087824482e-09, 2.7735806706896682e-11, 7.581864736039279e-08, 3.648092024377547e-05, 1.6175052408584634e-08, 6.302623312137712e-09, 1.7872121560458254e-09, 4.457382019040779e-09, 1.9307353227304702e-07, 4.34440650121104e-10, 3.289781034254702e-07, 1.0819572935361066e-06, 0.0014468962326645851, 6.238398686519986e-09, 1.278898098533432e-10, 4.14574014939717e-06, 1.0777977621501122e-07, 2.6535804664717944e-08, 4.0641015885967136e-08, 2.6264905272910255e-07, 1.7783570172014151e-07, 3.4179499834863236e-06, 1.9243330484641774e-08, 5.796496793664119e-08, 5.515912562259473e-06, 9.009672794491053e-07, 3.281074023586683e-10, 2.8728706524816516e-07, 4.0237158827949315e-05, 1.226377110974397e-09, 1.9859238697961246e-09, 3.322561559215842e-09, 2.207189595537784e-07, 4.30993862821083e-09, 1.6895652876769418e-08, 1.1071408678731132e-08, 4.170814463577699e-06, 1.605253174830068e-07, 5.7926353491666305e-08, 2.6490509785759286e-09, 3.214636024040374e-07, 3.107483714792636e-10, 1.0332448872191202e-12, 5.892464471912717e-08, 7.569832405351917e-08, 3.4984427657036576e-06, 3.193017050762137e-08, 2.5794864022543607e-09, 1.09481721466409e-08, 1.8455358130609056e-08, 1.8847229910079477e-07, 2.2071729688377673e-07, 1.8770502929754684e-09, 0.00038165433215908706, 5.34021182829747e-09, 3.5541330589694553e-07, 1.2393304160696061e-08, 9.56652639050315e-12, 2.3443330121608597e-07, 1.907883806495647e-08, 4.175781498361175e-08, 0.0001534424809506163, 9.480073117629217e-07, 2.6683974283514544e-06, 8.681678991706576e-06, 5.193990091356682e-07, 3.6692446858666017e-09, 0.09932297468185425, 2.205531927756965e-06, 1.9784115318222106e-11, 2.442915736367013e-08, 1.2745221056320588e-07, 1.079938005341119e-13, 2.3988955035747495e-06, 1.405667138243416e-08, 5.343290809811663e-10, 4.9636690313548115e-09, 2.489215278345114e-09, 4.186923092674988e-07, 9.87598713919624e-09, 8.490897585033963e-07, 0.00010483530058991164, 2.5233032374671893e-06, 5.662902822223259e-07, 7.043376513138355e-07, 1.9039083554162062e-06, 4.13790417042037e-07, 9.060792116599714e-09, 2.0305787273855458e-08, 1.9876216583725181e-07, 4.869181793765165e-05, 1.9385904579394264e-06, 3.682413591832301e-07, 1.4816026805419824e-06, 9.416424973096582e-09, 3.0274063078650215e-07, 1.5493804850219561e-10, 7.221709097393614e-08, 1.7861983891975797e-08, 8.640280384497601e-07, 1.727878498058999e-07, 6.650652358075604e-05, 2.3416304451018277e-09, 7.539703688053123e-07, 2.3041075714047565e-09, 1.8628481939231278e-06, 2.742824189994053e-09, 1.2204011412553228e-10, 7.99874833035119e-09, 1.0615600842811546e-07, 3.9178721635835245e-06, 4.051560753559613e-10, 1.1593671578324916e-09, 9.412794987895268e-09, 8.721077193740712e-08, 5.3763052676458756e-08, 6.141454832686577e-06, 5.449886884889565e-06, 1.2043216884194408e-05, 9.373351872454805e-07, 3.376693030077149e-06, 6.914089439868931e-11, 3.4401880839141086e-05, 1.2558255546579744e-09, 9.891094805425382e-07, 4.521851643124464e-11, 2.4303828283223083e-08, 0.00019700966367963701, 3.0821233849565033e-06, 2.543469292959344e-07, 5.2741717243565134e-11, 1.2460110720979856e-09, 5.1440235182553806e-08, 1.0089510169564164e-06, 2.932196139227017e-07, 4.3413677097703385e-09, 1.1153524610563181e-06, 0.00014638644643127918, 9.95791833702242e-06, 3.153315830672909e-08, 3.3956348843844353e-10, 3.1645446142647415e-05, 4.6072688775211645e-08, 3.929173875150127e-09, 8.568359355365374e-09, 1.7113074290975305e-09], [6.965742045395018e-07, 4.0360899333791167e-07, 1.1370045882586055e-07, 4.62155158587052e-09, 4.225102259169944e-07, 6.2130642852764595e-09, 4.511429452236371e-08, 6.47050590885101e-09, 7.966928450287014e-09, 4.0713231896916113e-07, 2.0475052209256717e-10, 0.00013106648111715913, 0.00028487524832598865, 1.477306227570807e-06, 8.925444126361981e-05, 0.0005541580612771213, 3.355910109803517e-07, 2.3981958293006755e-05, 3.0608913220930845e-05, 1.2176619179626869e-07, 2.3812873894968334e-09, 1.7599862678352451e-09, 1.4891978935338557e-05, 8.86048734383138e-10, 2.02682144845312e-07, 1.1332377880535205e-07, 1.5594940894203546e-09, 3.6273790637864067e-09, 2.310400093463727e-09, 6.597120955120772e-05, 2.856881131663158e-08, 4.3501270852175367e-07, 4.97437952162727e-07, 1.9713355925432552e-07, 3.1688352919445606e-06, 5.168109051822967e-09, 4.6007099285816366e-07, 2.164948682548129e-06, 4.76197465104633e-06, 2.8929765250040873e-09, 1.363557515787761e-07, 2.073243088318577e-08, 2.049586989016916e-08, 6.750169490032931e-08, 1.4569885298243435e-09, 1.8938018797598488e-07, 1.3160750313545577e-05, 7.408331725855533e-07, 5.2425845886716615e-09, 2.8955835773558647e-07, 4.124798124394147e-06, 2.3820959094145167e-10, 2.8534952178915773e-10, 4.3431803931071045e-08, 3.604811560364851e-09, 0.00013955100439488888, 1.7480623171195475e-07, 1.2640155944154685e-07, 6.504885163849394e-08, 1.0561410412890382e-08, 5.409055248151162e-08, 0.00014425875269807875, 3.543517479442926e-08, 2.348279485886451e-05, 3.5253628993814345e-07, 7.00603806080835e-08, 3.908163392907227e-08, 1.5024048671818946e-10, 1.8562292328216934e-11, 0.0004059282364323735, 2.4616715563752223e-06, 0.00095180090283975, 5.738556296819297e-07, 8.509525287081487e-06, 1.8570514725979592e-07, 0.00030719966161996126, 4.277910914396443e-09, 8.298529428429902e-05, 4.180301590994695e-11, 2.424373803933122e-07, 9.170695136617724e-08, 2.0199427908096368e-08, 3.5833352951541997e-10, 7.849279448635116e-09, 3.8942906144256995e-07, 0.0001667160977376625, 9.714796789239699e-08, 6.020979981258279e-06, 1.6014253811924561e-10, 1.1962792711983639e-07, 1.448161697226169e-07, 1.6289901395794004e-05, 1.1578799785638694e-07, 6.259142082853941e-06, 2.8187281714053825e-05, 6.271985171224514e-07, 3.295292799521121e-06, 1.7305909594256264e-08, 7.155673831249487e-09, 0.00015605096996296197, 1.4368637391015682e-09, 7.887149422458606e-08, 1.2627110663743224e-05, 1.066590129994438e-06, 1.1262530907885093e-09, 2.3826954986816418e-07, 1.2804392781617935e-06, 2.315344289627319e-07, 1.5074715520313475e-07, 8.044276000873651e-06, 6.50846004646155e-07, 1.547210501939844e-07, 1.3488130434780032e-06, 3.7832424693018396e-11, 1.1388283205349126e-08, 1.8517395801609382e-05, 5.7133203057446735e-08, 1.1973074833804276e-06, 3.531896481945296e-06, 1.0648992972150939e-11, 6.454806134037483e-10, 2.7400940894040104e-07, 6.358602036016237e-07, 1.594890086664691e-08, 0.00024101033341139555, 1.9019249748453149e-06, 2.6767148142425867e-07, 8.581448440736494e-09, 1.0576238235887603e-12, 2.7650811418311605e-08, 3.715477481236462e-09, 2.3361723666681655e-09, 8.75964456525935e-09, 1.3946173567092046e-05, 2.821099798211435e-09, 4.218220084339919e-08, 4.0235775600194756e-08, 1.7167962607089748e-09, 1.5317684898263906e-08, 1.3755730776665587e-07, 1.0930466487479862e-07, 2.1834245522711626e-10, 0.00022781456937082112, 0.0001581096148584038, 9.92166491187163e-08, 1.0131333283425192e-07, 6.414504927221287e-09, 1.2522369807754785e-09, 8.382252758565301e-07, 8.980234333932913e-09, 1.916932212964184e-08, 1.192616849721162e-07, 1.499805790672326e-07, 4.5437781182045e-07, 1.0307073949888945e-07, 0.0002338710764888674, 8.853113797613332e-10, 5.5326516701370565e-09, 1.1493946658447385e-05, 5.5014137245734673e-08, 4.853009158978239e-07, 1.1918930198362432e-07, 1.7232457238947063e-08, 1.5832862132469927e-08, 1.6225407684089532e-09, 9.276547530134849e-08, 0.00020771000708919019, 1.6664322174619883e-05, 0.000912281742785126, 4.5866704567743e-06, 1.7933980700490793e-07, 4.251157861290267e-06, 2.170608013329911e-07, 2.5240325385311735e-09, 5.1233945974615835e-09, 2.4590954126324505e-05, 2.4104204854324962e-08, 7.146168656824159e-10, 4.875176173868567e-08, 1.0666258276614826e-05, 6.684909294563113e-06, 2.7367576649339753e-07, 5.531083369092471e-10, 1.4921943147783168e-07, 2.2202470972843003e-08, 0.0033228974789381027, 7.680142033450466e-08, 1.302292389482318e-07, 2.770259355244775e-09, 2.1258335369811476e-10, 1.2054351827828214e-07, 2.322394578868625e-07, 7.80359041527845e-05, 2.869442461417293e-08, 9.003779410932111e-08, 0.00010612932965159416, 1.8371412124906783e-06, 2.8039746666763676e-06, 2.9059765438432805e-06, 4.090629224240416e-11, 9.573532224749215e-07, 0.002593802521005273, 1.5700422295594763e-08, 3.5571785517163335e-09, 1.2315068488533143e-05, 8.237228757934645e-05, 3.721386576671648e-07, 2.8095638662795075e-11, 1.0520478932463107e-09, 3.5608858084401618e-09, 6.282863118656223e-09, 2.0485844132167585e-09, 4.495087679856624e-08, 1.5840250000564993e-08, 8.622802738500468e-08, 2.3871785970186465e-07, 9.92704443092407e-08, 4.8768016291944605e-09, 1.0924972002612776e-06, 4.5855141905803976e-09, 3.4949536598105624e-08, 2.6325210455979686e-08, 2.3899499410617864e-06, 8.144245100538683e-08, 3.9544499941257527e-07, 1.1569586400028697e-09, 1.3171510317988577e-06, 1.483968361526422e-07, 8.301913112518378e-08, 7.236797500809189e-06, 3.9677119367809155e-09, 1.0525781135584111e-06, 8.698889525371456e-11, 3.6622225252358476e-09, 1.5874667951720767e-05, 8.83940742824052e-08, 6.315174494453402e-10, 1.907113255583681e-05, 3.210294039490691e-07, 4.4213660999048443e-07, 3.107998858276062e-10, 2.693998567337985e-06, 1.2168253533673123e-06, 3.651657110026463e-08, 1.1608875638557947e-07, 7.063021278952419e-09, 1.775048730223716e-07, 8.35613189753559e-10, 8.964293556346092e-07, 3.1620737672710675e-07, 2.5167086192823263e-09, 1.3518034336357232e-07, 4.655687391164065e-09, 8.814063789941429e-08, 6.022390919291354e-10, 2.861629155859191e-08, 0.00016696550301276147, 1.4276416777647682e-06, 1.6299985873047262e-05, 0.0002653873525559902, 2.6314230794355353e-09, 1.4087179067701072e-07, 4.976246081866975e-09, 8.475015783915296e-05, 2.4453571313642897e-05, 0.00011569014168344438, 7.686783431992694e-10, 1.9684050130308606e-05, 1.522835475498141e-07, 8.990581612522419e-09, 4.247719971317565e-07, 4.896105743057433e-10, 2.003429699470871e-06, 1.625654055814607e-09, 1.2191952336593204e-08, 1.7282894759773626e-06, 3.4472607239877107e-06, 2.946010454252246e-06, 1.9448016530532186e-07, 3.916122182090476e-08, 2.482530874203803e-07, 4.491555571917161e-08, 5.045168109063525e-06, 6.728964763169643e-07, 1.605448431973855e-07, 2.7901085104531376e-06, 1.4831690320349367e-09, 3.645338475521953e-09, 6.786682206438854e-06, 1.1767633623094298e-06, 1.5500023664571927e-07, 7.538833983744553e-07, 4.772632422600509e-08, 1.4483908294948833e-08, 4.0017198443820234e-07, 6.223745458555641e-06, 1.4904737327015027e-05, 7.197034506134514e-07, 1.2371215518669487e-07, 1.0243611514582085e-09, 3.731579454324674e-07, 2.6712744016776924e-08, 2.8186871077195974e-06, 6.151271048082663e-09, 4.437175604721233e-08, 2.3091479839365547e-09, 4.392145001475001e-06, 1.454870584893797e-06, 1.1452404996559196e-10, 6.680270558945267e-09, 1.6286544735066855e-07, 2.3159607735578902e-05, 0.00026130792684853077, 1.1433080544520635e-05, 6.559231735536741e-08, 1.1529326684467378e-06, 6.3247607151595275e-09, 6.27812187303789e-05, 1.0745542233792094e-09, 1.5953174892224808e-10, 5.41889846772392e-07, 7.701709364482667e-06, 2.4508297428837977e-05, 8.539726650269586e-07, 1.8110607697963133e-06, 2.7348823650186205e-08, 2.993578362975313e-08, 3.2570237618045894e-09, 4.2326627180955256e-07, 1.949732153505579e-09, 4.440591450105558e-09, 1.6256795909441735e-09, 7.976537972353981e-07, 6.746352028130786e-07, 2.4777710905254935e-07, 6.71814450470265e-06, 2.5827598393313167e-10, 2.048238769702948e-07, 1.6824518752400763e-05, 1.7140848285635002e-05, 2.782657881894579e-09, 4.6055455982241256e-07, 2.1860113719185392e-08, 6.620297909876172e-08, 1.1161088053768253e-08, 6.950413133921529e-09, 5.597673293777916e-07, 9.899074939312413e-05, 3.1030261027353845e-08, 9.646143439567822e-08, 2.1050571008540686e-10, 8.916660476643301e-08, 1.4754012056528154e-07, 8.970912546146792e-08, 3.0068194973864593e-06, 2.1735564587288536e-05, 2.9951316982135268e-09, 4.299491873638317e-09, 2.8589861589267684e-09, 2.607074378602192e-08, 3.0522633309715275e-09, 3.571389015633031e-07, 5.455634277673482e-10, 3.7611272429849407e-10, 5.947844439191385e-10, 1.1949521372400795e-09, 3.8167516436260485e-07, 7.393704777314269e-07, 3.8255782897067547e-07, 5.719161890738178e-07, 1.42965546956475e-07, 1.279911975871073e-05, 4.6311622980965694e-08, 1.0459548320795875e-06, 9.405380114912987e-05, 4.527047181568378e-09, 4.73910644060993e-09, 7.554278624866129e-08, 0.0001944589603226632, 0.0006812127539888024, 1.3370716089866619e-08, 0.00024100596783682704, 5.7337775416499426e-08, 1.8467264384369741e-09, 4.5507596269089845e-07, 7.951275904360955e-08, 6.684462050543516e-08, 1.0736466160565783e-09, 3.854517593993023e-09, 4.781191478286928e-07, 1.5867448155404418e-06, 3.616019483843047e-08, 9.290117564120237e-10, 9.249307453274014e-08, 1.2147712880050676e-07, 2.712492230827479e-09, 3.2355558232666226e-07, 3.488088751169016e-08, 4.989389253751142e-06, 9.358582353513611e-09, 1.941589289344847e-06, 1.2468026397982612e-05, 8.272921792062959e-10, 2.570252988221e-08, 4.754417415320233e-10, 8.396080630745928e-08, 7.424109327303086e-08, 3.4169509444126334e-09, 3.4236991268699057e-06, 2.745663323366898e-07, 1.4772362533221894e-07, 3.840739637439583e-08, 1.7669748997306556e-09, 1.366192736895755e-05, 2.436935808702856e-08, 2.5385797925991938e-06, 2.0039284901601784e-12, 9.501119535570979e-08, 4.133751674295638e-11, 1.3776936036435927e-08, 4.035576239402872e-06, 1.2308524333093374e-07, 1.558320361638721e-09, 7.408570581901586e-06, 1.2605457788694707e-09, 3.384059144018181e-10, 7.848694139056533e-08, 1.6962350968263706e-10, 0.010752140544354916, 6.930874224053696e-05, 4.081921467502525e-09, 3.780617407755926e-05, 1.4364867411131854e-06, 6.411253750115975e-10, 8.931626325647812e-06, 6.146357378611356e-08, 4.966294682162697e-07, 1.169068127637729e-06, 1.7463587482779985e-06, 2.378190533391944e-09, 8.50320157042006e-06, 6.510229866307782e-08, 0.0009087129728868604, 2.4945139287524398e-08, 1.1648312314704867e-09, 3.2700449992262293e-07, 1.1399186455873789e-10, 8.738430118393481e-12, 9.565751213358453e-08, 3.77367859982769e-06, 6.698191441500967e-07, 5.867976327067481e-09, 2.332240178759548e-08, 1.8207616525600656e-09, 4.063318925773274e-08, 2.5231592815089243e-08, 3.4025124477921054e-06, 4.00825790336512e-08, 0.0002880643878597766, 4.74017314289199e-09, 6.873104325677559e-07, 2.40131453210779e-06, 7.635656515958544e-07, 3.634844460975728e-06, 0.0019949167035520077, 9.85633050731849e-06, 4.161148808634607e-06, 9.19261401577387e-06, 5.45593081824336e-09, 5.948870840377651e-10, 2.2134347688052003e-08, 1.7689492093353465e-09, 1.5139055449253647e-06, 8.047869662353158e-12, 3.4135109672206454e-07, 1.1868010005855467e-06, 4.109307560895559e-09, 5.9233993852103595e-06, 1.468859949227408e-07, 1.219884666170401e-06, 1.2159300233349768e-08, 3.0707824407727458e-06, 0.0007164542912505567, 3.635024938830611e-07, 9.949658306140918e-07, 4.556902695185272e-06, 6.60020702980546e-07, 2.3929238590625346e-09, 1.5248264162437408e-06, 3.5504363338212386e-11, 1.434985733794747e-06, 4.487282989629193e-09, 2.124415686921566e-06, 4.136057395953685e-05, 1.4955331417709772e-09, 1.1851733461298863e-07, 7.449557415384334e-08, 3.296716144518541e-08, 3.563319594945824e-08, 8.695132391878246e-10, 0.00010860084148589522, 1.672839289312833e-08, 2.489833894614435e-10, 2.571512823124067e-06, 3.3715163993974784e-09, 6.69135076236671e-08, 1.4395594405414158e-07, 2.3188036213372953e-10, 1.1978240763710346e-06, 6.993547003730782e-07, 1.873668225016445e-05, 2.8740534951943175e-10, 1.4033172135441419e-07, 4.313753549922694e-07, 1.938829541359155e-07, 1.4818204363109544e-05, 1.6868010277448775e-07, 6.65946444655674e-08, 1.4605414833113173e-07, 5.441317085796982e-08, 8.534390758541122e-07, 7.541724578175035e-09, 2.9523130606889936e-08, 5.180801849746786e-07, 1.1308634384477045e-05, 2.656873903106316e-07, 1.0136342098121531e-05, 1.0510009751385496e-08, 5.820919213306297e-08, 3.6248911783332005e-06, 1.1630037377585722e-08, 2.8403819669620134e-05, 8.119038427678582e-12, 2.063647571048932e-06, 8.597300649171302e-09, 3.1448172421733034e-07, 1.4586910879188508e-07, 1.7360290485157748e-06, 3.24036852905607e-10, 3.138115900469529e-08, 2.4458679035888053e-05, 8.165662848114152e-07, 2.0666693245630086e-08, 1.1090918405898265e-06, 1.56668402695459e-08, 1.1566098773174716e-11, 8.751002589235668e-09, 3.2392068760600523e-07, 2.5298355077474355e-10, 8.005668905575192e-10, 4.483636644181388e-07, 2.691165718715638e-05, 1.932114912506222e-07, 8.542352936125042e-10, 3.5370607065488e-07, 1.487336476202472e-07, 6.704867416829075e-08, 1.5323570323744207e-07, 3.8252272460681525e-09, 3.750788124534665e-09, 1.1114674736134589e-09, 0.000136743372422643, 1.1227833965676837e-05, 1.1083222117846958e-09, 1.0196445465737725e-08, 2.5879487441926585e-09, 2.2262861421040725e-06, 1.9227309167035855e-06, 3.095876672887243e-05, 4.6022410060686525e-07, 2.125125497798308e-08, 3.391645577721647e-07, 1.2318201925154426e-06, 2.634294382630742e-08, 8.814079244245931e-09, 2.1172510411560097e-09, 1.3650577784574125e-05, 1.2845080732404313e-07, 7.56958979764022e-05, 5.8304799495090265e-06, 2.5847273832368955e-07, 1.5798410402112495e-07, 3.3338125149384723e-07, 3.8256091627886235e-09, 5.580550688222274e-09, 4.426057387263427e-07, 1.0302158015562668e-09, 9.421678987564519e-05, 2.2200518287718296e-05, 2.4679909671476707e-08, 9.996039551651847e-08, 5.088489274385211e-07, 6.380618700063678e-09, 7.235045895725989e-09, 5.0166182319344443e-08, 6.507133366540074e-05, 2.9772979814879363e-07, 7.728194759692997e-06, 1.8722370875856598e-11, 2.8206925435370067e-06, 1.0106715535584954e-06, 4.063900576056767e-07, 3.670080350737237e-10, 1.1542693982846686e-06, 1.8870925089231605e-07, 2.8877673230454093e-07, 2.620804129271903e-10, 0.00026784505462273955, 5.6462738484697184e-08, 2.1506347813726734e-09, 8.952504515491455e-08, 7.199921014944266e-07, 7.068512530850057e-08, 1.7950052555448792e-09, 7.440996796503896e-07, 3.733832636498846e-05, 1.7185982414957834e-08, 1.4161612682528357e-07, 5.848972364219662e-07, 4.3806312532979064e-07, 5.5854849634329184e-09, 3.754786121135112e-06, 4.618418980584238e-10, 9.158051028634873e-08, 1.306917738475022e-06, 2.1303811081452295e-05, 4.562090635817917e-11, 1.8175775039708242e-05, 1.2383316970954183e-05, 1.2289209649907207e-09, 0.0001866967068053782, 4.651982621339812e-08, 1.8839159565686714e-08, 3.159459538437659e-06, 3.0291791297543114e-09, 1.1123127478640527e-05, 1.8185099259770965e-11, 4.57662174824236e-08, 4.5129629200335586e-11, 1.841247831180226e-05, 6.644279437750811e-07, 2.0921601256995714e-11, 1.5829735744432583e-08, 0.00011394123430363834, 8.638161119733923e-08, 2.0099067267291548e-13, 2.1452440250868676e-06, 6.629784221612312e-11, 4.9671136537199345e-08, 0.001566613675095141, 2.67603354586754e-07, 2.6836664801521692e-06, 2.026589027082082e-05, 1.1955351153503102e-09, 1.4139915016642135e-08, 3.313777341418245e-08, 6.5221820477745496e-06, 1.6908465738652012e-08, 1.7957958675651753e-08, 5.2479546042150105e-08, 4.173276391128411e-09, 1.6165795102551783e-07, 5.523769441850845e-09, 2.162663008675736e-07, 1.354483515569882e-06, 4.428711619652859e-09, 1.3649199104293075e-08, 3.575489415652555e-07, 1.284929829203918e-09, 2.2590431399294175e-05, 3.189925212865319e-09, 2.703165193906898e-07, 1.540850860237697e-07, 1.8513880206683098e-08, 1.6861715721461223e-06, 1.6568718308462849e-07, 5.679230002941793e-11, 1.544402117303889e-08, 1.3098687304591294e-05, 9.594565142378997e-10, 0.017733590677380562, 0.000284622423350811, 7.936267643060546e-09, 8.222022734116763e-05, 7.06221248147898e-10, 1.0884825485391048e-08, 2.3512429834227078e-05, 0.0009180515771731734, 7.3234941737609915e-06, 1.5107693798199762e-05, 2.5819414162242538e-08, 5.79960861557538e-08, 1.0678384398943308e-08, 2.3840796004037657e-08, 8.257774766207149e-07, 1.023932964017149e-05, 0.016398919746279716, 3.61853637331766e-11, 3.033157502940753e-09, 1.209050310535531e-06, 5.394702839112142e-06, 4.279685583696846e-07, 8.106909810123852e-09, 5.642533107419467e-10, 0.00010082896187668666, 5.737931019211828e-08, 5.951851989038914e-08, 2.8474943292167154e-07, 8.67248309077695e-06, 0.00027620248147286475, 6.926714559085667e-05, 5.147156116436236e-06, 1.1556845436189178e-07, 2.188618509535445e-06, 3.8773117694290704e-07, 3.7247238626747503e-09, 9.973561532206077e-08, 4.506447659480273e-09, 4.2927847387908e-10, 1.2754115232382901e-05, 1.5208736670047074e-07, 3.6430411465282475e-10, 3.628163653957017e-07, 4.5329063169674555e-08, 4.803535347264187e-09, 2.758132779945299e-07, 0.0015016187680885196, 1.8688057252802537e-06, 1.1150827106121142e-07, 9.081752239126217e-08, 8.295334197327975e-08, 8.054364286635973e-09, 4.410172227409248e-09, 1.967771640920546e-05, 3.139374760152691e-09, 9.88173951554927e-07, 7.647437683111491e-10, 3.528073193592718e-06, 7.554404324316977e-10, 5.281304993332014e-07, 0.0003448008210398257, 2.5005932879906823e-09, 1.9814419829344843e-06, 1.1403901680751005e-06, 1.7071277724767242e-08, 5.6863129316298e-08, 5.320488980942173e-06, 9.358552821581156e-10, 2.4841639856276743e-10, 1.2296909268627587e-08, 1.4507796075591273e-09, 2.1232875013765806e-08, 2.157844875005921e-10, 2.4133015585903195e-07, 1.7573161414929928e-07, 2.3077352580003208e-06, 1.2009823358027916e-08, 9.45364354265621e-06, 7.053002718748758e-06, 4.090894890396157e-06, 4.301520561966754e-08, 7.366287491095136e-08, 0.000950856483541429, 5.44381562050944e-09, 3.137391235696896e-09, 9.99491021502763e-07, 8.061022960248465e-09, 2.6575099898451526e-09, 1.273581240468502e-08, 5.642730229737936e-06, 1.607670940817485e-10, 6.72072131280288e-09, 0.00013819285959471017, 7.946016289395175e-09, 5.312795203771259e-10, 9.302245640441242e-10, 5.598101182613391e-08, 6.75727051202557e-07, 3.965664241434297e-09, 2.8304886212637825e-10, 5.556665794159699e-09, 2.7038305461246637e-07, 1.889437584168263e-07, 8.412908937316388e-05, 2.1750231837813772e-07, 9.209857410041877e-08, 0.00040386951877735555, 9.383519682160113e-06, 3.0531359129781777e-07, 4.668221809112083e-09, 1.640905111344182e-06, 2.085582373467787e-09, 4.9758600795257735e-08, 5.495090604767938e-08, 1.0098126494995086e-06, 2.4330713131348602e-05, 4.890738924956395e-10, 0.0009875280084088445, 2.6393017549253273e-08, 0.00014086022565606982, 3.345560495304767e-10, 8.423104418397998e-07, 1.899351786960324e-09, 1.948795153028371e-10, 3.215971844383603e-08, 7.04986824473508e-09, 8.535666617959237e-11, 2.6499569205640228e-09, 1.5622913451807108e-06, 1.084873815671017e-06, 2.1135143413175683e-08, 1.6923390688816653e-08, 1.9069326526732766e-06, 8.27603496844631e-09, 1.017528061808548e-09, 8.169693614945572e-07, 1.678898975399079e-08, 2.1423949192467262e-07, 3.755148782147444e-06, 0.016938690096139908, 2.347193799323577e-07, 4.617918047955527e-09, 1.7572367141838185e-05, 1.5284078447308502e-09, 4.882635380454303e-07, 4.871013061347185e-06, 1.6312499795390067e-08, 3.730723108219536e-07, 5.794433732830839e-08, 1.855846254450455e-09, 1.6695872773198062e-06, 0.0006101008038967848, 3.219403515686281e-05, 9.653365218298404e-09, 4.4224114503776946e-07, 4.328543912190952e-10, 2.3186894395621493e-06, 2.0218683616235467e-08, 2.098881068945957e-09, 1.544910310791181e-09, 3.1682958478995715e-07, 1.9827612618428248e-07, 5.401672997784601e-10, 3.146079689031467e-05, 6.176659878009971e-11, 3.530217540514968e-08, 6.819824420745135e-07, 1.0769072105176747e-06, 1.078406530741205e-10, 1.7504867555473425e-09, 8.455986630906409e-07, 1.0716688848333433e-05, 1.1770208141470562e-09, 2.5866333075441617e-08, 6.920994565007277e-06, 4.338386361268931e-07, 3.230333689430154e-08, 2.5866903285987064e-08, 5.843433427799027e-06, 1.8955153109345702e-06, 1.974981245211893e-07, 5.00499345434946e-06, 1.1181353620770551e-08, 2.2402402066745708e-07, 9.801532542041969e-06, 1.4078668755246326e-06, 6.449757847803994e-07, 2.733840931412601e-10, 1.699188487691572e-06, 5.206516107136849e-09, 1.606149794497469e-06, 8.566116566077753e-11, 5.1604142754513305e-06, 1.457967435669616e-08, 8.481862781573035e-12, 1.9086333850282244e-05, 1.4004708503989605e-09, 2.5954871585298633e-09, 5.079134890451087e-09, 4.3645322023166955e-08, 9.536094148643315e-05, 1.2288387551961932e-06, 8.47588489705231e-06, 0.00010193590424023569, 3.1204765438275217e-09, 4.688394561469522e-09, 5.727562868429459e-09, 5.4065826589067e-06, 4.849763968195475e-07, 3.2101254987537686e-08, 3.5535686038201675e-05, 2.2548240963260469e-07, 1.0973445796480519e-06, 4.1726269728314946e-07, 7.927609857460993e-08, 2.5184558438695603e-08, 7.847943805927571e-08, 3.2413383088680803e-09, 1.127232189901406e-05, 2.340323135285871e-06, 2.765491899481276e-06, 9.156154305856035e-07, 1.2564920837121463e-07, 2.062376528236265e-12, 2.7297228371025994e-06, 8.080976385826943e-07, 6.524137674546182e-09, 1.595564782519432e-07, 2.3078108313256962e-07, 2.999717980856076e-05, 1.6903057087347406e-07, 7.430556170362479e-09, 5.3815612773178145e-05, 4.544668807948682e-10, 2.377325536428998e-08, 1.5875262909803567e-11, 0.006872961763292551, 2.0890328755740484e-07, 2.9981045424065655e-10, 8.380133253993449e-10, 1.2234623909535003e-06, 3.180047869477676e-08, 1.8838979265467515e-09, 2.4674278392922133e-05, 1.8375432375705714e-07, 1.3783242991394218e-07, 0.011574416421353817, 0.00011796280887210742, 1.669234528378638e-09, 3.932158776365213e-08, 3.8692427040132316e-08, 3.531790326860573e-08, 2.576722391012254e-09, 2.779302667477168e-05, 1.8837553170669707e-06, 2.0861479299583152e-07, 5.773263533903616e-10, 2.5607047859921295e-07, 1.4138937132202045e-08, 0.000237255371757783, 3.3539603805365914e-07, 1.2480157352001697e-07, 1.8307748632651055e-06, 8.273470086805901e-08, 1.1691575707573065e-08, 0.18854738771915436, 1.7927735740386197e-08, 1.2553938177006785e-05, 1.2319782694003578e-10, 1.8037734434983577e-06, 1.8616881789057516e-05, 1.2290150834814995e-06, 9.624557606002782e-06, 1.3230061490787648e-10, 1.4058469055555634e-08, 2.2700612500869966e-09, 2.5617139272071654e-06, 7.950988560878613e-07, 1.4237562027119566e-06, 9.332087991253957e-09, 1.022163587549585e-06, 1.1960950985212548e-07, 6.569591448624124e-08, 2.087581219001322e-09, 6.290697456279304e-07, 1.7427377088097273e-06, 7.203188374660385e-07, 3.5861000835524237e-09, 2.1363837277021958e-07, 0.0012512248940765858, 9.219788488223912e-09, 2.285308028149302e-06, 2.2077377082041494e-07, 1.6936553270952004e-09, 4.3143091876807205e-10, 4.1879147829604335e-06, 2.8066901336387673e-07, 1.4662881575588926e-08, 0.002308553783223033, 2.1668853378287167e-08, 1.1125753189844545e-06, 2.232782314592896e-08, 2.5752989074590005e-08, 3.3442583458054287e-07, 1.964366447282373e-09, 2.8160802685306408e-05, 3.618783050995944e-10, 9.181478162645362e-06, 4.27081801035456e-08, 6.804819463468448e-08, 1.7353218026983086e-06, 3.4505976600485155e-06, 3.236860779409767e-09, 1.0440230902020176e-09, 3.6695778362627607e-06, 1.3540721965910052e-06, 6.178862804517848e-06, 0.0001560944365337491, 3.3325391086691525e-06, 7.179907779431005e-10, 2.755094499207189e-07, 1.373332736420707e-07, 1.273348146924036e-07, 2.3945038265082985e-05, 1.457724749798217e-07, 3.5159551771357656e-05, 1.980049368910386e-08, 1.7650820154813118e-05, 4.061587455250759e-10, 2.494799105079437e-07, 8.53252117849479e-08, 7.374358901302003e-10, 3.668477077667376e-09, 6.110000185799436e-07, 7.974032996571623e-06, 1.3614203453471418e-06, 7.201616085694695e-07, 2.431871280350606e-06, 2.29066792201138e-08, 1.820501580596101e-07, 1.2515382508127004e-08, 2.054377574722821e-07, 1.9475592694107036e-08, 7.159824377822588e-08, 2.6103648322756134e-12, 5.541280767573653e-10, 8.003838047443423e-06, 2.453823810810718e-08, 1.0803956840277351e-08, 2.9052651839833743e-08, 3.908330199919874e-06, 2.9254813853185624e-05, 1.4606915101467166e-05, 1.825210711103864e-05, 4.630063887844926e-09, 3.5578472079578205e-07, 4.130488659437326e-11, 1.2238792201912929e-08, 3.2129449323292647e-07, 7.777152859489433e-06, 4.117441858397797e-05, 3.3001424509793154e-11, 1.231931747724957e-07, 0.0004053727025166154, 4.353838622961348e-09, 9.998524674870168e-09, 1.8359423847869039e-07, 0.0002685461367946118, 4.497083955223857e-11, 2.556347311610807e-08, 0.00029908487340435386, 7.958534098406744e-08, 1.7153986009432742e-09, 2.0098676056790055e-09, 2.8163167371531017e-05, 5.28642772223975e-07, 9.916341696225572e-06, 1.7695063434075564e-05, 1.7409606698493008e-06, 1.4451646848101518e-06, 2.4448510771435394e-07, 7.863877726776991e-06, 1.183808384652707e-09, 4.395169526105747e-06, 3.45140693802648e-09, 2.2741218685951026e-08, 2.764294094959041e-06, 2.0555379620645908e-08, 8.140545529333743e-12, 7.841797433627562e-09, 1.8093670561825093e-08, 1.0360981406165592e-07, 8.578643573287081e-09, 1.542143218102865e-05, 2.31154628771435e-09, 6.821827724934337e-08, 1.6359326115988893e-09, 1.5643729511793936e-06, 2.7094793040305376e-06, 3.0810113003099104e-06, 3.6398803331394447e-06, 2.2467659732683387e-07, 1.0479814591235481e-05, 2.156017409049582e-08, 1.9737509404649245e-09, 1.4042067952857451e-08, 4.125909526919713e-06, 3.566422265066649e-06, 3.847729956873991e-08, 9.588791982650946e-08, 7.506352761588175e-10, 2.5301760686602393e-08, 7.694264780866433e-09, 1.586129315001017e-06, 2.1120358795201355e-08, 0.009875841438770294, 7.329520485654939e-06, 2.983642000131681e-09, 7.036517502001516e-08, 1.440455567325305e-11, 1.1145242062582383e-08, 4.4824491851613857e-07, 1.0136384093195616e-10, 1.1403831479128712e-07, 1.3251873376418644e-08, 1.0102453096294539e-08, 4.6902795247660833e-07, 1.5339918571499744e-11, 6.6247141106146046e-09, 3.847884144647651e-08, 2.858309744624421e-05, 6.186307022204574e-10, 0.0001334758853772655, 1.8162604646931868e-06, 2.074051508316188e-08, 1.5591068631692906e-06, 7.574895022344208e-08, 8.962267372680799e-09, 3.833385926554911e-05, 5.486551213351731e-09, 6.423121521947905e-05, 5.9281596520577295e-08, 2.880319698306266e-07, 5.448517015338439e-08, 3.0312601211335277e-06, 1.503452295992247e-08, 5.797025437459524e-07, 4.367124972759484e-07, 2.6369958661121018e-08, 1.9026442714675795e-06, 8.75392647259332e-09, 3.1778727134224027e-06, 7.5325679063098505e-06, 0.00015366107982117683, 1.4118533897544694e-07, 1.7164779819722753e-07, 4.7018225757256005e-08, 6.9375607480992585e-09, 3.5806108078517696e-10, 3.311658656457439e-05, 2.700979440106721e-09, 8.952204311185596e-09, 3.8720216366527893e-07, 0.0008252899860963225, 1.7521192829939025e-10, 1.2245939817745466e-09, 4.18916840771999e-07, 8.447252639598446e-07, 1.3892874051180115e-07, 1.97920826394693e-08, 2.80797038954006e-08, 6.215859826852466e-08, 4.520050424616784e-06, 1.320864430454094e-05, 5.28936325849827e-08, 8.89813691173913e-07, 6.810675404267386e-05, 7.163727104853024e-07, 2.3413146976736243e-09, 8.932115633797366e-07, 2.026564338564185e-08, 1.5951016507642635e-07, 2.8072238933418703e-07, 3.0949067877372727e-06, 1.7888814340949466e-07, 1.724166031635832e-05, 1.8071602880809223e-06, 1.0216304872301407e-05, 6.35775307955555e-08, 7.842520426493138e-05, 1.7928240225728587e-08, 1.565150142823768e-07, 1.99576299930726e-10, 5.4803603433128956e-08, 8.599378634244204e-05, 8.047590371873525e-10, 1.5304703993024305e-07, 9.958876745486123e-08, 5.386648354033241e-06, 0.0049443552270531654, 5.671809333307465e-08, 1.6401916980157694e-07, 2.830284229204949e-09, 7.528113201260567e-06, 5.896520633541513e-06, 3.634431493537704e-07, 9.814941222430207e-06, 1.6771554101069341e-07, 1.38332916321815e-06, 1.2666104964864644e-07, 4.772786965645537e-08, 0.00016295324894599617, 9.328487982074307e-11, 2.9175821758542497e-09, 1.3696669576290788e-08, 3.134783526093088e-07, 2.1802284777550085e-07, 1.5852512660785578e-05, 5.63736250569491e-07, 8.810887379695487e-07, 4.9897702410817146e-05, 2.483625110016874e-07, 7.510068371630041e-06, 5.096394306747243e-05, 2.0621470753212634e-08, 3.11462972604204e-05, 0.000193550149560906, 0.0032188540790230036, 1.5233453609653225e-07, 3.026114427484572e-06, 7.404105417663231e-07, 4.509443442657357e-06, 3.3597880033653382e-09, 5.588757900909513e-09, 1.3533277609667493e-08, 3.6433227990073647e-08, 1.5371320216672757e-07, 1.299017057115881e-11, 4.5829925511498004e-05, 7.670214330346425e-08, 1.5250509743935936e-09, 2.463020543075345e-08, 2.634723689221019e-11, 2.32254751608707e-06, 0.00012761457765009254, 9.819540025546303e-08, 1.9496890217851615e-06, 1.2917932679101796e-07, 4.5476971877178585e-07, 6.287435553531395e-06, 6.890377903800982e-07, 1.3032944723434525e-09, 1.685197332790267e-07, 8.733114782444318e-07, 0.0002669954265002161, 3.7002501063199134e-10, 0.00010208674939349294, 1.11145670445012e-08, 1.1416470442782156e-05, 7.342514436459169e-05, 1.8504638932270723e-09, 1.4763331668277146e-09, 6.1121547823006495e-09, 2.284762423698794e-08, 0.0002728652616497129, 2.805864207644504e-09, 4.321314918342978e-06, 1.6576498203593748e-12, 3.551775817900449e-10, 6.979425926090244e-10, 1.5409015759360045e-05, 1.950943850914655e-09, 6.805236694162886e-07, 1.680674870385701e-07, 2.605323174975638e-07, 6.007582209477391e-10, 2.3159283379925455e-09, 3.2979976527514054e-09, 1.8344021412985967e-08, 1.7864197161543416e-06, 8.047088044804696e-07, 5.469977537586601e-08, 2.861055236280663e-06, 2.687112088395338e-09, 8.951594132611262e-09, 5.226052923745783e-10, 8.860303069013753e-07, 2.4444769763931617e-08, 2.489783490489117e-08, 1.3822928045248428e-11, 1.4662751368632598e-07, 5.186972202864126e-07, 7.84130691044993e-07, 2.6234946659542402e-08, 5.157883151696296e-07, 6.0579977798624896e-06, 1.3009860140300589e-06, 3.9729056311443856e-07, 0.001781659317202866, 8.5683075212728e-07, 3.702966932905838e-05, 6.833785981541496e-09, 0.0005646466743201017, 4.079172413185006e-07, 5.533255489353905e-07, 2.5981108930928087e-10, 3.924665747945255e-08, 2.450429192890624e-08, 4.549280674392975e-10, 1.863740408225567e-06, 2.6860811885853764e-07, 4.599457668064133e-07, 3.6903107684338465e-05, 1.4312081297873647e-08, 1.7269004004560884e-08, 4.420960806328367e-07, 0.0010258947731927037, 7.1114727688836865e-06, 6.679533370856916e-09, 5.356205479145615e-10, 2.3628910383877155e-08, 2.1634782569890376e-06, 2.0337009853932386e-09, 3.452235830536665e-07, 3.16918846721137e-09, 4.610188142351035e-08, 8.655594001538702e-07, 1.9252681227044377e-07, 3.8604641705575204e-10, 3.336976703849359e-07, 1.050136311242511e-09, 1.013946530292742e-06, 1.7487978993813158e-06, 2.2464745341732595e-11, 5.295217597733881e-10, 1.4469785583060002e-06, 4.024224509180385e-08, 1.1766430816351203e-06, 6.000675966788549e-06, 1.2323164355620975e-07, 1.8098374710007192e-07, 9.786335795070045e-07, 1.0377461023836076e-07, 6.310222344652061e-10, 1.518228920360798e-08, 2.5390498592514632e-08, 6.630482829450557e-10, 2.7589477369360793e-08, 3.4027046069695643e-08, 5.41767747108679e-07, 8.170421210706991e-07, 1.2345847011374644e-08, 5.354323651118875e-09, 9.094144104437873e-09, 7.704436939093284e-06, 3.483508273505187e-11, 3.390386154933367e-06, 1.8492510633905113e-08, 4.487697538024804e-07, 2.51603071710349e-09, 2.7372063868824625e-06, 3.985122987160139e-08, 3.8013843095541233e-07, 1.0614979117917756e-07, 1.0180330356490686e-08, 6.444615428335965e-05, 8.005421250345535e-07, 5.888405780751782e-07, 1.8450200514052995e-05, 3.6353448463338767e-12, 2.064520776556833e-09, 2.774066274469078e-07, 1.0000209016425288e-07, 2.0189729184494354e-05, 9.915456757880747e-07, 8.133979711999473e-09, 1.8932813645733404e-06, 2.5452711938100947e-08, 5.5557599409894465e-08, 2.748783799688681e-06, 1.6938696717261337e-05, 4.3765687962604716e-08, 4.616264050127938e-06, 7.507276578166966e-09, 1.0452316018927377e-05, 1.0062176443170756e-06, 2.6683116516323935e-07, 1.3475296327669639e-05, 4.2054980440298095e-06, 1.8290071466253721e-06, 4.8165492927410014e-08, 4.2898317786921325e-08, 1.0714529707911424e-07, 1.8205013475380838e-05, 5.230946786838331e-09, 4.2589334725562367e-08, 9.423170865829888e-08, 8.074154720816296e-06, 5.0738108825498784e-08, 2.954035904778607e-09, 4.6065952119533904e-07, 1.1690254098084552e-08, 3.757288311589946e-07, 1.3827168459101813e-06, 5.3185445722192526e-05, 1.0789965365631815e-08, 7.876282381857891e-08, 4.423763755312393e-08, 6.7427149588183966e-06, 1.1427898982674378e-07, 9.841679826649852e-08, 5.809425260849821e-07, 0.00108131964225322, 1.9297024778097693e-07, 1.9263830530746873e-09, 9.389594168851545e-08, 3.998018804907133e-09, 5.338244069008624e-09, 2.4653803620822146e-07, 7.787313620610803e-07, 7.130130796895173e-08, 1.5823451349206152e-07, 5.8990064388808605e-08, 8.363688408508096e-09, 2.3703173610556405e-06, 1.3082682926324196e-05, 5.852261963923411e-08, 3.606959708690738e-08, 1.8981352695846e-05, 8.669449016451836e-05, 5.722909079963756e-08, 4.4972814805532835e-08, 4.46720662239386e-07, 1.9978312337798343e-08, 8.333359460266365e-07, 1.530876814204163e-10, 2.2877044102642685e-05, 3.7453962704603327e-06, 1.455141136830207e-05, 4.57792930319556e-06, 9.99662120193534e-07, 2.9561881831341452e-08, 1.6252829482255038e-06, 2.4898870378819993e-06, 6.430745003171978e-08, 3.567063444620544e-08, 8.460088793071918e-06, 7.902027476802687e-09, 6.718619410817084e-11, 2.757555819243862e-07, 9.967740197680541e-07, 1.2336805355062097e-07, 1.583866287546698e-05, 5.570662438003637e-07, 4.533741230261512e-05, 2.0368541697735054e-07, 5.8858936426986475e-06, 3.7652085893569165e-09, 4.5416470584314084e-08, 9.768889867700636e-05, 0.00014489547174889594, 2.354308969998442e-09, 2.526302367300559e-08, 7.964982984276503e-08, 1.0174882464353274e-10, 5.868467667369259e-08, 1.4608101395197082e-08, 3.5670300047030423e-09, 1.9672462769904087e-08, 9.013208455144195e-07, 7.106537668732926e-05, 6.226392912367373e-08, 2.7747837805236486e-08, 3.101009937722665e-08, 5.451468950923299e-06, 1.5892897664571137e-08, 1.2194153669042862e-06, 5.3652279063953756e-08, 1.7719713696351391e-06, 1.9678694229696703e-07, 7.777948840548277e-10, 1.5916604922949773e-08, 3.1163552294266594e-10, 8.257788408627675e-08, 1.7093535689127748e-06, 1.1374482511428141e-07, 4.543289744418644e-09, 6.295820753621229e-08, 1.3316608828972676e-06, 1.930420921780751e-06, 3.7406416595331393e-06, 4.396897566039115e-07, 6.016603038005996e-07, 2.6263794694614262e-08, 0.0005774230812676251, 0.00012617011088877916, 6.258469056774629e-06, 5.342138251762663e-07, 4.312515500259906e-07, 4.383952922903944e-11, 1.740777832992535e-08, 2.125287892340566e-07, 1.0619903605402214e-06, 3.754659928745241e-06, 3.8582723682623055e-09, 8.086477464530617e-06, 1.1488675227155909e-05, 2.7244693683314836e-06, 3.456871411344764e-07, 2.979017881443724e-05, 1.1609272121404501e-07, 8.518689753600484e-09, 1.6143448533512128e-07, 3.793211931224505e-07, 7.621886470587924e-05, 6.502136074004738e-08, 5.942791858615237e-08, 5.222501044954697e-07, 6.8987505983386654e-06, 1.996211418386906e-09, 2.0387199128890643e-06, 7.5790893561134e-09, 9.957341262634145e-07, 2.1178118814191293e-08, 9.352702612375197e-09, 4.039595296490006e-06, 4.517341611887105e-09, 1.2696294788838713e-06, 1.167348045783001e-06, 1.7432849972465192e-06, 9.096255837448552e-08, 1.8935621071136666e-08, 1.5314698202928412e-06, 0.0008083663415163755, 1.3280207440402592e-06, 0.0005575964460149407, 1.443165365344612e-05, 1.5114596862986218e-05, 1.6387004507123493e-05, 5.730385237256996e-05, 1.5531813346569834e-07, 6.178047518012875e-10, 2.7045734896091744e-07, 1.3913957445765845e-05, 1.0748502177193586e-07, 4.685608132604102e-07, 0.0001970195007743314, 1.1266302863077726e-05, 1.9800189932084322e-07, 1.6827163662469502e-08, 6.247407213777478e-08, 3.512038304620546e-08, 1.0519425330812737e-08, 5.300545922182209e-07, 3.0242765092225454e-07, 2.434544832397023e-08, 9.248901200464843e-09, 4.2968656543962425e-07, 2.693248735141651e-09, 8.27627566479805e-09, 1.5740924119711508e-09, 2.08508436116972e-06, 2.294040086781024e-06, 1.6728664009590943e-09, 4.33215095740902e-11, 5.183464963920414e-07, 2.0591152047333416e-11, 3.051860708591647e-10, 2.6854235501971857e-10, 3.3497719414299354e-05, 6.688987468805863e-06, 4.540493758753428e-09, 5.152969606569968e-08, 1.4464932291957666e-06, 1.1679681666265651e-09, 2.6972607258812786e-08, 0.00014626960910391062, 5.396679512159608e-07, 2.335880999737583e-08, 0.003042592667043209, 6.015420240146341e-06, 1.347147193797582e-08, 3.1109510523208428e-09, 5.365127435652539e-05, 8.849560373391796e-08, 2.765605415788741e-07, 1.4600313988921698e-05, 9.390814348364529e-09, 8.746386015445751e-07, 1.7799358147385647e-06, 7.137959414649231e-07, 4.417929488909067e-08, 1.4020571370565449e-08, 5.029293426161985e-09, 5.94131824982469e-06, 0.0002972579968627542, 2.0079059481759032e-07, 2.910621299534455e-09, 7.867038220865652e-08, 3.260078074163175e-06, 3.6600110888684867e-06, 2.734802939663439e-10, 2.3009536675999698e-07, 2.9007867219377204e-09, 4.882320325805267e-08, 2.47913845896619e-07, 0.00019849487580358982, 1.248609464710171e-07, 1.12919806838363e-08, 6.503350050479639e-06, 5.719170985685196e-06, 2.394646614334306e-08, 4.2641337927307177e-07, 1.4822142624382195e-08, 5.344192310907658e-10, 3.997209674366786e-09, 6.363038096424134e-08, 1.1343624928272789e-09, 4.7576527606452146e-08, 4.129319997048242e-09, 1.299013234756785e-07, 8.74367106007412e-06, 1.5788241114478296e-07, 4.727499502976684e-10, 4.57001636533505e-09, 4.472272863154103e-08, 6.18997901824514e-08, 2.7096530175185762e-06, 6.8333396718855965e-09, 3.498406772606266e-11, 9.257791475647537e-07, 2.3632633201486897e-06, 4.368502004581387e-07, 2.418573587931183e-12, 1.4707887885379023e-07, 1.4667007164348433e-08, 8.579567634114937e-08, 2.3215636701934272e-06, 1.8008609913522378e-05, 8.445102395171489e-08, 1.8360495914748753e-06, 3.516699734973372e-06, 2.0836820269209966e-08, 6.634170296448971e-11, 1.5387238818220794e-05, 2.721422420748354e-09, 1.1627620324361487e-06, 3.1753461371408775e-07, 3.9401267713401467e-05, 1.2731267429177962e-10, 6.30239824772616e-08, 1.396272342191196e-08, 2.0428393199267703e-08, 2.9826134095856105e-07, 0.000676435767672956, 1.305045049804221e-08, 5.311765249871314e-08, 7.933519555081148e-06, 3.230538254683779e-07, 3.0064964562370733e-07, 1.8241576305033647e-10, 1.4823646097283927e-06, 1.1150125828862656e-05, 5.735786290372857e-10, 2.0968934677512152e-06, 6.399833440973168e-11, 1.91334038390778e-06, 1.5340603454205848e-08, 1.83036448220264e-08, 8.408660573389071e-11, 1.70044600622532e-08, 1.2034735846100375e-05, 2.1671713312798602e-08, 1.2838614615873212e-08, 1.0442744446947927e-09, 1.899799713100947e-07, 1.3093432471578126e-06, 5.392668089854169e-09, 1.5156764732182637e-11, 1.2408131055963167e-07, 0.00015212483413051814, 8.613038176008558e-07, 1.7135477481389216e-09, 1.0880739864660427e-05, 8.359886294329044e-08, 3.451531283005238e-09, 1.907993585348322e-08, 2.5301802753646685e-12, 4.2801293176353283e-08, 6.811931996253406e-08, 4.579456952535921e-11, 1.5376843132131057e-09, 1.6234749100618728e-09, 1.719303988068077e-08, 9.137438041761925e-07, 3.962536077040113e-09, 1.9746290713662518e-10, 9.589373348717345e-07, 4.700070732610584e-08, 7.087890480761416e-06, 1.5092907688085688e-06, 5.3935757193812606e-08, 7.888180334703065e-06, 3.228772271768321e-08, 1.380271612561046e-07, 4.848285772851568e-09, 4.1180101106874645e-06, 8.743011790102173e-07, 1.7224385828740196e-07, 3.3470598737039836e-08, 2.091909223622679e-08, 9.562286074071835e-09, 5.637904587629805e-10, 5.801582119602244e-06, 2.2681897462462075e-05, 3.2333460353584087e-07, 1.3705618584936019e-05, 4.3777415470458436e-08, 5.941162839473435e-11, 2.948818134118625e-10, 0.00011626145715126768, 8.09056643902295e-09, 4.0444000148909254e-08, 5.497232677953434e-07, 4.420220578449374e-10, 5.191799914427975e-07, 4.737454872838498e-09, 4.413387068780139e-05, 7.790696940901398e-07, 6.745553037035279e-06, 4.90914487727423e-07, 3.3492680850955026e-11, 5.374891642873081e-08, 6.531441609780586e-09, 3.0653217208964634e-07, 3.885865851316339e-09, 3.1942553846420196e-07, 7.588933037538936e-09, 7.47222827612859e-07, 7.88116100158831e-11, 2.5748187582053106e-08, 1.0930690450550173e-06, 4.818542720386176e-07, 2.8513733041357625e-10, 4.118822471355088e-05, 9.482325549470261e-05, 1.927052153405384e-06, 5.297232874568181e-09, 5.107055045527886e-08, 2.9274016100089284e-08, 5.0980681898238345e-09, 3.647497581749093e-11, 3.616309291951114e-12, 9.03149430087069e-06, 1.8245788169224397e-07, 1.2951015371243102e-08, 2.960008532681968e-05, 2.6354861404342955e-08, 4.61032634291314e-09, 7.0994769885146525e-06, 2.121651299091809e-09, 2.1477513655554503e-05, 3.1611403272790994e-08, 0.001438097795471549, 2.0670098965780426e-09, 3.6512889778350655e-07, 2.184579461772529e-10, 6.765705506950326e-07, 8.716304364497773e-06, 3.036815954260419e-08, 2.038731054199161e-06, 9.784445254013008e-09, 2.6631680611899355e-06, 1.3455526641337201e-05, 8.040490939720257e-08, 1.0973195442576245e-11, 3.729957853693122e-08, 1.6718446431696066e-06, 6.863316230010241e-05, 1.0520739124331158e-06, 3.150680827346264e-10, 2.439304536139275e-09, 2.738368038990302e-07, 0.00018076998821925372, 0.0002001765969907865, 1.597505239203656e-08, 3.0871262879372807e-07, 1.1608574368437985e-07, 3.85529697055631e-09, 4.6781185147892757e-08, 2.5801229597277597e-08, 1.0559517704678e-06, 0.00029623822774738073, 4.779711204605519e-08, 7.774093546686345e-08, 1.9700007669598563e-07, 6.740025426665852e-09, 3.051649244412147e-08, 1.5420816623645806e-07, 1.1901703089733928e-07, 1.215014311384266e-08, 8.767525514485897e-07, 6.385568849509582e-05, 3.211772861555495e-11, 6.976649480350261e-09, 3.991486664745025e-05, 4.071356099477619e-11, 0.00014999147970229387, 3.2124931603760842e-09, 1.594349754441282e-07, 8.950891416947826e-11, 7.872463356761727e-06, 7.000928121669858e-07, 2.5141872583844815e-07, 1.845814097123366e-07, 9.850954080237173e-11, 9.838567166298162e-06, 1.3846408364770468e-05, 7.18401634003385e-06, 8.561440836274414e-07, 6.833472434664145e-05, 1.6312304396137733e-08, 1.703451490975283e-09, 6.68091271194271e-09, 7.637014221018035e-08, 3.0223869629253386e-08, 8.035781320359092e-07, 7.133463805075735e-05, 2.3629203482755656e-08, 4.6518473517664916e-09, 1.6033272487447903e-08, 2.5920561697034827e-08, 1.4283547500326677e-07, 3.8148093040035747e-07, 2.104949770043163e-09, 1.1861396842505201e-07, 7.171010452111659e-08, 1.6207579278670892e-07, 2.2225357554361835e-08, 6.239189360712771e-07, 1.3742147530138027e-05, 1.609222408660571e-06, 3.709410520968959e-05, 2.088602968797204e-06, 7.733337952231523e-07, 1.053452933774679e-06, 3.7355343818212816e-10, 5.631811291095801e-06, 1.1888484152677847e-07, 3.758437401302217e-07, 2.4423703507636674e-05, 1.4378144896909362e-06, 6.738776505699207e-07, 1.5577312773018548e-09, 2.38337520386267e-07, 4.87226373024896e-07, 3.4501280765653064e-07, 8.910462412359266e-08, 8.155749964267045e-10, 1.3270295973200064e-09, 1.484264657847234e-07, 1.9660522099229638e-08, 4.2373254904859436e-10, 1.2474404229578795e-06, 2.436403008232446e-07, 2.220428086729953e-06, 1.1459808774816338e-05, 3.303576578961298e-10, 7.219867256935686e-05, 3.4842454965655634e-07, 0.00011338310287101194, 7.54499353661231e-07, 1.0466686184429363e-08, 7.792030373821035e-06, 1.8826561927198782e-06, 1.636106583546848e-09, 9.67631240200717e-06, 6.432804156020211e-08, 6.284389675315083e-10, 1.2752780775837635e-10, 1.350236367159141e-08, 2.817011761635513e-07, 6.248903332561895e-07, 7.610714965267107e-05, 3.719519838796259e-07, 1.873353767223307e-06, 7.891144377936143e-06, 9.560118030549347e-09, 2.9999064565799927e-08, 1.6443250094866357e-09, 3.0630029623779365e-09, 8.741435131298658e-09, 3.1419858714798465e-07, 8.096970871562803e-10, 6.734781436534831e-06, 1.604054773451935e-07, 2.0527353428168027e-10, 0.00017574286903254688, 2.9541380897057934e-08, 1.1538695332546922e-08, 8.726981945983425e-07, 7.553428105211424e-08, 8.570150384912267e-06, 5.6900404388215975e-09, 7.300790372255506e-08, 6.577811073604778e-10, 7.575670315418392e-05, 4.927111604047241e-06, 1.5495353611338913e-10, 5.957156758995552e-07, 2.531672294026066e-08, 1.2063651411153842e-07, 7.301018740690779e-06, 4.3994825205118104e-07, 8.911264504085636e-10, 3.279768634456559e-07, 3.572809825769241e-10, 2.5213162668791256e-09, 1.0520931027713232e-05, 3.1613313922207453e-07, 1.9063226091020624e-06, 8.572619947244675e-08, 2.0165603586974612e-07, 2.0461300209717592e-08, 2.276377664145457e-08, 9.326835765932628e-07, 1.3966690858069342e-05, 0.00016498277545906603, 1.7682146857822545e-09, 2.0897326535873617e-08, 0.004832662176340818, 4.588081026213331e-09, 3.9104534721445816e-07, 4.555613486445509e-06, 4.5588036300614476e-05, 1.7138590990839475e-08, 3.2758546986144665e-09, 2.86397261461957e-09, 5.617248888256654e-09, 3.6440053752251345e-10, 6.070863256546488e-11, 3.081720933550969e-05, 1.8423315850668587e-05, 5.764457661205924e-11, 6.615494263506605e-10, 6.895738841050658e-11, 8.707113607897554e-08, 2.805245591375183e-09, 1.502510713180527e-06, 4.179717683072681e-10, 1.6408471026352345e-08, 9.982248229789548e-06, 2.592988676042296e-05, 4.050623392259922e-09, 8.758777880757407e-08, 8.722330413490909e-09, 1.0455067034342846e-08, 1.4037367102037734e-10, 1.1094375906850473e-07, 3.6353156596113934e-10, 6.551762794515525e-07, 8.8754609217645e-10, 3.043994567519803e-08, 0.016845079138875008, 4.465187686264471e-08, 5.192698154132813e-06, 3.3187222925334936e-06, 9.0904528349256e-08, 1.5119891259018914e-06, 2.3012201921801534e-08, 1.3434311441073987e-08, 2.0652427338063717e-05, 2.598129489328471e-10, 4.276989873375214e-09, 4.3227259993727785e-07, 6.880588330204773e-07, 5.097051598568214e-07, 2.8372624001349322e-06, 0.0002280032349517569, 2.397200887571671e-06, 3.318779917549364e-08, 0.002621543360874057, 4.031325495645888e-09, 7.329601885430748e-07, 3.0423479707719636e-11, 8.012445323402062e-05, 6.28379037692639e-09, 3.3057986570383946e-08, 3.885263638592207e-11, 6.421454372684821e-07, 8.534458117992472e-08, 1.1597742322067006e-08, 3.1199831607153783e-10, 3.5307827772612654e-08, 0.0020170158240944147, 8.802063433677176e-08, 2.5636822442720586e-07, 3.782578417155236e-11, 2.253399022933422e-09, 2.2402204194804654e-05, 4.406650877530893e-10, 1.0724805071804511e-10, 2.8455360734369606e-06, 6.226995452607298e-08, 3.8144463587741484e-08, 3.339469185448252e-05, 1.0936417993434588e-06, 5.782428615930257e-07, 2.0235923159361846e-07, 3.044915956706973e-06, 1.0834107222468248e-10, 0.0006536075961776078, 3.228458567150483e-08, 6.815245967572992e-08, 1.613226174868032e-08, 2.218802919173868e-09, 6.049442049516074e-07, 1.267036395802279e-06, 2.0007175294267654e-07, 0.014325452037155628, 2.502497125078662e-07, 2.5675359616172955e-09, 2.434507450743695e-06, 2.188879122400067e-08, 9.592462447471917e-06, 9.622157449484803e-06, 8.551938321943453e-07, 3.118152744718827e-05, 1.079541107174009e-06, 1.2645673397315704e-09, 2.5601518682805136e-09, 6.281677542574471e-06, 7.609522612028741e-08, 9.077377853827784e-07, 7.16258364263922e-06, 9.434092618221257e-09, 0.0005142142181284726, 1.9117600558171688e-10, 5.998891197123157e-07, 1.1841392222322611e-07, 0.00014744796499144286, 4.921737328800191e-08, 1.3158024669834845e-09, 1.4202873543922578e-08, 4.6898767323000357e-05, 1.4978982676439045e-07, 1.5439859168964176e-07, 1.0369525682563108e-08, 0.0011537546524778008, 5.948777581643583e-10, 1.7727634915587487e-09, 2.7788246370619163e-07, 2.136615194103797e-06, 3.67926744626601e-10, 8.090345460232129e-08, 1.660612758414004e-10, 3.978565032980441e-09, 0.00012115282152080908, 2.555104856583057e-07, 1.7608015241421526e-06, 9.251317578673479e-07, 4.432910483131991e-09, 3.076910115584752e-10, 4.797813701884479e-08, 2.540687882301995e-09, 2.324818737520218e-08, 1.4677383660455234e-05, 2.5599747321969346e-10, 6.230246754057589e-07, 1.4762233035980898e-07, 6.560595267046665e-08, 1.1072963379543665e-10, 0.0009000836871564388, 1.2840824865634204e-06, 5.270713927529869e-07, 5.5835162129369564e-06, 2.58894292670675e-08, 1.546285943732073e-07, 5.070182305644266e-05, 1.619231304594848e-09, 9.02469673746964e-06, 1.1646134225884452e-06, 2.655807236351393e-07, 7.73330455228205e-10, 8.819300151285958e-11, 1.3766032225248637e-06, 3.6818537409999408e-06, 4.019883590444806e-07, 5.2741192746452725e-08, 1.5410767773005318e-08, 2.201350434916094e-05, 1.3323315215529874e-06, 1.1296966476948e-05, 1.4448108913711621e-06, 4.022887196697411e-07, 6.070056883800135e-07, 2.892777501983801e-06, 4.743301076359785e-07, 7.324085227189414e-10, 3.7192964263965678e-09, 3.3832344478040843e-12, 2.5045322260552894e-08, 4.735105196829181e-09, 1.9373349591234046e-09, 0.0005390789592638612, 8.870406418282073e-07, 1.730352039430727e-08, 0.0008449417655356228, 9.397846412184663e-08, 4.639361250724505e-08, 5.08463537940429e-09, 2.4620447902634623e-09, 0.0002927234163507819, 1.6753292584326118e-05, 2.457351300222399e-08, 6.628781079598411e-08, 6.412467890015705e-09, 5.8852576145795865e-09, 5.0822607590816915e-05, 1.2588977860161776e-08, 5.0238912081113085e-05, 3.914825867923355e-07, 3.5873131309926976e-06, 2.947145731013734e-06, 3.523358316215308e-07, 1.7831245713750832e-05, 6.477071767818643e-09, 2.0546663108689245e-06, 6.583986827957133e-11, 9.412712387302236e-09, 5.330180101736914e-07, 4.174255820998951e-07, 8.530139439244522e-08, 5.17101312880186e-08, 4.073586978847743e-07, 5.305966624291614e-06, 3.2425575682282215e-06, 5.267752953841409e-07, 3.12938794877482e-08, 1.2022962891933275e-06, 1.6436143823739258e-07, 1.2806083304894855e-07, 2.4134872919634454e-11, 7.479405894628144e-07, 7.62251720076712e-11, 9.923393662347735e-08, 3.5305249639350222e-06, 0.00027257917099632323, 6.880082281668365e-09, 1.1349453643560992e-07, 7.67840688808974e-08, 1.7483189367339946e-05, 2.6745422587737266e-07, 2.2462010917934094e-08, 3.872832508022839e-07, 3.865421760451682e-09, 5.556480608959191e-09, 3.022039563038703e-10, 1.7566085830367228e-07, 1.0208708545178524e-07, 4.505252837461171e-10, 3.584014640622968e-09, 1.5645820639065278e-08, 7.252958766912343e-08, 2.4355278682719472e-08, 3.9635950088268146e-05, 7.651872692804318e-06, 4.786970180248318e-07, 0.004419838078320026, 4.530929436441511e-05, 1.3573105206887703e-05, 4.911645135052822e-08, 2.199044502049219e-05, 2.3009125271755693e-08, 2.5313429432571866e-06, 2.7614122544150632e-08, 1.4626186384703033e-05, 1.2110550340338477e-09, 2.5206920781783992e-06, 2.5510746581858257e-06, 7.272145552406073e-08, 1.287291553353498e-07, 5.892880835745018e-06, 1.1417062495411301e-07, 6.058611688786186e-06, 6.960253813304007e-07, 5.793232503492618e-06, 0.0004305976035539061, 5.687465545634041e-06, 0.0223061665892601, 1.769575241894472e-08, 1.3496763706655202e-08, 1.5699441746619414e-08, 2.847175828435411e-08, 3.551765104248261e-08, 7.003694690865814e-07, 1.1170807390215032e-08, 1.6402601943354966e-07, 1.1814869793624894e-07, 2.790996177282068e-06, 4.173318982059193e-11, 9.745093620949774e-07, 1.8685308532440104e-05, 7.4953304647351615e-06, 6.752068770765618e-07, 0.00016100741049740463, 0.00017873963224701583, 6.169157131807879e-07, 8.29356707754414e-08, 1.223569938701985e-07, 7.1735868113764756e-12, 5.632259103549586e-09, 1.2444119192878134e-06, 4.7025210392348527e-08, 2.0276181089684542e-07, 5.741719724028371e-05, 1.993508340092376e-05, 6.797521336920909e-07, 1.369671372231096e-05, 3.706027040806248e-09, 8.651138159621041e-06, 6.222314663517636e-09, 2.2809794941736072e-08, 3.7337835578910017e-07, 4.81850861433486e-07, 8.018666286524478e-10, 7.566401949432588e-10, 0.00019342412997502834, 1.9775167459101795e-07, 5.393254127739056e-07, 3.1978483749917075e-10, 3.055096176041161e-10, 2.923234571072797e-11, 4.004261811019205e-08, 1.055433585861465e-05, 5.920345756749157e-06, 5.8683010450977235e-08, 0.00013824994675815105, 1.1688858592151519e-07, 1.1334302740806379e-07, 1.755462442076805e-08, 3.704310984176118e-06, 1.3818226562989366e-08, 3.191726989371091e-07, 2.907626594605972e-06, 3.763861968764104e-06, 8.390551364811927e-09, 3.1193845728694214e-08, 4.129529773688745e-10, 1.9026516895337409e-07, 3.461783359970383e-10, 1.9645484883312747e-07, 5.093159921587542e-11, 5.9514809436223e-07, 5.6970868023142884e-09, 4.385377838644899e-09, 9.22014908866231e-09, 1.0137868322601662e-08, 2.36065051950618e-08, 7.971872946654912e-06, 1.0009551409950745e-08, 2.4480696083628573e-05, 0.00030717666959390044, 2.1544634876136115e-07, 3.513730817417127e-08, 1.1688590717540137e-07, 2.2923886400239013e-10, 8.686585672990077e-09, 2.670558933459688e-05, 8.412816043801286e-08, 8.979422205790399e-10, 0.0001646028395043686, 1.740576294650964e-06, 4.644246098806093e-10, 1.7588488844921812e-05, 9.347178320240346e-08, 1.999802066166012e-07, 1.4657064184575574e-07, 4.824883490073262e-06, 6.143724249341176e-07, 1.2607429544786442e-09, 6.017955911374884e-07, 3.898271483393501e-09, 3.564038024705951e-06, 1.7450610130254063e-06, 8.054234399423876e-07, 7.215077868494291e-09, 0.00035718243452720344, 4.678814669034637e-09, 1.1984300218870203e-09, 7.640735930181108e-06, 2.621122696666589e-08, 2.336991656193277e-06, 5.645611755866753e-10, 2.877973201975692e-07, 1.271162659577385e-07, 1.078250897990074e-05, 7.64896554983352e-08, 3.330296067360905e-06, 4.6721938815608155e-06, 6.787477957459487e-08, 5.357349550649815e-07, 5.525679025453201e-09, 2.0346018203554195e-08, 1.7149467623767123e-08, 2.5416699855895786e-08, 9.061747796579311e-09, 7.158188236644492e-05, 1.5680448086641263e-06, 2.911055389631656e-06, 1.5382111939743481e-07, 0.00010264402953907847, 2.351702823943924e-05, 3.437193196731414e-09, 1.7127482351497747e-05, 3.986156968949217e-07, 1.91215281120094e-06, 3.2860052670002915e-06, 2.3477286958950572e-05, 6.665061835775532e-09, 0.0002455016947351396, 3.775860932364594e-06, 4.283026100893039e-06, 6.39828527937425e-08, 9.681181154519436e-07, 3.048014951545497e-09, 2.0535701423796127e-06, 9.154608494876815e-12, 1.2820524375456444e-07, 7.536345947301015e-08, 1.571555543478098e-07, 1.0762828406285507e-09, 6.490134296655015e-07, 3.564174448911217e-07, 2.8814664432275094e-08, 2.6342518140154425e-06, 2.697982381505426e-06, 1.981135788753363e-08, 5.374509015609874e-08, 2.6441080080985557e-07, 5.539977792068385e-07, 2.5855104468064383e-05, 2.4933813236316382e-08, 1.3800867009194917e-06, 1.6485450032632798e-05, 1.6676419534178422e-07, 5.926756863061655e-09, 0.00020473601762205362, 1.564985581126166e-07, 9.652286081518469e-09, 3.170331996926734e-09, 1.0416536078139416e-08, 1.2764596046821453e-09, 8.129148909574724e-07, 1.5786744356205418e-08, 2.2199477811568613e-09, 1.725503118166216e-08, 1.311269244297364e-07, 3.6906861566876614e-08, 1.1953213974180699e-09, 2.491737705057062e-09, 2.2590944226408283e-08, 5.365659916378718e-09, 1.0481230816594689e-07, 0.00019129112479276955, 1.3853797042884253e-07, 8.57811599530578e-09, 7.154777881268615e-10, 5.567541538908927e-09, 3.3940770194362813e-09, 1.0013969653499544e-09, 5.288688953442033e-07, 1.5331842462273926e-08, 2.9551441294017877e-09, 3.690873739969902e-08, 7.718400593148544e-05, 2.0116461030283972e-07, 3.937852710578227e-08, 9.887166640965006e-10, 9.150978641514484e-09, 4.209640849239804e-07, 2.2924393761059036e-06, 1.1404054021113552e-05, 3.7373560189735144e-05, 1.875117959571071e-05, 1.4601154729732002e-09, 2.457458414539815e-08, 1.2076357691626072e-08, 1.0631532987304126e-08, 8.981260180007666e-06, 6.915080952296648e-09, 3.4817792311514495e-06, 4.1507675518914766e-08, 2.6435669209234902e-09, 1.1693420987057834e-07, 0.00013481942005455494, 1.690963233151599e-12, 7.360233098552271e-07, 4.80313174193725e-05, 1.7581190947524306e-09, 0.057348113507032394, 1.3216128536441829e-05, 6.616454584218445e-08, 6.962686143197061e-07, 8.875704793354089e-08, 1.700235863211219e-08, 4.647160949389217e-06, 8.160032545501128e-12, 3.136996795660707e-08, 9.080833933694521e-07, 3.0723081856365297e-09, 4.790921082076238e-08, 1.3484915371009265e-06, 1.0607800504658371e-06, 4.7287110646720976e-05, 3.8913235300697124e-08, 7.93305687807333e-08, 3.8179841794772074e-05, 1.7213022118767185e-09, 1.4943452697480097e-08, 3.0829609698468374e-11, 2.2463641613512664e-08, 3.957179796998389e-05, 1.6858314211276593e-06, 3.3188698012054374e-07, 3.1707308778550214e-08, 0.0001557976647745818, 1.327363747805066e-07, 1.794888589756738e-06, 1.1043946201994004e-08, 1.2337903854131582e-06, 1.2973434593277489e-07, 2.068563276225177e-08, 9.00335635378724e-06, 1.7945814972941676e-11, 2.519149893487338e-05, 1.2914183855627925e-07, 2.8742459107888862e-05, 1.178383990918519e-05, 1.3214458704169374e-05, 3.853454302316095e-07, 1.800026439013891e-07, 0.0006564880604855716, 9.658409183543881e-09, 3.8419184500426695e-10, 8.468329371069672e-10, 6.956660945434123e-05, 6.749284686691226e-09, 1.118601088492821e-12, 2.7060044871518585e-08, 5.897346500027822e-10, 3.4311451599933207e-06, 2.6198705427304958e-08, 3.790857192598196e-07, 5.412212544797512e-07, 2.8487211238825694e-05, 6.138326625659829e-06, 9.686284130339118e-08, 6.472232194631999e-10, 8.76168030572444e-07, 6.828673043690614e-11, 4.500591543887822e-08, 2.071788962609844e-09, 5.015907618144411e-07, 1.0064672650855755e-08, 1.0087247574119829e-05, 0.00010976861085509881, 1.997300387301948e-06, 5.967195626510602e-10, 0.00029697324498556554, 1.5528485164395534e-07, 8.821147048365674e-07, 1.435207224176338e-07, 2.9496277420548722e-06, 6.075060809962451e-07, 8.229691275118967e-07, 3.6754030929841974e-07, 1.5449502370756818e-06, 8.707091936344113e-09, 1.0491957996849877e-10, 2.09154295589542e-05, 1.1320785375801279e-07, 7.466740115091852e-09, 1.3604792847843328e-08, 9.963737284124363e-07, 1.955625208438505e-07, 3.09310586033007e-08, 1.1384172715622753e-08, 6.161643000268668e-07, 4.433458045127736e-08, 4.510714290972828e-08, 1.1536857025262748e-09, 2.0709671844088007e-06, 4.096509292139672e-05, 7.538739055235055e-08, 6.076631642315533e-09, 9.759927976915606e-09, 6.473609346357989e-07, 3.994363396486733e-06, 2.364790308018172e-10, 9.171632342486191e-09, 9.708379167250314e-08, 2.0839687753237968e-09, 1.9495949743486563e-08, 6.811770703052389e-08, 7.625099556207715e-07, 1.8664671586066106e-07, 3.0851750043581205e-09, 5.022210203264876e-08, 1.412606422945828e-07, 1.5088001958929453e-09, 3.641130206233356e-06, 6.238877747755112e-10, 2.9481211640813854e-06, 9.674469083620352e-07, 1.3849428270873432e-08, 2.145952926468908e-08, 2.4536499054761407e-08, 2.30545396107118e-07, 1.452111428079661e-05, 1.274531427952752e-06, 1.4941515757982415e-07, 1.2699187479370266e-08, 1.6975680239283975e-07, 7.959117392930537e-11, 1.1528441064001527e-06, 7.011867069195432e-07, 1.3681670019138892e-08, 6.546664735651575e-06, 2.851422431504602e-10, 4.699000783148222e-05, 2.8335877644281027e-08, 1.303143370989801e-09, 7.299457593035186e-06, 5.290037918825874e-08, 9.290060276612166e-09, 2.4193314018816636e-11, 1.6192466811837392e-10, 2.705364749999717e-06, 1.9738206447073026e-06, 1.5709368881289265e-06, 8.268745688155832e-09, 1.130751243749728e-07, 2.0885781992774355e-08, 1.5319648327682955e-10, 2.12896411611041e-09, 3.4101148571608064e-07, 1.3776542573396e-06, 8.423815415881108e-06, 2.6266806507635465e-09, 3.5681824783750926e-07, 1.0897397260123398e-05, 1.8098705867330978e-09, 4.5950844906883503e-08, 1.2771198271366302e-05, 4.5923215230558867e-10, 1.878177499747835e-05, 6.906483163149346e-10, 2.5922757274088326e-08, 3.9888242042707134e-08, 7.93520598563191e-07, 9.915777354763122e-07, 1.6113883560819886e-08, 1.0583025300547888e-07, 2.7282291981123308e-08, 4.083115072717192e-06, 2.9442714932770286e-10, 1.5331928807427175e-05, 3.454851139395032e-06, 3.807108228670586e-08, 1.1868628302380557e-10, 1.4414354258807105e-11, 6.799178663641214e-05, 6.037760158505989e-08, 4.250507945613435e-09, 3.825924910216827e-09, 1.195575638490709e-08, 6.034033503965475e-07, 1.762071860866854e-06, 1.5967139832540056e-09, 2.3535348248060473e-07, 6.482662229245761e-08, 2.667061835381901e-06, 7.430577966260898e-07, 4.44213537775795e-06, 3.9629616367165e-05, 3.987220970058658e-11, 5.254386081787743e-08, 5.742377684470057e-09, 4.360247839940712e-05, 5.469597752494337e-08, 5.1919524679533424e-08, 1.4352217192481476e-08, 6.236197669373666e-10, 3.502530944388127e-06, 2.805475332934293e-06, 5.922021841797687e-07, 0.0007679779664613307, 8.736755034988164e-07, 1.0564237982180202e-06, 2.5755674926131178e-08, 2.113047292695569e-09, 1.6257912349715298e-08, 1.881682436533083e-07, 6.386935638147406e-06, 1.1760302953689461e-07, 3.9031902154818e-09, 1.6413961930084042e-05, 5.079436427024575e-09, 1.2052391866745893e-06, 4.254799623737426e-09, 4.754041782462082e-08, 0.00177368288859725, 8.57906101714434e-09, 4.216021753933319e-09, 0.0006038261926732957, 2.2900801255332226e-09, 7.975917704072799e-10, 1.324907861999236e-05, 1.9948720364482142e-07, 3.946302513213595e-06, 1.2087531331417267e-07, 6.796284196752822e-06, 2.4189059644186273e-09, 8.647162758279592e-05, 0.0011781902285292745, 3.237917781007127e-06, 6.636237230850384e-06, 1.0716728199300363e-10, 2.2581330995308235e-05, 5.113957035618455e-10, 1.3895531765228952e-06, 3.526118850030735e-12, 2.470873710080923e-07, 1.3140287746793433e-09, 1.0254944671217459e-09, 5.174620909542682e-08, 3.7061661259940593e-06, 2.1071109301828983e-09, 4.7118955848191035e-08, 3.881396878568921e-06, 8.406034979202559e-09, 3.719493335552215e-08, 3.1008480618766043e-06, 2.7444142958188422e-08, 3.7698802413110855e-11, 2.524771844036877e-05, 4.519107278611045e-06, 4.820117851522809e-07, 2.1200615663019562e-07, 2.5676951675990267e-09, 1.2730397429550067e-06, 1.2253914974280633e-05, 1.6659930679452373e-07, 5.001899783252384e-09, 3.926198602344222e-11, 8.042914487305097e-06, 1.457222680301129e-07, 5.424211080296004e-10, 5.0645965643525415e-08, 1.0434813013660005e-08, 2.497605322560048e-08, 2.0581174453582207e-07, 2.5317251584056066e-07, 1.3533745235605465e-09, 1.2120567305373697e-07, 0.0005168272764421999, 1.8208525887075666e-07, 6.674977612419752e-06, 2.165116308461279e-09, 1.2343457456154283e-07, 3.8624666132136554e-08, 2.0269769478338528e-13, 4.383927887374739e-09, 1.9652438822959084e-06, 2.179735858476306e-08, 1.5723610191642479e-09, 6.330453288683202e-06, 1.4856058783152548e-07, 1.605763827683404e-06, 4.754116034177969e-08, 4.097020791959949e-06, 1.6661594770539523e-08, 1.6302383443189683e-08, 4.494685526879039e-06, 8.228732895076973e-08, 1.6263051065834588e-06, 1.1479658112989455e-08, 4.1238212844518785e-09, 7.090454346325714e-06, 8.87530671178638e-09, 3.495405354048131e-10, 1.2560103131165068e-11, 0.0006178331095725298, 4.2391970489497055e-10, 3.750956878434408e-08, 6.479575063167431e-07, 7.946380220502647e-10, 9.063897721262038e-08, 2.2208088921615854e-05, 7.639439445483731e-07, 1.8095919585903175e-05, 0.0001131652170442976, 2.4083867629087763e-06, 8.33533817967691e-07, 1.2276681893297337e-09, 2.7849338390950606e-09, 2.9640997922797396e-07, 4.7083389631552564e-08, 2.8537117113813792e-09, 2.919007329182932e-07, 9.768151443267925e-08, 1.0393975680145218e-09, 9.197842487651542e-09, 3.4298332138860133e-06, 3.0307910492410883e-07, 1.0876762246425642e-07, 1.208678668263019e-07, 2.77444751617395e-08, 4.723832436326347e-09, 1.2180619535229198e-07, 1.0522257980483118e-06, 1.0355470436707037e-07, 3.877891785464271e-09, 2.0917743981385684e-08, 3.1868729877260193e-10, 9.023225175042171e-06, 2.523776493035257e-05, 7.529350654067457e-08, 2.9045269300809196e-08, 6.259289619947594e-08, 3.397530867754739e-10, 2.7715325146004943e-08, 1.180810727419157e-06, 4.41254123870749e-05, 2.1328997945602168e-07, 0.001567314611747861, 3.611585555063357e-07, 1.4868494879749505e-08, 2.1925754267737574e-10, 1.4041948759313527e-07, 0.00010364464833401144, 9.761834007804282e-08, 1.7401412533146754e-09, 5.1778128451474e-09, 2.64364075519552e-07, 1.0284826323925245e-08, 1.9512417566147633e-06, 1.1153911145811435e-06, 3.130333254830475e-07, 1.2231581081323384e-07, 4.748321558167845e-08, 5.853489710716531e-07, 6.971554103074595e-05, 3.257113911914189e-09, 1.1743518379958573e-09, 1.6371912181512016e-07, 1.620927491785551e-06, 1.5737869896170764e-09, 4.934695994052163e-07, 2.0506107034634624e-07, 2.4114825691867736e-08, 5.720823370580774e-09, 7.011829694647531e-08, 0.00010313823440810665, 1.2108935720789304e-07, 2.286880018687043e-08, 1.4935759296008655e-08, 1.4297301277110819e-05, 1.088215685740579e-05, 1.9581044341521192e-08, 0.0002112941729137674, 1.0182119467572193e-06, 7.540152751062124e-07, 2.301588210684713e-06, 3.7862748285988346e-05, 5.504909381670586e-07, 5.731591272706282e-07, 4.359010041810052e-09, 3.1867551797404303e-07, 8.381979138549767e-11, 9.007015933093498e-08, 7.543299318513164e-11, 6.181136935623499e-09, 3.8138904301376897e-07, 1.717616271434963e-08, 2.8932839768458507e-07, 1.4121651403797841e-08, 2.488548034307314e-06, 5.670006242297632e-09, 8.27428292549115e-10, 5.436301580630243e-05, 1.4231358363758773e-05, 8.580065546937021e-09, 1.0311403997320667e-07, 8.870142842454243e-09, 1.3006024346395861e-06, 3.180235125910258e-06, 2.702788606256945e-06, 1.5518353393417783e-05, 9.121709609871687e-08, 2.592744522189605e-06, 4.3610931754756166e-08, 6.448093448074843e-11, 4.104510367142211e-07, 7.691409109611413e-08, 9.706038056611632e-12, 8.803823448033654e-07, 2.3696176754128828e-07, 0.00043445618939585984, 3.1910225573028583e-08, 1.8443479348206893e-06, 3.216866417687925e-08, 0.0004355509881861508, 1.13093068243586e-09, 1.8657866363014364e-09, 8.653894474264234e-05, 3.465608355845973e-11, 5.306996513354534e-07, 4.7477652742600185e-07, 5.978473358730918e-12, 1.2112548297693593e-08, 1.1159100949953427e-06, 5.516781698133855e-09, 4.499104644395402e-09, 3.575517666831729e-06, 2.9343740607146174e-05, 7.910644939101985e-08, 7.779406285823853e-11, 1.3930798736794259e-09, 3.101443546427163e-09, 5.160508180779289e-07, 0.00012367991439532489, 2.8537585627930184e-09, 7.948779966682196e-06, 0.0002010581229114905, 2.5122329105897734e-08, 1.4222610936198699e-11, 1.4469606867351104e-05, 1.7097376314723078e-07, 0.00028055536677129567, 3.209705141671293e-07, 2.9865469741707784e-07, 1.6548505300306715e-05, 0.00012500131560955197, 6.692653187201358e-07, 2.1937015048933972e-07, 2.03355966732488e-06, 3.2478855160889e-08, 2.298046995752312e-11, 1.2625165481949807e-06, 1.1603885852196072e-08, 5.83000803189293e-09, 1.7868635495688068e-06, 7.583574301861518e-08, 4.031178946206637e-08, 1.0680104196580942e-06, 1.3414226884833624e-07, 9.165694869750496e-10, 1.9919418718927773e-06, 2.086021140712546e-06, 4.963157884674274e-09, 2.8794642275897786e-05, 8.863504596945404e-09, 2.6341719916445072e-08, 8.172576781362295e-05, 3.104686285837488e-09, 2.216341110639064e-09, 1.6584553463871998e-07, 3.1025215321278665e-06, 8.984630261998916e-10, 1.1947793154831743e-07, 6.148373898895443e-08, 1.0412053370600916e-06, 1.0319055832042068e-07, 8.089440370895318e-07, 2.1001524075359157e-08, 4.516342357874237e-07, 6.018163389853726e-07, 3.2778910963315866e-07, 2.440077366827609e-07, 8.077937962980286e-08, 7.210736612250912e-07, 1.2040069918839436e-07, 1.1110504183342584e-10, 1.0747909229280594e-09, 9.681800293037668e-06, 2.4669546405675646e-09, 9.54168058342475e-07, 1.6604838393163845e-08, 9.16932574313023e-09, 3.1563567972625606e-06, 7.983513569342904e-06, 0.00012031247752020136, 7.391968814829397e-09, 1.2109463605469717e-11, 2.0393596855683427e-07, 8.915450848689943e-07, 8.51314723604446e-07, 0.0001271476794499904, 3.913780346920248e-06, 3.1508749088970944e-05, 4.6042779899835296e-07, 1.6230780602199957e-05, 4.323365828895476e-06, 1.4375072510119935e-07, 2.0793477517599968e-07, 4.009081955302918e-09, 1.5317246493395942e-07, 1.2386884009174537e-05, 6.735391480106045e-07, 2.3935167519084644e-06, 3.473375143059343e-09, 7.869931373249983e-09, 2.328307768806326e-09, 2.908244027821638e-07, 3.4479640476092754e-08, 1.6948115444392897e-07, 3.3268690913246246e-08, 4.473126580251119e-08, 1.8480012613508734e-06, 1.3093028883304214e-08, 5.491072485597215e-09, 6.017868928870351e-12, 1.2294063708395697e-05, 6.546425765918684e-07, 7.716263894508302e-08, 3.85997047658293e-08, 3.4456297726137564e-06, 1.7411050379223525e-08, 5.075823423794645e-07, 3.0220058988561505e-07, 5.610363132291241e-06, 1.9772097292758417e-09, 7.264502777104553e-09, 1.3532623199807858e-07, 2.742716105785803e-06, 3.620005628590661e-08, 7.32903345124214e-07, 1.3258252806735982e-07, 1.6984698092414874e-09, 7.301698019546166e-07, 1.5443466736542177e-06, 3.847457264782861e-05, 0.00040786652243696153, 2.7225848242551365e-09, 5.887794395675883e-05, 3.1777472320193567e-10, 3.432925268498366e-07, 2.1987620613117542e-09, 6.513760086868103e-10, 4.2171218410658184e-06, 1.3255827070679516e-05, 5.72945557664184e-09, 1.2074698574338072e-08, 3.1510001008427935e-07, 7.835521675936263e-10, 6.014921041241905e-07, 1.713389607971294e-08, 4.7796593349858085e-09, 1.0574751740932697e-06, 5.724919649452431e-09, 2.3894621836717533e-09, 2.3522821024357654e-08, 1.792862576621701e-06, 1.4209905252471344e-08, 3.5870681358574075e-07, 5.218918886384927e-05, 0.0012071094242855906, 2.351981187587171e-08, 1.6575071413171827e-06, 4.314175683362009e-08, 4.766499159813975e-07, 1.517345594947983e-06, 2.390081590419868e-06, 3.1872201589067117e-07, 1.163000433734851e-06, 1.9039098333450966e-06, 6.416822628807495e-08, 1.7668821267191603e-11, 5.8207605402316176e-11, 7.421639480753583e-08, 5.7340919568105164e-08, 5.395079824666027e-06, 1.640641045241864e-07, 9.145020385403768e-08, 7.199971878435463e-05, 5.6963049388514264e-08, 1.1764951235448962e-09, 0.00012799419346265495, 1.1902972119059996e-06, 8.830623698941054e-08, 1.1263679225437251e-11, 3.464933584496066e-08, 1.507142144419049e-08, 2.735155157918001e-10, 1.2632257906375344e-08, 3.721694952218968e-09, 1.017507234024606e-08, 2.2701364343902242e-08, 1.524893961102336e-10, 2.3102557644705257e-09, 2.7780189260084853e-09, 9.783733823098828e-09, 6.207378611122749e-09, 1.1863242832532706e-07, 6.219949000296765e-08, 3.6532838976199855e-09, 2.4771402706846857e-08, 2.5887636923016544e-08, 2.036474633371199e-08, 2.7749756270623038e-08, 3.2129241844813805e-07, 2.160643589377287e-06, 6.513800343554976e-08, 5.06493913476902e-09, 3.171886575614735e-08, 2.3674738258705474e-06, 3.37433156971656e-08, 2.1640191860683444e-09, 3.2129531746249995e-07, 6.503478200414747e-09, 5.07495023782667e-09, 1.701096437045635e-07, 1.367323818612931e-07, 1.523222437072036e-08, 5.439493179437704e-06, 1.0442942084409879e-06, 3.061434472328983e-05, 2.9977761073496367e-07, 2.7236856325885128e-08, 9.793945537239779e-06, 0.0011751827551051974, 5.603068871096184e-07, 4.8229995996962316e-08, 4.733738308004831e-07, 4.168919431890572e-08, 6.907652050358593e-08, 1.3248906860496845e-08, 0.00037776801036670804, 9.786596422145522e-08, 4.288100541316453e-08, 3.4839919837992284e-09, 3.2181404208131426e-07, 6.664729426120175e-07, 5.17148379675092e-10, 5.173188810658758e-07, 1.3113385648466647e-06, 1.2873097432475333e-07, 7.821219696779735e-06, 3.9609592931810766e-05, 2.370066596313336e-07, 7.590474915275536e-09, 4.3088562051707413e-07, 3.138188020557209e-08, 2.381471131407409e-10, 6.45649667063708e-09, 3.3598439586057793e-08, 8.009475749304329e-09, 7.096952003848855e-08, 5.137075049788109e-07, 5.027962401982222e-07, 1.2155581430306484e-08, 3.8176764064701274e-05, 1.7805090735123486e-09, 3.69769139751952e-07, 1.2543101490791742e-07, 3.228316199965775e-05, 6.910514457558747e-06, 9.371002107627646e-08, 3.1876425055088475e-06, 5.056050578211568e-10, 4.0222818142865435e-07, 3.9238052806922497e-08, 5.608690734071331e-10, 7.443156391673256e-06, 5.145921022631228e-09, 4.9991012218697506e-08, 5.132798008844475e-08, 1.6382967871209075e-08, 5.8732920749093864e-09, 2.3471585564038833e-07, 4.446874513064358e-08, 7.432720394717762e-06, 1.2684868433421315e-10, 6.316659195704233e-09, 2.9222709585496887e-08, 8.384875371802991e-08, 2.594421788515433e-09, 3.1155455992859515e-10, 3.15250399580691e-05, 2.0542847778415307e-07, 1.8322056094532968e-09, 7.147333747070661e-08, 1.4038465945276357e-09, 1.147486088370897e-07, 1.7203611832883325e-06, 1.1106258170912042e-06, 1.1905988417382218e-09, 2.581374669574643e-09, 8.212058310164139e-05, 4.1060488001676276e-05, 2.0755182958964724e-06, 5.956860604783287e-07, 2.0489685994107276e-05, 9.159881528830738e-07, 1.9572468090700568e-07, 3.9707224459561985e-06, 2.0161827762876783e-08, 2.8423071398719912e-06, 4.762534615565528e-07, 4.859696645098666e-09, 1.4724315406056121e-05, 7.743126972936665e-11, 3.760438627153917e-10, 1.5073562398271179e-09, 1.1778142834373284e-05, 4.460850178134024e-08, 1.2005218195554335e-05, 2.471665716541338e-08, 1.0017777640314307e-05, 1.0199262545640408e-09, 2.9688371760983046e-09, 1.4464241360201413e-07, 3.821382506430382e-06, 2.2418411571578645e-10, 1.1781240800701198e-06, 5.905006332795892e-07, 5.470507744576025e-07, 2.0194146088670095e-07, 2.381279728069785e-06, 1.0919685337285046e-05, 2.822076794473105e-07, 2.4530534574296325e-05, 3.805290749170354e-09, 4.7314081541571795e-08, 1.082979338207224e-06, 5.143946509633679e-06, 7.253000262608111e-07, 4.150966788074584e-07, 7.659383300051559e-06, 1.3238512508451095e-07, 2.2810428390585002e-07, 1.3966426877232152e-06, 6.8818870602171955e-09, 0.001986159710213542, 8.539392304385274e-09, 5.5520192887570374e-08, 3.22551123588255e-08, 2.7812341159005882e-06, 9.076306923816446e-06, 4.309511325573112e-08, 2.711591662318824e-07, 2.787272933346685e-06, 8.80972805816782e-08, 6.990933343331562e-07, 7.659477887500543e-06, 4.772908113181984e-08, 4.863881599703745e-07, 1.8608249163776236e-08, 1.0184982812688759e-07, 3.6109000234318955e-09, 0.0038459186907857656, 3.2187863041599485e-08, 5.098461315355962e-06, 6.418099474103656e-07, 6.448432941397186e-06, 3.69502436114999e-06, 1.8733450346530844e-08, 0.0018403728026896715, 3.357650257385103e-06, 4.905772676977449e-09, 1.1330886749705282e-11, 1.1466828375716887e-08, 6.1471841945603956e-06, 1.4335106243379414e-05, 3.088885369706418e-09, 1.2410399108375714e-07, 4.482428650476322e-09, 8.99739534361288e-05, 6.069266873964807e-06, 2.935960641270441e-10, 1.3709018809038298e-08, 1.622029088821364e-07, 1.5939731667913293e-07, 0.00022245917352847755, 1.0818157534231432e-05, 4.0529235434405564e-07, 1.1013991496611197e-08, 9.5530558610335e-06, 1.3691806088900194e-06, 1.4514139934362902e-07, 1.546155363030266e-05, 8.438729537374456e-08, 7.460488476995764e-11, 4.789881165312426e-12, 9.479772984377632e-07, 1.0051945942279872e-08, 8.896452641238284e-07, 0.0006437922711484134, 5.317230034052045e-07, 8.901735782274045e-06, 2.5135449277513544e-08, 3.4016167660411156e-08, 2.725186298846438e-09, 1.2852328836743254e-06, 3.075578933930956e-05, 4.618563309577439e-08, 3.650826101875282e-06, 8.119101948977914e-06, 5.003494507604955e-09, 4.030824129586108e-05, 7.668035650887361e-10, 3.954511612391798e-06, 7.402279234014486e-09, 2.028249568297724e-09, 8.129735218354028e-10, 1.735930936774821e-06, 4.57287789856764e-08, 3.9810927887629077e-07, 2.8425038181012496e-05, 4.932452156936051e-06, 5.541657083085738e-05, 2.7520934509084327e-06, 1.33549292513635e-07, 5.4515658653997434e-09, 3.587552157569007e-07, 1.0557009488820768e-07, 1.1571827371881227e-06, 5.050599156675162e-07, 2.4817441612867697e-07, 3.845850733341649e-05, 1.2012503702862887e-07, 3.7135785646569275e-07, 6.785301565059854e-08, 6.856110303488094e-06, 3.7439070865730173e-07, 9.392722016832167e-11, 9.997319239118951e-08, 1.3692900211026426e-05, 7.392254169902301e-11, 5.839035566168604e-07, 3.2674206522642635e-06, 0.00014717366138938814, 1.1220612633477955e-10, 5.2316735832391714e-08, 1.269669667181006e-07, 7.596843352075666e-05, 5.9377647687597346e-08, 3.786932545324362e-09, 1.813247763493564e-06, 3.7687183152002035e-08, 8.793672634510585e-08, 0.00014715948782395571, 1.05070250810968e-06, 9.760224929777905e-05, 2.4675539833651783e-08, 2.251882506243419e-05, 0.0006029083742760122, 0.006532994098961353, 4.036419909425604e-08, 4.769881911670382e-07, 1.289907345380925e-07, 1.2848035169099603e-07, 2.3480858146740502e-08, 5.352185805662657e-09, 2.739657922745664e-08, 4.158972899404034e-07, 3.8952813952164433e-07, 3.485788013790625e-08, 9.328125116780939e-08, 1.5931996699691808e-07, 2.414367600067635e-06, 1.0093546052303282e-06, 1.0015543061570042e-08, 2.1623522972191722e-09, 2.613293418107787e-06, 1.3257520414811097e-08, 5.31185378349619e-07, 2.191631892856094e-06, 1.8669854995323476e-08, 3.3065904858631257e-07, 5.648759793253078e-10, 4.076261550522986e-09, 2.2608931260492682e-07, 7.301538262893814e-10, 4.665871983888792e-06, 3.049489905038172e-08, 2.4016328481479832e-08, 2.4333351120731095e-06, 2.0392577937400347e-07, 1.1494104557030482e-09, 2.621419081805243e-09, 2.2512070074753865e-07, 1.9234007595514413e-06, 2.8026312293150113e-07, 3.579742440251721e-07, 8.565088194245618e-09, 5.315298867714091e-09, 1.7883888858705177e-06, 2.8843423649504984e-08, 7.890847697478165e-11, 1.7733032819933214e-08, 1.3223641737170055e-08, 3.8386863132622295e-10, 2.1386514958976477e-07, 6.08114358868761e-09, 2.985064284644068e-08, 1.4960315297685156e-07, 0.00020926912839058787, 0.00010508351988391951, 3.184015184842792e-08, 2.8466995871667677e-08, 4.5714560314991104e-07, 1.3857322400667726e-08, 3.4563848316793155e-07, 1.6487705423173793e-08, 2.1596387966837938e-07, 2.05385418894366e-07, 7.64777574602249e-09, 5.423663083092833e-07, 2.6610085246403514e-09, 6.3336700328875395e-09, 4.1696793573464674e-08, 6.847702138657041e-08, 2.516114649964152e-09, 3.188232966522264e-08, 8.099989656784601e-08, 1.801620008734517e-08, 5.6219477961860775e-09, 7.510231171181658e-06, 5.558248972192814e-08, 3.9159335756266955e-06, 3.9675134644312493e-07, 0.00011814734170911834, 2.301797081827317e-07, 1.6827998194912652e-07, 4.252328653819859e-05, 1.3672378784690409e-09, 7.087983249220997e-06, 2.0959450353075226e-07, 5.148158948031778e-07, 3.616040444853752e-08, 2.2336671179346013e-07, 2.5071025788747647e-07, 0.001222127233631909, 5.259544466618138e-10, 6.538089110108558e-06, 2.5577039650670486e-06, 4.238356154928624e-07, 3.2132660976458283e-09, 2.2822971956770743e-08, 1.235816893263575e-09, 1.769756252656407e-08, 2.6330106095429073e-08, 0.00012970193347427994, 2.139679766699487e-09, 1.1643626507407134e-08, 9.936433897905772e-09, 4.3484998712983725e-08, 2.448688860567927e-07, 1.2913919533730223e-07, 4.715497198048979e-06, 2.6842116085390444e-07, 9.949490475946732e-08, 1.6964262385954498e-06, 0.00013370181841310114, 0.00021607766393572092, 9.933906852666041e-08, 1.0327015615985147e-06, 2.6898039351408443e-10, 7.050642125250306e-06, 0.00489719258621335, 2.6207991332682923e-09, 5.88800119771804e-10, 7.322316974978094e-08, 1.5183898582904476e-09, 1.388348096043046e-06, 0.00014864324475638568, 1.0749050538549909e-08, 2.977437816298334e-06, 2.0359784684842452e-06, 4.087152319698362e-06, 0.0001539646473247558, 8.751496807235526e-07, 2.1817244544308778e-07, 1.9529004475771217e-06, 6.450096012855511e-09, 1.9547664820152022e-08, 3.29478439198283e-06, 5.580463380283618e-08, 6.808768659993802e-08, 1.5199724145986693e-08, 5.818657200507005e-07, 5.174073391955858e-10, 7.160277164075524e-06, 3.828587978205178e-06, 8.067157185320806e-12, 2.4953455067588948e-05, 3.049987526537734e-06, 1.2943507377016772e-09, 3.10161518690677e-09, 3.976898366175874e-09, 2.4915298713068523e-09, 4.7474279085690796e-07, 3.691801353511437e-08, 1.5118321039508942e-09, 1.2507619430834893e-05, 6.737046192029084e-07, 7.470416221622145e-06, 4.792405761122609e-08, 0.002755146473646164, 8.11723793958663e-07, 3.8652285638818284e-07, 9.87477477565335e-08, 1.559140372364709e-07, 2.0219826524225937e-07, 3.5292480049520236e-09, 5.366554773900134e-07, 9.754965049069142e-07, 2.1763765900573162e-08, 1.6637677902053838e-08, 5.555346069741063e-05, 1.8264901200382155e-06, 8.946935992071303e-08, 1.0102279901502698e-08, 1.5796724994743272e-07, 4.971605349624042e-10, 2.001063315049123e-08, 1.7286350129097627e-08, 4.001098474759601e-08, 1.817266195303091e-07, 4.294864538678667e-06, 4.938916390528902e-06, 1.1895474472112255e-07, 5.525790243154916e-07, 5.594176855083788e-08, 9.05340158396939e-09, 2.6409098907720363e-08, 4.400193631681759e-07, 1.6672531728545437e-06, 3.71525929949712e-05, 0.0005382457165978849, 7.197628804078704e-08, 3.438793000354323e-11, 1.0230940006294986e-06, 5.761100041468126e-08, 2.1966156964481343e-06, 6.51553089259238e-10, 6.103723990236176e-06, 4.317002549214521e-06, 3.1193123084527485e-10, 6.474259066635568e-07, 4.205293002712551e-09, 1.1590861603849589e-10, 2.811826149695662e-08, 1.3378521901419305e-10, 0.0002988272754009813, 5.67263462869505e-08, 0.0016850886167958379, 5.046588739787694e-06, 6.291329555097036e-07, 4.574204126583936e-09, 3.5954548227579153e-09, 0.00016435924044344574, 1.083052580952426e-08, 2.559789491485276e-09, 1.2293648069316987e-05, 1.6370311186619801e-06, 2.974850758619141e-06, 2.33140690397704e-06, 1.0115397230947565e-07, 5.260975012788549e-05, 3.1674517231294885e-08, 6.17405548908323e-09, 2.1030255314968826e-09, 1.561358864421436e-08, 3.1510969389358934e-09, 7.847989036235958e-05, 1.023829354629413e-09, 6.199523340910673e-05, 7.049927915753207e-12, 0.0008653585100546479, 3.8899639065448355e-09, 3.86311839974951e-05, 3.5389803088037297e-06, 9.540897008264437e-05, 3.4376765323251846e-10, 1.1435163287387695e-05, 0.00010466860112501308, 0.0011152668157592416, 6.934715770512412e-07, 1.6631665857858025e-06, 7.425914549941126e-09, 7.503365395677974e-08, 1.2520334102816832e-08, 2.321107395175659e-08, 9.950871238117998e-09, 1.1969686353197062e-10, 1.8439682492044085e-07, 9.776880416367817e-10, 6.200051028137921e-10, 3.524179348346479e-08, 1.2990641895527233e-09, 1.4302303265323335e-11, 2.5580477540643187e-06, 5.780547667200153e-07, 1.0500154701276188e-07, 1.738467403811228e-06, 1.7600096380299135e-10, 3.0835789033289984e-08, 3.3855023957585217e-06, 3.328955600068184e-08, 0.00010969881259370595, 1.298478657929536e-08, 1.5984436174676375e-07, 5.364145749808813e-07, 9.690507596360476e-08, 1.277944150501753e-08, 0.0002654682321008295, 2.2784171960665844e-05, 2.8716401629935717e-06, 2.6127043639689873e-08, 1.5274054021574557e-08, 9.960339042436317e-08, 7.66502671467606e-06, 7.368058140855283e-05, 6.388679594238056e-06, 1.994917056435952e-06, 6.501428373439921e-08, 1.1132946120540055e-08, 5.375833025261789e-12, 2.917350583331313e-09, 5.7285416232844e-07, 6.174156741423076e-10, 6.685223752356251e-08, 6.14455641922973e-09, 1.567947438552153e-09, 5.007716197269474e-08, 2.6340496788179735e-06, 0.0032094805501401424, 1.3041269994573668e-05, 1.2031156074954197e-05, 1.524773883543329e-11, 4.6190247182664734e-09, 1.7928100533026736e-06, 2.3310612959903665e-06, 1.107271607736493e-08, 1.3071226323546625e-09, 2.3125187453842955e-06, 6.011904929437151e-07, 5.710645289980221e-09, 1.064672596839955e-05, 4.4114120356653075e-08, 5.9269869012723575e-08, 1.605531110726588e-06, 3.3043929761333857e-06, 2.063797843732118e-08, 6.881417657922384e-10, 7.964841231000719e-09, 0.022275108844041824, 7.66182324696274e-07, 7.754828743600228e-07, 2.3876808086242818e-08, 1.3069824111866524e-09, 2.35883179655616e-09, 1.31131884018032e-07, 1.0762382096629608e-09, 7.0335854616132565e-06, 2.886659977718864e-08, 2.5460620236117393e-05, 7.521860879933229e-06, 1.4608688161388272e-06, 1.698171217867639e-05, 4.6537657908629626e-05, 1.1186671144969296e-08, 3.2037430486298035e-08, 2.0983635806715029e-07, 4.851934409799696e-09, 1.8493197251334692e-10, 7.630355298715585e-07, 3.553455098881386e-05, 0.004623930901288986, 5.216009526520793e-07, 2.1272047945330996e-07, 0.00011937682575080544, 1.186827830679249e-06, 6.837957243988058e-06, 6.734315149969916e-08, 1.7360986248604604e-06, 1.8510746269129186e-09, 4.081580559645026e-11, 2.0034130798762817e-08, 6.63714854454156e-06, 2.8824556963513714e-08, 2.674503775779158e-06, 3.2324660423910245e-06, 1.1816343736814616e-10, 3.017717986608659e-08, 4.635576544842479e-08, 4.412243015394779e-08, 1.4440719496633392e-05, 2.0000666722808091e-07, 0.00012901713489554822, 3.889245370203298e-09, 6.206615239534585e-07, 3.612486088044875e-09, 8.987436928009629e-08, 2.4574689071882316e-12, 1.3505670359847954e-08, 9.750185370194231e-08, 1.717846698223724e-11, 5.221169274705062e-08, 2.9462768225130276e-08, 9.517303851680481e-07, 6.364953151205555e-06, 3.406304216468925e-08, 3.4635903034541116e-07, 4.922220000480593e-07, 4.412627276906278e-06, 3.9813085095374845e-06, 4.583720055961749e-06, 6.275308805925306e-08, 4.395085184683012e-08, 7.08214251132766e-11, 4.721490043380072e-08, 4.495208543175977e-07, 5.027609405183764e-12, 4.0051585159517344e-08, 5.5813323740494525e-08, 3.8477722341667686e-07, 1.117081538382081e-09, 8.801154035609215e-05, 4.141355418596504e-07, 5.349219009076478e-06, 4.720315160966493e-08, 3.599907813622849e-06, 7.777626933602733e-07, 4.1441012399445754e-07, 6.262545149127163e-09, 2.5071363779716194e-05, 1.5690259091982739e-09, 3.536790609359741e-05, 1.1269711563954843e-07, 3.2652711524860933e-05, 3.486336481728358e-07, 1.5285549181953684e-07, 1.4564274124495569e-06, 7.579846084126984e-08, 1.710869810267468e-07, 3.535251025255093e-08, 1.3387460739977541e-06, 2.1769631075585494e-06, 2.6513522488613717e-09, 3.055771413684738e-09, 2.415625931462273e-05, 1.8064628193315002e-06, 1.0231573366326074e-09, 3.3339256355624514e-10, 2.2839972135813014e-09, 3.431556615396403e-05, 5.9636100502302725e-09, 2.3098364181350917e-06, 1.7660013895692828e-07, 3.693530175041815e-07, 1.0491613267049615e-07, 7.32368921063653e-09, 3.7458171675552876e-08, 1.0529177416174207e-05, 1.6971706884305604e-07, 1.3098211404383164e-08, 6.941834271856351e-07, 9.054801193997264e-05, 1.0630612834461317e-08, 4.0914738264064e-09, 3.822226233296533e-07, 7.026919320196612e-06, 1.5892337046352623e-07, 5.042764539275879e-10, 1.5717241694801487e-05, 1.611615019214696e-08, 1.7004900265682465e-10, 1.658076870247882e-10, 4.768947903244225e-09, 2.256849596093957e-09, 3.0946725360081473e-07, 4.8719297041088794e-08, 4.837839573390568e-10, 6.119675077798092e-08, 9.71388924853045e-09, 9.848277748858436e-09, 2.0339537343261327e-07, 1.818247982732224e-11, 0.005284599494189024, 2.1075149561511353e-05, 1.5429086897000843e-09, 2.952575073322805e-07, 1.4810616733029747e-08, 8.976979006547481e-05, 1.993061147231856e-07, 2.9667293688362406e-07, 2.31909891290627e-09, 3.200826062532691e-11, 4.878188519796822e-06, 6.0655471756376755e-09, 4.349596565589309e-06, 1.1322773616484483e-06, 5.269129488283397e-08, 1.915827496645761e-08, 6.961304279684555e-07, 7.160867880884325e-06, 1.3704827324545477e-05, 1.7229381228389684e-06, 1.7402450591674778e-09, 1.914879987907625e-08, 1.1247794873270323e-06, 0.00010848449164768681, 9.90360433661408e-08, 1.2548485756269656e-05, 3.5917094010073924e-07, 5.679771106770204e-07, 2.7445830710348673e-06, 1.0535480214457493e-05, 5.219201852924016e-07, 6.581947786798992e-07, 4.8852987627467925e-12, 2.365872475706965e-08, 7.703019333504812e-10, 4.862467335442489e-07, 6.16184649970819e-07, 3.3013694178407604e-07, 1.3464234172033684e-09, 1.1211639616703906e-07, 1.1555283813136086e-10, 3.01683797943042e-07, 2.9749973595727397e-09, 5.1458513894431235e-08, 4.031484422739595e-06, 7.228646836665575e-07, 4.628197998457484e-12, 3.8231449117631655e-09, 2.642581442557912e-08, 2.5892063493238027e-10, 6.492143711511744e-07, 1.7354373085254338e-06, 6.481951686510001e-08, 6.446827569561719e-07, 8.082093927441747e-07, 2.7982546271232422e-06, 1.3479979799058128e-08, 1.3915324359459191e-07, 1.4008024962208765e-08, 1.099660039471928e-06, 4.141070153451665e-09, 1.3647511877934448e-05, 7.583343375472396e-08, 2.996872472404988e-10, 1.4581332807850345e-09, 1.7833933441124827e-07, 1.5390613725685398e-06, 3.1765071639711095e-07, 2.4303665213665226e-07, 1.872159458571332e-07, 2.3091962120247445e-08, 2.0434406167169072e-08, 4.8233923799489986e-11, 3.8322656337186345e-07, 7.427508563750962e-08, 3.320528321637539e-06, 6.540982440128573e-07, 3.3190078738698503e-06, 9.237359677172208e-07, 7.335286795751017e-07, 2.961921381938737e-06, 2.8814726160675264e-09, 2.7247283540532408e-08, 1.658247811064939e-06, 2.142532053994728e-07, 3.214315569266546e-08, 1.147829140762946e-11, 3.180370185873471e-05, 2.0424533175855686e-09, 0.001673801918514073, 4.3027007734508516e-08, 4.586586510413326e-05, 1.2509396583482157e-05, 2.868586790327754e-08, 9.815568091653404e-07, 1.0830915719850509e-08, 0.0013253167271614075, 8.740780543803339e-08, 5.644575917784778e-09, 2.883072802717379e-08, 1.422462077016462e-07, 2.3752235733809357e-09, 8.000637308214209e-08, 3.391832308352605e-07, 1.250592504220549e-06, 3.0791691418841083e-09, 1.4137295067939704e-07, 8.551677699308868e-10, 3.1523399446342637e-09, 5.924356969444489e-07, 2.855380376587391e-10, 7.242828814923996e-06, 1.0691821898944909e-06, 1.1041045411275263e-08, 2.254891455777397e-07, 3.942569187787015e-11, 9.417248492127328e-08, 5.0016033128486015e-06, 5.5859384673340173e-08, 7.573823381790135e-07, 3.678633175852042e-09, 1.6447191910629044e-06, 1.197652022000284e-08, 7.248311885632575e-05, 2.3621657518901884e-08, 6.169194044503001e-09, 5.375237014959566e-05, 7.252182854244893e-08, 7.699978987751877e-10, 1.125516568833973e-08, 9.595528638328688e-08, 0.00023920844250824302, 1.6642315303627697e-09, 1.1687384358083364e-06, 1.8744760836852947e-06, 1.802385014570973e-07, 6.316901846048495e-08, 8.081620705979731e-08, 9.04371466603493e-10, 3.008091766787402e-07, 9.127045075274509e-08, 1.0617843599902699e-06, 1.4453520691404265e-07, 4.847130252727538e-09, 2.4547325665480457e-05, 5.880812992131723e-08, 1.7090222570459446e-07, 2.1303826758245492e-11, 1.2871472776510018e-09, 1.2119285202061292e-06, 1.349339981970843e-07, 8.180000676816235e-09, 1.21088760351995e-07, 3.1297386904327595e-10, 4.85418968310114e-06, 2.4614873836981133e-06, 9.123625943630032e-08, 3.5999357805849286e-07, 1.5186305063252803e-05, 1.0373132397489826e-07, 4.087754787440723e-11, 2.7030310434383864e-07, 2.8695521336885577e-07, 1.1335007599200253e-07, 2.2163273438735587e-09, 1.5075462215463631e-05, 6.5951462069335776e-09, 1.1553698442412497e-07, 3.6182903784265363e-08, 1.6622456300297017e-08, 7.743438618490472e-05, 7.095054543881929e-10, 9.54997303779237e-05, 6.305599375977522e-10, 1.0176386844307217e-08, 5.405564706961741e-07, 3.083557658101199e-06, 4.118833203392569e-06, 1.597430987487769e-08, 2.0511199405603975e-09, 1.6012827472877689e-06, 6.328091694740579e-05, 1.6766745147833717e-07, 3.1486641205447086e-07, 1.484198946855031e-06, 5.979935849609319e-06, 6.163448915685876e-07, 1.5017332771094516e-05, 1.3578736570707406e-06, 1.131096638573581e-08, 1.2111911473766668e-06, 3.266121151312973e-09, 1.475795414762615e-07, 4.262170023139333e-06, 1.8134227275368175e-07, 2.6158990294788964e-05, 3.646979607196954e-09, 2.4927248887252063e-06, 1.3126780686434358e-05, 2.4083863081614254e-06, 1.292815809961212e-08, 2.8712105404338217e-07, 3.589499101508409e-05, 6.876844926839709e-11, 9.729161476457193e-09, 4.6248658236436313e-07, 6.373945211635146e-07, 1.3151057828508783e-05, 1.5994726709323004e-05, 4.334194443345041e-07, 1.3674918974970751e-08, 3.2213606004916073e-07, 6.833855081822549e-08, 2.379710082323072e-07, 1.6801604942884296e-06, 4.960983801538532e-08, 1.3830417922022775e-09, 2.2498680252169834e-08, 1.1349774240443367e-06, 2.765002307114628e-07, 5.065927233260936e-09, 3.483459067865624e-06, 1.2880303756901412e-07, 1.1670945241348818e-06, 2.9183345304772956e-06, 7.585398265064214e-08, 1.8814732527516753e-07, 9.036158488129331e-09, 2.1227881674690252e-08, 3.269105519621007e-08, 4.7769876942993506e-08, 0.0003877870040014386, 6.356530857232201e-10, 6.640052561124321e-07, 9.510935342404991e-06, 3.7118245472811395e-06, 1.8451488870141475e-07, 2.3454751385543204e-07, 3.415422966668302e-08, 7.018211789500128e-08, 2.9119843247826793e-07, 1.8005624724537483e-07, 4.89110796308978e-08, 7.271684353327146e-06, 0.0009982496267184615, 5.6663846947913044e-08, 5.537579454539809e-06, 2.814014976593171e-07, 8.000306195299345e-08, 4.5013018312722863e-10, 1.0401134398208e-09, 1.279410533072678e-08, 5.165595595713057e-08, 1.3695145018033372e-09, 5.253216386336135e-07, 2.556643430295935e-08, 9.503350328898819e-10, 1.2484692888392601e-06, 1.057950385074946e-06, 2.2273142974427174e-07, 5.088491434435127e-06, 5.653378321035518e-10, 7.021402836926427e-08, 7.67347842156596e-07, 2.393734632732958e-07, 1.2469255628033693e-09, 1.2910765008200542e-06, 5.946312285232125e-06, 3.981338068115292e-06, 2.12898314089216e-07, 1.1036375724415848e-07, 3.1416209911316884e-11, 4.4086944399168715e-05, 1.4488671595813685e-08, 2.693089129479631e-07, 8.18069523234044e-09, 5.898672043258557e-06, 0.0006484147743321955, 1.4634216313424986e-05, 4.518034529610304e-06, 5.7918295937042785e-08, 1.2057468268267257e-07, 1.3388326181029697e-07, 4.614760837284848e-05, 1.5825936117153105e-08, 3.6765407003258588e-06, 3.68924617077937e-07, 7.057472117821817e-08, 0.0001686198520474136, 1.2315156709519215e-05, 1.84264388281008e-06, 8.054555109993089e-06, 5.6851431509130634e-06, 2.53841881203698e-06, 2.4482540084136417e-07, 7.295911075289041e-08, 3.88517173632863e-06, 0.014415272511541843, 5.547203159039782e-07, 1.3297696277447812e-12, 1.9945812823607412e-07, 0.0010986727429553866, 1.6738232488933136e-06, 5.735759089908754e-10, 1.8233084119856358e-05, 3.0016281016287394e-07, 9.45983686051477e-07, 4.6556269950315254e-08, 3.439585896103381e-07, 1.8530810219630212e-09, 3.380407759268067e-11, 1.9009416973858606e-06, 2.065950535645733e-11, 1.3942926813115264e-09, 1.3830317868723796e-07, 5.708884032173955e-07, 5.913538458912626e-08, 9.314009366789833e-05, 1.4049696517304255e-08, 4.1113000293080404e-07, 2.1030414909528616e-10, 7.777040345047226e-09, 3.228587885928391e-08, 0.0001672485377639532, 2.1455317522622863e-09, 1.0985138487740187e-06, 1.1031805115635507e-05, 2.200508106398047e-06, 0.00011913668276974931, 5.375371259219719e-08, 8.102777542262629e-07, 5.936401521466905e-07, 1.472319127060473e-06, 8.864415690368332e-08, 1.0357636710978113e-05, 2.5806772100622766e-05, 1.712243999918428e-07, 1.5826641686089715e-07, 8.344882758137828e-07, 0.00042296553147025406, 3.309174445575991e-08, 7.446096361718446e-08, 1.8496814080393165e-09, 7.062876647978555e-07, 7.563457415926678e-08, 7.089402398463562e-09, 5.658756307980184e-08, 2.0139430123577995e-09, 1.3701051102543715e-05, 1.628391328267753e-05, 1.0789703992486466e-06, 1.2575459322761162e-06, 1.2052181830313202e-07, 1.065931428456679e-06, 4.643887496769139e-09, 1.0572603903469258e-09, 1.8281078155268915e-05, 3.856188612871847e-08, 1.4425641836623981e-07, 6.902773293404607e-06, 7.861075914661342e-07, 1.3062086134940643e-10, 9.487821080256253e-05, 8.414759555819273e-09, 1.2920228775747944e-10, 5.6868293540901504e-06, 1.7499868221193537e-08, 5.334828756531351e-07, 1.7624080328459968e-07, 1.644666269839945e-07, 2.9341833851503907e-07, 1.995862852766095e-08, 1.0813544122356689e-06, 4.9818406344659394e-11, 7.50748370137444e-07, 1.1948903022585e-07, 6.585189993302265e-08, 4.224805252306396e-07, 1.3808344192511868e-06, 6.124781066318974e-05, 9.448342802587106e-11, 1.6654021237627603e-06, 2.639609017474065e-12, 6.637566940526085e-08, 1.2657312254304998e-05, 1.5222468391584698e-05, 7.304198357260816e-10, 2.352898445678875e-05, 2.039189865854496e-07, 1.7316090783481286e-09, 5.773799330199836e-06, 2.726324410673442e-08, 5.424234394979521e-10, 1.319861979709458e-07, 4.854302005696809e-06, 1.8294667825102806e-06, 5.59094985419506e-07, 3.569648754364607e-08, 1.3158842193661258e-06, 1.437444563379131e-08, 1.0600396116444699e-08, 1.2005160199279885e-09, 1.65500338766833e-07, 0.00012598092143889517, 3.1212111935019493e-06, 1.586048625767944e-07, 2.2232758567497513e-07, 1.5406667941419983e-08, 2.439426616263063e-08, 2.108098229314237e-08, 5.833897830598289e-06, 1.9450050103841932e-07, 1.8982246956511517e-06, 1.1029472801737938e-07, 5.5811806305428036e-06, 3.180436760885641e-05, 1.5337930392433918e-07, 3.71095779883035e-07, 4.6727786866540555e-06, 5.095953383715823e-07, 0.0006629715790040791, 4.313313161502519e-12, 2.123001468135044e-05, 4.505432684709376e-07, 1.26432431191148e-09, 0.00011444709525676444, 6.978209057706408e-06, 1.0677011232473888e-05, 0.00012147027882747352, 1.975536861209548e-06, 3.65295136361965e-08, 2.3774630264483676e-08, 1.9729444034055632e-07, 4.210661153081219e-09, 0.00028618230135180056, 2.199574510086677e-06, 4.8885551223065704e-05, 0.00027934127137996256, 1.2763325685227755e-06, 2.693977023682237e-07, 1.8097583431853081e-09, 1.4486725774531806e-07, 4.418177468323847e-08, 0.00021707215637434274, 3.277916766819544e-05, 2.219912147438663e-07, 1.721814584243475e-07, 8.704196474695891e-09, 0.005893339402973652, 1.104162492993055e-06, 8.032991445361404e-07, 7.579454397443897e-09, 1.2171008165751118e-05, 1.0856459198294033e-07, 2.7383330234442838e-06, 8.62030674397829e-07, 3.498535807011649e-05, 5.201470685278764e-06, 1.2482049669415574e-06, 4.0181311056208813e-10, 1.8630886188475415e-05, 3.616410282347715e-08, 1.4123162372925435e-07, 6.9375219027278945e-06, 9.028872938188215e-08, 9.524208621769503e-07, 2.5233305223082425e-06, 5.816124803992295e-10, 4.191509361817225e-10, 1.1201130973859108e-06, 1.5537000308540883e-07, 2.3341713978330958e-11, 3.9492924486239644e-08, 1.253895106856362e-06, 1.4108465506978973e-07, 1.803566192393191e-05, 1.4361130062923166e-09, 7.450197614389253e-08, 1.5107434592209756e-05, 1.2875813126811408e-06, 2.9632401492563076e-06, 2.0633299235850444e-10, 2.1899999591568076e-09, 1.4326509699458256e-05, 1.4613494414561501e-08, 2.4959337707741724e-08, 6.559130724781426e-06, 9.565593472871114e-07, 4.060336511457763e-09, 1.2848464336912002e-07, 3.467620786068437e-07, 4.466412804049469e-07, 8.173038634762264e-11, 9.4051870291878e-08, 1.4697255892315297e-06, 2.98120630759513e-05, 3.47659588442184e-05, 1.793026740415371e-06, 1.8224990299131605e-06, 6.971891508555927e-08, 2.945048152014351e-07, 3.4076019801432267e-06, 2.914871954917686e-10, 6.618594561302871e-09, 5.312740086083068e-06, 2.7885413422801264e-10, 1.2287084246054292e-05, 5.033954764144255e-08, 1.9804469175710437e-08, 1.8449645722284913e-05, 2.0439069885469507e-06, 0.000515305029693991, 5.078287124149483e-09, 4.0634470788170063e-10, 2.526137450331589e-07, 1.6673904923436567e-08, 2.4741050097532025e-08, 4.473633907764452e-06, 6.020524523364656e-08, 3.775388449867023e-06, 1.0594940924590901e-08, 2.448970235491288e-07, 1.4531774183978996e-07, 1.11166241367755e-06, 1.007389638374434e-08, 3.869895071062501e-09, 1.8998962403316e-08, 2.0694520230790658e-07, 2.0985440940535227e-08, 2.827032972163579e-07, 2.6727584554464556e-06, 3.746821186645377e-10, 8.173466348182501e-09, 2.0278077883517653e-08, 1.0104634490248898e-10, 5.4871605925654876e-08, 2.4496503669979575e-07, 5.620527645078255e-06, 6.341498988149397e-07, 1.2669288480537944e-06, 1.7925384510064646e-09, 7.821088665593834e-09, 5.58346414436528e-07, 0.00011961677228100598, 6.516954158541921e-07, 3.627143385642739e-08, 3.7326914025470614e-05, 0.0007453051512129605, 2.9132116718955103e-09, 2.568819379433762e-09, 1.6436439409517334e-06, 5.372440825546221e-10, 8.342290129803587e-06, 5.230899091657193e-08, 1.4933721104171127e-07, 2.7198473162570735e-07, 2.2877106875540676e-09, 5.001109002478188e-07, 2.211182170697157e-08, 3.6436291761532402e-09, 2.5718161822396723e-08, 1.50481760385901e-08, 5.122985413663628e-08, 1.7055752365990884e-09, 4.948930836690124e-06, 2.98670366216669e-10, 3.608366023399867e-05, 7.413839853143145e-08, 2.2398109755794593e-10, 1.8284117553335477e-09, 3.9889215486255125e-08, 1.3944508880925355e-09, 2.893738383136224e-06, 7.207517915830408e-10, 1.1653557285828953e-10, 5.8742333663985846e-08, 2.3964451756341987e-08, 3.1489594221056905e-07, 0.0007365202181972563, 6.644008954115321e-11, 1.9483771040995634e-08, 4.83847131249604e-08, 4.752328459289856e-05, 1.27225271984166e-10, 6.998472599661909e-06, 0.0002246889634989202, 2.100568785579071e-08, 5.114765855296355e-08, 2.1155838680897432e-07, 4.763337813074031e-08, 2.217019678951715e-09, 3.9259589357243385e-06, 2.0188366534057423e-08, 3.334504339314037e-10, 1.825539243327512e-06, 4.744304169435054e-05, 1.8654183753241682e-10, 3.9694043191262907e-11, 2.8940860374859767e-07, 2.713134605869527e-08, 3.6667585634475586e-10, 0.0002759270719252527, 4.261992092580158e-09, 2.2152219116833294e-06, 0.0019390294328331947, 9.636949016567087e-08, 3.7033139506092994e-06, 0.00013001916522625834, 4.1995215638301886e-10, 5.694010951629025e-07, 3.3324340620310977e-07, 1.295526317335316e-07, 3.3121559681603685e-05, 2.851760143585125e-07, 1.1897211038558453e-07, 6.538484740303829e-07, 2.7480873541207984e-06, 4.722180690919231e-08, 3.914444732799893e-06, 3.0478026769031885e-09, 2.4514518415230668e-08, 4.552232812926604e-09, 2.076443479381851e-07, 1.2811148053515353e-06, 3.3384292663640736e-08, 6.079115166812699e-08, 7.78754980501617e-08, 2.1446577136430278e-07, 8.083276270554052e-10, 3.6384759027896507e-07, 2.880406782423961e-06, 1.6763716814693908e-07, 2.1553150930664344e-10, 8.935476785154606e-07, 6.804849128627666e-09, 6.18126279050557e-08, 1.1526553045371202e-08, 3.9482483771280386e-07, 1.1763474742909619e-11, 3.175320273385296e-08, 5.28189270099233e-10, 0.00012521953613031656, 8.31967668091238e-08, 1.0726114441084178e-10, 7.80316986492835e-06, 1.79597190452796e-08, 1.8073506025118036e-09, 1.2420099665177986e-05, 1.6099220090382005e-07, 6.323167554000975e-07, 4.713193169081364e-10, 1.6109778471218306e-06, 5.1970946515211836e-06, 5.030568445363315e-06, 2.91841084631983e-09, 4.493212600209517e-06, 0.00307569932192564, 1.5970115896379866e-07, 2.092360045935493e-05, 6.222879164852202e-05, 1.1166973763465649e-06, 1.4106250034728873e-07, 1.34806623464101e-05, 7.169359150793753e-08, 3.6619942420657026e-06, 2.3794219927708582e-08, 1.0069034942716826e-05, 5.4604484489573224e-08, 1.0392285787474975e-07, 4.903147754475867e-08, 6.655940243405212e-09, 1.072465124707378e-06, 2.476815268437349e-07, 7.531539836236334e-09, 5.609289814856311e-07, 1.0510471838642843e-05, 5.434874310594751e-07, 2.7884388131838023e-08, 0.0011822872329503298, 2.095865703211075e-09, 2.575870666987612e-06, 1.208838718014249e-08, 1.2001761206192896e-05, 7.358099685106367e-10, 0.006193619687110186, 2.2737869365130337e-09, 2.521370697650127e-05, 3.207163246088385e-08, 1.2542695060346887e-07, 7.238634935902155e-08, 9.512263865474324e-09, 7.926058970042504e-06, 0.348932147026062, 8.385695338120058e-08, 2.5050113094948756e-09, 3.142265267430666e-11, 6.955851858947426e-07, 2.647948895173613e-05, 1.6589104534503463e-09, 4.812086899619317e-06, 1.3083240446576383e-07, 1.3873163062783078e-08, 9.891799379602162e-08, 4.688361514126882e-06, 3.2263988032354973e-06, 1.8024116798187606e-05, 5.351121767915856e-09, 1.2324132967478363e-06, 2.8209007352231197e-10, 4.566815334783314e-07, 1.563504952173389e-06, 1.0139490314031718e-06, 1.2521542913646044e-08, 2.1624376955742264e-08, 1.3190729619694253e-09, 1.9048119526132723e-08, 1.6488191367791671e-10, 1.3409348298409896e-08, 4.399873603233573e-07, 0.0018298878567293286, 2.002004322321227e-07, 2.475221663189586e-07, 1.1146270573192396e-09, 1.1764598639274482e-05, 5.337038146535633e-06, 4.6298549705170444e-07, 8.829273312471742e-09, 2.4205670356991504e-08, 2.046811387046432e-09, 4.74316983259393e-12, 2.5695827687854944e-09, 2.122115984093398e-05, 1.515240377614191e-08, 1.275661816180218e-07, 6.6322445491096e-07, 9.618389418264428e-10, 1.4089609067013953e-05, 6.027774190897617e-08, 9.779805623111315e-06, 2.1137459043529816e-05, 9.667992344475351e-06, 0.00025345050380565226, 3.4695170825216337e-07, 0.00012220561620779335, 3.484039439172193e-07, 3.103169490259461e-07, 3.3698510648605406e-09, 1.3242364730103873e-05, 8.455159239417753e-09, 2.5967956673866865e-09, 0.0001047887490130961, 1.571746111039829e-08, 1.6503109989685072e-08, 7.752326003362953e-10, 4.749406645743193e-09, 8.908457260758951e-08, 2.418903932266403e-05, 7.296091508735003e-10, 1.2240496971571702e-07, 8.377805329473631e-07, 3.154634242719112e-08, 7.440544891323952e-08, 7.175069640652509e-07, 8.00988875226949e-09, 1.4757504551710099e-08, 2.919604114026697e-08, 1.071718092049423e-08, 5.187833721720381e-06, 3.561140271557406e-10, 3.6711054196558734e-09, 2.526803939417732e-07, 2.356335393471909e-08, 1.7069113766865485e-07, 4.3815893135956685e-09, 1.9870240066666156e-06, 1.2641079294439805e-09, 1.3301910257723648e-06, 1.4974105289056183e-10, 6.638610106080023e-09, 5.5986029146026794e-09, 1.4594282049529284e-07, 1.6293014937218686e-08, 5.143593284628878e-07, 1.5812142351023795e-07, 2.1852401914657094e-06, 2.2697564361351397e-07, 0.0001342813193332404, 2.3920350145090197e-09, 7.830711547285318e-05, 9.089157538255677e-05, 2.4157233990962368e-09, 4.923300434001021e-09, 0.0019442252814769745, 4.973469192037783e-09, 2.9536337264879364e-10, 1.0131619987419072e-09, 2.678184500837233e-07, 2.178205988911941e-07, 3.5207108062262904e-11, 4.887093041361368e-07, 6.235432920220774e-06, 1.7347283574054018e-05, 1.3294500149640953e-06, 4.6329692082736074e-08, 1.8864272988139419e-06, 1.2865971399378395e-08, 2.1043642846052535e-06, 0.014668933115899563, 8.801488010634628e-11, 3.1927873351378366e-05, 1.271871070684938e-07, 6.73197453338048e-09, 0.0001329070219071582, 3.638761469915153e-08, 3.275494542265278e-09, 1.5323976185754873e-05, 5.197009159019217e-06, 2.7105505068902858e-05, 6.39461213722825e-05, 9.009998969133903e-09, 3.6043431350663013e-07, 8.529137858204194e-07, 1.1793464871345805e-08, 2.514722208246667e-09, 1.67816524481168e-05, 9.883161737889168e-07, 9.083135228138417e-05, 1.532572735385429e-09, 8.278005407191813e-05, 5.738764130569507e-08, 7.092120313245687e-08, 1.5237629824582655e-08, 1.5539966625510715e-05, 4.8619654080539476e-08, 5.032152330386452e-07, 4.202912757406141e-12, 3.0913827053780096e-10, 2.1913011849505892e-08, 9.29357520362828e-06, 1.0868120625673328e-06, 5.5167074606288224e-05, 1.1193095588168944e-06, 2.2216558193122182e-07, 1.8607256606628653e-06, 1.720245847991464e-07, 8.957785624374992e-09, 5.442900146590546e-05, 3.034982043459422e-08, 4.027211630841521e-08, 5.8335057673275514e-08, 1.9571979592569733e-09, 3.7542242353083566e-05, 6.258580106077716e-05, 4.4717299307883707e-10, 8.150672359263922e-10, 4.682832965841044e-08, 1.989628600762572e-11, 3.911089550001634e-07, 5.541043037737836e-07, 5.8686239867711265e-08, 1.2436840961527196e-06, 1.1479825161586632e-06, 7.424852288551165e-09, 7.964757031686531e-08, 1.7135894481157266e-08, 4.740402204106431e-08, 3.0598366063117055e-09, 1.2977723784501904e-08, 5.145860200173047e-07, 7.554292608125124e-11, 1.2047955806337995e-06, 1.0740911875473103e-06, 7.586383787838713e-08, 0.00014927292068023235, 1.3766526762992726e-07, 9.193062311396716e-09, 1.792922557797283e-05, 2.1852837317482e-09, 4.055634761357396e-08, 8.187058142539172e-09, 3.834595696527998e-10, 0.0006430632784031332, 3.440311502345139e-07, 6.860427959054505e-08, 3.227952447559801e-07, 1.5992584394552978e-06, 9.508973164429335e-09, 4.330563907473106e-09, 1.8894417053161305e-07, 1.777458055585157e-05, 7.087959374985076e-07, 7.402487511853906e-08, 4.956142947776243e-05, 8.086809089036251e-07, 2.4676207885931944e-06, 1.0681344519980485e-07, 0.0001708447962300852, 7.241529687007642e-08, 5.568434602309935e-09, 1.9417701935253717e-07, 0.0010566789424046874, 3.4866751708051424e-09, 1.2820877600461245e-05, 0.0001370318204862997, 1.4597884728573263e-05, 0.000163822784088552, 2.6718740286924003e-07, 0.00023747835075482726, 2.0879245710148098e-07, 3.762787414984814e-08, 4.524335750488717e-08, 3.4254397096589173e-09, 3.9262031350517645e-05, 1.262788373423973e-06, 1.6493567045472446e-07, 1.256358217460729e-07, 6.943540711290552e-07, 2.2868878346571364e-08, 2.1399905563157517e-06, 1.5415145071528968e-06, 1.624312517378712e-06, 5.7124311751977075e-06, 7.324463513214141e-05, 4.4083883210532804e-08, 4.047529245099213e-08, 5.490599294666332e-12, 4.814161229660385e-07, 7.858043886699306e-07, 8.741062096362384e-09, 2.620040788769984e-07, 4.1525323467794806e-05, 1.239766067584469e-08, 3.056130015721692e-09, 1.9673690985655412e-05, 2.8714556066233854e-08, 0.00023486983263865113, 3.834591552731581e-06, 2.245224663965928e-07, 1.2233033430675277e-07, 2.4699345431145048e-06, 2.9759017472485993e-09, 7.533725643327216e-09, 1.0943417283471035e-09, 4.8859888579499966e-08, 1.3809584231694316e-07, 7.518340083834119e-08, 1.0168268005372738e-07, 9.092276513911202e-07, 1.8202459539651272e-09, 1.9207575405744137e-06, 2.5048967344787343e-09, 2.168161614690689e-07, 1.7284703517361777e-06, 2.8188996026301538e-08, 6.412816111467379e-11, 1.1094231311403746e-08, 1.9478036804992938e-12, 1.4011588334028602e-08, 9.244678221875802e-05, 5.495930377463765e-09, 5.510549350162819e-09, 4.661575019326847e-07, 3.929757630416475e-10, 2.4349557092850915e-11, 5.592762919448546e-10, 5.19180087810156e-10, 2.2106354080619894e-08, 1.274845089938026e-06, 2.1200233391027723e-07, 1.4008310245117173e-07, 1.1451349024582669e-07, 1.6669390845436283e-07, 6.743076852444574e-08, 7.080474006215809e-06, 2.0491230316110887e-05, 1.6405600433699874e-07, 1.3788763908451074e-06, 6.73304800802299e-10, 7.4992394729633816e-06, 1.2039839020872023e-05, 1.492239107392379e-06, 1.219201664071079e-07, 6.53528573479889e-08, 1.8246962829593372e-09, 6.061384727473751e-10, 2.083049954748617e-09, 3.6624150823172386e-08, 1.875670313511435e-10, 0.00019141110533382744, 1.026410600957206e-07, 0.00014902628026902676, 1.4891672606154316e-07, 9.873620854250476e-08, 4.359246830176744e-08, 3.402802661867099e-08, 4.1411540863123264e-10, 4.8161727939088905e-09, 2.6646799256013765e-07, 5.119109118822962e-07, 1.5673913367209025e-07, 1.1480842943001335e-09, 4.401572856194136e-11, 2.2597176212002523e-05, 5.876198638787855e-09, 4.327811893745093e-06, 7.118606815126327e-10, 4.840982015252848e-08, 1.0358691042711143e-06, 1.5708604905739776e-07, 1.2440127648005728e-05, 5.330409749149112e-07, 9.774578302312875e-07, 9.749396667757537e-08, 3.8701062976542744e-07, 8.799901918266073e-10, 1.1202803307241993e-07, 1.452218567621344e-09, 7.52462092634687e-09, 4.6124991825990946e-08, 5.168146230971615e-07, 1.6421677173639182e-07, 6.950858164600504e-07, 1.4130825398694924e-09, 1.1974322660535108e-05, 7.701311588448334e-09, 3.206096221841648e-10, 0.00016480633348692209, 5.79007632950379e-07, 8.59114557272278e-09, 1.581119306592882e-07, 3.129563879156194e-07, 2.3170024121554889e-07, 7.657939837812933e-10, 1.1081560558068304e-07, 1.4064299875826691e-06, 6.293084152275696e-05, 6.996245360824105e-08, 3.2107184466667604e-08, 2.2394208372134017e-06, 2.64278874340107e-08, 5.746283804342056e-08, 3.0657236038678093e-07, 2.234432017189647e-08, 1.4286558047693632e-10, 5.320477214354469e-08, 1.0934793408523547e-06, 7.645903679076582e-06, 3.648594670835337e-08, 8.545119101199816e-08, 1.4491493560697677e-09, 5.824408066956721e-09, 2.2639446797256824e-06, 7.758687026182542e-09, 0.0024261504877358675, 0.002781449118629098, 7.688198411237579e-10, 5.893376169296971e-07, 8.595192775828764e-05, 2.1664394012077537e-07, 2.3369934876171783e-09, 3.436929674194289e-08, 3.513929641485447e-06, 3.4352265743109456e-07, 2.85248290765594e-07, 1.2381518388338009e-07, 4.008497569429892e-07, 3.0815263535322046e-11, 8.996311606779273e-08, 1.8736832529953062e-08, 7.746414667053614e-06, 0.007327436935156584, 9.946744539135466e-10, 4.279058529732538e-08, 6.815903930146305e-07, 2.4638195100123994e-05, 8.685155080456752e-06, 1.0761027624539565e-07, 2.348210959013386e-09, 7.80090795160504e-06, 1.0416822249226243e-07, 1.2927261039408222e-08, 9.979801234294428e-07, 3.6927083328919252e-06, 8.578114648116753e-05, 3.551532179457695e-09, 2.3488286160500138e-07, 0.000186405741260387, 3.820850338343007e-07, 3.801201557962486e-07, 3.46682526242148e-07, 7.925657428131672e-07, 4.1428056647419e-06, 8.016478147965245e-08, 3.6900221545010936e-08, 1.883911681943573e-05, 5.0256232952961e-07, 8.252298755451193e-08, 5.885178566700233e-08, 9.20413398830533e-08, 9.905079423333518e-08, 2.0488448271294146e-08, 1.320769627000118e-07, 1.923638990319887e-07, 1.0011995073000435e-06, 1.1079148709569608e-09, 3.118072709185071e-05, 2.4863752723369714e-10, 4.765639971537894e-08, 9.746096111484803e-06, 1.4432240291739618e-08, 8.784616767343323e-08, 4.725339022115804e-05, 2.2116362075053075e-08, 1.1513182074374129e-10, 7.293715631462305e-10, 1.2160337803379662e-07, 3.540109537425451e-05, 1.143777467405016e-06, 7.90459608879246e-09, 6.347988801280735e-08, 1.1106138231298246e-08, 0.000542359659448266, 2.0725035938085057e-05, 7.145804374886211e-07, 3.6230524358416005e-08, 1.934609095144424e-08, 1.5782185280865946e-10, 5.9070274005534884e-08, 9.856481483438984e-05, 9.861830241320035e-10, 0.00010900489724008366, 7.53704625822138e-06, 1.5952316971379332e-05, 9.162288188235834e-05, 1.2335946486530247e-08, 4.55353692530025e-08, 6.13382127312434e-08, 1.2747474187335683e-09, 3.904631640239131e-08, 1.849110958573874e-06, 8.25833433755474e-13, 1.1497401253279804e-08, 1.205311184193647e-09, 1.9468220813223525e-08, 4.775179604621371e-06, 1.5142611557023011e-12, 2.9911817467365154e-10, 1.2412675459927414e-05, 6.9253549561665295e-09, 0.0022446613293141127, 8.743890944629129e-09, 3.369481873960467e-06, 3.1539400424662745e-07, 4.137434572726306e-08, 1.1116338752281685e-11, 1.1997033366739629e-09, 6.420070586266036e-10, 7.576478999737901e-09, 2.1678308712580474e-06, 7.048579675483779e-08, 4.7270368952467834e-08, 7.454044492760659e-08, 1.1678708688123152e-05, 6.005121235830302e-07, 6.030246857413601e-10, 1.3797329501130662e-08, 2.6899746785602474e-07, 1.3269906951052235e-08, 2.1132016172487056e-06, 4.9750635611189864e-08, 1.0767858826810084e-10, 4.2606288275237603e-07, 1.0651929338578725e-09, 1.0268706773786107e-07, 9.444796233992747e-08, 1.6086706011719798e-07, 3.263967528255307e-06, 2.2194464577296458e-07, 5.068645805295091e-06, 4.017343826490105e-07, 2.3453678466012207e-07, 4.6014879444555845e-06, 1.192881455835959e-07, 3.264461412300079e-08, 2.3479492128331003e-08, 8.269468594335194e-07, 1.0204639693256468e-05, 1.8011775182458223e-07, 2.8647713179452694e-07, 2.0080247509213223e-07, 4.56667648141007e-10, 1.689868986431975e-05, 1.0665329064352136e-08, 5.156492095294285e-13, 2.262876996894647e-08, 2.652250941537204e-06, 2.194870830862783e-06, 1.7486152046330972e-06, 5.5950573063512365e-09, 8.993368716403438e-09, 1.322303688766624e-07, 3.917089319571687e-09, 4.76267741760239e-05, 6.93675694662943e-09, 2.3260547266090725e-07, 1.2355619105619553e-07, 8.430806608394903e-10, 1.4465184676737408e-06, 1.7901952489851602e-11, 2.953121656901203e-05, 2.380376784572036e-09, 7.963613825268112e-06, 1.6468467265440268e-06, 6.871193818369647e-07, 3.6371724831951724e-07, 5.262628999247454e-09, 4.3514467029659443e-10, 1.9854722268064506e-05, 9.285798228120257e-07, 7.095209042518036e-08, 6.681174227196607e-07, 1.4381521395989694e-07, 2.2317924219805718e-07, 2.796096243429247e-09, 9.84032567430404e-07, 4.9712070904206485e-08, 3.999888065209234e-08, 3.668208955787122e-05, 9.492871555494276e-08, 1.0002895578509197e-06, 6.283687525865389e-07, 3.222824318527273e-08, 4.774027729581576e-06, 2.4227233552664984e-07, 3.308418285996595e-07, 6.054655023035593e-05, 1.680537309312058e-07, 6.175703219923889e-07, 6.1692322361750485e-09, 2.363553797124496e-10, 3.282660543391103e-08, 4.208223813861878e-08, 2.3561214845813083e-07, 7.429525328284114e-10, 9.236605365003925e-06, 9.313747995065569e-08, 2.0065467065677467e-09, 1.3517368415705278e-06, 1.984416257982957e-06, 3.3590455927878793e-07, 7.055140759798517e-12, 3.072547727356323e-08, 1.5233330486807972e-05, 6.433174348785542e-07, 0.00015494957915507257, 2.8891855574642022e-09, 6.03109526764456e-08, 6.443556799240469e-07, 3.5279018595346656e-10, 8.246825746027753e-05, 1.0682907003456421e-07, 2.9468907314367243e-07, 1.3960376188393298e-09, 2.0315344499977073e-06, 4.6550921450894123e-10, 1.2106231572772685e-09, 1.3198708757045097e-06, 3.296257546026027e-06, 1.7616929426367278e-06, 2.7329928542485504e-08, 3.986207275374909e-07, 8.716544819264982e-09, 5.400755953566261e-10, 1.0775141889851625e-10, 1.1621384032878268e-07, 4.5899425458628684e-05, 1.577024875132338e-07, 1.6805979612399824e-06, 0.00017407209088560194, 4.6068146519750997e-10, 2.0848905478487723e-05, 6.425038918678183e-06, 0.0001139948217314668, 8.808736311038956e-05, 2.5202622211395465e-08, 2.8552494768518955e-07, 6.249725856832811e-07, 1.82761250577812e-09, 6.88283541272483e-10, 1.241356795134152e-08, 4.795495556209062e-07, 2.687728795081057e-07, 6.07008203701298e-08, 1.3895797565055545e-05, 1.1774577615142334e-05, 1.1257417753540722e-07, 2.7885032977792434e-05, 8.38740277231409e-07, 5.947390491201077e-07, 2.0456307758820458e-09, 1.1354920388839673e-05, 2.7468680663389478e-09, 7.046735106541746e-08, 2.0218251393089304e-06, 1.4541635096065875e-07, 1.0976838993315141e-08, 5.385144330460889e-09, 1.3697138456336688e-05, 1.1516544162759601e-07, 9.022604330333195e-10, 1.0329306121548143e-07, 1.4434383688310959e-09, 5.27599830490999e-08, 9.387642057845369e-05, 1.0646230919064692e-07, 8.722454758469667e-09, 5.2067942846178994e-08, 4.0441710780214635e-07, 1.1546427147735017e-09, 7.260253909180392e-08, 1.263747321900155e-06, 3.906874610493105e-07, 2.1599222410628727e-10, 1.6473526709415154e-10, 1.1090780844824621e-06, 6.020403359868753e-13, 0.0001374187268083915], [1.670293841016246e-07, 1.115247982852452e-08, 2.394769808233832e-06, 1.0191275805482292e-06, 1.611151128599886e-05, 1.9490934732857568e-07, 1.8972021109675552e-07, 5.1497700326308404e-08, 3.750634789412288e-07, 0.00018050741346087307, 6.415002280846238e-05, 1.5781961337779649e-06, 9.547716763336211e-05, 2.430860668312107e-08, 2.519495296837704e-07, 1.0379748971445224e-07, 1.0889654049606179e-06, 4.131775650784153e-12, 9.062151207217539e-08, 8.080959901235474e-07, 7.080891009536572e-06, 6.083432140258083e-07, 1.2179174291304662e-07, 5.196621941649937e-07, 3.762652855954229e-09, 3.82532903131505e-08, 1.089091938411002e-06, 2.540539867368352e-07, 3.8872794902999885e-06, 6.391384643755771e-10, 1.7608269287094913e-09, 1.8766086213872768e-05, 1.8505563275539316e-05, 1.1399010624302264e-08, 1.3699390422061697e-07, 3.9538991813969915e-07, 7.671112456364426e-08, 5.331106112116402e-10, 2.511189052256668e-07, 2.465961301822972e-07, 1.9608885850175284e-05, 6.896243298637472e-11, 1.5362975318566896e-05, 1.8732288289968757e-11, 7.4336649049655534e-06, 3.284931437974592e-07, 5.643912073804991e-11, 2.537862542340008e-07, 2.398902552158688e-06, 2.776298970275093e-06, 2.4931486564128136e-07, 2.538358785386663e-05, 8.107554094749503e-08, 6.521812867532617e-09, 4.284622150407813e-07, 5.104274225509187e-10, 1.617106448748018e-07, 1.6698472791176755e-06, 9.488373420296625e-10, 3.507344104036747e-07, 5.311699879939624e-08, 2.5325971364509314e-07, 4.160666833286086e-08, 1.981385508997846e-07, 8.43446457565733e-07, 2.2952049221203197e-06, 0.0001323727919952944, 9.824592528406839e-11, 6.13030959328853e-10, 5.801276969918945e-08, 2.423703620024753e-07, 2.850413522992312e-07, 4.2753161011432894e-09, 1.8274820545727266e-10, 4.223492942401208e-05, 1.630623228265904e-05, 7.481446573365247e-08, 3.132388519588858e-05, 2.6554497267738952e-08, 4.2014102064058534e-07, 1.2199006249602462e-08, 1.2928997421113309e-05, 3.412698035276662e-08, 6.721552758826022e-10, 5.313622319924605e-10, 9.507929377150504e-08, 3.309141183294173e-09, 9.898242581130035e-08, 7.640121566510061e-07, 2.179066029839305e-07, 7.315260882023722e-05, 5.438865628093481e-06, 5.1467659994974824e-11, 1.799500211063787e-07, 9.92305558611406e-07, 3.7177989575809534e-09, 2.923762565387733e-07, 2.226693140983116e-08, 8.779799100011587e-05, 1.1245985120922342e-07, 1.4018654610481462e-06, 5.117332693771459e-05, 2.6119249014300294e-05, 4.728062386050169e-10, 2.269062633786234e-06, 2.0231592401387388e-08, 0.0001612303894944489, 0.00019710692868102342, 7.471815933968173e-06, 1.4818848512732075e-06, 5.1278764345852323e-08, 0.0011065752478316426, 0.0022958223707973957, 7.695406907259894e-07, 7.469673164450796e-06, 8.75836758496007e-06, 1.7992105583175544e-09, 9.71079106015793e-10, 3.3606390843488043e-06, 0.00011187861673533916, 1.3624001926615392e-09, 2.1459984012039968e-08, 5.69910480407998e-06, 5.108661298436346e-07, 7.027185056074359e-09, 1.2523146608600655e-07, 1.356735168656087e-07, 1.3600182455775212e-06, 1.0151030949145934e-07, 1.4766915512609557e-07, 1.1826210843945972e-10, 9.63469037884579e-09, 6.078321179714408e-10, 1.151733158621937e-05, 1.6289702671201667e-06, 3.467943487933667e-09, 2.559997653861501e-07, 2.6455904844624456e-06, 5.224987035035156e-06, 4.261156050233694e-07, 3.4853203656481924e-10, 4.5008369253807246e-10, 1.8434240800502266e-09, 1.3055838280706666e-05, 0.0004115727497264743, 2.3092253798040474e-07, 2.702022250389291e-08, 1.2077592828063644e-06, 0.0015340305399149656, 7.64121637075732e-07, 4.102898572000413e-07, 1.0764089175552272e-08, 9.152310853632883e-11, 1.3796735975901697e-10, 7.059091149130836e-05, 7.911776629043743e-05, 7.570268856227358e-09, 5.244588052732979e-08, 3.7689337659685407e-06, 3.0131292305668467e-07, 9.557421520867138e-10, 1.1990388237848038e-08, 3.971638662392252e-08, 2.6323589281673776e-06, 4.987267061551393e-07, 7.4185763878631406e-06, 8.743099044750124e-08, 8.662078698762343e-07, 8.05860711494688e-09, 4.107361746719107e-05, 5.427367000265804e-07, 8.420063757341723e-09, 8.697227713128086e-06, 1.4844299300875718e-07, 1.6937760083379771e-09, 2.0765476449469134e-08, 1.2196856857826788e-08, 0.00032857907353900373, 1.8722486856859177e-05, 9.60136503636022e-07, 1.7663199969319976e-06, 1.153134365772246e-09, 1.4248810202843742e-06, 5.716946702705172e-07, 6.576977966687991e-07, 4.461855951376492e-06, 1.4888374266774917e-07, 4.378343305688759e-07, 4.2627996776900545e-07, 7.007739441178273e-06, 0.006653199903666973, 7.884049857409536e-09, 3.3125189702332136e-08, 1.435737090105249e-06, 1.8551983885117806e-05, 1.6575157246734307e-07, 2.9083517461003794e-07, 8.162639460351784e-06, 5.139026426626003e-10, 2.625394301958295e-07, 2.841352264582042e-09, 1.0940697393380105e-05, 0.02631572261452675, 8.156387565350087e-09, 3.170288032094959e-09, 6.377214134545284e-08, 6.077839898033233e-10, 2.3400021476049915e-08, 6.63467858430522e-08, 7.761676634743253e-09, 4.553049620881211e-06, 4.3621270151561475e-08, 1.5001016606674966e-07, 2.571494405856356e-06, 4.544440628251323e-07, 2.050487696303449e-10, 7.241504499688745e-05, 6.127521823273696e-10, 8.114649062918033e-06, 2.481670691167892e-08, 1.063987298266511e-08, 5.020917459575003e-09, 0.0001816603180486709, 0.00011694637942127883, 6.431126166717149e-06, 1.0061409483341777e-09, 3.463716211626888e-07, 4.7538385672396544e-08, 8.567373477319506e-09, 2.6169873379444653e-08, 2.0036287651237217e-07, 1.143229837907711e-06, 1.4361168254595214e-08, 7.932482731121127e-06, 2.6271879960404476e-07, 8.897955289244308e-11, 2.3038784036089055e-07, 4.4622307626696056e-08, 2.8146364456915762e-06, 3.232477174819337e-09, 2.9068985440972028e-08, 0.0018682677764445543, 1.817850898078177e-05, 7.099961862877535e-08, 1.8344225694022498e-08, 0.0002501052222214639, 9.114258681108822e-09, 3.544452198411818e-08, 6.407751061487943e-05, 2.624179273880145e-08, 6.5298886511300225e-06, 6.532618499477394e-07, 0.00018606414960231632, 4.0964715708469157e-07, 6.87519150233129e-06, 8.631413292436996e-12, 5.5240252549992874e-06, 2.434692980557429e-08, 2.369475851082825e-07, 4.827261368411939e-10, 9.203201756236012e-08, 3.804477843871723e-10, 2.9597828632432766e-09, 2.7381428235884187e-09, 9.684976248536259e-06, 1.418171996192541e-05, 9.104509146595774e-09, 1.2644874036737974e-10, 0.00031955877784639597, 7.110881483640696e-08, 3.934278236528144e-09, 2.4391309125348926e-05, 7.02847771094639e-08, 3.4034954854433863e-09, 5.275903092183398e-08, 3.1958113595464965e-06, 2.4020081923481484e-08, 2.6280455589500207e-09, 4.155838269070955e-06, 1.149057879956672e-05, 8.840189691383671e-11, 2.0043762560817413e-06, 9.593762939630324e-08, 2.1227981872318225e-10, 1.565579488271851e-08, 4.467260168894427e-06, 3.758605870984866e-08, 8.740537396079162e-07, 7.027214921073721e-11, 9.568339010002092e-09, 3.6361707316245884e-06, 5.831038265569077e-07, 7.788525380192368e-08, 1.6070328001660528e-07, 5.980839379304825e-09, 3.936800396786566e-07, 1.0482583689963576e-07, 3.978851168540132e-07, 6.22423002027972e-08, 4.681992322730366e-06, 0.00011975142842857167, 5.209572861986089e-08, 0.0004563705006148666, 3.1591562787980365e-07, 1.0492511304249774e-09, 2.23417426781225e-08, 8.317364859067311e-07, 3.3897400442128856e-09, 4.973327083490631e-09, 4.692479738110933e-09, 2.3377639024602104e-07, 6.536907820597904e-11, 4.0159150671570387e-07, 2.7624240672707856e-08, 5.736362709285459e-07, 4.7349595888590557e-07, 1.450947806347358e-08, 0.0020106497686356306, 1.7792497430946241e-07, 1.6211759046314e-08, 2.564813712524483e-07, 3.5488591265675495e-07, 7.601604011142626e-06, 1.2526345017249696e-05, 0.03096146322786808, 1.8496609754947713e-07, 5.578497734859411e-07, 3.033703706023516e-06, 3.877843937516445e-06, 3.4654178193704865e-07, 1.0614588319413087e-07, 2.606968640961327e-09, 6.653139280388132e-06, 6.15083450838938e-08, 2.450408764786971e-08, 9.348436265099735e-07, 4.9529688112670556e-05, 1.5542949540758855e-06, 2.7293944881989773e-11, 3.313276319971692e-09, 4.903616623863627e-09, 1.8506845833599073e-08, 2.565526756370673e-06, 3.674893150673597e-06, 3.7436304864968406e-06, 4.540305781119969e-06, 3.1987283932721766e-07, 4.062266611981613e-07, 1.2787551462167812e-08, 1.0833830401679734e-06, 3.190251845808234e-06, 4.980158701073378e-05, 1.68345025031158e-07, 8.182145393220708e-06, 0.000880353560205549, 2.9202985274423554e-07, 4.332977823651163e-06, 1.4818079989709076e-06, 3.8102837551790003e-10, 2.25328666836333e-09, 2.7088129517238713e-09, 1.4843867290892376e-07, 2.6147681797539235e-09, 2.1936730210114774e-09, 8.251208072351801e-08, 1.208422584220159e-09, 1.688700467639137e-05, 7.239881597342901e-06, 3.069663216592744e-05, 4.2756931861731573e-07, 3.194132659700699e-05, 1.1797558485682202e-09, 1.0288945162528762e-07, 1.9721394295402206e-08, 4.593348990056256e-08, 1.9630899259936996e-05, 3.377568091877947e-08, 8.802658157947008e-06, 1.1960778465436306e-05, 4.641301984520396e-06, 1.985848152585845e-09, 7.578197624980021e-08, 4.021039057278131e-09, 9.683410553407157e-07, 4.64708360681243e-09, 1.8762182918408143e-08, 3.3793693887673726e-07, 1.4079368781949597e-07, 1.617531553144147e-09, 1.6965293525572633e-06, 4.8351914301747456e-05, 3.2240063774224836e-07, 3.8189401641020027e-10, 1.3068126918369671e-06, 9.205411515722517e-06, 3.005593316629529e-05, 1.4528696468119051e-08, 7.327091156028587e-10, 4.157437061280689e-08, 6.805187149439007e-05, 2.829762593137275e-07, 2.7984662665403448e-05, 6.407333899005607e-07, 2.592677406099142e-09, 8.727130875740841e-07, 1.0239971715009233e-07, 1.9397759842831874e-07, 1.592440490583158e-08, 2.1100225566783593e-08, 4.048279933499543e-09, 9.715154014600103e-09, 5.034315790908295e-07, 3.874724541219621e-08, 2.954954254619224e-07, 6.734667823415919e-10, 3.025951400559279e-08, 3.19300230700037e-09, 2.306337898438926e-10, 2.0882841909042327e-06, 2.0743561535141453e-08, 1.4728444739375846e-06, 2.1355168655645684e-07, 1.5975010203561624e-10, 5.2443056119955145e-08, 7.82060425308373e-08, 1.1887013897649013e-06, 1.49255890846689e-06, 4.593871949509776e-08, 3.5194198062526993e-06, 7.358898528764257e-06, 0.00017842331726569682, 0.0002582166052889079, 1.0869204743357841e-05, 9.056913552463186e-10, 5.704195586986316e-07, 0.00038684537867084146, 1.2365805268643726e-08, 7.77406583551965e-08, 3.3601784252823563e-06, 1.9695581277012764e-10, 1.003557713374903e-06, 1.6088941734437867e-08, 0.0005776443285867572, 1.67722546962068e-07, 3.846028448606376e-06, 4.508946949499659e-05, 4.9095294940570966e-08, 8.341760803887155e-06, 4.352047540123749e-07, 2.489475434686028e-07, 4.302796412503085e-07, 0.00012478955613914877, 0.0036591989919543266, 5.006841092836112e-05, 5.845391393144439e-10, 6.721717363689095e-05, 4.4128087295369767e-10, 8.855309374666831e-09, 7.867622571211541e-07, 1.8075605794365401e-06, 1.2156034472354804e-06, 7.343608103838051e-06, 9.505561671119267e-09, 0.00010367635695729405, 2.481449155311566e-05, 1.084905943571357e-05, 1.4007556226047768e-09, 5.811621184292903e-10, 2.379932084295433e-06, 0.0013591068563982844, 1.4439870632543261e-08, 5.693551941021724e-08, 1.1237766273097805e-08, 7.809439206418745e-10, 0.00021274447499308735, 3.863400266368444e-09, 0.0013281942810863256, 2.1446283426485024e-05, 2.2352418527216145e-10, 8.4893838447897e-07, 2.3137157467711944e-10, 3.4673189475142863e-07, 7.675726010347717e-06, 8.606643575603812e-08, 6.9792808685131824e-12, 3.778271473464656e-09, 3.9663743223172787e-07, 2.1252836290841515e-07, 2.9143373012630036e-06, 3.8303804572592526e-11, 2.8918207828354525e-09, 3.2220356160905794e-08, 7.633650156435579e-09, 4.0182939642363635e-08, 0.0002322212530998513, 3.6517456464935094e-05, 2.5471100228990906e-10, 2.3273410079127643e-06, 3.475714765954763e-05, 1.5410089870826482e-10, 1.4299275790108368e-06, 1.9442586429718745e-11, 7.595403076265939e-06, 1.5624333173036575e-05, 2.158068584945383e-10, 3.936711845398122e-09, 2.4943348762462847e-05, 5.225314453127794e-05, 1.012752818496665e-06, 3.29212582528271e-07, 3.110382174043025e-09, 4.239774398229201e-08, 9.061477612704039e-08, 7.985028787516057e-06, 2.2996975530986674e-05, 8.739005075142359e-09, 4.590608071453062e-08, 7.078726049769557e-09, 1.61910975293722e-06, 9.179977666917694e-08, 3.5331604308908027e-09, 4.0903649911605555e-10, 1.957200197466591e-07, 1.55315007077661e-07, 2.0942108847066265e-07, 1.1312054626344548e-09, 5.366993718780577e-06, 7.941360308905132e-06, 9.5534090860383e-08, 2.3219908218408136e-09, 2.086488493091565e-08, 4.509459028412266e-09, 5.2739607819818346e-11, 8.207977657548327e-07, 4.183593205198122e-08, 8.479162829644338e-07, 1.7757489331415854e-05, 4.922599217138668e-08, 4.137054020247888e-06, 2.9326231469894992e-06, 7.352583963893267e-08, 3.971228679233718e-09, 2.1235689928289503e-05, 5.901938937569184e-10, 2.424262035560787e-08, 8.887792500900105e-05, 7.065509066705999e-07, 9.0006169557455e-06, 3.4850484098569723e-07, 7.414105773762003e-09, 2.2125277610030025e-07, 2.6383650038042106e-06, 1.3607420896732947e-06, 2.102069345255586e-07, 7.109546285022361e-09, 1.558927010592015e-06, 1.4589872421311156e-07, 8.169522658363348e-08, 9.102631537416528e-09, 1.1069618039982743e-06, 2.5007727000314617e-09, 5.106380318586901e-10, 7.221283794933697e-06, 1.1186221948733532e-09, 6.342134292935953e-05, 2.998400759679498e-06, 7.759134064144746e-07, 1.5749146768939681e-06, 2.028100425377488e-06, 3.1226540797568703e-10, 3.419498284529254e-08, 1.7353656289742503e-08, 2.2547894786839606e-06, 1.6115817516038078e-06, 1.214869911336791e-07, 6.872681410641235e-07, 4.619516769110987e-09, 5.7635944017420115e-08, 4.499282113101799e-06, 1.9571093901049608e-07, 5.366836974829425e-10, 9.997303095587995e-06, 2.370865558987134e-06, 6.15536337136291e-05, 8.100819570699969e-09, 5.817829329402002e-09, 1.0196844186793896e-06, 0.0005025867139920592, 5.231944442130043e-07, 8.826713019516319e-05, 3.0730497257991374e-08, 1.5749068325021653e-06, 8.396037753932717e-10, 6.057043755447467e-11, 3.4327456432947656e-06, 1.4574584383808542e-07, 5.5152241280886916e-12, 5.318134554954668e-08, 9.97068923425104e-07, 4.942234568261483e-07, 3.049658062082017e-06, 5.459349381453649e-07, 0.0014201432932168245, 1.8900905729424267e-07, 5.302319550537504e-05, 1.3654478436819772e-08, 2.74698539470819e-08, 1.0042701045764879e-11, 5.4193009418668225e-05, 2.0169639292078045e-09, 0.0005679525202140212, 8.520920005139487e-07, 7.082832098603831e-07, 4.209571358160247e-08, 6.967666195123456e-07, 6.4488772295590024e-06, 5.457109431006302e-09, 1.070935354618996e-06, 2.7124229529107424e-09, 1.7702964427712686e-08, 0.0003487929352559149, 4.9236692944987226e-08, 2.1264912675178493e-07, 9.855687793658419e-11, 1.3261096682981588e-05, 3.8612452044617385e-06, 7.789322609141891e-08, 5.320984655554639e-06, 3.2185598684009165e-05, 1.7400354579422128e-07, 2.076480853929752e-07, 0.00020190331269986928, 1.3283066380154196e-07, 2.611423497000942e-06, 1.1630627572145613e-09, 4.904997581434145e-07, 7.989385153450712e-09, 1.9710247323168062e-11, 4.013728016616369e-09, 2.540392927130597e-07, 1.848798456194345e-05, 2.7191377739654854e-05, 1.5503283066209406e-05, 2.8491359671534155e-07, 1.9597926836922852e-07, 2.75056422083253e-09, 6.87574175373129e-09, 0.00046398938866332173, 3.1394972666021204e-07, 9.466401138524816e-07, 0.0006325601716525853, 2.8456549117095165e-08, 9.392920885531453e-10, 1.0878246081702514e-09, 7.220101599925144e-11, 1.575526084707235e-06, 3.316588390589459e-07, 7.096483386703767e-06, 9.742184658989572e-08, 7.468152489309432e-06, 4.304531557863811e-06, 6.541712309626746e-07, 1.7651410644248244e-07, 4.209870894555934e-06, 8.16106933143601e-08, 2.028330392533917e-08, 1.4468718667615121e-08, 1.1906721653076602e-07, 2.551516445237212e-06, 3.3290200462943176e-07, 3.214848121046998e-08, 3.172557967445755e-07, 1.0992735610670934e-07, 8.685805141794845e-07, 9.782828019888612e-11, 0.0012095619458705187, 4.552004995161951e-08, 6.531086782501916e-09, 2.4361629380464933e-10, 2.3768191113049397e-06, 3.584833621061989e-06, 4.171652745466048e-10, 6.231995826055936e-07, 1.7141211401394685e-06, 4.448084608976721e-11, 1.8721937067311956e-07, 6.113183417255641e-07, 6.220776640475378e-07, 1.83474901405134e-06, 1.305022867548189e-09, 1.425832039103625e-07, 1.3769327722457092e-07, 1.549536705169885e-06, 3.171416392433457e-05, 3.817169513808949e-08, 5.150691606559121e-08, 7.783614641709846e-09, 4.748206450244652e-08, 7.19526660475367e-11, 6.157516274640784e-09, 4.95866814276269e-09, 9.006465018224219e-11, 7.3077330853266176e-09, 2.7850720130118134e-07, 4.856635769101558e-06, 2.4332222348100174e-10, 8.92804473551223e-06, 4.922647462990426e-07, 5.688315929397447e-10, 4.917018827654829e-07, 1.9675966541399248e-05, 4.321986580180237e-06, 1.2476147048801067e-06, 5.42758460397863e-09, 5.965565463839084e-08, 3.669362147462607e-09, 4.539847608953096e-09, 3.9670126739110856e-07, 6.2296354741420146e-09, 7.125941579033679e-07, 5.572761807570714e-08, 6.439709210326328e-08, 4.177270973571012e-09, 0.0017768611432984471, 4.070136583322892e-08, 1.4110266420175321e-05, 1.8035170796792954e-06, 4.380442675255836e-09, 2.1550134476910898e-07, 7.482314572371251e-07, 0.0031320196576416492, 0.00024448856129311025, 9.075095874777617e-08, 7.882739083697743e-08, 5.070384556660201e-09, 9.110048715399444e-09, 5.847898592037382e-07, 0.0036824019625782967, 1.8063392417388968e-06, 4.0600019701741985e-07, 4.45943351223832e-06, 1.550340655853688e-08, 2.0315178517194e-06, 1.9697838069987483e-05, 2.606464875043457e-07, 3.478744110907428e-05, 7.271229947036773e-07, 0.0042166635394096375, 1.9773381154664094e-08, 1.1032532256649574e-06, 3.205824228302845e-08, 3.677295353554655e-06, 1.8065145923173986e-05, 6.0068182392569724e-06, 2.4196010528498846e-08, 2.5741140774471205e-08, 1.7560657789017498e-11, 2.318432279935223e-06, 3.3799003062995325e-07, 5.10361569467932e-05, 8.357365999245303e-08, 1.6620692377955493e-08, 6.372219196748574e-09, 3.000203321334993e-07, 3.652829491329612e-06, 1.5608902685926296e-06, 0.0001462925283703953, 5.233701472207031e-08, 7.408702913380694e-07, 9.343153806185e-08, 7.128938772638094e-09, 3.628647959885711e-08, 6.977295561227947e-05, 1.395271953530397e-10, 2.9146382075850852e-05, 1.5364124195116347e-08, 1.2367006263502844e-07, 3.673716619800871e-08, 1.8223749620460694e-08, 0.0001785054337233305, 2.7357966246199794e-05, 8.339284818248416e-09, 2.3851935111451894e-06, 1.3262256418934693e-13, 3.544762421370251e-07, 2.7622547804639908e-08, 2.419019892840879e-06, 1.3261401754505187e-08, 4.691406374490725e-08, 9.764423991587279e-11, 4.4676838228951965e-07, 2.5384997570654377e-06, 1.7085523040805128e-07, 5.269331268209498e-06, 3.1238292308444215e-07, 0.0003656918997876346, 3.7702790223193006e-08, 6.573463995351858e-09, 6.12683606959763e-06, 3.389292047017989e-08, 1.7217614001197035e-08, 3.63698093863718e-09, 4.25093737987936e-09, 1.7693063680823684e-10, 2.6253646367990768e-08, 3.3980537494926466e-08, 2.6544451969812144e-08, 2.939142689228902e-07, 6.071542202334967e-08, 7.610763645971019e-07, 3.4451619335129635e-09, 3.0696995168000285e-07, 6.674894414970822e-09, 1.7130736296167015e-06, 7.912533961373924e-10, 1.501042419249643e-07, 3.2834560670380597e-07, 9.398511480185334e-08, 3.258881179135642e-06, 3.3688003497900354e-09, 5.4588930709087435e-08, 1.4523683375955443e-06, 4.443636498763226e-06, 2.356134018555167e-06, 4.5971429329938474e-08, 5.939124321230338e-07, 6.7204584119906485e-09, 5.735441277465725e-07, 3.638475476464009e-08, 7.580763394798851e-06, 1.751220679580001e-08, 8.573334042694114e-08, 6.7420864979794715e-06, 1.8237400922771485e-07, 3.150502436710667e-08, 6.396169851541345e-07, 3.1419205015481566e-07, 2.5096771327781653e-09, 7.025671777682874e-08, 1.626359136253086e-08, 3.9959133573574945e-05, 8.12063216670822e-09, 5.053349241279648e-07, 4.222264204933879e-10, 7.620215014547682e-10, 1.0923442772536873e-07, 3.5881751614397217e-07, 7.527741985313696e-08, 6.318834380181215e-07, 1.047114128738258e-08, 6.584879863602566e-10, 1.9254882133168394e-09, 9.674438842921518e-06, 9.679583357069532e-09, 0.0023106401786208153, 1.953804229515299e-09, 1.0760502846096642e-05, 7.282891800741709e-08, 1.4500449651677627e-05, 1.0102528904099017e-05, 2.2330921112256874e-08, 4.7862727114988957e-08, 2.0630182007153053e-06, 4.2355185314590926e-07, 2.4673302290167953e-10, 1.3116707009430684e-07, 1.5555366772446177e-08, 8.978753953670093e-07, 1.3889444065370848e-12, 2.5154570266749943e-06, 3.889656952082987e-08, 3.8083209801698104e-05, 1.1335119509681135e-08, 9.58705683729022e-08, 1.4368264800168618e-08, 4.67310419480782e-05, 2.746501763795095e-07, 2.4541379843867617e-06, 2.4136193488288882e-08, 0.00022108227130956948, 7.142833169382357e-07, 4.85498006241869e-08, 7.815284561729641e-07, 0.00010046755051007494, 2.9409238777589053e-06, 6.472791369560582e-07, 1.7949228947600204e-07, 6.121379791945714e-11, 0.00018444759189151227, 2.111753133471339e-11, 4.759177585356156e-08, 2.1369415126315516e-09, 3.8837453075757367e-07, 8.594504663506086e-08, 2.300245796504896e-06, 1.4089167332809183e-10, 3.949323570395791e-07, 6.464609896283946e-07, 9.782963328319738e-11, 2.256845164083643e-07, 1.776770318429044e-06, 9.575177273291047e-07, 1.8829178216606124e-08, 5.383163426131432e-08, 1.611166061676883e-11, 1.2668807336524424e-08, 7.703570190642495e-07, 2.3111637048600642e-08, 2.936490495208943e-10, 5.569059339904925e-06, 5.625415724352933e-07, 6.254149975681855e-10, 2.410957222309662e-06, 0.0007053065346553922, 2.048425784551e-08, 0.0001645202428335324, 0.0004154896305408329, 7.163364443840692e-07, 1.3254067198431585e-05, 1.692829528110451e-06, 6.623101711511481e-08, 2.5989334972109646e-05, 2.1001014829380438e-06, 2.4830424649735505e-07, 8.154069632837491e-07, 4.528775025391951e-05, 1.1269418109804974e-07, 9.823553082100034e-10, 5.053683338474002e-08, 1.1307562663986914e-09, 8.109161342417792e-09, 1.0168846875657778e-09, 3.862491666950518e-06, 8.511910891684238e-06, 0.0005857491050846875, 8.554763009094302e-10, 1.7647239758389333e-07, 1.2128718935855431e-06, 2.99979831197561e-07, 1.042962026076566e-06, 0.005597739014774561, 3.082820285271737e-06, 3.7108791040019184e-10, 1.5898271144010323e-08, 7.271181675605476e-05, 1.8355325037688885e-09, 0.0005643894546665251, 0.0008436335483565927, 8.781282367920085e-09, 4.8484460421605036e-05, 1.4415313387416973e-10, 6.008487218878145e-08, 1.0272080999129685e-06, 8.280853336373184e-09, 3.451134489296237e-07, 4.831386846149144e-09, 5.177018465474248e-06, 0.00011009211448254064, 0.00022847593936603516, 1.830160201166109e-08, 9.23526895348914e-05, 2.717826852460803e-09, 4.939544240301075e-08, 3.07686877931701e-06, 2.992236680654514e-09, 1.512354401711491e-07, 1.265652915805049e-09, 0.00025204289704561234, 1.8038494031813457e-09, 6.768308230675757e-05, 5.2743667765753344e-05, 3.971748085973559e-08, 6.757184110028902e-07, 5.347551734757872e-09, 1.1124447141241944e-08, 0.0015314725460484624, 1.3571167073678225e-05, 5.136464142907471e-09, 2.56501953010968e-09, 7.840560556360288e-08, 3.5655300933967737e-10, 1.6370611319871387e-06, 0.03348715603351593, 6.769400329176278e-08, 1.4293379990704125e-07, 2.9504363396881672e-09, 4.24061425974287e-07, 7.787440381434863e-08, 1.6669482647557743e-06, 0.00020745150686707348, 5.3095220664545195e-08, 1.327551558460982e-06, 3.528191783175316e-08, 1.9936756412430867e-10, 1.8615559156387462e-06, 4.476522917684633e-06, 8.004695928320871e-07, 1.7237502447642328e-07, 3.016203953265517e-11, 9.857864142759354e-07, 1.071773840521928e-05, 1.1104018815544237e-09, 4.317061197411931e-08, 2.9351092507567955e-06, 3.3119113007629153e-10, 5.325546936774117e-08, 2.8140303243162634e-07, 2.537726695450715e-10, 1.1582302249735221e-05, 5.035860522184521e-05, 1.9547026219868258e-09, 4.896519953945244e-07, 2.2137503492558608e-06, 5.533814828595496e-07, 5.8800122104685215e-08, 2.8209669835632667e-06, 1.101713942297522e-09, 2.9072241836125556e-10, 2.3417089778376976e-07, 7.1749042263036245e-09, 4.808891753782518e-05, 3.1892532206256874e-06, 1.7396669704794476e-07, 3.687623123527217e-11, 1.5682623200063972e-08, 4.293271558708511e-05, 0.00020258735457900912, 6.039999078666369e-08, 3.0151352348184446e-07, 9.160407353192568e-05, 8.691794391779695e-06, 9.604438133692383e-08, 2.8746138923452236e-05, 1.3074881455299447e-10, 1.5896311822416465e-08, 9.059414551870759e-09, 1.1769982847908977e-05, 0.003262122394517064, 4.1290491026302334e-08, 1.0410996083010104e-06, 1.8324168422623188e-06, 1.6587787854405178e-07, 2.025297485275246e-09, 1.5069930668687448e-05, 1.4703107176217145e-08, 0.00028950042906217277, 0.0006991780246607959, 8.441280442639254e-06, 1.1109290198874078e-06, 2.0944296920788474e-05, 7.113308697626053e-08, 3.7649954265361885e-07, 1.5010236893431284e-06, 2.730920556359706e-08, 8.061056178121362e-08, 0.0003000003343913704, 0.000132950532133691, 8.87738877963784e-08, 4.116994205105584e-06, 0.008953798562288284, 8.801271178526804e-08, 6.829034100519493e-05, 0.0010388108203187585, 3.2310808251168055e-08, 1.7721446056384593e-05, 1.3647908758684935e-07, 2.5453908425454586e-10, 1.3701393982046284e-05, 0.002123379847034812, 0.0001447840331820771, 6.31747889201506e-06, 4.95227645558316e-08, 6.271873331797906e-08, 4.196120784172308e-09, 1.1067396599173662e-06, 6.626474942095228e-07, 6.231921929611417e-07, 1.404639737856428e-09, 2.1024191028118366e-06, 0.00016211377806030214, 3.7362767670856556e-06, 2.4715225208638003e-06, 2.0723878435546794e-07, 9.77596573648043e-06, 4.536497363005765e-05, 1.794985138303673e-08, 1.580332167350207e-07, 4.744019292957091e-07, 4.39880313933827e-05, 9.112061660765391e-10, 3.840205681626685e-05, 3.4652381145861e-05, 0.00029546490986831486, 0.00011366841499693692, 3.361850353655882e-09, 1.04107780174445e-08, 0.0001315661211265251, 4.74046935039496e-09, 4.146093246504279e-08, 0.0004892504657618701, 0.0006020633154548705, 5.224486621102642e-09, 2.1045884750492405e-06, 6.818106044192973e-07, 3.0950192408951693e-10, 7.838434790130577e-09, 1.651908974054095e-07, 1.9617252267067897e-10, 9.477518148059971e-08, 1.0923469062618096e-06, 3.692505856633943e-07, 5.233839146967512e-06, 3.8217211795199546e-07, 5.699287797256147e-08, 1.2754901135281216e-08, 3.257838798731427e-08, 1.8871395468522678e-07, 4.253547203347807e-08, 1.735195875873785e-09, 4.1310693632112816e-05, 2.5911521106536384e-07, 1.0424812124654181e-08, 3.3990357906077406e-07, 2.9205311875557527e-06, 0.0010624945862218738, 3.507029759930447e-05, 1.102668978347765e-08, 1.8247700950269063e-07, 5.680784624928492e-07, 1.8816537306065584e-07, 7.112757494098787e-09, 2.776657659353532e-08, 0.003928372636437416, 1.8562289483270433e-07, 1.9639688986217152e-08, 1.2160771802882664e-05, 1.4305951481219381e-05, 3.546713145397007e-08, 6.60430714560789e-08, 1.7175678124203841e-07, 4.167012568334627e-10, 9.16953535323728e-09, 2.5725469754434016e-07, 0.0005855622584931552, 3.0334536859299988e-05, 1.1110329296570853e-06, 4.430978515301831e-05, 5.764260549767641e-06, 2.427726686349274e-09, 5.509172638085147e-07, 1.606143551491357e-09, 3.3289431211613874e-09, 3.500152345736751e-08, 1.265748323930893e-07, 4.848961054904066e-08, 6.311617539722647e-08, 8.705684506615796e-10, 2.1915946035733214e-07, 1.8472792362445034e-05, 2.8558413966983665e-10, 1.015207999444101e-06, 6.406336865438789e-07, 1.962082579742841e-10, 2.033645841947873e-06, 4.0998273220793635e-08, 1.2745119626345058e-08, 1.9083579445577925e-06, 5.154886935088143e-07, 3.3937718058041355e-07, 9.704238124186304e-08, 7.622121600547871e-09, 6.435622344724834e-05, 2.2698778412433285e-09, 1.685605077739183e-08, 3.539989279488509e-07, 3.901306389053616e-09, 4.988691216567531e-05, 3.725995867398524e-08, 8.838247467224392e-09, 3.179954788379291e-08, 6.961351027712226e-05, 3.445812168934026e-08, 1.5703610188211314e-05, 2.436791874060873e-05, 2.5475610527792014e-05, 2.6691050152294338e-06, 1.0696334356907755e-05, 5.37623520813213e-07, 2.163226264784157e-09, 1.0308350510968012e-06, 1.0363453839090653e-05, 3.440878936089575e-05, 1.0317564580475391e-08, 1.9237087656165386e-07, 6.07222818871378e-06, 1.8483466419638717e-06, 4.902585715171881e-06, 1.6522754719972e-07, 8.994113159133121e-06, 3.746308436802792e-07, 2.8040009203422755e-10, 7.868295597290853e-07, 4.5906609069668036e-10, 2.0665749111969944e-09, 2.1932338611918567e-08, 4.641731790600012e-11, 3.221289716748288e-06, 2.0083579954643938e-07, 3.928693459442911e-08, 1.1936613919516503e-09, 3.5366927022550954e-06, 3.8194326634766185e-07, 7.37941945772036e-07, 8.973778664600474e-11, 1.4054576946698205e-10, 1.4534284673572984e-06, 1.0904279562629071e-10, 7.1153118952338446e-09, 1.075064460565045e-06, 3.577120197273409e-11, 2.0679037149307078e-08, 8.458231945951411e-07, 2.8003755687677767e-06, 2.4443515940220095e-05, 4.806052402273053e-06, 2.4970438161631137e-08, 3.009359872407913e-08, 3.464772078132228e-07, 2.0022273474751273e-06, 1.5462833857782243e-07, 1.9914983795388252e-07, 4.3511805415619165e-05, 1.091995909519028e-05, 2.033371401921613e-06, 5.026430827115291e-08, 6.480122920038411e-06, 3.5050733426800207e-09, 4.256040426753316e-07, 3.5439448708984855e-08, 1.3125255549084613e-08, 6.535269494634122e-05, 4.004658826772811e-09, 1.6449462236778345e-07, 1.8160446302317723e-07, 2.637043361453095e-10, 1.410062733242512e-08, 4.95078866791232e-10, 3.223728981538443e-07, 1.5736935310428635e-08, 0.0020548300817608833, 5.446990769542026e-08, 3.253384761592315e-07, 2.887333323542407e-07, 7.787217555232928e-07, 7.739852492250066e-08, 8.155354862537934e-07, 6.400024510355706e-09, 4.445723243406974e-05, 1.2706608458756818e-06, 1.6088870324892923e-06, 0.0001350224920315668, 1.2331510106378119e-06, 2.271612231652398e-07, 5.451014430946088e-07, 2.393072236372973e-06, 1.708061603267197e-07, 1.0035864761448465e-06, 2.0388453947361995e-07, 4.686064425385439e-09, 1.5217038651371695e-07, 1.4655025779575226e-06, 4.275334219983051e-08, 9.13669282454066e-07, 1.8164031700962369e-07, 1.4965766240493394e-05, 9.123727409132698e-07, 2.070615039428958e-07, 8.754126312737753e-09, 7.414215019707626e-07, 1.367180857414496e-07, 1.9679644083225867e-06, 1.3990494380777818e-06, 4.915598310617497e-07, 6.656730420218082e-07, 2.2938660748650364e-08, 5.513451469596475e-06, 5.187119178629018e-09, 2.6973978037858615e-06, 0.0005872537149116397, 0.00012924449401907623, 1.4371819645475625e-07, 6.247629968925139e-09, 3.785503679409885e-07, 1.2263416465430055e-05, 1.8382777966508002e-07, 3.818765605956287e-07, 3.548124638541594e-08, 3.901778200088302e-06, 1.3655029107439987e-08, 1.1781385589326732e-10, 8.571314680239084e-08, 3.2701256714062765e-05, 1.923890067700995e-06, 1.5450155842700042e-05, 1.573384565745073e-06, 3.090429174790188e-07, 9.442265422876517e-07, 3.9084966374502983e-07, 1.027767780215072e-06, 1.3100387094566734e-11, 8.143979934871481e-11, 9.726157657041767e-09, 5.9895130107179284e-05, 2.0265517377993092e-05, 0.0005107350298203528, 5.912758727077971e-09, 1.5817613885360515e-10, 2.3390377918985905e-06, 1.5225286631448398e-07, 3.0246243909459736e-07, 3.360627033544006e-06, 2.5622164390171065e-09, 6.582610723171456e-08, 9.386064334648836e-07, 1.9624732772172138e-07, 5.080715936855995e-08, 4.37969947597594e-06, 9.853844318286065e-08, 5.801862457133211e-08, 9.189971024170518e-07, 2.2704369939674507e-08, 1.069227096195391e-06, 1.0444871634263109e-08, 2.105698285959079e-06, 1.4770972001088012e-08, 7.277323188503715e-10, 5.297458027797575e-09, 1.272832196264062e-05, 3.375325297039211e-10, 1.3029125001118302e-10, 2.1831390313309385e-06, 8.281796226583538e-08, 2.2576220999326324e-06, 9.008140067212622e-11, 3.767160805523417e-09, 7.353850417501917e-09, 2.495432624982641e-07, 4.209212889350056e-08, 1.2059220289017958e-10, 2.602580025268253e-06, 5.13869927090127e-05, 2.1001731511205435e-05, 9.363746444890353e-10, 0.0002401792153250426, 1.9038330947296345e-06, 4.6284264954010723e-07, 1.3971517773825326e-06, 0.00018903992895502597, 2.5664878648967715e-06, 1.3103099716360589e-09, 1.7557292425474103e-11, 1.3832998035923083e-07, 1.4025066263911867e-07, 1.456940901256587e-09, 6.382644642144442e-05, 4.359090780781116e-06, 5.645893907058053e-06, 6.670353513982263e-07, 8.640792970027178e-08, 3.85558769266936e-06, 9.604379869188051e-08, 1.477272615346692e-08, 4.925162215840828e-07, 2.823898830683902e-05, 3.3930631637701936e-09, 1.359790621791035e-05, 1.5214298709764762e-09, 1.4109942103601725e-09, 1.7642631746639381e-06, 1.2393290660384082e-07, 1.8152089751310996e-06, 4.403618580894886e-10, 1.6276304677376174e-06, 6.343180575640872e-05, 9.199218879984983e-07, 4.342255000011619e-09, 1.9628947711680667e-07, 3.0883438029150057e-08, 6.155480747338515e-08, 6.8603712861659005e-06, 1.0919986692670136e-07, 6.589974677062571e-10, 2.4023655953442358e-08, 1.4466937196810914e-11, 7.854554132791236e-05, 9.632282171878614e-08, 8.630586800784101e-10, 1.0611281631156544e-08, 1.372761282425472e-08, 6.613618097617291e-08, 1.8752603025973258e-08, 2.539459664774313e-08, 3.99596054023732e-08, 2.9337823836961263e-10, 7.257420975292916e-07, 3.6158311900180706e-08, 0.0009242547093890607, 2.298590288773994e-06, 6.53157883334643e-08, 3.3188942438755475e-07, 1.095865709288546e-08, 2.712451760089607e-06, 2.2891687478932e-07, 0.0027892354410141706, 1.1869749627635429e-08, 1.3127971598692056e-08, 3.208137089316665e-09, 7.367364940336074e-09, 3.599846240831539e-05, 2.4085838390419667e-07, 1.9501253945009012e-08, 6.599710786758806e-07, 1.1923123111046152e-06, 1.000073374807231e-11, 7.1633658080827445e-06, 9.461921581532806e-05, 2.204230398206164e-08, 7.844132596801501e-07, 8.552447252441198e-05, 3.0176950531313196e-05, 1.9828744370897766e-06, 1.1562042345758528e-07, 1.2882003019853983e-08, 1.3888426408925625e-08, 0.0012474291725084186, 7.656054457072514e-09, 3.2639988489790994e-08, 7.594032013003016e-06, 3.800586455326993e-06, 2.0767961572687454e-08, 1.0385197128925938e-06, 1.5930555719023687e-07, 1.3103402807246312e-08, 4.403253228701942e-08, 1.1943501476707752e-07, 4.183797136647627e-05, 0.0002187638747273013, 7.751773667408202e-10, 0.0022956712637096643, 1.4834239436822827e-07, 1.4945295561119565e-06, 5.708775279344991e-05, 2.02690875994449e-06, 8.436474630002522e-09, 1.137891114666445e-09, 5.124019480717834e-06, 6.711484275001567e-07, 2.6127660035513145e-08, 0.0011423933319747448, 5.254336201687693e-07, 4.21878985434887e-06, 3.1386465648708395e-10, 1.2709090697171632e-05, 5.232589361803264e-10, 4.165778832998512e-09, 5.508781431018406e-09, 3.2199716226699593e-09, 2.4568616030506973e-09, 1.1880851502610312e-07, 1.501121227320823e-09, 0.0013165103737264872, 3.5678189569399166e-11, 1.1107571481261402e-05, 4.902491923530761e-08, 1.0622702006912732e-07, 1.0096731450914831e-08, 2.459405179511265e-10, 2.1909306369138903e-08, 4.21688525875652e-07, 1.8761472801998025e-06, 0.00026514960336498916, 8.28126616170266e-08, 9.514116161568609e-10, 4.2648784503995785e-10, 1.0095890701222743e-07, 1.8525743439301579e-10, 4.923600727124722e-07, 4.206561789033003e-05, 5.167804850714219e-09, 2.2573723981622607e-05, 2.0309837509557838e-06, 2.1228945268347843e-09, 1.0972551535815e-05, 1.0793321791879862e-07, 1.668775439611636e-06, 8.273551657111966e-07, 1.3204167759184315e-09, 1.5165242984949145e-05, 4.582489444328319e-13, 0.0002318577899131924, 6.588877567992313e-07, 9.253714949863934e-08, 1.0917488907580264e-05, 9.036556036789989e-08, 7.503912797801604e-07, 1.2015784989216627e-07, 3.2072358635559794e-07, 7.718036876269707e-09, 4.774889816872019e-08, 1.9141509710607352e-08, 5.6061839198706664e-11, 3.6528128077861766e-08, 0.0001211769413203001, 1.8036999449577706e-07, 1.7918952988082992e-09, 9.371569831273519e-08, 4.425136523877882e-08, 3.3923478781616723e-07, 3.296516410955519e-07, 1.3259514375363324e-08, 3.803967047133483e-05, 4.141098575161095e-09, 8.036905637709424e-05, 3.231604495113061e-08, 7.892112741103574e-09, 1.9078891799750863e-09, 9.002516890177503e-06, 8.31070656737154e-10, 1.5111650819576994e-09, 3.819823390927013e-08, 1.1476304706548035e-07, 0.00020650069927796721, 1.2576811059261672e-05, 6.580025857516603e-09, 3.748449017848543e-08, 2.1551966256083688e-06, 6.798931053708657e-07, 2.730077830470634e-10, 4.880259751871563e-08, 4.701327105394171e-10, 2.0159211544523714e-07, 9.203604349750094e-07, 2.4268766196655633e-07, 5.604069315268134e-07, 5.668557605531532e-07, 1.2837993779157841e-07, 5.9362605497881304e-06, 3.773831025455365e-08, 9.590495153588563e-08, 6.126139595608038e-10, 2.6316197363485117e-06, 6.201891888935052e-09, 4.819399990196871e-08, 1.5341805692514754e-07, 7.796652062097564e-05, 3.456625563558191e-07, 3.1032032499211937e-09, 6.472862423834158e-08, 2.412658406392154e-10, 1.7171119637282573e-08, 4.466684004000854e-06, 8.329091542691458e-06, 4.412640919326805e-06, 2.7278033165600846e-09, 2.2643602548555464e-09, 1.9852070636261487e-06, 6.247931310099375e-07, 2.7100384158984525e-07, 7.45146344627301e-09, 1.2394609257171396e-05, 9.688204954727553e-06, 0.00017787412798497826, 8.979747434523233e-08, 3.545629123635763e-09, 4.814220346816e-07, 1.3636027688335162e-05, 6.710551819083532e-12, 1.925178139572381e-06, 1.4223456901163445e-07, 3.057771755266003e-05, 0.0002855795610230416, 3.84682685705684e-09, 5.389063773009184e-08, 2.894695967370353e-07, 1.3351493066693365e-07, 0.00015338147932197899, 9.48416678525632e-10, 3.246926280553453e-05, 5.192819116928149e-06, 2.600871266622562e-05, 3.5493744832137963e-08, 1.620477263486464e-07, 6.469747404480586e-06, 5.914604361834108e-09, 7.706801342932668e-12, 6.807491015337064e-09, 1.9978537238785066e-05, 4.3298278740167007e-08, 9.187289151668665e-08, 2.2354228690346645e-07, 9.717958704413832e-08, 1.003148895506456e-06, 2.8946533348062076e-06, 0.000307058566249907, 3.6015449040860403e-06, 2.487951314833481e-05, 3.966611620853655e-05, 6.140941621879392e-08, 7.766740600345656e-05, 6.524399941554293e-05, 8.57190607348457e-05, 1.3288549780554604e-05, 1.6511309297584376e-07, 3.129209558139223e-09, 2.0660155541918357e-07, 2.136637755612014e-09, 6.970850008336527e-10, 0.00018608864047564566, 9.217258423177555e-08, 2.1003461192492523e-09, 2.573569872765802e-06, 1.8265213341805975e-08, 7.072821972542442e-07, 0.00019757884729187936, 0.00012764862913172692, 2.6192151381110307e-07, 1.0700810889829881e-05, 7.243841637638582e-10, 0.00023169432824943215, 1.467522281473066e-07, 8.862986788926719e-09, 0.005297770723700523, 4.7379600687236234e-08, 2.394936748917331e-10, 1.662611914809986e-08, 9.55788891587872e-07, 3.254391049267724e-05, 2.1716987319564396e-08, 1.203073622946249e-07, 1.1848134029435187e-08, 8.802742377156392e-05, 5.147490433898838e-09, 8.410337315467586e-09, 7.3658474875060165e-09, 2.4661775910317374e-07, 9.084870100650733e-08, 2.286492062353318e-08, 6.341420430544531e-06, 0.0015932116657495499, 4.054076896409242e-07, 7.77359243642195e-09, 8.65237379912287e-05, 4.4428637124838133e-07, 1.711746439037043e-10, 1.243012093254947e-06, 8.120480998741186e-08, 2.398877768428065e-06, 0.00013956289330963045, 5.873612707318898e-09, 8.675368690092e-08, 4.2674130895647977e-10, 3.1545725505566224e-05, 7.2716375143500045e-06, 1.898272239486687e-05, 2.0160796339041553e-06, 2.032911200489096e-10, 9.394187117095498e-08, 1.8211980545856932e-07, 7.492820230936559e-08, 1.5963948385433469e-07, 3.602914875955321e-05, 1.0552706486421926e-09, 1.9577193199893372e-08, 2.595139676486724e-07, 7.511946478189202e-06, 7.634500320818915e-08, 2.9431280523795067e-08, 9.975356078939512e-05, 0.00010790515079861507, 5.241149736434636e-08, 4.754609562951373e-06, 2.0245599330337427e-07, 8.65297070107296e-11, 7.404366186847255e-08, 0.00018041618750430644, 1.776953517662605e-08, 5.240594305178092e-07, 2.927613529379869e-08, 2.1302421160385165e-08, 1.7301712773587496e-07, 4.827624024983379e-07, 6.0729940742021427e-05, 0.00024564919294789433, 0.0167985912412405, 1.1502648833427997e-09, 6.599677249141678e-08, 1.8184771022333734e-07, 1.308325954596512e-05, 4.7080212084438244e-07, 3.779837332018587e-09, 3.5739128634304507e-06, 2.5611578166717663e-05, 0.0016175773926079273, 1.247917147395583e-08, 0.03170733153820038, 1.9348907198679655e-10, 1.1806461088781361e-06, 8.117533980112057e-06, 2.1172958497572836e-07, 1.4976997420035332e-08, 2.7636653612717055e-05, 1.009039518606869e-09, 7.000559207881452e-07, 4.5731712861041274e-10, 9.891320651433944e-09, 2.305480428788087e-08, 6.811314179344663e-09, 1.532359164002628e-07, 5.1099689812872384e-08, 5.976134616503259e-06, 2.2718618311046157e-06, 3.0403477069285145e-08, 1.5512834750097682e-07, 1.2951929306836973e-08, 9.10067740278464e-07, 9.110093235342731e-11, 9.638608389650472e-06, 1.3360986761412619e-09, 3.0873874123926726e-08, 3.475157086541003e-07, 1.0176838038944425e-08, 1.3843783790434827e-06, 1.3134282550453236e-10, 2.1559182641794905e-05, 7.488830888746634e-09, 5.636295554722892e-07, 2.8110866878705565e-06, 6.908541649863764e-07, 4.636243888800351e-10, 1.7292280940495175e-09, 0.0006165295490063727, 2.2467122562375152e-06, 7.706534961471334e-05, 8.458274081135642e-10, 9.272601175780437e-08, 3.33927545170809e-07, 2.284984645939403e-08, 8.232025265897391e-07, 1.600184589278797e-07, 7.353755382411009e-09, 9.464835581951547e-10, 8.778546884968819e-07, 6.757922506039904e-07, 1.8912869881315686e-10, 2.4204600990174185e-08, 8.869441639469677e-12, 5.235873956621617e-09, 5.059735485701822e-06, 1.1593982662816416e-08, 2.5455702257204393e-07, 1.2184166175988054e-10, 3.644616697329184e-08, 4.900085059489356e-06, 3.981715508416528e-06, 6.568467370016151e-08, 1.0187362704527914e-06, 9.50838511926122e-05, 1.9347902835420427e-08, 9.382439020555466e-05, 3.2594411614184082e-09, 2.4923965113998747e-09, 4.128695607619193e-08, 1.0429736185812999e-08, 9.414669221996519e-08, 7.5567200497062e-08, 1.0121351756708918e-08, 9.282701284973882e-06, 8.458770142660832e-12, 5.638554648612626e-05, 1.634233399272489e-06, 0.00011430281301727518, 3.062104525497489e-08, 2.9153227387723746e-08, 4.4709773305839917e-07, 2.518518171790163e-10, 2.53696327945363e-07, 2.9372053177212365e-05, 2.9006168915657327e-05, 2.9701985315711e-10, 4.657620593206957e-06, 3.478435246506706e-07, 2.993587600030878e-08, 8.296241844618635e-07, 2.511390262416313e-10, 0.0003055839042644948, 6.523067242608649e-12, 2.7335612884371585e-08, 1.3108955032592462e-09, 0.00024426868185400963, 5.554007884711609e-07, 0.00011657729919534177, 2.168080825981633e-09, 0.0010879243491217494, 1.216677816273659e-07, 2.977573565488001e-08, 1.0723185368988197e-05, 0.0011702118208631873, 4.829992576560471e-06, 4.035569389770899e-08, 1.8163188997277757e-07, 8.066902410064358e-06, 6.400636038961238e-07, 7.955025527195403e-11, 1.1983268821680326e-09, 0.00010544951510382816, 7.34318164177239e-05, 2.0864153071897817e-08, 5.431787020171441e-09, 8.593468159290296e-10, 1.8562838377533808e-08, 0.000559103733394295, 1.7498562598916578e-08, 3.0221913220884744e-06, 8.20220975583652e-06, 9.460948717787687e-07, 7.008442935330095e-06, 8.093166692368925e-10, 0.00011305208317935467, 1.6139120595504153e-10, 1.5765820080559934e-06, 6.782400987503934e-07, 7.818235872036894e-07, 4.0519180402043276e-06, 3.675701009342447e-06, 1.6500902688676433e-07, 6.674325163658068e-07, 4.702239095877303e-07, 1.1534580153238494e-06, 3.0299745503725717e-06, 1.836026854107331e-06, 2.2149480860544912e-10, 1.4615258123740205e-06, 1.2225001455590245e-06, 7.710021510121123e-11, 7.643561548320577e-05, 6.365418698806025e-07, 1.0665908434237537e-10, 7.777602561986896e-09, 1.672649929673753e-08, 4.389509644653344e-09, 5.0395360773336506e-08, 1.4391804370461614e-07, 3.178542465320788e-05, 4.461889147933107e-06, 2.684502078409423e-06, 4.366948136436122e-09, 6.484105688286945e-05, 1.9287613728380393e-09, 3.3219468065226465e-08, 3.1039686291478574e-05, 1.2991937126116682e-07, 1.9806548152701e-06, 3.353945343675946e-09, 3.888331320922589e-06, 1.3023002232159797e-08, 2.5248168000757687e-08, 4.747008404137887e-07, 0.00013045920059084892, 4.6033657441491016e-10, 1.3105375273880782e-06, 2.834382257788093e-06, 3.8391299028717185e-08, 1.4827848644927144e-05, 3.700810680129507e-07, 2.4489350547440836e-09, 9.599335770715811e-10, 1.2030655582861982e-08, 3.1110960208025062e-06, 1.7537588048455177e-10, 8.480190558657341e-07, 2.0733664314320777e-06, 3.803178216799097e-08, 1.1307161429385815e-08, 1.5168720892688725e-06, 3.278947770013474e-05, 1.7031675625389653e-08, 1.5660290841879032e-08, 0.00010331052908441052, 2.4518483314750483e-06, 1.3102899174555205e-06, 0.00010552567255217582, 0.00019034225260838866, 7.75833886024202e-09, 5.924618380959146e-05, 1.661119739537753e-07, 1.7296930536758737e-06, 1.4893561228745966e-07, 1.4367569356465992e-08, 3.5332777770236135e-07, 4.230607530963937e-10, 5.692837476090062e-06, 1.279248235119823e-10, 9.699755310066394e-07, 6.210796876793268e-10, 1.4102514001024247e-07, 3.5904275108578076e-08, 2.196495901829465e-10, 2.3421698358561116e-07, 0.0003683109534904361, 1.3728232488574577e-06, 2.590563097371046e-09, 8.44856174353481e-07, 3.0166686926236252e-09, 8.734019019129846e-08, 3.535695070056022e-10, 3.571619444642238e-08, 1.1074814398881472e-09, 1.6630522736704734e-08, 2.888482981688867e-07, 1.2541609976324253e-05, 1.7637414202908985e-05, 0.0007915082387626171, 0.0006125670624896884, 9.580119808560994e-09, 7.108500454933164e-09, 0.003606249578297138, 1.2255449064468849e-06, 1.0430266002003918e-06, 4.016870207124157e-06, 1.3569872862717602e-05, 4.317481838711501e-08, 3.844578699840895e-08, 1.4227866813598666e-05, 2.335218169946529e-07, 5.007008585522499e-09, 6.162145837151911e-06, 2.5059588182330117e-08, 8.58941251458134e-10, 1.9807275464245322e-07, 5.0528942807659405e-08, 9.043807949637994e-06, 1.275249239540699e-09, 1.88551575774909e-06, 0.0002522338181734085, 1.2257325465725444e-07, 2.9538774981574534e-08, 2.9412567936759615e-09, 2.443499761284329e-05, 3.5398965536614924e-08, 6.95104172336869e-05, 8.527471351271743e-08, 2.739651563388179e-06, 2.128553973079761e-07, 5.728450958031317e-08, 7.54220224052915e-08, 7.187193205027143e-07, 1.1895140232809354e-06, 9.689647695321213e-11, 5.381473044963059e-08, 4.207495749142254e-06, 1.1007952593899972e-07, 6.366621674303019e-10, 3.768966294614984e-08, 9.114688737099641e-07, 5.161581952961569e-07, 4.82446296246053e-07, 0.0004454924783203751, 7.066795593146935e-09, 1.0029809800471412e-06, 4.144478538137264e-08, 1.5814574094719092e-09, 8.423080544162076e-06, 0.0002092722716042772, 8.027297553780954e-06, 1.1399167760828277e-06, 7.517888533925543e-10, 9.180351412396703e-08, 9.927421551481075e-09, 1.032014552038163e-06, 4.2595454941007915e-11, 9.141930767953e-09, 2.6721984000555032e-11, 1.3749706795351813e-06, 4.3261384234938305e-06, 4.986974033727165e-08, 1.0629199920231258e-07, 4.961707156247996e-10, 2.0033610780956224e-05, 9.945407164480002e-09, 1.015875650267617e-08, 3.012997922269278e-07, 3.0815067164624566e-10, 5.97730220874837e-08, 3.138211468467489e-06, 6.722273155901348e-07, 5.924034596205274e-08, 6.434364081542299e-07, 4.449929136285391e-08, 2.1391133486758918e-05, 8.895633385463952e-08, 4.960754813509993e-05, 4.0552885138822603e-07, 2.08017490876955e-06, 3.2280183859256795e-07, 4.2189240048173815e-06, 1.1159716206687875e-10, 0.0006473114481195807, 3.2874109479053004e-07, 1.729312248954784e-09, 3.819985749942134e-07, 4.814725840240897e-10, 5.110631207116967e-08, 1.933665743081292e-07, 1.9172509269083093e-07, 9.661771382951656e-10, 5.051066231942514e-10, 6.500206239934414e-08, 1.2593722203746438e-05, 1.5071162806634675e-06, 9.952668733603787e-06, 7.228062486319686e-07, 7.572568279101688e-07, 4.026082081054483e-07, 1.8867356565976934e-09, 4.288643040695206e-08, 0.0004223714640829712, 4.163418907410232e-06, 7.992729251782293e-07, 2.632419580095302e-07, 0.020231764763593674, 3.2135151286638575e-06, 3.7293787613634777e-07, 3.090120381799011e-10, 0.005182075779885054, 7.057093798223946e-10, 1.687234316705144e-06, 5.085337151378155e-10, 1.40267863990573e-09, 4.125396299059503e-05, 1.832811506730092e-11, 8.939627491599822e-07, 3.276145434938371e-05, 4.469956820685184e-06, 1.4335572728896295e-08, 3.976855168730253e-06, 4.2870325955846056e-08, 2.642190288781876e-08, 6.4544283304712735e-06, 7.871611160226166e-06, 1.322478571097463e-09, 7.780582222949306e-08, 2.427431809337577e-06, 1.7382225792061945e-07, 0.01562727801501751, 0.00020297693845350295, 9.745002171257511e-05, 6.729020896045768e-08, 3.6743422526797076e-08, 5.612441000835133e-08, 0.00015859540144447237, 1.5484805771848187e-05, 0.0003038378490600735, 4.855121815694474e-08, 2.2683821043756325e-07, 6.778763772530283e-09, 8.849201371674553e-09, 1.3092363304600951e-10, 3.246495516862069e-09, 7.870735134929419e-05, 3.2450589060317725e-07, 2.1070452930871397e-05, 3.4022432373603806e-06, 2.907609939484246e-07, 0.0001798207376850769, 4.018222898594104e-05, 2.009009486547697e-10, 3.622516260293196e-06, 9.319649763028792e-08, 5.304173544828927e-09, 7.301141977222869e-06, 3.4725738373708737e-07, 4.9242335903310064e-11, 6.397999907648e-09, 8.270090745554626e-08, 2.5682589388509314e-10, 2.7070589325717265e-08, 1.187475209007971e-05, 2.7494006644701585e-06, 2.4738717741001892e-08, 9.893163951346651e-05, 6.775242695766792e-07, 1.774010911503865e-07, 5.726289487029135e-07, 8.34888851386495e-08, 3.676193500723457e-07, 1.7334613744424132e-08, 4.714414814088741e-08, 5.1989194199686395e-11, 3.7770481187138216e-10, 3.8072269035183126e-06, 1.3400041076749858e-09, 5.345919635146856e-05, 1.84710177109082e-06, 2.6552372673904756e-06, 6.60526424667296e-08, 1.5041293409012724e-05, 2.9378709243133017e-08, 1.1267614752341615e-07, 1.9157459973939694e-05, 8.695933502167463e-07, 1.3614650518078975e-09, 2.6664718610208693e-12, 1.0665618388472353e-10, 0.00040790686034597456, 0.0009901556186378002, 7.270354274169222e-08, 6.370285632328887e-09, 1.3861031220585573e-05, 4.867132918917605e-09, 0.00076391821494326, 3.968738838011632e-06, 1.8121777429769281e-06, 4.5409613846914e-09, 5.330683672255532e-10, 5.664516322667623e-08, 7.635941983608063e-06, 9.126505915446614e-07, 1.1462667481865196e-09, 5.334216623964494e-08, 6.199649646987382e-07, 8.134708462392837e-10, 0.0016482877545058727, 1.6549091697015683e-06, 5.083570795250125e-05, 4.148370813528146e-10, 5.91785919823451e-07, 1.1886399079230614e-05, 1.2333680388110224e-07, 3.544746718375791e-09, 7.167004514485598e-05, 2.56017695932087e-08, 0.00014349717821460217, 8.905859516517012e-08, 7.296480180229992e-05, 1.3545531363234886e-09, 3.51940229847969e-07, 3.3295730617055597e-08, 2.153972659957315e-11, 5.169873020349769e-06, 0.0007338091963902116, 2.0076688755921168e-08, 1.6679214240866713e-06, 1.9994885747109947e-08, 1.093465300527896e-07, 2.079213710926009e-11, 3.487659938627985e-08, 4.8063806407094845e-11, 1.0166299091451947e-07, 7.313497007999104e-08, 1.322209008947084e-08, 4.329117928136839e-06, 1.2911951898786356e-06, 3.31598855041193e-08, 1.580283792712578e-09, 4.693237860919908e-05, 1.546916905681428e-08, 8.87247182390638e-08, 3.7460097246366786e-08, 2.756850108198705e-06, 3.336380541441031e-05, 1.7528745956951752e-05, 1.4070666054522007e-07, 6.5100231871895176e-09, 1.1974163882655375e-09, 1.2410809269169931e-08, 6.484056029876228e-06, 2.6178584300318164e-10, 2.265561002690042e-11, 2.4756378479651175e-05, 5.23740891367197e-06, 1.3353554706441173e-08, 1.0119587834367394e-08, 6.000698249408742e-07, 3.7073082381766653e-09, 3.2216027623377386e-09, 4.7089487864582225e-09, 8.300659004589761e-08, 8.485363800492252e-11, 0.00012473941023927182, 9.896988827431952e-13, 1.8876338003792625e-07, 2.068330360316395e-07, 0.00012497416173573583, 1.0633621067013266e-10, 8.062826054811012e-06, 7.658040885871742e-07, 3.83800013992186e-09, 4.3842337760224837e-08, 1.5785776668053586e-06, 2.2004646780260373e-06, 1.7522158657357068e-07, 8.137845952660427e-08, 7.708116100957341e-08, 9.25730728340568e-06, 5.824308004775958e-07, 5.1278948376420885e-05, 6.869321822478014e-08, 4.493697815632913e-06, 7.323115625013088e-08, 2.7107324385156062e-08, 5.196950425556679e-09, 2.071075222431773e-08, 5.31285024862882e-07, 3.119604574663981e-10, 6.879068426002277e-09, 2.4288834055141706e-08, 5.976416559860809e-06, 3.404390369610155e-08, 1.3717774471899702e-08, 3.5478802828947664e-07, 1.564987428537279e-06, 2.3627117684554833e-07, 1.9378938304726034e-05, 1.0282514040227397e-06, 2.0526097159745404e-06, 5.189708107700142e-10, 4.09514456123361e-07, 1.3180242675048248e-08, 7.586447026142196e-10, 1.8642602128693397e-07, 8.179667077001795e-08, 3.309285148134222e-07, 0.0015546197537332773, 2.608638638434968e-08, 7.048306827073247e-08, 1.4128511338640237e-07, 2.9263402367973868e-08, 1.4657131064410578e-10, 0.004476354923099279, 9.912574927550821e-11, 2.6158855082059063e-09, 2.038458859487946e-07, 4.1011336548990585e-09, 1.4821861213931697e-06, 2.373453071413678e-06, 2.116952828146168e-06, 2.808093086059671e-06, 6.77378977798071e-07, 2.9441756851156242e-05, 2.069869786680556e-08, 1.632507519389037e-06, 1.3530528519822838e-07, 1.9123083347949432e-06, 1.298416707484762e-09, 3.4858395281389676e-08, 1.269949194693254e-07, 5.018706517034843e-08, 3.13224859382899e-06, 3.426479088375345e-05, 1.3255367470321744e-08, 3.3013898814715503e-07, 5.7692668633535504e-05, 2.8001568352920003e-07, 9.969656389330339e-08, 1.2218386302720319e-08, 0.00029564305441454053, 3.0637056624982506e-05, 9.62124335757153e-09, 7.31669133529067e-05, 3.328095203869452e-07, 1.4292353966993687e-09, 1.7539025787272067e-10, 6.025015295563207e-07, 1.9144883012245373e-08, 2.1728240540141996e-08, 1.5399735275423154e-05, 3.074925416512997e-06, 2.166115700674709e-05, 1.7701689003501997e-09, 1.7950194219906734e-08, 1.6333022434533717e-10, 0.0013899048790335655, 1.367397306495377e-09, 5.743183010054054e-06, 6.946409030206269e-07, 4.778472728617089e-09, 3.8896894238860114e-07, 0.00113818165846169, 8.389703793909575e-07, 8.644006470603927e-07, 6.913762149451941e-07, 1.5752774515931378e-06, 1.0996678838637308e-06, 7.572104010478142e-08, 6.262539500312414e-06, 2.655737851853246e-08, 2.3442012775376497e-08, 1.486502583247784e-07, 2.4063799841655964e-08, 2.7365608445961698e-08, 2.9151320291020966e-07, 6.566577326339029e-07, 9.349221363663673e-05, 9.555717328524338e-11, 4.73485806651297e-06, 2.745785820934543e-08, 6.102914085204247e-07, 5.414381121227052e-08, 0.0023327749222517014, 2.728109826932723e-08, 7.907385111138865e-07, 9.933673883466554e-10, 1.0512688497499312e-08, 0.004340701270848513, 3.7389789042663324e-08, 1.4906866454111878e-05, 6.332751217996702e-05, 6.459068117692368e-06, 5.973331695940942e-08, 0.0001225938758580014, 1.954495019163005e-05, 1.3586583236246952e-06, 1.8717448424521166e-10, 3.021306440587068e-07, 4.402681952342391e-06, 1.703405594355445e-08, 3.679446649584861e-07, 9.808719392268017e-10, 7.393572332148324e-08, 6.814973829705195e-08, 5.741028508055024e-05, 9.005578249343671e-06, 5.91330899624154e-06, 5.987919848848833e-07, 2.885021466170201e-08, 8.639760434858701e-11, 2.4635461670641234e-08, 4.115005140192807e-05, 1.729880807488371e-07, 1.6428127747047938e-09, 2.1374582956923405e-06, 3.491750931061688e-06, 5.278877335257448e-09, 1.9664807950903196e-06, 1.5548421288258396e-05, 1.7731101706885966e-06, 2.897059814266356e-10, 4.668701876653358e-06, 2.771616358643314e-08, 1.57894135099923e-06, 3.560947106961976e-06, 1.5292019952539704e-06, 2.898195532452519e-07, 2.7177857191418298e-06, 4.249078483553603e-05, 5.030318206422635e-08, 1.7787266104463129e-09, 2.8046039449169946e-11, 2.5198565456463484e-09, 1.2952546590838665e-08, 9.975349968271985e-08, 5.72347280680674e-09, 1.2832594542544484e-08, 2.7112876388457607e-09, 2.341703186914401e-08, 0.00013676786329597235, 1.7199928379341145e-06, 1.2484495659492723e-09, 1.607551425308884e-08, 6.408961326087592e-07, 6.0866058859687655e-09, 1.0789476618811022e-06, 4.48141773690125e-10, 2.4639016160676874e-08, 6.025643415341619e-07, 5.467409303072657e-10, 1.7192224959217128e-06, 1.3727951227338053e-05, 5.533202056540176e-06, 2.893780788326694e-07, 4.274178593277611e-07, 1.1300588198537298e-07, 2.2839715541067562e-07, 8.429741349402775e-09, 8.796664729970871e-08, 1.2832700804210617e-06, 1.128214677237338e-07, 2.5459079289191777e-09, 1.0209893162027583e-06, 1.9050299115974667e-08, 6.707201100653037e-05, 4.83709712284508e-08, 1.7383687236360856e-06, 4.607392838806845e-06, 1.6530842913198285e-05, 1.8742341012512043e-09, 4.3248091969871894e-05, 2.2932040337764192e-06, 2.0920396082146908e-07, 1.9735546530341708e-09, 0.0005001083482056856, 1.92815559785231e-06, 2.1065479813842103e-05, 7.438671673298813e-06, 8.950319170253351e-05, 6.057416612748057e-07, 6.027908966643736e-06, 6.698493426604557e-10, 1.3572417856266838e-07, 8.447314030490816e-06, 1.0337339517718647e-06, 5.670662176271435e-06, 1.895535950424687e-09, 1.5430247003678232e-05, 4.330883207614988e-09, 3.3745186556188855e-07, 0.00011051562614738941, 0.0019745773170143366, 1.992467879574633e-08, 9.015928299049847e-06, 7.93578180946497e-07, 1.031974079523934e-06, 9.556413260725094e-07, 2.1943058481355138e-09, 1.37456606807973e-06, 8.30282260722015e-08, 8.564344788908329e-09, 1.4594080539609422e-06, 9.587963756274576e-10, 8.357437764061615e-07, 7.918761184555478e-06, 2.9686366698200573e-09, 1.2569573160092773e-09, 1.0573054964879702e-07, 8.239052107228417e-09, 0.00021809374447911978, 8.066980861087814e-09, 1.563654628000677e-08, 1.7718838307700935e-06, 6.330161795631284e-06, 2.2665444987524097e-07, 9.000833961181343e-05, 1.8730007766976087e-08, 0.00014684702910017222, 3.060704745205811e-10, 8.684055501362309e-05, 1.4301313967735041e-05, 1.6802988511699368e-07, 7.517620588259888e-07, 1.5035083151815343e-06, 3.625023187441706e-15, 6.948068342893521e-09, 7.136757851355924e-09, 2.715402729336347e-07, 8.916070441955526e-07, 6.78655567298847e-07, 6.689354449918028e-06, 7.992030504055947e-08, 9.744456974658533e-08, 1.0045083875809269e-08, 9.370383635687318e-11, 2.4933620807132684e-05, 1.9862855538121948e-07, 4.973277327735559e-07, 2.6460435037733987e-05, 3.949332860742061e-09, 4.463688128453214e-06, 2.0455892979498458e-08, 2.7071616841567447e-06, 1.5137767150008585e-05, 1.700686880212743e-05, 1.7462544121826795e-07, 2.3484615496727201e-07, 4.038959741592407e-05, 1.5089375438037678e-07, 1.4793257832934614e-05, 3.6859600527350267e-07, 0.00011312450806144625, 1.5738740444248833e-07, 1.5711283367636497e-06, 3.2502622815400173e-10, 4.295336353266066e-08, 5.4944965086178854e-05, 2.203693671987139e-09, 1.018398378960228e-07, 2.5621495296945795e-05, 2.4771620865671196e-10, 0.07912230491638184, 5.039626103098271e-06, 2.8148858177701186e-07, 0.0006306723807938397, 1.2072611070834682e-06, 9.645858334295099e-09, 2.9768793652351633e-08, 0.00017572531942278147, 1.8361948832534836e-06, 2.0915599918680527e-07, 2.917512347266893e-06, 1.4281664739712596e-10, 6.826996354902803e-07, 2.576810320675804e-09, 3.2709939201680527e-09, 4.4466693793765444e-07, 2.3368046697669342e-07, 7.2966108746186364e-06, 5.871039654437027e-09, 1.8275436275416723e-07, 2.8188301470777333e-08, 1.6761847291490994e-05, 2.0542297534120735e-06, 7.651991817958503e-10, 1.4602999272028683e-06, 7.095432920323219e-06, 5.246325258667639e-07, 6.314922416095214e-07, 0.00012729766604024917, 7.984602234500926e-06, 2.6244176609679926e-08, 7.771105003939738e-08, 1.4443971352307017e-08, 3.758696909272885e-09, 1.1830615598285021e-08, 4.869210897595622e-06, 1.3427787735054153e-06, 6.412546440515143e-07, 1.3277830224200926e-10, 1.960276385659654e-08, 3.659408321254887e-05, 1.0428087282576826e-09, 3.7331469115997606e-07, 3.033813131270158e-11, 4.3450995690363925e-07, 0.00010883176582865417, 5.024402867093158e-07, 5.3734878747491166e-05, 1.4725821529282257e-05, 9.59817612056213e-07, 1.5731111489003524e-06, 2.038882618293769e-09, 1.3687341038348677e-07, 2.0113765231144498e-07, 1.6701776985428296e-05, 4.433275535120629e-06, 3.641878834059753e-07, 6.032590249560599e-08, 1.128044213594137e-09, 4.620690674528305e-07, 1.0758125057463985e-07, 1.2676466987215917e-08, 1.5955103904730095e-08, 1.2368659429995432e-09, 5.27951488038525e-05, 4.592340285825003e-09, 1.9691308583702494e-09, 1.6941118019531132e-06, 0.00034479942405596375, 9.96366111394309e-07, 2.3143614669152157e-07, 2.428985901303804e-08, 7.341716923292552e-08, 2.738180793215861e-09, 2.332059434451139e-09, 1.731532029225491e-05, 2.2561579271496157e-08, 7.968750104225819e-10, 7.849123573322458e-10, 4.5351121116254944e-06, 1.7371490685036406e-05, 5.911871267016977e-05, 2.0981568912836757e-11, 9.720189382278477e-07, 3.4421364603076654e-07, 0.00018009266932494938, 1.072061550644321e-07, 8.061960543592761e-10, 1.3031271009822376e-05, 0.0001613837666809559, 4.109093509896411e-08, 5.1142190926611875e-08, 2.265120747324545e-05, 1.5642626749468036e-07, 2.4383675634176427e-11, 2.4474084057146683e-06, 0.00043209566501900554, 0.00021409419423434883, 2.2272681476920297e-08, 1.018403472663465e-09, 3.3880843375300174e-07, 2.0358313577162335e-06, 2.2921436539036222e-05, 5.7501216360833496e-05, 1.3895935353502864e-06, 3.4705024964942766e-10, 4.516109584074002e-07, 8.75016592516431e-09, 8.356753511407078e-09, 1.2625734598259442e-05, 3.640709067553871e-08, 1.3571451518146205e-06, 2.2204174854323355e-07, 5.6205053944324845e-09, 2.2528518456965685e-05, 3.926628835415613e-07, 2.224110460247175e-07, 1.0789369753183564e-07, 2.144708446394361e-07, 1.0200558453465902e-10, 5.4566477558637416e-08, 6.932829865746726e-09, 9.987724189364222e-12, 5.439744654722745e-06, 4.440274778971798e-07, 4.0500134446119773e-07, 8.517823668618973e-10, 9.85755150395562e-07, 4.802562330041837e-07, 8.246518518717494e-06, 0.00016428106755483896, 1.0181568477207747e-08, 1.4533892453982844e-06, 2.1569750430217027e-09, 1.139672008321213e-06, 4.4165204826640547e-07, 6.025216407579137e-06, 8.166774279061428e-08, 0.0004053348966408521, 2.8270662824070314e-06, 7.777572363920626e-09, 1.704420782289162e-08, 8.128853892230836e-07, 2.339090315217618e-06, 1.278470107557439e-09, 9.833868830355641e-09, 4.098881163372425e-06, 9.080151630591615e-10, 2.9720954586309745e-09, 4.249847336268431e-07, 1.632448345390003e-07, 3.8049358408898115e-05, 3.728491719812155e-07, 5.67713827337446e-11, 5.381898038336885e-09, 1.9628004110927577e-08, 2.7107279976235077e-08, 7.461157913724037e-09, 7.708031546371785e-09, 3.843132105885161e-07, 3.5755892895394936e-05, 1.403237769037613e-11, 0.0002896311634685844, 5.308865638653515e-06, 2.6661592933407974e-09, 1.0267073236036595e-07, 7.666358214919455e-09, 1.6777775044829468e-06, 9.635618525294376e-09, 1.0060003940992601e-09, 1.234765067970045e-10, 0.0001519896468380466, 0.00293781328946352, 9.614760099196928e-10, 6.400526899597025e-07, 2.663548137604721e-08, 1.2178062434031744e-06, 4.780431027029408e-06, 3.033152893294755e-07, 1.360594126254e-07, 3.594979887111549e-07, 6.417768076971697e-07, 6.654336630163016e-06, 6.341965672618244e-07, 5.038826600411994e-08, 1.584597564274759e-09, 2.496047500244458e-06, 2.858593870769255e-05, 5.132950731123742e-10, 1.7383119939040625e-06, 5.698954191757366e-05, 2.9761061171029723e-08, 9.56800477069919e-07, 1.0312724896266445e-08, 7.185419121924497e-07, 2.0904657503706403e-05, 1.6260176494142797e-07, 2.0554529189809045e-09, 2.6421184884384274e-07, 9.292711600217274e-10, 3.5074060633633053e-07, 1.065483593265526e-05, 2.6439892053531366e-07, 2.072772176120452e-08, 6.130753718025517e-07, 2.8432328491589942e-08, 9.686263524599781e-08, 3.287319583211712e-10, 1.8356161035626428e-08, 0.1303459107875824, 7.414520041493233e-06, 7.966071691178911e-10, 6.341415428323671e-07, 2.9760380826360233e-09, 3.6138288805886987e-08, 4.4236426788302197e-07, 7.841379101591883e-07, 1.5084681947996614e-08, 2.1640722547289215e-09, 4.632282468719495e-08, 1.8239519894436285e-09, 2.651083832461154e-06, 5.41318968316773e-07, 7.182686090345669e-08, 2.100743934363436e-08, 1.6019865825001034e-06, 1.2471773613853543e-09, 0.0008646179339848459, 1.6044947415139177e-07, 4.1322095967188943e-07, 3.1397658517562377e-07, 7.873991307860706e-06, 2.8895899006897707e-08, 2.0706745544885052e-06, 7.690131837989611e-07, 8.125023668981157e-06, 8.769663395469252e-08, 0.00018426052702125162, 2.4750789862082456e-07, 7.068982085911557e-06, 3.538839621342049e-08, 3.7165747812650807e-07, 1.2654882993956562e-05, 3.69108690279063e-08, 1.1565104429678286e-08, 1.1697211732553114e-07, 7.418314908136381e-06, 3.3934663861145964e-06, 3.1195722840493545e-05, 0.0003523711930029094, 4.171278600306749e-11, 2.0382956790854223e-05, 2.4485191829626274e-07, 7.394788781311945e-07, 2.604271003292524e-07, 3.857727914891029e-09, 1.6501150668091213e-08, 6.576925670742639e-07, 7.501387955244354e-09, 3.0447478138739825e-07, 2.1198589195137174e-07, 7.283344075403875e-06, 6.789474760182657e-09, 3.2956006634776713e-07, 1.5399223229906056e-07, 3.395359371438644e-08, 7.564215047750622e-05, 1.346081024422574e-08, 1.666158340185575e-07, 1.5149343823850359e-07, 4.3122628312630695e-07, 3.10499599436298e-05, 8.798159979050979e-05, 1.2360882983841748e-08, 3.485387424007058e-05, 6.955273761377612e-07, 2.3345014810161047e-10, 8.225994861277286e-06, 5.965414118236367e-08, 3.989744436694309e-05, 1.4583811491775123e-08, 1.1649083717202302e-05, 0.00010267322795698419, 0.00013139433576725423, 9.227155146618315e-07, 1.0098082725562563e-07, 5.107088640876611e-10, 8.815005685391952e-07, 1.8984057987836422e-06, 2.963295173685765e-06, 3.599605591375621e-08, 1.6084229059742938e-07, 0.0013757138513028622, 4.3347270661797666e-07, 0.0022382554598152637, 1.947608689079061e-05, 1.4929157714504981e-06, 9.524758297629887e-07, 3.558457137842197e-07, 4.544511284620967e-06, 1.8035304094610183e-07, 1.724397264979416e-07, 4.4590262149313276e-08, 2.889610868805903e-06, 1.4798447189878772e-11, 2.3521078418298202e-09, 3.3145984161819797e-06, 0.00013293519441504031, 4.205975585591659e-07, 2.0190487504123666e-08, 9.065188777412914e-08, 4.543925768984991e-08, 5.235552634985652e-06, 3.317052588158731e-08, 1.358610168811225e-10, 9.64737552067163e-08, 1.2731768528340126e-08, 5.708985284513801e-08, 3.6294711662776535e-06, 1.224934749188833e-06, 3.066259068873478e-08, 1.096930657418227e-09, 1.1481554906822566e-07, 1.3029447245571646e-06, 8.704531495595802e-08, 8.096612447161533e-08, 2.0116351606702665e-07, 3.802452130940992e-09, 3.889593926942325e-07, 5.4883280142803414e-08, 8.421791736212825e-11, 7.045541483563511e-09, 5.957288828906115e-11, 6.313921829814717e-09, 5.026026705934328e-09, 1.9874459411539647e-08, 1.186308118406032e-08, 1.8866353812541092e-08, 2.277806061101728e-06, 1.2561851690406911e-05, 1.4792940419283696e-05, 5.908421307765366e-09, 4.922107166294154e-08, 3.1888823741610395e-06, 0.00023987452732399106, 2.1277254802498646e-07, 0.00014289228420238942, 6.829594349255785e-05, 4.154496124897378e-08, 1.7180136424599368e-08, 6.938066832162804e-08, 1.9764044623116206e-08, 2.8581240485436865e-07, 9.327797670266591e-06, 5.597341896645958e-06, 1.778385012585204e-06, 2.2870483462611446e-06, 6.738837328157388e-07, 1.717522354738321e-05, 1.6657833839417435e-05, 2.0313223103585187e-06, 9.634393336455105e-07, 1.967082852161184e-08, 1.4214941984391771e-05, 1.3565790141001344e-05, 5.867918298463337e-05, 1.3905876983244525e-07, 4.4209882616996765e-06, 1.600650634259182e-08, 2.530065046357777e-07, 1.1736156579900126e-07, 3.4787732590413345e-12, 5.707111085939687e-06, 1.69766390172299e-05, 1.314455175815965e-06, 1.9311099386243313e-09, 1.0223825258615094e-11, 9.457108535571024e-05, 6.425833021239669e-08, 3.490304933961852e-08, 0.0003201032232027501, 1.077815738881327e-07, 5.262473905531806e-07, 6.159209533507237e-06, 8.559372588479164e-08, 2.79227206192445e-05, 7.463850693056884e-07, 8.968517306584545e-08, 6.0493784985737875e-05, 2.5175850169034675e-06, 4.4785317121487367e-10, 9.538567610434256e-06, 4.676307341355823e-08, 6.595539758791347e-08, 6.12386575085111e-05, 9.24730236828708e-10, 2.3127910253606387e-08, 1.232260819961084e-05, 1.6340277397830505e-06, 8.572024671593681e-05, 3.5830471922793095e-10, 1.858919313235674e-05, 1.4296753647613514e-07, 1.1634149643668934e-08, 1.821608819341236e-08, 4.96472523536795e-07, 1.2499373269747593e-07, 1.994849796460585e-08, 1.2151983810326783e-06, 2.5661524016840076e-09, 1.212438633046986e-06, 1.6351273113279774e-10, 1.2667102055274881e-05, 3.4650923907975084e-08, 2.8541521714942064e-06, 3.729569542088029e-08, 4.3304746100147895e-07, 3.40052878300412e-07, 1.513829950283707e-08, 2.8980564366065664e-06, 0.0010686834575608373, 6.706365951458793e-08, 2.407421484384997e-10, 0.00034646879066713154, 4.92249455419369e-06, 2.4896729122758643e-09, 1.2398805665725376e-05, 4.260132300260011e-06, 7.238052290858832e-08, 1.3352013183975942e-07, 2.735297357503441e-07, 4.4937172560821637e-07, 2.5460217329964507e-06, 6.039027084625559e-06, 1.911207391458447e-06, 3.3529188314673775e-09, 2.1322655641142774e-08, 0.0002630054368637502, 4.8849230438463564e-08, 1.7301058505836409e-06, 8.098387297650333e-06, 7.604951690609596e-08, 1.3851607860715376e-08, 4.661749652967728e-09, 1.0073863450088538e-06, 1.7850603398983367e-05, 9.896778465190437e-06, 1.4070158727008675e-07, 1.070994130714098e-06, 2.7074966055806726e-06, 8.202569733839482e-05, 3.080907845287584e-05, 7.123394851760168e-08, 7.584151830997143e-07, 4.1328594192757695e-12, 1.6518840766366338e-06, 1.6252620776979754e-11, 0.0038341970648616552, 3.9319436950790987e-07, 3.3706303526059855e-08, 0.000131826353026554, 9.452135429910413e-08, 1.1868276487803087e-05, 0.0006673773168586195, 1.5522526840283657e-12, 2.2256485543437066e-09, 1.6780349332279343e-09, 1.1768222520913696e-06, 9.247152092939359e-07, 6.114268119583599e-11, 5.868854486834607e-07, 7.04823833075352e-07, 7.794137957262137e-08, 1.7028844467859017e-07, 3.706674434056367e-08, 2.2955275369440642e-07, 2.940143986052135e-07, 6.546105737470498e-07, 7.275895086422679e-08, 3.511338491080096e-07, 2.7539028835121826e-09, 1.2357116929706535e-06, 2.8068736355635338e-05, 9.614461760065751e-08, 2.2936872401402297e-09, 4.3502911495352237e-08, 1.4295039596490255e-09, 9.93227115136941e-12, 5.940067694609752e-06, 2.704201165215636e-07, 3.0397972295759246e-05, 3.4099302865797654e-06, 1.1646039638435468e-05, 1.1368666896771629e-08, 1.047437876877666e-06, 6.75223532198288e-07, 1.812761354358372e-07, 2.8151532660558587e-07, 1.1831872370748897e-06, 4.4611856537812855e-06, 9.235223696890671e-10, 3.4857694117818028e-06, 2.5597412189881652e-08, 7.376997928076889e-07, 2.5449531676713377e-05, 9.075450946305352e-10, 1.1745268091445382e-09, 7.399786227324512e-06, 1.4196356801221555e-07, 2.128562846337445e-05, 7.259566814354912e-07, 6.781304850989045e-09, 1.445351283102525e-09, 7.924531018943526e-06, 1.62944652402075e-05, 6.852378220401079e-08, 2.3489791445285846e-08, 5.6242338359879795e-06, 1.157060225409623e-07, 2.1318122890079394e-05, 2.1005382677685702e-06, 7.860480266685954e-09, 5.441034733877359e-09, 2.3280223615529394e-07, 5.336174835335328e-10, 1.8840935922526114e-07, 3.491704319458222e-06, 1.0916790138537635e-08, 3.937219389626989e-06, 6.298321864051104e-07, 2.1720023823945667e-07, 1.1967589671257883e-05, 2.037282911260263e-07, 2.9083296340104425e-06, 8.408322038633287e-09, 5.689659883501008e-05, 6.830330967488862e-09, 1.9328862777001632e-07, 9.486446828077533e-08, 0.000599187274929136, 7.044435506031732e-07, 1.6368542787859042e-07, 0.024520881474018097, 2.2484258010990743e-08, 2.7339517316704587e-08, 2.8391175987962924e-07, 1.5148968657285877e-07, 1.9591217537140437e-09, 9.762517584022135e-05, 9.071590056919376e-07, 5.999433483339089e-07, 1.6960115090114414e-08, 9.757825864653569e-06, 1.5866055491642328e-06, 2.391347152297385e-05, 1.0087570672112633e-06, 8.599204193160404e-06, 1.771589808186036e-08, 2.7320952167286805e-09, 2.5508417820674367e-05, 4.059121039290403e-09, 4.154844646109268e-07, 1.1378556052932254e-07, 4.460877778278416e-10, 7.351159752033709e-07, 7.669741508564698e-10, 3.669828220154159e-05, 2.4979695467663987e-07, 1.9517749478836777e-06, 1.349608780287781e-08, 8.258376027470149e-08, 1.0842533468280457e-11, 5.331860961632628e-07, 0.0009515914716757834, 1.7397580904798815e-06, 2.3130492081691045e-06, 2.211094624726684e-06, 1.3815757483826019e-05, 1.0930354619631544e-05, 1.1297448310187974e-07, 2.640299271661206e-06, 9.016410729145719e-08, 1.1412536338184509e-07, 0.0005073523498140275, 8.537794542462507e-07, 2.846970881265065e-09, 7.721921429038048e-05, 7.698744042272665e-08, 3.886089228188894e-09, 4.480125781469724e-08, 1.795199295884231e-06, 1.7806037533318886e-08, 2.3785712528479053e-06, 1.7941725616310578e-07, 1.1096985872427467e-06, 3.488328914613703e-08, 2.4605708404124016e-06, 1.2800835946791267e-09, 0.0003797013487201184, 5.635257821268169e-06, 1.8016289571320954e-09, 1.7519212747174606e-08, 0.00030074428650550544, 6.859284695792667e-09, 1.3233453444172483e-08, 3.0280660212156363e-05, 1.3460054049119208e-07, 1.5344260972938173e-08, 3.0579478789150016e-06, 1.8677526192334426e-09, 6.806759756727843e-06, 7.463322404532846e-09, 8.917139666664298e-07, 2.4645705476444846e-09, 4.2314720616332124e-09, 1.4617096866231805e-08, 5.878676461179566e-07, 2.1460616039803426e-07, 5.1131338746346344e-12, 1.1847818370824825e-07, 1.7702408513287082e-05, 4.3146326333953766e-07, 1.0448514942140719e-08, 1.9733102817554027e-05, 8.772541804091816e-09, 1.5373479982372373e-06, 0.00014584585733246058, 6.986706768685735e-09, 1.0881316681032871e-10, 6.225189281394705e-05, 1.2625216641026782e-06, 0.00016520086501259357, 2.7128235160489567e-06, 6.679084663119284e-08, 1.8138736734840677e-08, 5.922605055275199e-07, 9.82887058853521e-07, 0.00022287787578534335, 4.4401718923836597e-07, 4.459297088033054e-06, 7.071349727993947e-08, 1.1520362441785892e-09, 1.3052133818192146e-09, 8.620021141325651e-09, 2.339448428756441e-06, 3.3404924124624813e-06, 9.986051452415268e-08, 5.986704820770683e-08, 0.0004376971919555217, 2.687354871966363e-08, 1.2874690469288907e-07, 1.6275905068141583e-08, 3.5615957472145965e-07, 8.230790058405546e-07, 1.1975465668001561e-06, 0.0031925630755722523, 1.8105068875229335e-06, 3.588445906643756e-05, 4.425936506180506e-09, 0.002625837689265609, 6.774991234692607e-09, 3.0392333428608254e-05, 1.375536573533509e-08, 2.278610963912797e-06, 1.723796105856934e-11, 4.8791772400136324e-08, 4.3219264278526737e-10, 0.0007095352048054338, 3.522721669924067e-07, 2.8262872930895355e-08, 1.4311872575945017e-07, 1.4122722404863453e-06, 7.469644458524272e-09, 2.4206789817071694e-07, 3.3730975701473653e-07, 5.391246418184892e-07, 1.2797877388948109e-05, 4.6483876303682337e-07, 1.8164428183808923e-05, 1.7446416222810512e-06, 3.1128666933000204e-07, 1.4383795132744126e-06, 2.3363692889688537e-05, 6.58607657300081e-09, 7.933733172649227e-07, 1.2616160915968067e-07, 5.987284890096589e-09, 1.6560212756644432e-08, 4.858177362621063e-07, 1.1573929015185058e-07, 5.957623017138758e-08, 7.229678885778412e-06, 9.229228048468485e-09, 2.32196771321469e-06, 3.4441278273789067e-08, 3.535777670649054e-09, 3.455682235653512e-05, 2.6252172347085434e-07, 2.160166978626421e-08, 6.519692874462635e-07, 2.2152526071295142e-05, 0.0007330985390581191, 3.999993623438058e-06, 1.6193965848287917e-06, 5.3427571401698515e-05, 1.1678139344439842e-08, 7.109053694875911e-05, 2.7282509584836134e-09, 6.813686013629194e-06, 4.6859383928676834e-07, 4.080527027383596e-10, 5.626525734214738e-08, 0.00022522600193042308, 3.5408671550385407e-08, 1.4217903299140744e-06, 4.246787284500897e-06, 1.8623831010700087e-07, 3.71437681678799e-06, 3.5999929082208837e-07, 0.010157561860978603, 7.301042933249846e-05, 5.986394158163932e-10, 3.412133864344469e-08, 3.0880485724082973e-07, 4.694180599784659e-09, 4.720201104646549e-06, 5.631486596491264e-11, 1.7974083675653674e-05, 2.252594697438326e-07, 3.6890726584637434e-10, 1.415036308571871e-06, 1.975747181859333e-06, 3.89198978600902e-10, 4.1267638152930886e-05, 6.61726451411937e-09, 2.7172998670721427e-05, 2.118691918440163e-05, 5.213848271523602e-05, 1.4300112205489768e-08, 1.175968655786619e-09, 6.210457286215387e-07, 1.3681418664646117e-08, 1.6807673031138393e-09, 1.7002328604576178e-05, 0.00014674833801109344, 4.6539593512306965e-08, 1.3209860982854593e-09, 5.3229065599680325e-09, 1.1200793625931738e-09, 1.734294263755487e-09, 9.959801900549792e-06, 1.21879693892879e-07, 1.1927416210255615e-07, 9.73239977497542e-09, 3.4223033296143512e-09, 1.2809961980053686e-08, 3.76095908904972e-06, 1.1291856338857542e-08, 1.542805512144696e-05, 1.0990432883772883e-06, 3.102255141129717e-05, 2.281486104038777e-06, 1.2130819868616527e-06, 0.02878021076321602, 6.847251938779664e-07, 0.00011979077680734918, 1.5946687526025016e-08, 2.4691560611245222e-05, 5.542869416785834e-07, 1.2748255229233507e-10, 2.1702308572457696e-07, 5.0185104960576155e-09, 9.594174343874329e-10, 3.3223602713405853e-07, 4.6034955403229105e-08, 1.7146095387943205e-07, 4.185379154364455e-08, 2.3809178628653171e-07, 1.0360681699239649e-05, 2.8823382569598266e-10, 6.3932117377873965e-09, 5.586987317229841e-09, 8.984534227707286e-10, 3.749460665858351e-05, 1.3224325634553225e-08, 2.3202115784215493e-08, 4.625215979103814e-07, 5.174055317525017e-08, 5.958052184951157e-08, 3.856970920423919e-08, 4.106933317871153e-07, 1.9478071067169367e-07, 9.728905752126593e-07, 6.467203661486565e-09, 1.1601609912759159e-05, 2.1220565304957972e-09, 1.6574666972246632e-08, 0.0007361271418631077, 7.129184086807072e-05, 1.6023809834564418e-08, 4.142892180425406e-07, 4.959283828043226e-08, 1.1051724868593737e-07, 0.0007910479325801134, 2.156921530271916e-09, 9.113656496140266e-10, 1.1148210177225337e-07, 2.357349330850411e-06, 7.26652737981226e-09, 1.8456437089753308e-07, 2.7327027396495396e-07, 4.975965750553257e-10, 5.967824989738801e-08, 1.5232060377456946e-06, 1.9500797066029918e-07, 0.033474065363407135, 1.3734652384300716e-06, 6.791658790916699e-08, 3.14871285809204e-05, 1.1731062343756093e-08, 3.898216505149321e-07, 1.843690733949188e-05, 4.616152082803637e-08, 1.7108318095537811e-09, 8.077357733782264e-07, 5.3047770620651136e-09, 8.76287575835022e-09, 9.980534514397732e-07, 2.1357008606059935e-08, 5.7418772485107183e-05, 4.3605368205135164e-07, 1.530094436930085e-06, 0.00015349348541349173, 5.596720242806441e-08, 5.589726015386987e-07, 4.0496363595821094e-08, 6.684508502274866e-09, 3.14819601499039e-07, 1.599987031752903e-09, 3.5662068853525852e-09, 2.6515741069488286e-07, 2.166584955887174e-08, 4.0301856074620446e-08, 1.9721958324225852e-06, 5.172170330913772e-11, 1.3364136066229548e-05, 4.676610032561257e-08, 4.080102300463295e-08, 0.005229527596384287, 1.465592873728383e-07, 1.469952167099109e-06, 5.442317529968932e-08, 9.228544513462111e-06, 8.288393473776523e-06, 1.0266022400173824e-05, 7.5193788973138e-09, 5.568952019530116e-07, 1.3891435628465842e-06, 4.1134587402957834e-10, 7.4085032792936545e-06, 1.7281074178754352e-05, 2.959363314403163e-07, 4.918452987112687e-07, 0.003511725692078471, 2.1865314892011156e-08, 3.0515763249638894e-09, 7.958978187616594e-08, 1.4411594140284478e-10, 5.7730435401026625e-06, 2.8281178856559563e-06, 4.384299245430157e-06, 0.006305650342255831, 4.861529845356927e-09, 3.827294676739257e-06, 2.7434333560449886e-07, 6.947335862150794e-08, 1.8518274373491295e-05, 2.6940543979697695e-12, 4.199038812657818e-05, 1.6532505469513126e-05, 5.992275475819042e-08, 8.734811485222593e-11, 1.7139907413366018e-06, 1.6320888107657083e-06, 5.600665645033587e-06, 4.786771228282305e-07, 1.884456324319217e-08, 2.976016958200489e-06, 1.5633886505383998e-06, 2.039644186879741e-07, 1.6056221241456115e-08, 3.7835512500805635e-08, 9.473558826300632e-09, 1.2612791806532186e-06, 1.2212456113047665e-07, 2.3103357005282987e-09, 4.733879450213863e-06, 1.3076913774057175e-07, 4.052575164337213e-08, 1.1010852318804609e-07, 3.3906968899266587e-10, 1.2435757525963709e-06, 1.1908870192200993e-06, 0.00017319689504802227, 0.0001503037492511794, 1.3225101014313623e-09, 3.5387230923333846e-07, 1.2867302778829526e-08, 1.3339630640984979e-05, 2.4888791813282296e-05, 1.7690851450424816e-08, 8.564363270124886e-06, 1.6651284795443644e-08, 1.6594486895726845e-10, 2.5956190086162678e-08, 3.8628493825854093e-07, 4.408763743413147e-06, 8.59603233038797e-08, 1.2249010069353972e-05, 1.3381042052174053e-08, 1.781963447911039e-07, 4.6074242732174753e-07, 1.9019154251509462e-06, 1.0278617423864489e-07, 2.4606466467957944e-05, 9.509046321909409e-06, 7.449802410519624e-07, 2.7086836246192902e-11, 1.0492531998806953e-07, 2.512180685698695e-09, 2.2714988290317706e-07, 4.759046934310618e-09, 4.730601688152092e-08, 3.055770321225282e-06, 1.254951431661766e-08, 6.291361387411598e-07, 0.00014118061517365277, 0.00016005961515475065, 2.8937371098436415e-05, 6.11862773988725e-12, 1.5264136621340185e-08, 1.2495144474655717e-08, 3.30960858718754e-08, 1.437202143961258e-07, 6.176217937081674e-08, 1.977119801210847e-09, 6.555168213795426e-11, 1.8936393644253258e-06, 1.3649661401160529e-09, 8.765511665842496e-06, 7.46860564504459e-07, 4.267750597364284e-08, 4.178088630624188e-08, 5.0786443495098865e-08, 1.7926607753793178e-08, 0.0001759278675308451, 2.8357648673704716e-08, 3.786786706427847e-08, 4.099340600305368e-08, 7.122042688934016e-07, 4.49080789621803e-06, 3.995772317466617e-07, 5.003963110539189e-07, 2.2124417853319756e-07, 2.2789401477041338e-09, 1.197422875520715e-06, 5.867372010470717e-08, 1.7634144811040642e-08, 8.907698578752843e-09, 6.382495598700189e-07, 0.00018867412291001529, 3.0121702366159298e-05, 8.708164855875111e-09, 1.5997857190086506e-06, 6.9678781073889695e-06, 4.840539524764154e-09, 1.6716954931439432e-08, 9.397401186106435e-07, 9.48102929498873e-09, 8.25450933916727e-06, 1.1076108086260916e-10, 1.0253556226302862e-08, 7.497393283983911e-08, 2.562353529356187e-07, 2.3346660782408435e-07, 5.7773874573285866e-08, 1.1528810262007028e-07, 8.514810523330141e-10, 2.473474758346583e-10, 2.1699715802014907e-08, 2.2245441044788095e-09, 3.2002991368074163e-09, 0.0020403291564434767, 3.018731575821221e-08, 2.4942840681774214e-08, 2.2588622528019187e-07, 6.643281722062966e-06, 9.243991883067793e-08, 1.1401836275126698e-07, 4.36617213495083e-08, 2.2030367574643606e-07, 9.245606946706175e-09, 5.504166220027251e-11, 8.872678591842487e-09, 8.733461527299369e-07, 1.7507258143201199e-10, 9.172911041854803e-11, 1.78189054622635e-08, 5.673238320014207e-06, 7.786245259921998e-05, 3.1224044505506754e-06, 8.715966032468714e-06, 5.739499556511873e-06, 6.466730724241643e-07, 1.5936806221361621e-06, 8.536140967407846e-07, 2.1654967952144943e-07, 2.1752404044672602e-10, 6.770392957378135e-08, 8.423699238147009e-11, 1.7747544234225643e-07, 1.930547568917973e-06, 6.216998826857889e-07, 1.4043071416836028e-07, 1.383933412313354e-08, 5.913294298665051e-10, 3.7129944985281327e-07, 4.0485524266387074e-08, 1.2673523563933031e-08, 2.023756717761671e-09, 2.009037598327268e-05, 1.0510469792279764e-06, 4.942401155005882e-09, 1.3436000699584838e-05, 1.7876176627851237e-07, 9.4812840245595e-08, 1.6809737601874986e-08, 2.799449794110842e-05, 1.6556472473894246e-05, 1.7329271031485405e-06, 3.471185118542053e-05, 6.936929253242852e-08, 2.422054194539669e-06, 5.332110308842175e-09, 5.794856150487249e-09, 1.7112062877799872e-08, 4.793372136191465e-05, 1.7230601088158437e-06, 2.4349542400159407e-06, 8.52597804623656e-06, 1.656714943010229e-07, 6.826894161093833e-09, 7.288682990491679e-10, 1.3511965990531394e-09, 3.006337934152725e-08, 8.140177953919192e-08, 2.9209807905772323e-09, 5.7343346071547785e-08, 2.3988857122958507e-08, 1.6238433090620674e-05, 9.76927836404684e-08, 2.2767746486351825e-05, 5.854038320762811e-08, 1.8043401439626905e-08, 1.732256649233932e-08, 2.5932418878937824e-08, 5.3626752816171575e-08, 7.046244832054072e-08, 2.9436766908474965e-06, 6.183596269693226e-06, 1.8334902051719837e-05, 4.8252783017233014e-05, 3.5611446236316624e-08, 2.635168527831411e-07, 2.2351819950472418e-07, 9.663875033538716e-09, 1.9153045371922417e-08, 4.009026724816067e-06, 1.1937645467696711e-05, 6.181169709407186e-08, 2.736965598160168e-06, 2.289168833158328e-06, 5.08073194627201e-10, 3.5669960318784888e-09, 3.8986276429398e-09, 7.988077044274178e-08, 6.398851837730035e-06, 5.598270945483819e-06, 8.647626600577496e-06, 8.071809133980423e-05, 2.1934852156846318e-07, 1.9184332344934774e-08, 2.9837674446753226e-06, 7.021682222330128e-07, 1.6020846871356298e-08, 1.1020456440746784e-06, 2.480707621543843e-07, 5.505528921645464e-09, 8.264475582109299e-06, 3.353217081780713e-08, 0.0008635450503788888, 3.5408345411269693e-07, 0.0010784383630380034, 1.187931264468034e-08, 7.533635937306826e-09, 5.410531755956072e-08, 8.25879808985519e-08, 7.567760462734441e-08, 1.244286295332131e-06, 3.088501399073351e-10, 1.3467681014844857e-07, 8.272449747437349e-08, 1.9587695909706326e-09, 2.8921695593986385e-10, 8.128146156138882e-09, 2.2822462142357836e-06, 6.194683521698607e-08, 6.460557955279e-08, 3.58641932507453e-06, 1.1471057405287866e-05, 6.369719631038606e-05, 2.2271383386396337e-06, 7.473093432963651e-07, 1.0803425993799465e-06, 1.287277626715877e-07, 2.3040533960738685e-06, 2.3157768737291917e-05, 8.549805352231488e-05, 1.7395995755009608e-08, 1.2592843745551363e-07, 5.3317627646265464e-08, 1.0242665604565104e-09, 2.6830068691197084e-06, 1.3095805115881376e-05, 1.7795237283735332e-07, 3.7136547170746326e-09, 2.8197206347613246e-07, 4.0746877516539826e-07, 3.008693072459323e-09, 6.920154760337027e-08, 2.4986366042867303e-07, 1.1794826377808931e-06, 5.934898683612744e-10, 5.6883429522258666e-08, 1.1051373149939536e-07, 4.0436702875013e-06, 5.59034106117906e-07, 5.530690714294906e-07, 1.627815618121531e-05, 0.0005706712836399674, 3.593368091969751e-07, 1.7020749965013238e-06, 2.591859882272729e-08, 2.3372481727790273e-09, 2.0462382963160053e-05, 8.623957000963856e-06, 1.3510152712115087e-06, 8.508740023671635e-08, 1.6906958535400918e-06, 8.598491945122078e-07, 3.229353387723677e-05, 2.4869919457159995e-07, 5.636238711304031e-07, 1.598429122395828e-08, 2.8899482913402608e-06, 5.488950591825414e-07, 2.431584107398521e-07, 7.439936666742142e-08, 1.8332873423787532e-06, 1.2430553397280164e-05, 1.73954759929984e-07, 1.8000143882090924e-06, 6.3160605350276455e-06, 0.06028861179947853, 7.411448876837312e-08, 6.729137247418748e-10, 8.966473119542684e-10, 1.4987134591137874e-06, 1.660950132986727e-08, 2.6712916323390345e-09, 2.2755653844797052e-05, 2.429607093290542e-07, 3.67775015774896e-07, 9.453795968283885e-09, 2.4532889710826566e-06, 0.0003906023921445012, 1.452189962947159e-06, 1.7320130041298398e-07, 1.1564869950575485e-08, 1.7168289900837408e-08, 1.133706239997423e-09, 6.527898666064402e-09, 4.792491381522268e-08, 2.3394506570184603e-05, 1.0615495149579601e-09, 8.336447621104526e-08, 3.52668990899474e-07, 1.5969224875789223e-07, 2.9697498575842474e-06, 1.2410400529461185e-07, 1.3229845535533968e-05, 1.9374809312466823e-07, 4.2156173662988294e-07, 2.0582687554338008e-08, 1.457744897015445e-10, 7.893763154243061e-08, 5.4860691307112575e-05, 4.964365416526562e-07, 1.611816514923703e-05, 2.9112377433193615e-06, 3.820425220624202e-08, 0.0004862226196564734, 2.4833104816934792e-06, 1.5966394428801323e-08, 6.629451490880456e-06, 2.180657929784502e-07, 2.8323111109784804e-06, 1.0468658729223534e-05, 1.738723085509264e-06, 5.137327025295235e-05, 1.028378733280988e-06, 1.1883717343508238e-09, 6.1053265199007e-07, 3.001496651222624e-08, 1.1126646271009122e-08, 2.2418948901758995e-06, 6.311108791123843e-07, 3.2628108215249085e-08, 4.021982302759852e-09, 4.821786081521395e-09, 4.980201424586994e-07, 2.7592907958506885e-09, 7.729732409700318e-09, 1.0067144984304832e-07, 3.036855673599348e-07, 1.7203269919718878e-08, 2.3910381059977226e-05, 2.4072553514997708e-06, 1.8144115188079013e-07, 5.701291456716717e-07, 6.642172678539282e-08, 1.8909677990119889e-10, 7.91111898212904e-10, 7.546203505626181e-06, 8.024127851058438e-07, 0.00010306834155926481, 7.173579774644168e-07, 1.4063590469959308e-06, 0.0022611215244978666, 7.352556963269308e-07, 5.322013407749182e-07, 2.4559568601034698e-08, 1.1206425369891804e-06, 2.9822365377185633e-07, 5.378332224026394e-10, 4.780688982464198e-07, 3.289419292507034e-10, 7.405225915135816e-05, 5.810271819228774e-09, 0.0001481483195675537, 1.4814066418011862e-08, 9.58872590217652e-08, 5.8961365567711255e-08, 7.975017410899454e-07, 4.187187130355596e-08, 1.1003505733242491e-06, 5.9554770004410784e-09, 1.5062590819070465e-07, 9.482315022069088e-07, 3.850680663219919e-09, 3.188976052115322e-07, 6.321934779407457e-05, 1.0394921901024645e-06, 3.030528148428857e-07, 3.951086910092272e-06, 8.458363822683168e-07, 3.9637640725231904e-07, 1.4247389934496368e-09, 7.130448921088828e-06, 2.516245785955107e-06, 9.197354415846348e-07, 0.007222848013043404, 3.6165434380563966e-07, 2.645913355081575e-08, 3.469616416396093e-08, 7.566918611701112e-06, 1.5380544937215745e-05, 2.7942617109033563e-08, 7.078491265355424e-11, 2.252958353210488e-07, 0.021487897261977196, 0.000535817991476506, 1.1953697139688302e-05, 1.1880670172104146e-05, 6.061763002662701e-08, 7.782902684994042e-06, 1.4114355517680366e-10, 4.773055906071022e-08, 1.1510892505839365e-07, 2.3705894018621887e-12, 2.0854195106889684e-11, 2.6332260816275266e-09, 8.414031071879435e-08, 4.415386278822098e-09, 8.62075580698729e-07, 3.4819894523252515e-08, 3.842711976176361e-06, 1.9049259663006524e-06, 1.0469173503224738e-05, 3.379493307420489e-08, 7.199267315627367e-08, 1.385070902415464e-08, 6.647661621173029e-07, 6.797292950722067e-09, 7.953531167004257e-10, 7.289063574944521e-09, 5.307155674927344e-07, 2.870152400191728e-07, 1.204418276756769e-05, 4.94217156088439e-09, 1.0230000953015406e-06, 6.108721208875068e-07, 1.5723907509368473e-08, 2.198303405975821e-08, 1.4554618132933683e-07, 2.8846382065239595e-06, 2.6151361964821263e-08, 3.428909423064397e-08, 3.5088569916297274e-07, 2.6435127864488095e-08, 1.114442316207942e-08, 1.9601000822433434e-09, 3.283812930021668e-06, 4.8830298027269237e-08, 1.4995955098129343e-05, 6.995444579160903e-08, 2.1070530042965885e-10, 5.037721351186519e-10, 2.0764775399584323e-05, 6.109097284934251e-06, 2.361502993153408e-09, 1.912774116874516e-08, 3.066061404766174e-09, 5.1575349061749876e-05, 8.352886879947619e-07, 5.0818982799683e-07, 2.9262411771924235e-05, 1.0908453305091825e-06, 7.178800132123797e-08, 6.434913935038367e-09, 3.966127223975491e-06, 2.641142193837709e-09, 0.002474396489560604, 1.0901739187829662e-05, 1.5911605544260965e-07, 5.192925911501334e-08, 3.6651495172179693e-09, 4.797493513564177e-09, 7.141087721151962e-11, 2.5908462703227997e-05, 2.1965328414808027e-05, 1.5476649650736363e-07, 1.0658514071337777e-08, 1.3341689975021609e-09, 1.8514220982979168e-06, 6.200285497470759e-06, 5.105973173158418e-07, 2.3842696464271285e-05, 5.209303921560604e-08, 1.975273153220769e-05, 2.711565416646522e-10, 4.869269076834826e-08, 2.9259272196213715e-05, 1.5193582214578782e-07, 8.745991181058343e-06, 2.096278137742047e-07, 8.02025557078423e-09, 1.9300439646485756e-07, 1.9929559869069635e-07, 3.158257300128753e-07, 1.231863677730871e-07, 4.108115717826877e-06, 2.61177257598888e-09, 3.8310159311549796e-07, 2.500122036508401e-06, 0.00016350465011782944, 1.299962715251013e-08, 1.3281269140019702e-10, 2.725525882851798e-05, 3.265247627837198e-08, 2.038610546151176e-05, 7.444802552214469e-09, 8.892867526810733e-07, 6.684775257781439e-07, 3.098548768321052e-05, 0.001156667829491198, 5.162085585652676e-07, 6.096050810810993e-07, 1.499837480878341e-07, 7.950759495223281e-10, 5.7808858144881015e-08, 4.889999072332785e-09, 2.3644006432732567e-06, 3.1704563298262656e-05, 1.822031117626466e-05, 4.306835035094991e-05, 1.232450586030609e-06, 1.6351552289961546e-07, 1.1805454960267525e-06, 6.274880348655643e-08, 4.84982308535109e-07, 1.1370683061784348e-08, 1.261595095058965e-08, 1.6539144780836068e-05, 5.285852111569511e-08, 4.101835600067716e-07, 3.6996183894189016e-07, 3.276075233316078e-08, 1.0289855708833784e-05, 2.7881480235691924e-08, 6.7279373183737334e-09, 1.8702335513598456e-10, 2.542062702559633e-07, 1.0415273209218867e-05, 5.10883921833738e-08, 6.802660657001525e-09, 0.0012035141699016094, 1.753961441863794e-05, 1.002342742140172e-05, 3.2542806227553456e-09, 3.253008307169125e-09, 2.0228839275659993e-05, 2.2745834371562523e-07, 2.6289473709084632e-08, 1.12429328069652e-08, 2.09120884164804e-07, 1.190986836263619e-06, 6.860171652078861e-06, 7.288154801887714e-11, 7.22420212539987e-09, 9.289659885780566e-08, 1.1266334922765964e-06, 1.937578431032705e-10, 2.642638946781517e-06, 5.883912024273741e-08, 2.6203812453218234e-09, 2.4292543457704596e-05, 8.787197316451056e-07, 8.085633274035331e-10, 1.616711301721807e-06, 1.6479156927928784e-09, 1.0334012118251579e-10, 9.696615279608523e-07, 7.614104013953238e-09, 3.3641919117144425e-07, 6.528974028086054e-10, 1.5265846968759433e-06, 2.6630023057805374e-06, 9.963146280211888e-11, 1.38383961711952e-06, 7.986821870531458e-09, 2.1342501099752553e-09, 4.087914646788704e-07, 6.095082198953605e-07, 3.6172828288272285e-08, 6.800020457831124e-08, 0.00010967038542730734, 2.0932760946834605e-07, 1.1966465535806492e-05, 1.196365883515682e-05, 1.266355837969968e-07, 3.575883056328166e-08, 1.9160738702339586e-06, 5.819873649670626e-07, 6.015705480422184e-07, 0.00013091048458591104, 1.5001431208361282e-08, 2.8441771604548194e-08, 0.00015832761710043997, 3.1116223908611573e-07, 9.011966176331043e-05, 1.8135153823095607e-06, 0.0007346207275986671, 1.1163621820742264e-05, 1.1687894385659092e-07, 1.9343147528161353e-08, 6.007433661636696e-09, 1.4258626833907329e-05, 1.4269534887034752e-08, 2.287154757141252e-06, 2.6583542478419986e-08, 2.647069550221204e-06, 7.630858167573251e-10, 1.9712424546014518e-06, 9.010761026218006e-10, 1.4522310465281407e-08, 1.5564684190394473e-06, 3.0209776014089584e-05, 2.1717418348998763e-05, 2.8545221297804346e-08, 1.2064770089637022e-06, 1.908210395251686e-10, 3.7647369026672095e-06, 0.0001822908961912617, 5.706571073460509e-07, 6.757180926797446e-06, 0.00044068103306926787, 2.4962176325971086e-08, 3.2240382097370457e-07, 7.771993841743097e-05, 0.00014217769785318524, 6.033291227725002e-11, 1.0222951232208288e-06, 1.1179428582863693e-07, 2.5438737338845385e-08, 9.37715685722651e-06, 7.85941922654132e-10, 7.039297543087741e-06, 3.6512125234366977e-07, 2.309148158019525e-06, 7.358177640526264e-07, 0.00045685691293329, 1.3286585271998774e-05, 1.781719106475066e-06, 7.670799817560692e-08, 2.3403945306199603e-05, 6.1209898376546334e-06, 6.102416350017847e-10, 7.582235639347346e-07, 1.4534929704268507e-08, 1.122829985433782e-06, 6.039762467935361e-09, 2.4917845564687013e-09, 5.184737347008195e-06, 2.641549826876144e-06, 3.888993660439155e-07, 4.535460362831145e-08, 3.05156326874112e-08, 2.0924939558852884e-09, 1.685908870285857e-07, 6.857764844880876e-08, 2.0787751964235213e-08, 3.681123672549802e-08, 0.0023898151703178883, 2.5161278927043895e-07, 3.969636054534931e-06, 4.7325666940878364e-08, 9.822185287333696e-10, 8.719389654743281e-10, 1.489066448812082e-06, 2.4196182835112268e-09, 3.7236088701320114e-07, 1.3304252144408224e-10, 3.888807896146318e-06, 1.5976023837183106e-09, 0.00016808428335934877, 1.7856331396615133e-05, 7.606057450537662e-11, 5.21042409218353e-07, 2.607480276139995e-08, 1.7213507135238615e-06, 1.649254591029603e-05, 1.9488154066493735e-05, 0.0011037191143259406, 4.628331851108669e-07, 1.8166435467037445e-08, 7.58046007831581e-05, 5.791650270481341e-09, 7.500538146132385e-08, 9.806064582562612e-08, 3.363701637226768e-07, 1.5721849422334344e-06, 8.356102476625438e-09, 2.6033862923213746e-06, 0.00019464045180939138, 5.69979238207452e-06, 1.874359156772698e-07, 0.0001856652379501611, 5.26064418693295e-08, 8.355701197615417e-08, 5.897175924474141e-06, 4.987634838471422e-06, 1.9388378404983087e-06, 1.9548902230326348e-07, 7.402450208360278e-09, 2.2918040798458605e-08, 7.301569127093899e-08, 2.6108107675781866e-08, 1.6135049962784365e-09, 4.926615559952552e-08, 1.2713239315453873e-11, 7.020710768301797e-07, 0.0011733922874554992, 1.0609078060497268e-08, 4.758582416997115e-10, 1.0738602895798977e-07, 3.939642738259863e-06, 6.143108799427921e-10, 5.232953981249011e-07, 0.0004141173558309674, 7.633638721138425e-10, 3.74430015881444e-08, 1.5983820276233018e-06, 1.0307795728081715e-09, 2.0134197598054016e-07, 4.6793342335149646e-05, 2.166203216802387e-07, 6.37794585145457e-07, 2.5882073373395542e-08, 0.0002855716447811574, 0.0002691247209440917, 2.029701295924724e-09, 2.807226395784568e-10, 3.6499759659136544e-08, 2.150452704796635e-09, 2.198723996116314e-05, 1.1662003771562013e-06, 9.613557239163129e-08, 5.7268543969257735e-06, 5.551257717395508e-11, 1.2536544025110175e-09, 4.139242282263922e-09, 1.045348199113505e-05, 0.002013543387874961, 1.231466200124487e-07, 2.299978405062575e-05, 1.2624001101357862e-05, 6.386575250871829e-07, 9.767466508492362e-06, 2.1351211216824595e-06, 1.3182867405703291e-05, 2.5736655970831634e-06, 3.7902005978196485e-09, 1.8639047993929125e-05, 2.4157191091944696e-07, 3.0347904612426646e-05, 3.5167934129276546e-06, 5.833143301714472e-09, 0.001051361788995564, 3.5855842725140974e-05, 1.4327084727483452e-06, 0.05767182633280754, 4.136198867854546e-07, 2.845541857698919e-10, 1.6102952304919427e-09, 3.969917372614873e-08, 3.504877810200924e-08, 7.556860509794205e-05, 5.417167812993284e-06, 8.093439163303628e-08, 3.202560401405208e-05, 2.9163376780161343e-07, 1.7162539167614455e-09, 4.277360243776229e-09, 5.321010121406289e-07, 8.142548608702782e-07, 1.7155066700524912e-08, 1.7676479728834238e-06, 8.700382022652775e-05, 1.042384951688291e-06, 1.1350120132647135e-07, 1.830944347602781e-05, 5.747688192059286e-08, 9.728239092510194e-06, 6.333569217531476e-06, 2.4799874154268764e-08, 6.010019859559179e-08, 3.680309191622655e-06, 1.3627560520035331e-06, 4.841020473378421e-09, 9.337687956190166e-09, 2.8647818339777587e-08, 1.193639462826468e-07, 7.162668111959647e-07, 1.54400804603938e-05, 2.8480206637482297e-08, 1.1386728147044778e-05, 1.5620045132891391e-06, 2.5339939035973202e-09, 3.4520740882726386e-05, 1.865437937453862e-08, 1.8138353041763366e-08, 8.794025916358805e-07, 1.0651234561009915e-06, 2.0954366846126504e-05, 7.459744466586926e-08, 9.671629186414066e-07, 7.466724838423033e-08, 3.86460726531368e-07, 0.019743183627724648, 1.9818309738184325e-05, 3.507996115814649e-09, 0.00034945434890687466, 5.556431688091834e-07, 1.320765079526609e-07, 1.019438666816086e-08, 8.463669018965447e-07, 7.455818376911338e-06, 9.78375828708522e-05, 1.1542484571691602e-05, 3.979116911523306e-07, 1.047568365386553e-09, 3.129870529416934e-10, 3.101202139532688e-08, 4.138451714652547e-08, 4.234922471368918e-06, 3.890405366746563e-07, 9.757859515957534e-07, 1.9995188267785124e-05, 4.2594800220285833e-07, 3.294399775199963e-08, 9.766289998935918e-09, 5.586487383801853e-10, 7.078287467265909e-08, 1.6632384358672425e-07, 2.227693446377188e-10, 1.0629615644575097e-05, 7.404696589219384e-08, 6.585325422747701e-07, 7.403258450722205e-08, 1.3778726781765727e-07, 4.904948696093925e-07, 5.0029619336200426e-11, 5.559357759921113e-06, 2.8424909032764845e-05, 9.458166459808126e-06, 2.3287277661765415e-10, 1.7350595271636848e-07, 8.427271658284496e-10, 1.0784023288579192e-05, 1.3277654886678647e-07, 2.6090876681372777e-10, 1.0765363185782917e-05, 6.388821693903424e-10, 6.158280996260146e-09, 2.1303856101440033e-06, 1.8902287592936773e-06, 2.9766755460514105e-07, 3.923336677758016e-08, 5.564166372096224e-08, 6.014766427142604e-07, 0.000290483731077984, 2.1271166872338654e-07, 3.6161189598260535e-09, 2.216821258116397e-06, 3.205009591056296e-07, 2.6564143809082452e-06, 1.3434595658168291e-09, 3.244949198233371e-09, 9.73885903476912e-07, 6.337479163676107e-08, 4.28837711297092e-06, 7.889123132542863e-09, 2.635976770193338e-08, 2.319384293514304e-06, 9.248522954419247e-12, 5.733783225991829e-09, 5.170091412765032e-07, 8.24292601464549e-06, 2.3955234951245075e-07, 0.00012930730008520186, 3.5881271287507843e-09, 2.345656184843392e-07, 3.9683840213911026e-07, 1.770355879671115e-07, 1.0646197523556111e-07, 9.304907955254293e-09, 4.2938827959915216e-07, 9.329050865147792e-10, 3.206812152711791e-06, 2.4315761493198806e-07, 2.998605586057579e-09, 1.320918272540439e-07, 6.028784582667868e-08, 7.409943769687288e-10, 1.688759653006855e-06, 3.231944045722912e-09, 3.922980340576032e-07, 7.17488868318128e-09, 3.671429160689854e-09, 3.792488598719501e-07, 5.0803400881704874e-06, 8.887666602674926e-09, 1.5061283775708034e-08, 2.578901512606535e-05, 1.1357838957337663e-05, 5.919058025938284e-07, 5.537302740776795e-07, 4.733053792643771e-12, 2.957074363152401e-09, 1.0586923560040873e-09, 2.9743851825969614e-09, 4.6758322014284204e-07, 8.507539632773842e-07, 2.0248408816314623e-07, 3.817121069005225e-06, 2.6730214131021057e-07, 0.00011698246089508757, 3.215892138541676e-05, 0.0011507034068927169, 7.630067955233244e-08, 2.8748833175162858e-11, 0.0009831821080297232, 1.962704300240148e-05, 5.377248868398965e-09, 9.657611371949315e-06, 4.093223651580047e-06, 3.434269535418366e-10, 3.1702546143819177e-10, 1.5717438373030745e-06, 6.85031636749045e-06, 6.956614015507512e-06, 1.2873402344126816e-09, 8.158679065672914e-08, 9.855899207877883e-09, 8.050106359291931e-09, 0.00011904340499313548, 6.277483066696732e-07, 8.137243275996298e-05, 2.5497145088593243e-06, 5.724393528638849e-11, 1.2936623994264096e-09, 8.751622772251721e-06, 1.683971930788175e-07, 3.8530951762538734e-09, 5.232413241174072e-05, 2.7315841180097777e-06, 0.0001729231735225767, 5.09470599041606e-09, 4.764352024011487e-09, 1.1365861141143796e-08, 0.000764900934882462, 0.0004903572844341397, 1.2877358130936045e-05, 3.761975619909208e-07, 1.4483379828789111e-08, 1.0781331184261944e-05, 2.0108272380525705e-08, 2.5861947960947873e-07, 1.308304604208388e-06, 8.79243344797942e-10, 2.7715356676338843e-09, 1.462876525692991e-06, 1.4613409093922058e-10, 3.444590765866451e-05, 4.533448727528366e-09, 1.6676832501616445e-06, 0.0036857537925243378, 1.5568488379358314e-05, 9.035344419316971e-07, 1.40501583700825e-06, 2.4547816792619415e-06, 7.293867838598089e-06, 5.47118361282628e-05, 1.428062432751176e-07, 4.226914995797415e-07, 1.2970367890829948e-08, 1.3818615141047985e-09, 1.3361000128497835e-07, 2.481049705238547e-06, 1.0257606390950968e-06, 3.7745473413508535e-10, 1.5413694200105965e-05, 5.052098117630521e-07, 1.0378358638263308e-05, 8.935625572803474e-08, 2.8501478954723325e-10, 5.1294457570350005e-09, 5.571162198236834e-09, 3.4846572294355838e-09, 5.791805506305536e-06, 3.83725939911983e-08, 1.66497102327412e-05, 1.0068650226457976e-05, 2.5895864382619038e-05, 5.670581231242977e-05, 2.57247620538692e-06, 1.673944005631256e-08, 2.420542841718998e-07, 2.4882260163394676e-07, 2.7044969042044897e-10, 2.4839619250371925e-10, 3.684304061835064e-08, 2.8395356821420137e-07, 4.9206483332397966e-09, 3.2822890716488473e-06, 1.3720581115705954e-08, 2.2027319346307195e-07, 3.0053697628318332e-05, 3.7732213513663737e-06, 8.184820821099947e-09, 9.677142998043564e-08, 3.196203397237696e-05, 1.1898886441485956e-05, 9.682770141239416e-10, 1.4733942954592294e-08, 3.5431544347375166e-06, 9.766192476945434e-08, 2.497362316944418e-09, 4.1431235331401695e-08, 0.00042993438546545804, 6.352078116833582e-07, 9.780975318562923e-08, 3.80228414087469e-07, 1.4871570641616927e-10, 6.226516234164592e-06, 0.00037856801645830274, 1.5902742234175093e-05, 2.4305030210669543e-10, 1.7798903684251854e-07, 1.6370773892049328e-07, 6.2852589799433645e-09, 1.1204946304133045e-07, 1.0698341839088243e-06, 1.266547045020161e-08, 3.11382973450236e-05, 1.2235605861832255e-09, 1.0072909390146378e-05, 6.435063551180065e-05, 1.4017753358075424e-07, 2.2802126409260381e-07, 1.6419011983348497e-10, 1.2303929430856897e-08, 0.0012704795226454735, 1.0100926317591075e-08, 3.775519644477754e-06, 9.487596980761737e-05, 7.482683486159658e-07, 3.6490190868931904e-09, 8.546877872106506e-09, 1.2115427550085656e-09, 8.615796964761557e-09, 3.4801705961484686e-09, 3.2708336220821366e-05, 4.572840771288611e-05, 1.0659987310646102e-06, 5.982646484881116e-07, 1.1980995084925894e-09, 0.0012664294335991144, 5.103492715236024e-12, 2.328400512396911e-07, 6.426287768590555e-07, 8.08293307841268e-08, 5.176544618734624e-06, 2.8464922863236097e-08, 5.0117311189978864e-08, 2.07569155463716e-05, 3.5261685127352393e-08, 1.69433744812153e-10, 2.939830601178528e-09, 4.013059218266335e-09, 6.562230225881649e-08, 2.4748249671802114e-08, 1.4953973614950655e-09, 2.7160260152037097e-10, 8.891906588814891e-08, 0.000821647874545306, 0.0014904094859957695, 2.436525505800091e-07, 1.2493697454374342e-07, 1.4748514409745894e-08, 5.819053967570653e-07, 1.5431049178005196e-05, 8.844899639370851e-07, 4.597564497998974e-07, 1.5342956203312497e-06, 3.8243058497755555e-08, 1.1319281156829675e-06, 0.00013245777518022805, 1.1823532430899064e-10, 2.2922333187125332e-07, 3.534566772600556e-08, 3.029248407671048e-09, 6.978474687002745e-08, 4.919563139083039e-07, 2.2161213308891092e-08, 3.862488640038464e-08, 2.5106214884829114e-09, 9.478967832876606e-09, 1.287296186092135e-06, 5.6633878557477146e-05, 1.0899314474954735e-05, 1.7606414985493757e-05, 1.0515227444329867e-07, 5.703637562248787e-09, 5.373480860271229e-08, 4.902697128272848e-07, 2.3896600396255963e-05, 1.1879197700181976e-05, 4.628669322825907e-11, 1.8193311916547827e-05, 1.9995610500700423e-08, 5.734198111895239e-06, 1.0919240594375879e-05, 2.448797431497951e-08, 2.1103413018863648e-05, 5.245043510626601e-09, 1.3499054096755003e-09, 1.56004347218186e-08, 1.977649844775442e-05, 2.350973815623547e-08, 2.5648216706031235e-06, 1.0117468036696664e-06, 1.309608137489704e-06, 2.1959453988529276e-07, 1.292489032467259e-10, 1.233191824212554e-07, 1.707535375317093e-05, 1.0658448346134719e-08, 4.202256604912691e-06, 3.434602376728435e-06, 1.3469195891957497e-06, 7.448620635841507e-07, 9.40771042223787e-06, 3.0295783659539666e-08, 3.1206462836053106e-07, 2.2333659217110835e-05, 2.555092635248002e-07, 8.250193417325136e-08, 2.9426844960767085e-08, 5.276440262491633e-08, 1.93161653783136e-07, 2.977225790345983e-07, 4.992465255782008e-05, 1.2479935140863319e-10, 1.282974722016661e-07, 2.5090747257650037e-08, 3.797739012156853e-09, 2.1116239423690786e-08, 1.334411470210739e-09, 2.153723471565172e-06, 7.042584115879436e-08, 3.6228243516234215e-08, 1.0696437069412923e-09, 4.168821376993037e-08, 6.364133753322676e-08, 3.8633357624107134e-10, 1.323853666690411e-06, 2.8241169047760195e-07, 2.1600304478397447e-07, 4.7478758702368395e-09, 6.413441155928012e-08, 4.070417162438389e-06, 2.053337055940574e-08, 6.813820618845057e-09, 1.9157742769948527e-07, 1.5298101061489433e-06, 1.3838014290001865e-08, 0.0006932826363481581, 2.3397037693939637e-06, 1.4114295843192792e-10, 1.634237989378562e-08, 7.30002414073283e-09, 3.57679574847225e-08, 2.0377507681246243e-08, 2.0120301087445114e-06, 6.697569165936557e-09, 3.5416580779212836e-10, 2.799148660415085e-06, 1.3530403464301344e-08, 4.571497669303426e-08, 4.098220074411074e-08, 3.2461510812709093e-08, 8.504076554061157e-09, 5.236379001871683e-05, 6.132733432195892e-08, 7.04391766248591e-07, 7.840990861041064e-07, 0.002126887906342745, 1.1383635012407467e-07, 0.000424190831836313, 7.690330676268786e-06, 2.1421330984594533e-06, 0.010969994589686394, 0.0003513698757160455, 7.138288538044435e-07, 1.3583087365987012e-06, 6.459785709012067e-06, 2.0821353530209308e-08, 2.1151417968212627e-05, 9.91052615972876e-07, 4.0490718674845994e-05, 4.4337480176182e-07, 4.3040722097487105e-09, 1.6504085920132638e-07, 4.0358045225730166e-05, 6.790868756212376e-09, 4.453165391993252e-09, 0.002424224279820919, 0.000617553130723536, 4.6093731609353483e-10, 2.492373312179552e-07, 2.0843022241479048e-07, 6.430066605389584e-06, 6.192699686380365e-08, 0.00010438905155751854, 5.063367552793352e-06, 4.0297136287559265e-12, 1.0459639376847463e-08, 6.038725519630361e-09, 3.934528791660341e-08, 6.007727093582105e-10, 4.7300642569414464e-11, 5.0395286962157115e-05, 2.1600596866733213e-09, 1.2627145906662918e-07, 6.841428898951563e-07, 2.519568109704551e-09, 8.868769185710335e-08, 6.159974645925104e-07, 2.1770742364424223e-07, 1.488574952190902e-07, 6.062201407530665e-09, 2.5028854544473234e-09, 1.3138044323568465e-06, 1.3881527820558404e-06, 1.3509326890925877e-05, 6.137686470175652e-10, 6.59387796986266e-06, 2.9022587000326894e-07, 2.2219087725261488e-07, 2.8565619913933915e-07, 6.46018975203333e-07, 7.109151738404762e-07, 6.252071638179757e-10, 0.0013160333037376404, 0.00036760157672688365, 2.1735640984843485e-06, 1.6600397145793977e-07, 3.0412540041879765e-08, 7.488996978111118e-09, 3.4866920941567514e-06, 8.247181426668249e-07, 0.0002350469003431499, 9.792413948161993e-06, 2.4687885513685615e-08, 8.821930919111765e-08, 5.703914496280049e-08, 2.296772327881058e-09, 1.9992105748656286e-09, 1.8619059005686722e-07, 1.969581608918247e-09, 2.3928883763346676e-08, 2.35459850728148e-07, 7.401187303912593e-06, 3.5369492223935595e-08, 1.6066352415133878e-10, 3.0229901426537253e-07, 3.995160824388222e-08, 2.8296109899628163e-09, 6.55334531529661e-07, 2.580866909074331e-10, 3.2140259342838817e-09, 2.987499647133518e-05, 4.2820600398840725e-09, 1.8487085640117584e-07, 6.127214646767243e-07, 2.6181186285612057e-07, 1.9203905594622483e-06, 3.596021480589684e-09, 2.930709115389618e-07, 1.224503023422585e-07, 8.340731993961015e-10, 6.370429961322088e-09, 6.981330180622081e-09, 5.4925219927781654e-09, 1.1959561163621402e-07, 3.924363909391104e-07, 2.6548102027845744e-07, 1.3361346873352886e-07, 4.4495834572444437e-07, 1.1306827474300007e-08, 1.4975123363569764e-08, 4.82080793062778e-07, 2.4845668633588502e-08, 2.37755539274076e-05, 2.3779099578291607e-08, 2.731380011056217e-09, 1.4148675120395637e-07, 8.829334774418385e-08, 7.316009487112751e-07, 3.440184173086891e-06, 1.5019237589730672e-11, 0.002359505509957671, 9.459211725015848e-08, 4.156896693530143e-07, 7.99241473004031e-09, 9.650091669755057e-05, 8.942966189806612e-08, 6.16679187714908e-08, 6.018459771439666e-06, 7.788325092406012e-06, 0.006489973980933428, 3.1654167287342716e-06, 0.0014535341178998351, 1.3866384485083927e-08, 4.0608424001220556e-07, 2.3605782644153805e-06, 1.1857738968501508e-07, 3.165936141158454e-05, 2.771850802218978e-07, 1.4163552997104034e-08, 3.4553386285551824e-06, 4.691998037742451e-05, 6.276939384930458e-11, 6.616257053337904e-08, 7.097658905053095e-09, 9.045834303833544e-08, 1.1634178918029647e-06, 3.579905387596227e-05, 7.483913577743806e-07, 4.2076833778992295e-05, 7.304860716317307e-09, 4.761582204082515e-07, 3.704677098426146e-08, 1.791267641237937e-05, 1.0266214545140429e-08, 1.5729579416756678e-08, 3.4277172211716334e-09, 1.6399162916513887e-08, 2.6640107535058632e-05, 3.493430540402187e-07, 5.154789040062724e-09, 1.4757501958229113e-06, 3.6234590083950025e-07, 2.5387321329617407e-06, 7.0227514470389e-07, 8.307517873618053e-07, 2.91653100248368e-06, 1.540776253250442e-07, 1.0659269719326403e-05, 1.692159479205202e-08, 6.884698905196274e-06, 1.1673637345666066e-07, 7.682540854148101e-06, 4.164471878453924e-09, 6.493829918241545e-10, 1.338916710835747e-08, 4.3546380390502293e-10, 2.06957331272406e-08, 1.4505410206311353e-08, 9.857943794600033e-09, 8.399149820093044e-09, 4.218042359838137e-09, 1.5369621905847453e-05, 6.2100198192638345e-06, 1.405410330335144e-07, 9.430049976799637e-05, 2.5819537086135824e-07, 9.38225093705114e-06, 3.1685624435340287e-07, 6.901814049342647e-05, 2.2793559040223954e-08, 4.9370804333648266e-08, 5.121665935803321e-08, 8.593649909016676e-06, 6.483229526565992e-07, 2.146801080016303e-06, 2.5112100843216467e-07, 0.03903934732079506, 5.382083600125043e-06, 1.8514410839998163e-06, 2.3134298317017965e-06, 6.857382231828524e-06, 3.4838328701880528e-06, 6.421223952202126e-05, 3.99105829274049e-06, 1.0598554034402241e-08, 9.226085239788517e-06, 1.3935316900415273e-08, 3.687405492058815e-09, 1.5194254046946298e-05, 5.634077737681764e-08, 2.8978746513530496e-07, 2.6592309154693794e-07, 8.854332554619759e-05, 2.890078398820606e-09, 2.1185128318279567e-08, 7.015322580627981e-07, 6.026815668747076e-08, 3.2952216511006327e-09, 1.4033817441472252e-09, 1.1817670610980713e-06, 1.601811163709499e-07, 5.341051974028233e-07, 2.7872260943695437e-08, 1.763469299476128e-07, 9.147560376732145e-06, 0.00040804556920193136, 2.8719295830370584e-09, 1.9743018242479593e-07, 4.382370661915047e-07, 3.9922470023157075e-05, 1.8699543034017552e-06, 1.8814341729012085e-07, 1.5561088275717339e-06, 4.029944433714405e-10, 1.5041695405670907e-05, 2.5734773316798965e-06, 1.2983819885903358e-07, 0.0003109215758740902, 2.4164377165902806e-08, 1.594960167494719e-06, 6.215349799276737e-07, 1.719801758781614e-07, 3.7894217541634134e-08, 5.455514838104136e-06, 3.644855439688399e-07, 8.695276483194903e-05, 4.161352649134642e-07, 1.4359530231544682e-10, 6.563809620274696e-07, 1.2297451057463604e-08, 8.114685812188327e-08, 3.166245505781262e-06, 4.697994882008061e-07, 4.18768722738605e-05, 4.663249342229392e-08, 4.535201014732593e-07, 1.2032316476506821e-08, 4.77792355013662e-07, 2.891838164487126e-07, 0.00022900395561009645, 9.853112770130679e-10, 1.1989499393294523e-09, 0.0012812025379389524, 3.676040449818174e-08, 3.3200981874870195e-07, 7.214039214886725e-07, 1.7825042277763714e-06, 0.00029611046193167567, 2.7164878702024e-07, 7.815871754246473e-07, 1.1494523555199976e-08, 4.2004605347756296e-05, 6.148619286250323e-05, 1.2617361599964738e-09, 5.169848993347159e-09, 4.522224816838616e-09, 4.7477934117523546e-07, 7.771125964950443e-09, 8.444468235779823e-09, 1.6122829293863106e-08, 1.4822364846622804e-06, 1.1950773114222102e-05, 7.614094954533357e-08, 4.0333816286874935e-09, 1.1298777913282265e-08, 9.273457180825062e-06, 5.629658161865336e-08, 2.6489811716601253e-05, 9.049494110513479e-06, 6.492047077699681e-08, 0.00011911665933439508, 6.3392411320251085e-09, 2.0500099708442576e-05, 0.0006059604347683489, 6.619377757033362e-08, 6.037835786898427e-10, 0.0004827812372241169, 1.747219613434936e-07, 3.278427485042812e-08, 1.9220574021261427e-09, 1.0168685804501365e-07, 2.9298925952048194e-08, 6.125916115706787e-05, 6.620238224286368e-08, 2.504044616102874e-08, 4.6096384487270825e-09, 7.66288110298774e-07, 2.6479667347700797e-09, 0.00038406506064347923, 2.743425397966348e-08, 1.2799586102119065e-07, 5.202395527703629e-07, 4.9042256478060153e-08, 3.8427406252594665e-06, 1.172156816053871e-09, 2.6123095722141443e-06, 3.39627304057899e-09, 5.070538477980335e-08, 9.662925570808056e-09, 1.319566766966318e-08, 5.320968284650007e-06, 2.764504870356177e-06, 1.9086240854448988e-07, 4.6209949755571245e-10, 6.23309528990923e-11, 0.001967532094568014, 5.396422693593195e-06, 6.780995249755506e-07, 1.0807183343786164e-06, 0.0006950037786737084, 2.578146904852474e-07, 3.124504291918129e-05, 2.2703606106233565e-08, 2.782901674436289e-06, 0.0010356701677665114, 2.817752697836795e-10, 4.6723172886231623e-07, 9.627336112316698e-06, 2.531063785227161e-07, 2.639970659856772e-07, 1.7058200185715577e-08, 6.227246274193021e-09, 1.1920060671855026e-07, 3.0235578378778882e-06, 1.034182606218792e-08, 1.243262886418961e-05, 5.960459475318203e-06, 6.826506364632223e-07, 1.0408451345256253e-07, 2.47664206654008e-06, 5.15260811795315e-09, 2.1027150864938449e-07, 5.224229937539349e-10, 3.038140334865602e-07, 5.161248282092856e-06, 0.0018062820890918374, 2.9226622899614085e-08, 4.7029398047016e-06, 1.11810898317799e-07, 1.8166192161661598e-10, 5.145968651198984e-10, 8.404927029914688e-07, 1.8472507967715046e-09, 5.988074917695485e-06, 1.58125050830904e-08, 6.710380694130436e-05, 1.1385830021026777e-06, 4.2951504042321176e-07, 5.925673818296673e-08, 7.38457905935519e-10, 8.213566502490721e-07, 0.00020667466742452234, 2.6873129172599874e-05, 4.006190010841237e-06, 1.1083055824201438e-06, 3.8412332514781156e-07, 5.541268706110714e-07, 3.679521753952031e-08, 8.658348482626366e-10, 3.269585713283618e-10, 1.2104490743070073e-08, 1.5534460828803276e-07, 4.713657242305658e-10, 2.003448025789112e-05, 1.7584012024229878e-10, 2.40223729797151e-09, 1.1950344287470216e-06, 2.0370525533053296e-07, 1.2550184180781798e-07, 8.017948260885532e-08, 2.6553756526936922e-08, 0.0003352602943778038, 5.653288826579228e-05, 0.009269271977245808, 9.133237499625579e-10, 2.3205753677757457e-05, 1.2499729109549662e-06, 1.5103312378528244e-08, 1.5709822491771774e-06, 1.548353800728819e-08, 3.6031772010503005e-10, 2.984162108532473e-07, 3.1341610196022884e-08, 9.715536180010531e-07, 3.6712179962705704e-08, 5.657965175487334e-06, 4.553643151439246e-08, 0.00026549739413894713, 0.00020741620392072946, 2.02587475683913e-08, 1.0440900366504025e-09, 5.0675968310542885e-08, 1.7090931336838366e-08, 1.0967178809551115e-07, 5.852178830423327e-08, 1.3412826405101441e-08, 6.075991265674929e-09, 1.3831219503046555e-09, 2.0065278931724606e-06, 5.098501674183353e-07, 5.527951794231933e-12, 1.2103390645279433e-06, 0.00013923989899922162, 3.355013822314845e-09, 3.424987369271548e-07, 1.2643201898754342e-06, 6.658168461015634e-10, 6.075287473095159e-08, 3.391743330638519e-09, 2.2234932117726203e-08, 4.241871465637814e-06, 2.1618353329699858e-08, 1.8761494580132876e-09, 5.266414526694518e-10, 6.740887847911381e-10, 2.0245289888976004e-09, 1.2149932445026934e-05, 2.8966960030629707e-07, 9.296816472215141e-08, 1.2645097413610529e-08, 9.175697641694569e-07, 4.052590441006032e-07, 2.743514393444002e-08, 1.7559904919028924e-10, 1.4395812542034037e-08, 6.473752819147194e-06, 6.69620533244597e-07, 1.1779268227485318e-08, 6.672216841252521e-06, 2.4253199448764917e-09]], "dtype": "float32"}}, {"a": [{"__nd__": [[4.2234320640563965, 1.2493547201156616, 2.0593960285186768, 5.417830944061279, 2.7157974243164062, -3.5502166748046875, 7.050563812255859, 5.506555080413818, 3.3873589038848877, -0.26290664076805115, 0.9910125732421875, -1.2603952884674072, 7.642894744873047, 4.970369815826416, 4.297702789306641, 1.8949295282363892, -0.966750979423523, 1.9724856615066528, -4.318467140197754, 1.6474518775939941, 5.308629989624023, 0.08070340752601624, 8.54131031036377, 2.009141206741333, -1.3203977346420288, 1.9320539236068726, 2.001314640045166, 4.923953056335449, -3.937187671661377, 1.9233700037002563, 3.022216558456421, -6.161691665649414, -1.662185788154602, 7.467103481292725, -12.242238998413086, -6.478738307952881, 6.726426124572754, 7.714914798736572, 5.079604625701904, -1.1719528436660767, -0.11505960673093796, 1.0446290969848633, 6.560758590698242, 0.13404373824596405, 7.47062873840332, -0.9568029046058655, -0.05689271166920662, -0.6553658843040466, 2.76741099357605, -1.037451148033142, 1.658294677734375, 3.192279815673828, 2.5800461769104004, 7.841561794281006, -4.411533832550049, -3.5940945148468018, -3.410902976989746, 1.227234959602356, 0.7783933281898499, -2.2916183471679688, -7.331368446350098, 6.9531941413879395, -4.458621978759766, -5.970935821533203, -1.2781827449798584, 1.5881130695343018, 0.49570029973983765, 6.890801429748535, 5.840797424316406, -0.8902703523635864, -1.3125420808792114, 1.3121424913406372, 1.7642844915390015, 3.131042957305908, 3.9469103813171387, 7.116906642913818, -0.21469473838806152, 3.6286590099334717, 4.708251476287842, 0.904545247554779, 4.315161228179932, -5.541357040405273, -1.671318769454956, -0.05970402806997299, -1.1355818510055542, -5.872666358947754, -3.734105110168457, 3.456397771835327, 1.384861707687378, 1.8128913640975952, 6.543035507202148, 5.197755336761475, 0.052730485796928406, 6.422513484954834, -5.64851713180542, -4.659593105316162, -1.906635046005249, 2.8308815956115723, 2.919851303100586, 3.3982737064361572, -2.3563034534454346, -4.111919403076172, 1.8281519412994385, -1.3125598430633545, 6.731423854827881, 3.2871460914611816, -2.266711950302124, 1.2603219747543335, 2.48315167427063, -6.705044269561768, -0.7250155806541443, -0.7245174646377563, 2.6784281730651855, 3.466393232345581, -3.176565647125244, 3.487865447998047, -3.4431240558624268, -3.585984706878662, 1.9366552829742432, -2.608713150024414, 3.3744635581970215, 3.812988042831421, 0.5938137173652649, -1.719234585762024, -0.886197030544281, -3.5243234634399414, 0.3672511577606201, -6.740555286407471, -5.762979984283447, 2.017198324203491, -5.775432109832764, -4.363646984100342, -1.4781337976455688, 2.666585683822632, -6.792537689208984, 0.5450276732444763, 2.589521884918213, -4.341961860656738, 1.7023694515228271, 0.6618782877922058, -0.8003953099250793, 1.3689199686050415, -9.206378936767578, 0.49359244108200073, 0.5099819898605347, 1.933440089225769, -0.29539740085601807, 6.878866672515869, -1.0489424467086792, -0.15518121421337128, 0.585224986076355, 4.149982452392578, 5.417242527008057, 2.2324392795562744, 4.64932918548584, -1.765182375907898, -2.8331711292266846, -3.4544973373413086, 9.582839965820312, -4.033164978027344, -0.15614719688892365, 5.805666446685791, 0.00017955267685465515, -1.1061872243881226, -1.118586540222168, 1.939778208732605, -0.45335012674331665, 6.06797456741333, 2.92434024810791, -3.6573102474212646, -3.317359209060669, 1.6785175800323486, 2.1373748779296875, -4.775317192077637, -4.507817268371582, 8.694818496704102, -7.657325744628906, -1.1729036569595337, -4.398735046386719, 0.30786916613578796, -0.37954333424568176, 0.5643357038497925, -0.8233785033226013, -5.871382713317871, 4.209634304046631, -2.406517267227173, -5.214478969573975, 0.04612937569618225, 3.442167282104492, 1.5322989225387573, 2.220710039138794, 1.2270079851150513, -2.0455446243286133, -1.4440842866897583, -5.390488624572754, 2.5876550674438477, 4.560370445251465, -0.6703753471374512, -1.7717604637145996, 3.652409791946411, -5.277456760406494, 5.344705104827881, 0.3766017258167267, 8.962408065795898, 4.694502830505371, 2.797135353088379, 5.851546764373779, 1.8465557098388672, -1.5874433517456055, 1.612579584121704, -3.828080415725708, -4.515137672424316, -6.086464881896973, -3.145538806915283, 3.8171820640563965, 1.315560221672058, -2.0956830978393555, -2.0387089252471924, -6.364821910858154, -4.951267719268799, 4.679579257965088, -0.898228645324707, -1.4114302396774292, -3.9749019145965576, 6.321427345275879, 2.84273624420166, -2.684108257293701, -3.0871498584747314, -3.027782440185547, 1.3462457656860352, -7.94134521484375, 1.6514989137649536, 5.537271022796631, -1.2036322355270386, 3.4748971462249756, -3.1610734462738037, 0.49169978499412537, 8.225441932678223, 1.543204426765442, 3.7180418968200684, -9.704151153564453, -4.537503242492676, 2.721675395965576, 1.569157361984253, -5.928340435028076, -5.579094409942627, -4.038370132446289, -5.2985734939575195, 1.5301389694213867, -2.676887273788452, 0.340353399515152, 5.331160068511963, 1.5314217805862427, -3.6668806076049805, -2.9937632083892822, 2.380751132965088, -1.0218641757965088, -1.4800323247909546, 1.1377718448638916, 0.09404344111680984, -0.3643692433834076, -4.150486469268799, 6.31311559677124, 5.359620571136475, -2.347109794616699, 1.764475703239441, 4.409941673278809, 2.2986273765563965, 1.9658807516098022, -0.1993635892868042, -3.931831121444702, 1.1273380517959595, 2.4337987899780273, -0.8720054626464844, -2.300916910171509, -6.608798027038574, -0.37386834621429443, -0.4980342388153076, -2.1071653366088867, 1.7874191999435425, 1.7880816459655762, 0.3136948347091675, -1.198691964149475, -6.883574962615967, -1.315007209777832, 5.480441570281982, 0.6332064270973206, -4.383832931518555, -2.321983575820923, 5.097784042358398, -3.347554922103882, 1.9007610082626343, 2.5812265872955322, -0.22803977131843567, -4.421660900115967, -10.771788597106934, 3.2566230297088623, -2.644984006881714, 2.324476718902588, 5.035706996917725, 3.364841938018799, -3.4819533824920654, 4.817603588104248, 2.0154194831848145, 3.8831748962402344, -0.08313394337892532, -5.149158000946045, -5.7657270431518555, -1.997820496559143, 0.5179604887962341, 0.5951889753341675, 0.9584388732910156, 1.7348250150680542, 7.02697229385376, 1.3383378982543945, 5.222921848297119, 0.3260161578655243, 0.046633530408144, 5.053713321685791, 0.3331884443759918, -0.9427154660224915, -0.17530645430088043, 1.6989191770553589, 2.1802663803100586, -0.033582232892513275, -0.3235098123550415, 5.182171821594238, 4.185640335083008, 0.9686139225959778, -5.404376029968262, 1.0438389778137207, 3.445725917816162, 4.273226737976074, -4.432849884033203, 16.3703670501709, -1.0095672607421875, -8.29343318939209, 3.0771899223327637, 0.10681281983852386, -2.137737989425659, -3.9741251468658447, 2.523465871810913, -4.7417683601379395, -7.771134853363037, 3.622795581817627, -4.073310375213623, -4.0342841148376465, 3.0451502799987793, 1.5400935411453247, -0.9551502466201782, 4.340272426605225, -5.205580711364746, 0.39669671654701233, 3.206543445587158, -2.4556713104248047, -0.6116181015968323, -3.877561569213867, -1.2701576948165894, 0.7019041776657104, 4.984241008758545, -1.0137057304382324, -1.3733028173446655, 1.088639259338379, 0.7177529335021973, 7.79380989074707, 5.617984771728516, -1.4250972270965576, 1.9037750959396362, -1.9391149282455444, 0.3800587058067322, -0.6313937902450562, -2.054368019104004, -3.0328755378723145, 2.408506155014038, 0.2973257303237915, 4.317121982574463, 2.794603109359741, 3.5516202449798584, 7.5088605880737305, -3.8006985187530518, 2.7656641006469727, 3.1915431022644043, -3.878164291381836, 4.08501672744751, -3.1812894344329834, -0.6274522542953491, -4.78887939453125, 0.25365960597991943, 3.2226171493530273, -0.3045819401741028, -5.809976100921631, 5.011395454406738, 2.5375564098358154, -3.2140519618988037, 1.7726027965545654, -7.589479446411133, 8.321797370910645, -2.53330659866333, 0.04226873070001602, 6.791311740875244, -2.7412843704223633, -0.7150443196296692, 1.3601396083831787, 2.2795023918151855, 0.44318637251853943, 0.5534289479255676, -1.4480020999908447, 2.638918876647949, 4.8072428703308105, 2.045908212661743, -2.13038969039917, -3.228666067123413, -1.094475269317627, -5.215014457702637, 0.8519036173820496, -2.359477996826172, -9.490561485290527, 7.108680248260498, -3.9734840393066406, -0.8144682049751282, -4.196252346038818, -0.631300687789917, -5.776798248291016, -3.971914291381836, 0.8470052480697632, 0.7407289147377014, -2.6325600147247314, -4.707901477813721, 8.31021785736084, -2.6913328170776367, -1.1266999244689941, -5.32444953918457, -3.3321290016174316, -2.1258273124694824, -0.6554548144340515, -0.4327279031276703, -0.3366923928260803, 1.7804213762283325, -4.705851078033447, -5.494569778442383, 3.13602876663208, 0.6949769854545593, -2.5594420433044434, 1.3164571523666382, 3.6802752017974854, 2.9420697689056396, -2.8977527618408203, 1.4236565828323364, -1.708717703819275, -1.8768516778945923, -4.332589149475098, 2.9261035919189453, 0.9261487126350403, 1.6247344017028809, -0.6626641750335693, -2.1020491123199463, -0.8143759965896606, 0.6992276906967163, -2.769461154937744, -0.28236091136932373, -0.6543883085250854, 7.235359191894531, -6.125737190246582, -1.2268742322921753, -1.043243169784546, 1.782220721244812, 0.13206250965595245, -2.1458888053894043, 4.838528633117676, -7.248376846313477, 1.0182958841323853, -3.774610757827759, 0.195480614900589, -7.1109514236450195, 2.2526695728302, 2.851544141769409, -6.95382022857666, 0.7028428316116333, 4.127792835235596, -2.863323450088501, -1.0740400552749634, 1.6007524728775024, -2.459224224090576, 6.998191833496094, -2.9374005794525146, 1.197472095489502, 4.2418951988220215, -2.120898962020874, 1.193660855293274, -0.8232558369636536, -0.14427347481250763, -1.1372873783111572, -0.9523996710777283, 1.1134454011917114, -4.378269672393799, -0.3807121515274048, 5.551519393920898, 0.879406213760376, 2.5905606746673584, 2.639659881591797, -0.2924058437347412, -3.0192787647247314, 0.5515177845954895, 1.4817850589752197, 9.154324531555176, 0.7245725393295288, 8.703011512756348, 4.8320488929748535, 1.3646471500396729, -0.275713711977005, -0.343916118144989, 1.6098072528839111, -2.830920934677124, -2.7908310890197754, 1.4054360389709473, -1.1084188222885132, 3.8265528678894043, 1.3744783401489258, 2.7574667930603027, 1.691499948501587, 8.16336727142334, -4.709992408752441, 10.210142135620117, 4.456656455993652, -1.242829442024231, 2.0828235149383545, 0.4032079577445984, 5.995516777038574, -0.3166724443435669, -4.095829010009766, 2.7237160205841064, 4.223389148712158, 3.5589406490325928, 1.632474660873413, 5.051271438598633, 1.525854468345642, 1.1644662618637085, -0.8228392004966736, 1.346641182899475, 2.988510847091675, -3.5123190879821777, -0.3785431385040283, -0.43597346544265747, -2.2269554138183594, 0.9610122442245483, 7.72530460357666, -2.9536116123199463, -1.2346141338348389, 3.17502498626709, 0.0775366798043251, 6.4682111740112305, 4.226944923400879, -3.080641746520996, -2.770151138305664, 2.8597097396850586, -2.9549200534820557, -3.696157932281494, -1.1489925384521484, 0.5413450598716736, -0.7826180458068848, 4.161913871765137, 5.80289363861084, -3.941243886947632, 1.7272897958755493, 2.1691927909851074, 3.0286684036254883, 3.805360794067383, 5.75822639465332, -1.7553677558898926, 5.124037265777588, 3.8098318576812744, -3.826448678970337, 2.7647459506988525, 2.4475367069244385, -3.851396322250366, -0.7766244411468506, 1.6033353805541992, -0.30697160959243774, -4.50587797164917, -5.959840774536133, 1.4809099435806274, 3.10351824760437, -0.6312881112098694, 1.2849982976913452, 1.392574667930603, -1.1520359516143799, 3.524557590484619, 0.4541483521461487, -3.670346736907959, 4.175017356872559, 4.578048229217529, 1.232683777809143, -2.784841299057007, -2.255195379257202, 0.3126867115497589, 0.9244490265846252, 1.9458997249603271, 4.302304267883301, 5.1888322830200195, -7.824631690979004, 0.33536458015441895, 2.210240125656128, 0.5931892991065979, -0.7164409160614014, 0.27309104800224304, -2.2588067054748535, -3.53177809715271, 2.2758820056915283, 3.297914743423462, -4.459871768951416, -0.44118934869766235, -0.9217286705970764, 9.815204620361328, 1.1561518907546997, 0.6501184701919556, 4.034605026245117, -1.4516769647598267, -3.6793670654296875, -3.56199049949646, -2.1304969787597656, -4.240163326263428, -1.5110706090927124, 1.5524431467056274, 1.973286747932434, -0.24872857332229614, 6.200883388519287, 5.491456508636475, -5.381007671356201, -2.2743635177612305, 1.0913681983947754, 0.32730016112327576, 4.819174766540527, 2.024967670440674, -6.014010906219482, -1.5840154886245728, -1.2593234777450562, -9.724126815795898, -0.17340821027755737, -5.6426520347595215, 5.819756031036377, 4.6943769454956055, -0.4609496593475342, 3.959275007247925, -5.889639854431152, -3.0354487895965576, 5.481388092041016, 1.1515545845031738, 4.7332963943481445, -7.447391033172607, -3.2130179405212402, -2.729684829711914, 1.2630932331085205, 3.1801271438598633, 3.392232656478882, 3.380425214767456, -5.873343467712402, 4.328609466552734, -6.585883140563965, 4.625871181488037, 1.869149088859558, -0.604320228099823, 2.565520763397217, -3.0669727325439453, 2.2572388648986816, 1.6435176134109497, -4.09404993057251, 1.938133716583252, 3.8767850399017334, 5.1649088859558105, 1.5338659286499023, 0.24064427614212036, -1.4134204387664795, -4.132077693939209, 2.682372808456421, -0.08443860709667206, 1.8154059648513794, 4.889734745025635, -0.5869209170341492, -1.8247458934783936, -3.2480204105377197, 1.6344046592712402, -1.1404165029525757, -7.017206192016602, 2.3789162635803223, 0.6858075857162476, -1.2584564685821533, 9.2720365524292, 4.479163646697998, -2.8692054748535156, -1.9968818426132202, 1.1617649793624878, -3.619188070297241, -2.106532096862793, 6.149182319641113, 2.2772185802459717, 3.1080965995788574, 5.289214134216309, -5.763869762420654, 3.801240921020508, 7.4996724128723145, 2.634812355041504, -1.4306910037994385, 3.6153080463409424, 2.116852045059204, 3.77018666267395, 1.0230365991592407, -0.22717460989952087, -0.06256664544343948, -0.8147872090339661, -3.782801628112793, 3.260610342025757, -2.9859814643859863, -4.336307048797607, 0.5922769904136658, -0.37730008363723755, -3.617177963256836, 0.3037181496620178, 1.3573634624481201, -2.3800318241119385, -8.400016784667969, 1.8957737684249878, -3.2337913513183594, -3.9745612144470215, 6.853271007537842, 6.862155437469482, 7.015835285186768, -2.4670214653015137, -1.0118341445922852, -1.0401649475097656, 3.3075649738311768, 2.9338409900665283, 10.056909561157227, 0.8217990398406982, -1.7514081001281738, 1.5438791513442993, 0.9991715550422668, 0.3378347158432007, -3.222625255584717, -7.283980846405029, 0.9951568841934204, 1.7336605787277222, -2.3960115909576416, 3.162766933441162, -3.153090476989746, 0.07716122269630432, 1.5837750434875488, -4.673421859741211, 1.090172290802002, -1.2637912034988403, -6.473382472991943, 1.0856350660324097, 2.498487710952759, 2.9477550983428955, -3.834522247314453, 7.143996715545654, 2.5675294399261475, 3.1057322025299072, -3.9665329456329346, 2.3352622985839844, -3.444016456604004, -4.2213664054870605, 0.5027232766151428, 1.0823400020599365, -1.192886471748352, 3.8086342811584473, -5.8954854011535645, -1.830966830253601, 1.4372364282608032, -0.5817050337791443, -7.439647674560547, 5.097438335418701, 0.2066323161125183, -2.001488447189331, 9.805301666259766, -7.477235794067383, 3.5603864192962646, 1.1218775510787964, -3.355247974395752, -0.7717389464378357, -6.119186878204346, -0.3104492425918579, 2.3045730590820312, -2.884330987930298, 4.126824378967285, -3.7158589363098145, 2.682102918624878, -0.9861769676208496, 0.0982559397816658, -0.5284393429756165, 5.335381507873535, 2.025405168533325, 1.710651159286499, -1.2687411308288574, 1.1428613662719727, 6.51735258102417, 1.238208293914795, 3.9544143676757812, -0.1539769172668457, -7.04682731628418, 3.6310462951660156, -0.8795719742774963, -1.5057094097137451, -1.401997685432434, 7.272477626800537, 1.898756980895996, 6.065330982208252, 0.3511330485343933, -0.44637027382850647, -2.2944276332855225, 0.5020341873168945, -3.905822515487671, -1.4551111459732056, 2.022481918334961, 3.2571568489074707, 3.1055550575256348, 0.9687911868095398, -0.8768904805183411, 0.5583454966545105, -1.803059697151184, -3.018833875656128, 0.1559450328350067, -5.782719135284424, 2.8322150707244873, 4.423182964324951, -4.018660545349121, -1.5170968770980835, 3.485370635986328, 4.215877056121826, 1.6273199319839478, 6.087425708770752, -4.788241863250732, 5.944531440734863, -1.1619470119476318, -4.936446666717529, 0.8466178178787231, -5.018459320068359, -1.479213833808899, -2.0266215801239014, -1.6984552145004272, -1.0624748468399048, -3.399122953414917, -3.3626463413238525, 2.03855299949646, -3.242645263671875, -2.6986608505249023, 1.657716989517212, -2.037890911102295, 3.0964627265930176, -3.0699706077575684, -0.5477821230888367, 2.912921667098999, -1.2914838790893555, 0.5115600228309631, 7.7633748054504395, -3.9236834049224854, 0.5449908971786499, 0.6365500092506409, -0.8095990419387817, -8.998567581176758, -1.25327730178833, 0.37429192662239075, -6.649525165557861, -6.05774450302124, 4.907351016998291, 1.2366600036621094, 4.373739242553711, -0.28379252552986145, 0.4669921398162842, -2.636828660964966, -0.9081181883811951, -4.717737197875977, 9.738455772399902, -0.14093302190303802, 5.635190010070801, 4.313760280609131, 2.2155256271362305, 2.396165132522583, -3.5895180702209473, 2.512667655944824, 3.718032121658325, -4.8793792724609375, -1.816372036933899, 0.0516517199575901, 6.8879194259643555, 3.7520012855529785, 3.86588978767395, -0.665032148361206, -2.5264945030212402, -1.9800218343734741, -4.967949867248535, 3.452881336212158, -3.1937167644500732, 12.605925559997559, 1.7627397775650024, -1.4754525423049927, 1.602144718170166, 2.2368507385253906, 0.2355802357196808, 5.554189205169678, -7.723141193389893, -3.52250599861145, 2.845106601715088, -1.3887109756469727, 4.586686134338379, -8.999334335327148, -2.3810150623321533, 5.2004714012146, -4.828869819641113, 7.811128616333008, -0.7487157583236694, -3.4891531467437744, 8.092203140258789, 1.828572154045105, 1.4100959300994873, 0.5285302400588989, -4.557489395141602, 2.146613121032715, 8.644103050231934, -4.736231327056885, -1.290257453918457, -0.598885715007782, -1.1358463764190674, -9.271934509277344, 0.8100035190582275, 2.268794059753418, 4.145431995391846, -1.2485888004302979, -1.464762568473816, -4.7560834884643555, -6.080331802368164, 5.381588459014893, -0.014193259179592133, 2.323025941848755, 1.7527433633804321, -7.495926380157471, -5.605381488800049, 3.6887543201446533, 4.554793357849121, -3.0238986015319824, -1.3681102991104126, 8.93228530883789, 1.516574501991272, 2.739563226699829, 1.7707993984222412, 5.975716590881348, 0.5186623334884644, 3.2300782203674316, -3.0005524158477783, -3.2942867279052734, -1.0769126415252686, -0.21330668032169342, -1.9914817810058594, 3.9131171703338623, -0.7487604022026062, -3.24552321434021, -1.0506913661956787, 4.528998851776123, 6.06328010559082, 7.559633731842041, 1.7398208379745483, 2.601910352706909, -8.580221176147461, -0.36636292934417725, -4.806960105895996, -14.904939651489258, 6.374774932861328, 1.7817747592926025, -3.0324811935424805, 4.92169189453125, -1.2081975936889648, -3.891115427017212, -2.025343418121338, -2.294757843017578, -1.4937818050384521, 2.9561636447906494, 3.042471170425415, 0.6219298839569092, 1.905110239982605, 6.486572742462158, 0.43610474467277527, 0.5632149577140808, 0.06971824169158936, 0.7625821828842163, -1.0187948942184448, 0.07915057241916656, -0.028959637507796288, 5.704864978790283, 2.6643123626708984, 1.8739864826202393, -2.1713571548461914, -1.2846099138259888, 2.015619993209839, -0.7223582863807678, -3.4047067165374756, 1.233997106552124, 3.305877923965454, 2.739372491836548, -0.6590619683265686, -1.4039939641952515, -0.42207980155944824, -6.375594615936279, 3.8994176387786865, -6.880584239959717, 1.4035401344299316, 3.5217947959899902, 1.1421509981155396, 6.281534671783447, 0.10953910648822784, -4.697499752044678, -4.4014458656311035, 1.882527470588684, -2.3971338272094727, -1.9531307220458984, -3.1309707164764404, -4.468140125274658, 6.06815242767334, 4.641443729400635, -1.818250060081482, -2.773505687713623, 1.7725403308868408, 5.72283411026001, -4.653161525726318, 4.4809064865112305, -3.6898069381713867, -2.2476694583892822, 2.1437935829162598, -0.033089976757764816, -2.870635986328125, -1.812215805053711, -6.445957660675049, 2.0807271003723145, -5.997028350830078, 4.703373908996582, -2.8796372413635254, -3.0520129203796387, 1.0670812129974365, -2.971776247024536, 2.382662057876587, -0.45206886529922485, -4.420325756072998, 1.5905872583389282, -1.9272093772888184, -5.025033473968506, -7.647532939910889, 4.769960403442383, 0.9681003093719482, 6.938663005828857, -1.265320897102356, 5.815293312072754, -3.1583244800567627, 3.7174415588378906, 2.3426783084869385, 5.414059162139893, -0.18196628987789154, 0.4580491781234741, -3.424576759338379, -2.6740481853485107, -6.448776721954346, -4.2027692794799805, -5.734192371368408, -2.173501968383789, -5.515153408050537, 0.644404411315918, 0.44423142075538635, 3.970994234085083, 2.5379750728607178, -1.044418454170227, -4.666868209838867, 5.120816707611084, 1.1245986223220825, -3.5302257537841797, 4.899628162384033, 4.211165904998779, -0.1260930448770523, -2.833237409591675, -3.1413543224334717, 0.9520490169525146, 2.77907395362854, 2.4878132343292236, -6.458727836608887, 11.504639625549316, -1.8636434078216553, 0.7956988215446472, 4.805478096008301, 5.078848361968994, -3.38118314743042, -2.0547337532043457, -6.302219390869141, 1.257628321647644, 4.117371559143066, -0.27940604090690613, -8.147388458251953, 5.140049934387207, -0.3042166829109192, 2.5991950035095215, -1.3189419507980347, -2.8479971885681152, 4.995729446411133, -2.074004650115967, 3.2891879081726074, -0.729414701461792, 2.503068208694458, 2.3802270889282227, -4.270445823669434, 5.976463317871094, -2.0693650245666504, -8.265216827392578, 5.3054585456848145, -8.925116539001465, 2.586197853088379, -1.388753056526184, -2.886354684829712, 7.0998711585998535, -0.06256232410669327, -3.3141086101531982, -7.718225002288818, 2.573103427886963, -3.8969438076019287, 2.4665372371673584, 7.964488506317139, -9.01746940612793, 0.9618533253669739, -1.044624924659729, -5.749630928039551, -0.8387975692749023, -4.650193214416504, 0.3622288405895233, 9.068943977355957, 2.827288866043091, 0.6696363091468811, 2.252653121948242, -4.490538120269775, -0.5204776525497437, 6.368897914886475, -0.286848247051239, -2.767392635345459, -4.216109752655029, 4.7284369468688965, 2.59879207611084, 2.6732819080352783, -0.24837519228458405, -1.7560906410217285, 1.0640076398849487, 4.063935279846191, 5.237116813659668, -1.8706326484680176, 12.731736183166504, 0.6097314953804016, 1.9687658548355103, -2.9944729804992676, -2.2622246742248535, -9.243793487548828, -1.1840033531188965, 2.3945727348327637, 8.316300392150879, 0.586995542049408, -4.11552095413208, 3.225830078125, 1.8294984102249146, 10.171517372131348, 5.984091281890869, 3.181215524673462, -2.1239449977874756, 0.5370218753814697, -5.390129089355469, -2.63114333152771, -1.5268068313598633, 0.5391498804092407, 4.038338661193848, 5.902439117431641, -2.328866958618164, -0.14424648880958557, 3.467439889907837, 1.5261194705963135, -0.8968546986579895, -0.08944110572338104, 5.866945266723633, 3.2069199085235596, -6.628146648406982, -1.0834780931472778, 1.1123682260513306, -3.944329023361206, -2.892885446548462, -6.8569135665893555, -5.721790313720703, 4.576887130737305, 0.8604044914245605, 1.9764477014541626, -8.65481948852539, -3.347593307495117, -8.521023750305176, 0.3113107681274414, -0.8219963312149048, 2.05602765083313, -0.35521090030670166, 2.8229801654815674, -6.175443649291992, -6.755290508270264, -2.5390875339508057, -5.683323860168457, -7.831744194030762, 9.064746856689453, -2.210909366607666, -1.2033953666687012, 0.9421703815460205, 2.2018024921417236, 2.7019574642181396, -7.6071600914001465, -2.55058217048645, -1.002394676208496, -0.6028674840927124, 3.0853841304779053, 2.3589766025543213, -2.750347852706909, -1.804043173789978, -0.46897995471954346, 1.3271300792694092, 3.4650890827178955, -8.39106559753418, -1.4841634035110474, -4.319955348968506, -2.0090720653533936, -4.667806148529053, 2.9721453189849854, 5.458168983459473, -1.8545598983764648, 2.8716330528259277, -1.6990628242492676, -5.755194187164307, -9.117173194885254, 3.2627055644989014, 3.3127799034118652, -5.355149745941162, -2.9897384643554688, -6.7238006591796875, 1.6392860412597656, -0.8565127849578857, 1.5840176343917847, -5.898509502410889, 2.437978744506836, 3.5583667755126953, 3.1299819946289062, 4.213918685913086, 5.238126277923584, 1.8951334953308105, 4.436854839324951, 3.941070079803467, 5.1863250732421875, 2.484189987182617, -6.441786289215088, 4.560989856719971, -7.404213905334473, -2.1310811042785645, -4.086379528045654, 1.397423267364502, -1.4066749811172485, -6.707771301269531, -0.3533259332180023, 4.58944845199585, 3.6006035804748535, -0.8365576267242432, -1.1047495603561401, 0.7223278880119324, -0.08838297426700592, -0.8544749021530151, 0.9543567299842834, -0.6688051223754883, 2.1272804737091064, 9.19100570678711, -1.6298887729644775, -1.4149034023284912, 1.0363093614578247, 5.742776870727539, -3.7082009315490723, 3.1813087463378906, 5.378591537475586, -3.6156973838806152, -1.0266345739364624, -2.4624431133270264, 5.901355743408203, 3.6160964965820312, -2.7521209716796875, -1.6447803974151611, -8.355752944946289, -1.8211408853530884, -2.176321506500244, 1.325407862663269, -1.7756710052490234, -4.314027786254883, 5.258797645568848, 9.622147560119629, -10.973100662231445, 7.057530403137207, -3.6541595458984375, -5.6199750900268555, 0.1953461915254593, -2.852869987487793, 1.8925902843475342, 0.16961179673671722, 1.8698114156723022, 1.8417859077453613, -2.052030324935913, 7.762177467346191, 4.05662202835083, 6.288693428039551, -5.245065689086914, -3.9681758880615234, 1.3032679557800293, -5.849452495574951, -0.9546891450881958, 7.284778118133545, -4.976530075073242, 3.5598630905151367, -1.041957139968872, -1.3530703783035278, -5.702335834503174, -8.102109909057617, -3.5023069381713867, -6.850703716278076, 6.400044918060303, -6.903570652008057, -3.4894206523895264, 4.856353759765625, -2.9908576011657715, 7.063409328460693, 0.689043402671814, -2.924062967300415, -4.023932933807373, -4.866250514984131, -2.663604497909546, -2.7318427562713623, -6.04975700378418, 2.9248194694519043, 2.978644609451294, -3.46877384185791, 3.4792768955230713, 0.7199409008026123, 1.861057996749878, 0.8433153033256531, 9.736774444580078, 0.9295895099639893, 6.195370674133301, -2.0381269454956055, 0.6664369702339172, -0.5974192023277283, 2.7608723640441895, -0.7032371163368225, -2.753995656967163, 1.057719349861145, 3.3937761783599854, -0.12454119324684143, 0.6937824487686157, 1.3848282098770142, 2.344811201095581, 2.2558207511901855, -3.6974802017211914, 2.4570393562316895, -4.6470513343811035, -2.1869592666625977, 0.8854730129241943, -1.2877334356307983, -0.5116680860519409, -3.168875217437744, -2.89202618598938, -5.1251912117004395, 3.4725255966186523, -6.002261638641357, -0.7305927276611328, 6.015583038330078, 2.3600080013275146, 6.147135257720947, 3.772291421890259, 1.1486287117004395, -0.17146557569503784, -1.5119168758392334, -3.275158405303955, 2.2510101795196533, 4.55671501159668, 3.1678929328918457, 10.602609634399414, 2.5470476150512695, 0.2441493719816208, -2.0330593585968018, -9.1133451461792, 2.2757654190063477, -2.2953360080718994, 0.4692370593547821, 5.750129222869873, 1.4267759323120117, 4.717372417449951, 2.032193660736084, 1.8839750289916992, 1.0890977382659912, -3.9927690029144287, 3.074052572250366, 1.2945340871810913, -1.3022204637527466, 2.7681386470794678, -0.9788496494293213, 0.4036997854709625, -1.1730539798736572, -4.013840198516846, 3.9644510746002197, -3.916368246078491, 2.3779666423797607, -0.5876732468605042, 0.8087016344070435, 2.4254724979400635, 2.9399492740631104, -5.299556732177734, -1.6762638092041016, 0.6189676523208618, -0.25640779733657837, 7.115555763244629, 1.7464864253997803, 3.595090389251709, -5.4952545166015625, 0.7337369322776794, 2.147601842880249, -0.9691974520683289, -1.135245442390442, -4.457712173461914, -4.353250503540039, -1.522348403930664, 4.646824359893799, 0.7011833786964417, 0.33770591020584106, 3.65549635887146, 4.70846700668335, 0.8491783142089844, 2.9269490242004395, -0.7601690888404846, 5.312174320220947, 1.0161962509155273, 3.117246389389038, 6.162302494049072, 6.099605560302734, 1.3670334815979004, 3.7433743476867676, -2.9063827991485596, 9.913641929626465, 1.9780367612838745, -0.848554253578186, 3.333329200744629, -2.414613723754883, -2.995798349380493, -1.5493563413619995, 1.4645588397979736, 6.972012519836426, -2.0172410011291504, 6.827520847320557, 6.282376766204834, -9.18703842163086, 1.5473878383636475, 2.487178325653076, 2.9435653686523438, -2.677511692047119, 0.9690083265304565, 3.8004002571105957, 0.913618803024292, -0.10057386010885239, -3.434375524520874, -0.3736259639263153, -0.2222350835800171, -3.297473669052124, 1.1814613342285156, -0.28555843234062195, 1.4934005737304688, 1.3617674112319946, -2.3383655548095703, -1.6422079801559448, -0.12293598800897598, 3.0083365440368652, -1.1737325191497803, 0.475868284702301, 2.134448766708374, 5.111170291900635, 0.14983032643795013, -5.137503147125244, 1.2403161525726318, 2.848008871078491, 1.5140776634216309, -5.285163402557373, -1.4953923225402832, -1.8477379083633423, 0.6411842703819275, -0.8903945684432983, 5.7973551750183105, -1.7396496534347534, -2.913539409637451, 5.949392318725586, -1.20461905002594, 3.286611795425415, -3.076475143432617, -2.170130729675293, 6.999063491821289, 7.788489818572998, 5.07742977142334, 1.3362455368041992, -4.251554489135742, 0.6781461834907532, -1.7909176349639893, -1.6925734281539917, 0.9164854288101196, 1.192929744720459, 4.306854248046875, -0.027442168444395065, -6.078782081604004, -0.9853528141975403, -3.085998296737671, 3.478102445602417, -6.423081874847412, -5.549626350402832, -9.765755653381348, 0.5579404234886169, -2.3063387870788574, -2.1407105922698975, -3.0124282836914062, -2.7474865913391113, 9.110590934753418, -0.6032695770263672, -5.669584274291992, -6.3113789558410645, 5.721226215362549, -5.3962178230285645, 0.029667947441339493, -2.490232467651367, -3.6235575675964355, 1.6706634759902954, -4.555075645446777, -0.5628423690795898, 4.777282238006592, 8.201659202575684, 2.9630961418151855, -1.5584256649017334, -6.464096546173096, -2.510690927505493, -0.8475455641746521, -1.4507453441619873, -0.049317386001348495, 6.020775318145752, -3.0595195293426514, -2.605987071990967, 1.843812108039856, -1.5966113805770874, 1.3600528240203857, -4.902627944946289, -0.6431818008422852, 2.3227312564849854, 6.740424633026123, -2.053138494491577, 3.5675413608551025, -1.1015723943710327, -0.21713586151599884, -1.954257845878601, -5.945681095123291, -2.6301121711730957, -0.1489429473876953, -4.627312660217285, 6.50405216217041, -0.07970846444368362, -1.3821638822555542, 2.6621053218841553, -7.246746063232422, -4.732264518737793, 1.3033957481384277, -3.4509239196777344, -5.787459373474121, 0.9948642253875732, -2.646932601928711, 5.440334796905518, -1.8230345249176025, 1.5084017515182495, 5.566751003265381, 0.9317286014556885, -1.6627960205078125, -1.3733668327331543, -5.703019618988037, -1.7360358238220215, 0.05276281759142876, -2.8457696437835693, -1.0443081855773926, 2.2018299102783203, -9.156115531921387, 3.704556941986084, 3.9546496868133545, 4.107061862945557, -5.3943071365356445, -1.8636479377746582, -5.819223880767822, -3.122849464416504, -2.5505590438842773, -2.294398546218872, -3.4742472171783447, -9.362196922302246, -1.7725825309753418, 6.361058712005615, 0.8957045078277588, 3.5150206089019775, -2.3223249912261963, 5.111695289611816, -1.961311936378479, -1.2952812910079956, -0.17951270937919617, -8.758309364318848, -4.884505748748779, 5.3249664306640625, -4.301511764526367, -2.5310263633728027, -3.9376742839813232, -4.88963508605957, -1.1079763174057007, 3.8830788135528564, 2.2995684146881104, -4.957516670227051, 1.7943367958068848, 2.3369140625, -3.145641565322876, 4.346141815185547, 1.0022847652435303, 1.4107800722122192, 0.9927710294723511, -5.350117206573486, -1.70658278465271, -4.020463943481445, 3.032578706741333, -5.142092704772949, 0.7570077180862427, -2.024226665496826, -1.960305094718933, 3.1632614135742188, -1.1244181394577026, 2.2625954151153564, 8.112114906311035, 3.157630205154419, 1.1924360990524292, -1.2328474521636963, 0.35561084747314453, 0.9016169905662537, -3.0652589797973633, -1.6285990476608276, 2.652057409286499, -4.671477317810059, -1.2732521295547485, -7.504828929901123, 0.819764256477356, -3.6439120769500732, 0.7202042937278748, -3.042983055114746, 6.203157901763916, 3.9075701236724854, 2.633624315261841, -3.0131754875183105, -5.864737510681152, 2.8719024658203125, 5.017621994018555, -4.943881988525391, -0.2269529104232788, -3.7568016052246094, 3.2971086502075195, 1.493327260017395, -3.7475266456604004, -2.584285259246826, -2.2332069873809814, -0.5735516548156738, -4.36004114151001, -1.7631094455718994, 2.44014573097229, -3.8641598224639893, 9.994209289550781, -2.9611895084381104, 0.1507636159658432, 7.319316387176514, -5.456377029418945, -6.5384321212768555, -4.436521053314209, 0.919439971446991, 0.11922904849052429, -0.7580530643463135, -2.18402361869812, -3.8074207305908203, 9.37110424041748, -1.520235538482666, -6.329061031341553, 3.5120115280151367, -1.2709790468215942, 2.393357038497925, 2.11741042137146, 3.013141632080078, 6.37246036529541, 2.5639264583587646, 1.861840009689331, -5.910253524780273, -3.942113161087036, -4.655893802642822, -2.188798189163208, 2.7000391483306885, 3.5818474292755127, -0.3681911528110504, 4.644777774810791, -0.958766758441925, -0.12660710513591766, -1.2471189498901367, 4.16719913482666, 4.4139227867126465, 2.0642449855804443, 1.8060762882232666, -2.0899860858917236, -2.056486129760742, -2.5893352031707764, -4.969425678253174, -2.9628729820251465, 2.7509243488311768, 2.14595627784729, 3.760010242462158, 2.0952835083007812, 2.346965789794922, -1.2355178594589233, 7.644450664520264, 1.0496660470962524, 2.766234874725342, -2.476271152496338, 5.413235187530518, -1.0890812873840332, 5.336252689361572, -1.1826804876327515, 4.7213134765625, 2.9546689987182617, 1.0813146829605103, -5.786599636077881, 0.9964088201522827, 6.1019816398620605, 5.468465328216553, -6.951910018920898, 6.118138790130615, -5.675096035003662, 3.703620433807373, -0.44464993476867676, 0.20239099860191345, 3.2311904430389404, 9.851165771484375, -5.235588073730469, 2.8342723846435547, 0.23842701315879822, 3.3258817195892334, 2.933897018432617, -0.2787882685661316, -0.961240828037262, 6.094198703765869, -4.237892150878906, 1.2456040382385254, 4.97934627532959, -3.687943935394287, 8.269037246704102, 3.9188551902770996, 3.646745443344116, -3.5238332748413086, 1.8521161079406738, 0.12592457234859467, 4.182986736297607, -0.2831728160381317, 5.520606517791748, 7.661952018737793, -0.44097715616226196, 6.585136413574219, -1.8937501907348633, 4.414253234863281, 0.9313223958015442, -5.329806804656982, 3.963146209716797, -9.327596664428711, -0.6473107933998108, -0.5647755861282349, 0.36276912689208984, 8.116495132446289, -2.234673500061035, -0.289886474609375, -4.035630226135254, -2.3956191539764404, -1.258249044418335, -2.938671350479126, 3.093751907348633, 4.559482574462891, 4.4507575035095215, -0.7175551652908325, 8.655247688293457, 4.300224781036377, -0.31230950355529785, -5.989750385284424, 2.2055821418762207, -5.315362453460693, 2.0047898292541504, -5.940179824829102, 3.9211511611938477, 4.048663139343262, -0.45802703499794006, -2.0041544437408447, -0.7509843707084656, -2.5300846099853516, 0.7352907657623291, 1.1609833240509033, -0.8131830096244812, -0.8978599309921265, 2.18916654586792, 2.1527516841888428, -6.0873003005981445, -6.57730770111084, -3.4720284938812256, 3.3717870712280273, -6.195117950439453, -4.854286193847656, 5.343537330627441, 0.5187868475914001, 4.461184024810791, -5.886721134185791, -1.6460438966751099, -0.6344138383865356, 0.3882484436035156, 4.021881103515625, -0.6076125502586365, -1.7758843898773193, -3.1388065814971924, -0.04656783863902092, 0.6262817978858948, -5.401689052581787, 1.2468689680099487, 0.6461656093597412, 7.437700271606445, -4.123518943786621, 0.7159989476203918, -2.7582340240478516, 0.7478136420249939, 2.444662094116211, 3.2282676696777344, -3.237497568130493, 3.198852777481079, -2.5051980018615723, -1.1561896800994873, 1.792688250541687, 0.5979088544845581, -1.0749731063842773, 2.51967453956604, 2.886481761932373, -1.9819341897964478, -3.203040599822998, 11.385693550109863, 7.353436470031738, 1.0512471199035645, 0.898971438407898, -2.040196180343628, 9.30925464630127, 1.00259530544281, -0.3484565019607544, -5.259443283081055, -5.13713264465332, -4.034519195556641, -3.6410672664642334, -6.9988884925842285, 4.24087381362915, 4.185462474822998, -5.501072883605957, -4.061407566070557, -5.330751419067383, 0.004967821296304464, -5.185999870300293, 0.8237104415893555, -1.5735381841659546, -2.118438720703125, 2.111926317214966, 3.837925672531128, -2.6914572715759277, 2.667207956314087, 0.18069642782211304, 1.5744962692260742, 0.3061489760875702, 1.4398937225341797, 0.19206859171390533, 6.0074663162231445, 8.897066116333008, 1.746622085571289, 9.607210159301758, -0.6727832555770874, 3.880688428878784, 1.461320400238037, 1.6037101745605469, 2.4000961780548096, -3.80704927444458, 1.259519100189209, -1.2262996435165405, 3.644364833831787, -5.046170711517334, -6.910304069519043, 0.19861137866973877, -4.594931125640869, 2.2960357666015625, 8.496121406555176, -2.5925939083099365, 1.7176949977874756, 4.999183177947998, -3.4064462184906006, 4.075101375579834, -1.9893434047698975, -0.8957132697105408, -0.08624478429555893, 2.139346122741699, 5.022213459014893, -4.496915340423584, -1.4211091995239258, -5.814278602600098, -0.693776547908783, -0.5490253567695618, 3.843632936477661, -4.336023807525635, -3.250659942626953, -2.8190884590148926, -2.169934034347534, -12.674684524536133, 0.3114206790924072, 2.976827383041382, -10.775410652160645, -13.04185962677002, -0.16859273612499237, 0.14715293049812317, 2.3108716011047363, 3.4584598541259766, -6.837027072906494, 2.2766590118408203, -6.765358924865723, -1.0236624479293823, -3.7670021057128906, 2.8881595134735107, 2.5619258880615234, 0.471682071685791, 0.01069970615208149, -1.542993187904358, -0.003030151827260852, 4.235509395599365, -3.8305628299713135, 2.7688493728637695, 4.093763828277588, -2.0485432147979736, 3.7335753440856934, -4.474318504333496, -0.09133195877075195, 14.46582317352295, -3.4043989181518555, 8.106555938720703, -3.644970655441284, 0.4161883294582367, 3.746910810470581, 6.907534599304199, -1.2790664434432983, 0.48524391651153564, 6.747149467468262, 6.5572357177734375, -1.0491323471069336, -2.87109112739563, -2.4098362922668457, -1.3581101894378662, 6.068997859954834, 0.02120833843946457, -0.4065132439136505, -2.2188591957092285, -1.4035253524780273, -3.285756826400757, -3.7542641162872314, 2.9986519813537598, -0.557023286819458, 0.345597505569458, 0.9724592566490173, -1.6445403099060059, 2.175628662109375, 4.58142614364624, 1.7790940999984741, 5.03640604019165, -3.262873411178589, -2.4975600242614746, 5.232794284820557, -0.4146577715873718, 3.2382285594940186, 3.9603655338287354, 3.7706220149993896, 0.2347985804080963, -4.125167369842529, 5.883339881896973, 1.173801302909851, 2.8395023345947266, -7.303686141967773, 4.183091640472412, 1.2842698097229004, 0.9731816649436951, -1.0936418771743774, 6.669026851654053, -1.2327426671981812, -1.1365962028503418, 4.499297618865967, 0.8114914298057556, -7.701622009277344, 9.411323547363281, 3.42006516456604, -6.401336193084717, 1.4461452960968018, -1.9426627159118652, 2.790858030319214, -2.0762219429016113, 4.438493251800537, 1.5058817863464355, 0.6372278332710266, 0.35260483622550964, -0.722766101360321, -3.890505313873291, -4.117739677429199, -3.079780340194702, 7.393026351928711, -1.9681572914123535, 0.6808818578720093, -1.0536326169967651, 4.243442058563232, 5.2136688232421875, -2.0964598655700684, -0.7903991341590881, 0.6418474316596985, -0.09659487009048462, 1.2702964544296265, 1.9303526878356934, 1.4489874839782715, 4.214512825012207, 2.7978484630584717, 3.8088948726654053, -0.8019537925720215, -2.4353113174438477, 1.8423218727111816, 1.9009793996810913, -2.1334919929504395, -8.099536895751953, 3.012847423553467, 2.04471492767334, 1.000587821006775, 0.46002107858657837, -5.807648658752441, -0.524124264717102, 1.6096482276916504, 0.09851767122745514, 1.9812363386154175, -2.7288734912872314, -1.9933459758758545, 0.9608446359634399, -3.9733731746673584, 1.2540478706359863, 1.1121485233306885, -5.507266521453857, -2.737623691558838, -11.983744621276855, -1.780870795249939, 2.734508514404297, -2.11207914352417, 5.3820977210998535, 1.858454942703247, 3.7421321868896484, -2.454132556915283, 3.451476573944092, 7.213163375854492, -1.008004069328308, -6.951940059661865, -1.2225085496902466, 8.214133262634277, -3.638204574584961, -1.4428503513336182, 1.5604339838027954, -0.6100091338157654, 3.7629611492156982, 0.5502160787582397, -5.072583198547363, -4.312097549438477, 2.329066514968872, 5.54557466506958, 6.9923996925354, -1.2311902046203613, -6.344871997833252, 1.5850656032562256, 1.2344707250595093, -2.038986921310425, -1.2549114227294922, 0.16757774353027344, -3.853158473968506, -0.798948347568512, 0.06905438005924225, -1.768515706062317, -4.773878574371338, 1.5328407287597656, 1.4949175119400024, -3.140536069869995, 7.7980146408081055, -3.9703783988952637, -0.4463552236557007, -1.7977107763290405, -2.1175050735473633, 4.146946907043457, 1.0774085521697998, 9.22253131866455, 6.1323561668396, 2.96098256111145, 1.695009469985962, -2.7402474880218506, 5.664239883422852, 1.8331431150436401, -4.764430999755859, -3.7034738063812256, -9.019235610961914, 1.5135875940322876, 4.204579830169678, 5.385741710662842, 0.8390487432479858, -5.545003890991211, 1.7569224834442139, -4.435750484466553, 4.648539066314697, 8.63571834564209, 4.702471733093262, -1.388827919960022, -6.237865924835205, -0.11629963666200638, 7.63609504699707, -2.852553367614746, -0.857062578201294, 7.78233528137207, 5.210522174835205, 0.29810696840286255, -1.9560449123382568, 1.8996185064315796, 1.3191044330596924, 0.17631599307060242, -3.3667452335357666, 2.3453562259674072, -4.8322367668151855, 0.39629435539245605, -3.9621129035949707, -0.2948661148548126, -1.8123118877410889, 1.0967826843261719, 2.6838276386260986, -2.2981786727905273, 6.194550514221191, -0.983768105506897, 2.986830711364746, 0.38677555322647095, -5.211970806121826, 9.756288528442383, -2.762692451477051, 4.628668308258057, 4.236577987670898, -5.252001762390137, -1.0325703620910645, -4.236546993255615, -1.8984794616699219, 1.0895315408706665, 4.622847557067871, -2.0185112953186035, -2.008179187774658, -1.8435864448547363, -0.5909862518310547, 3.9928717613220215, -3.834134578704834, 0.23203830420970917, 0.5962089896202087, 2.818373203277588, -5.843860626220703, 5.3452534675598145, 0.21464557945728302, 3.4903645515441895, -6.164791107177734, 0.6490396857261658, -2.368149518966675, -1.1447943449020386, -1.1862475872039795, -2.481760263442993, 0.45447051525115967, -0.16852597892284393, 3.305051803588867, -1.6243340969085693, -0.1377234011888504, -0.8561951518058777, 1.4620319604873657, -0.8921377658843994, 3.664290189743042, -5.194657802581787, 0.35047557950019836, -0.8653076887130737, 3.636427402496338, -0.4532509744167328, -4.848232269287109, 0.7859143018722534, -2.0999388694763184, -1.9751801490783691, 0.021068748086690903, -1.3164243698120117, -4.4183878898620605, 2.283618688583374, 4.388657569885254, 0.1593504548072815, 1.8736401796340942, -4.248893737792969, -7.444990634918213, -3.534406900405884, -2.8802106380462646, -3.0660359859466553, -0.4315451383590698, -3.77765154838562, 4.866544723510742, 2.6554269790649414, -10.469419479370117, 0.9438033699989319, -1.121747612953186, -2.231137275695801, -2.8688924312591553, -1.6033166646957397, 8.562499046325684, 4.54442834854126, 4.448423862457275, -4.680930137634277, 0.28692927956581116, 2.4706523418426514, -4.0297746658325195, -1.0305379629135132, -2.039577007293701, 8.817511558532715, 1.419023036956787, 7.214712619781494, -1.4866688251495361, 2.192986011505127, -4.846751689910889, -8.351884841918945, -8.935087203979492, 1.7083097696304321, -5.255655288696289, -6.222029685974121, 3.4926962852478027, 7.660424709320068, -3.0508341789245605, 0.961209237575531, -1.3123571872711182, -2.5125670433044434, -6.091404438018799, 6.739809513092041, -3.284907341003418, -1.3279320001602173, -1.3712332248687744, -1.6034287214279175, -1.6612249612808228, -4.711147785186768, 10.010324478149414, 0.03048878163099289, -1.7268470525741577, -4.99428653717041, 3.004668712615967, -3.3382325172424316, -6.895091533660889, 2.1272966861724854, 4.716544151306152, -0.796664834022522, -2.7355706691741943, 6.667721271514893, 2.2705955505371094, -1.8838608264923096, 0.27944880723953247, -3.045138120651245, -3.282183885574341, 5.598196983337402, -7.600775241851807, -6.556766510009766, 3.7817647457122803, -3.541317939758301, -0.971446692943573, 2.628552198410034, 3.7695419788360596, 0.10769011080265045, -1.1033250093460083, -3.040233850479126, 3.760296106338501, -2.1295526027679443, 2.3839151859283447, -3.477771520614624, -2.4414727687835693, 2.612269401550293, -2.8673315048217773, -3.118969678878784, -6.648177623748779, -1.5119969844818115, 5.093153953552246, 0.19264203310012817, 0.4553917944431305, 2.2829811573028564, 3.101905345916748, 2.5819950103759766, 5.644764423370361, -7.2324042320251465, -1.4715569019317627, -2.3537676334381104, 0.5261656641960144, -3.286923885345459, -2.211714029312134, -5.507052898406982, -1.183990478515625, -0.8873232007026672, 2.6975982189178467, -1.896424651145935, -0.20581750571727753, 2.0149729251861572, -1.8047499656677246, 2.2427828311920166, -0.5621168613433838, 3.4229774475097656, 1.8122762441635132, -2.347965717315674, 3.7322394847869873, -2.822331666946411, 1.1539525985717773, 1.1359273195266724, -4.522316932678223, 9.258561134338379, 3.011479139328003, 4.013589382171631, 6.285445690155029, -5.572636127471924, 0.7695658206939697, -2.1825835704803467, -1.8181875944137573, -2.095244884490967, -0.27051040530204773, -1.248966097831726, -2.9706690311431885, -7.952827453613281, 0.8863796591758728, 1.0801292657852173, -8.513665199279785, 2.825686454772949, -0.9484190940856934, 0.2177276313304901, -1.7823446989059448, 7.2036519050598145, -0.6082229614257812, 3.2218799591064453, 2.8908276557922363, 1.5658799409866333, 0.25419119000434875, 2.835322856903076, 1.2253497838974, -1.2082713842391968, 10.198481559753418, -1.815706729888916, -0.009725632146000862, -1.6804189682006836, -1.5374255180358887, 2.9302849769592285, -0.9320374131202698, -5.359589099884033, -4.62004280090332, -6.005904197692871, 5.335451126098633, 5.24581241607666, -2.307358980178833, 2.8702704906463623, 4.456345558166504, -1.454697847366333, 2.2355284690856934, -3.15020751953125, 6.4589924812316895, 3.1384410858154297, -0.32217860221862793, -1.9591385126113892, -2.4921956062316895, -5.107854843139648, -1.2747929096221924, -3.973479747772217, 9.429705619812012, 3.739037275314331, -5.189061164855957, 4.767029285430908, 0.4805271625518799, -4.051088333129883, -3.0665998458862305, -0.06697438657283783, -1.233345866203308, -2.5819599628448486, -5.069194316864014, 1.2700120210647583, -4.847099781036377, 0.036061640828847885, -0.0487293042242527, -4.455685615539551, -2.778895854949951, 8.95258903503418, 3.6815924644470215, -4.978290557861328, -0.5636678338050842, -5.165798664093018, -4.86819314956665, -6.875214576721191, 6.022593975067139, 1.280212640762329, 1.2799148559570312, 9.216792106628418, -7.059273719787598, 3.967303514480591, -4.079941272735596, 3.764158248901367, -5.077703952789307, 4.950606822967529, -6.429859161376953, -3.257197618484497, -0.5723266005516052, -0.8943333029747009, 3.335571765899658, -0.11216489970684052, -3.7603142261505127, 8.536623001098633, -4.74902868270874, 1.3552812337875366, -3.3324546813964844, -3.5156338214874268, 0.9058835506439209, -3.017024040222168, 4.380371570587158, -2.628048896789551, -5.4867072105407715, 0.27953749895095825, -2.7514874935150146, 0.5315627455711365, -1.711196780204773, 0.9081254005432129, 3.399545669555664, 6.685586452484131, 5.4983110427856445, -1.224600911140442, 8.127918243408203, -1.3293143510818481, 2.215932607650757, -0.20284929871559143, -1.1684212684631348, 0.6320295929908752, -6.442585468292236, -0.3964961767196655, 5.444937705993652, -0.6688445806503296, -0.48865875601768494, -0.5587066411972046, 5.986804962158203, -3.9529829025268555, 2.573115348815918, 3.355881452560425, -2.8807809352874756, 0.3883686661720276, -0.9795457720756531, -2.1681597232818604, -1.796650767326355, 2.248872756958008, 0.21741800010204315, 0.38424962759017944, -0.17879432439804077, 3.927734851837158, 0.06836756318807602, 1.8374103307724, 10.186594009399414, 3.297398090362549, -2.749114513397217, -2.247140884399414, 0.7366185188293457, -1.912549614906311, 5.113735198974609, -3.3764538764953613, -4.315485954284668, -1.8714030981063843, -0.729260265827179, -2.0615181922912598, 0.41808781027793884, -2.2910337448120117, 1.4742929935455322, 2.3092353343963623, 5.622244834899902, -3.561248779296875, -5.496681213378906, 4.668588161468506, -2.8840322494506836, 5.334104537963867, 3.020582914352417, 5.094013214111328, 4.611120223999023, 1.8747479915618896, -3.630985975265503, 1.5698412656784058, 3.2548863887786865, 0.05578969419002533, 0.3415186405181885, 6.2540082931518555, 5.944632053375244, -1.3097443580627441, 1.6090952157974243, -4.800492286682129, 10.552583694458008, 1.0272082090377808, 2.325084924697876, 6.757601737976074, -1.539076328277588, -1.1594491004943848, -3.2407679557800293, 0.8799391388893127, 3.096691370010376, -3.1606552600860596, -0.6918128728866577, -2.9901793003082275, -0.6275634169578552, -6.278295516967773, -0.18194176256656647, 1.2191818952560425, 2.8891868591308594, -2.9233431816101074, 0.0137705709785223, 2.2892751693725586, -2.53936505317688, -3.0383923053741455, -0.8673365116119385, 2.6222965717315674, 2.8200032711029053, -3.650372266769409, -0.05304737016558647, -1.0462353229522705, -2.224766969680786, -3.2387444972991943, 0.18218769133090973, 0.8751997947692871, -0.7628313302993774, 4.281660556793213, 2.1673338413238525, 2.1158313751220703, -0.3526449501514435, -4.345393180847168, 0.7774988412857056, -0.6284586191177368, -3.1475539207458496, 3.7723283767700195, -2.5410282611846924, -4.116459846496582, 4.332398414611816, -0.7793610692024231, -3.5144684314727783, -2.5439534187316895, -6.496212005615234, -5.497274398803711, 7.635554313659668, 1.9438070058822632, -1.0447062253952026, -0.9046248197555542, 0.556280255317688, -1.116841435432434, -0.850938081741333, -1.932492733001709, 0.8164958953857422, -6.877628803253174, -5.5945868492126465, 1.8172204494476318, 6.277177810668945, 2.183072805404663, -7.15985107421875, 8.501326560974121, 0.7963108420372009, -5.41193962097168, -5.4426093101501465, 0.2321614772081375, -1.763061285018921, -3.511887550354004, 1.3349703550338745, 5.126923084259033, 3.220125198364258, -2.476794958114624, -2.8368308544158936, 1.0540640354156494, -3.4716267585754395, 1.1491811275482178, 4.786190986633301, -1.566253662109375, 2.23397159576416, 6.677834987640381, 7.037532329559326, 6.495123863220215, -1.8797475099563599, -0.5090416073799133, -2.281226873397827, -0.5997872948646545, -1.8179749250411987, -0.364338219165802, 2.3239223957061768, -2.3878695964813232, -2.939356803894043, -1.1962871551513672, -0.8242617249488831, 2.578247547149658, -0.7540161609649658, 6.356310844421387, 0.4984580874443054, -3.523358106613159, 3.9270267486572266, -1.769963264465332, -0.539711058139801, -8.310250282287598, -0.9212165474891663, -3.122051954269409, 4.9131855964660645, -0.46195122599601746, -4.211363315582275, -4.597149848937988, 1.3133480548858643, -0.18374605476856232, 0.1709270030260086, 3.4387362003326416, 1.817740797996521, -3.081789493560791, -0.9562311172485352, 1.2226446866989136, -3.1518962383270264, -5.250067234039307, -9.034113883972168, 4.270321369171143, -0.4871120750904083, 0.316763311624527, 2.779158115386963, -0.8933770656585693, 2.910086154937744, -0.7561739087104797, 2.493258237838745, -2.737682342529297, 5.131467819213867, -5.366952419281006, -9.20811939239502, -1.7700058221817017, -4.905304908752441, -7.6821208000183105, 1.7031121253967285, -0.9661977887153625, 3.7079381942749023, -1.9213027954101562, -3.411121129989624, 0.5881562829017639, 2.0655407905578613, 4.839926242828369, -1.3178294897079468, -3.5158960819244385, -8.385828018188477, 5.073389053344727, 1.076133370399475, 3.3314578533172607, 2.7080514430999756, -1.9235981702804565, -4.6289215087890625, 3.206061601638794, 5.3294501304626465, -5.471710205078125, -0.9192764163017273, 5.084212779998779, -0.8053635358810425, -0.20512980222702026, 1.1149240732192993, 0.4837539494037628, -0.23251934349536896, -4.290746212005615, -5.651077747344971, 2.483701467514038, 2.7543604373931885, 1.1473828554153442, 3.9390671253204346, 0.40306368470191956, -0.881379246711731, -2.110982656478882, 5.888415813446045, 11.644394874572754, -4.099481582641602, 5.095839500427246, -0.46541306376457214, -3.762627601623535, -0.6867902278900146, 0.4013136923313141, -1.5320348739624023, 4.757821083068848, 1.3156434297561646, 0.3061162829399109, 3.8307769298553467, 2.2407312393188477, -1.808007001876831, 0.8662860989570618, 2.8968002796173096, -0.48004162311553955, -0.08188877254724503, -6.254203796386719, -5.7674431800842285, -0.8185974359512329, -2.3595900535583496, -2.442854404449463, 5.30882453918457, 0.8337158560752869, 4.866797924041748, -5.3294758796691895, 2.879476547241211, 0.4466499388217926, -6.643548488616943, 0.9122652411460876, -5.927031517028809, 2.9944512844085693, 3.079029083251953, -2.2869834899902344, -1.7914568185806274, -1.743546962738037, -0.7318083643913269, -4.660019874572754, 0.22771401703357697, 1.2896515130996704, -3.4487249851226807, 7.758432865142822, -3.3827662467956543, 0.8506330847740173, 1.5453425645828247, 4.183310031890869, 5.819145679473877, -0.9350067973136902, -2.8996105194091797, -2.513667345046997, -4.773609638214111, -2.292931318283081, -3.9420721530914307, -4.630252361297607, -6.089692115783691, 3.9533824920654297, 0.9401699304580688, 4.982771396636963, -2.2595608234405518, -3.5408432483673096, 6.167953014373779, -1.840392827987671, -1.0753726959228516, -2.8237574100494385, 5.917280197143555, -3.739624500274658, 4.198363304138184, 0.5062344074249268, -5.744068145751953, 1.0409300327301025, 1.2431820631027222, 2.622469663619995, -0.8739399313926697, -2.607593297958374, 2.6886861324310303, 5.099462509155273, -1.4776930809020996, 11.611800193786621, 5.716353416442871, 1.118718147277832, 5.569639205932617, -1.9071849584579468, -3.066944122314453, -0.7272364497184753, 0.3234104812145233, -6.346644878387451, -5.919148921966553, 2.3346638679504395, 3.1710562705993652, 5.778671741485596, -1.3933119773864746, 2.368931531906128, 3.280846357345581, 2.171987771987915, 5.392348766326904, -2.611670732498169, -0.2848657965660095, 4.682639122009277, 1.9354575872421265, -0.7430810928344727, -4.171407699584961, 4.356657028198242, -1.4662638902664185, -0.2705076038837433, -6.069024085998535, 7.908382892608643, 6.099843978881836, -3.043182611465454, -5.353861331939697, 3.70699405670166, 3.8202404975891113, 6.749668598175049, 0.13175326585769653, 3.5603270530700684, -5.7607316970825195, 1.4449142217636108, 2.9928653240203857, -2.826054573059082, -5.225595474243164, 1.9845305681228638, -0.1849057525396347, 3.5247714519500732, 2.4649360179901123, 3.9914839267730713, -2.8241751194000244, -4.965856075286865, -4.928529262542725, -2.009615182876587, 0.5690396428108215, -1.2676726579666138, 5.279450416564941, 4.054128170013428, 0.3218674659729004, -3.4711527824401855, 4.938323974609375, -1.3694254159927368, 1.1588131189346313, -2.3730692863464355, -3.1204326152801514, -0.44749441742897034, -2.0069329738616943, 3.3441598415374756, -4.941697597503662, 0.16241474449634552, 0.4355849027633667, -2.3004376888275146, -6.946254730224609, -2.1688072681427, -1.6720327138900757, -2.084271192550659, -1.4574156999588013, -0.2102254033088684, -3.0943455696105957, -2.388577461242676, 4.586543083190918, -0.46590980887413025, 0.9919223785400391, -0.5865589380264282, -4.057643890380859, 2.30452036857605, 1.6603370904922485, -5.882049083709717, -1.2272881269454956, -0.8673081994056702, 2.6418280601501465, 2.412785530090332, 1.375338077545166, -5.418211460113525, 1.205949068069458, -6.230384826660156, -4.959380626678467, -6.630678176879883, -2.2729275226593018, -1.304069995880127, -1.2697827816009521, 1.1469109058380127, -6.624316215515137, 5.016386985778809, 5.019649505615234, 3.7856929302215576, 3.4656801223754883, -0.9441767930984497, -3.304621934890747, -3.9790797233581543, -3.9769134521484375, 3.164459705352783, 2.320378541946411, 4.062882900238037, 7.056545257568359, 7.1200995445251465, 1.8382548093795776, 3.772204637527466, 5.495570182800293, 2.9055275917053223, 2.3910815715789795, 3.809242010116577, 3.444288492202759, 5.709272861480713, -0.5141560435295105, 7.925128936767578, -8.547221183776855, 4.858285903930664, 4.938972473144531, -6.565774917602539, -3.03019380569458, 1.357740044593811, -2.2599761486053467, 3.224945545196533, 4.871371746063232, 1.8493473529815674, -5.4198126792907715, 3.18281626701355, -4.526892185211182, -2.9252161979675293, -2.348925828933716, 3.797767162322998, -0.22574597597122192, -1.7730733156204224, -2.6523079872131348, 4.433095455169678, -2.9095652103424072, 0.992922306060791, -5.28494930267334, -2.4946529865264893, -10.355816841125488, 4.155524730682373, -7.179975509643555, 4.32620906829834, 8.254857063293457, -0.8308591842651367, 5.913372039794922, 1.640080213546753, -1.0368551015853882, 1.7111588716506958, 3.202160120010376, 2.8510186672210693, 1.9447369575500488, -0.3620622158050537, -5.605122089385986, 0.365727961063385, -3.7178289890289307, -4.747134208679199, -0.5563182830810547, -0.28654244542121887, -3.917086124420166, 4.246619701385498, -0.1389523297548294, 3.114912986755371, 2.8201711177825928, 3.9170725345611572, 8.399635314941406, -4.038735866546631, 1.267775535583496, -3.8989009857177734, -3.788954257965088, 5.692419528961182, -1.7807650566101074, -5.402249336242676, 2.737464666366577, -2.9743239879608154, -5.683318138122559, 3.3709609508514404, 2.8383445739746094, -1.343889594078064, 0.43036994338035583, 3.304931879043579, 2.8260300159454346, 4.018680572509766, -2.0311195850372314, -1.1426291465759277, -3.242069959640503, -5.151797771453857, 2.8249387741088867, -2.166172742843628, -0.05997734144330025, 0.35505247116088867, -2.9759368896484375, -4.989224910736084, 2.0069403648376465, 8.199602127075195, 0.6839501261711121, -7.8105621337890625, 6.206231117248535, 3.833847761154175, -1.322081446647644, 5.778895378112793, -1.7269995212554932, -8.219541549682617, -3.8171892166137695, -0.1258806735277176, -4.643458843231201, -2.6481964588165283, -2.244955062866211, -1.1701298952102661, 0.26366907358169556, 0.5482003688812256, 5.940546989440918, 4.728292942047119, 3.4695351123809814, 2.500676393508911, 0.48785650730133057, -0.237350732088089, -2.6434128284454346, -4.2875657081604, 2.529505491256714, 12.785896301269531, -2.4873268604278564, 2.9410440921783447, -0.628404974937439, 1.6768993139266968, 0.9263680577278137, 1.5479704141616821, 0.4855632483959198, -0.6861615180969238, 4.508479118347168, -1.2050033807754517, -1.3300124406814575, -4.125497341156006, -2.760462760925293, 4.95358943939209, -6.386228084564209, 1.2856371402740479, 1.3123615980148315, 8.780355453491211, -1.52359938621521, 3.739004611968994], [-2.758476972579956, -0.33250629901885986, 3.358335256576538, 6.398940563201904, -4.324512958526611, -2.752894401550293, 1.5376783609390259, -2.350566864013672, 0.8105167150497437, 6.578622817993164, 4.03793478012085, -0.5099883675575256, -2.2225279808044434, 2.105469226837158, -2.0332179069519043, -0.6910538673400879, 0.2278616577386856, 0.06568422168493271, 2.387410879135132, 1.230234980583191, -1.9298957586288452, 2.710500717163086, -2.958634376525879, -4.39357852935791, -4.184762477874756, -0.5639891028404236, -1.8903666734695435, 4.871150970458984, -1.9154853820800781, 1.7405203580856323, 0.8581520318984985, 1.250035285949707, -0.8880412578582764, -5.682384490966797, -3.9105331897735596, 5.024569034576416, -2.8985679149627686, -2.3096258640289307, -5.974162578582764, -2.387845754623413, -0.6932469606399536, 3.0932908058166504, -3.1014888286590576, -3.396836996078491, 4.210669040679932, -8.042584419250488, 5.494256496429443, 9.374540328979492, 0.0962938740849495, 1.1388459205627441, -0.7736898064613342, -3.899808645248413, -4.318267822265625, -4.462707042694092, -0.59175705909729, 4.976724147796631, -1.83187735080719, -8.843541145324707, -2.065850019454956, -5.923741340637207, -3.012923002243042, 1.8982892036437988, 0.3225356936454773, 3.951385974884033, 4.542745590209961, -3.0810837745666504, 2.971189022064209, 1.7021459341049194, 4.471303462982178, -3.6082301139831543, -4.128689289093018, -3.466231346130371, 1.9854189157485962, -2.3776438236236572, 3.136542320251465, -2.018934726715088, -2.642204761505127, -4.578882694244385, -1.4462121725082397, 9.538406372070312, -2.0234012603759766, -4.425649642944336, -1.939941167831421, -1.7072936296463013, -2.4529457092285156, 3.866971492767334, 0.13526172935962677, 9.016297340393066, 0.34645384550094604, 3.4123454093933105, 0.04453113302588463, 2.058043956756592, -9.494955062866211, 1.679132342338562, -6.295340061187744, -1.9895309209823608, -2.4350779056549072, -3.4508142471313477, -1.6535450220108032, 1.6250559091567993, -4.33172082901001, -1.7376044988632202, 4.824427604675293, 4.290928840637207, -0.1423872858285904, -2.0927834510803223, 3.0710344314575195, -1.8358818292617798, -4.229925632476807, -1.3595319986343384, 1.5483943223953247, -2.8827314376831055, -5.8614606857299805, -2.407710313796997, -4.133083343505859, -3.3930323123931885, 6.593428134918213, -2.583113193511963, 5.605087757110596, 3.376173257827759, -0.043018341064453125, -2.226882219314575, -8.270536422729492, -0.6287847757339478, -2.6163644790649414, -2.7155051231384277, 2.743704080581665, -2.9319334030151367, 2.4811134338378906, -1.6450321674346924, 0.8840502500534058, -1.9116204977035522, -0.29762834310531616, 3.5496461391448975, 4.214672565460205, -0.18158014118671417, -1.5063225030899048, -4.141171932220459, 2.991676092147827, 9.905501365661621, -5.166377067565918, -1.7716907262802124, -0.38478219509124756, 2.98777174949646, 0.15790022909641266, -4.199671268463135, -3.198291778564453, -1.380383849143982, -1.032267451286316, 4.728816032409668, -6.9524617195129395, -1.9221136569976807, 3.5343759059906006, -3.529118299484253, -1.0263054370880127, 3.201087713241577, -0.7824113368988037, -4.592657566070557, 0.4335305392742157, 4.7377119064331055, 3.026015281677246, -2.842688798904419, -5.453699111938477, -2.31787109375, 7.483558654785156, 0.21275557577610016, -1.6272621154785156, 9.007061958312988, -3.3585097789764404, -2.4640731811523438, 1.2797449827194214, 1.0456210374832153, -1.3445061445236206, 6.581717491149902, 1.7032129764556885, -3.42844295501709, 0.302291601896286, 2.530663251876831, 4.698570728302002, 4.150859355926514, 3.9019389152526855, 4.256097793579102, 2.7651193141937256, 10.174419403076172, 5.275232315063477, -1.1892935037612915, -0.9585875868797302, 0.11688142269849777, -2.9753801822662354, 3.2490451335906982, 2.1205313205718994, -8.267932891845703, -2.463994026184082, 2.4336845874786377, 1.2906885147094727, 5.023483753204346, -1.9595839977264404, -2.278348207473755, 2.488646984100342, 6.129337310791016, 0.5197840332984924, -4.216189861297607, 0.3962649703025818, -4.7733354568481445, 7.029444694519043, -0.6766392588615417, -1.7804887294769287, -4.1728739738464355, 6.222195625305176, 0.5543318390846252, -6.522449493408203, 4.145895004272461, 6.1199727058410645, -1.568862795829773, 10.005369186401367, -2.059014320373535, -0.5901786088943481, 1.0840396881103516, -5.8717217445373535, 1.0072699785232544, 7.800667762756348, 2.417491912841797, -0.04521183669567108, 4.092277526855469, -6.523150444030762, 0.2768670618534088, -4.117203235626221, 2.875702142715454, 5.554638385772705, -0.6105718016624451, 4.184418678283691, 9.422805786132812, -5.493828773498535, -0.7267561554908752, 7.923589706420898, 4.455309867858887, 2.419184446334839, -9.200386047363281, -3.984342098236084, 0.3175235390663147, -7.0309062004089355, -1.2723493576049805, -3.964029550552368, 1.931776762008667, 1.9006279706954956, -1.3416916131973267, -0.8230369091033936, 9.86886978149414, 1.324332356452942, 3.103756904602051, -1.3636542558670044, 2.3161230087280273, -9.24291706085205, -0.7673585414886475, -2.0428433418273926, -3.8170506954193115, -2.0246193408966064, -1.566785216331482, 3.6390645503997803, 1.4003010988235474, -2.6498823165893555, -0.7920604348182678, -2.3143296241760254, 3.4428298473358154, 1.7547203302383423, -0.2692084312438965, 1.9085625410079956, -7.343617916107178, -8.623017311096191, 1.115236759185791, -5.351328372955322, 2.874615430831909, -1.2212761640548706, -1.6196423768997192, 3.7485833168029785, -0.9843490123748779, 2.412985324859619, 2.7138943672180176, -4.770808219909668, -5.971920013427734, -1.382871150970459, 0.7868598699569702, 1.1409456729888916, -1.2833722829818726, -6.710453033447266, 2.1387217044830322, -4.944640159606934, -0.4658808410167694, -3.246455430984497, -2.4288322925567627, -5.365068435668945, 2.444873571395874, 1.727785348892212, 2.032139301300049, 4.977697849273682, 1.3883496522903442, -4.772608280181885, -2.9403204917907715, 1.6789238452911377, 8.371004104614258, 1.605193018913269, 5.625034332275391, 0.2546275854110718, 4.681654453277588, 7.7922773361206055, 0.22230249643325806, 2.2497401237487793, 2.3872673511505127, 2.5045359134674072, 3.4633712768554688, 1.3062868118286133, 5.927835464477539, -5.850792407989502, -1.7812868356704712, -2.809877634048462, -0.7881119847297668, 8.140734672546387, 6.391213893890381, -0.7399060726165771, 6.474343299865723, -0.3853653371334076, 1.1751130819320679, -6.349667072296143, -1.1050469875335693, -1.3237067461013794, 0.4489745795726776, -3.4572336673736572, -1.1197185516357422, -2.6590075492858887, 0.12161028385162354, 7.9706268310546875, -2.1481008529663086, 3.5463953018188477, -0.4374222159385681, -7.146884918212891, 0.06367344409227371, -3.3875784873962402, 1.0037684440612793, 1.114414095878601, -1.8297752141952515, -1.8300906419754028, 1.2824835777282715, -1.5651601552963257, -2.181480646133423, 0.1780366748571396, -2.7734925746917725, -1.8278685808181763, -6.684410095214844, 2.528735399246216, 0.7639493346214294, -0.18612565100193024, 10.055389404296875, -4.8484787940979, -0.7773494720458984, -3.758721351623535, -3.038876533508301, 2.972904682159424, -0.5982397794723511, -2.7861742973327637, 5.205507278442383, 1.5052611827850342, -0.13370780646800995, -6.494699001312256, -0.04126638174057007, 5.367435455322266, -0.741527259349823, -1.8419573307037354, 0.4340801537036896, -0.6824716925621033, -6.57806921005249, -6.201546669006348, -2.6837475299835205, 1.5511635541915894, 3.8943731784820557, 2.1104724407196045, -0.15844660997390747, -0.9497901797294617, 3.850766181945801, 4.547069549560547, 4.503929138183594, -3.5558979511260986, 1.0799311399459839, -0.5290445685386658, -3.5115718841552734, 1.3447479009628296, 1.3829729557037354, 4.363185882568359, 6.634045600891113, 3.4863088130950928, -1.8734116554260254, 4.727128982543945, -0.0829157829284668, 2.274458169937134, -1.2617748975753784, -0.27265235781669617, 0.5756825804710388, 0.6461095213890076, -1.766283631324768, 5.829996585845947, -5.850315093994141, -0.11863257735967636, 0.6177685260772705, -0.6416086554527283, 3.816737651824951, -0.8691301941871643, -4.383944988250732, 3.857494354248047, -9.162086486816406, 2.232351303100586, -1.6694947481155396, -5.335681438446045, 0.15342313051223755, -11.84113597869873, -4.1783037185668945, 7.785393238067627, 4.047639846801758, -7.339375972747803, -7.216527462005615, -4.518865585327148, 4.332782745361328, 2.882596254348755, 0.8764277696609497, -5.859201908111572, -4.881533145904541, 0.9035586714744568, 1.5111905336380005, 1.9907968044281006, -0.8219552040100098, 3.144918918609619, 1.4117817878723145, -1.0854696035385132, -0.22227740287780762, 1.4931750297546387, 5.712613582611084, 4.987534523010254, -5.561093807220459, 1.5326844453811646, -6.909411907196045, 2.6181159019470215, -6.186581611633301, 2.9006643295288086, 3.791513442993164, -1.527490258216858, -3.1358392238616943, 4.936491966247559, -5.387601375579834, -0.27911874651908875, -0.24041713774204254, 2.008939266204834, 0.14964693784713745, 1.579793930053711, 5.748732089996338, -2.1886191368103027, 2.461095094680786, -6.218151092529297, 1.9839116334915161, 7.700675010681152, -3.8725297451019287, -0.3977765440940857, -3.239013195037842, -6.283461093902588, 1.5053361654281616, 2.0044026374816895, -6.965487957000732, -1.944779872894287, -4.672513961791992, -0.49047020077705383, -0.8929036855697632, -4.921108245849609, 1.5020017623901367, 6.143533229827881, 3.2090959548950195, 0.5060397386550903, 6.048508644104004, 0.6383209824562073, -4.137282848358154, -2.374497890472412, -4.649104118347168, -4.225315093994141, -2.936990737915039, -2.051907539367676, -1.5941777229309082, -0.8433688282966614, 4.2482476234436035, -1.9679889678955078, -0.38740330934524536, -3.3572754859924316, 5.3356404304504395, -2.5459823608398438, -6.2542033195495605, 0.7443253397941589, -6.525758266448975, -1.7117010354995728, 7.0897345542907715, 1.625892996788025, 5.401468276977539, 2.0870354175567627, 1.6853474378585815, 3.817749500274658, 1.5853663682937622, 2.360520124435425, 4.319761276245117, 3.4965953826904297, -6.731762409210205, -8.512591361999512, -0.494092732667923, 6.770846843719482, -0.7471250295639038, 0.40915966033935547, -0.8424453139305115, -2.180549144744873, -7.763489246368408, -3.1324052810668945, 3.7448008060455322, -3.1911189556121826, 3.3119311332702637, 8.285154342651367, 2.3293423652648926, 3.2971794605255127, -3.522650957107544, 1.5862140655517578, -1.9024986028671265, 3.2048540115356445, -0.20231521129608154, 2.973602533340454, -2.3652830123901367, 2.4639601707458496, -12.853134155273438, 3.3561134338378906, 0.3694857060909271, 1.6588904857635498, 3.042701005935669, 1.3646364212036133, 4.511162281036377, -4.107690334320068, 4.099316596984863, 7.350646018981934, 5.034813404083252, 0.11686652898788452, -2.033885955810547, 1.323533058166504, 0.2946338355541229, 0.10471685230731964, 5.761620044708252, -2.0152904987335205, -3.57216215133667, 2.2359578609466553, -1.7595081329345703, -10.268972396850586, 4.28961706161499, 1.8508120775222778, -0.8117873668670654, -8.251443862915039, -4.982847690582275, -1.659050703048706, 3.6523778438568115, -8.37320327758789, -7.189403533935547, 4.11063289642334, -2.1419451236724854, -2.8029792308807373, -1.5105863809585571, 1.0952610969543457, 2.4110987186431885, 1.8083051443099976, -1.7993264198303223, 2.4084463119506836, -7.698807239532471, 0.34804269671440125, 0.2563653886318207, -4.490677356719971, -2.9064509868621826, -2.123552083969116, -0.8846623301506042, 0.8008750677108765, -4.556567192077637, 0.38263946771621704, 5.316380500793457, -1.9428271055221558, 4.577024936676025, -3.9871788024902344, 0.7834283709526062, 3.460925579071045, 4.043193340301514, 2.9890761375427246, -1.1500787734985352, 2.9198691844940186, 6.430581569671631, -6.30438756942749, 3.037079334259033, 1.0281050205230713, 0.13899007439613342, 0.44360432028770447, -1.8252677917480469, 1.4601993560791016, 4.141808032989502, -2.1402993202209473, 1.2093133926391602, 4.87720251083374, -6.929882526397705, -1.0778424739837646, -6.646069526672363, 4.032994747161865, 2.0395305156707764, 0.34704679250717163, 0.19491727650165558, 2.470547914505005, -0.924095094203949, 0.6893613338470459, -2.2717905044555664, 5.224372863769531, 4.952253818511963, -6.98502779006958, -3.292141914367676, 3.5657777786254883, -0.7076768279075623, 2.038254499435425, 0.5791544318199158, -1.4856300354003906, 4.078759670257568, -0.9390925168991089, -2.206568717956543, -3.0619006156921387, 7.959512233734131, 4.922073841094971, -0.16118814051151276, -1.389744520187378, -5.138289451599121, -4.775795936584473, 0.9387128353118896, 2.4012234210968018, 5.939112663269043, -6.566479206085205, -2.085958957672119, -2.95454740524292, -1.4159703254699707, -2.4877164363861084, 1.4311636686325073, 1.5695229768753052, 1.4969871044158936, -0.36176249384880066, 5.771963119506836, 1.4703748226165771, 2.502519369125366, -6.021974563598633, -3.0582242012023926, 2.137529134750366, -2.276601552963257, -9.866918563842773, -2.844170570373535, 1.9015159606933594, 0.6214386820793152, 2.3697774410247803, 2.547675371170044, 4.595037937164307, 7.345618724822998, 2.475071907043457, -0.09941767156124115, 3.4691383838653564, -0.35978057980537415, -3.4560139179229736, -5.116968154907227, 7.027108192443848, -6.381624221801758, 7.355325222015381, -7.930233955383301, 2.3926939964294434, 0.9088808298110962, 3.870736837387085, 5.465512275695801, -0.1802401840686798, -0.00424622930586338, 0.1258314996957779, -9.784175872802734, -6.361967086791992, -12.005492210388184, -1.0544962882995605, 1.5090192556381226, 5.685558319091797, -1.6286587715148926, -1.5703190565109253, 5.494572162628174, -5.937548637390137, 0.19778795540332794, -2.8052196502685547, -3.320678234100342, 5.120193958282471, -2.514686346054077, 8.611773490905762, -4.902491092681885, -4.588294982910156, -5.650025367736816, -4.198029041290283, 0.18439127504825592, -0.8078314661979675, -0.9503872394561768, -6.784914493560791, 1.6391806602478027, 4.442667484283447, 2.6061532497406006, 2.0684704780578613, -3.4891083240509033, 1.7092435359954834, -0.0768071785569191, -3.4515247344970703, 2.7912676334381104, 1.537784457206726, -2.2822582721710205, 0.5530795454978943, -6.283288955688477, 5.932327747344971, 3.8483431339263916, -1.3776110410690308, 3.6394381523132324, -5.082612991333008, -1.1403084993362427, 0.37551718950271606, -2.4364640712738037, 3.3777012825012207, 5.914908409118652, -1.7368582487106323, 0.8805986642837524, 2.144747495651245, -6.476025104522705, 1.4705559015274048, -3.724545955657959, -3.895308494567871, 1.384183406829834, -2.0220155715942383, -3.015467405319214, -5.27782678604126, -1.5849504470825195, -3.2858083248138428, 0.5164244174957275, 0.8877990245819092, -0.18230465054512024, 4.329789638519287, 3.6240556240081787, 1.1169241666793823, -5.86702823638916, 0.3359302282333374, -0.5073976516723633, 7.7985429763793945, -2.9032371044158936, 4.537341594696045, 3.532006025314331, -0.6142655611038208, -1.4281758069992065, 1.2562438249588013, -0.6399348378181458, 0.6464322209358215, -5.179744243621826, -0.1920391172170639, 0.11130516976118088, -1.280813455581665, -4.442986488342285, 3.9818994998931885, -0.31018131971359253, 3.3449270725250244, 6.5931925773620605, -6.347193717956543, 8.24008560180664, -3.0817627906799316, -0.7940376400947571, -0.5065770745277405, 4.766356945037842, 4.494274139404297, -0.0509287565946579, -1.3428022861480713, 2.280628204345703, -2.5765202045440674, -1.413292407989502, -7.057924270629883, -2.1033411026000977, -6.530954360961914, -4.4801836013793945, 2.8132266998291016, -3.532109260559082, -3.5341720581054688, 6.4183149337768555, -5.563525199890137, 5.2316789627075195, 1.4999606609344482, -0.26871562004089355, -4.793802261352539, 0.6803393363952637, 3.483308792114258, -1.410465121269226, -6.720125198364258, 4.238523006439209, 0.7431178689002991, -0.8133682608604431, 2.0941736698150635, 1.3365938663482666, 7.048168659210205, -3.325397253036499, -0.8581297993659973, 0.2529308795928955, 0.08933469653129578, 4.546207904815674, 2.9393444061279297, -3.8359711170196533, -9.409464836120605, -3.0055644512176514, 0.13636867702007294, 1.4696801900863647, -5.353062152862549, 6.154592514038086, 1.221374273300171, 0.09438712894916534, 5.670688629150391, 4.585854530334473, -3.388976573944092, 5.637359142303467, -5.517645359039307, -1.3384910821914673, 1.746734380722046, 0.3276136815547943, 2.264626979827881, -7.832647323608398, -11.753727912902832, -4.439144134521484, -9.695882797241211, -4.900967597961426, -1.078113079071045, -4.85954475402832, 0.9712997078895569, 7.6233229637146, 1.4664030075073242, -5.568395137786865, 11.260119438171387, 1.2922592163085938, -0.6823610067367554, 1.4297831058502197, -4.377814292907715, 6.579679012298584, -6.621805191040039, -3.494777202606201, -0.8619801998138428, -3.991560220718384, 3.6799840927124023, 6.22395658493042, -4.7557268142700195, -0.18345707654953003, -2.1679415702819824, -7.923032283782959, -1.1845771074295044, -4.589217185974121, 1.444064736366272, -4.325095176696777, -0.5964524745941162, 1.3213508129119873, -0.9504183530807495, 0.8112817406654358, 3.6081364154815674, 1.7190983295440674, -2.1699302196502686, -9.45102596282959, 0.5682655572891235, 4.673946380615234, -5.766836166381836, 4.5572733879089355, 4.518328666687012, -4.304108619689941, 0.41605284810066223, 1.8948230743408203, 1.2452611923217773, -6.765087127685547, -3.746122121810913, 3.259814977645874, 0.4259259104728699, 8.789915084838867, -0.8722577691078186, 7.316352367401123, 1.1663341522216797, 0.6343604326248169, 0.5229785442352295, 3.166928291320801, -2.4534757137298584, 5.649110317230225, -1.5460584163665771, 0.32693761587142944, 3.3412652015686035, -2.9731969833374023, -1.9836138486862183, -7.698205947875977, -2.154416084289551, 2.560354709625244, -4.758866786956787, -2.720726728439331, -2.731038808822632, -2.442883014678955, 5.576786994934082, -4.232798099517822, -0.8701759576797485, -0.9389122128486633, 3.8660590648651123, -0.4707051217556, -2.3546645641326904, -1.520664095878601, -5.681529521942139, -0.32897379994392395, 0.0710153728723526, -8.649356842041016, -2.407470941543579, -0.6779018640518188, -1.0774973630905151, -7.205893516540527, -0.5244632363319397, 0.5939730405807495, 2.1311604976654053, -2.818645715713501, 5.132180690765381, -7.47539758682251, -3.9094483852386475, -4.7066216468811035, -2.3867993354797363, -0.4513559639453888, 3.905256748199463, 8.258758544921875, 6.335861682891846, -0.8514281511306763, -0.7650622725486755, -0.7165603637695312, 13.97880744934082, -1.434759259223938, 1.3649309873580933, -4.05377721786499, 3.1602861881256104, 1.630112648010254, -0.05147939175367355, -2.145533800125122, 0.008310256525874138, 0.22337250411510468, 1.2010397911071777, -3.421663999557495, -0.9010487198829651, 1.220990538597107, -5.9287943840026855, 2.8088090419769287, 1.3341737985610962, 0.9314093589782715, 1.4051803350448608, 2.5192155838012695, -0.3774549961090088, 1.0722482204437256, 2.6400930881500244, 3.0048999786376953, 6.297821521759033, 2.3696601390838623, 5.214084625244141, 3.085937261581421, -1.012836217880249, 4.252696990966797, 3.7454893589019775, -11.03594970703125, 0.5300524234771729, -1.2294639348983765, -4.047792434692383, -0.9486887454986572, -0.40378740429878235, -2.588519334793091, 2.7962067127227783, -0.4813985526561737, -1.4045910835266113, 5.591978073120117, 0.3211221396923065, 2.7472639083862305, 3.352482795715332, 5.463661193847656, 8.035905838012695, -3.3943161964416504, -4.526743412017822, -4.613033771514893, 1.835931658744812, 4.651186466217041, 0.8391948938369751, -1.081994891166687, 1.6044998168945312, 2.309035062789917, 2.7204718589782715, -0.7364084720611572, -9.1517333984375, 2.018646478652954, 4.480189323425293, -4.226564884185791, 0.40176814794540405, -1.6839338541030884, 0.31047505140304565, -2.295940399169922, 0.9365216493606567, 0.5929319262504578, -2.7551827430725098, 3.644789218902588, -0.48691532015800476, -2.4498796463012695, 2.000612258911133, -1.3016088008880615, 0.9920780062675476, -7.777496337890625, -3.5546352863311768, -2.0292065143585205, 1.4153093099594116, -1.3118770122528076, 2.249387264251709, -0.2247513085603714, 1.3872087001800537, 0.6642577052116394, -4.5454277992248535, -1.9861409664154053, 3.130993366241455, -5.370640754699707, 1.3530136346817017, -0.14282502233982086, -0.9167670011520386, -4.794486045837402, -0.1954176127910614, 8.67042064666748, -2.0035054683685303, -3.055253267288208, 0.206720769405365, -3.3900887966156006, -2.313659429550171, -5.626789569854736, -3.1618857383728027, -1.078589916229248, 3.9980974197387695, -3.605055332183838, 4.851892948150635, -3.8894081115722656, -6.421253681182861, 4.513006210327148, 6.518487453460693, -7.094040393829346, -1.4521057605743408, 4.718570232391357, 0.1301565170288086, -1.8667629957199097, -0.8861127495765686, 1.2644342184066772, -0.7475528717041016, -1.3145477771759033, 4.591390609741211, -9.001298904418945, -1.3266186714172363, 5.568612575531006, -7.122086524963379, -0.3187963664531708, 3.4009695053100586, -1.9674445390701294, -2.381930112838745, -7.457711696624756, -3.1842262744903564, 4.136822700500488, -3.079110622406006, 5.319814205169678, 8.246397972106934, -5.673947811126709, -3.8344857692718506, -0.20575283467769623, 0.29586055874824524, -4.608280658721924, -5.341364860534668, 4.116537094116211, 0.36773625016212463, -2.4327638149261475, 3.048964738845825, -4.432109355926514, -4.582411766052246, -2.757218360900879, -0.5111803412437439, 1.6814814805984497, -2.4802405834198, -7.788475513458252, 0.7993914484977722, 1.7618054151535034, -2.7698421478271484, 5.37948751449585, 3.9663400650024414, 3.56974720954895, 3.7544662952423096, 3.1923279762268066, -2.2693192958831787, -2.8317809104919434, 6.546425819396973, -1.0329269170761108, 3.4395129680633545, 1.5878374576568604, -6.2256035804748535, 7.698212623596191, 5.3539228439331055, 5.504227161407471, 5.137426376342773, 2.4298129081726074, 2.475555658340454, 3.596921443939209, 1.3635207414627075, 4.970361709594727, 1.8675521612167358, 0.2769659757614136, 0.7722954154014587, 4.855499744415283, -2.485161304473877, -8.57275676727295, -2.2257909774780273, -0.9181264638900757, 4.489521026611328, -9.64005184173584, -8.8646879196167, -1.4960670471191406, 0.32189565896987915, -1.6746176481246948, 0.5559512376785278, -8.494806289672852, 4.2027082443237305, -0.16335788369178772, -5.1038079261779785, 0.8187175989151001, 0.0163503959774971, -6.873922824859619, -4.037623882293701, -5.47397518157959, -3.0157129764556885, 4.436094284057617, -2.592392683029175, -8.296092987060547, 6.328553676605225, -0.5297367572784424, -5.583409309387207, -9.56436824798584, -1.021126627922058, 1.1146390438079834, 1.3398983478546143, 3.634202718734741, 4.369666576385498, 3.286740779876709, 9.287456512451172, 1.908374547958374, 4.033920764923096, -2.6771016120910645, 0.9371125102043152, 4.520631313323975, 4.890690326690674, -2.57649302482605, -1.7135175466537476, -3.2902865409851074, -4.5304951667785645, 1.221340298652649, 5.6988019943237305, 1.6225277185440063, 2.457350730895996, 0.6047924160957336, 1.1651394367218018, 1.7656223773956299, -2.9626898765563965, 0.7570160627365112, 0.03381264582276344, 7.113746643066406, -4.839709758758545, -1.3490318059921265, -2.111398458480835, 6.524071216583252, 5.050350666046143, -2.3745105266571045, 4.986367702484131, 2.0654618740081787, 1.1665040254592896, 1.362992525100708, 4.06924295425415, 2.723785877227783, -6.711119174957275, -0.8554403185844421, 0.9305039644241333, 1.4047725200653076, 6.007575511932373, 2.1360576152801514, -2.1602530479431152, -2.3653719425201416, -6.201617240905762, 5.206823348999023, -5.995349407196045, -1.0930345058441162, 5.457366943359375, 6.153891563415527, -2.7290995121002197, -12.986682891845703, 1.406162142753601, 1.4856675863265991, 1.6606793403625488, 7.231594085693359, 0.08692952245473862, 9.14163589477539, 2.1067309379577637, -0.5326699018478394, -7.805420398712158, 0.5928064584732056, -0.44110095500946045, -3.1308343410491943, -1.260366439819336, -6.468232154846191, -1.446560263633728, -4.4186201095581055, -3.1297965049743652, -0.9164994955062866, -1.062429666519165, -2.2658252716064453, -8.425589561462402, 2.3487472534179688, 2.7691380977630615, 4.049398899078369, 2.313206911087036, 1.0918760299682617, 5.99114990234375, -4.278219223022461, 1.3416996002197266, -3.4459357261657715, 1.9460930824279785, 8.875575065612793, 1.584977388381958, 1.9635035991668701, -4.072495937347412, 3.398268461227417, -7.385570526123047, 2.2183139324188232, 2.9680261611938477, -4.118935585021973, 1.2678223848342896, -0.2515755593776703, -0.9428099393844604, 2.583683729171753, -2.742542028427124, 6.523646354675293, 1.8424917459487915, -4.6950578689575195, 3.765416383743286, 2.657156467437744, 5.156623840332031, 3.742199182510376, 3.219132423400879, 1.056752324104309, 5.092011451721191, -2.5177857875823975, -2.0614047050476074, -5.719792366027832, 5.383638858795166, 1.2199126482009888, 4.394274711608887, -2.298997402191162, -0.5069913864135742, 2.3358211517333984, -0.2649383544921875, 3.3647189140319824, -1.9069576263427734, 3.6543936729431152, -0.6785604357719421, 6.896001815795898, -7.5557451248168945, 0.9725961089134216, 3.4517579078674316, 7.829615592956543, -3.0575618743896484, 4.25801420211792, -7.224454879760742, 2.0890660285949707, -0.26880335807800293, 0.28230050206184387, 3.371629238128662, -5.960128307342529, -2.350275754928589, 1.110121726989746, -1.8984410762786865, -9.130880355834961, 3.401252031326294, -2.7690224647521973, -3.729208469390869, 7.49824857711792, 6.490496635437012, 1.4128201007843018, 6.444572925567627, 8.19819450378418, 2.7521204948425293, 0.10647998750209808, -0.7609082460403442, 1.5301457643508911, 1.5284126996994019, 0.707021951675415, -5.007310390472412, 5.519338130950928, 2.3174567222595215, -0.4372771978378296, 2.310424327850342, 1.7295949459075928, 2.060554027557373, -3.7112698554992676, -1.3260281085968018, 2.658024549484253, -3.394406318664551, 1.8713895082473755, -9.665432929992676, 5.256492614746094, -1.1658223867416382, 2.7052695751190186, 5.120584487915039, -3.7028591632843018, 9.520234107971191, -0.6349502801895142, 1.3800005912780762, 3.355055332183838, -1.6702672243118286, -0.19631922245025635, 1.4529657363891602, 4.639548301696777, 3.893629789352417, -4.257949352264404, 1.6471436023712158, 2.2400100231170654, -4.936738014221191, -7.3927001953125, -0.9387004375457764, 8.802538871765137, 6.957688808441162, 5.021069049835205, 6.327868461608887, -4.761626243591309, 6.071029186248779, -3.3816418647766113, -2.2736976146698, -6.534251689910889, -0.48138272762298584, -2.929637908935547, -3.9304540157318115, -5.818287372589111, -3.1399478912353516, -7.159388542175293, -2.5164904594421387, 8.722054481506348, 0.25555917620658875, -0.48472410440444946, 1.3799220323562622, -2.549546957015991, -1.7830067873001099, -6.224616527557373, -3.047679901123047, 1.092083215713501, 5.269080638885498, 5.0076680183410645, -0.2953026294708252, 7.48966646194458, -5.030957221984863, -2.435208797454834, 4.491990089416504, -0.6538597941398621, -0.502084493637085, -1.4088283777236938, -4.403620719909668, 1.4810714721679688, 1.8356173038482666, -2.1100871562957764, 1.3435572385787964, -6.6773786544799805, 3.501920700073242, -2.9361627101898193, -1.0937172174453735, -1.7486798763275146, 4.729093074798584, 0.9912053942680359, 3.407410144805908, 0.08260956406593323, -3.585704803466797, 3.155424118041992, 1.671250581741333, 0.19409416615962982, 5.13674783706665, -5.759332180023193, -8.935083389282227, -2.2367892265319824, 3.2001259326934814, 1.5623778104782104, -1.7638462781906128, -2.6314408779144287, -9.294434547424316, -0.7624775171279907, 4.452943325042725, 1.0993895530700684, 3.3954598903656006, 5.360567569732666, -4.008674621582031, -3.5966787338256836, -2.4081881046295166, 7.533828258514404, 10.284327507019043, -2.9065370559692383, 6.639883995056152, -6.65068244934082, -2.9969794750213623, -10.279688835144043, -4.2363128662109375, 0.42915675044059753, 3.758869171142578, -0.48156848549842834, -4.749187469482422, 2.0866539478302, -4.9037957191467285, 1.59127676486969, 2.047804355621338, 4.831448078155518, -0.8775779604911804, 8.249728202819824, -0.3089541792869568, 0.24876254796981812, 3.50780987739563, -2.9260241985321045, -3.1948931217193604, -0.8979539275169373, -2.405771255493164, 1.0103659629821777, 0.06678003817796707, -7.892909526824951, -4.916921138763428, -4.022322654724121, -1.0916857719421387, 5.015470027923584, -2.941326379776001, 2.1487314701080322, -4.175074100494385, 2.9028968811035156, -1.5693327188491821, 2.521949052810669, -3.3639628887176514, -0.4262499213218689, 5.998659133911133, 2.370216131210327, 1.9994593858718872, 4.071410179138184, -3.2401957511901855, -1.0658358335494995, 0.0056914519518613815, 0.607914924621582, -3.347562551498413, -7.693525314331055, -4.382955074310303, -2.1611921787261963, 4.978556156158447, -0.3268390893936157, -1.1777591705322266, -0.09542489796876907, 5.315682888031006, 3.5650742053985596, 3.048121213912964, -4.558755874633789, -3.1439037322998047, -2.268968105316162, -0.7376869320869446, 0.32553717494010925, 0.829643726348877, 0.4689013957977295, -3.1045610904693604, 1.0332199335098267, 4.470856666564941, 1.0815566778182983, -0.884551465511322, 7.931187629699707, 4.139603614807129, 0.9535887837409973, 5.730717658996582, 0.2859674394130707, 0.8726257085800171, -5.419949054718018, -2.787536859512329, 0.13944977521896362, 5.248020648956299, -0.8032612800598145, -1.8282256126403809, -0.2983192503452301, 6.175710678100586, 6.73114538192749, 11.0319185256958, -6.879545211791992, -2.1312475204467773, -0.7626676559448242, -1.696397304534912, -2.014193296432495, 4.421221733093262, 3.6332311630249023, 0.4594762623310089, -3.6030805110931396, 2.987900733947754, 0.03940592706203461, -1.052130937576294, 6.880795955657959, -2.3287441730499268, 2.2931466102600098, -2.8865206241607666, -1.1033904552459717, 2.5381438732147217, 6.894838333129883, -3.3913111686706543, 2.0814287662506104, 6.926971912384033, -5.894586086273193, 2.338487148284912, 3.9843268394470215, -3.0853042602539062, 1.8891767263412476, -1.1060333251953125, 0.972052276134491, 2.4887125492095947, 0.8401485681533813, 2.115741014480591, 0.6761406660079956, -4.141171455383301, -3.485637903213501, -0.5707941055297852, 2.6510159969329834, 5.142910480499268, -4.1361589431762695, -0.5421832203865051, -1.4754717350006104, -0.6949440836906433, -9.077491760253906, 2.704395294189453, 8.05050277709961, 2.529757499694824, 2.6228652000427246, -1.7663333415985107, -10.284472465515137, -0.8258794546127319, -6.2163190841674805, 2.6069326400756836, -3.986372709274292, 0.2732684910297394, 2.9056456089019775, -0.28374651074409485, -8.254433631896973, -1.2284834384918213, -7.459161281585693, -1.964125156402588, -3.1310596466064453, 6.512897491455078, -1.1914172172546387, 4.72244119644165, 3.3715217113494873, -0.17373968660831451, 5.701409339904785, 4.5273613929748535, 6.529614448547363, 1.331156849861145, 7.772974491119385, 2.2579026222229004, -2.8041675090789795, -0.596800684928894, 4.580185413360596, 4.3304762840271, 2.8691627979278564, -3.5937442779541016, 8.005627632141113, 6.783235549926758, -5.901083946228027, 2.9773502349853516, -0.49901315569877625, 9.883622169494629, 1.067071557044983, 3.0302436351776123, -5.624842643737793, -3.8813316822052, 0.47297683358192444, -0.5886525511741638, -4.9552459716796875, -2.5870413780212402, -6.9527506828308105, -3.2933247089385986, -1.318947196006775, -6.0112481117248535, -3.484546184539795, -3.149087429046631, 0.007162462919950485, 2.3490469455718994, -4.903083324432373, -2.7454311847686768, -0.5332677960395813, -6.016636371612549, -4.347541332244873, 3.1186978816986084, 2.4530017375946045, 0.3394187390804291, -4.773931980133057, 0.043528445065021515, -4.681440353393555, -4.329133987426758, 2.799758195877075, 2.336557149887085, -3.891508102416992, 3.305436849594116, 5.515542030334473, -7.400055408477783, -0.9599158763885498, -0.05080389603972435, 1.273232102394104, -4.991893768310547, -2.8414595127105713, -0.3346118927001953, 4.162083625793457, 0.5561091899871826, -1.5824793577194214, 2.4802300930023193, -1.7379919290542603, 6.843037128448486, -1.1978976726531982, -1.069352388381958, 3.0097920894622803, 5.865948677062988, -0.7833762168884277, 4.670937538146973, 3.1611218452453613, -2.2186801433563232, 0.45976072549819946, -6.545234680175781, -8.15341854095459, 0.6232503652572632, -6.697839736938477, 3.6727399826049805, -0.08728402107954025, -4.73215389251709, -1.4304713010787964, -5.6199259757995605, -2.704486608505249, 1.0257827043533325, -4.733372211456299, 1.0255913734436035, 5.341277122497559, 7.508883476257324, -2.069465398788452, 1.496882438659668, 1.217753529548645, -2.382479190826416, 1.6423351764678955, -1.6157710552215576, 2.397679090499878, -1.697331428527832, 3.2244701385498047, -0.296687513589859, -0.7003660202026367, 3.6508026123046875, -8.264801025390625, 2.271251678466797, -4.066760063171387, -5.230356693267822, -1.6413284540176392, 4.714512348175049, -2.237321376800537, 3.6673150062561035, -3.22936749458313, -2.1257169246673584, 0.21814019978046417, 5.032769203186035, -6.577577590942383, 2.137239694595337, 0.1620934158563614, 1.6690702438354492, -6.418412208557129, -1.2870523929595947, 0.741696834564209, 2.532215118408203, 2.1331043243408203, -5.636307716369629, 1.5359786748886108, -4.913295745849609, 1.1687028408050537, 4.91805362701416, 3.0795319080352783, -0.6726323962211609, -2.5787558555603027, 3.8359673023223877, -1.1323907375335693, 6.090343952178955, -6.403285503387451, -0.6251289248466492, 2.853210687637329, -6.296736240386963, -0.22711101174354553, -1.4704011678695679, -4.808862209320068, -0.3181648254394531, 0.4722588062286377, 4.479406833648682, 5.585064888000488, 4.933856010437012, -2.558037042617798, -1.9961204528808594, -2.110323190689087, -7.700229644775391, -2.183732271194458, -4.16917085647583, 1.6097795963287354, -2.425701856613159, 0.06140376627445221, 4.32147216796875, 0.37028324604034424, 1.7079836130142212, 4.764199256896973, -7.4901299476623535, -2.293886661529541, 4.96043062210083, -3.1593964099884033, 0.5640465617179871, -3.1952149868011475, -2.1402807235717773, -4.36691427230835, 4.116676330566406, 0.07040726393461227, -0.12407854199409485, 6.264153003692627, -3.298713207244873, 7.172287940979004, -3.043020725250244, -0.35016047954559326, 4.0715484619140625, -2.267716407775879, 1.2794477939605713, -0.8120086193084717, 1.5726079940795898, -4.834161281585693, 1.5104559659957886, -1.73697030544281, -0.9713799953460693, 2.6865177154541016, 2.3393971920013428, -2.4966328144073486, -1.3700685501098633, -0.5544379949569702, 7.438514709472656, -2.6274898052215576, -3.300100564956665, -0.8620870113372803, -4.893858909606934, -6.313747406005859, 0.1922193169593811, 0.9686689972877502, -1.249336838722229, -5.574650764465332, 7.868316173553467, 5.691610336303711, -0.1667739301919937, 5.434932708740234, 2.0188348293304443, -6.357203960418701, -4.151395320892334, 12.355823516845703, 1.1647239923477173, -3.811978340148926, -1.8265351057052612, -1.0185937881469727, 6.412591934204102, 2.0299265384674072, 0.16912700235843658, 0.6538618206977844, 3.151850700378418, 4.394265651702881, -4.074286460876465, -4.340865612030029, -1.0967530012130737, -0.5565274357795715, -3.128157615661621, -2.0145938396453857, 0.9791668653488159, -0.04874817654490471, -2.6094517707824707, -3.875993490219116, -1.394840121269226, -7.0765886306762695, 2.3923206329345703, 2.054748296737671, -6.883955001831055, -1.952571153640747, 0.10490533709526062, -1.5081990957260132, 0.1829037070274353, -1.5682100057601929, 0.9343096017837524, 3.3061933517456055, -3.026512384414673, -0.8557131290435791, -2.046286106109619, 1.5513417720794678, 0.03896180912852287, 1.4749246835708618, -3.6437759399414062, -4.645318508148193, 1.8481405973434448, 3.0021438598632812, -2.9108736515045166, -1.3371007442474365, -0.30988702178001404, 0.6126992702484131, 3.732353448867798, 5.5979132652282715, -3.084632635116577, 4.649171352386475, 10.919880867004395, -0.1708345264196396, -0.019149957224726677, -4.969673156738281, 2.1326427459716797, 5.675805568695068, 6.33345365524292, -1.6496760845184326, -0.13535897433757782, 2.95889949798584, 1.4461199045181274, -1.3846063613891602, 3.0463924407958984, -1.0199586153030396, 0.09615711122751236, -0.8006448149681091, 2.700458288192749, -1.5711758136749268, -3.948928117752075, -5.976497650146484, 3.965299129486084, 0.1420149952173233, -1.788121223449707, 8.07742977142334, -2.8596014976501465, -1.5690839290618896, -6.5662617683410645, 3.272625207901001, -0.42425817251205444, -7.247320652008057, -0.5431665778160095, 4.522609710693359, -0.09277882426977158, -3.169434070587158, 1.7603431940078735, -0.1341298371553421, 0.12921734154224396, -1.4159841537475586, 2.2875442504882812, -6.624317646026611, -2.7599830627441406, 2.618147611618042, -2.6069843769073486, 1.8790624141693115, -1.750952959060669, 5.355045318603516, 0.6633718609809875, -6.573459148406982, -3.2072854042053223, 4.303415298461914, -4.055258750915527, -7.0566935539245605, -3.537097215652466, 2.2487094402313232, 0.9765322804450989, -5.582675933837891, 0.19515360891819, -0.6080777049064636, 0.9303045868873596, 3.893249034881592, 0.838447630405426, -2.492830753326416, -2.3807454109191895, 3.1034159660339355, -4.219695568084717, -3.4534192085266113, 0.2989681363105774, -1.0117766857147217, -9.332469940185547, -0.2480926513671875, 3.5552866458892822, -1.1103410720825195, -0.8081794381141663, -0.07442867755889893, -2.2594223022460938, -3.397491455078125, 3.0636353492736816, 0.5623761415481567, 0.43965500593185425, 0.1035962700843811, -5.995411396026611, -4.9559550285339355, 2.896235466003418, 2.8105030059814453, 4.627583026885986, 0.48967793583869934, -5.63869571685791, -5.150657653808594, -0.8295727968215942, 3.034120559692383, 0.5718615651130676, -1.225844144821167, -7.9596028327941895, -3.6972594261169434, 0.3987226188182831, -2.4725451469421387, -1.9401793479919434, 0.16084454953670502, 4.336878776550293, 1.8407082557678223, 2.3488378524780273, 7.923192024230957, -2.493396282196045, -3.583320140838623, 7.093721866607666, -2.794994831085205, 1.964286208152771, -0.9821362495422363, 1.4449447393417358, 0.029968533664941788, -2.3752493858337402, -0.18013852834701538, 4.885205268859863, -2.9137587547302246, -1.4563429355621338, -2.487705945968628, -3.301161527633667, -2.772761583328247, 3.579089641571045, 5.572563171386719, 0.679286777973175, -0.04692696034908295, 6.156122207641602, -0.23914691805839539, 5.163187026977539, -4.995095252990723, -4.360209941864014, 6.257612228393555, -1.8685262203216553, 3.7916295528411865, -3.0336382389068604, -1.8907020092010498, -0.34607136249542236, -2.632606267929077, -1.3067270517349243, 0.8323667645454407, -2.1162524223327637, -1.363470196723938, 6.7505621910095215, -1.9515652656555176, 4.995888710021973, 0.48355746269226074, -1.0065020322799683, 1.623745322227478, -1.3245919942855835, 3.3945329189300537, -2.841308116912842, -0.25551679730415344, -3.467508554458618, 4.071984767913818, -5.241499423980713, -5.1025800704956055, -2.928840160369873, 0.32673534750938416, 5.264484882354736, 6.685585021972656, 1.2010186910629272, -2.053711175918579, -1.5662153959274292, 4.882288455963135, -0.5490037202835083, -2.9212112426757812, -2.1284384727478027, 0.6507802605628967, 2.153048515319824, -5.156432628631592, 1.1554994583129883, -3.6037168502807617, -3.1444337368011475, 3.5891456604003906, -1.3126966953277588, -5.456382751464844, 1.2569866180419922, -2.999056577682495, -4.835071086883545, -5.259854316711426, 6.813762664794922, 2.9899821281433105, -1.1153066158294678, 1.884891390800476, 1.9814226627349854, -3.277521848678589, -3.3131463527679443, 6.600866317749023, 4.475849628448486, -1.3412984609603882, 4.230310916900635, -1.6636043787002563, 1.8218854665756226, -8.36279010772705, 4.564592361450195, -4.642319202423096, 4.681410789489746, -0.28148478269577026, -1.2401386499404907, -7.655614852905273, 1.0673511028289795, -2.8398690223693848, 2.181680202484131, 3.311002254486084, -2.070622682571411, 5.5778279304504395, -1.6035107374191284, -0.6132863163948059, 3.4288389682769775, -0.6412236094474792, -1.1537234783172607, -0.6990137696266174, 3.035343885421753, 0.4690515100955963, -3.1603171825408936, -4.071429252624512, 4.534809112548828, 4.780436992645264, -8.277043342590332, -0.44753170013427734, -0.5443808436393738, 4.052167892456055, -3.3015472888946533, -6.183491230010986, -4.656569957733154, -3.1084086894989014, -8.858942031860352, 0.7982494235038757, 0.8298959136009216, 8.839990615844727, -4.737908363342285, 1.422772765159607, -1.7101504802703857, 5.292778491973877, 0.5863257646560669, -2.339890956878662, -3.96779465675354, 6.00053596496582, -0.0822732001543045, 5.017404079437256, -1.8363674879074097, -2.286512613296509, -6.8730149269104, -3.5316576957702637, 6.276215553283691, -6.324771404266357, 1.6779569387435913, 7.820117950439453, -2.9793527126312256, 1.692882776260376, -2.6836187839508057, 1.8496381044387817, -6.318033218383789, -5.808257579803467, 2.7646360397338867, 3.9573659896850586, -10.685443878173828, 3.6335389614105225, 3.2698569297790527, 2.3953816890716553, -4.169885635375977, 2.3972790241241455, 3.552171230316162, 1.203810691833496, -1.4818518161773682, 0.07263496518135071, -3.7745630741119385, -4.619298934936523, -4.957827091217041, -5.428615093231201, -1.2283073663711548, 0.9725478887557983, 5.188869953155518, -1.0497466325759888, 0.2699074447154999, -2.5603108406066895, -9.365928649902344, -1.0343414545059204, 5.801625728607178, 3.9032678604125977, 1.6114168167114258, -8.835075378417969, 2.4529612064361572, -6.24576473236084, -0.4972565472126007, 3.9971938133239746, 1.3676027059555054, 3.753767251968384, 2.1722209453582764, 0.6288618445396423, 2.5784060955047607, -6.047997951507568, 4.14370584487915, -6.489250183105469, -2.3076016902923584, -4.793293476104736, -2.0176284313201904, -6.1534857749938965, -3.6926679611206055, -0.46308648586273193, 1.7469478845596313, -4.4483323097229, -1.4239639043807983, 2.6656079292297363, -0.6928899884223938, -2.071640968322754, -2.3123011589050293, 6.077510833740234, -3.6898550987243652, -0.2570553719997406, 3.602324962615967, 3.199061632156372, 0.09938342124223709, 0.5236876606941223, -6.025161266326904, 3.282949686050415, -6.5442118644714355, -0.04997522756457329, 2.590277910232544, 5.64623498916626, 3.6309444904327393, 3.5542523860931396, 3.074280261993408, 0.7236101031303406, 6.263434410095215, 0.9075740575790405, 2.417017936706543, -1.6050013303756714, 1.5592893362045288, -2.4447391033172607, -0.8701596856117249, -5.225686550140381, -2.9267735481262207, 3.3882510662078857, 0.36437439918518066, 2.3011178970336914, 4.694097995758057, 1.7081801891326904, 0.9115719199180603, 6.1111602783203125, 2.2846555709838867, 10.075764656066895, 6.158210754394531, 4.223341941833496, 4.242311477661133, -0.40179121494293213, -2.025803327560425, 5.460140228271484, 2.421607494354248, -3.2701756954193115, -1.8371644020080566, -2.6594319343566895, 1.1222317218780518, 6.352953910827637, -9.289629936218262, -7.277328968048096, 1.636033535003662, 0.9452869892120361, 0.031878676265478134, -5.375181198120117, -3.2241687774658203, 4.093510150909424, -1.3802696466445923, -1.7008849382400513, 1.1575270891189575, -3.748502254486084, 1.2549453973770142, 1.4846705198287964, 6.532020568847656, 2.332629919052124, -4.191583633422852, -2.3584113121032715, -2.585515260696411, 2.6994452476501465, -2.7607710361480713, -2.946194648742676, 3.289893865585327, 8.452269554138184, 1.4841827154159546, -2.8048596382141113, 0.8691390156745911, 5.269802093505859, -5.420851230621338, -7.102870464324951, 2.8979227542877197, 0.3664092421531677, 2.46077823638916, 2.611626148223877, -6.005642414093018, -7.336912155151367, 2.2392563819885254, 3.172651529312134, -7.883371353149414, 0.04156487435102463, 4.196549892425537, 13.528018951416016, -2.3725109100341797, 8.845004081726074, -6.332461357116699, 10.141554832458496, -6.99678373336792, -1.1082301139831543, -0.7405193448066711, -1.1906694173812866, 2.931734800338745, -2.2438251972198486, 2.1398887634277344, -4.454836845397949, 0.7842771410942078, 4.013739585876465, -5.978775978088379, -1.5235536098480225, -5.562032222747803, -1.0789902210235596, 6.397664546966553, -1.4433108568191528, -3.1958425045013428, 1.6030510663986206, 1.1141349077224731, 0.7425758838653564, 4.705258846282959, -0.5108550786972046, -4.960629940032959, 5.751594066619873, 2.1777377128601074, 8.205032348632812, 1.393519401550293, 3.7464511394500732, -1.1965628862380981, -1.1464214324951172, -2.5361664295196533, 3.830723524093628, -3.65572452545166, -5.0223002433776855, 6.738443374633789, 1.87617826461792, -4.647336006164551, -0.36449915170669556, 2.639509916305542, -0.8865689635276794, -1.5442521572113037, -1.1972227096557617, 3.5374457836151123, -7.010381698608398, -1.263218879699707, 1.4022101163864136, -5.879209518432617, -4.354215145111084, 0.43107616901397705, -3.6067099571228027, -3.2655720710754395, -4.299464225769043, -6.647404193878174, 0.534268319606781, 0.5234751105308533, 2.9335110187530518, 0.47443902492523193, 5.743286609649658, 4.12678861618042, 1.231208086013794, 1.927406907081604, -1.1102796792984009, 5.212915897369385, 2.0882298946380615, 11.56927490234375, 0.8865373730659485, -3.3735079765319824, 2.2087604999542236, -6.885209560394287, 0.40556392073631287, -1.0151394605636597, 9.267452239990234, -0.05623794347047806, -4.434800148010254, -3.4537112712860107, 3.6396124362945557, -1.5498548746109009, 0.3033260703086853, -0.4921663701534271, -7.655850410461426, 5.523403167724609, -6.19824743270874, 6.105393409729004, 5.224589824676514, 3.329932689666748, 2.5182454586029053, 1.2335784435272217, -2.8018276691436768, 2.02156662940979, -3.50390625, -3.6568243503570557, -1.191867470741272, -1.9296212196350098, -4.316049575805664, -7.744803428649902, 3.8158011436462402, -1.277888298034668, -0.3487820327281952, -3.0587172508239746, -0.7245285511016846, 2.562394380569458, -1.161607027053833, 5.497824668884277, -1.5439339876174927, -1.1900367736816406, -8.79708480834961, 1.052582859992981, 2.641597032546997, 3.923921585083008, 2.9362142086029053, -5.701591491699219, 0.7800403237342834, 5.980006217956543, 6.1436381340026855, -1.1576054096221924, 1.8685107231140137, 0.6185336709022522, -2.9298605918884277, 9.107821464538574, 4.674645900726318, 1.7387956380844116, 10.765871047973633, -2.3618695735931396, 2.519392728805542, -5.434156894683838, -1.571165919303894, -0.23718950152397156, 1.502122402191162, -1.5885041952133179, -1.9455349445343018, -1.0708742141723633, -0.09947766363620758, 0.17202536761760712, 4.672547817230225, 0.19774229824543, -3.293745517730713, -7.660051345825195, 0.6535602807998657, 3.739389657974243, -1.6351518630981445, -2.025062322616577, 1.7148979902267456, 0.8558580875396729, -8.086738586425781, -1.8221393823623657, -2.3289072513580322, 0.2585473358631134, -2.3575186729431152, 0.6989750862121582, 0.20936408638954163, 1.6797723770141602, -2.9279298782348633, -2.6318109035491943, -1.15389883518219, -1.5829973220825195, -2.7596919536590576, -4.196961879730225, 1.1906145811080933, -3.630751609802246, -5.987004280090332, -7.176140785217285, 1.683990478515625, 1.4251012802124023, 2.8647289276123047, -3.720612049102783, -0.6334825754165649, 6.047001838684082, 4.189503192901611, 3.4551122188568115, -0.5112518072128296, 0.898330569267273, -2.4484431743621826, -3.8164455890655518, 3.16290020942688, -2.584085464477539, 3.113147020339966, -0.905170738697052, 5.459139347076416, 3.18652606010437, 4.7406535148620605, 4.8973188400268555, 2.076908826828003, 5.10727596282959, 0.22165276110172272, 7.004494667053223, 2.1492538452148438, -4.12821626663208, 4.994661331176758, 1.8323633670806885, -6.340590000152588, -3.4134016036987305, 3.403618335723877, -3.220392942428589, 7.46351432800293, 0.5043705105781555, 4.7104315757751465, -0.04751495271921158, -1.6073634624481201, 3.9269320964813232, 0.3060050904750824, 8.504693984985352, -3.65358567237854, -0.5041685104370117, -0.2883915305137634, -2.625990152359009, -3.237215757369995, -0.3560217320919037, -3.5640225410461426, 8.919577598571777, -1.5965440273284912, -0.20526471734046936, 1.2667157649993896, 11.507805824279785, 2.009915828704834, -0.6320834755897522, 3.691683292388916, 1.9193025827407837, 5.62816047668457, -0.57918781042099, -3.6420793533325195, -4.009459018707275, -4.325214385986328, -3.4427807331085205, -6.1933112144470215, -1.6262623071670532, -5.727046489715576, 5.897985458374023, 7.460803508758545, 5.284383296966553, 6.223201274871826, 5.950898170471191, -1.1246978044509888, -7.519406318664551, 1.9953486919403076, 0.466497540473938, -0.7875587344169617, 3.2478837966918945, 2.4640209674835205, 3.4424569606781006, 10.506491661071777, -7.454115390777588, 0.7895016670227051, 0.5672246217727661, -2.516676902770996, -2.554978132247925, -4.693129062652588, -7.993396282196045, 0.9385732412338257, -0.041640203446149826, -5.171135425567627, -3.0398969650268555, -9.129502296447754, 9.976485252380371, -5.134744167327881, -3.095822334289551, -5.01683235168457, 0.49563702940940857, -3.972721576690674, 2.92349910736084, -2.4910292625427246, 3.4096996784210205, 3.4934215545654297, 4.195245742797852, 2.9945478439331055, 1.115024447441101, -1.8542101383209229, 1.400033950805664, -1.268417477607727, 2.449887275695801, -1.4603626728057861, -1.251831293106079, 2.4947407245635986, 0.682338535785675, -2.262538194656372, 5.4222612380981445, 3.5520823001861572, 0.7029500007629395, -1.3247590065002441, 2.8494374752044678, 2.367356777191162, 5.547586917877197, 2.234281301498413, -3.674099922180176, -0.7335077524185181, 1.1076136827468872, -0.8988597393035889, -3.547395706176758, -1.4161866903305054, -8.068435668945312, -3.3532490730285645, 7.5081963539123535, 0.8606279492378235, 0.15139290690422058, 8.633255004882812, -2.1390862464904785, -0.5684227347373962, -0.9585285782814026, -5.4363484382629395, 3.9580135345458984, 1.1038514375686646, -2.1974380016326904, 2.8961215019226074, 6.718468189239502, 2.7615349292755127, 0.11868385970592499, 3.6881515979766846, 4.990325927734375, -2.826307535171509, -1.871383547782898, -0.3314023017883301, -3.9644312858581543, 4.291318893432617, -2.4285645484924316, -3.6435539722442627, 2.827397108078003, -3.771099328994751, 5.090515613555908, 1.311306118965149, 1.1631537675857544, 2.0473384857177734, 0.9832562804222107, 0.4993162453174591, 1.6714353561401367, 0.22918681800365448, 2.0201356410980225, -1.7615272998809814, 0.13455599546432495, 1.52936589717865, 6.027299880981445, 0.05619518831372261, 3.5116829872131348, 3.2671425342559814, 2.714928388595581, 3.003197193145752, -1.897566318511963, -3.235135316848755, -2.605756998062134, -3.9342126846313477, -3.064178943634033, 0.7901884317398071, 1.9253106117248535, -1.2738279104232788, -2.3518054485321045, -6.0340704917907715, -5.806081771850586, -6.787715435028076, 2.5143795013427734, 8.729795455932617, -5.487206935882568, 0.8552299737930298, 5.031566619873047, -0.10460405796766281, -7.837841510772705, -0.6940230131149292, -1.9990473985671997, 0.11832354962825775, 1.9527173042297363, 5.751193046569824, 0.5747697949409485, -5.2645263671875, 2.6611454486846924, -1.2823169231414795, 3.697139024734497, -5.764069557189941, -2.013399362564087, -6.766597747802734, 1.1290191411972046, 2.351499080657959, -0.9619734883308411, 2.994462013244629, -5.0912089347839355, 0.4433048963546753, -0.44006866216659546, -3.1451406478881836, 2.171316623687744, -4.673916339874268, -0.8882188200950623, -0.2671457827091217, -0.888729989528656, -0.37815061211586, 1.5228676795959473, 0.15806929767131805, -0.09539654850959778, -5.4189887046813965, 7.533965587615967, -1.4934903383255005, -3.0753610134124756, 4.395316123962402, -4.441091537475586, -0.8747466206550598, 2.2168540954589844, 2.5076282024383545, 1.0984745025634766, -4.764727592468262, 1.8372911214828491, 6.181361675262451, -4.363367557525635, 1.9363999366760254, 1.4289138317108154, -2.1304240226745605, 5.4998297691345215, -2.2073354721069336, 1.4949885606765747, 4.068599224090576, 0.8938670754432678, -2.6039886474609375, 0.9434314966201782, -8.87206745147705, -0.34136858582496643, 1.2950551509857178, -0.3574371635913849, 7.666615009307861, -0.4577541649341583, 7.087643623352051, -5.893746376037598, -4.503411293029785, -5.942641735076904, 3.879979133605957, 4.742551803588867, 1.4242078065872192, -0.9389866590499878, -2.5131001472473145, 5.906795024871826, -7.971466541290283, 4.184790134429932, 0.6249435544013977, 4.291713237762451, 1.0044914484024048, -0.7418369054794312, -8.015416145324707, -3.5288991928100586, -1.6899521350860596, 1.109678030014038, -2.7009129524230957, -3.741858720779419, -1.8746757507324219, -2.2996723651885986, -2.1477837562561035, 5.228559970855713, -0.5416426062583923, 3.344506025314331, 3.3603944778442383, 3.17897629737854, -2.4056649208068848, 3.3140435218811035, 3.962872266769409, -11.317049980163574, 3.6353437900543213, -4.360769271850586, 1.9821089506149292, 2.265148639678955, -5.84749698638916, -1.363162875175476, -0.9791840314865112, 1.1770468950271606, -1.207689642906189, 0.812005877494812, 2.968742847442627, 0.6120105981826782, 1.2744885683059692, 0.9376225471496582, -0.9072900414466858, -1.3407652378082275, -3.0712411403656006, -0.24413475394248962, 5.372461795806885, 0.006898272782564163, -0.34031376242637634, -7.131025791168213, -2.3403453826904297, 1.0377699136734009, -4.4911699295043945, 0.005402766168117523, -1.9667984247207642, 0.24801117181777954, 4.407003402709961, -10.20324420928955, 6.123022079467773, 1.185611605644226, -3.7455215454101562, -3.5338211059570312, -4.628661155700684, -1.769203782081604, 0.09435433149337769, 1.1781301498413086, -7.495953559875488, -1.2122999429702759, -4.522957801818848, -0.8080803155899048, -5.445008277893066, -3.2250351905822754, -2.3482887744903564, 1.1149663925170898, -6.504042148590088, -0.7235135436058044, 1.274996280670166, 2.411268472671509, 2.067141532897949, -5.369289875030518, 0.5676187872886658, 5.09661340713501, -0.08372458815574646, -1.1618653535842896, 4.994297504425049, 0.8930251002311707, 1.9353642463684082, 6.543984889984131, -4.695345878601074, 3.284303903579712, -2.5775883197784424, -3.1279385089874268, -1.0520468950271606, 8.879929542541504, 4.072081089019775, -3.3526611328125, -1.2258740663528442, -1.8441824913024902, -6.080013751983643, -1.1580448150634766, -0.29662269353866577, 2.755633592605591, -0.45029565691947937, -5.143157482147217, -3.6409056186676025, 0.7205934524536133, -1.2919187545776367, -0.11225999146699905, 7.4873046875, 2.2333712577819824, -0.6271061897277832, 7.5542120933532715, 2.8383090496063232, 2.9883320331573486, -2.048820734024048, 1.2666914463043213, -7.243699550628662, 2.397655487060547, -0.5517138242721558, -4.480961799621582, 2.1236579418182373, 5.859184265136719, 5.660594463348389, -7.906214714050293, -4.09015417098999, 5.881071090698242, -2.1413679122924805, 5.517378330230713, -0.8087223172187805, -2.776801824569702, -1.1780498027801514, 2.7076497077941895, 2.518421173095703, 5.451047420501709, -2.814755916595459, -0.7298212647438049, 3.754878044128418, 1.0515719652175903, 2.846806287765503, 2.6971147060394287, 2.638601541519165, -9.237905502319336, 1.1847418546676636, -4.520915985107422, -1.4226890802383423, -8.20555591583252, -4.43378210067749, -2.4412639141082764, -2.9108946323394775, -0.5826728343963623, 0.4950144588947296, 4.624966144561768, -0.579334557056427, -3.9767611026763916, -1.5878616571426392, -3.115676164627075, -1.2739907503128052, -5.996994495391846, -1.2081315517425537, 1.7253913879394531, -1.856839895248413, -0.29549938440322876, -1.1400364637374878, 6.70836877822876, -2.9195902347564697, 10.340789794921875, -1.6207340955734253, -0.625855028629303, 0.5902217030525208, -7.282996654510498, -2.2354326248168945, -2.7599639892578125, -4.967517852783203, -5.435335159301758, -2.8520960807800293, 3.1338624954223633, -3.676591396331787, 6.75177526473999, -6.668103218078613, 3.215832471847534, 1.4376388788223267, 1.3354713916778564, -4.604982376098633, -0.6037712097167969, -2.0201430320739746, -1.121439814567566, 4.894336700439453, -4.5850443840026855, -1.852431058883667, -4.303778171539307, 2.268549680709839, -3.418382406234741, -1.438443899154663, 3.2420103549957275, -7.184665679931641, 5.941910266876221, 4.762669563293457, -3.379704475402832, -2.8494985103607178, 2.873721122741699, 1.672885537147522, 6.3403449058532715, -6.407344341278076, 9.791056632995605, -1.5553306341171265, -2.5206172466278076, -3.2281382083892822, 6.3662333488464355, 2.221203565597534, -0.24207651615142822, -2.502239465713501, 0.7275384068489075, 1.8701914548873901, 5.281519889831543, 3.7237329483032227, 7.643941402435303, 2.4504358768463135, -0.9156104922294617, -1.86051344871521, -7.326320648193359, -0.28200942277908325, 5.688233375549316, -2.7764508724212646, 7.622278213500977, -0.9072315096855164, -6.5976667404174805, -0.023677142336964607, -10.514442443847656, 0.9935920834541321, -1.409482479095459, -3.462367057800293, -6.472252368927002, 2.877196788787842, 1.007190227508545, -0.9377370476722717, -7.977027416229248, 5.436900615692139, -2.1559484004974365, 7.345513820648193, 2.042973041534424, 2.7624168395996094, 1.6696103811264038, -2.83577823638916, -5.675567150115967, -3.3840222358703613, 4.6793060302734375, 2.376030445098877, -2.0503082275390625, -1.782525658607483, -0.6653441786766052, -0.7040117979049683, -3.6333940029144287, -0.3469725549221039, -3.558588981628418, 4.01607084274292, 5.618244647979736, 1.4834513664245605, -1.085079312324524, -2.8262531757354736, 1.332342505455017, -1.7578879594802856, 1.1984739303588867, 2.5579495429992676, -3.110140562057495, 3.9163482189178467, 0.5191136002540588, 7.6150336265563965, -1.8216520547866821, 7.034151554107666, -0.5226921439170837, -4.901454448699951, -3.270193099975586, -3.533672571182251, 0.16255499422550201, 5.360978126525879, 6.902613639831543, -1.286069393157959, -0.10298355668783188, -4.8349289894104, -3.969043016433716, 1.924088954925537, 4.8031511306762695, -1.0432604551315308, -4.960836410522461, 2.7848868370056152, 4.597407817840576, -2.8722264766693115, 1.5677131414413452, 3.3958981037139893, -2.119792938232422, -3.450807809829712, 0.9064644575119019, -4.2339935302734375, -1.194127082824707, -3.6194801330566406, -0.17034585773944855, -0.5508182644844055, 1.971266508102417, -6.06443452835083, 1.998369574546814, -4.590253829956055, -3.588470697402954, -1.3832292556762695, 1.0972321033477783, 3.719717502593994, -3.7622339725494385, 3.0981361865997314, 2.672283411026001, -2.8954079151153564, 0.49827516078948975, -3.2944040298461914, 1.9787997007369995, 5.325236797332764, 3.1640586853027344, 2.973233938217163, 2.227017879486084, 8.544093132019043, 3.467310905456543, -1.294985294342041, -3.1155951023101807, -2.4607863426208496, -3.603567600250244, -1.2826920747756958, 2.3357770442962646, 0.12317260354757309, 1.543928623199463, 1.5105692148208618, -2.001659393310547, 3.491880416870117, 0.23614609241485596, -3.027306318283081, 2.9805805683135986, 4.020495414733887, 1.0866189002990723, -0.5335778594017029, 0.5485814213752747, -0.21768726408481598, -4.836250305175781, -2.8026154041290283, 3.2501742839813232, 0.2949049472808838, -6.048654556274414, 8.954526901245117, -0.8566542863845825, -3.393019676208496, -4.577755451202393, 5.704519271850586, 4.155969142913818, 3.642357349395752, -3.6097402572631836, -9.099432945251465, 3.9969370365142822, -0.3669077754020691, -3.4274868965148926, -1.294273853302002, -5.075517177581787, -4.367071628570557, -2.9516990184783936, -1.4147557020187378, -1.9370890855789185, 1.75510835647583, 4.378402233123779, 2.045762538909912, 4.80137825012207, -2.8420021533966064, 2.985757350921631, 5.095546245574951, 0.24414312839508057, -8.7882661819458, -0.8745993971824646, 0.7266359329223633, 4.430667877197266, -8.553750991821289, 3.959246873855591, -2.214052438735962, 1.9082354307174683, 1.0791864395141602, 0.3976459503173828, 5.0442914962768555, 0.4011900722980499, -2.9264867305755615, -8.089739799499512, 1.2863820791244507, 6.136681079864502, -0.976020872592926, 4.3974432945251465, -1.5536361932754517]], "dtype": "float32"}, 0.1], "e": {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.346580992500528e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2504587739891937e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.973293556784574e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.155845839320551e-39, 0.0, 0.0, 1.3088127656793791e-42, 2.569499056640294e-38, 0.0, 0.0, 0.0, 0.0, 2.4943112664981744e-43, 0.0, 2.2332003171482096e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.11736628932906e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.263971214820985e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 6.772475478081841e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.497260459688447e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0879347118439774e-43, 0.0, 0.0, 6.305843089461677e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3760750919669704e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.011570411953465e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.327701879607218e-30, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.629308893534293e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.724235436306426e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2420775429197073e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.236816624685057e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.1019476964872256e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.6432692932559024e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.655688806682134e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.273154354268372e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1104679318802179e-35, 0.0, 0.0, 2.504120355748448e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.984245091899131e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.890467696771533e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.124046173269628e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.9822768076338862e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.582743190928605e-32, 0.0, 5.0245596854486715e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2774707996460344e-36, 0.0, 1.7639372804629033e-27, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.8508268191947657e-38, 0.0, 0.0, 0.0, 0.0, 9.949219096706201e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.398531489199066e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4871074413255536e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 2.985813900263175e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.6537122000227175e-42, 5.085312127034761e-42, 2.364691158548129e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 3.81068013618607e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.51891375416986e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.078103991921978e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6114932339735396e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.078708778060196e-40, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.171647591674637e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5775213933571858e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.580497588469341e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.479590651580511e-17, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.725116074142748e-38, 0.0, 0.0, 1.1178690119127755e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7878010248776947e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.975324655577457e-33, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 5.4384029062845426e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.923635700109488e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1911036946760945e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0930128021733573e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.38608471318065e-22, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.479637514895765e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.1169749881084696e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.951292998247753e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 3.6433760072445244e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5762565817375323e-16, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.051073109500593e-35, 0.0, 0.0, 0.0, 0.0, 1.1987756419913966e-27, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8710897533751175e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.613476979025691e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.930093272772435e-30, 0.0, 3.5881648477501266e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.12199678448668e-38, 0.0, 1.6815581571897805e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 3.4816801774460837e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.044674471569341e-44, 0.0, 0.0, 0.0, 0.0, 3.805366109720473e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.55121981070104e-29, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.931981137358716e-26, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.410099695207443e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.203895392974451e-45, 2.802596928649634e-45, 0.0, 0.0, 0.0, 9.094924300673678e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5257337679568608e-41, 0.0, 3.5971331579218054e-42, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.999652908591514e-41, 5.362666087404101e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.959326717725485e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.340046950758748e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5049945506848535e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4153114489680652e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.363116314379561e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.364164189544862e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0356292288308315e-28, 0.0, 0.0, 4.917983077409735e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.004864584466202e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.783505853677006e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2700670032706444e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 4.869322960789726e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 6.551966882686383e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5129830729702765e-38, 0.0, 3.1809475140173348e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4252456461452339e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.11646557194776e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6066489451823687e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2482159669919227e-22, 6.917804141816968e-40, 0.0, 0.0, 0.0, 2.157637513349571e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 3.498375513170763e-33, 0.0, 4.246027782966778e-30, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.347251068960983e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.353909759975295e-09, 0.0, 1.290400017749007e-36, 0.0, 0.0, 0.0, 8.007019425152005e-42, 0.0, 0.0, 1.6100919355092148e-42, 2.4102333586386854e-43, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.370829922348538e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 5.987672938907806e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.027790766449823e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7012604135981874e-40, 0.0, 0.0, 0.0, 3.783787458616397e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8707334498736308e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.898566901193383e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.064499662178068e-32, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.563583513064257e-34, 0.0, 0.0, 0.0, 0.0, 1.1682577652928213e-38, 0.0, 0.0, 5.042569721275926e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 1.8854801493404122e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2327076353399756e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5789390730458665e-33, 0.0, 1.7278150194971427e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4900506834717544e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4965867598989046e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3915863419072858e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.2727390298458e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2996287877083876e-31, 0.0, 0.0, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5468933747681656e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5697912161170259e-27, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.96831017167883e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.195987562202792e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0953606522958875e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 8.558916015406349e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.516601374109394e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.702063463457114e-43, 0.0, 0.0, 1.59771629525401e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3938463453035744e-27, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1210387714598537e-44, 0.0, 0.0, 0.0, 0.0, 5.41611559704534e-26, 0.0, 0.0, 1.7880568404784666e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1619575273972169e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-44, 0.0, 0.0, 6.68638520697617e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.057466169867698e-43, 2.9376821006105465e-41, 1.2891945871788317e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.2229864679470793e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.9878832032407646e-21, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.970498483872824e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.605193857299268e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.156784387991264e-21, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.778565005200516e-37, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 1.6507295909746345e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.5529922562955737e-41, 6.708155878569332e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.1027947533829407e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.6857526002473905e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.41853227413915e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.272041689656535e-36, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7092015298705815e-16, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0889262632174302e-33, 0.0, 0.0], [0.0, 0.0, 0.0, 1.1920537601475119e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 7.188626969226248e-33, 6.58610278232664e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7617911173684955e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2807853950944185e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.7414668997472616e-43, 0.0, 1.4037923271758067e-37, 9.980709572276547e-21, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.93758508624471e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-44, 1.0349990457503099e-41, 0.0, 0.0, 0.0, 5.065693948534214e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.1383373977240103e-20, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2611686178923354e-44, 0.0, 2.7754403613792025e-22, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7308978761186573e-40, 8.337725862732662e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.335749294523501e-33, 0.0, 4.252427403468491e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.8956097308229915e-43, 0.0, 0.0, 0.0, 0.0, 2.018812246708397e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.653365108614231e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.272318640306503e-41, 0.0, 0.0, 0.0, 0.0, 6.11973147255037e-29, 0.0, 0.0, 2.5305978788022844e-22, 0.0, 0.0, 0.0, 0.0, 0.0, 7.41456947308177e-33, 0.0, 0.0, 0.0, 0.0, 4.916876051622918e-41, 2.059908742557481e-43, 1.6815581571897805e-44, 5.885453550164232e-43, 0.0, 2.97161193988327e-17, 1.5706989133398084e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2669601844453898e-39, 0.0, 0.0, 0.0, 8.043106004001084e-35, 0.0, 0.0, 0.0, 0.0, 6.524398487318119e-31, 0.0, 0.0, 0.0, 2.0356462708246687e-34, 0.0, 0.0, 1.961817850054744e-43, 7.324092614212813e-35, 0.0, 5.4804516476552765e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 1.458544591407861e-27, 0.0, 0.0, 1.14906474074635e-43, 0.0, 0.0, 0.0, 0.0, 2.56766329960009e-37, 0.0, 2.872661851865875e-43, 1.6172444031745792e-20, 0.0, 0.0, 4.986122462825735e-27, 4.3174005685847614e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3996097589671538e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.015245047137591e-40, 0.0, 0.0, 0.0, 0.0, 4.37380083202877e-25, 0.0, 5.191152536356846e-37, 0.0, 4.1516269602551355e-41, 1.3411599316694367e-27, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0722903377047507e-35, 0.0, 0.0, 0.0, 0.0, 4.373322831624013e-26, 1.1034161596367342e-33, 0.0, 2.5337617264791246e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.98072936293807e-27, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.03756444169818e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.821336168241252e-39, 0.0, 0.0, 0.0, 0.0, 3.949354011084934e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 9.80908925027372e-45, 1.0806813756872989e-41, 7.020505306267334e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7179919172622257e-42, 1.251249818944999e-32, 0.0, 0.0, 6.541961880700408e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.030914267650703e-36, 0.0, 0.0, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 1.1210387714598537e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2519393973622508e-27, 7.286752014489049e-44, 0.0, 0.0, 0.0, 1.2681751102139594e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2462800571270262e-36, 8.84376276416964e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.605193857299268e-45, 0.0, 0.0, 5.308384829570629e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7884480381681552e-36, 0.0, 0.0, 0.0, 0.0, 5.366074640782812e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.269918030076457e-35, 0.0, 0.0, 3.584176120054791e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.4510510262235384e-43, 0.0, 0.0, 0.0, 2.87370413766364e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1922738273836652e-30, 0.0, 5.550456897205198e-38, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 1.1126309806739048e-42, 0.0, 0.0, 0.0, 0.0, 4.914329870018021e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.203895392974451e-45, 0.0, 0.0, 1.853609272008353e-25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.547393528853465e-42, 0.0, 1.233142648605839e-43, 1.6199445142806218e-29, 1.4189492197814525e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0344545258958154e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 8.239634970229924e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 1.3732724950383207e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3702666448779844e-38, 0.0, 1.4581911819764046e-41, 0.0, 0.0, 0.0, 7.006492321624085e-44, 0.0, 0.0, 0.0, 1.635727572476076e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.877739942195255e-43, 0.0, 0.0, 2.9340807635572317e-40, 0.0, 0.0, 0.0, 6.305843089461677e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.445243505310245e-39, 6.2146325724187745e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.949219096706201e-44, 0.0, 0.0, 0.0, 7.141221122411541e-27, 4.595628378676454e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.200296822447157e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.256147027917037e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.746017886548722e-41, 1.5405182605944308e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.373723151231978e-31, 0.0, 1.6975464007343367e-29, 0.0, 0.0, 0.0, 1.2611686178923354e-44, 1.0530967593651263e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.508589875013311e-37, 0.0, 0.0, 1.4082306317758934e-37, 0.0, 0.0, 0.0, 0.0, 3.332497942934064e-39, 0.0, 4.858555266641403e-24, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.8045253306418783e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1215590051962924e-35, 9.80908925027372e-45, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 9.422603152016882e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2303400516771894e-42, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 1.4278806089088079e-27, 0.0, 9.80628665334507e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.783505853677006e-44, 0.0, 0.0, 8.316136905964e-33, 0.0, 1.1811012827433979e-25, 0.0, 0.0, 0.0, 9.684513816795243e-41, 6.374506714213593e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4468971701504413e-33, 0.0, 1.016115568034608e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.946583579066604e-43, 0.0, 0.0, 0.0, 0.0, 7.8678785040367565e-31, 0.0, 0.0, 0.0, 0.0, 1.0714328058227551e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0353942504809832e-34, 0.0, 0.0, 8.194790728878479e-37, 1.5928559643980196e-41, 0.0, 5.872049962197149e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.4758313324996444e-28, 0.0, 0.0, 1.5421529071107942e-12, 0.0, 0.0, 0.0, 0.0, 7.264955490844986e-33, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 2.071810630034205e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 3.8436215577965407e-41, 0.0, 1.1968490183798263e-41, 8.107912914583392e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.885135698544636e-23, 0.0, 1.1496474681889494e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 6.604264507255307e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.204263775217174e-37, 0.0, 0.0, 0.0, 1.2611686178923354e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.756882584153299e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8216880036222622e-44, 1.4235827036449133e-25, 6.343774751048292e-34, 0.0, 0.0, 0.0, 0.989098310470581, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.336520801393309e-34, 0.0, 8.52181023809562e-39, 0.0, 0.0, 5.689271765158757e-43, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.7299437422154257e-37, 0.0, 0.0, 0.0, 1.0337824944755217e-37, 1.5330125023410406e-26, 0.0, 0.0, 0.0, 0.0, 3.0612766251639954e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.536530232547352e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.733905855451546e-24, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.484155085839415e-44, 0.0, 2.2779928225603524e-40, 0.0, 0.0, 7.687523375285946e-42, 3.93986547740069e-33, 0.0, 0.0, 6.005404698710436e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6835199750398352e-41, 0.0, 0.0, 2.952753570061807e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.793662034335766e-43, 0.0, 2.4530682500961664e-38, 1.2580602107259474e-25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4573504028978098e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.4552031385276924e-38, 3.2229864679470793e-44, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 5.209741371676453e-33, 0.0, 0.0, 0.0, 0.0, 5.235554849934175e-28, 3.450178427448676e-38, 1.5509754132466823e-37, 3.9592174707156236e-39, 0.0, 0.0, 1.401298464324817e-45, 0.0, 7.448293701456414e-40, 0.0, 0.0, 0.0, 2.361664353865187e-40, 0.0, 0.0, 0.0, 0.0, 6.078832738241056e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.44719422223905e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.562100696313685e-42, 0.0, 0.0, 5.896706239604248e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 1.8328983913368607e-42, 0.0, 4.177930542569438e-21, 0.0, 6.445972935894159e-44, 0.0, 0.0, 8.297088207267242e-42, 3.357749341261197e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0855071337075776e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.515860338775198e-30, 0.0, 0.0, 0.0, 4.1661161811813174e-33, 1.6574642313941796e-39, 0.0, 8.74117370359642e-40, 0.0, 0.0, 0.0, 9.10844001811131e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 2.380230558550907e-35, 0.0, 0.0, 0.0, 0.0, 7.924950979290358e-39, 0.0, 0.0, 9.707191310103105e-38, 1.028161975569801e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 4.9256568630528834e-30, 0.0, 9.720091133772657e-22, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.42688186092153e-44, 0.0, 0.0, 2.0196844006346334e-35, 0.0, 0.0, 0.0, 0.0, 6.794896321154429e-23, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.1484532765342587e-33, 0.0, 0.0, 4.203895392974451e-45, 0.0, 4.797144906935613e-39, 2.802596928649634e-45, 0.0, 0.0, 2.514055561860511e-39, 0.0, 0.0, 0.0, 4.6440466037351906e-38, 0.0, 2.344372330815419e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 1.7179298630417196e-31, 0.0, 0.0, 0.0, 1.9482223909005237e-27, 0.0, 5.997557427310217e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.088101085650424e-29, 2.9779597358197584e-33, 0.0, 1.8813719126643555e-33, 7.768844559358013e-26, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8039770483835994e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3022891608085614e-38, 0.0, 0.0, 3.3455384264430705e-39, 0.0, 4.284513168393679e-20, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7249649937260393e-41, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.273336674358793e-23, 3.1835276385381867e-31, 1.2367327752714392e-39, 5.856439129603854e-34, 0.0, 4.489467482306229e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.463697646390844e-23, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4769865180187006e-38, 1.0816230482553252e-39, 0.0, 6.505163619348834e-29, 0.0, 0.0, 6.230172972388137e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.671722118496887e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.932442860957769e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.2165070791533746e-42, 0.0, 0.0, 3.6872215151544765e-38, 0.0, 0.0, 0.0, 1.0116970183176912e-28, 8.919016059593407e-17, 0.0, 1.3264774901041915e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.856790530153599e-40, 0.0, 1.3006619043553095e-25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1693905749713815e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.1771867248274335e-35, 0.0, 0.0, 9.248569864543793e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.084343074413448e-40, 0.0, 0.0, 0.0, 2.3537890564956817e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0432731731050166e-42, 0.0, 0.0, 5.379727847048065e-27, 1.8357009882655104e-43, 0.0, 1.4936218365561476e-36, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1965040186979095e-38, 0.0, 0.0, 0.0, 1.2788533650381835e-34, 3.3040228036888586e-32, 1.5742129008764622e-13, 0.0, 0.0, 0.0, 0.0, 0.0, 3.070244935335674e-42, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4755951198513405e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6980578959421987e-31, 0.0, 0.0, 2.3415685250159554e-31, 0.0, 0.0, 3.923635700109488e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.182409283620959e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7739418702264128e-26, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.725670937555684e-33, 0.0, 6.242364269027763e-41, 0.0, 0.0, 1.1141822292843e-36, 8.874423174569067e-42, 4.4035741215710814e-33, 0.0, 1.1057306922971438e-27, 0.0, 0.0, 0.0, 1.5049945506848535e-41, 1.2387478424631383e-42, 0.0, 0.0, 1.1325284667293367e-26, 5.562471006438537e-32, 0.0, 0.0, 0.0, 1.6220931162055149e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.73677985444863e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2981294814927e-43, 0.0, 0.0, 0.0, 0.0, 1.0115392199560368e-31, 0.0, 0.0, 0.0, 5.774860295183564e-36, 0.0, 3.729695992646933e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.040338143250361e-38, 7.883456587755775e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 5.766623440389487e-41, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 1.3902380155459013e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.414538578131758e-40, 0.0, 0.0, 0.0, 8.407790785948902e-45, 0.0, 5.446002190456049e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.493089980153283e-42, 3.4807969109980505e-37, 5.170286865911418e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1322491591744522e-42, 0.0, 0.0, 9.4776821634609e-41, 0.0, 0.0, 6.744141223133193e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4573504028978098e-43, 0.0, 0.0, 3.096855249686923e-34, 0.0, 2.7220796668997304e-30, 0.0, 0.0, 9.248569864543793e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9003987400509986e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.868884311865676e-27, 1.0101822068925454e-36, 0.0, 7.756448762590998e-38, 0.0, 0.0, 0.0, 8.845243826272053e-08, 0.0, 0.0, 0.0, 0.0, 1.3664162147374241e-33, 0.0, 0.0, 0.0, 0.0, 2.344372330815419e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 3.95802468546279e-37, 0.0, 3.0001800121194333e-41, 5.13440472863641e-14, 0.0, 0.0, 0.0, 0.0, 8.62502657591341e-37, 6.192839439692787e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.2229864679470793e-44, 0.0, 0.0, 2.3221049813785655e-26, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.46244142605757e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.489123874888577e-38, 0.0, 0.0, 0.0, 9.458764634192515e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5414283107572988e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.417660240499607e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3214244518583025e-42, 0.0, 0.0, 4.966332762275178e-27, 0.0, 0.0, 1.2407740660217563e-30, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.178537280658696e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0717397284335388e-37, 0.0, 0.0, 1.0513543413582805e-34, 0.0, 5.1225530350226575e-39, 0.0, 0.0, 2.9007793481386594e-34, 0.0, 5.605193857299268e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.012076242308497e-32, 0.0, 9.614336789701856e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.388699710976274e-44, 0.0, 0.0, 0.0, 0.0, 1.4106440040430921e-38, 2.0949655912250992e-32, 0.0, 0.0, 0.0, 3.0871586078000747e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.548233613282627e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.979419144307077e-33, 5.301112090540783e-42, 0.0, 4.5542200090556555e-43, 0.0, 0.0, 0.0, 1.287793288714507e-41, 0.0, 4.141537611311997e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.237792251209504e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.561059422088227e-42, 1.1148730582168245e-40, 0.0, 0.0, 0.0, 7.707141553786494e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.760378923745661e-23, 0.0, 0.0, 0.0, 1.8719532801868255e-38, 0.0, 0.0, 0.0, 2.218434781419758e-35, 0.0, 1.192227536044483e-39, 0.0, 0.0, 0.0, 0.0, 3.493879385032112e-34, 0.0, 0.0, 1.7717026299986726e-27, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.942726775082116e-44, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.622560364472979e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 3.035221980136336e-36, 1.6815581571897805e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 4.484155085839415e-44, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 1.9197788961249994e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.7900964317119975e-35, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.417074350027958e-37, 1.401298464324817e-45, 0.0, 0.0, 0.0, 3.0746813395508355e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.701776737349059e-41, 0.0, 0.0, 6.706284237078053e-35, 0.0, 1.1080005722989613e-17, 1.0735432895568638e-34, 4.245934346904196e-43, 5.1287523794288305e-43, 0.0, 0.0, 9.980166493031122e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 7.526244926158741e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1630777253895982e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.510814471196599e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.858034814758977e-25, 0.0, 0.0, 0.0, 1.4876808075088883e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.2510124372335756e-43, 0.010901590809226036, 0.0, 5.005122704046118e-23, 0.0, 2.139257643470968e-17, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.184804318001823e-44, 0.0, 0.0, 0.0, 0.0, 1.1769395660253258e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 5.256831059068119e-41, 0.0, 0.0, 1.8403724016677344e-36, 0.0, 8.318649179120366e-26, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 8.407790785948902e-45, 0.0, 0.0, 3.5541668260494905e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6936626854284954e-36, 1.6114932339735396e-43, 0.0, 0.0, 0.0, 8.422793087307964e-39, 0.0, 3.394474631934763e-11, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.420454033011424e-21, 0.0, 0.0, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 1.8788202872593107e-37, 0.0, 6.330481907362534e-35, 9.465757113529496e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.006492321624085e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4547863630197268e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 2.1019476964872256e-44, 0.0, 0.0, 0.0, 1.8067043967057536e-35, 9.279647426922545e-35, 0.0, 0.0, 0.0, 0.0, 6.93132268340982e-22, 3.8706666181580097e-41, 0.0, 1.1006113182561884e-14, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.790232086305765e-41, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.5305746108066265e-35, 3.026804682941605e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.880775195553491e-38, 0.0, 7.489379772430417e-41, 3.587856562087975e-40, 0.0, 2.9286521333064374e-39, 0.0, 5.083746465187627e-31, 0.0, 0.0, 9.49704810823787e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 5.008191838503214e-29, 0.0, 5.535969713161622e-41, 0.0, 0.0, 2.2420775429197073e-44, 0.0, 1.66520135723515e-24, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0550751443333051e-22, 0.0, 0.0, 0.0, 1.8357593223328195e-11, 0.0, 0.0, 1.401298464324817e-45, 0.0, 5.355998597206966e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.955646612727984e-36, 4.874252438667696e-29, 1.7212150438600192e-38, 2.056221095383905e-34, 1.350431213481794e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.225524774687894e-16, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.1055848253635385e-18, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.208973483303831e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.833314812728353e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 2.392841563138257e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.8294727714146e-29, 0.0, 0.0, 6.0228240326044215e-24, 0.0, 0.0, 0.0, 0.0, 2.942726775082116e-44, 0.0, 0.0, 0.0, 2.9106280438860316e-32, 0.0, 0.0, 1.401298464324817e-45, 9.094104734821268e-40, 0.0, 0.0, 0.0, 0.0, 8.379764816662406e-43, 0.0, 0.0, 0.0, 0.0, 2.476729174666291e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.899217915115233e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.581502978532929e-23, 0.0, 0.0, 1.373618615759009e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.833006907889938e-36, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0130872989962957e-28, 0.0, 0.0, 2.3695957031732657e-42, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3532023495783614e-34, 0.0, 0.0, 0.0, 0.0, 1.4842507371538833e-37, 0.0, 0.0, 8.96831017167883e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.817121649626316e-28, 0.0, 1.1676031161858954e-30, 0.0, 0.0, 0.0, 1.401298464324817e-44, 7.632872735177279e-41, 0.0, 0.0, 0.0, 8.688302308963487e-36, 0.0, 2.886674836509123e-43, 0.0, 8.407790785948902e-43, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.849121544802551e-39, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0828566215145976e-44, 0.0, 1.401298464324817e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.152935492761122e-38, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.6638683806814773e-42, 0.0, 7.550871762397208e-35, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.6324540717082435e-39, 0.0, 0.0, 9.462562153030835e-40, 0.0, 0.0, 5.0841148210361365e-33, 0.0, 0.0, 0.0, 0.0, 0.0, 7.09731466886868e-23, 9.388699710976274e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.353326311798882e-29, 0.0, 0.0, 1.2404416437425076e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.397144128077408e-36, 7.407974421002089e-37, 0.0, 0.0, 6.717648879727605e-36, 0.0, 1.768966895447031e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 9.112724067776445e-38, 0.0, 0.0, 4.203895392974451e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.35516232899072e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.631028190208907e-32, 0.0, 1.5686625343187035e-16, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.061042047239175e-32, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.4824508916014624e-40, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2343504288732794e-35, 9.333908941021174e-41, 0.0, 0.0, 0.0, 0.0, 6.634651746514229e-34, 0.0, 6.427894912094047e-19, 0.0, 0.0, 0.0, 8.595086123517945e-34, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6726286629855635e-38, 2.802596928649634e-45, 3.042746626482013e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 9.766383054067202e-37, 0.0, 2.4500997047364157e-28, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.910600001457514e-38, 0.0, 1.538903037316853e-29, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.055217625909588e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.324934164434305e-44, 4.8503890357496686e-37, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.961817850054744e-44, 0.0, 2.278877069701856e-28, 0.0, 6.838834762603755e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 3.7023908512904855e-38, 1.8353556361182419e-31, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3991684906590433e-40, 0.0, 0.0, 0.0, 1.787916710632034e-41, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.802596928649634e-45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5897607113956593e-38, 0.0, 0.0, 0.0, 2.4693069994495577e-24, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.605193857299268e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.496464088494466e-22, 0.0, 0.0, 0.0, 1.1493770173065279e-36, 2.1579996350602183e-43, 1.401298464324817e-45, 0.0, 0.0, 4.344025239406933e-44, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0010542070558388e-42, 0.0, 1.3745757026101428e-40, 0.0, 0.0, 2.6045107790311423e-39, 0.0, 0.0, 0.0, 0.0, 3.3743267020941595e-42, 0.0, 3.0828566215145976e-44, 0.0, 0.0, 0.0, 0.0, 1.5600179361850318e-39, 0.0, 0.0, 0.0, 0.0, 8.656002330035094e-35, 0.0, 2.420042447888959e-42, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0], [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026], [0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026, 0.00048804294783622026]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6976044178009033, -1.5158472061157227, 0.2100277692079544, 0.42401695251464844, 0.3815732002258301, -0.643405556678772, -0.8977649211883545, 1.4742765426635742, -0.20580404996871948, -0.1471862643957138, 0.6989075541496277, 2.0974721908569336, -0.2993490695953369, -0.5388371348381042, 0.9557773470878601, 0.8983821272850037, -0.6630536913871765, 0.07315444946289062, 0.6455310583114624, -0.8833655118942261, -2.163800001144409, -0.8622226119041443, 1.1012532711029053, -0.6190475821495056, -1.2486802339553833, 0.28921300172805786, 1.2756537199020386, 0.38562411069869995, -0.7048953771591187, -0.11915625631809235, -0.9023851752281189, -0.9758181571960449, 0.16654175519943237, -0.7260059118270874, -0.5770264863967896, -0.13618957996368408, -0.05792469158768654, 0.13622985780239105, -1.597974181175232, 0.5605955719947815, -1.3339931964874268, 0.30140671133995056, -0.7326043844223022, 0.15733227133750916, 0.12371503561735153, 0.39530321955680847, 0.8461267948150635, 0.34414854645729065, 1.461633563041687, 1.0611023902893066, -0.5813695192337036, -1.5271751880645752, 0.6752774715423584, 1.6052857637405396, 0.10280914604663849, -0.6147125959396362, -0.9381895661354065, 0.5793716311454773, -0.6602746248245239, 1.6444883346557617, 0.848953366279602, 1.660659670829773, 0.4782233238220215, -0.5981929302215576, -0.398374080657959, 0.1479705572128296, 0.14762040972709656, 0.5285834074020386, -0.9613871574401855, 0.6396903991699219, 0.5625585317611694, -1.6659389734268188, -1.0284239053726196, 0.6193081140518188, -0.45226985216140747, 0.715520977973938, 0.07222634553909302], [0.7779216170310974, -0.18909643590450287, -0.23486697673797607, 0.48579245805740356, 0.771953821182251, -0.07887998968362808, 2.231961727142334, 1.0411746501922607, -0.09424888342618942, -1.0287636518478394, 0.5675925612449646, -0.1157090961933136, -2.740443706512451, -1.0794293880462646, 0.26136332750320435, -1.627306342124939, 0.27552932500839233, 0.1286071389913559, -1.189565896987915, 0.3953530490398407, 0.42519134283065796, 0.21383433043956757, 0.45949774980545044, 1.1996979713439941, -0.5333204865455627, -1.0936493873596191, 1.4467010498046875, -1.1074159145355225, 0.040867071598768234, -0.9017786383628845, -0.9802924990653992, 0.20612488687038422, -0.32408323884010315, 0.6025149822235107, -0.14442461729049683, 1.7875109910964966, -0.668490469455719, 0.4060991108417511, -0.12807400524616241, -0.21099808812141418, 0.010213920846581459, -2.022808074951172, 0.462968111038208, 0.7797217965126038, -0.30421727895736694, 1.1583045721054077, -0.15693844854831696, 0.32020169496536255, -0.04454267397522926, 0.7416521906852722, 0.5125260353088379, -0.655063271522522, 0.826246440410614, -1.52482008934021, 0.6294963955879211, 0.16552843153476715, -0.14210717380046844, 0.8529902100563049, 0.3045676648616791, -0.13334117829799652, 1.4828463792800903, 0.7790729999542236, -1.9399396181106567, 0.8737785816192627, -0.9460664987564087, -0.35553812980651855, -1.0753380060195923, 1.30824875831604, 0.8122845888137817, 2.0504002571105957, -2.265106678009033, -1.6286756992340088, 1.2671457529067993, -0.47854289412498474, 0.6653746962547302, -0.37862661480903625, 0.514690637588501], [1.000436544418335, -0.558817446231842, -0.7675531506538391, -0.016572488471865654, 0.8271105885505676, -0.17515762150287628, -0.0025638809893280268, 1.331427812576294, -0.27227622270584106, 0.4149535596370697, -0.05917771905660629, 0.7623157501220703, -0.6949419975280762, -2.215031385421753, 0.5330620408058167, -0.8823159337043762, -0.19829145073890686, -0.6195881962776184, 1.1161069869995117, -0.5759096145629883, -0.8681803345680237, 0.710225522518158, 0.47154712677001953, -0.7482173442840576, 1.2250021696090698, 0.9454332590103149, -1.3531593084335327, 0.1427237093448639, 0.6269236207008362, -1.0408284664154053, 1.5063458681106567, 0.3623112142086029, 0.37270817160606384, -0.6791490912437439, -0.11292365193367004, 0.25818565487861633, -2.087367534637451, -0.13670235872268677, 0.24994602799415588, 1.1044394969940186, -1.9085416793823242, -0.8535199761390686, -1.2418620586395264, 1.605710744857788, -0.6957688331604004, 0.3047436773777008, -0.12974773347377777, 0.9756499528884888, 0.8916864395141602, 0.11655397713184357, -1.576377511024475, -1.5499368906021118, 0.09282809495925903, -1.6377288103103638, -0.45024609565734863, -0.020434364676475525, -1.1834760904312134, 1.036583423614502, 0.32275786995887756, 1.0267040729522705, 0.4869495630264282, 0.8821724057197571, -2.8250255584716797, -2.134843349456787, 1.5203285217285156, -0.41217929124832153, -0.14400364458560944, 0.11939817667007446, 0.28555285930633545, -0.14685893058776855, 0.23358571529388428, 0.01307538989931345, 1.3052352666854858, 1.6933531761169434, -0.49172893166542053, 0.4042787551879883, -0.7299262285232544], [-1.2435038089752197, 1.4839330911636353, 0.5400910377502441, -0.3927037715911865, 0.36067190766334534, -0.9962633848190308, 0.5946658849716187, -1.4215680360794067, 0.44153279066085815, -1.8203457593917847, 1.07827889919281, -0.5192613005638123, 1.3286399841308594, 2.4958724975585938, 1.7403851747512817, 1.086497187614441, 0.7320279479026794, 1.1328086853027344, -1.274891972541809, 0.586864173412323, -1.787940502166748, 0.08079630136489868, -0.9425048232078552, -0.5882256031036377, 1.586480975151062, -0.6222463846206665, 0.9126509428024292, -0.7889233231544495, -1.2709486484527588, 0.19911855459213257, -2.2960352897644043, -0.5204546451568604, 0.11449082940816879, 1.113391399383545, 1.6101816892623901, -0.45701682567596436, 1.3949403762817383, 0.55235755443573, -0.9798246026039124, -0.46875330805778503, -0.49927595257759094, -1.0947333574295044, -0.22140483558177948, -0.8156910538673401, -0.025018513202667236, -0.5049547553062439, -0.4149991273880005, -1.3764164447784424, 0.6366759538650513, -0.36787834763526917, 0.3768474757671356, -0.08772978186607361, -0.16032497584819794, 0.2525297701358795, -0.2998753488063812, -0.7374903559684753, 1.0299352407455444, -1.1348507404327393, 0.6438685059547424, -1.3544690608978271, 2.336376667022705, 0.15558625757694244, -0.5534050464630127, 0.2538798153400421, -1.6171767711639404, 0.5137333869934082, 0.6920157074928284, -0.8167852163314819, 1.1739301681518555, 0.3159199059009552, 1.702481746673584, -0.5742741823196411, -0.07475826889276505, 2.181307077407837, 0.04453099146485329, 0.6786401867866516, 0.43422865867614746], [-2.049834728240967, -0.2912953495979309, 0.19436709582805634, 1.0785174369812012, -0.4768225848674774, 0.3525863587856293, -1.059457540512085, -0.04409317299723625, -1.1968343257904053, 0.8399340510368347, 1.9165046215057373, 1.3750604391098022, -0.7581815719604492, -1.5265161991119385, -1.378976583480835, -0.220421701669693, 1.0957626104354858, -0.7896960377693176, 0.04839714244008064, 0.9903232455253601, -2.2183971405029297, 0.6151841878890991, 1.6323096752166748, -0.7674844264984131, -1.8014119863510132, -1.5672402381896973, 2.0585923194885254, 0.49538615345954895, -0.07423781603574753, -1.5174528360366821, 0.737056314945221, 0.32641974091529846, -0.6564961671829224, -0.22860106825828552, -0.6318085789680481, 0.08918794989585876, 1.8636711835861206, -0.049245115369558334, -1.526317834854126, -1.44350004196167, -0.18570682406425476, 0.3077988624572754, 0.4299106299877167, -0.031090425327420235, 1.137417197227478, -1.2476272583007812, -0.09676463901996613, -1.3264150619506836, -0.08335331827402115, -0.009277714416384697, 1.2034354209899902, -0.07189720869064331, 0.21306563913822174, -1.548718810081482, -1.3979806900024414, -1.6211345195770264, -0.8388335108757019, -0.6018869280815125, -1.4861762523651123, -0.6444334387779236, 0.9176667928695679, 0.039012860506772995, -0.6285542845726013, 1.2208744287490845, 0.004189527127891779, 0.005233116913586855, 1.8452928066253662, 0.18026533722877502, 0.32325631380081177, 0.5103498697280884, 0.660235583782196, 1.4112266302108765, 0.8290190696716309, -1.8974332809448242, -0.4370259642601013, -0.14010323584079742, 0.6184121370315552]], "dtype": "float32"}, 2.0], "e": {"__nd__": [[0.016690826043486595, 0.0055186511017382145, 0.013079824857413769, 0.014556906186044216, 0.014251235872507095, 0.008536526933312416, 0.0075170560739934444, 0.02461104467511177, 0.010624418966472149, 0.01094041671603918, 0.016701705753803253, 0.033608973026275635, 0.01013893075287342, 0.008994726464152336, 0.01899063028395176, 0.018453389406204224, 0.008453073911368847, 0.012214628048241138, 0.016261860728263855, 0.007571372203528881, 0.003991459961980581, 0.0076518370769917965, 0.020423447713255882, 0.008641129359602928, 0.0063073583878576756, 0.01360807754099369, 0.022284334525465965, 0.01428013015538454, 0.008278066292405128, 0.011094826273620129, 0.0074997106567025185, 0.00722934165969491, 0.012798499315977097, 0.008191147819161415, 0.00882460456341505, 0.011000736616551876, 0.011439756490290165, 0.012605988420546055, 0.005296625662595034, 0.015585715882480145, 0.006043966393917799, 0.013691297732293606, 0.00816416833549738, 0.012739701196551323, 0.012527354061603546, 0.014349407516419888, 0.01797749102115631, 0.013987042009830475, 0.024455957114696503, 0.020017527043819427, 0.008805463090538979, 0.005487481597810984, 0.016505535691976547, 0.026277154684066772, 0.01239708811044693, 0.008659878745675087, 0.0073666442185640335, 0.015732724219560623, 0.008464828133583069, 0.026797302067279816, 0.018002916127443314, 0.027014855295419693, 0.014956839382648468, 0.008731704205274582, 0.009649151936173439, 0.012680207379162312, 0.012677988037467003, 0.015338235534727573, 0.007281693629920483, 0.016214441508054733, 0.0156010203063488, 0.00511965760961175, 0.007041668053716421, 0.016050036996603012, 0.009392599575221539, 0.016841018572449684, 0.012208960950374603], [0.01694236323237419, 0.010446930304169655, 0.010210564360022545, 0.014639928936958313, 0.016891883686184883, 0.011038800701498985, 0.035052161663770676, 0.019325852394104004, 0.010954299010336399, 0.006865264382213354, 0.015251115895807743, 0.01083738636225462, 0.0029172301292419434, 0.006693532224744558, 0.013085936196148396, 0.005089620128273964, 0.013178952969610691, 0.01224551722407341, 0.006334896199405193, 0.013992659747600555, 0.014202984049916267, 0.012778621166944504, 0.014448711648583412, 0.02091999538242817, 0.008795112371444702, 0.0066461097449064255, 0.02366996370255947, 0.006600520107895136, 0.011719919741153717, 0.007315292954444885, 0.0070336805656552315, 0.012729457579553127, 0.009765099734067917, 0.015519757755100727, 0.01068289764225483, 0.028067488223314285, 0.008220336399972439, 0.014068045653402805, 0.010770590975880623, 0.010333151556551456, 0.011541662737727165, 0.004176411312073469, 0.014473805204033852, 0.016957620158791542, 0.00986257940530777, 0.020491469651460648, 0.010616264306008816, 0.013476632535457611, 0.01122995838522911, 0.016637887805700302, 0.01483693066984415, 0.008275710046291351, 0.017356717959046364, 0.0053572263568639755, 0.01573054865002632, 0.012473677285015583, 0.01069528330117464, 0.017590368166565895, 0.013371695764362812, 0.010742262937128544, 0.024101633578538895, 0.016952119767665863, 0.004353092983365059, 0.017774159088730812, 0.007155084051191807, 0.009612721391022205, 0.006707238964736462, 0.022086814045906067, 0.017235973849892616, 0.032010260969400406, 0.003699889872223139, 0.005086136516183615, 0.021637529134750366, 0.00903932936489582, 0.01601528748869896, 0.00950238760560751, 0.014852997846901417], [0.019802648574113846, 0.00908103957772255, 0.008181052282452583, 0.011909198947250843, 0.018158752471208572, 0.011001355946063995, 0.011992907151579857, 0.023366684094071388, 0.010479900985956192, 0.01477703358978033, 0.01165818516165018, 0.017579883337020874, 0.008483528159558773, 0.003967283759266138, 0.01567596197128296, 0.007724826689809561, 0.010874837636947632, 0.008809258230030537, 0.02098170481622219, 0.00900376308709383, 0.007779617793858051, 0.017127923667430878, 0.015201147645711899, 0.008260529488325119, 0.022155780345201492, 0.019265463575720787, 0.006104448344558477, 0.012896542437374592, 0.016429182142019272, 0.007136219646781683, 0.025502342730760574, 0.014393158257007599, 0.014468175359070301, 0.008550782687962055, 0.011349067091941833, 0.013662982732057571, 0.004228780511766672, 0.01121493335813284, 0.013606810010969639, 0.02085965871810913, 0.004624307155609131, 0.00783685315400362, 0.006453782320022583, 0.02680136263370514, 0.008480021730065346, 0.01398477517068386, 0.01125399861484766, 0.01955874264240265, 0.0187546294182539, 0.012728892266750336, 0.005459780804812908, 0.005532439798116684, 0.012578781694173813, 0.00529484124854207, 0.009587636217474937, 0.011886225081980228, 0.0066449642181396484, 0.020163804292678833, 0.014111305586993694, 0.020064447075128555, 0.015318667516112328, 0.018665624782443047, 0.002924386179074645, 0.004129580222070217, 0.025681262835860252, 0.009771868586540222, 0.011174065992236137, 0.012747006490826607, 0.0138512272387743, 0.011158124543726444, 0.013495958410203457, 0.01208705548197031, 0.02306266315281391, 0.028001945465803146, 0.009390822611749172, 0.014698372222483158, 0.008336422964930534], [0.005950459279119968, 0.023270508274435997, 0.014516208320856094, 0.009105406701564789, 0.013270668685436249, 0.006733456626534462, 0.014917771331965923, 0.005443576257675886, 0.01381820160895586, 0.004459547810256481, 0.018998494371771812, 0.008547079749405384, 0.021532000973820686, 0.03859630599617958, 0.026454145088791847, 0.019076723605394363, 0.015978343784809113, 0.019523613154888153, 0.0058578010648489, 0.014859693124890327, 0.00453239306807518, 0.011537683196365833, 0.006916901096701622, 0.008257381618022919, 0.024494798853993416, 0.008118107914924622, 0.017488539218902588, 0.0074689812026917934, 0.005869362037628889, 0.012240860611200333, 0.003515573451295495, 0.008541980758309364, 0.0117337079718709, 0.019334981217980385, 0.024786796420812607, 0.008817266672849655, 0.02225775457918644, 0.014605512842535973, 0.006789029575884342, 0.00876567605882883, 0.008632916025817394, 0.006409963592886925, 0.009919652715325356, 0.007369683589786291, 0.010943121276795864, 0.008608438074588776, 0.009004466235637665, 0.005567867308855057, 0.01523443404585123, 0.009219134226441383, 0.013378433883190155, 0.01060531660914421, 0.010227271355688572, 0.01257216464728117, 0.009537988342344761, 0.007663548458367586, 0.018544772639870644, 0.006282669026404619, 0.015289319679141045, 0.005629303865134716, 0.03563786298036575, 0.011977302841842175, 0.008402403444051743, 0.01258065365254879, 0.004936378449201584, 0.014326156117022038, 0.015661854296922684, 0.007365652825683355, 0.01992918737232685, 0.012977021746337414, 0.025957515463232994, 0.008315184153616428, 0.010674322955310345, 0.0329790934920311, 0.011330359615385532, 0.015557462349534035, 0.013767829164862633], [0.004325494170188904, 0.010420720092952251, 0.013284889981150627, 0.020670395344495773, 0.009497536346316338, 0.014378542080521584, 0.007097303867340088, 0.011791715398430824, 0.0066261678002774715, 0.018345985561609268, 0.03142790123820305, 0.02397409826517105, 0.00825115293264389, 0.005619179457426071, 0.0060493783093988895, 0.010796617716550827, 0.020849397405982018, 0.00812215730547905, 0.012349830940365791, 0.019778696820139885, 0.003975876607000828, 0.016395974904298782, 0.027264857664704323, 0.00821286253631115, 0.004897563252598047, 0.005505918990820646, 0.03374188393354416, 0.015442706644535065, 0.011615319177508354, 0.005644701886922121, 0.017426149919629097, 0.014191647991538048, 0.008681511506438255, 0.01075255312025547, 0.00878933910280466, 0.012604296207427979, 0.03060854971408844, 0.011761379428207874, 0.0056197368539869785, 0.005857329349964857, 0.010985655710101128, 0.014060131274163723, 0.014945332892239094, 0.011868627741932869, 0.02128818817436695, 0.006460005417466164, 0.011485225521028042, 0.006210467778146267, 0.011562500149011612, 0.011998779140412807, 0.02200261875987053, 0.011628921143710613, 0.01340967696160078, 0.0055571445263922215, 0.00599216902628541, 0.005359531380236149, 0.007925037294626236, 0.008921823464334011, 0.005733668804168701, 0.008734031580388546, 0.019073065370321274, 0.012292019091546535, 0.008803651668131351, 0.02219530940055847, 0.012079847045242786, 0.012086152099072933, 0.030328569933772087, 0.013191549107432365, 0.014169218949973583, 0.015558680519461632, 0.016769496724009514, 0.02441156655550003, 0.018246134743094444, 0.004667983390390873, 0.009688414633274078, 0.011239025741815567, 0.016422457993030548]], "dtype": "float32"}}, {"a": [{"__nd__": [[-3.0]], "dtype": "float32"}, 0.7], "e": {"__nd__": [[1.0]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(online_softmax, json.loads(_TESTS), 'triton', *[1e-05, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")